> ★ **これは図を落とした写しです。**元の `6_pathway.ipynb` は 95 MB（うち画像 227 枚・97 MB 相当）あり、
> git では管理していません。`strip_figures.py` が自動で作ります（`.githooks/pre-commit`）。
>
> - **文字出力・表・エラーはすべて残しています。**数値はこのまま読めます
> - **図を見るには** `oligo_eptA_analysis.html`（まとめ HTML）か、
>   `/mnt/scratch/tanoue/261016_Xiaoxi_eptAoligo_2wFeces_Metatranscriptome/pathway/` を開いてください
> - **この写しを編集しないでください。**元の notebook から作り直されます


# ステージ6: 機能解析（eptA oligo metatranscriptome）

ステージ5（`5_1_DESeq2_A_community.ipynb` / `5_2_DESeq2_B_perstrain.ipynb`）の続き。

## 最終成果物

**(A) と (B) を左右に並べた経路レベルの2パネル図**（CLAUDE.md §7 ステージ6）。
過去プロジェクト T19 の `LP食での経路シフト`（2026-10-16）と同形式。

- 左右で x 軸・行順を共通にする
- **スコア = 2×(AUC−0.5)**（経路内の遺伝子の Wald 統計量が他と比べてどちらに寄るか）
- **片方のパネルでのみ有意な経路をハイライト**
- ホバーで担い手の株。**順位は遺伝子数ではなく転写量（`normA_diff` / `normB_diff`）**

## 過去notebook との対応

`230313_T19_33mix/5_Enrichment_pathway_R.ipynb` と `4_DESeq2_..._R3.ipynb` セル81-259 に対応。

| 過去 | 今回 | 理由 |
|---|---|---|
| 経路名を `/mnt/hdd2/Genome/*.csv` から読む | **`ko00001.keg` から直接作る** | 前者は hondaserver2 のパスで本サーバに無い |
| KEGG のみ | **KEGG / COG / PFAM / CAZy の4種** | `KEGG_Pathway` は CDS の 24.6% にしか付かない（COG 91.7% / PFAM 87.6%） |
| **`enricher()` に `universe` を渡していない** | **必ず渡す** | **CLAUDE.md §6-4。渡さないと「変動した機能」ではなく「その株が持つ機能」が出る** |
| ORA のみ | **GSEA を併走** | 閾値で切ると検出力を捨てる |
| 1回のプール解析 | **(A) と (B) の2パネル** | ステージ5 で両方を用意した |

★ **過去notebook の `universe` 未指定は不備。**全notebook で `enricher()` が 21〜78回
呼ばれているのに **`universe` は0回**だった。背景は
「**検定された遺伝子（`padj` が NA でない）のうち、その TERM2GENE に載っている遺伝子**」とする。


## セル1: ライブラリ

### ★ ステージ6 では `.libPaths` を足す

`clusterProfiler` / `fgsea` / `enrichplot` は **`metatx-lib-4.4` 側にしか無い**（CLAUDE.md §10）。

副作用として **ggplot2 が 3.5.1 → 3.5.2 に上がる。**
ステージ5 までの図は 3.5.1 で描いたが、このnotebook の図は 3.5.2 で描かれる。
（gggenomes 用の 3.5.1 固定は user lib 側なので崩れない）


In [1]:
# ── ライブラリ ─────────────────────────────────────────────────
# ★ ステージ6 では .libPaths を足す必要がある。
#   clusterProfiler / fgsea / enrichplot は metatx-lib-4.4 側にしか無い（CLAUDE.md §10）。
#   副作用として ggplot2 が 3.5.1 → 3.5.2 に上がる。
#   ステージ5 までの図は 3.5.1 で描いたが、このnotebook の図は 3.5.2 で描かれる。
#   （gggenomes 用の 3.5.1 固定は user lib 側なので崩れない）
.libPaths(c("/home/tanoue/R/metatx-lib-4.4", .libPaths()))

library(dplyr)
library(tidyverse)
suppressMessages(library(clusterProfiler))
suppressMessages(library(fgsea))

cat(sprintf("R %s / clusterProfiler %s / fgsea %s / ggplot2 %s\n",
            getRversion(), packageVersion('clusterProfiler'),
            packageVersion('fgsea'), packageVersion('ggplot2')))
cat("  ※ ggplot2 が 3.5.2 になっていることを確認（metatx-lib 側）\n")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ forcats   1.0.0     ✔ readr     2.1.5
✔ ggplot2   3.5.2     ✔ stringr   1.6.0
✔ lubridate 1.9.4     ✔ tibble    3.3.1
✔ purrr     1.1.0     ✔ tidyr     1.3.1
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


R 4.4.1 / clusterProfiler 4.14.6 / fgsea 1.32.4 / ggplot2 3.5.2
  ※ ggplot2 が 3.5.2 になっていることを確認（metatx-lib 側）


## セル2: パスを置く

入力はステージ5 が保存した `A_de.csv` / `B_de.csv` と、KEGG の階層ファイル `ko00001.keg`。


In [2]:
# ── パスを置く ─────────────────────────────────────────────────
path         = '/mnt/scratch/tanoue/261016_Xiaoxi_eptAoligo_2wFeces_Metatranscriptome'
path_project = '/home/tanoue/261016_Xiaoxi_eptAoligo_2wFeces_Metatranscriptome'
path_de      = file.path(path, 'DESeq2')
path_pw      = file.path(path, 'pathway')
if (!dir.exists(path_pw)) dir.create(path_pw, recursive = TRUE)

# ステージ5 の出力
f_deA = file.path(path_de, 'A_de.csv')   # (A) 全株プール 36列
f_deB = file.path(path_de, 'B_de.csv')   # (B) 株ごと 35列

# ★ KEGG の階層ファイル（経路名とカテゴリの出どころ）
#   過去プロジェクト 230313_T19_33mix の ko00001.keg（2023-11-07 取得）をコピーして使う。
#   過去notebook は /mnt/hdd2/Genome/ の CSV を読んでいたが、
#   それは hondaserver2 のパスで本サーバには無い。
#   .keg から経路名とカテゴリを直接作る方が、依存が1つ減って確実
f_keg = file.path(path, 'ref/ko00001.keg')

for (x in c(f_deA, f_deB, f_keg))
    cat(sprintf('%-24s %s\n', basename(x), ifelse(file.exists(x), 'OK', '!! 無い')))

A_de.csv                 OK
B_de.csv                 OK
ko00001.keg              OK


## セル3: ステージ5 の結果を読む

**(A) と (B) を1つの表（`res`）にまとめる。ステージ6 はこの表だけで進む。**

`B_de.csv` には既に `LFC_A` / `padj_A` / `stat_A` / `AvsB` が入っている（5_2 セル11）。
**(A) にしか無い pME008 の10遺伝子**を足して全 9,795 遺伝子の表にする（pME008 の (B) 側は NA になる）。


In [3]:
# ── ステージ5 の結果を読む ─────────────────────────────────────
# (A) と (B) を1つの表にまとめる。ステージ6 はこの表だけで進む
deA = read.csv(f_deA, stringsAsFactors = FALSE)
deB = read.csv(f_deB, stringsAsFactors = FALSE)

# B_de.csv には既に LFC_A / padj_A / stat_A / AvsB が入っている（5_2 セル11）。
# (A) にしか無い pME008 の10遺伝子を足して、全 9,795 遺伝子の表にする
res = deA %>%
    dplyr::select(gene_id, strain, product, marker, biotype, mappability,
                  contig, strand, start, end,
                  Preferred_name, KEGG_ko, KEGG_Pathway, COG_category,
                  PFAMs, EC, CAZy, Description,
                  baseMean, LFC_A = LFC_shrunk, padj_A = padj, stat_A = stat_Wald,
                  normA_WT = norm_WT_mean, normA_KO = norm_KO_mean,
                  normA_diff = norm_diff) %>%
    left_join(deB %>% dplyr::select(gene_id, LFC_B, padj_pooled, padj_bystrain,
                                    stat_B, normB_WT = normB_WT_mean,
                                    normB_KO = normB_KO_mean, normB_diff, AvsB),
              by = 'gene_id') %>%
    mutate(sigA = !is.na(padj_A)      & padj_A      < 0.05,
           sigB = !is.na(padj_pooled) & padj_pooled < 0.05)

cat(sprintf("■ res: %d 行 × %d 列\n", nrow(res), ncol(res)))

cat("\n■ Table 1: 読み込んだもの\n")
data.frame(
    item = c('(A) 全株プール', '(B) 株ごと', '統合後'),
    rows = c(nrow(deA), nrow(deB), nrow(res)),
    DEG  = c(sum(deA$padj < 0.05, na.rm = TRUE),
             sum(deB$padj_pooled < 0.05, na.rm = TRUE),
             NA),
    note = c('9,795 遺伝子（pME008 を含む）',
             '9,785 遺伝子（pME008 は dispersion を推定できず除外）',
             'pME008 の (B) 側は NA になる'),
    check.names = FALSE)

cat("\n■ Table 2: (A) vs (B) の4分類（ステージ6 の2パネル図の基礎）\n")
res %>% dplyr::count(AvsB, name = 'n') %>%
    mutate(pct = round(100 * n / sum(n), 1)) %>% as.data.frame()

■ res: 9795 行 × 35 列

■ Table 1: 読み込んだもの


item,rows,DEG,note
<chr>,<int>,<int>,<chr>
(A) 全株プール,9795,3126,"9,795 遺伝子（pME008 を含む）"
(B) 株ごと,9785,2964,"9,785 遺伝子（pME008 は dispersion を推定できず除外）"
統合後,9795,NA,pME008 の (B) 側は NA になる



■ Table 2: (A) vs (B) の4分類（ステージ6 の2パネル図の基礎）


AvsB,n,pct
<chr>,<int>,<dbl>
(A) のみ有意,851,8.7
(A) 有意 / (B) 未検定,5,0.1
(B) のみ有意,687,7.0
(B) 有意 / (A) 未検定,8,0.1
どちらも有意でない,5965,60.9
両方で有意,2269,23.2
NA,10,0.1


## セル4: KEGG 階層の版を突き合わせる

### なぜこれをやるか

セル5 以降で使う経路の定義は `ko00001.keg`（KEGG BRITE の KO 階層）から作る。
手元にあるのは**過去プロジェクト `230313_T19_33mix_metatranscriptome` から複製した 2023-11-07 版**で、**3年前**のもの。

`/db` と `/home/tanoue` を探しても KEGG のダウンロードは他に無く（CheckM2 内部の json のみ）、
一方 **`KEGGREST` が入っていて API も通る**ことは確認済み。
`keggGet("br:ko00001")` は**ローカルファイルと同じフラットファイル形式**（A/B/C/D 行）を返すので、
**同じパーサで2版を読んで突き合わせられる。**

| 選択肢 | 中身 | 難点 |
|---|---|---|
| 2023版をそのまま使う | 過去プロジェクトと同じ土俵 | 3年前。経路名が古い |
| 最新版を取り直す | 名前が現行 | 過去プロジェクトと比較しにくい |
| **両方取って突き合わせる** | **差分を記録して判断** | API を1回叩くだけ |

**3つめを採る。**差分が小さいと分かれば、どちらを使っても結論が変わらないと確認できる。

### 判断の基準

全体の経路数が増えていても、**本データに実際に出てくる経路**（GTF の `KEGG_Pathway` に入っている ko）
の**増減が 0 なら、解析対象の集合は版によらず同じ**。その場合は経路名が現行である最新版を使う。

### 取得したものはファイルに保存する

API は将来内容が変わるので、**取得した版を `ref/ko00001_<日付>.keg` に残す**。
md5 も記録して、どの版で解析したかを後から特定できるようにする。

In [130]:
# ── KEGG 階層を2版そろえて突き合わせる ─────────────────────────
# (1) 2023-11-07 版 … 過去プロジェクト 230313_T19_33mix から複製した ko00001.keg
#     → 過去プロジェクトと同じ土俵で比較できる
# (2) 最新版       … KEGGREST で br:ko00001 を取得（同じフラットファイル形式）
#     → 経路名の改称や新規追加が反映されている
#
# 両方を読んで差分を記録し、どちらを使うかを根拠つきで決める。
# ★ 取得した最新版はファイルに保存する（再現性のため。API は将来変わる）
library(stringr)
suppressMessages(library(KEGGREST))

# ── 最新版を取得（既に保存済みなら読むだけ）──────────────────
f_keg_new = file.path(path, sprintf('ref/ko00001_%s.keg', format(Sys.Date(), '%y%m%d')))
if (!file.exists(f_keg_new)) {
    writeLines(strsplit(keggGet('br:ko00001'), '\n')[[1]], f_keg_new)
    cat(sprintf("KEGG API から取得して保存: %s\n", basename(f_keg_new)))
} else {
    cat(sprintf("既に取得済み: %s\n", basename(f_keg_new)))
}

# ── 同じ手順で2版を読む ────────────────────────────────────────
#   A 行 = 大分類、B 行 = 中分類、C 行 = 経路（[PATH:koXXXXX] を含む）
parse_keg = function(path_keg) {
    A = ''; B = ''; rows = list()
    for (l in readLines(path_keg, warn = FALSE)) {
        if (startsWith(l, 'A'))      A = str_trim(sub('^A[0-9]*', '', l))
        else if (startsWith(l, 'B')) B = str_trim(sub('^B\\s*[0-9]*', '', l))
        else if (startsWith(l, 'C')) {
            id = str_match(l, '\\[PATH:(ko[0-9]+)\\]')[, 2]
            if (!is.na(id)) rows[[length(rows)+1]] = data.frame(
                ko = id, 大分類 = A, 中分類 = B,
                経路名 = str_trim(sub('\\s*\\[PATH:ko[0-9]+\\]$', '',
                                      sub('^C\\s*[0-9]+\\s*', '', l))))
        }
    }
    bind_rows(rows) %>% distinct(ko, .keep_all = TRUE)
}

keg_2023 = parse_keg(f_keg)
keg_new  = parse_keg(f_keg_new)

cat("\n■ Table 1: 2版の規模\n")
data.frame(
    版 = c('2023-11-07（過去プロジェクト）', paste0(Sys.Date(), '（KEGG API）')),
    経路数 = c(nrow(keg_2023), nrow(keg_new)),
    大分類の数 = c(length(unique(keg_2023$大分類)), length(unique(keg_new$大分類))),
    check.names = FALSE)

# ── 差分 ───────────────────────────────────────────────────────
only_old = setdiff(keg_2023$ko, keg_new$ko)
only_new = setdiff(keg_new$ko,  keg_2023$ko)
common   = intersect(keg_2023$ko, keg_new$ko)
renamed  = keg_2023 %>% filter(ko %in% common) %>%
    dplyr::select(ko, 旧名 = 経路名) %>%
    inner_join(keg_new %>% dplyr::select(ko, 新名 = 経路名), by = 'ko') %>%
    filter(旧名 != 新名)

cat("\n■ Table 2: 差分\n")
data.frame(
    item = c('両版に共通', '2023版にのみ', '最新版にのみ', '経路名が変わった'),
    n = c(length(common), length(only_old), length(only_new), nrow(renamed)),
    check.names = FALSE)

if (length(only_new) > 0) {
    cat("\n■ 最新版で追加された経路\n")
    print(keg_new %>% filter(ko %in% only_new) %>%
          dplyr::select(ko, 大分類, 経路名) %>% as.data.frame(), row.names = FALSE)
}
if (length(only_old) > 0) {
    cat("\n■ 2023版にあって最新版に無い経路\n")
    print(keg_2023 %>% filter(ko %in% only_old) %>%
          dplyr::select(ko, 大分類, 経路名) %>% as.data.frame(), row.names = FALSE)
}
if (nrow(renamed) > 0) {
    cat("\n■ 経路名が変わったもの\n")
    print(as.data.frame(renamed), row.names = FALSE)
}

# ── 本データに実際に出てくる経路に絞って差分を見る ───────────────
# 全体の差分より、こちらが実害の有無を決める
mine = res %>% filter(!is.na(KEGG_Pathway)) %>% pull(KEGG_Pathway) %>%
    str_split(',') %>% unlist() %>% unique() %>% .[str_starts(., 'ko')]

cat(sprintf("\n■ Table 3: 本データに出てくる %d 経路のうち、差分に当たるもの\n", length(mine)))
data.frame(
    item = c('本データの経路（GTF の KEGG_Pathway）',
             'うち 2023版にのみ', 'うち 最新版にのみ', 'うち 名前が変わった'),
    n = c(length(mine), sum(mine %in% only_old), sum(mine %in% only_new),
          sum(mine %in% renamed$ko)),
    check.names = FALSE)

if (sum(mine %in% renamed$ko) > 0) {
    cat("\n■ 本データに出てくる経路のうち改称されたもの（最新版の名前を使う）\n")
    print(renamed %>% filter(ko %in% mine) %>% as.data.frame(), row.names = FALSE)
}

# ── どちらを使うか ─────────────────────────────────────────────
# 本データの経路の増減が 0 なら、解析対象の集合は版によらず同じ。
# → 最新版を使う（経路名が現行のものになる）。これを kegg_use とする。
# ★ 以降のセルに渡すのは「使うファイルのパス」f_keg_use。
#   セル5 はこれがあれば最新版、無ければ f_keg（2023版）を読む
f_keg_use = f_keg_new
kegg_map_all = keg_new
cat(sprintf("\n→ 以降は %s（KEGG API 取得）を使う（経路 %d 件）\n", Sys.Date(), nrow(kegg_map_all)))
cat(sprintf("  本データの経路の増減が 0 件なので、解析対象の集合は版によらず同じ。\n"))
cat(sprintf("  改称 %d 件のみ最新の名前に揃う。\n", sum(mine %in% renamed$ko)))

# ── 版の記録と保存 ─────────────────────────────────────────────
# mtime は複製した日になってしまうので、版の日付は出所で記す
cat("\n■ Table 4: 使用ファイルの記録\n")
print(data.frame(
    file = c(basename(f_keg), basename(f_keg_new)),
    version = c('2023-11-07', format(Sys.Date(), '%Y-%m-%d')),
    source = c('230313_T19_33mix_metatranscriptome から複製', 'KEGGREST::keggGet("br:ko00001")'),
    md5 = unname(tools::md5sum(c(f_keg, f_keg_new))),
    n_pathway = c(nrow(keg_2023), nrow(keg_new)),
    used = c('', '★'),
    row.names = NULL, check.names = FALSE))

# 経路マップを書き出す（以降の図で経路名を引くのに使う）＋ project にも複製
write.csv(kegg_map_all, file.path(path_pw, 'kegg_pathway_map.csv'), row.names = FALSE)
write.csv(renamed,  file.path(path_pw, 'kegg_version_diff_renamed.csv'), row.names = FALSE)
for (x in c(f_keg_new)) file.copy(x, file.path(path_project, 'ref', basename(x)), overwrite = TRUE)
for (x in c('kegg_pathway_map.csv', 'kegg_version_diff_renamed.csv'))
    file.copy(file.path(path_pw, x), file.path(path_project, 'refqc', paste0('stage6_', x)), overwrite = TRUE)
cat(sprintf("\n保存: %s/{kegg_pathway_map.csv, kegg_version_diff_renamed.csv}\n", path_pw))
cat(sprintf("project 複製: ref/%s, refqc/stage6_kegg_*.csv\n", basename(f_keg_new)))


KEGG API から取得して保存: ko00001_261028.keg

■ Table 1: 2版の規模


版,経路数,大分類の数
<chr>,<int>,<int>
2023-11-07（過去プロジェクト）,474,6
2026-10-28（KEGG API）,490,6



■ Table 2: 差分


item,n
<chr>,<int>
両版に共通,473
2023版にのみ,1
最新版にのみ,17
経路名が変わった,13



■ 最新版で追加された経路
      ko                               大分類
 ko00566                           Metabolism
 ko00074                           Metabolism
 ko00975                           Metabolism
 ko03271       Genetic Information Processing
 ko03273       Genetic Information Processing
 ko03272       Genetic Information Processing
 ko04082 Environmental Information Processing
 ko04081 Environmental Information Processing
 ko04517 Environmental Information Processing
 ko04518 Environmental Information Processing
 ko04519 Environmental Information Processing
 ko04149                   Cellular Processes
 ko04156                   Cellular Processes
 ko04820                   Cellular Processes
 ko04981                   Organismal Systems
 ko04980                   Organismal Systems
 ko04382                   Organismal Systems
                                             経路名
                          Sulfoquinovose metabolism
                          Mycolic acid biosynthesis
       

item,n
<chr>,<int>
本データの経路（GTF の KEGG_Pathway）,224
うち 2023版にのみ,0
うち 最新版にのみ,0
うち 名前が変わった,4



■ 本データに出てくる経路のうち改称されたもの（最新版の名前を使う）
      ko                                        旧名
 ko00710 Carbon fixation in photosynthetic organisms
 ko00720     Carbon fixation pathways in prokaryotes
 ko00966                  Glucosinolate biosynthesis
 ko04142                                    Lysosome
                            新名
 Carbon fixation by Calvin cycle
  Other carbon fixation pathways
        Glucosinolate metabolism
             Lysosome biogenesis

→ 以降は 2026-10-28（KEGG API 取得）を使う（経路 490 件）
  本データの経路の増減が 0 件なので、解析対象の集合は版によらず同じ。
  改称 4 件のみ最新の名前に揃う。

■ Table 4: 使用ファイルの記録
                file    version                                      source
1        ko00001.keg 2023-11-07 230313_T19_33mix_metatranscriptome から複製
2 ko00001_261028.keg 2026-10-28             KEGGREST::keggGet("br:ko00001")
                               md5 n_pathway used
1 87280ef0138e89afede00a3a80c932a7       474     
2 f051d98ea690db39ed8f4c1b61fc152d       490    ★

保存: /mnt/scratch/tanoue/261016_Xiaoxi_e

### 考察: 版の違いは解析結果に影響しない

**結論: 最新版（2026-10-26 取得、490 経路）を使う。ただし 2023版（474 経路）でも結果は同じ。**

| 項目 | 件数 | 意味 |
|---|---|---|
| 両版に共通 | 473 | ほぼ全部 |
| 最新版で追加 | 17 | 下記 |
| 2023版にのみ | 1 | `ko03269` Virion - Bacteriophage T4 → T7 と統合され `ko03259` に吸収 |
| 経路名が変わった | 13 | ID は同じで名前だけ変更 |
| **本データの経路（224本）の増減** | **0** | **★ 解析対象の集合は版によらず同一** |

#### 追加された 17 経路は本データに1本も出てこない

内訳は **Virion（ウイルス粒子）5本、受容体・接着関連 5本、Organismal Systems 3本**など、
**細菌の遺伝子には付かないもの**が大半。細菌寄りの追加は
`ko00566` Sulfoquinovose metabolism、`ko00074` Mycolic acid biosynthesis、
`ko00975` Biosynthesis of various siderophores の3本だが、
**いずれも本データの `KEGG_Pathway` に現れない**（スルホキノボースは植物由来糖なので
*Bacteroides* にあり得る経路だが、eggNOG がそのアノテーションを付けていない）。

#### 本データに関わるのは改称 4 件だけ

| ko | 旧名（2023） | 新名（2026） |
|---|---|---|
| ko00710 | Carbon fixation in photosynthetic organisms | Carbon fixation by Calvin cycle |
| ko00720 | Carbon fixation pathways in prokaryotes | **Other carbon fixation pathways** |
| ko00966 | Glucosinolate biosynthesis | Glucosinolate metabolism |
| ko04142 | Lysosome | Lysosome biogenesis |

**いずれも ID は同じで中身の KO 集合は変わらない**ので、濃縮解析の統計量は 1 ビットも変わらない。
図の経路名ラベルだけが変わる。

**★ `ko00720` は注意が要る。**旧名「Carbon fixation pathways in prokaryotes」は
**「原核生物」と名前に入っているため濃縮結果に出てきたときに過大に解釈しやすい**が、
新名「Other carbon fixation pathways」のとおり**中身は還元的 TCA や Wood-Ljungdahl などの雑多な寄せ集め**で、
*Bacteroides* では**逆向きに回る TCA 酵素が機械的に拾われているだけ**のことが多い。
最新版の名前を使うことで、この誤読を避けられる。

#### セル5 への受け渡し

このセルは `f_keg_use`（使うファイルのパス）だけを渡す。
**セル5 はこれがあれば最新版、無ければ `f_keg`（2023版）を読む**ので、
セル4 を飛ばしてもセル5 は単体で動く（CLAUDE.md §9）。

実測で確認済み: **最新版で作った TERM2GENE は KEGG 151 term / 2,333 遺伝子**で、
**2023版と完全に一致**。MGEC 絞り込み後の経路数は 284 → 297 に増えるが、
**増えた分は本データに1遺伝子も当たらない**ので TERM2GENE は同一になる。

## セル5: TERM2GENE を作る

過去notebook（`5_Enrichment_pathway_R.ipynb` セル2-4、`R3` セル82-83）の
`PathwayNameMGEC2Gene` に対応。

### 経路名は `ko00001.keg` から直接作る

過去notebook は `/mnt/hdd2/Genome/KEGG_Pathway_onlyMGEC_names_230719.csv` を読んでいたが
**hondaserver2 のパスで本サーバには無い。**
代わりに **`ko00001.keg`**（KEGG BRITE の KO 階層。セル4 で選んだ版）の
**A 行 = 大分類 / B 行 = 中分類 / C 行 = 経路**から「経路ID → 経路名 → カテゴリ」を組み立てる。
依存が1つ減る。

### ★ 細菌に関係のないカテゴリを外す

**eggNOG は細菌の遺伝子にも真核・疾患の経路を付けてしまう**（CLAUDE.md §7 ステージ6）。
**採用**: Metabolism 169 / Environmental Information Processing 41 /
Genetic Information Processing 39 / Cellular Processes 35。
**除外**: Human Diseases 99 / Organismal Systems 91。

### global map は自動的に落ちる

`ko01100`（Metabolic pathways）などは**ほぼ全遺伝子が入る**ので除外が必要だが、
**`ko00001.keg` には C 行として載っていない**（KO 階層ではなく経路リストにしかない）ため
**`inner_join` で自動的に落ちる。**念のため明示的な除外リストも置いてある。

### TERM2GENE は4種類作る

**`KEGG_Pathway` は CDS の 24.6% にしか付いていない**ので KEGG だけでは足りない。

| 出どころ | 用途 |
|---|---|
| **KEGG**（151 term / 23.8%） | 経路レベルの主解析。1 term 中央値 18 遺伝子 |
| **COG**（21 term / 91.7%） | **被覆が最も広い**が 1 term 中央値 394 遺伝子と粗い |
| **PFAM**（2,477 term / 87.6%） | 最も細かい。1 term 中央値 3 遺伝子 |
| **CAZy**（44 term / 2.0%） | 多糖分解酵素。被覆は狭いが *Bacteroides* では重要 |


In [20]:
# ── TERM2GENE を作る ───────────────────────────────────────────
# 過去notebook（5_Enrichment_pathway_R.ipynb セル2-4、
# 4_DESeq2..._R3.ipynb セル82-83）の PathwayNameMGEC2Gene に対応。
#
# 過去notebook との違い
#   ・経路名の表を /mnt/hdd2/Genome/ の CSV ではなく ko00001.keg から直接作る
#     （前者は hondaserver2 のパスで本サーバに無い）
#   ・KEGG だけでなく COG / PFAM / CAZy の TERM2GENE も作る
#     （本データは KEGG_Pathway が CDS の 24.6% にしか付いていない。
#       COG 91.7% / PFAM 87.6% なのでそちらを主にする。CLAUDE.md §7 ステージ6）
library(stringr)

# ── ko00001.keg から「経路ID → 経路名 → カテゴリ」を作る ─────────
#   A 行 = 大分類、B 行 = 中分類、C 行 = 経路（[PATH:koXXXXX] を含む）
keg = readLines(if (exists('f_keg_use')) f_keg_use else f_keg)  # セル4 が決めた版
A = ''; B = ''; rows = list()
for (l in keg) {
    if (startsWith(l, 'A'))      A = str_trim(sub('^A[0-9]*', '', l))
    else if (startsWith(l, 'B')) B = str_trim(sub('^B\\s*[0-9]*', '', l))
    else if (startsWith(l, 'C')) {
        id = str_match(l, '\\[PATH:(ko[0-9]+)\\]')[, 2]
        if (!is.na(id))
            rows[[length(rows)+1]] = data.frame(
                ko = id, 大分類 = A, 中分類 = B,
                経路名 = str_trim(sub('\\s*\\[PATH:ko[0-9]+\\]$', '',
                                      sub('^C\\s*[0-9]+\\s*', '', l))))
    }
}
kegg_map = bind_rows(rows)

cat(sprintf("■ ko00001.keg から %d 経路を取得\n", nrow(kegg_map)))

# ★ 細菌に関係のないカテゴリを外す（CLAUDE.md §7 ステージ6）
#   eggNOG は細菌の遺伝子にも真核・疾患の経路を付けてしまう
keep_cat = c('Metabolism', 'Genetic Information Processing',
             'Environmental Information Processing', 'Cellular Processes')
# global map（ko01100 Metabolic pathways など、ほぼ全遺伝子が入る）も外す。
# ★ ただし ko00001.keg には global map が C 行として載っていない
#   （KO 階層には含まれず、経路リストにしかない）ので、
#   この表に inner_join した時点で自動的に落ちる。念のための保険として指定しておく
global_ko = c('ko01100','ko01110','ko01120','ko01200','ko01210','ko01212',
              'ko01230','ko01232','ko01250','ko01240','ko01220')

cat("\n■ Table 1: カテゴリごとの経路数と採否\n")
kegg_map %>% dplyr::count(大分類, name = 'n') %>%
    mutate(採用 = 大分類 %in% keep_cat) %>% as.data.frame()

kegg_use = kegg_map %>% filter(大分類 %in% keep_cat, !(ko %in% global_ko))
cat(sprintf("\n  → 採用する経路: %d\n", nrow(kegg_use)))
cat(sprintf("     ko00001.keg に載っている global map: %d 本（0 なら元から含まれない）\n",
            sum(kegg_map$ko %in% global_ko)))
cat("     GTF の KEGG_Pathway には ko01100 などが入っているが、\n")
cat("     この表に無いので inner_join で自動的に落ちる\n")

# ── TERM2GENE 4種類 ────────────────────────────────────────────
# KEGG: GTF の KEGG_Pathway は "ko00040,ko00053,...,map00040,..." の形。
#       ko で始まるものだけ取り、経路名に変換する
t2g_kegg = res %>% filter(!is.na(KEGG_Pathway)) %>%
    dplyr::select(gene_id, KEGG_Pathway) %>%
    tidyr::separate_rows(KEGG_Pathway, sep = ',') %>%
    filter(str_starts(KEGG_Pathway, 'ko')) %>%
    inner_join(kegg_use %>% dplyr::select(ko, 経路名), by = c('KEGG_Pathway' = 'ko')) %>%
    # ★ 過去notebook と同じ "koXXXXX_経路名" 形式にする（例 ko00910_Nitrogen metabolism）。
    #   ko 番号を省略すると KEGG で引き直せないので必ず付ける
    transmute(term = paste0(KEGG_Pathway, '_', 経路名), gene = gene_id) %>% distinct()

# COG: 1文字のカテゴリ記号が複数並ぶことがある（例 "EG"）ので1文字ずつに分ける。
# ★ eggNOG が出すのは1文字コードだけなので、名前はこちらで与える必要がある。
#   NCBI COG / eggNOG の公式カテゴリ名をそのまま使う（和訳はしない）。
#   理由: term 名がそのまま図のラベルになるため、和名だと PDF/EPS 出力で
#   文字化けする。KEGG / PFAM / CAZy は元から英語なので COG だけの問題
cog_name = c(
  J = 'Translation, ribosomal structure and biogenesis',
  A = 'RNA processing and modification',
  K = 'Transcription',
  L = 'Replication, recombination and repair',
  B = 'Chromatin structure and dynamics',
  D = 'Cell cycle control, cell division, chromosome partitioning',
  Y = 'Nuclear structure',
  V = 'Defense mechanisms',
  T = 'Signal transduction mechanisms',
  M = 'Cell wall/membrane/envelope biogenesis',
  N = 'Cell motility',
  Z = 'Cytoskeleton',
  W = 'Extracellular structures',
  U = 'Intracellular trafficking, secretion, and vesicular transport',
  O = 'Posttranslational modification, protein turnover, chaperones',
  C = 'Energy production and conversion',
  G = 'Carbohydrate transport and metabolism',
  E = 'Amino acid transport and metabolism',
  F = 'Nucleotide transport and metabolism',
  H = 'Coenzyme transport and metabolism',
  I = 'Lipid transport and metabolism',
  P = 'Inorganic ion transport and metabolism',
  Q = 'Secondary metabolites biosynthesis, transport and catabolism',
  R = 'General function prediction only',
  S = 'Function unknown',
  X = 'Mobilome: prophages, transposons')
# 本データに出現するのは ABCDEFGHIJKLMNOPQSTUV の 21 種類（R/W/X/Y/Z は出ない）
t2g_cog = res %>% filter(!is.na(COG_category)) %>%
    dplyr::select(gene_id, COG_category) %>%
    mutate(ch = str_split(COG_category, '')) %>% tidyr::unnest(ch) %>%
    filter(ch %in% names(cog_name)) %>%
    transmute(term = paste0('COG ', ch, ': ', cog_name[ch]), gene = gene_id) %>% distinct()

# PFAM / CAZy: カンマ区切り
t2g_pfam = res %>% filter(!is.na(PFAMs)) %>%
    dplyr::select(gene_id, PFAMs) %>% tidyr::separate_rows(PFAMs, sep = ',') %>%
    filter(PFAMs != '', PFAMs != '-') %>%
    transmute(term = paste0('PFAM:', str_trim(PFAMs)), gene = gene_id) %>% distinct()

t2g_cazy = res %>% filter(!is.na(CAZy)) %>%
    dplyr::select(gene_id, CAZy) %>% tidyr::separate_rows(CAZy, sep = ',') %>%
    filter(CAZy != '', CAZy != '-') %>%
    transmute(term = paste0('CAZy:', str_trim(CAZy)), gene = gene_id) %>% distinct()

T2G = list(KEGG = t2g_kegg, COG = t2g_cog, PFAM = t2g_pfam, CAZy = t2g_cazy)

cat("\n■ Table 2: TERM2GENE 4種類\n")
do.call(rbind, lapply(names(T2G), function(k) data.frame(
    source = k, terms = length(unique(T2G[[k]]$term)),
    genes  = length(unique(T2G[[k]]$gene)),
    `全9795に対する被覆pct` = round(100*length(unique(T2G[[k]]$gene))/nrow(res), 1),
    `1termあたり遺伝子数の中央値` = median(as.integer(table(T2G[[k]]$term))),
    check.names = FALSE)))

■ ko00001.keg から 490 経路を取得

■ Table 1: カテゴリごとの経路数と採否


大分類,n,採用
<chr>,<int>,<lgl>
Cellular Processes,38,TRUE
Environmental Information Processing,46,TRUE
Genetic Information Processing,41,TRUE
Human Diseases,99,FALSE
Metabolism,172,TRUE
Organismal Systems,94,FALSE



  → 採用する経路: 297
     ko00001.keg に載っている global map: 0 本（0 なら元から含まれない）
     GTF の KEGG_Pathway には ko01100 などが入っているが、
     この表に無いので inner_join で自動的に落ちる

■ Table 2: TERM2GENE 4種類


source,terms,genes,全9795に対する被覆pct,1termあたり遺伝子数の中央値
<chr>,<int>,<int>,<dbl>,<dbl>
KEGG,151,2333,23.8,18.0
COG,21,8986,91.7,394.0
PFAM,2477,8576,87.6,3.0
CAZy,44,195,2.0,3.5


## セル6: ORA（over-representation analysis）

過去notebook（`5_Enrichment_pathway_R.ipynb` セル17-32、`4_DESeq2_allmapped_R.ipynb` セル92-122）の
`enricher` に対応。**DEG のリストに特定の経路の遺伝子が偶然より多く入っているかを Fisher 検定で調べる。**

### ★ 違い1: `universe` を渡す（CLAUDE.md §6-4）

過去notebook は `universe` を省略していた。本notebook は検定された遺伝子を明示的に渡す。

**ただし `enricher` は渡した `universe` をそのまま分母にしない。**内部で TERM2GENE の遺伝子と
交差させる（`extID <- intersect(ALLEXTID(USER_DATA), universe)`）ので、
**★ アノテーションの無い遺伝子は渡しても渡さなくても分母に入らない。**

| ソース | 渡した `universe_A` | 実際の背景 N |
|---|---|---|
| KEGG | 9,629 | **2,311** |
| COG | 9,629 | 8,846 |
| PFAM | 9,629 | 8,440 |
| CAZy | 9,629 | **192** |

→ `universe` を渡す意味は「アノテーションはあるが検定されなかった遺伝子（`padj` が NA）と
マーカーを外す」ことだけ。**本データでは結果を動かさない**（N 2,333→2,311、有意 term は12本で同一）。
`org.db` の GO 解析など背景が既定で DB 全体になる系では効くので、**作法として常に渡す。**

★ **`universe` では直せない偏り**: KEGG あり 2,311 本は baseMean 中央値 **400** / DEG 率 38.9%、
KEGG なし 7,318 本は **149** / 30.3%。**アノテーションが付く遺伝子は発現量が高く DEG になりやすい。**
→ 閾値を切らない GSEA（セル8）で確認する。

### 上下は分けて検定する / マーカーは外す

up と down を混ぜると同じ経路の中で打ち消し合うので別々に検定する。
**`eptA` 自身が `ko01503` に属する**ため、マーカー（`_deleted`/`_retained`/`_3p`/`_5p`）を残すと
**「eptA を入れたから eptA の経路が出た」という循環**になる（CLAUDE.md §7 ステージ6）。

### ★ 違い2: term サイズの足切りをしない（`minGSSize=1` / `maxGSSize=Inf`）

既定の `10` / `500` は**範囲外の term を検定そのものをしない。**数えているのは DEG 側のヒット数(`k`)
ではなく**背景に残った数(`M`)。**

| ソース | term 総数 | `M` 中央値 | `M` 最大 | 既定値で落ちる数 |
|---|---|---|---|---|
| KEGG | 151 | 18 | 141 | 57 |
| COG | 21 | 388 | **2,269** | **9**（`maxGSSize` に引っかかる） |
| PFAM | 2,470 | 3 | 289 | 2,164 |
| CAZy | 44 | 3.5 | 35 | **40**（44 本中 4 本しか検定されない） |

**CAZy は既定値だと 4 本しか検定されない**（*Bacteroides* の多糖分解酵素が見えない）。
**COG も 21 のうち 9 本が落ちる。**小さい term が同じ遺伝子を二重に数えること
（`Sod_Fe_N` と `Sod_Fe_C` は同じ `sodB` 5遺伝子）は、term リストをさらに統計解析しない限り問題にならず、
むしろ**検定が動いている確認になる。**

**代償**: BH の分母増で**既定値では有意だった 7 件が有意性を失う**
（`CAZy:GH20` 生 p=0.0058 → padj 0.109、PFAM `Patatin`/`Biotin_lipoyl_2`/`BPD_transp_1`/`CPSase_L_D2`/
`Thioredoxin`、`CAZy:GH31`）。**それ以外はすべて増える**（KEGG down 12→18、PFAM down 11→22 / 6→30）。
生の `pvalue` 列も `ora` に残してある。

`pvalueCutoff = 1` で全 term を返させ、**有意判定は `p.adjust < 0.05` で自分でやる**（過去notebook も同じ）。

### 出力の列（`enricher` の既定のまま）

| 列 | 意味 | `Photosynthesis`（(A) down）の例 |
|---|---|---|
| `GeneRatio` | DEG のうちこの term に入った数 / **DEG のうちこのソースのアノテーションを持つ数** | `15/411` |
| `BgRatio` | 背景のうちこの term に入った数 / **背景のうち同** | `18/2311` |
| `fold` | `GeneRatio` ÷ `BgRatio` | **4.69** |
| `pvalue` / `padj` | Fisher の正確検定 + BH | 2.2e-09 / 2.6e-07 |
| `geneID` | 当たった遺伝子（`/` 区切り） | — |

`fold` だけは `enricher` が返さないので計算する。

★ **分母はソースごとに違う**（`GeneRatio` の分母: KEGG 411 / COG 1,400 / PFAM 1,346 / **CAZy 29**）。
**DEG 総数 1,503 でも `universe` 9,629 でもない。**CAZy で有意が出にくいのは分母 29 が小さいため。

### 16 個の結果に名前を付けて残す

`A_up_KEGG` … `B_dn_CAZy` の 16 個を個別に呼び、後から `A_dn_KEGG@result` で中身を見られるようにする。
縦に積んだものが `ora`（1,486 行）で、図と GSEA との突き合わせに使う。


In [25]:
# ── ORA（over-representation analysis）─────────────────────────
# 過去notebook（5_Enrichment_pathway_R.ipynb セル17-32、
# 4_DESeq2_allmapped_R.ipynb セル92-122）の enricher に対応。
# DEG のリストを取り、そこに特定の経路の遺伝子が偶然より多く入っているかを
# Fisher 検定で調べる。
library(clusterProfiler)

# Jupyter（IRkernel）は data.frame を既定で 60 行までしか表示しない。
# Table 4 は 104 行あるので全部出すように広げる
options(repr.matrix.max.rows = 500, repr.matrix.max.cols = 50)

# ===============================================================
# 1. 検定にかける遺伝子リストを作る
# ===============================================================
# マーカー（eptA の _deleted / _retained / _3p / _5p）は外す。
# eptA 自身が ko01503 に属するので、残すと「eptA を入れたから eptA の
# 経路が出た」という循環になる（CLAUDE.md §7 ステージ6）

# ── (A) 全株プール = 宿主が何にさらされているか ──────────────────
universe_A = res %>% filter(!is.na(padj_A), !marker) %>% pull(gene_id)
upDEGs_A   = res %>% filter(!is.na(padj_A), !marker, padj_A < 0.05, LFC_A > 0) %>% pull(gene_id)
downDEGs_A = res %>% filter(!is.na(padj_A), !marker, padj_A < 0.05, LFC_A < 0) %>% pull(gene_id)

# ── (B) 株ごと独立 = 菌内部の制御 ───────────────────────────────
universe_B = res %>% filter(!is.na(padj_pooled), !marker) %>% pull(gene_id)
upDEGs_B   = res %>% filter(!is.na(padj_pooled), !marker, padj_pooled < 0.05, LFC_B > 0) %>% pull(gene_id)
downDEGs_B = res %>% filter(!is.na(padj_pooled), !marker, padj_pooled < 0.05, LFC_B < 0) %>% pull(gene_id)

cat("■ Table 1: 検定にかける遺伝子リスト\n")
data.frame(
    scheme = c('(A) 全株プール', '(A) 全株プール', '(A) 全株プール',
               '(B) 株ごと',     '(B) 株ごと',     '(B) 株ごと'),
    object = c('universe_A', 'upDEGs_A', 'downDEGs_A',
               'universe_B', 'upDEGs_B', 'downDEGs_B'),
    meaning = rep(c('検定された遺伝子（背景）', 'KO で高い（padj<0.05, LFC>0）',
                    'KO で低い（padj<0.05, LFC<0）'), 2),
    n = c(length(universe_A), length(upDEGs_A), length(downDEGs_A),
          length(universe_B), length(upDEGs_B), length(downDEGs_B)),
    check.names = FALSE)

# ===============================================================
# 2. enricher の引数を決める
# ===============================================================
# ★ universe（背景）に実際に何が入るか ── 誤解しやすいので Table 5 で実測する
#   clusterProfiler の enricher は、渡した universe をそのまま分母にはしない。
#   内部で「TERM2GENE に出てくる遺伝子」と交差させてから使う
#     （enricher_internal: extID <- intersect(ALLEXTID(USER_DATA), universe)）。
#   つまり ★ アノテーションが付いていない遺伝子は分母に入らない。
#   KEGG の背景 N は渡した 9,629 ではなく 2,311（universe_A のうち KEGG を持つもの）。
#   ソースごとに別の背景になる: COG 8,846 / PFAM 8,440 / CAZy 192。
#
#   では universe を渡す意味は何か:
#     ・アノテーションはあるが検定されなかった遺伝子（padj が NA。低発現や
#       Cook's distance 外れ値で DESeq2 が落としたもの）を背景から外す
#     ・マーカーを外す
#   いずれも「論理的に正しい背景にする」ためで、本データでは結果を動かさない
#   （Table 5: 有意 term は universe あり12本 / なし12本で完全に同一、N の差は 22）。
#   背景が既定でゲノム全体やアノテーション DB 全体になる解析系（org.db を使う
#   GO 解析など）では効果が大きいので、作法として常に渡しておく。
#
#   ★ universe では直せない偏りが残る（Table 5 下段）
#   アノテーションが付く遺伝子は発現量が高い（baseMean 中央値 400 対 149）。
#   高発現＝検出力が高いので DEG になりやすく、DEG 率も 38.9% 対 30.3%。
#   → ORA で出てくる経路は「高発現遺伝子の集まり」に偏る。背景の取り方では
#     直らないので、次のセルで fgsea（閾値を切らない）をかけて確認する
#
# ★ minGSSize / maxGSSize は使わない（既定値を無効化する）
#   clusterProfiler の既定は minGSSize=10, maxGSSize=500 で、
#   この範囲外の term は検定そのものをしない。これを
#   minGSSize=1, maxGSSize=Inf にして ★ 全 term を検定する。
#
#   数えているサイズは、DEG 側で当たった数(k)ではなく
#   universe と交差させた後の背景に残った数(M)。
#
#   なぜ足切りをしないか:
#   ・KEGG / COG / PFAM / CAZy はそれぞれ別の切り口なので、
#     ★ 同じ遺伝子が複数のソースで別々にヒットすること自体が
#     生物学的に重要な情報。足切りで片方を消すべきでない
#   ・小さい term が同じ遺伝子を二重に数える（PFAM:Sod_Fe_N と
#     PFAM:Sod_Fe_C は同じ sodB 5遺伝子。Table 5-5）のは、
#     この term リストをさらに統計解析する場合には問題になるが、
#     その予定はない。むしろ ★ 検定が正しく動いていることの確認になる
#   ・既定値のままだと CAZy は 44 term のうち 4 本しか検定されない。
#     Bacteroides では多糖分解酵素が重要なので致命的
#
#   ★ 代償: 検定する term が増えるので BH 補正の分母が増え、
#   既定値では有意だった 7 件が有意性を失う（Table 5-5）。
#   生の pvalue 列も ora に残してあるので、必要ならそちらで確認できる
#
# pvalueCutoff = 1, qvalueCutoff = 1
#   全 term を返させ、有意判定は p.adjust < 0.05 で自分でやる
#   （過去notebook 5_Enrichment_pathway_R.ipynb セル17 も pvalueCutoff = 1）

# ★ DEG が1つも当たらなかった term（k=0）は enricher の結果に現れないので、
#   実際の結果の行数は term 総数より少ない
cat("\n■ Table 2: term の大きさ（universe_A と交差させた後の遺伝子数 M）\n")
sz_kegg = t2g_kegg %>% filter(gene %in% universe_A) %>% dplyr::count(term, name = 'sz')
sz_cog  = t2g_cog  %>% filter(gene %in% universe_A) %>% dplyr::count(term, name = 'sz')
sz_pfam = t2g_pfam %>% filter(gene %in% universe_A) %>% dplyr::count(term, name = 'sz')
sz_cazy = t2g_cazy %>% filter(gene %in% universe_A) %>% dplyr::count(term, name = 'sz')
data.frame(
    source = c('KEGG', 'COG', 'PFAM', 'CAZy'),
    term総数 = c(nrow(sz_kegg), nrow(sz_cog), nrow(sz_pfam), nrow(sz_cazy)),
    M最小 = c(min(sz_kegg$sz), min(sz_cog$sz), min(sz_pfam$sz), min(sz_cazy$sz)),
    M中央値 = c(median(sz_kegg$sz), median(sz_cog$sz), median(sz_pfam$sz), median(sz_cazy$sz)),
    M最大 = c(max(sz_kegg$sz), max(sz_cog$sz), max(sz_pfam$sz), max(sz_cazy$sz)),
    既定値10_500で落ちる数 =
        c(sum(sz_kegg$sz < 10 | sz_kegg$sz > 500), sum(sz_cog$sz < 10 | sz_cog$sz > 500),
          sum(sz_pfam$sz < 10 | sz_pfam$sz > 500), sum(sz_cazy$sz < 10 | sz_cazy$sz > 500)),
    check.names = FALSE)

# ===============================================================
# 3. enricher を1回ずつ回す（2スキーム × 上下 × 4ソース = 16 回）
# ===============================================================
# 過去notebook と同じく、1回ごとに名前を付けて別々に持つ。
# 名前の付け方: <スキーム>_<向き>_<ソース>

# ── (A) KO で高い ───────────────────────────────────────────────
A_up_KEGG = enricher(upDEGs_A, universe = universe_A, TERM2GENE = t2g_kegg,
                     pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)
A_up_COG  = enricher(upDEGs_A, universe = universe_A, TERM2GENE = t2g_cog,
                     pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)
A_up_PFAM = enricher(upDEGs_A, universe = universe_A, TERM2GENE = t2g_pfam,
                     pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)
A_up_CAZy = enricher(upDEGs_A, universe = universe_A, TERM2GENE = t2g_cazy,
                     pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)

# ── (A) KO で低い ───────────────────────────────────────────────
A_dn_KEGG = enricher(downDEGs_A, universe = universe_A, TERM2GENE = t2g_kegg,
                     pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)
A_dn_COG  = enricher(downDEGs_A, universe = universe_A, TERM2GENE = t2g_cog,
                     pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)
A_dn_PFAM = enricher(downDEGs_A, universe = universe_A, TERM2GENE = t2g_pfam,
                     pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)
A_dn_CAZy = enricher(downDEGs_A, universe = universe_A, TERM2GENE = t2g_cazy,
                     pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)

# ── (B) KO で高い ───────────────────────────────────────────────
B_up_KEGG = enricher(upDEGs_B, universe = universe_B, TERM2GENE = t2g_kegg,
                     pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)
B_up_COG  = enricher(upDEGs_B, universe = universe_B, TERM2GENE = t2g_cog,
                     pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)
B_up_PFAM = enricher(upDEGs_B, universe = universe_B, TERM2GENE = t2g_pfam,
                     pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)
B_up_CAZy = enricher(upDEGs_B, universe = universe_B, TERM2GENE = t2g_cazy,
                     pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)

# ── (B) KO で低い ───────────────────────────────────────────────
B_dn_KEGG = enricher(downDEGs_B, universe = universe_B, TERM2GENE = t2g_kegg,
                     pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)
B_dn_COG  = enricher(downDEGs_B, universe = universe_B, TERM2GENE = t2g_cog,
                     pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)
B_dn_PFAM = enricher(downDEGs_B, universe = universe_B, TERM2GENE = t2g_pfam,
                     pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)
B_dn_CAZy = enricher(downDEGs_B, universe = universe_B, TERM2GENE = t2g_cazy,
                     pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)

# ===============================================================
# 4. 16 個の結果を1つの表に積む
# ===============================================================
# 後のセル（図・fgsea との突き合わせ）で使うので縦に並べておく。
# どれがどの結果かを rbind の行で明示する
ora = rbind(
    as.data.frame(A_up_KEGG) %>% mutate(scheme = 'A', direction = 'up',   source = 'KEGG'),
    as.data.frame(A_up_COG ) %>% mutate(scheme = 'A', direction = 'up',   source = 'COG' ),
    as.data.frame(A_up_PFAM) %>% mutate(scheme = 'A', direction = 'up',   source = 'PFAM'),
    as.data.frame(A_up_CAZy) %>% mutate(scheme = 'A', direction = 'up',   source = 'CAZy'),
    as.data.frame(A_dn_KEGG) %>% mutate(scheme = 'A', direction = 'down', source = 'KEGG'),
    as.data.frame(A_dn_COG ) %>% mutate(scheme = 'A', direction = 'down', source = 'COG' ),
    as.data.frame(A_dn_PFAM) %>% mutate(scheme = 'A', direction = 'down', source = 'PFAM'),
    as.data.frame(A_dn_CAZy) %>% mutate(scheme = 'A', direction = 'down', source = 'CAZy'),
    as.data.frame(B_up_KEGG) %>% mutate(scheme = 'B', direction = 'up',   source = 'KEGG'),
    as.data.frame(B_up_COG ) %>% mutate(scheme = 'B', direction = 'up',   source = 'COG' ),
    as.data.frame(B_up_PFAM) %>% mutate(scheme = 'B', direction = 'up',   source = 'PFAM'),
    as.data.frame(B_up_CAZy) %>% mutate(scheme = 'B', direction = 'up',   source = 'CAZy'),
    as.data.frame(B_dn_KEGG) %>% mutate(scheme = 'B', direction = 'down', source = 'KEGG'),
    as.data.frame(B_dn_COG ) %>% mutate(scheme = 'B', direction = 'down', source = 'COG' ),
    as.data.frame(B_dn_PFAM) %>% mutate(scheme = 'B', direction = 'down', source = 'PFAM'),
    as.data.frame(B_dn_CAZy) %>% mutate(scheme = 'B', direction = 'down', source = 'CAZy'))
rownames(ora) = NULL

# 列名は enricher の既定のまま使う（過去notebook の出力と同じ並び）。
#   GeneRatio = 「DEG のうちこの term に入った数 / DEG のうちこのソースの
#                アノテーションを持つ数」   例 "15/411"
#   BgRatio   = 「背景のうちこの term に入った数 / 背景のうちこのソースの
#                アノテーションを持つ数」   例 "18/2311"
#   Count     = GeneRatio の分子（この term に入った DEG の数）
#
#   ★ 分母はソースごとに違う。DEG 総数 1,503 や universe 9,629 ではなく、
#     そのソースのアノテーションを持つ遺伝子だけを数える
#     （KEGG 411/2311、COG 1400/8846、PFAM 1346/8440、CAZy 29/192）
#
# fold enrichment = GeneRatio ÷ BgRatio。
#   DEG の中でのその term の割合が、背景での割合の何倍かを表す。
#   1 より大きければ偶然より多く入っている。
#   例: ko00195_Photosynthesis は (15/411) / (18/2311) = 4.69 倍
ora$fold = round(
    (as.integer(sub('/.*', '', ora$GeneRatio)) / as.integer(sub('.*/', '', ora$GeneRatio))) /
    (as.integer(sub('/.*', '', ora$BgRatio))   / as.integer(sub('.*/', '', ora$BgRatio))), 2)

ora = ora %>%
    dplyr::select(scheme, direction, source, ID, term = Description,
                  GeneRatio, BgRatio, Count, fold, pvalue, padj = p.adjust, geneID) %>%
    # ★ 並び順はセル7 の dotplot の y 軸と揃える。
    #   dotplot は GeneRatio の大きい順に上から並べる（同値は padj の小さい順）。
    #   GeneRatio の分母は scheme × direction × source の中では一定なので、
    #   Count の降順が GeneRatio の降順と一致する
    arrange(scheme, direction, source, desc(Count), padj)

cat(sprintf("\n■ ora: %d 行（16 回の結果を積んだもの）\n", nrow(ora)))

# ===============================================================
# 5. 結果を見る
# ===============================================================
cat("\n■ Table 3: 有意 term 数 / 検定した term 数（padj<0.05）\n")
data.frame(
    source = c('KEGG', 'COG', 'PFAM', 'CAZy'),
    A_up = c(sprintf('%d / %d', sum(A_up_KEGG@result$p.adjust < 0.05), nrow(A_up_KEGG@result)),
             sprintf('%d / %d', sum(A_up_COG @result$p.adjust < 0.05), nrow(A_up_COG @result)),
             sprintf('%d / %d', sum(A_up_PFAM@result$p.adjust < 0.05), nrow(A_up_PFAM@result)),
             sprintf('%d / %d', sum(A_up_CAZy@result$p.adjust < 0.05), nrow(A_up_CAZy@result))),
    A_down = c(sprintf('%d / %d', sum(A_dn_KEGG@result$p.adjust < 0.05), nrow(A_dn_KEGG@result)),
               sprintf('%d / %d', sum(A_dn_COG @result$p.adjust < 0.05), nrow(A_dn_COG @result)),
               sprintf('%d / %d', sum(A_dn_PFAM@result$p.adjust < 0.05), nrow(A_dn_PFAM@result)),
               sprintf('%d / %d', sum(A_dn_CAZy@result$p.adjust < 0.05), nrow(A_dn_CAZy@result))),
    B_up = c(sprintf('%d / %d', sum(B_up_KEGG@result$p.adjust < 0.05), nrow(B_up_KEGG@result)),
             sprintf('%d / %d', sum(B_up_COG @result$p.adjust < 0.05), nrow(B_up_COG @result)),
             sprintf('%d / %d', sum(B_up_PFAM@result$p.adjust < 0.05), nrow(B_up_PFAM@result)),
             sprintf('%d / %d', sum(B_up_CAZy@result$p.adjust < 0.05), nrow(B_up_CAZy@result))),
    B_down = c(sprintf('%d / %d', sum(B_dn_KEGG@result$p.adjust < 0.05), nrow(B_dn_KEGG@result)),
               sprintf('%d / %d', sum(B_dn_COG @result$p.adjust < 0.05), nrow(B_dn_COG @result)),
               sprintf('%d / %d', sum(B_dn_PFAM@result$p.adjust < 0.05), nrow(B_dn_PFAM@result)),
               sprintf('%d / %d', sum(B_dn_CAZy@result$p.adjust < 0.05), nrow(B_dn_CAZy@result))),
    check.names = FALSE)

# ── 有意 term ごとに、当たった遺伝子を株で数える ─────────────────
# ★ geneID 列（"/" 区切りの遺伝子名）を1行1遺伝子にばらして株を付け、
#   term ごとに数える。これを Table 4 の右に列として足す。
#   4株の合計は Count（= GeneRatio の分子）と一致するはず
hit_strain = ora %>% filter(padj < 0.05) %>%
    dplyr::select(scheme, direction, source, term, geneID) %>%
    tidyr::separate_rows(geneID, sep = '/') %>%
    left_join(res %>% dplyr::select(gene_id, strain), by = c('geneID' = 'gene_id')) %>%
    dplyr::count(scheme, direction, source, term, strain) %>%
    tidyr::pivot_wider(names_from = strain, values_from = n, values_fill = 0)

cat("\n■ Table 4: 有意になった term（padj<0.05）すべて ＋ 当たった遺伝子の株ごとの内訳\n")
ora %>% filter(padj < 0.05) %>%
    left_join(hit_strain, by = c('scheme', 'direction', 'source', 'term')) %>%
    mutate(scheme = paste0('(', scheme, ')'), padj = signif(padj, 2),
           VPI_pct = round(100 * Bth_VPI  / Count, 1),
           St8_pct = round(100 * Bth_St8  / Count, 1),
           Bun_pct = round(100 * Bun_8492 / Count, 1),
           Odo_pct = round(100 * Odo_St23 / Count, 1)) %>%
    dplyr::select(scheme, direction, source, term, GeneRatio, BgRatio, fold, padj,
                  Bth_VPI, Bth_St8, Bun_8492, Odo_St23,
                  VPI_pct, St8_pct, Bun_pct, Odo_pct) %>%
    as.data.frame()

# 検算: 4株の合計が Count と一致するか
cat(sprintf("\n  検算: 4株の合計 ≠ Count となる term の数 = %d（0 なら整合）\n",
    ora %>% filter(padj < 0.05) %>%
        left_join(hit_strain, by = c('scheme', 'direction', 'source', 'term')) %>%
        filter(Bth_VPI + Bth_St8 + Bun_8492 + Odo_St23 != Count) %>% nrow()))

cat("\n■ Table 4-2: 比較用 — DEG 全体の株ごとの内訳（Table 4 の割合を読む基準）\n")
data.frame(
    scheme = c('A', 'A', 'B', 'B'),
    direction = c('up', 'down', 'up', 'down'),
    Bth_VPI = c(length(intersect(upDEGs_A,   res$gene_id[res$strain == 'Bth_VPI'])),
                length(intersect(downDEGs_A, res$gene_id[res$strain == 'Bth_VPI'])),
                length(intersect(upDEGs_B,   res$gene_id[res$strain == 'Bth_VPI'])),
                length(intersect(downDEGs_B, res$gene_id[res$strain == 'Bth_VPI']))),
    Bth_St8 = c(length(intersect(upDEGs_A,   res$gene_id[res$strain == 'Bth_St8'])),
                length(intersect(downDEGs_A, res$gene_id[res$strain == 'Bth_St8'])),
                length(intersect(upDEGs_B,   res$gene_id[res$strain == 'Bth_St8'])),
                length(intersect(downDEGs_B, res$gene_id[res$strain == 'Bth_St8']))),
    Bun_8492 = c(length(intersect(upDEGs_A,   res$gene_id[res$strain == 'Bun_8492'])),
                 length(intersect(downDEGs_A, res$gene_id[res$strain == 'Bun_8492'])),
                 length(intersect(upDEGs_B,   res$gene_id[res$strain == 'Bun_8492'])),
                 length(intersect(downDEGs_B, res$gene_id[res$strain == 'Bun_8492']))),
    Odo_St23 = c(length(intersect(upDEGs_A,   res$gene_id[res$strain == 'Odo_St23'])),
                 length(intersect(downDEGs_A, res$gene_id[res$strain == 'Odo_St23'])),
                 length(intersect(upDEGs_B,   res$gene_id[res$strain == 'Odo_St23'])),
                 length(intersect(downDEGs_B, res$gene_id[res$strain == 'Odo_St23']))),
    check.names = FALSE)

# term ごとの内訳はセル8 の Table 6 で出す（(A) の各 term × 株）

# ===============================================================
# 6. 背景（universe）が実際にどうなったかを確かめる
# ===============================================================
# 渡した universe_A は 9,629 だが、enricher はそれを
# 「そのソースの term を1つ以上持つ遺伝子」に絞ってから分母にする。
# アノテーションの無い遺伝子が分母に入って p 値が甘くなることは起きない
cat("\n■ Table 5-1: 渡した universe と実際に使われた背景 N\n")
data.frame(
    source = c('KEGG', 'COG', 'PFAM', 'CAZy'),
    universe_A_渡した = length(universe_A),
    N_実際の背景 = c(length(intersect(universe_A, t2g_kegg$gene)),
                     length(intersect(universe_A, t2g_cog$gene)),
                     length(intersect(universe_A, t2g_pfam$gene)),
                     length(intersect(universe_A, t2g_cazy$gene))),
    アノテーション無しで分母から外れた数 =
        length(universe_A) - c(length(intersect(universe_A, t2g_kegg$gene)),
                               length(intersect(universe_A, t2g_cog$gene)),
                               length(intersect(universe_A, t2g_pfam$gene)),
                               length(intersect(universe_A, t2g_cazy$gene))),
    check.names = FALSE)

# universe を省略した過去notebook の呼び方と結果を比べる
A_dn_KEGG_noU = enricher(downDEGs_A, TERM2GENE = t2g_kegg,
                         pvalueCutoff = 1, qvalueCutoff = 1,
                         minGSSize = 1, maxGSSize = Inf)

cat("\n■ Table 5-2: universe を渡す／渡さない（KEGG, (A) down）\n")
data.frame(
    呼び方 = c('universe = universe_A', 'universe 省略（過去notebook）'),
    N = c(as.integer(sub('.*/', '', A_dn_KEGG@result$BgRatio[1])),
          as.integer(sub('.*/', '', A_dn_KEGG_noU@result$BgRatio[1]))),
    有意term = c(sum(A_dn_KEGG@result$p.adjust < 0.05),
                 sum(A_dn_KEGG_noU@result$p.adjust < 0.05)),
    check.names = FALSE)

cat(sprintf("  → 有意 term は %d 本で共通（違い %d 本）。N の差 %d は\n",
    length(intersect(A_dn_KEGG@result$ID[A_dn_KEGG@result$p.adjust < 0.05],
                     A_dn_KEGG_noU@result$ID[A_dn_KEGG_noU@result$p.adjust < 0.05])),
    length(setdiff(A_dn_KEGG_noU@result$ID[A_dn_KEGG_noU@result$p.adjust < 0.05],
                   A_dn_KEGG@result$ID[A_dn_KEGG@result$p.adjust < 0.05])),
    as.integer(sub('.*/', '', A_dn_KEGG_noU@result$BgRatio[1])) -
    as.integer(sub('.*/', '', A_dn_KEGG@result$BgRatio[1]))))
cat("     「KEGG は付いているが検定されなかった（padj が NA）遺伝子」\n")

# ★ universe では直せない偏り: アノテーションが付く遺伝子は発現量が高い
kegg_yes = res %>% filter(!is.na(padj_A), !marker,  gene_id %in% t2g_kegg$gene)
kegg_no  = res %>% filter(!is.na(padj_A), !marker, !gene_id %in% t2g_kegg$gene)

cat("\n■ Table 5-3: ★ universe では直せない偏り（発現量）\n")
data.frame(
    群 = c('KEGG あり', 'KEGG なし'),
    n = c(nrow(kegg_yes), nrow(kegg_no)),
    baseMean中央値 = round(c(median(kegg_yes$baseMean), median(kegg_no$baseMean))),
    DEG率pct = round(100 * c(mean(kegg_yes$padj_A < 0.05),
                             mean(kegg_no$padj_A  < 0.05)), 1),
    check.names = FALSE)

# 有意になった term は、その KEGG あり集団の中でもさらに高発現に偏っている
atp_genes = strsplit(ora %>% filter(scheme == 'A', direction == 'down',
                                    term == 'ko00195_Photosynthesis') %>% pull(geneID), '/')[[1]]
rib_genes = strsplit(ora %>% filter(scheme == 'A', direction == 'up',
                                    term == 'ko03010_Ribosome') %>% pull(geneID), '/')[[1]]

cat("\n■ Table 5-4: 有意 term は高発現遺伝子の集まりか\n")
data.frame(
    集団 = c('KEGG あり全体', 'ko00195_Photosynthesis（実体は ATP synthase）',
              'ko03010_Ribosome'),
    n = c(nrow(kegg_yes), length(atp_genes), length(rib_genes)),
    baseMean中央値 = round(c(median(kegg_yes$baseMean),
                             median(res$baseMean[res$gene_id %in% atp_genes]),
                             median(res$baseMean[res$gene_id %in% rib_genes]))),
    check.names = FALSE)

# ── 足切りをしない判断の裏付け ───────────────────────────────────
# clusterProfiler の既定値（minGSSize=10, maxGSSize=500）でも同じ16回を回し、
# 足切りをしないことで何が増え、何を失うかを並べて確認する。
# 名前は本番の結果に _d10 を付けたもの

A_up_KEGG_d10   = enricher(upDEGs_A, universe = universe_A, TERM2GENE = t2g_kegg,
                  pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 10, maxGSSize = 500)
A_up_COG_d10    = enricher(upDEGs_A, universe = universe_A, TERM2GENE = t2g_cog ,
                  pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 10, maxGSSize = 500)
A_up_PFAM_d10   = enricher(upDEGs_A, universe = universe_A, TERM2GENE = t2g_pfam,
                  pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 10, maxGSSize = 500)
A_up_CAZy_d10   = enricher(upDEGs_A, universe = universe_A, TERM2GENE = t2g_cazy,
                  pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 10, maxGSSize = 500)
A_dn_KEGG_d10   = enricher(downDEGs_A, universe = universe_A, TERM2GENE = t2g_kegg,
                  pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 10, maxGSSize = 500)
A_dn_COG_d10    = enricher(downDEGs_A, universe = universe_A, TERM2GENE = t2g_cog ,
                  pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 10, maxGSSize = 500)
A_dn_PFAM_d10   = enricher(downDEGs_A, universe = universe_A, TERM2GENE = t2g_pfam,
                  pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 10, maxGSSize = 500)
A_dn_CAZy_d10   = enricher(downDEGs_A, universe = universe_A, TERM2GENE = t2g_cazy,
                  pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 10, maxGSSize = 500)
B_up_KEGG_d10   = enricher(upDEGs_B, universe = universe_B, TERM2GENE = t2g_kegg,
                  pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 10, maxGSSize = 500)
B_up_COG_d10    = enricher(upDEGs_B, universe = universe_B, TERM2GENE = t2g_cog ,
                  pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 10, maxGSSize = 500)
B_up_PFAM_d10   = enricher(upDEGs_B, universe = universe_B, TERM2GENE = t2g_pfam,
                  pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 10, maxGSSize = 500)
B_up_CAZy_d10   = enricher(upDEGs_B, universe = universe_B, TERM2GENE = t2g_cazy,
                  pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 10, maxGSSize = 500)
B_dn_KEGG_d10   = enricher(downDEGs_B, universe = universe_B, TERM2GENE = t2g_kegg,
                  pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 10, maxGSSize = 500)
B_dn_COG_d10    = enricher(downDEGs_B, universe = universe_B, TERM2GENE = t2g_cog ,
                  pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 10, maxGSSize = 500)
B_dn_PFAM_d10   = enricher(downDEGs_B, universe = universe_B, TERM2GENE = t2g_pfam,
                  pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 10, maxGSSize = 500)
B_dn_CAZy_d10   = enricher(downDEGs_B, universe = universe_B, TERM2GENE = t2g_cazy,
                  pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 10, maxGSSize = 500)

cat("\n■ Table 5-5: 既定値(10/500) と 採用(足切り無し) の比較\n")
rbind(
    data.frame(scheme = 'A', direction = 'up', source = 'KEGG',
               既定値_term = nrow(A_up_KEGG_d10@result),
               採用_term = nrow(A_up_KEGG@result),
               既定値_有意 = sum(A_up_KEGG_d10@result$p.adjust < 0.05),
               採用_有意 = sum(A_up_KEGG@result$p.adjust < 0.05),
               失う = length(setdiff(A_up_KEGG_d10@result$Description[A_up_KEGG_d10@result$p.adjust < 0.05],
                                     A_up_KEGG@result$Description[A_up_KEGG@result$p.adjust < 0.05])),
               check.names = FALSE),
    data.frame(scheme = 'A', direction = 'up', source = 'COG',
               既定値_term = nrow(A_up_COG_d10@result),
               採用_term = nrow(A_up_COG@result),
               既定値_有意 = sum(A_up_COG_d10@result$p.adjust < 0.05),
               採用_有意 = sum(A_up_COG@result$p.adjust < 0.05),
               失う = length(setdiff(A_up_COG_d10@result$Description[A_up_COG_d10@result$p.adjust < 0.05],
                                     A_up_COG@result$Description[A_up_COG@result$p.adjust < 0.05])),
               check.names = FALSE),
    data.frame(scheme = 'A', direction = 'up', source = 'PFAM',
               既定値_term = nrow(A_up_PFAM_d10@result),
               採用_term = nrow(A_up_PFAM@result),
               既定値_有意 = sum(A_up_PFAM_d10@result$p.adjust < 0.05),
               採用_有意 = sum(A_up_PFAM@result$p.adjust < 0.05),
               失う = length(setdiff(A_up_PFAM_d10@result$Description[A_up_PFAM_d10@result$p.adjust < 0.05],
                                     A_up_PFAM@result$Description[A_up_PFAM@result$p.adjust < 0.05])),
               check.names = FALSE),
    data.frame(scheme = 'A', direction = 'up', source = 'CAZy',
               既定値_term = nrow(A_up_CAZy_d10@result),
               採用_term = nrow(A_up_CAZy@result),
               既定値_有意 = sum(A_up_CAZy_d10@result$p.adjust < 0.05),
               採用_有意 = sum(A_up_CAZy@result$p.adjust < 0.05),
               失う = length(setdiff(A_up_CAZy_d10@result$Description[A_up_CAZy_d10@result$p.adjust < 0.05],
                                     A_up_CAZy@result$Description[A_up_CAZy@result$p.adjust < 0.05])),
               check.names = FALSE),
    data.frame(scheme = 'A', direction = 'down', source = 'KEGG',
               既定値_term = nrow(A_dn_KEGG_d10@result),
               採用_term = nrow(A_dn_KEGG@result),
               既定値_有意 = sum(A_dn_KEGG_d10@result$p.adjust < 0.05),
               採用_有意 = sum(A_dn_KEGG@result$p.adjust < 0.05),
               失う = length(setdiff(A_dn_KEGG_d10@result$Description[A_dn_KEGG_d10@result$p.adjust < 0.05],
                                     A_dn_KEGG@result$Description[A_dn_KEGG@result$p.adjust < 0.05])),
               check.names = FALSE),
    data.frame(scheme = 'A', direction = 'down', source = 'COG',
               既定値_term = nrow(A_dn_COG_d10@result),
               採用_term = nrow(A_dn_COG@result),
               既定値_有意 = sum(A_dn_COG_d10@result$p.adjust < 0.05),
               採用_有意 = sum(A_dn_COG@result$p.adjust < 0.05),
               失う = length(setdiff(A_dn_COG_d10@result$Description[A_dn_COG_d10@result$p.adjust < 0.05],
                                     A_dn_COG@result$Description[A_dn_COG@result$p.adjust < 0.05])),
               check.names = FALSE),
    data.frame(scheme = 'A', direction = 'down', source = 'PFAM',
               既定値_term = nrow(A_dn_PFAM_d10@result),
               採用_term = nrow(A_dn_PFAM@result),
               既定値_有意 = sum(A_dn_PFAM_d10@result$p.adjust < 0.05),
               採用_有意 = sum(A_dn_PFAM@result$p.adjust < 0.05),
               失う = length(setdiff(A_dn_PFAM_d10@result$Description[A_dn_PFAM_d10@result$p.adjust < 0.05],
                                     A_dn_PFAM@result$Description[A_dn_PFAM@result$p.adjust < 0.05])),
               check.names = FALSE),
    data.frame(scheme = 'A', direction = 'down', source = 'CAZy',
               既定値_term = nrow(A_dn_CAZy_d10@result),
               採用_term = nrow(A_dn_CAZy@result),
               既定値_有意 = sum(A_dn_CAZy_d10@result$p.adjust < 0.05),
               採用_有意 = sum(A_dn_CAZy@result$p.adjust < 0.05),
               失う = length(setdiff(A_dn_CAZy_d10@result$Description[A_dn_CAZy_d10@result$p.adjust < 0.05],
                                     A_dn_CAZy@result$Description[A_dn_CAZy@result$p.adjust < 0.05])),
               check.names = FALSE),
    data.frame(scheme = 'B', direction = 'up', source = 'KEGG',
               既定値_term = nrow(B_up_KEGG_d10@result),
               採用_term = nrow(B_up_KEGG@result),
               既定値_有意 = sum(B_up_KEGG_d10@result$p.adjust < 0.05),
               採用_有意 = sum(B_up_KEGG@result$p.adjust < 0.05),
               失う = length(setdiff(B_up_KEGG_d10@result$Description[B_up_KEGG_d10@result$p.adjust < 0.05],
                                     B_up_KEGG@result$Description[B_up_KEGG@result$p.adjust < 0.05])),
               check.names = FALSE),
    data.frame(scheme = 'B', direction = 'up', source = 'COG',
               既定値_term = nrow(B_up_COG_d10@result),
               採用_term = nrow(B_up_COG@result),
               既定値_有意 = sum(B_up_COG_d10@result$p.adjust < 0.05),
               採用_有意 = sum(B_up_COG@result$p.adjust < 0.05),
               失う = length(setdiff(B_up_COG_d10@result$Description[B_up_COG_d10@result$p.adjust < 0.05],
                                     B_up_COG@result$Description[B_up_COG@result$p.adjust < 0.05])),
               check.names = FALSE),
    data.frame(scheme = 'B', direction = 'up', source = 'PFAM',
               既定値_term = nrow(B_up_PFAM_d10@result),
               採用_term = nrow(B_up_PFAM@result),
               既定値_有意 = sum(B_up_PFAM_d10@result$p.adjust < 0.05),
               採用_有意 = sum(B_up_PFAM@result$p.adjust < 0.05),
               失う = length(setdiff(B_up_PFAM_d10@result$Description[B_up_PFAM_d10@result$p.adjust < 0.05],
                                     B_up_PFAM@result$Description[B_up_PFAM@result$p.adjust < 0.05])),
               check.names = FALSE),
    data.frame(scheme = 'B', direction = 'up', source = 'CAZy',
               既定値_term = nrow(B_up_CAZy_d10@result),
               採用_term = nrow(B_up_CAZy@result),
               既定値_有意 = sum(B_up_CAZy_d10@result$p.adjust < 0.05),
               採用_有意 = sum(B_up_CAZy@result$p.adjust < 0.05),
               失う = length(setdiff(B_up_CAZy_d10@result$Description[B_up_CAZy_d10@result$p.adjust < 0.05],
                                     B_up_CAZy@result$Description[B_up_CAZy@result$p.adjust < 0.05])),
               check.names = FALSE),
    data.frame(scheme = 'B', direction = 'down', source = 'KEGG',
               既定値_term = nrow(B_dn_KEGG_d10@result),
               採用_term = nrow(B_dn_KEGG@result),
               既定値_有意 = sum(B_dn_KEGG_d10@result$p.adjust < 0.05),
               採用_有意 = sum(B_dn_KEGG@result$p.adjust < 0.05),
               失う = length(setdiff(B_dn_KEGG_d10@result$Description[B_dn_KEGG_d10@result$p.adjust < 0.05],
                                     B_dn_KEGG@result$Description[B_dn_KEGG@result$p.adjust < 0.05])),
               check.names = FALSE),
    data.frame(scheme = 'B', direction = 'down', source = 'COG',
               既定値_term = nrow(B_dn_COG_d10@result),
               採用_term = nrow(B_dn_COG@result),
               既定値_有意 = sum(B_dn_COG_d10@result$p.adjust < 0.05),
               採用_有意 = sum(B_dn_COG@result$p.adjust < 0.05),
               失う = length(setdiff(B_dn_COG_d10@result$Description[B_dn_COG_d10@result$p.adjust < 0.05],
                                     B_dn_COG@result$Description[B_dn_COG@result$p.adjust < 0.05])),
               check.names = FALSE),
    data.frame(scheme = 'B', direction = 'down', source = 'PFAM',
               既定値_term = nrow(B_dn_PFAM_d10@result),
               採用_term = nrow(B_dn_PFAM@result),
               既定値_有意 = sum(B_dn_PFAM_d10@result$p.adjust < 0.05),
               採用_有意 = sum(B_dn_PFAM@result$p.adjust < 0.05),
               失う = length(setdiff(B_dn_PFAM_d10@result$Description[B_dn_PFAM_d10@result$p.adjust < 0.05],
                                     B_dn_PFAM@result$Description[B_dn_PFAM@result$p.adjust < 0.05])),
               check.names = FALSE),
    data.frame(scheme = 'B', direction = 'down', source = 'CAZy',
               既定値_term = nrow(B_dn_CAZy_d10@result),
               採用_term = nrow(B_dn_CAZy@result),
               既定値_有意 = sum(B_dn_CAZy_d10@result$p.adjust < 0.05),
               採用_有意 = sum(B_dn_CAZy@result$p.adjust < 0.05),
               失う = length(setdiff(B_dn_CAZy_d10@result$Description[B_dn_CAZy_d10@result$p.adjust < 0.05],
                                     B_dn_CAZy@result$Description[B_dn_CAZy@result$p.adjust < 0.05])),
               check.names = FALSE))

cat("\n■ Table 5-6: 足切りをしないことで有意性を失う term（唯一の代償）\n")
cat("   検定する term が増えて BH 補正の分母が大きくなるため。生の pvalue は ora に残してある\n")

# 16 通りそれぞれで「既定値では有意だったが、足切り無しでは有意でなくなった term」
lost_A_up_KEGG    = setdiff(A_up_KEGG_d10@result$Description[A_up_KEGG_d10@result$p.adjust < 0.05],
                        A_up_KEGG@result$Description[A_up_KEGG@result$p.adjust < 0.05])
lost_A_up_COG     = setdiff(A_up_COG_d10@result$Description[A_up_COG_d10@result$p.adjust < 0.05],
                        A_up_COG@result$Description[A_up_COG@result$p.adjust < 0.05])
lost_A_up_PFAM    = setdiff(A_up_PFAM_d10@result$Description[A_up_PFAM_d10@result$p.adjust < 0.05],
                        A_up_PFAM@result$Description[A_up_PFAM@result$p.adjust < 0.05])
lost_A_up_CAZy    = setdiff(A_up_CAZy_d10@result$Description[A_up_CAZy_d10@result$p.adjust < 0.05],
                        A_up_CAZy@result$Description[A_up_CAZy@result$p.adjust < 0.05])
lost_A_dn_KEGG    = setdiff(A_dn_KEGG_d10@result$Description[A_dn_KEGG_d10@result$p.adjust < 0.05],
                        A_dn_KEGG@result$Description[A_dn_KEGG@result$p.adjust < 0.05])
lost_A_dn_COG     = setdiff(A_dn_COG_d10@result$Description[A_dn_COG_d10@result$p.adjust < 0.05],
                        A_dn_COG@result$Description[A_dn_COG@result$p.adjust < 0.05])
lost_A_dn_PFAM    = setdiff(A_dn_PFAM_d10@result$Description[A_dn_PFAM_d10@result$p.adjust < 0.05],
                        A_dn_PFAM@result$Description[A_dn_PFAM@result$p.adjust < 0.05])
lost_A_dn_CAZy    = setdiff(A_dn_CAZy_d10@result$Description[A_dn_CAZy_d10@result$p.adjust < 0.05],
                        A_dn_CAZy@result$Description[A_dn_CAZy@result$p.adjust < 0.05])
lost_B_up_KEGG    = setdiff(B_up_KEGG_d10@result$Description[B_up_KEGG_d10@result$p.adjust < 0.05],
                        B_up_KEGG@result$Description[B_up_KEGG@result$p.adjust < 0.05])
lost_B_up_COG     = setdiff(B_up_COG_d10@result$Description[B_up_COG_d10@result$p.adjust < 0.05],
                        B_up_COG@result$Description[B_up_COG@result$p.adjust < 0.05])
lost_B_up_PFAM    = setdiff(B_up_PFAM_d10@result$Description[B_up_PFAM_d10@result$p.adjust < 0.05],
                        B_up_PFAM@result$Description[B_up_PFAM@result$p.adjust < 0.05])
lost_B_up_CAZy    = setdiff(B_up_CAZy_d10@result$Description[B_up_CAZy_d10@result$p.adjust < 0.05],
                        B_up_CAZy@result$Description[B_up_CAZy@result$p.adjust < 0.05])
lost_B_dn_KEGG    = setdiff(B_dn_KEGG_d10@result$Description[B_dn_KEGG_d10@result$p.adjust < 0.05],
                        B_dn_KEGG@result$Description[B_dn_KEGG@result$p.adjust < 0.05])
lost_B_dn_COG     = setdiff(B_dn_COG_d10@result$Description[B_dn_COG_d10@result$p.adjust < 0.05],
                        B_dn_COG@result$Description[B_dn_COG@result$p.adjust < 0.05])
lost_B_dn_PFAM    = setdiff(B_dn_PFAM_d10@result$Description[B_dn_PFAM_d10@result$p.adjust < 0.05],
                        B_dn_PFAM@result$Description[B_dn_PFAM@result$p.adjust < 0.05])
lost_B_dn_CAZy    = setdiff(B_dn_CAZy_d10@result$Description[B_dn_CAZy_d10@result$p.adjust < 0.05],
                        B_dn_CAZy@result$Description[B_dn_CAZy@result$p.adjust < 0.05])

# rbind は NULL を無視するので、該当が無い組み合わせは if が NULL を返して飛ばされる
rbind(
    if (length(lost_A_up_KEGG) > 0) data.frame(scheme = 'A', direction = 'up',
        source = 'KEGG', term = lost_A_up_KEGG, check.names = FALSE),
    if (length(lost_A_up_COG) > 0) data.frame(scheme = 'A', direction = 'up',
        source = 'COG', term = lost_A_up_COG, check.names = FALSE),
    if (length(lost_A_up_PFAM) > 0) data.frame(scheme = 'A', direction = 'up',
        source = 'PFAM', term = lost_A_up_PFAM, check.names = FALSE),
    if (length(lost_A_up_CAZy) > 0) data.frame(scheme = 'A', direction = 'up',
        source = 'CAZy', term = lost_A_up_CAZy, check.names = FALSE),
    if (length(lost_A_dn_KEGG) > 0) data.frame(scheme = 'A', direction = 'down',
        source = 'KEGG', term = lost_A_dn_KEGG, check.names = FALSE),
    if (length(lost_A_dn_COG) > 0) data.frame(scheme = 'A', direction = 'down',
        source = 'COG', term = lost_A_dn_COG, check.names = FALSE),
    if (length(lost_A_dn_PFAM) > 0) data.frame(scheme = 'A', direction = 'down',
        source = 'PFAM', term = lost_A_dn_PFAM, check.names = FALSE),
    if (length(lost_A_dn_CAZy) > 0) data.frame(scheme = 'A', direction = 'down',
        source = 'CAZy', term = lost_A_dn_CAZy, check.names = FALSE),
    if (length(lost_B_up_KEGG) > 0) data.frame(scheme = 'B', direction = 'up',
        source = 'KEGG', term = lost_B_up_KEGG, check.names = FALSE),
    if (length(lost_B_up_COG) > 0) data.frame(scheme = 'B', direction = 'up',
        source = 'COG', term = lost_B_up_COG, check.names = FALSE),
    if (length(lost_B_up_PFAM) > 0) data.frame(scheme = 'B', direction = 'up',
        source = 'PFAM', term = lost_B_up_PFAM, check.names = FALSE),
    if (length(lost_B_up_CAZy) > 0) data.frame(scheme = 'B', direction = 'up',
        source = 'CAZy', term = lost_B_up_CAZy, check.names = FALSE),
    if (length(lost_B_dn_KEGG) > 0) data.frame(scheme = 'B', direction = 'down',
        source = 'KEGG', term = lost_B_dn_KEGG, check.names = FALSE),
    if (length(lost_B_dn_COG) > 0) data.frame(scheme = 'B', direction = 'down',
        source = 'COG', term = lost_B_dn_COG, check.names = FALSE),
    if (length(lost_B_dn_PFAM) > 0) data.frame(scheme = 'B', direction = 'down',
        source = 'PFAM', term = lost_B_dn_PFAM, check.names = FALSE),
    if (length(lost_B_dn_CAZy) > 0) data.frame(scheme = 'B', direction = 'down',
        source = 'CAZy', term = lost_B_dn_CAZy, check.names = FALSE))

# ── 既定値で「検定すらされなかった」term を確認する ───────────────
# 失うのは有意性だけではない。minGSSize=10 では term 内の遺伝子数が 10 未満の term は
# ★ 結果の表に行として現れない（padj が NA）ので、見落としにも気づけない。
# 4株の定義群集では、生物学的に重要な機能ほど数遺伝子で代表されるため
# この影響が大きい（例: bile salt hydrolase は全データで 5-7 遺伝子しかない）
M_kegg = t2g_kegg %>% filter(gene %in% universe_A) %>% dplyr::count(term, name = 'term内遺伝子数')
M_pfam = t2g_pfam %>% filter(gene %in% universe_A) %>% dplyr::count(term, name = 'term内遺伝子数')

cat("\n■ Table 5-7: (A) down で有意な term のうち、既定値 minGSSize=10 では検定されないもの\n")
rbind(
    A_dn_KEGG@result %>% filter(p.adjust < 0.05) %>%
        transmute(source = 'KEGG', term = Description, GeneRatio, BgRatio,
                  padj_足切り無し = signif(p.adjust, 3)) %>%
        left_join(M_kegg, by = 'term') %>%
        left_join(A_dn_KEGG_d10@result %>% transmute(term = Description,
                  padj_既定値 = signif(p.adjust, 3)), by = 'term'),
    A_dn_PFAM@result %>% filter(p.adjust < 0.05) %>%
        transmute(source = 'PFAM', term = Description, GeneRatio, BgRatio,
                  padj_足切り無し = signif(p.adjust, 3)) %>%
        left_join(M_pfam, by = 'term') %>%
        left_join(A_dn_PFAM_d10@result %>% transmute(term = Description,
                  padj_既定値 = signif(p.adjust, 3)), by = 'term')) %>%
    mutate(既定値での扱い = ifelse(term内遺伝子数 < 10,
                                   '★ 検定されない（表に出ない）', '検定された')) %>%
    dplyr::select(source, term, term内遺伝子数, GeneRatio, BgRatio,
                  padj_足切り無し, padj_既定値, 既定値での扱い) %>%
    arrange(source, term内遺伝子数) %>% as.data.frame()

cat(sprintf("\n  (A) down の有意 term %d 本のうち、既定値では %d 本が検定対象から外れる\n",
    sum(A_dn_KEGG@result$p.adjust < 0.05) + sum(A_dn_PFAM@result$p.adjust < 0.05),
    sum(M_kegg$term内遺伝子数[match(A_dn_KEGG@result$Description[A_dn_KEGG@result$p.adjust < 0.05],
                                     M_kegg$term)] < 10) +
    sum(M_pfam$term内遺伝子数[match(A_dn_PFAM@result$Description[A_dn_PFAM@result$p.adjust < 0.05],
                                     M_pfam$term)] < 10)))

# ── 小さい term の二重計上を確認する ───────────────────────────
# ★ 足切りをしないので PFAM:Sod_Fe_N と PFAM:Sod_Fe_C は両方とも検定される。
#   中身が同じであることを確かめておく。これは欠点ではなく、
#   同じ生物学的事実が別のドメイン名で独立に拾えている＝検定が正しく
#   動いていることの確認になる
sod_n = strsplit(A_dn_PFAM@result$geneID[A_dn_PFAM@result$Description == 'PFAM:Sod_Fe_N'], '/')[[1]]
sod_c = strsplit(A_dn_PFAM@result$geneID[A_dn_PFAM@result$Description == 'PFAM:Sod_Fe_C'], '/')[[1]]

cat(sprintf("\n■ PFAM:Sod_Fe_N（%d遺伝子）と PFAM:Sod_Fe_C（%d遺伝子）の一致数: %d\n",
            length(sod_n), length(sod_c), length(intersect(sod_n, sod_c))))
cat("   → 同じ SOD タンパクの N 末／C 末ドメイン。中身は同一\n")
print(res %>% filter(gene_id %in% union(sod_n, sod_c)) %>%
    dplyr::select(gene_id, strain, Preferred_name, product, LFC_A) %>%
    mutate(LFC_A = round(LFC_A, 2)) %>% as.data.frame(), row.names = FALSE)

# ===============================================================
# 7. 保存
# ===============================================================
write.csv(ora, file.path(path_pw, 'ora_all.csv'), row.names = FALSE)
invisible(file.copy(file.path(path_pw, 'ora_all.csv'),
          file.path(path_project, 'refqc', 'stage6_ora_all.csv'), overwrite = TRUE))
cat(sprintf("\n保存: %s/ora_all.csv（%d 行、geneID 列に当たった遺伝子）\n", path_pw, nrow(ora)))
cat("project 複製: refqc/stage6_ora_all.csv\n")


■ Table 1: 検定にかける遺伝子リスト


scheme,object,meaning,n
<chr>,<chr>,<chr>,<int>
(A) 全株プール,universe_A,検定された遺伝子（背景）,9629
(A) 全株プール,upDEGs_A,"KO で高い（padj<0.05, LFC>0）",1616
(A) 全株プール,downDEGs_A,"KO で低い（padj<0.05, LFC<0）",1503
(B) 株ごと,universe_B,検定された遺伝子（背景）,9638
(B) 株ごと,upDEGs_B,"KO で高い（padj<0.05, LFC>0）",1425
(B) 株ごと,downDEGs_B,"KO で低い（padj<0.05, LFC<0）",1533



■ Table 2: term の大きさ（universe_A と交差させた後の遺伝子数 M）


source,term総数,M最小,M中央値,M最大,既定値10_500で落ちる数
<chr>,<int>,<int>,<dbl>,<int>,<int>
KEGG,151,1,18.0,141,57
COG,21,1,388.0,2269,9
PFAM,2470,1,3.0,289,2164
CAZy,44,1,3.5,35,40



■ ora: 4692 行（16 回の結果を積んだもの）

■ Table 3: 有意 term 数 / 検定した term 数（padj<0.05）


source,A_up,A_down,B_up,B_down
<chr>,<chr>,<chr>,<chr>,<chr>
KEGG,1 / 110,18 / 120,2 / 111,18 / 115
COG,4 / 20,3 / 19,3 / 20,3 / 19
PFAM,0 / 1263,22 / 898,0 / 1031,30 / 897
CAZy,0 / 19,0 / 19,0 / 12,0 / 19


... (15 outputs left out)

scheme,direction,source,term
<chr>,<chr>,<chr>,<chr>
A,up,CAZy,CAZy:GH20
A,down,PFAM,PFAM:Patatin
A,down,PFAM,PFAM:Biotin_lipoyl_2
A,down,PFAM,PFAM:BPD_transp_1
A,down,PFAM,PFAM:CPSase_L_D2
A,down,PFAM,PFAM:Thioredoxin
B,down,CAZy,CAZy:GH31



■ Table 5-7: (A) down で有意な term のうち、既定値 minGSSize=10 では検定されないもの


source,term,term内遺伝子数,GeneRatio,BgRatio,padj_足切り無し,padj_既定値,既定値での扱い
<chr>,<chr>,<int>,<chr>,<chr>,<dbl>,<dbl>,<chr>
KEGG,ko04013_MAPK signaling pathway - fly,5,5/411,5/2311,2.43e-03,NA,★ 検定されない（表に出ない）
KEGG,ko00120_Primary bile acid biosynthesis,5,4/411,5/2311,3.04e-02,NA,★ 検定されない（表に出ない）
KEGG,ko04016_MAPK signaling pathway - plant,6,6/411,6/2311,7.37e-04,NA,★ 検定されない（表に出ない）
KEGG,ko04068_FoxO signaling pathway,6,6/411,6/2311,7.37e-04,NA,★ 検定されない（表に出ない）
KEGG,ko01051_Biosynthesis of ansamycins,6,5/411,6/2311,8.24e-03,NA,★ 検定されない（表に出ない）
KEGG,ko00121_Secondary bile acid biosynthesis,7,6/411,7/2311,2.43e-03,NA,★ 検定されない（表に出ない）
KEGG,ko00660_C5-Branched dibasic acid metabolism,15,9/411,15/2311,3.58e-03,4.52e-03,検定された
KEGG,ko00480_Glutathione metabolism,17,11/411,17/2311,7.37e-04,6.69e-04,検定された
KEGG,ko00195_Photosynthesis,18,15/411,18/2311,2.63e-07,1.99e-07,検定された



  (A) down の有意 term 40 本のうち、既定値では 22 本が検定対象から外れる

■ PFAM:Sod_Fe_N（5遺伝子）と PFAM:Sod_Fe_C（5遺伝子）の一致数: 5
   → 同じ SOD タンパクの N 末／C 末ドメイン。中身は同一
        gene_id   strain Preferred_name                   product LFC_A
  CE91St8_30310  Bth_St8           sodB      superoxide dismutase -1.56
     BT_RS03240  Bth_VPI           sodB      superoxide dismutase -1.23
  NQ510_RS12460 Bun_8492           sodB      superoxide dismutase -1.03
 CE91St23_07220 Odo_St23           <NA> superoxide dismutase [Fe] -2.41
 CE91St23_11810 Odo_St23           sodB      superoxide dismutase -2.38

保存: /mnt/scratch/tanoue/261016_Xiaoxi_eptAoligo_2wFeces_Metatranscriptome/pathway/ora_all.csv（4692 行、geneID 列に当たった遺伝子）
project 複製: refqc/stage6_ora_all.csv


### 考察: 下がる側は4株そろって一貫、上がる側は経路に乗らない

#### 1. 有意になるのは圧倒的に down 側（Table 3）

| ソース | (A) down | (A) up | (B) down | (B) up |
|---|---|---|---|---|
| KEGG | **18 / 120** | 1 / 110 | **18 / 115** | 2 / 111 |
| COG | 3 / 19 | 4 / 20 | 3 / 19 | 3 / 20 |
| PFAM | **22 / 898** | 0 / 1263 | **30 / 897** | 0 / 1031 |
| CAZy | 0 / 19 | 0 / 19 | 0 / 19 | 0 / 12 |

DEG 数は up 1,616 / down 1,503 とほぼ同数なのに、**束になるのは down 側だけ。**
= KO で下がる遺伝子は機能的にまとまり、上がる遺伝子はばらばら。

#### 2. ★ (A) と (B) がほぼ一致する — これが一番重要

KEGG は **(A) 18 / (B) 18 本で 17 本が共通**（違いは (A) のみ `Lysine degradation`、(B) のみ
`Pyruvate metabolism`）、PFAM は **(A) 22 / (B) 30 本で 19 本が共通**。
`Bun_8492` の「見かけの一斉上昇」は (A) にしか出ない組成シフトだったのに、
**その違いが down の結果をほとんど動かさない** → **この down の束は組成シフトの産物ではなく、
各菌の内部で実際に起きている転写変化。**

#### 3. ★★ 経路名に騙されない — `geneID` から実体を全部引いた

| 出てきた term | 実体 |
|---|---|
| **Photosynthesis**（15遺伝子） | **15個すべて ATP synthase**（`atpA`–`atpH`、4株） |
| **Peroxisome** / **FoxO signaling**（10 / 6） | **`sodB` ×4株 ＋ `katA`**（FoxO の 6/6 が Peroxisome と重複） |
| **MAPK signaling - plant**（6） | **`copA`/`actP` 重金属排出 P型 ATPase ×5 ＋ `katA`** |
| **Carbon fixation by Calvin cycle**（15） | transketolase・`fbaB`・`rpiA/B`・`tpiA` = **PPP と解糖の共有酵素** |
| **Biosynthesis of ansamycins**（5） | transketolase のみ（**5/5 が PPP と重複**） |
| **Glutathione metabolism**（11） | `zwf`・`gnd`・`icd`（NADPH）＋ glutathione peroxidase |

eggNOG は細菌にも真核の経路名を付ける。セル5 の MGEC 4カテゴリの網では**これらは落ちない。**
→ **最終図では経路名ではなく中身の遺伝子名で書く。**

#### 4. ★ 一方、胆汁酸の2本は本物の細菌機能

`Secondary`（6遺伝子）/ `Primary`（4遺伝子、4/4 重複）の実体は **BSH（胆汁酸脱共役酵素）×4
＋ `7α-HSDH`**（LFC −0.80〜−2.28）。*Bacteroides* / *Odoribacteraceae* は実際に BSH を持つ。
脱抱合胆汁酸は宿主の FXR/TGR5 のリガンドなので、**(A)（宿主が何にさらされているか）の観点では
最も直接的な所見。** → 胆汁酸の LC-MS 実測があれば突き合わせる価値がある。

#### 5. ★ 足切り（`minGSSize=10`）を外して初めて見えたもの

`msrA`/`msrB`（`PMSR`/`SelR`）、`sodB`（`Sod_Fe_N/C`）、`sufB`/`sufD`（`UPF0051`）、
(4Fe-4S) 結合タンパク（`zf-CDGSH`/`Fer4_19`）、`HMA`、`6PGD`、`Thioredoxin` がいずれも
**4株そろって低下**。**SOD・カタラーゼ・Msr・チオレドキシン・NADPH 供給・Fe-S 組み立て・
重金属排出が系全体として下がっている。**

★ **既定値では失うのは有意性だけではない。**`M < 10` の term は `padj` が NA で表に現れず、
**見落としに気づけない。(A) down の有意 40 本のうち 22 本が既定値では検定対象外**（Table 5-7）。
**4株の定義群集では、生物学的に重要な機能ほど数遺伝子で代表される**（BSH は全データで 5〜7 個）。
19株・33株なら `M ≥ 10` を超えるので、`minGSSize = 10` は**定義群集に固有の落とし穴。**

#### 6. term の冗長性 — 本数を成果として数えない

有意18 term のヒット **239 個の実体は 142 遺伝子。**`Photosynthesis` 15/15 が
`Oxidative phosphorylation` に入れ子、`FoxO` 6/6 = `Peroxisome`、`Primary bile acid` 4/4 =
`Secondary`、`ansamycins` 5/5 = PPP、`Sod_Fe_N`=`Sod_Fe_C`、`PMSR`=`SelR`。
→ **「18本の経路が下がった」ではなく「数系統のまとまりが下がった」。**
ただし同じ事実が別ソース・別ドメインで独立に拾えている＝**検定が動いている確認**でもある。

#### 7. 実体のまとめ

**down**（4株すべてが寄与。St8 99 / VPI 107 / Bun 68 / Odo 120 遺伝子）
= エネルギー産生（ATP synthase 全サブユニット・酸化的リン酸化・TCA・propanoate）／
NADPH 供給（`zwf`・`gnd`・`icd`・PPP）／酸化ストレス応答／Fe-S クラスター／重金属排出／
胆汁酸変換／外膜・表層（`OmpA` fold 3.8、`P_gingi_FimA` は Odo 5遺伝子で最大 −4.79）

**up** = 翻訳と生合成のみ（`Ribosome` fold 1.73、COG J/H/F/I）。
**典型的な「増殖に振った」パターン。**

★ **`Bth` の ATP synthase が term 内で 1 遺伝子ずつしかないのはアノテーションの差**
（`atp` オペロンで `ko00195` が付くのは `atpG` だけ）。発現は下がっている（VPI −1.62 / St8 −1.79）ので、
**「`Bth` では ATP synthase が下がっていない」と読んではいけない。**

#### 8. up が経路に乗らない理由 / CAZy が 0 になった理由

up の正体は **`Bth_St8` の CPS locus switching**（ステージ5 PC1 42.96%）。CPS クラスタは株特異的で
KEGG 経路に対応が無く多くが hypothetical → **KEGG 1-2本・PFAM 0本**になる。
`CAZy:GH20` は (A) up 生 p=0.0058 / fold 3.06、(B) up 生 p=0.0278 / fold 3.11 と両スキーム同方向だが、
検定 term が 4→19 本に増えた BH の分母増で padj 0.109 / 0.111。**「有意でないが一致する候補」**として記録。

#### 9. ★ ORA の構造的な偏り: 高発現遺伝子に寄る

baseMean 中央値は KEGG あり全体 400 に対し **ATP synthase 2,665（6.7倍）/ Ribosome 3,428（8.6倍）**。
高発現遺伝子は検出力が高く DEG になりやすいので、**「エネルギーが下がって翻訳が上がった」が
本当の変化か検出力の差かを ORA では切り分けられない。**
ただし LFC 自体は 0.5〜1.8 の実質的な低下なので「わずかな差が高発現ゆえに有意」だけではない。
→ 閾値を切らない GSEA（セル8）で確認する。

#### 10. ★ 名前が紛らわしいもの

- **`CPSase_L_D2`** = carbamoyl-phosphate synthase（ピリミジン／アルギニン合成）。**CPS とは無関係**
- **`GNVR`** = Wzc 型チロシンキナーゼのドメインで**莢膜多糖の重合に関わる**（CPS と関係あり）
- **`P_gingi_FimA`** = Bacteroidetes 共通の表層フィンブリリン様ドメイン。当たったのは
  `Odo_St23` の5遺伝子のみで **Odo 特異的な表層構造の低下**

#### 保存

`pathway/ora_all.csv`（4,692 行、`geneID` 付き）→ project の `refqc/stage6_ora_all.csv`


## セル7: ORA の図（dotplot）

過去notebook（`5_Enrichment_pathway_R.ipynb` セル27 ほか）と同じ `dotplot(x, showCategory, label_format)`。

### ★ 違い1: 有意な term を全部出す

セル6 は `pvalueCutoff = 1` なのでオブジェクトに有意でない term も入っている。`showCategory = 10` だと
**「上位10本」に有意でないものが混ざる。**→ `@result` を `padj < 0.05` で絞ったコピー（`*_sig`）を作り、
`showCategory = nrow(...)` で全件表示する。

### ★ 違い2: PFAM 名のアンダースコアを保つ

`label_format` に**数値**を渡すと enrichplot の既定 labeller が `gsub("_", " ", str)` を実行する。
**`Pyr_redox_2` が `Pyr redox 2` になると Pfam で検索できない。**
→ **折り返しだけする関数 `lab60` を渡す。**

### ★ 違い3: KEGG は `koXXXXX_経路名` で表記する

**ko 番号を省略すると KEGG で引き直せない**ので必ず付ける（過去notebook の出力と同じ形式）。

### 図にするのは有意 term がある10通り

| scheme | direction | KEGG | COG | PFAM | CAZy |
|---|---|---|---|---|---|
| (A) | down | 18 | 3 | 22 | 0 |
| (A) | up | 1 | 4 | 0 | 0 |
| (B) | down | 18 | 3 | 30 | 0 |
| (B) | up | 2 | 3 | 0 | 0 |

**PFAM の up と CAZy は有意 0 なので図を作らない**（`dotplot` は 0 行だとエラーになる）。

### 図の読み方

y 軸 = term（`GeneRatio` 順）／x 軸 = `GeneRatio`／点の大きさ = `Count`／点の色 = `padj`。

★ **`GeneRatio` の分母はソースごとに違う**（KEGG 411 / COG 1,400 / PFAM 1,346）ので、
**ソースをまたいで x 軸の値を比べてはいけない。**

### ★ 違い4: 色を固定し、向きで色を変える

`dotplot` の既定は**パネルごとに自動スケール**するので、有意 term が2本しかない図では
「`padj` 0.0485 だけ」の目盛りになり**ほかの図と色が比べられない。**
→ **`padj = 0` を濃色、`0.05` を白に固定**する（過去notebook も `limits` を固定していた）。
白に近いほど「ぎりぎり有意」と読める。

| 向き | 色 |
|---|---|
| **down in KO**（WT で高い） | **青** 〜 白 |
| **up in KO**（WT で低い） | **赤** 〜 白 |

★ **`scale_colour_*` ではなく `scale_fill_*` を使う。** enrichplot 1.26.6 の `dotplot` は
`padj` を `fill` にマップしている（`enrichplot_point_shape` が **21** = 縁つき塗りつぶし円）。
**過去notebook の `scale_colour_gradient(...)` をそのまま足しても何も起きない。**
点の縁が黒いのも `shape 21` の既定で、このバージョンのままにする。

### ★ 違い5: `Count` の目盛りを実値に合わせる

`scale_size` の既定の目盛りは `pretty()` 由来で**実値の範囲を外れる**（図7-1 は実値 4〜31 なのに
目盛りが 0〜35、図7-4 は実値 41 のみなのに 40・50）。
→ **最小値と最大値を必ず含む4点**を `breaks` に渡す。
`limits` は指定しない（term が1本だと `limits = c(41, 41)` が範囲ゼロになって描けなくなる）。


In [79]:
# ── ORA の図（dotplot）─────────────────────────────────────────
# 過去notebook（5_Enrichment_pathway_R.ipynb セル27、
# 4_DESeq2_allmapped_fractionmode_R.ipynb セル107-108）と同じ書き方:
#   dotplot(x, showCategory=10, label_format = 60) + ggtitle(...)
#
# ★ 過去notebook との違い: showCategory を 10 に固定せず、有意な term を全部出す。
#   セル6 は pvalueCutoff = 1 で呼んでいるので、オブジェクトには
#   有意でない term も入っている。そのまま dotplot に渡すと
#   「上位10本」に有意でないものが混ざるので、
#   ★ @result を padj < 0.05 で絞ったコピーを作ってから渡す。
#
# 図の読み方
#   y 軸 = term（有意なものだけ。padj の小さい順ではなく GeneRatio 順に並ぶ）
#   x 軸 = GeneRatio（DEG のうちこの term に入った割合。セル6 Table 4 と同じ）
#   点の大きさ = Count（この term に入った DEG の数）
#   点の色 = padj
library(enrichplot)

# ── 有意 term だけに絞ったコピーを作る ──────────────────────────
# dotplot は enrichResult オブジェクトを取るので、@result を差し替える
A_dn_KEGG_sig = A_dn_KEGG; A_dn_KEGG_sig@result = A_dn_KEGG@result %>% filter(p.adjust < 0.05)
A_dn_COG_sig  = A_dn_COG ; A_dn_COG_sig@result  = A_dn_COG@result  %>% filter(p.adjust < 0.05)
A_dn_PFAM_sig = A_dn_PFAM; A_dn_PFAM_sig@result = A_dn_PFAM@result %>% filter(p.adjust < 0.05)
A_up_KEGG_sig = A_up_KEGG; A_up_KEGG_sig@result = A_up_KEGG@result %>% filter(p.adjust < 0.05)
A_up_COG_sig  = A_up_COG ; A_up_COG_sig@result  = A_up_COG@result  %>% filter(p.adjust < 0.05)
B_dn_KEGG_sig = B_dn_KEGG; B_dn_KEGG_sig@result = B_dn_KEGG@result %>% filter(p.adjust < 0.05)
B_dn_COG_sig  = B_dn_COG ; B_dn_COG_sig@result  = B_dn_COG@result  %>% filter(p.adjust < 0.05)
B_dn_PFAM_sig = B_dn_PFAM; B_dn_PFAM_sig@result = B_dn_PFAM@result %>% filter(p.adjust < 0.05)
B_up_KEGG_sig = B_up_KEGG; B_up_KEGG_sig@result = B_up_KEGG@result %>% filter(p.adjust < 0.05)
B_up_COG_sig  = B_up_COG ; B_up_COG_sig@result  = B_up_COG@result  %>% filter(p.adjust < 0.05)

# PFAM の up と CAZy は有意 term が 0 なので図を作らない
cat("■ Table 1: 図にする組み合わせ（有意 term の数）\n")
data.frame(
    scheme    = c(rep('(A) 全株プール', 8), rep('(B) 株ごと', 8)),
    direction = rep(c(rep('down', 4), rep('up', 4)), 2),
    source    = rep(c('KEGG', 'COG', 'PFAM', 'CAZy'), 4),
    有意term  = c(nrow(A_dn_KEGG_sig@result), nrow(A_dn_COG_sig@result),
                  nrow(A_dn_PFAM_sig@result), sum(A_dn_CAZy@result$p.adjust < 0.05),
                  nrow(A_up_KEGG_sig@result), nrow(A_up_COG_sig@result),
                  sum(A_up_PFAM@result$p.adjust < 0.05), sum(A_up_CAZy@result$p.adjust < 0.05),
                  nrow(B_dn_KEGG_sig@result), nrow(B_dn_COG_sig@result),
                  nrow(B_dn_PFAM_sig@result), sum(B_dn_CAZy@result$p.adjust < 0.05),
                  nrow(B_up_KEGG_sig@result), nrow(B_up_COG_sig@result),
                  sum(B_up_PFAM@result$p.adjust < 0.05), sum(B_up_CAZy@result$p.adjust < 0.05)),
    check.names = FALSE) %>%
    mutate(図 = ifelse(有意term > 0, '作る', '作らない（有意 0）'))

# ── y 軸ラベルの指定 ───────────────────────────────────────────
# ★ label_format に数値を渡すと、enrichplot の既定 labeller が
#   gsub("_", " ", str) でアンダースコアをスペースに置き換えてしまう。
#   PFAM 名は識別子なので Pyr_redox_2 が "Pyr redox 2" になると検索できない。
#   → 折り返しだけする関数を渡してアンダースコアを保つ
#   （過去notebook は KEGG 経路名だけだったので label_format = 60 で問題なかった）
lab60 = function(x) stringr::str_wrap(x, width = 60)

# ── 点の色（padj）の目盛り ──────────────────────────────────────
# 過去notebook（4_DESeq2_allmapped_fractionmode_R.ipynb セル107）と同じく
# 上限を固定する: + scale_colour_gradient(low="red", high="#F1D4E5", limits = c(0, 0.1))
#
# ★ ただし scale_colour_* ではなく scale_fill_* で指定する。
#   enrichplot 1.26.6 の dotplot は padj を colour ではなく fill にマップしており
#   （enrichplot:::enrichplot_point_shape = 21 の塗りつぶし点を使うため）、
#   scale_colour_gradient を足しても ★ 何も起きずに既定の配色のままになる。
#   点の縁が黒く描かれるのも shape 21 の既定（過去のバージョンは縁無しだった）。
#
# padj = 0 を濃色、0.05 を白に固定する。
#   ・dotplot の既定はパネルごとに自動スケールするので、図7-9 のように
#     有意 term が2本しかないと「padj 0.0485 だけ」の目盛りになり、
#     ほかの図と色が比べられない
#   ・有意判定の閾値が 0.05 なので、白に近いほど「ぎりぎり有意」と読める
#
# ★ 向きで色を変える: KO で低い（＝WT で高い）を青、KO で高い（＝WT で低い）を赤
col_down = scale_fill_gradient(low = 'blue', high = 'white',
                               limits = c(0, 0.05), name = 'p.adjust')
col_up   = scale_fill_gradient(low = 'red',  high = 'white',
                               limits = c(0, 0.05), name = 'p.adjust')

# ===============================================================
# (A) 全株プール = 宿主が何にさらされているか
# ===============================================================
# ── 図7-1: (A) KO で低い × KEGG（18 term、Count 4-31） ──
options(repr.plot.width = 9, repr.plot.height = 6.5)
dotplot(A_dn_KEGG_sig, showCategory = nrow(A_dn_KEGG_sig@result), label_format = lab60) +
    ggtitle('(A) pooled - down in KO - KEGG pathway') + col_down +
    scale_size_continuous(breaks = unique(round(seq(min(A_dn_KEGG_sig@result$Count),
                                                    max(A_dn_KEGG_sig@result$Count), length.out = 4))))

# ── 図7-2: (A) KO で低い × COG（3 term、Count 75-201） ──
options(repr.plot.width = 11, repr.plot.height = 3)
dotplot(A_dn_COG_sig, showCategory = nrow(A_dn_COG_sig@result), label_format = lab60) +
    ggtitle('(A) pooled - down in KO - COG category') + col_down +
    scale_size_continuous(breaks = unique(round(seq(min(A_dn_COG_sig@result$Count),
                                                    max(A_dn_COG_sig@result$Count), length.out = 4))))

# ── 図7-3: (A) KO で低い × PFAM（22 term、Count 4-20） ──
options(repr.plot.width = 8, repr.plot.height = 7.5)
dotplot(A_dn_PFAM_sig, showCategory = nrow(A_dn_PFAM_sig@result), label_format = lab60) +
    ggtitle('(A) pooled - down in KO - PFAM domain') + col_down +
    scale_size_continuous(breaks = unique(round(seq(min(A_dn_PFAM_sig@result$Count),
                                                    max(A_dn_PFAM_sig@result$Count), length.out = 4))))

# ── 図7-4: (A) KO で高い × KEGG（1 term、Count 41） ──
options(repr.plot.width = 9, repr.plot.height = 2.5)
dotplot(A_up_KEGG_sig, showCategory = nrow(A_up_KEGG_sig@result), label_format = lab60) +
    ggtitle('(A) pooled - up in KO - KEGG pathway') + col_up +
    scale_size_continuous(breaks = unique(round(seq(min(A_up_KEGG_sig@result$Count),
                                                    max(A_up_KEGG_sig@result$Count), length.out = 4))))

# ── 図7-5: (A) KO で高い × COG（4 term、Count 47-123） ──
options(repr.plot.width = 11, repr.plot.height = 3.2)
dotplot(A_up_COG_sig, showCategory = nrow(A_up_COG_sig@result), label_format = lab60) +
    ggtitle('(A) pooled - up in KO - COG category') + col_up +
    scale_size_continuous(breaks = unique(round(seq(min(A_up_COG_sig@result$Count),
                                                    max(A_up_COG_sig@result$Count), length.out = 4))))

# ===============================================================
# (B) 株ごと独立 = 菌内部の制御
# ===============================================================
# ── 図7-6: (B) KO で低い × KEGG（18 term、Count 4-33） ──
options(repr.plot.width = 9, repr.plot.height = 6.5)
dotplot(B_dn_KEGG_sig, showCategory = nrow(B_dn_KEGG_sig@result), label_format = lab60) +
    ggtitle('(B) per-strain - down in KO - KEGG pathway') + col_down +
    scale_size_continuous(breaks = unique(round(seq(min(B_dn_KEGG_sig@result$Count),
                                                    max(B_dn_KEGG_sig@result$Count), length.out = 4))))

# ── 図7-7: (B) KO で低い × COG（3 term、Count 72-199） ──
options(repr.plot.width = 11, repr.plot.height = 3)
dotplot(B_dn_COG_sig, showCategory = nrow(B_dn_COG_sig@result), label_format = lab60) +
    ggtitle('(B) per-strain - down in KO - COG category') + col_down +
    scale_size_continuous(breaks = unique(round(seq(min(B_dn_COG_sig@result$Count),
                                                    max(B_dn_COG_sig@result$Count), length.out = 4))))

# ── 図7-8: (B) KO で低い × PFAM（30 term、Count 4-19） ──
options(repr.plot.width = 8, repr.plot.height = 9.5)
dotplot(B_dn_PFAM_sig, showCategory = nrow(B_dn_PFAM_sig@result), label_format = lab60) +
    ggtitle('(B) per-strain - down in KO - PFAM domain') + col_down +
    scale_size_continuous(breaks = unique(round(seq(min(B_dn_PFAM_sig@result$Count),
                                                    max(B_dn_PFAM_sig@result$Count), length.out = 4))))

# ── 図7-9: (B) KO で高い × KEGG（2 term、Count 4-20） ──
options(repr.plot.width = 9, repr.plot.height = 2.8)
dotplot(B_up_KEGG_sig, showCategory = nrow(B_up_KEGG_sig@result), label_format = lab60) +
    ggtitle('(B) per-strain - up in KO - KEGG pathway') + col_up +
    scale_size_continuous(breaks = unique(round(seq(min(B_up_KEGG_sig@result$Count),
                                                    max(B_up_KEGG_sig@result$Count), length.out = 4))))

# ── 図7-10: (B) KO で高い × COG（3 term、Count 47-101） ──
options(repr.plot.width = 11, repr.plot.height = 3)
dotplot(B_up_COG_sig, showCategory = nrow(B_up_COG_sig@result), label_format = lab60) +
    ggtitle('(B) per-strain - up in KO - COG category') + col_up +
    scale_size_continuous(breaks = unique(round(seq(min(B_up_COG_sig@result$Count),
                                                    max(B_up_COG_sig@result$Count), length.out = 4))))

options(repr.plot.width = 7, repr.plot.height = 7)   # 既定に戻す

■ Table 1: 図にする組み合わせ（有意 term の数）


scheme,direction,source,有意term,図
<chr>,<chr>,<chr>,<int>,<chr>
(A) 全株プール,down,KEGG,18,作る
(A) 全株プール,down,COG,3,作る
(A) 全株プール,down,PFAM,22,作る
(A) 全株プール,down,CAZy,0,作らない（有意 0）
(A) 全株プール,up,KEGG,1,作る
(A) 全株プール,up,COG,4,作る
(A) 全株プール,up,PFAM,0,作らない（有意 0）
(A) 全株プール,up,CAZy,0,作らない（有意 0）
(B) 株ごと,down,KEGG,18,作る


Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Scale for size is already present.
Adding another scale for size, which will replace the existing scale.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Scale for size is already present.
Adding another scale for size, which will replace the existing scale.


plot without title

Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Scale for size is already present.
Adding another scale for size, which will replace the existing scale.


plot without title

... (11 outputs left out)

plot without title

Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Scale for size is already present.
Adding another scale for size, which will replace the existing scale.


plot without title

plot without title

### 考察: 図で見ると (A) と (B) がほぼ重なる

#### 1. (A) と (B) の KEGG 図（図7-1 と図7-6）はほぼ同じ絵になる

18 本中 17 本が共通で `GeneRatio` の順序もほぼ同じ（`Pentose phosphate pathway` が最も右、
`Oxidative phosphorylation` がその次）。
セル6 考察の「down の束は組成シフトの産物ではない」が図でも確認できる。

#### 2. ★ x 軸（`GeneRatio`）は重要度の順ではない

- `Pentose phosphate pathway`（31遺伝子、0.075）が最大だが **fold は 2.56 と中程度**
- `Photosynthesis`（= ATP synthase、15遺伝子）は 0.036 と中位だが **fold 4.69 で padj 最小**（2.6e-07）
- **胆汁酸の2本は左端**だが、これは term 内遺伝子数が 7 / 5 しかないため。**fold は 4.8 / 4.5 と高い**

**小さい term は必ず左に来る。** fold と padj を併せて読む必要がある。

#### 3. (B) の up に出る2本も中身を引いた（図7-9）

`Aminoacyl-tRNA biosynthesis`（20遺伝子）= **aminoacyl-tRNA 合成酵素**（`gltX`・`glnS`・`ileS`・
`proS`・`fmt`、LFC_B +0.50〜+1.08）。
`Ribosome biogenesis in eukaryotes`（4遺伝子）= **`rnc` RNase III のみ**（4株すべて、+0.85〜+1.52）。
後者は**また真核の経路名**だが、実体は rRNA 前駆体プロセシングの細菌酵素なので**方向としては正しい。**
→ **(A) の `Ribosome` と同じ「翻訳に振った」像**を別の角度から支持する。

#### 4. COG の図は点が3-4個しかない

21 カテゴリしかないので図としては情報量が乏しいが、**`Count` が 100-200 と大きい**
（COG M 201 / COG J 123 遺伝子）ので、**「機能カテゴリ全体が動いている」ことを示す図**として意味がある。

#### 5. 色を 0-0.05 に固定したので図をまたいでも図の中でも読める

パネルごとの自動スケールだと図7-9 は「0.0485 だけ」の目盛りになっていた。
固定したので**全図で色の意味が同じ**になり、**白に近いほど「ぎりぎり有意」**と読める。
図7-1 では `ko00030`（4.1e-06）と `ko00195`（2.6e-07）が真っ青、`ko04146`（0.035）・
`ko00310`（0.030）・`ko00120`（0.030）は薄い → **胆汁酸は fold は高いが padj はぎりぎり。**

#### 6. ★ CAZy の図が無いこと自体が所見

**4 スキーム全部で有意 0。**`GH20`（(A) up 生 p=0.0058）と `GH31`（(B) down 生 p=0.0222）は
**生の p では候補だが BH 後は届かない。**
*Bacteroides* の多糖分解酵素は **CDS の 2.0% にしかアノテーションが付いていない**ので、
**CAZy で群間差を検出するだけの検定力が元から無い。**
→ 多糖利用を見たいなら **PUL 単位でまとめる**など別の集約が要る。


## セル7-2: ORA の図（株ごとの内訳を入れた横棒）

セル7 の `dotplot` は**「どの株の遺伝子が当たっているか」を表せない。**
セル6 Table 4 に足した株ごとの列を図にする。**セル7 はそのまま残して併用する。**

### 図の作り

棒の長さ = **`GeneRatio`**（セル7 の x 軸と同じなので**点の x 位置と棒の右端が一致する**）／
棒の区分 = **各株の遺伝子数 ÷ `GeneRatio` の分母**（足すと棒全体になる）／
区分の塗り = **`padj`**（0-0.05 の固定スケール）／
区分の縁 = **株**（`Bth_VPI` 青 / `Bth_St8` 橙 / `Bun_8492` 緑 / `Odo_St23` 紫）／
y 軸の並び = **セル7 と同じ。**

### ★ 塗りの濃さをセル7 から落としてある

セル7 と同じ `low = 'blue'` / `'red'` にすると**塗りが濃すぎて株の縁が読めない。**実測した問題:
**down（青）は `Bth_VPI` の青の縁が塗りと同化して完全に見えなくなり、
up（赤）は `Bth_St8` の橙の縁が埋もれる。**
この図の目的は**株の内訳を読むこと**なので、塗りを薄く（`#8fb8f5` / `#f08080` 始まり）して縁を優先した。
**`padj` の強弱を読むのはセル7 の役目**という分担にする。

### 見た目の指定

| 指定 | 値 | 理由 |
|---|---|---|
| 縁の線幅 | `linewidth = 1.2` | 塗りを薄くしたのでこの太さでも区分が潰れない |
| 棒の太さ | `width = 0.45` | 既定 0.9 の半分。term ごとに読みやすい |
| x 軸の余白 | `expansion(mult = c(0, 0.05))` | **0 の左に余白を作らない**（棒の左端を軸に付ける） |
| 区分の並び | `position_stack(reverse = TRUE)` | **`geom_col` は既定で factor の levels を逆順に積む**ので、指定しないと右から `Bth_VPI` になる |


In [81]:
# ── ORA の図（株ごとの内訳を入れた横棒）─────────────────────────
# セル7 の dotplot は「どの株の遺伝子が当たっているか」を表せない。
# セル6 Table 4 の株ごとの列（Bth_VPI / Bth_St8 / Bun_8492 / Odo_St23）を
# 図にする。セル7 はそのまま残し、こちらを併用する。
#
# 図の作り
#   棒の長さ  = GeneRatio（dotplot の x 軸と同じ）
#   棒の区分  = 各株の遺伝子数 ÷ GeneRatio の分母。足すと棒全体になる
#   区分の塗り = padj（セル7 と同じ。down は青〜白、up は赤〜白）
#   区分の縁  = 株（ステージ5 と同じ色。Bth_VPI 青 / Bth_St8 橙 /
#                   Bun_8492 緑 / Odo_St23 紫）
#   区分の並び = 左から Bth_VPI → Bth_St8 → Bun_8492 → Odo_St23。
#               ★ geom_col は既定で factor の levels を逆順に積むので
#                 position_stack(reverse = TRUE) を指定して levels 順にする
#   y 軸の並び = セル7 と同じ（GeneRatio の降順、同値は padj の昇順）
library(ggplot2)

# ステージ5（5_1 / 5_2）と同じ株の色を使う
col_strain = c(Bth_VPI = '#2a78d6', Bth_St8 = '#eb6834', Bun_8492 = '#3f9e5f',
               Odo_St23 = '#9b59b6')

# ── 棒グラフ用のデータを作る ────────────────────────────────────
# セル6 の hit_strain（term × 株の遺伝子数）を縦持ちに戻し、
# GeneRatio の分母で割って「棒のうちその株が占める長さ」にする
bar_A = ora %>% filter(scheme == 'A', padj < 0.05) %>%
    left_join(hit_strain, by = c('scheme', 'direction', 'source', 'term')) %>%
    tidyr::pivot_longer(c(Bth_VPI, Bth_St8, Bun_8492, Odo_St23),
                        names_to = 'strain', values_to = 'n_gene') %>%
    mutate(denom = as.integer(sub('.*/', '', GeneRatio)),
           width = n_gene / denom,
           strain = factor(strain, levels = names(col_strain)))

bar_B = ora %>% filter(scheme == 'B', padj < 0.05) %>%
    left_join(hit_strain, by = c('scheme', 'direction', 'source', 'term')) %>%
    tidyr::pivot_longer(c(Bth_VPI, Bth_St8, Bun_8492, Odo_St23),
                        names_to = 'strain', values_to = 'n_gene') %>%
    mutate(denom = as.integer(sub('.*/', '', GeneRatio)),
           width = n_gene / denom,
           strain = factor(strain, levels = names(col_strain)))

cat("■ Table 1: 図にするデータ（1行 = term × 株）\n")
data.frame(
    scheme = c('(A) 全株プール', '(B) 株ごと'),
    行数 = c(nrow(bar_A), nrow(bar_B)),
    term数 = c(nrow(distinct(bar_A, direction, source, term)),
               nrow(distinct(bar_B, direction, source, term))),
    check.names = FALSE)

# y 軸の並びはセル6 Table 4 の順（= セル7 の図の順）をそのまま使う。
# ggplot は factor の levels 順に下から積むので、levels を逆順にして上から並べる
ord_A_dn_KEGG = ora %>% filter(scheme=='A', direction=='down', source=='KEGG', padj<0.05) %>% pull(term)
ord_A_dn_COG  = ora %>% filter(scheme=='A', direction=='down', source=='COG',  padj<0.05) %>% pull(term)
ord_A_dn_PFAM = ora %>% filter(scheme=='A', direction=='down', source=='PFAM', padj<0.05) %>% pull(term)
ord_A_up_KEGG = ora %>% filter(scheme=='A', direction=='up',   source=='KEGG', padj<0.05) %>% pull(term)
ord_A_up_COG  = ora %>% filter(scheme=='A', direction=='up',   source=='COG',  padj<0.05) %>% pull(term)
ord_B_dn_KEGG = ora %>% filter(scheme=='B', direction=='down', source=='KEGG', padj<0.05) %>% pull(term)
ord_B_dn_COG  = ora %>% filter(scheme=='B', direction=='down', source=='COG',  padj<0.05) %>% pull(term)
ord_B_dn_PFAM = ora %>% filter(scheme=='B', direction=='down', source=='PFAM', padj<0.05) %>% pull(term)
ord_B_up_KEGG = ora %>% filter(scheme=='B', direction=='up',   source=='KEGG', padj<0.05) %>% pull(term)
ord_B_up_COG  = ora %>% filter(scheme=='B', direction=='up',   source=='COG',  padj<0.05) %>% pull(term)

# 棒の塗り（padj）。セル7 と同じ 0-0.05 の固定スケールだが、
# ★ 濃さを落としてある。セル7 のように low='red' / low='blue' にすると
#   塗りが濃すぎて株の縁が読めなくなる。実測で確認した問題:
#     ・down（青）: Bth_VPI の青の縁が青い塗りと同化して ★ 完全に見えなくなる
#     ・up（赤）  : Bth_St8 の橙の縁が赤い塗りに埋もれる
#   この図の目的は株の内訳を読むことなので、塗りを薄くして縁を優先する。
#   セル7 側は濃いまま（padj の強弱を読む図）で、役割を分ける
fill_down = scale_fill_gradient(low = '#8fb8f5', high = '#eaf1fd',
                                limits = c(0, 0.05), name = 'p.adjust')
fill_up   = scale_fill_gradient(low = '#f08080', high = '#fdeaea',
                                limits = c(0, 0.05), name = 'p.adjust')

# 縁は株。凡例は縁の色だけが分かるように塗りを白にして見せる
col_edge = scale_colour_manual(values = col_strain, name = 'strain',
    guide = guide_legend(override.aes = list(fill = 'white', linewidth = 1)))

thm_bar = theme_bw() +
    theme(panel.grid.major.y = element_blank(),
          panel.grid.minor = element_blank(),
          legend.key.size = unit(0.9, 'lines'))

# x 軸は 0 の左に余白を作らない（棒の左端を軸に付ける）。右だけ少し空ける
x_bar = scale_x_continuous(expand = expansion(mult = c(0, 0.05)))
# ===============================================================
# (A) 全株プール = 宿主が何にさらされているか
# ===============================================================
# ── 図7-2-1: (A) KO で低い × KEGG（18 term） ──
options(repr.plot.width = 11, repr.plot.height = 6.5)
ggplot(bar_A %>% filter(direction == 'down', source == 'KEGG') %>%
           mutate(term = factor(term, levels = rev(ord_A_dn_KEGG))),
       aes(x = width, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45,
             position = position_stack(reverse = TRUE)) + fill_down + col_edge + thm_bar + x_bar +
    labs(x = 'GeneRatio (stacked by strain)', y = NULL,
         title = '(A) pooled - down in KO - KEGG pathway')

# ── 図7-2-2: (A) KO で低い × COG（3 term） ──
options(repr.plot.width = 13, repr.plot.height = 2.8)
ggplot(bar_A %>% filter(direction == 'down', source == 'COG') %>%
           mutate(term = factor(term, levels = rev(ord_A_dn_COG))),
       aes(x = width, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45,
             position = position_stack(reverse = TRUE)) + fill_down + col_edge + thm_bar + x_bar +
    labs(x = 'GeneRatio (stacked by strain)', y = NULL,
         title = '(A) pooled - down in KO - COG category')

# ── 図7-2-3: (A) KO で低い × PFAM（22 term） ──
options(repr.plot.width = 10, repr.plot.height = 7.5)
ggplot(bar_A %>% filter(direction == 'down', source == 'PFAM') %>%
           mutate(term = factor(term, levels = rev(ord_A_dn_PFAM))),
       aes(x = width, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45,
             position = position_stack(reverse = TRUE)) + fill_down + col_edge + thm_bar + x_bar +
    labs(x = 'GeneRatio (stacked by strain)', y = NULL,
         title = '(A) pooled - down in KO - PFAM domain')

# ── 図7-2-4: (A) KO で高い × KEGG（1 term） ──
options(repr.plot.width = 11, repr.plot.height = 2.2)
ggplot(bar_A %>% filter(direction == 'up', source == 'KEGG') %>%
           mutate(term = factor(term, levels = rev(ord_A_up_KEGG))),
       aes(x = width, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45,
             position = position_stack(reverse = TRUE)) + fill_up + col_edge + thm_bar + x_bar +
    labs(x = 'GeneRatio (stacked by strain)', y = NULL,
         title = '(A) pooled - up in KO - KEGG pathway')

# ── 図7-2-5: (A) KO で高い × COG（4 term） ──
options(repr.plot.width = 13, repr.plot.height = 3.0)
ggplot(bar_A %>% filter(direction == 'up', source == 'COG') %>%
           mutate(term = factor(term, levels = rev(ord_A_up_COG))),
       aes(x = width, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45,
             position = position_stack(reverse = TRUE)) + fill_up + col_edge + thm_bar + x_bar +
    labs(x = 'GeneRatio (stacked by strain)', y = NULL,
         title = '(A) pooled - up in KO - COG category')

# ===============================================================
# (B) 株ごと独立 = 菌内部の制御
# ===============================================================
# ── 図7-2-6: (B) KO で低い × KEGG（18 term） ──
options(repr.plot.width = 11, repr.plot.height = 6.5)
ggplot(bar_B %>% filter(direction == 'down', source == 'KEGG') %>%
           mutate(term = factor(term, levels = rev(ord_B_dn_KEGG))),
       aes(x = width, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45,
             position = position_stack(reverse = TRUE)) + fill_down + col_edge + thm_bar + x_bar +
    labs(x = 'GeneRatio (stacked by strain)', y = NULL,
         title = '(B) per-strain - down in KO - KEGG pathway')

# ── 図7-2-7: (B) KO で低い × COG（3 term） ──
options(repr.plot.width = 13, repr.plot.height = 2.8)
ggplot(bar_B %>% filter(direction == 'down', source == 'COG') %>%
           mutate(term = factor(term, levels = rev(ord_B_dn_COG))),
       aes(x = width, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45,
             position = position_stack(reverse = TRUE)) + fill_down + col_edge + thm_bar + x_bar +
    labs(x = 'GeneRatio (stacked by strain)', y = NULL,
         title = '(B) per-strain - down in KO - COG category')

# ── 図7-2-8: (B) KO で低い × PFAM（30 term） ──
options(repr.plot.width = 10, repr.plot.height = 9.5)
ggplot(bar_B %>% filter(direction == 'down', source == 'PFAM') %>%
           mutate(term = factor(term, levels = rev(ord_B_dn_PFAM))),
       aes(x = width, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45,
             position = position_stack(reverse = TRUE)) + fill_down + col_edge + thm_bar + x_bar +
    labs(x = 'GeneRatio (stacked by strain)', y = NULL,
         title = '(B) per-strain - down in KO - PFAM domain')

# ── 図7-2-9: (B) KO で高い × KEGG（2 term） ──
options(repr.plot.width = 11, repr.plot.height = 2.5)
ggplot(bar_B %>% filter(direction == 'up', source == 'KEGG') %>%
           mutate(term = factor(term, levels = rev(ord_B_up_KEGG))),
       aes(x = width, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45,
             position = position_stack(reverse = TRUE)) + fill_up + col_edge + thm_bar + x_bar +
    labs(x = 'GeneRatio (stacked by strain)', y = NULL,
         title = '(B) per-strain - up in KO - KEGG pathway')

# ── 図7-2-10: (B) KO で高い × COG（3 term） ──
options(repr.plot.width = 13, repr.plot.height = 2.8)
ggplot(bar_B %>% filter(direction == 'up', source == 'COG') %>%
           mutate(term = factor(term, levels = rev(ord_B_up_COG))),
       aes(x = width, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45,
             position = position_stack(reverse = TRUE)) + fill_up + col_edge + thm_bar + x_bar +
    labs(x = 'GeneRatio (stacked by strain)', y = NULL,
         title = '(B) per-strain - up in KO - COG category')

options(repr.plot.width = 7, repr.plot.height = 7)   # 既定に戻す

■ Table 1: 図にするデータ（1行 = term × 株）


scheme,行数,term数
<chr>,<int>,<int>
(A) 全株プール,192,48
(B) 株ごと,224,56


plot without title

plot without title

plot without title

plot without title

... (2 outputs left out)

plot without title

plot without title

plot without title

plot without title

### 考察: 棒の区分で「どの菌の応答か」が一目で分かる

#### 1. (A) down KEGG（図7-2-1）— 区分の幅が株ごとの寄与

セル6 Table 4 の数字がそのまま棒の区分になっている。

- **`ko00020_Citrate cycle` と `ko00640_Propanoate metabolism`** は
  **青（VPI）と橙（St8）の区分が棒の 8 割**を占める → **`Bth` 2株の経路**
- **`ko00195_Photosynthesis`（ATP synthase）と `ko00190_Oxidative phosphorylation`** は
  **緑（Bun）と紫（Odo）が大半** → **`Bun`/`Odo` の経路**
- **`ko00030_Pentose phosphate pathway` と `ko00480_Glutathione metabolism`** は
  **4色がほぼ均等** → **4株共通の応答**
- **`ko00120_Primary bile acid biosynthesis` と `ko01051_Biosynthesis of ansamycins`** は
  **青と橙の区分が無い**（幅 0）→ `Bth` 2株には当たっていない

#### 2. (A) up COG（図7-2-5）— `Bun_8492` の緑が棒を占める

`COG J`（翻訳）は **緑（Bun）の区分が棒の 8 割**。
`COG H`・`COG F`・`COG I` も同様で、**up 側は `Bun_8492` 1株の現象**であることが
セル6 Table 4 の数字（Bun 77.6-82.9%）より直感的に分かる。

#### 3. (B) と見比べる（図7-2-6 以降）

(B) の down は**4色がより均等**になる。
(B) の up（図7-2-10）でも緑は大きいが、(A) ほど極端ではない（Bun 50.7%）。
**(A) で Bun に偏るのが正規化範囲の違いによる組成シフト**だということが、
2つの図を並べると色の面積で見える。

#### 4. この図で読めないこと

- **棒の長さ（`GeneRatio`）はソースをまたいで比べられない。**分母が違う（KEGG 411 / COG 1,400 / PFAM 1,346）
- **`fold` が入っていない。**棒が短くても fold が高い term がある
  （`ko00120_Primary bile acid biosynthesis` は `GeneRatio` 4/411 だが fold 4.50）。
  **fold と padj の数値はセル6 Table 4 で確認する**
- **区分の幅は「その株の遺伝子が何個当たったか」であって「その株でどれだけ強く変動したか」ではない。**
  株ごとの濃縮そのものはセル8（(A)）とセル10 以降（(B)）で検定する

In [149]:
# ── セル7-3: ORA の図（株を「転写量」で順位づける）───────────────
# セル7-2 の棒は GeneRatio（遺伝子数）で株を積み上げている。
# ★ これだと「少数の高発現遺伝子を持つ株」が埋もれる。
#   本データは株の大きさが16倍違う（Table 1）ので、遺伝子数の順位は
#   転写量の順位とほぼ裏返しになる（CLAUDE.md §7 ステージ6 残件3）。
#
# セル7-2 は「制御の広がり（何本の遺伝子が動いたか）」の図として残し、
# こちらは「効果の大きさ（群集の転写プールが実際にどれだけ動いたか）」を出す。
# 2つは問うていることが違うので、どちらかに置き換えるのではなく併用する。
#
# 図の作り（1つの図＝左右2枚組）
#   左  = 正味リード増減（norm_diff の和 = KO − WT）。リニア軸なので
#         ★ 棒の長さがそのまま「量」。down は負なので左に伸びる
#   右  = 同じ棒を 100% に正規化した構成比。
#         ★ 左だけだと量の小さい term（PFAM:GNVR の −582 など）が
#           線になって株の内訳が読めないので、構成比を必ず隣に置く
#   区分の塗り = padj（セル7-2 と同じ）／区分の縁 = 株（同じ色）
#   y 軸の並び = セル7・セル7-2 と完全に同じ（GeneRatio 降順）
library(ggplot2)
library(patchwork)

options(repr.matrix.max.rows = 500, repr.matrix.max.cols = 50)

# ステージ5（5_1 / 5_2）・セル7-2 と同じ株の色
col_strain = c(Bth_VPI = '#2a78d6', Bth_St8 = '#eb6834', Bun_8492 = '#3f9e5f',
               Odo_St23 = '#9b59b6')

# ===============================================================
# 1. 株の「大きさ」── 遺伝子数の順位が歪む理由
# ===============================================================
# ★ この表を先に出しておかないと、以降の図が読めない。
#   Odo_St23 は DEG 数が最多なのに群集の転写の 3.8% しか占めていない
cat("■ Table 1: 株の大きさ（scheme A の正規化値、マーカー除外・検定されたCDSのみ）\n")
sz = res %>% filter(!marker, !is.na(padj_A))
data.frame(
    strain = c('Bth_VPI', 'Bth_St8', 'Bun_8492', 'Odo_St23', 'pME008'),
    検定CDS数 = sapply(c('Bth_VPI','Bth_St8','Bun_8492','Odo_St23','pME008'),
                       function(s) sum(sz$strain == s)),
    WT転写量 = sapply(c('Bth_VPI','Bth_St8','Bun_8492','Odo_St23','pME008'),
                      function(s) round(sum(sz$normA_WT[sz$strain == s]))),
    KO転写量 = sapply(c('Bth_VPI','Bth_St8','Bun_8492','Odo_St23','pME008'),
                      function(s) round(sum(sz$normA_KO[sz$strain == s]))),
    占有率WT_pct = sapply(c('Bth_VPI','Bth_St8','Bun_8492','Odo_St23','pME008'),
                      function(s) round(100*sum(sz$normA_WT[sz$strain == s])/sum(sz$normA_WT), 1)),
    占有率KO_pct = sapply(c('Bth_VPI','Bth_St8','Bun_8492','Odo_St23','pME008'),
                      function(s) round(100*sum(sz$normA_KO[sz$strain == s])/sum(sz$normA_KO), 1)),
    一CDSあたりWT = sapply(c('Bth_VPI','Bth_St8','Bun_8492','Odo_St23','pME008'),
                      function(s) round(sum(sz$normA_WT[sz$strain == s])/sum(sz$strain == s))),
    down_DEG数 = sapply(c('Bth_VPI','Bth_St8','Bun_8492','Odo_St23','pME008'),
                      function(s) sum(sz$strain == s & sz$padj_A < 0.05 & sz$LFC_A < 0)),
    up_DEG数 = sapply(c('Bth_VPI','Bth_St8','Bun_8492','Odo_St23','pME008'),
                      function(s) sum(sz$strain == s & sz$padj_A < 0.05 & sz$LFC_A > 0)),
    down正味 = sapply(c('Bth_VPI','Bth_St8','Bun_8492','Odo_St23','pME008'),
                      function(s) round(sum(sz$normA_diff[sz$strain == s & sz$padj_A < 0.05 & sz$LFC_A < 0]))),
    up正味 = sapply(c('Bth_VPI','Bth_St8','Bun_8492','Odo_St23','pME008'),
                      function(s) round(sum(sz$normA_diff[sz$strain == s & sz$padj_A < 0.05 & sz$LFC_A > 0]))),
    row.names = NULL, check.names = FALSE)

# ===============================================================
# 2. 有意 term ごとに、当たった遺伝子を株別に集計する
# ===============================================================
# セル6 の hit_strain は遺伝子数だけ。ここは正規化リード値も必要なので
# geneID をばらして res から値を引き直す。
# ★ scheme A は normA_*、scheme B は normB_* を使う
hit_long = ora %>% filter(padj < 0.05) %>%
    dplyr::select(scheme, direction, source, term, geneID) %>%
    tidyr::separate_rows(geneID, sep = '/') %>%
    left_join(res %>% dplyr::select(gene_id, strain, product, Preferred_name,
                                    normA_WT, normA_KO, normA_diff,
                                    normB_WT, normB_KO, normB_diff),
              by = c('geneID' = 'gene_id')) %>%
    mutate(WT   = ifelse(scheme == 'A', normA_WT,   normB_WT),
           KO   = ifelse(scheme == 'A', normA_KO,   normB_KO),
           diff = ifelse(scheme == 'A', normA_diff, normB_diff))

# 株ごとに集計し、★ 当たり遺伝子が0本の株も行として残す（0 を明示する）
hit_net = hit_long %>%
    group_by(scheme, direction, source, term, strain) %>%
    summarise(n_gene = n(), WT = sum(WT), KO = sum(KO), net = sum(diff), .groups = 'drop') %>%
    tidyr::complete(tidyr::nesting(scheme, direction, source, term),
                    strain = names(col_strain),
                    fill = list(n_gene = 0, WT = 0, KO = 0, net = 0)) %>%
    group_by(scheme, direction, source, term) %>%
    mutate(net_tot = sum(net), n_tot = sum(n_gene),
           sh_net  = 100 * net / net_tot,       # down は分子分母とも負なので正になる
           sh_gene = 100 * n_gene / n_tot) %>%
    ungroup() %>%
    mutate(strain = factor(strain, levels = names(col_strain))) %>%
    left_join(ora %>% filter(padj < 0.05) %>%
                  dplyr::select(scheme, direction, source, term, GeneRatio, padj),
              by = c('scheme', 'direction', 'source', 'term'))

cat(sprintf("\n  集計: 有意 term %d 本 × 4株 = %d 行\n",
            nrow(distinct(hit_net, scheme, direction, source, term)), nrow(hit_net)))
# 検算: 株別の遺伝子数の合計が Count（GeneRatio の分子）と一致するか
cat(sprintf("  検算: 4株の遺伝子数合計 ≠ GeneRatio の分子 となる term = %d（0 なら整合）\n",
    hit_net %>% distinct(scheme, direction, source, term, n_tot, GeneRatio) %>%
        filter(n_tot != as.integer(sub('/.*', '', GeneRatio))) %>% nrow()))
# 検算: 株別の正味の合計が、term の全遺伝子の norm_diff の和と一致するか
cat(sprintf("  検算: 正味の合計がずれる term = %d（0 なら整合）\n",
    hit_net %>% group_by(scheme, direction, source, term) %>%
        summarise(s = sum(net), t = first(net_tot), .groups = 'drop') %>%
        filter(abs(s - t) > 1e-6) %>% nrow()))

# ===============================================================
# 3. Table 2: 遺伝子数での1位 と 転写量での1位 を並べる
# ===============================================================
cat("\n■ Table 2: term ごとに「遺伝子数で1位の株」と「転写量で1位の株」\n")
cat("  ★ 一致 列が『違う』の行が、セル7-2 の図と順位が逆になる term\n")
hit_net %>% group_by(scheme, direction, source, term) %>%
    summarise(GeneRatio = first(GeneRatio), padj = signif(first(padj), 2),
              正味合計 = round(first(net_tot)),
              # 遺伝子数1位
              数1位 = as.character(strain[which.max(n_gene)]),
              数1位_n = n_gene[which.max(n_gene)],
              数1位_pct = round(sh_gene[which.max(n_gene)]),
              # 転写量1位（down は net が最小、up は最大）
              量1位 = as.character(strain[if (first(direction) == 'down') which.min(net) else which.max(net)]),
              量1位_net = round(net[if (first(direction) == 'down') which.min(net) else which.max(net)]),
              量1位_pct = round(sh_net[if (first(direction) == 'down') which.min(net) else which.max(net)]),
              .groups = 'drop') %>%
    mutate(一致 = ifelse(数1位 == 量1位, '', '★ 違う')) %>%
    arrange(scheme, direction, source, term) %>% as.data.frame()

cat("\n■ Table 2-2: 株ごとの集計 ── 遺伝子数で1位になった回数 と 転写量で1位になった回数\n")
rk = hit_net %>% group_by(scheme, direction, source, term) %>%
    mutate(順位_数 = rank(-n_gene, ties.method = 'min'),
           順位_量 = ifelse(first(direction) == 'down', rank(net, ties.method = 'min'),
                                                        rank(-net, ties.method = 'min'))) %>%
    ungroup()
data.frame(
    strain = names(col_strain),
    遺伝子数1位 = sapply(names(col_strain), function(s) sum(rk$順位_数[rk$strain == s] == 1)),
    転写量1位 = sapply(names(col_strain), function(s) sum(rk$順位_量[rk$strain == s] == 1)),
    転写量4位 = sapply(names(col_strain), function(s) sum(rk$順位_量[rk$strain == s] == 4)),
    row.names = NULL, check.names = FALSE)

# ===============================================================
# 4. Table 3: term の正味増減のうち「最大の1遺伝子」が占める割合
# ===============================================================
# ★ 割合が高い term は、遺伝子数で数えても実体が見えない。
#   KEGG のラベルを字義通りに読めない例（ko04146 Peroxisome など）が
#   ここで一気に露出する
cat("\n■ Table 3: term の正味増減のうち、最大の1遺伝子が占める割合（降順）\n")
conc = hit_long %>% group_by(scheme, direction, source, term) %>%
    summarise(n_gene = n(), 正味合計 = round(sum(diff)),
              最大遺伝子 = geneID[which.max(abs(diff))],
              株 = as.character(strain[which.max(abs(diff))]),
              name = Preferred_name[which.max(abs(diff))],
              product = substr(product[which.max(abs(diff))], 1, 40),
              その遺伝子の正味 = round(diff[which.max(abs(diff))]), .groups = 'drop') %>%
    mutate(占有率_pct = round(100 * その遺伝子の正味 / 正味合計))
conc %>% arrange(desc(占有率_pct)) %>% as.data.frame()

cat(sprintf("\n  1遺伝子で term の正味増減の 50%% 以上を作っている term: %d / %d 本\n",
            sum(conc$占有率_pct >= 50), nrow(conc)))
cat(sprintf("  同 80%% 以上: %d 本\n", sum(conc$占有率_pct >= 80)))

# ===============================================================
# 5. 図7-3: 正味リード増減の積み上げ棒（左）＋ 構成比（右）
# ===============================================================
# y 軸の並びはセル6 Table 4 の順（= セル7・セル7-2 の図の順）をそのまま使う。
# ggplot は factor の levels 順に下から積むので levels を逆順にして上から並べる
ord_A_dn_KEGG = ora %>% filter(scheme=='A', direction=='down', source=='KEGG', padj<0.05) %>% pull(term)
ord_A_dn_COG  = ora %>% filter(scheme=='A', direction=='down', source=='COG',  padj<0.05) %>% pull(term)
ord_A_dn_PFAM = ora %>% filter(scheme=='A', direction=='down', source=='PFAM', padj<0.05) %>% pull(term)
ord_A_up_KEGG = ora %>% filter(scheme=='A', direction=='up',   source=='KEGG', padj<0.05) %>% pull(term)
ord_A_up_COG  = ora %>% filter(scheme=='A', direction=='up',   source=='COG',  padj<0.05) %>% pull(term)
ord_B_dn_KEGG = ora %>% filter(scheme=='B', direction=='down', source=='KEGG', padj<0.05) %>% pull(term)
ord_B_dn_COG  = ora %>% filter(scheme=='B', direction=='down', source=='COG',  padj<0.05) %>% pull(term)
ord_B_dn_PFAM = ora %>% filter(scheme=='B', direction=='down', source=='PFAM', padj<0.05) %>% pull(term)
ord_B_up_KEGG = ora %>% filter(scheme=='B', direction=='up',   source=='KEGG', padj<0.05) %>% pull(term)
ord_B_up_COG  = ora %>% filter(scheme=='B', direction=='up',   source=='COG',  padj<0.05) %>% pull(term)

# 棒の塗り（padj）・縁（株）・テーマはセル7-2 と完全に同じにする。
# セル7-2 と見比べる図なので、色の意味をずらさない
fill_down = scale_fill_gradient(low = '#8fb8f5', high = '#eaf1fd',
                                limits = c(0, 0.05), name = 'p.adjust')
fill_up   = scale_fill_gradient(low = '#f08080', high = '#fdeaea',
                                limits = c(0, 0.05), name = 'p.adjust')
col_edge = scale_colour_manual(values = col_strain, name = 'strain',
    guide = guide_legend(override.aes = list(fill = 'white', linewidth = 1)))
thm_bar = theme_bw() +
    theme(panel.grid.major.y = element_blank(),
          panel.grid.minor = element_blank(),
          legend.key.size = unit(0.9, 'lines'))

# ★ 1つの図＝左右2枚組を作る。10通り（scheme × direction × source）で
#   同じことを書くので、下では同じ書き方を10回くり返す（CLAUDE.md §9:
#   読みやすさを優先し、関数にまとめない）

# ── 図7-3-1: (A) KO で低い × KEGG（18 term） ──
options(repr.plot.width = 15, repr.plot.height = 6.5)
d = hit_net %>% filter(scheme == 'A', direction == 'down', source == 'KEGG') %>%
    mutate(term = factor(term, levels = rev(ord_A_dn_KEGG)))
# 左: 正味リード増減そのもの（リニア軸。棒の長さ = 量）
g_L = ggplot(d, aes(x = net, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
    fill_down + col_edge + thm_bar +
    scale_x_continuous(labels = scales::comma,
                       expand = expansion(mult = c(0.05, 0))) +
    labs(x = 'net normalized reads (KO - WT)', y = NULL,
         title = '(A) pooled - down in KO - KEGG pathway', subtitle = 'transcript amount')
# 右: 同じ棒を 100% に正規化した構成比（量の小さい term でも内訳が読める）
g_R = ggplot(d, aes(x = sh_net, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
    fill_down + col_edge + thm_bar +
    # ★ limits = c(0, 100) は使わない。position_stack の累積和が
    #   浮動小数で 100 をわずかに超えることがあり、その棒が黙って消える
    scale_x_continuous(breaks = c(0, 25, 50, 75, 100),
                       expand = expansion(mult = c(0, 0.02))) +
    labs(x = 'strain composition (%)', y = NULL, subtitle = 'composition')
g_L + (g_R + theme(axis.text.y = element_blank())) +
    plot_layout(widths = c(1, 0.55), guides = 'collect')

# ── 図7-3-2: (A) KO で低い × COG（3 term） ──
options(repr.plot.width = 17, repr.plot.height = 2.8)
d = hit_net %>% filter(scheme == 'A', direction == 'down', source == 'COG') %>%
    mutate(term = factor(term, levels = rev(ord_A_dn_COG)))
# 左: 正味リード増減そのもの（リニア軸。棒の長さ = 量）
g_L = ggplot(d, aes(x = net, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
    fill_down + col_edge + thm_bar +
    scale_x_continuous(labels = scales::comma,
                       expand = expansion(mult = c(0.05, 0))) +
    labs(x = 'net normalized reads (KO - WT)', y = NULL,
         title = '(A) pooled - down in KO - COG category', subtitle = 'transcript amount')
# 右: 同じ棒を 100% に正規化した構成比（量の小さい term でも内訳が読める）
g_R = ggplot(d, aes(x = sh_net, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
    fill_down + col_edge + thm_bar +
    # ★ limits = c(0, 100) は使わない。position_stack の累積和が
    #   浮動小数で 100 をわずかに超えることがあり、その棒が黙って消える
    scale_x_continuous(breaks = c(0, 25, 50, 75, 100),
                       expand = expansion(mult = c(0, 0.02))) +
    labs(x = 'strain composition (%)', y = NULL, subtitle = 'composition')
g_L + (g_R + theme(axis.text.y = element_blank())) +
    plot_layout(widths = c(1, 0.55), guides = 'collect')

# ── 図7-3-3: (A) KO で低い × PFAM（22 term） ──
options(repr.plot.width = 14, repr.plot.height = 7.5)
d = hit_net %>% filter(scheme == 'A', direction == 'down', source == 'PFAM') %>%
    mutate(term = factor(term, levels = rev(ord_A_dn_PFAM)))
# 左: 正味リード増減そのもの（リニア軸。棒の長さ = 量）
g_L = ggplot(d, aes(x = net, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
    fill_down + col_edge + thm_bar +
    scale_x_continuous(labels = scales::comma,
                       expand = expansion(mult = c(0.05, 0))) +
    labs(x = 'net normalized reads (KO - WT)', y = NULL,
         title = '(A) pooled - down in KO - PFAM domain', subtitle = 'transcript amount')
# 右: 同じ棒を 100% に正規化した構成比（量の小さい term でも内訳が読める）
g_R = ggplot(d, aes(x = sh_net, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
    fill_down + col_edge + thm_bar +
    # ★ limits = c(0, 100) は使わない。position_stack の累積和が
    #   浮動小数で 100 をわずかに超えることがあり、その棒が黙って消える
    scale_x_continuous(breaks = c(0, 25, 50, 75, 100),
                       expand = expansion(mult = c(0, 0.02))) +
    labs(x = 'strain composition (%)', y = NULL, subtitle = 'composition')
g_L + (g_R + theme(axis.text.y = element_blank())) +
    plot_layout(widths = c(1, 0.55), guides = 'collect')

# ── 図7-3-4: (A) KO で高い × KEGG（1 term） ──
options(repr.plot.width = 15, repr.plot.height = 2.2)
d = hit_net %>% filter(scheme == 'A', direction == 'up', source == 'KEGG') %>%
    mutate(term = factor(term, levels = rev(ord_A_up_KEGG)))
# 左: 正味リード増減そのもの（リニア軸。棒の長さ = 量）
g_L = ggplot(d, aes(x = net, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
    fill_up + col_edge + thm_bar +
    scale_x_continuous(labels = scales::comma,
                       expand = expansion(mult = c(0, 0.05))) +
    labs(x = 'net normalized reads (KO - WT)', y = NULL,
         title = '(A) pooled - up in KO - KEGG pathway', subtitle = 'transcript amount')
# 右: 同じ棒を 100% に正規化した構成比（量の小さい term でも内訳が読める）
g_R = ggplot(d, aes(x = sh_net, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
    fill_up + col_edge + thm_bar +
    # ★ limits = c(0, 100) は使わない。position_stack の累積和が
    #   浮動小数で 100 をわずかに超えることがあり、その棒が黙って消える
    scale_x_continuous(breaks = c(0, 25, 50, 75, 100),
                       expand = expansion(mult = c(0, 0.02))) +
    labs(x = 'strain composition (%)', y = NULL, subtitle = 'composition')
g_L + (g_R + theme(axis.text.y = element_blank())) +
    plot_layout(widths = c(1, 0.55), guides = 'collect')

# ── 図7-3-5: (A) KO で高い × COG（4 term） ──
options(repr.plot.width = 17, repr.plot.height = 3.0)
d = hit_net %>% filter(scheme == 'A', direction == 'up', source == 'COG') %>%
    mutate(term = factor(term, levels = rev(ord_A_up_COG)))
# 左: 正味リード増減そのもの（リニア軸。棒の長さ = 量）
g_L = ggplot(d, aes(x = net, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
    fill_up + col_edge + thm_bar +
    scale_x_continuous(labels = scales::comma,
                       expand = expansion(mult = c(0, 0.05))) +
    labs(x = 'net normalized reads (KO - WT)', y = NULL,
         title = '(A) pooled - up in KO - COG category', subtitle = 'transcript amount')
# 右: 同じ棒を 100% に正規化した構成比（量の小さい term でも内訳が読める）
g_R = ggplot(d, aes(x = sh_net, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
    fill_up + col_edge + thm_bar +
    # ★ limits = c(0, 100) は使わない。position_stack の累積和が
    #   浮動小数で 100 をわずかに超えることがあり、その棒が黙って消える
    scale_x_continuous(breaks = c(0, 25, 50, 75, 100),
                       expand = expansion(mult = c(0, 0.02))) +
    labs(x = 'strain composition (%)', y = NULL, subtitle = 'composition')
g_L + (g_R + theme(axis.text.y = element_blank())) +
    plot_layout(widths = c(1, 0.55), guides = 'collect')

# ── 図7-3-6: (B) KO で低い × KEGG（18 term） ──
options(repr.plot.width = 15, repr.plot.height = 6.5)
d = hit_net %>% filter(scheme == 'B', direction == 'down', source == 'KEGG') %>%
    mutate(term = factor(term, levels = rev(ord_B_dn_KEGG)))
# 左: 正味リード増減そのもの（リニア軸。棒の長さ = 量）
g_L = ggplot(d, aes(x = net, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
    fill_down + col_edge + thm_bar +
    scale_x_continuous(labels = scales::comma,
                       expand = expansion(mult = c(0.05, 0))) +
    labs(x = 'net normalized reads (KO - WT)', y = NULL,
         title = '(B) per-strain - down in KO - KEGG pathway', subtitle = 'transcript amount')
# 右: 同じ棒を 100% に正規化した構成比（量の小さい term でも内訳が読める）
g_R = ggplot(d, aes(x = sh_net, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
    fill_down + col_edge + thm_bar +
    # ★ limits = c(0, 100) は使わない。position_stack の累積和が
    #   浮動小数で 100 をわずかに超えることがあり、その棒が黙って消える
    scale_x_continuous(breaks = c(0, 25, 50, 75, 100),
                       expand = expansion(mult = c(0, 0.02))) +
    labs(x = 'strain composition (%)', y = NULL, subtitle = 'composition')
g_L + (g_R + theme(axis.text.y = element_blank())) +
    plot_layout(widths = c(1, 0.55), guides = 'collect')

# ── 図7-3-7: (B) KO で低い × COG（3 term） ──
options(repr.plot.width = 17, repr.plot.height = 2.8)
d = hit_net %>% filter(scheme == 'B', direction == 'down', source == 'COG') %>%
    mutate(term = factor(term, levels = rev(ord_B_dn_COG)))
# 左: 正味リード増減そのもの（リニア軸。棒の長さ = 量）
g_L = ggplot(d, aes(x = net, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
    fill_down + col_edge + thm_bar +
    scale_x_continuous(labels = scales::comma,
                       expand = expansion(mult = c(0.05, 0))) +
    labs(x = 'net normalized reads (KO - WT)', y = NULL,
         title = '(B) per-strain - down in KO - COG category', subtitle = 'transcript amount')
# 右: 同じ棒を 100% に正規化した構成比（量の小さい term でも内訳が読める）
g_R = ggplot(d, aes(x = sh_net, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
    fill_down + col_edge + thm_bar +
    # ★ limits = c(0, 100) は使わない。position_stack の累積和が
    #   浮動小数で 100 をわずかに超えることがあり、その棒が黙って消える
    scale_x_continuous(breaks = c(0, 25, 50, 75, 100),
                       expand = expansion(mult = c(0, 0.02))) +
    labs(x = 'strain composition (%)', y = NULL, subtitle = 'composition')
g_L + (g_R + theme(axis.text.y = element_blank())) +
    plot_layout(widths = c(1, 0.55), guides = 'collect')

# ── 図7-3-8: (B) KO で低い × PFAM（30 term） ──
options(repr.plot.width = 14, repr.plot.height = 9.5)
d = hit_net %>% filter(scheme == 'B', direction == 'down', source == 'PFAM') %>%
    mutate(term = factor(term, levels = rev(ord_B_dn_PFAM)))
# 左: 正味リード増減そのもの（リニア軸。棒の長さ = 量）
g_L = ggplot(d, aes(x = net, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
    fill_down + col_edge + thm_bar +
    scale_x_continuous(labels = scales::comma,
                       expand = expansion(mult = c(0.05, 0))) +
    labs(x = 'net normalized reads (KO - WT)', y = NULL,
         title = '(B) per-strain - down in KO - PFAM domain', subtitle = 'transcript amount')
# 右: 同じ棒を 100% に正規化した構成比（量の小さい term でも内訳が読める）
g_R = ggplot(d, aes(x = sh_net, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
    fill_down + col_edge + thm_bar +
    # ★ limits = c(0, 100) は使わない。position_stack の累積和が
    #   浮動小数で 100 をわずかに超えることがあり、その棒が黙って消える
    scale_x_continuous(breaks = c(0, 25, 50, 75, 100),
                       expand = expansion(mult = c(0, 0.02))) +
    labs(x = 'strain composition (%)', y = NULL, subtitle = 'composition')
g_L + (g_R + theme(axis.text.y = element_blank())) +
    plot_layout(widths = c(1, 0.55), guides = 'collect')

# ── 図7-3-9: (B) KO で高い × KEGG（2 term） ──
options(repr.plot.width = 15, repr.plot.height = 2.5)
d = hit_net %>% filter(scheme == 'B', direction == 'up', source == 'KEGG') %>%
    mutate(term = factor(term, levels = rev(ord_B_up_KEGG)))
# 左: 正味リード増減そのもの（リニア軸。棒の長さ = 量）
g_L = ggplot(d, aes(x = net, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
    fill_up + col_edge + thm_bar +
    scale_x_continuous(labels = scales::comma,
                       expand = expansion(mult = c(0, 0.05))) +
    labs(x = 'net normalized reads (KO - WT)', y = NULL,
         title = '(B) per-strain - up in KO - KEGG pathway', subtitle = 'transcript amount')
# 右: 同じ棒を 100% に正規化した構成比（量の小さい term でも内訳が読める）
g_R = ggplot(d, aes(x = sh_net, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
    fill_up + col_edge + thm_bar +
    # ★ limits = c(0, 100) は使わない。position_stack の累積和が
    #   浮動小数で 100 をわずかに超えることがあり、その棒が黙って消える
    scale_x_continuous(breaks = c(0, 25, 50, 75, 100),
                       expand = expansion(mult = c(0, 0.02))) +
    labs(x = 'strain composition (%)', y = NULL, subtitle = 'composition')
g_L + (g_R + theme(axis.text.y = element_blank())) +
    plot_layout(widths = c(1, 0.55), guides = 'collect')

# ── 図7-3-10: (B) KO で高い × COG（3 term） ──
options(repr.plot.width = 17, repr.plot.height = 2.8)
d = hit_net %>% filter(scheme == 'B', direction == 'up', source == 'COG') %>%
    mutate(term = factor(term, levels = rev(ord_B_up_COG)))
# 左: 正味リード増減そのもの（リニア軸。棒の長さ = 量）
g_L = ggplot(d, aes(x = net, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
    fill_up + col_edge + thm_bar +
    scale_x_continuous(labels = scales::comma,
                       expand = expansion(mult = c(0, 0.05))) +
    labs(x = 'net normalized reads (KO - WT)', y = NULL,
         title = '(B) per-strain - up in KO - COG category', subtitle = 'transcript amount')
# 右: 同じ棒を 100% に正規化した構成比（量の小さい term でも内訳が読める）
g_R = ggplot(d, aes(x = sh_net, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
    fill_up + col_edge + thm_bar +
    # ★ limits = c(0, 100) は使わない。position_stack の累積和が
    #   浮動小数で 100 をわずかに超えることがあり、その棒が黙って消える
    scale_x_continuous(breaks = c(0, 25, 50, 75, 100),
                       expand = expansion(mult = c(0, 0.02))) +
    labs(x = 'strain composition (%)', y = NULL, subtitle = 'composition')
g_L + (g_R + theme(axis.text.y = element_blank())) +
    plot_layout(widths = c(1, 0.55), guides = 'collect')


■ Table 1: 株の大きさ（scheme A の正規化値、マーカー除外・検定されたCDSのみ）


strain,検定CDS数,WT転写量,KO転写量,占有率WT_pct,占有率KO_pct,一CDSあたりWT,down_DEG数,up_DEG数,down正味,up正味
<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>,<dbl>,<dbl>
Bth_VPI,1669,1716052,829178,6.8,4.0,1028,407,126,-905161,20015
Bth_St8,1792,4270740,2747058,17.0,13.2,2383,401,199,-1354362,44786
Bun_8492,3321,18172974,16541667,72.4,79.5,5472,222,1173,-3672052,1834693
Odo_St23,2837,947595,689895,3.8,3.3,334,473,117,-197567,21845
pME008,10,2935,3518,0.0,0.0,293,0,1,0,68



  集計: 有意 term 104 本 × 4株 = 416 行
  検算: 4株の遺伝子数合計 ≠ GeneRatio の分子 となる term = 0（0 なら整合）
  検算: 正味の合計がずれる term = 0（0 なら整合）

■ Table 2: term ごとに「遺伝子数で1位の株」と「転写量で1位の株」
  ★ 一致 列が『違う』の行が、セル7-2 の図と順位が逆になる term


scheme,direction,source,term,GeneRatio,padj,正味合計,数1位,数1位_n,数1位_pct,量1位,量1位_net,量1位_pct,一致
<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<int>,<dbl>,<chr>,<dbl>,<dbl>,<chr>
A,down,COG,COG C: Energy production and conversion,127/1400,5.2e-06,-1334308,Odo_St23,48,38,Bun_8492,-780054,58,★ 違う
A,down,COG,COG M: Cell wall/membrane/envelope biogenesis,201/1400,7.4e-06,-558326,Odo_St23,59,29,Bun_8492,-220462,39,★ 違う
A,down,COG,"COG O: Posttranslational modification, protein turnover, chaperones",75/1400,4.0e-05,-792580,Odo_St23,26,35,Bun_8492,-532397,67,★ 違う
A,down,KEGG,ko00020_Citrate cycle (TCA cycle),24/411,9.6e-04,-295309,Bth_St8,9,38,Bth_St8,-127337,43,
A,down,KEGG,ko00030_Pentose phosphate pathway,31/411,4.1e-06,-566342,Bun_8492,10,32,Bun_8492,-415837,73,
A,down,KEGG,ko00120_Primary bile acid biosynthesis,4/411,3.0e-02,-40522,Bun_8492,2,50,Bun_8492,-39334,97,
A,down,KEGG,ko00121_Secondary bile acid biosynthesis,6/411,2.4e-03,-46400,Bun_8492,2,33,Bun_8492,-39334,85,
A,down,KEGG,ko00190_Oxidative phosphorylation,27/411,4.8e-03,-155993,Odo_St23,12,44,Bun_8492,-128552,82,★ 違う
A,down,KEGG,ko00195_Photosynthesis,15/411,2.6e-07,-106223,Odo_St23,7,47,Bun_8492,-98077,92,★ 違う



■ Table 2-2: 株ごとの集計 ── 遺伝子数で1位になった回数 と 転写量で1位になった回数


strain,遺伝子数1位,転写量1位,転写量4位
<chr>,<int>,<int>,<int>
Bth_VPI,39,27,0
Bth_St8,45,27,0
Bun_8492,51,27,0
Odo_St23,50,27,0


... (9 outputs left out)

plot without title

plot without title

plot without title

plot without title

### 考察: 担い手の株は、遺伝子数と転写量でほぼ裏返しになる

#### 1. 有意 term 104 本のうち 50 本（48%）で1位の株が入れ替わる（Table 2-2）

| 株 | 遺伝子数で1位 | **転写量で1位** | **転写量で最下位** |
|---|---|---|---|
| Bth_VPI | 39 | **1** | 5 |
| Bth_St8 | 45 | 27 | 15 |
| **Bun_8492** | 51 | **75** | 11 |
| **Odo_St23** | **50** | **1** | **73** |

`Odo_St23` は down DEG が最多（473本）だが群集の転写の **3.8%** しか占めず、1CDSあたりの発現量は
`Bun_8492` の **1/16**（Table 1）。逆に `Bun_8492` は down DEG 最少（222本）で正味の減少は 18.6 倍。
→ **セル7-2 の図は担い手の順位をほぼ裏返しに見せている。**

#### 2. ★ 両方の尺度が正しい。置き換えず併用する

遺伝子数 = **制御の広がり**（セル7-2）／正味リード = **効果の大きさ**（セル7-3）。
最終成果物の担い手の順位は転写量で付ける（CLAUDE.md §7 ステージ6）が、両方の図を残す。

**up 側は入れ替わらない。** `COG F/H/I/J`・`ko03010` はいずれも `Bun_8492` が転写量シェア 93〜98% で、
もともと単独。量を動かしているのは `tuf`(+50,941)・`fusA2`(+22,880)・`fusA`(+17,898) の伸長因子3本。

#### 3. ★★ 1遺伝子が term を作っている（Table 3）

**正味増減の 50% 以上を1遺伝子が作る term が 37 / 104 本、80% 以上が 8 本。**

最も極端なのが `sodB`（`NQ510_RS12460` は 953,711 → 442,474、正味 −511,238）で、
**4株の `sodB` だけで 6 term を同時に有意にしている。**

| term | 遺伝子数 | sodB の占有率 |
|---|---|---|
| **`ko04013` MAPK - fly** / **`ko04068` FoxO** | 5 / 6 | **77%** |
| **`ko04146` Peroxisome** | 10 | **71%** |
| `PFAM:Sod_Fe_C` / `Sod_Fe_N` | 5 | 77% |

**細菌に MAPK もペルオキシソームもない。**CLAUDE.md §7「KEGG のラベルを鵜呑みにしない」の実例。
同様に `ko00195 Photosynthesis` は `atpD` が 32%、`ko00120/00121 胆汁酸` は BSH が 44〜53%、
`PFAM:6PGD` と `NAD_binding_2` は**同じ `gnd` 1遺伝子**で 79%。
→ **有意 term の本数を「所見の数」として数えてはいけない。**

転写量を動かした遺伝子は **酸化ストレス防御**（`sodB`/`tpx`/thioredoxin×2/`ahpC`）と
**PPP 酸化相**（`gnd`/`zwf`/`pgl`）の2系統に分かれ、どちらも `Bun_8492` が主。

#### 4. ★ scheme B で読むときの注意

(B) は株ごとに正規化するため、`Bun_8492` の WT→KO が **(A) ×0.91 / (B) ×0.68** になる。
**(B) は菌量増加を取り除くぶん `Bun_8492` の減少を大きく見せる。**
担い手の順位づけは **(A) で読む**のが筋。

#### 5. 限界

`norm_diff` の単純和なので**正負が混ざる term では相殺する**（`COG M` は `Bun_8492` の +23,997 が
他3株の減少を一部打ち消す）。相殺の中身は右パネルの構成比で読む。


In [151]:
# ── セル7-4: PFAM の結果をまとめる（term 間の関係と COG による束ね）────
# セル7 の dotplot とセル7-2・7-3 の棒は「有意 term の一覧」。
# ★ PFAM は term が小さく（1termあたり中央値3遺伝子、セル5 Table 2）数が多い
#   （2,477 term）ので、一覧だけでは「22〜30 本がどう関係しているか」が読めない。
#
# GO / PFAM で一般に使われるのは emapplot（term 間の遺伝子共有ネットワーク）。
# ★ ただし本データで機能するかは分からないので、まず共有の有無を実測してから使う。
#   emapplot は enrichplot にあるが ggraph が未導入（CLAUDE.md §10）なので、
#   ネットワークは igraph のレイアウトだけ借りて ggplot2 で自前に描く。
#   クラスタリングのパラメータを入れずに済むよう、グループは
#   ★ Jaccard > 0 の連結成分（＝遺伝子を1本でも共有していれば同じ組）とする。
#   閾値を置かないので「どこで切るか」の恣意性が入らない
library(ggplot2)
library(ggrepel)
suppressMessages(library(igraph))

options(repr.matrix.max.rows = 500, repr.matrix.max.cols = 50)

# ステージ5・セル7-2・セル7-3 と同じ株の色
col_strain = c(Bth_VPI = '#2a78d6', Bth_St8 = '#eb6834', Bun_8492 = '#3f9e5f',
               Odo_St23 = '#9b59b6')

# ===============================================================
# 1. 有意 PFAM term の構成遺伝子を引き直す
# ===============================================================
# ★ up 側は PFAM に有意 term が1本も無い（セル6 Table 3）。
#   「無い」ことも結果なので、0 であることを明示してから down だけを扱う
cat(sprintf("■ 有意 PFAM term の数: (A) down %d / (A) up %d / (B) down %d / (B) up %d\n",
    sum(ora$scheme=='A' & ora$direction=='down' & ora$source=='PFAM' & ora$padj<0.05),
    sum(ora$scheme=='A' & ora$direction=='up'   & ora$source=='PFAM' & ora$padj<0.05),
    sum(ora$scheme=='B' & ora$direction=='down' & ora$source=='PFAM' & ora$padj<0.05),
    sum(ora$scheme=='B' & ora$direction=='up'   & ora$source=='PFAM' & ora$padj<0.05)))
cat("  → up は 0 本なので、以下は down のみを扱う\n")

pf_long = ora %>% filter(direction == 'down', source == 'PFAM', padj < 0.05) %>%
    dplyr::select(scheme, term, GeneRatio, padj, geneID) %>%
    tidyr::separate_rows(geneID, sep = '/') %>%
    left_join(res %>% dplyr::select(gene_id, strain, product, Preferred_name, COG_category,
                                    normA_diff, normB_diff),
              by = c('geneID' = 'gene_id')) %>%
    mutate(diff = ifelse(scheme == 'A', normA_diff, normB_diff))

# ── 各 term の「主たる COG カテゴリ」を決める ───────────────────
# ★ COG_category は "CP" のように2文字並ぶことがある。セル5 の TERM2GENE と
#   同じく1文字ずつにばらしてから数える（"CP" は C と P の両方に数える）
pf_cog = pf_long %>% filter(!is.na(COG_category), COG_category != '') %>%
    mutate(ch = str_split(COG_category, '')) %>% tidyr::unnest(ch) %>%
    dplyr::count(scheme, term, ch) %>%
    group_by(scheme, term) %>%
    slice_max(n, n = 1, with_ties = FALSE) %>%   # 最頻の1文字
    ungroup() %>%
    transmute(scheme, term, COG = ch,
              # cog_name はセル5 で作った「1文字 → 正式名」の対応表
              COG名 = paste0(ch, ': ', cog_name[ch]))

cat(sprintf("\n  COG_category が付いている構成遺伝子: (A) %.1f%% / (B) %.1f%%\n",
    100*sum(pf_long$scheme=='A' & !is.na(pf_long$COG_category) & pf_long$COG_category!='')/sum(pf_long$scheme=='A'),
    100*sum(pf_long$scheme=='B' & !is.na(pf_long$COG_category) & pf_long$COG_category!='')/sum(pf_long$scheme=='B')))

# ===============================================================
# 2. term 間の遺伝子共有（Jaccard）を実測する
# ===============================================================
# ★ emapplot が意味を持つかどうかは、ここで決まる。
#   ペアごとに Jaccard = |共通| / |和集合| を全組み合わせで計算する
#   （term 数が 22〜30 本なので総当たりで十分）
jac_all = list(); comp_all = list()
for (sch in c('A', 'B')) {
    tb = ora %>% filter(scheme == sch, direction == 'down', source == 'PFAM', padj < 0.05)
    sets = strsplit(tb$geneID, '/')
    names(sets) = tb$term
    n = length(sets)
    J = matrix(0, n, n, dimnames = list(names(sets), names(sets)))
    for (i in 1:n) {
        for (j in 1:n) {
            J[i, j] = length(intersect(sets[[i]], sets[[j]])) /
                      length(union(sets[[i]], sets[[j]]))
        }
    }
    # 上三角の 0 より大きいペアだけをエッジにする（0 は「共有なし＝線なし」）
    ij = which(upper.tri(J) & J > 0, arr.ind = TRUE)
    el = data.frame(scheme = sch,
                    from = rownames(J)[ij[, 1]], to = colnames(J)[ij[, 2]],
                    共通 = NA_integer_, 和集合 = NA_integer_, jaccard = J[ij])
    for (k in seq_len(nrow(el))) {
        el$共通[k]   = length(intersect(sets[[el$from[k]]], sets[[el$to[k]]]))
        el$和集合[k] = length(union(sets[[el$from[k]]], sets[[el$to[k]]]))
    }
    jac_all[[sch]] = el

    # 連結成分（Jaccard>0 を辺とみなしたときの組）
    gr = graph_from_data_frame(el[, c('from', 'to')], directed = FALSE,
                               vertices = data.frame(name = names(sets)))
    cm = components(gr)
    comp_all[[sch]] = data.frame(scheme = sch, term = names(cm$membership),
                                 組 = as.integer(cm$membership))
    cat(sprintf("\n  scheme %s: %d term / Jaccard>0 のペア %d（全 %d ペア中 %.1f%%）/ 連結成分 %d 個\n",
                sch, n, nrow(el), n*(n-1)/2, 100*nrow(el)/(n*(n-1)/2), cm$no))
}
jac = bind_rows(jac_all)
comp = bind_rows(comp_all)

cat("\n★ 共有はほとんど無い。emapplot（ネットワーク）は本データでは情報を持たない。\n")
cat("  代わりに COG カテゴリで束ねる（図7-4-3 / 7-4-4）\n")

cat("\n■ Table 1: 遺伝子を共有している term の組（Jaccard > 0 のペアすべて）\n")
cat("  ★ jaccard = 1 は『同じ遺伝子集合』＝ 同じタンパク質の別ドメインを\n")
cat("    2つの term として数えている。独立な所見ではないので考察で1本に数える\n")
jac %>% mutate(jaccard = round(jaccard, 3)) %>%
    arrange(scheme, desc(jaccard)) %>% as.data.frame()

# ===============================================================
# 3. Table 2: 有意 PFAM term の一覧（COG・正味リード・株内訳つき）
# ===============================================================
cat("\n■ Table 2: 有意 PFAM term の一覧（down のみ）\n")
pf_long %>% group_by(scheme, term) %>%
    summarise(GeneRatio = first(GeneRatio), padj = signif(first(padj), 2),
              正味リード = round(sum(diff)),
              Bth_VPI = sum(strain == 'Bth_VPI'), Bth_St8 = sum(strain == 'Bth_St8'),
              Bun_8492 = sum(strain == 'Bun_8492'), Odo_St23 = sum(strain == 'Odo_St23'),
              代表遺伝子 = geneID[which.max(abs(diff))],
              代表の株 = as.character(strain[which.max(abs(diff))]),
              代表name = Preferred_name[which.max(abs(diff))],
              代表product = substr(product[which.max(abs(diff))], 1, 36),
              .groups = 'drop') %>%
    left_join(pf_cog, by = c('scheme', 'term')) %>%
    left_join(comp, by = c('scheme', 'term')) %>%
    dplyr::select(scheme, term, COG, GeneRatio, padj, 正味リード,
                  Bth_VPI, Bth_St8, Bun_8492, Odo_St23,
                  代表遺伝子, 代表の株, 代表name, 代表product, 組) %>%
    arrange(scheme, COG, term) %>% as.data.frame()

cat("\n■ Table 3: 主たる COG カテゴリごとの集計\n")
pf_long %>% left_join(pf_cog, by = c('scheme', 'term')) %>%
    group_by(scheme, COG, COG名) %>%
    summarise(term数 = n_distinct(term), 延べ遺伝子 = n(),
              正味リード = round(sum(diff)),
              term一覧 = paste(sort(unique(sub('^PFAM:', '', term))), collapse = ', '),
              .groups = 'drop') %>%
    arrange(scheme, 正味リード) %>% as.data.frame()

# ===============================================================
# 4. 図7-4-1 / 7-4-2: term 間の遺伝子共有ネットワーク
# ===============================================================
# ★ これは「ネットワークにならない」ことを示すための図。
#   emapplot を使わなかった理由を図として残す（Table 1 の数字の可視化）。
#   ノードの大きさ = term の遺伝子数、色 = padj、線の太さ = Jaccard。
#   レイアウトは igraph の Fruchterman-Reingold（再現のため seed を固定）

# ── 図7-4-1: (A) KO で低い × PFAM の共有ネットワーク ──
options(repr.plot.width = 9, repr.plot.height = 7)
tb = ora %>% filter(scheme == 'A', direction == 'down', source == 'PFAM', padj < 0.05)
el = jac %>% filter(scheme == 'A')
gr = graph_from_data_frame(el[, c('from', 'to')], directed = FALSE,
                           vertices = data.frame(name = tb$term))
set.seed(1)
xy = layout_with_fr(gr)
nd = data.frame(term = V(gr)$name, x = xy[, 1], y = xy[, 2]) %>%
    left_join(tb %>% transmute(term, Count = as.integer(sub('/.*', '', GeneRatio)), padj),
              by = 'term') %>%
    left_join(pf_cog %>% filter(scheme == 'A') %>% dplyr::select(term, COG), by = 'term')
eg = el %>% left_join(nd %>% dplyr::select(term, x, y), by = c('from' = 'term')) %>%
    rename(x1 = x, y1 = y) %>%
    left_join(nd %>% dplyr::select(term, x, y), by = c('to' = 'term')) %>%
    rename(x2 = x, y2 = y)
ggplot() +
    geom_segment(data = eg, aes(x = x1, y = y1, xend = x2, yend = y2, linewidth = jaccard),
                 colour = 'grey55') +
    scale_linewidth_continuous(range = c(0.4, 2.2), limits = c(0, 1), name = 'Jaccard') +
    geom_point(data = nd, aes(x = x, y = y, size = Count, fill = padj),
               shape = 21, colour = 'grey25', stroke = 0.6) +
    scale_size_continuous(range = c(3, 11), name = 'gene count') +
    scale_fill_gradient(low = '#8fb8f5', high = '#eaf1fd', limits = c(0, 0.05), name = 'p.adjust') +
    geom_text_repel(data = nd, aes(x = x, y = y, label = sub('^PFAM:', '', term)),
                    size = 3.2, max.overlaps = Inf, min.segment.length = 0,
                    segment.colour = 'grey70', box.padding = 0.5) +
    theme_void() +
    labs(title = '(A) pooled - down in KO - PFAM domain',
         subtitle = 'edges = shared genes (Jaccard > 0). Most terms share no gene at all.')

# ── 図7-4-2: (B) KO で低い × PFAM の共有ネットワーク ──
options(repr.plot.width = 9, repr.plot.height = 8)
tb = ora %>% filter(scheme == 'B', direction == 'down', source == 'PFAM', padj < 0.05)
el = jac %>% filter(scheme == 'B')
gr = graph_from_data_frame(el[, c('from', 'to')], directed = FALSE,
                           vertices = data.frame(name = tb$term))
set.seed(1)
xy = layout_with_fr(gr)
nd = data.frame(term = V(gr)$name, x = xy[, 1], y = xy[, 2]) %>%
    left_join(tb %>% transmute(term, Count = as.integer(sub('/.*', '', GeneRatio)), padj),
              by = 'term') %>%
    left_join(pf_cog %>% filter(scheme == 'B') %>% dplyr::select(term, COG), by = 'term')
eg = el %>% left_join(nd %>% dplyr::select(term, x, y), by = c('from' = 'term')) %>%
    rename(x1 = x, y1 = y) %>%
    left_join(nd %>% dplyr::select(term, x, y), by = c('to' = 'term')) %>%
    rename(x2 = x, y2 = y)
ggplot() +
    geom_segment(data = eg, aes(x = x1, y = y1, xend = x2, yend = y2, linewidth = jaccard),
                 colour = 'grey55') +
    scale_linewidth_continuous(range = c(0.4, 2.2), limits = c(0, 1), name = 'Jaccard') +
    geom_point(data = nd, aes(x = x, y = y, size = Count, fill = padj),
               shape = 21, colour = 'grey25', stroke = 0.6) +
    scale_size_continuous(range = c(3, 11), name = 'gene count') +
    scale_fill_gradient(low = '#8fb8f5', high = '#eaf1fd', limits = c(0, 0.05), name = 'p.adjust') +
    geom_text_repel(data = nd, aes(x = x, y = y, label = sub('^PFAM:', '', term)),
                    size = 3.2, max.overlaps = Inf, min.segment.length = 0,
                    segment.colour = 'grey70', box.padding = 0.5) +
    theme_void() +
    labs(title = '(B) per-strain - down in KO - PFAM domain',
         subtitle = 'edges = shared genes (Jaccard > 0). Most terms share no gene at all.')

# ===============================================================
# 5. 図7-4-3 / 7-4-4: COG カテゴリで束ねた PFAM の dotplot
# ===============================================================
# ★ ネットワークが使えないので、term を「構成遺伝子の主たる COG カテゴリ」で
#   グループ化して並べる。COG は 91.5% の遺伝子に付いていて（セル5 Table 2）、
#   有意 PFAM term の構成遺伝子に限れば 97〜100%（上の出力）なので束ねられる。
#   x 軸・点の大きさ・色はセル7 の dotplot と同じ意味にしてある
thm_dot = theme_bw() +
    theme(panel.grid.minor = element_blank(),
          # ★ グレーの帯の上の文字は細いと読めないので太字にし、大きめにする
          strip.text.y.right = element_text(angle = 0, hjust = 0, size = 11, face = 'bold'),
          strip.background = element_rect(fill = 'grey92', colour = 'grey60'),
          legend.key.size = unit(0.9, 'lines'))

# ── 図7-4-3: (A) KO で低い × PFAM（COG カテゴリでグループ化） ──
options(repr.plot.width = 12, repr.plot.height = 10)
dd = ora %>% filter(scheme == 'A', direction == 'down', source == 'PFAM', padj < 0.05) %>%
    mutate(Count = as.integer(sub('/.*', '', GeneRatio)),
           ratio = Count / as.integer(sub('.*/', '', GeneRatio))) %>%
    left_join(pf_cog %>% filter(scheme == 'A') %>% dplyr::select(term, COG, COG名), by = 'term') %>%
    # ★ COG が1つも付かない term があれば 'unassigned' として残す（落とさない）
    mutate(COG名 = ifelse(is.na(COG名), 'unassigned', COG名)) %>%
    arrange(COG名, ratio) %>%
    mutate(term = factor(sub('^PFAM:', '', term), levels = sub('^PFAM:', '', term)))
# ★ padj の色はセル7 の dotplot と同じ「0 = 青 / 0.05 = 白」に統一する。
#   白い点は背景に溶けるので、colour ではなく shape = 21 の fill に割り当て、
#   灰色の縁を付けて輪郭が残るようにする（セル7 の dotplot も fill 指定）
ggplot(dd, aes(x = ratio, y = term, size = Count, fill = padj)) +
    geom_point(shape = 21, colour = 'grey30', stroke = 0.5) +
    # ★ space = 'free_y' は使わない。1 term しかないカテゴリの帯が低くなりすぎ、
    #   'D: Cell cycle control, cell division, chromosome partitioning' のような
    #   長い名前が黙って切り取られる。帯の高さを揃えて全文を出す
    facet_grid(COG名 ~ ., scales = 'free_y',
               labeller = labeller(COG名 = label_wrap_gen(26))) +
    scale_fill_gradient(low = 'blue', high = 'white', limits = c(0, 0.05), name = 'p.adjust') +
    scale_size_continuous(range = c(2, 7), name = 'gene count',
                          guide = guide_legend(override.aes = list(fill = 'grey70'))) +
    thm_dot +
    labs(x = 'GeneRatio', y = NULL,
         title = '(A) pooled - down in KO - PFAM domain, grouped by COG category')

# ── 図7-4-4: (B) KO で低い × PFAM（COG カテゴリでグループ化） ──
options(repr.plot.width = 12, repr.plot.height = 11)
dd = ora %>% filter(scheme == 'B', direction == 'down', source == 'PFAM', padj < 0.05) %>%
    mutate(Count = as.integer(sub('/.*', '', GeneRatio)),
           ratio = Count / as.integer(sub('.*/', '', GeneRatio))) %>%
    left_join(pf_cog %>% filter(scheme == 'B') %>% dplyr::select(term, COG, COG名), by = 'term') %>%
    mutate(COG名 = ifelse(is.na(COG名), 'unassigned', COG名)) %>%
    arrange(COG名, ratio) %>%
    mutate(term = factor(sub('^PFAM:', '', term), levels = sub('^PFAM:', '', term)))
# ★ padj の色はセル7 の dotplot と同じ「0 = 青 / 0.05 = 白」に統一する。
#   白い点は背景に溶けるので、colour ではなく shape = 21 の fill に割り当て、
#   灰色の縁を付けて輪郭が残るようにする（セル7 の dotplot も fill 指定）
ggplot(dd, aes(x = ratio, y = term, size = Count, fill = padj)) +
    geom_point(shape = 21, colour = 'grey30', stroke = 0.5) +
    # ★ space = 'free_y' は使わない。1 term しかないカテゴリの帯が低くなりすぎ、
    #   'D: Cell cycle control, cell division, chromosome partitioning' のような
    #   長い名前が黙って切り取られる。帯の高さを揃えて全文を出す
    facet_grid(COG名 ~ ., scales = 'free_y',
               labeller = labeller(COG名 = label_wrap_gen(26))) +
    scale_fill_gradient(low = 'blue', high = 'white', limits = c(0, 0.05), name = 'p.adjust') +
    scale_size_continuous(range = c(2, 7), name = 'gene count',
                          guide = guide_legend(override.aes = list(fill = 'grey70'))) +
    thm_dot +
    labs(x = 'GeneRatio', y = NULL,
         title = '(B) per-strain - down in KO - PFAM domain, grouped by COG category')

# ===============================================================
# 6. 図7-4-5 / 7-4-6: 同じ並びで「株ごとの遺伝子数の割合」を見る
# ===============================================================
# 図7-4-3 / 7-4-4 は term の大きさと有意性しか出していない。
# ★ 同じ y 軸の並び・同じ COG グループ・同じ x 軸（GeneRatio）のまま、
#   棒を株で区切る。棒の長さは図7-4-3 の点の位置と一致し、
#   区分の幅がその株の寄与になる。どの菌の応答かが読める。
#   ここは ★ 遺伝子数のみ（転写量はセル7-3 で別に出してある）
#   区分の塗り = padj（図7-4-3 と同じ）／区分の縁 = 株（セル7-2・7-3 と同じ色）

# 株ごとの当たり遺伝子数。★ 0 本の株も行として残す（0 を明示する）
pf_share = pf_long %>%
    dplyr::count(scheme, term, strain, name = 'n_gene') %>%
    tidyr::complete(tidyr::nesting(scheme, term), strain = names(col_strain),
                    fill = list(n_gene = 0)) %>%
    ungroup() %>%
    mutate(strain = factor(strain, levels = names(col_strain))) %>%
    left_join(ora %>% filter(direction == 'down', source == 'PFAM', padj < 0.05) %>%
                  dplyr::select(scheme, term, GeneRatio, padj),
              by = c('scheme', 'term')) %>%
    # ★ x 軸は図7-4-3 / 7-4-4 と同じ GeneRatio のまま。
    #   棒を株で区切るので、各株の遺伝子数を GeneRatio の分母で割る。
    #   足すと棒全体＝その term の GeneRatio になる（セル7-2 と同じ作り）
    mutate(denom = as.integer(sub('.*/', '', GeneRatio)),
           width = n_gene / denom)

col_edge = scale_colour_manual(values = col_strain, name = 'strain',
    guide = guide_legend(override.aes = list(fill = 'white', linewidth = 1)))

# ★ 棒の塗りだけは図7-4-3 / 7-4-4 の濃い青（0 = blue）を使えない。
#   セル7-2 で実測したとおり、濃い青で塗ると Bth_VPI の青い縁が塗りと同化して
#   完全に見えなくなる（Bth_St8 の橙も赤い塗りに埋もれる）。
#   この図の目的は株の内訳を読むことなので、セル7-2・セル7-3 の棒と同じ
#   薄い青（0 = #8fb8f5 / 0.05 = #eaf1fd）にして縁を優先する。
#   意味は同じ「0 = 青 / 0.05 = 白」で、濃さだけを落としてある
fill_down_bar = scale_fill_gradient(low = '#8fb8f5', high = '#eaf1fd',
                                    limits = c(0, 0.05), name = 'p.adjust')

# ── 図7-4-5: (A) KO で低い × PFAM の株内訳（COG カテゴリでグループ化） ──
options(repr.plot.width = 12, repr.plot.height = 10)
dd = pf_share %>% filter(scheme == 'A') %>%
    left_join(pf_cog %>% filter(scheme == 'A') %>% dplyr::select(term, COG名), by = 'term') %>%
    mutate(COG名 = ifelse(is.na(COG名), 'unassigned', COG名),
           ratio = as.integer(sub('/.*', '', GeneRatio)) / as.integer(sub('.*/', '', GeneRatio)))
# y 軸の並びは図7-4-3 と完全に同じにする（COG名 → GeneRatio の昇順）
ord = dd %>% distinct(term, COG名, ratio) %>% arrange(COG名, ratio) %>% pull(term)
dd = dd %>% mutate(term = factor(sub('^PFAM:', '', term), levels = sub('^PFAM:', '', ord)))
ggplot(dd, aes(x = width, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.0, width = 0.45, position = position_stack(reverse = TRUE)) +
    facet_grid(COG名 ~ ., scales = 'free_y',
               labeller = labeller(COG名 = label_wrap_gen(26))) +
    fill_down_bar + col_edge +
    # x 軸は 0 の左に余白を作らない（棒の左端を軸に付ける）。セル7-2 と同じ
    scale_x_continuous(expand = expansion(mult = c(0, 0.05))) +
    thm_dot +
    labs(x = 'GeneRatio (stacked by strain)', y = NULL,
         title = '(A) pooled - down in KO - PFAM domain, stacked by strain within COG category')

# ── 図7-4-6: (B) KO で低い × PFAM の株内訳（COG カテゴリでグループ化） ──
options(repr.plot.width = 12, repr.plot.height = 11)
dd = pf_share %>% filter(scheme == 'B') %>%
    left_join(pf_cog %>% filter(scheme == 'B') %>% dplyr::select(term, COG名), by = 'term') %>%
    mutate(COG名 = ifelse(is.na(COG名), 'unassigned', COG名),
           ratio = as.integer(sub('/.*', '', GeneRatio)) / as.integer(sub('.*/', '', GeneRatio)))
ord = dd %>% distinct(term, COG名, ratio) %>% arrange(COG名, ratio) %>% pull(term)
dd = dd %>% mutate(term = factor(sub('^PFAM:', '', term), levels = sub('^PFAM:', '', ord)))
ggplot(dd, aes(x = width, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.0, width = 0.45, position = position_stack(reverse = TRUE)) +
    facet_grid(COG名 ~ ., scales = 'free_y',
               labeller = labeller(COG名 = label_wrap_gen(26))) +
    fill_down_bar + col_edge +
    scale_x_continuous(expand = expansion(mult = c(0, 0.05))) +
    thm_dot +
    labs(x = 'GeneRatio (stacked by strain)', y = NULL,
         title = '(B) per-strain - down in KO - PFAM domain, stacked by strain within COG category')


■ 有意 PFAM term の数: (A) down 22 / (A) up 0 / (B) down 30 / (B) up 0
  → up は 0 本なので、以下は down のみを扱う

  COG_category が付いている構成遺伝子: (A) 97.4% / (B) 100.0%

  scheme A: 22 term / Jaccard>0 のペア 5（全 231 ペア中 2.2%）/ 連結成分 17 個

  scheme B: 30 term / Jaccard>0 のペア 9（全 435 ペア中 2.1%）/ 連結成分 22 個

★ 共有はほとんど無い。emapplot（ネットワーク）は本データでは情報を持たない。
  代わりに COG カテゴリで束ねる（図7-4-3 / 7-4-4）

■ Table 1: 遺伝子を共有している term の組（Jaccard > 0 のペアすべて）
  ★ jaccard = 1 は『同じ遺伝子集合』＝ 同じタンパク質の別ドメインを
    2つの term として数えている。独立な所見ではないので考察で1本に数える


scheme,from,to,共通,和集合,jaccard
<chr>,<chr>,<chr>,<int>,<int>,<dbl>
A,PFAM:Sod_Fe_C,PFAM:Sod_Fe_N,5,5,1.000
A,PFAM:6PGD,PFAM:NAD_binding_2,4,4,1.000
A,PFAM:PMSR,PFAM:SelR,4,4,1.000
A,PFAM:zf-CDGSH,PFAM:Fer4_19,4,6,0.667
A,PFAM:OmpA,PFAM:OMP_b-brl,5,28,0.179
B,PFAM:Biotin_carb_C,PFAM:Biotin_carb_N,5,5,1.000
B,PFAM:Sod_Fe_C,PFAM:Sod_Fe_N,5,5,1.000
B,PFAM:6PGD,PFAM:NAD_binding_2,4,4,1.000
B,PFAM:PMSR,PFAM:SelR,4,4,1.000



■ Table 2: 有意 PFAM term の一覧（down のみ）


scheme,term,COG,GeneRatio,padj,正味リード,Bth_VPI,Bth_St8,Bun_8492,Odo_St23,代表遺伝子,代表の株,代表name,代表product,組
<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<int>
A,PFAM:Ligase_CoA,C,4/1346,2.9e-02,-26502,2,2,0,0,CE91St8_32130,Bth_St8,sucC,succinate--CoA ligase [ADP-forming],14
A,PFAM:Pyr_redox_2,C,14/1346,2.7e-03,-124230,3,3,3,5,NQ510_RS14720,Bun_8492,ndh,NAD(P)/FAD-dependent oxidoreductase,2
A,PFAM:Sod_Fe_C,C,5/1346,9.2e-03,-665655,1,1,1,2,NQ510_RS12460,Bun_8492,sodB,superoxide dismutase,10
A,PFAM:Sod_Fe_N,C,5/1346,9.2e-03,-665655,1,1,1,2,NQ510_RS12460,Bun_8492,sodB,superoxide dismutase,10
A,PFAM:6PGD,H,4/1346,2.9e-02,-186762,1,1,1,1,NQ510_RS11780,Bun_8492,gnd,decarboxylating NADP(+)-dependent ph,13
A,PFAM:NAD_binding_2,H,4/1346,2.9e-02,-186762,1,1,1,1,NQ510_RS11780,Bun_8492,gnd,decarboxylating NADP(+)-dependent ph,13
A,PFAM:DUF3987,L,5/1346,2.9e-02,-161,3,2,0,0,CE91St8_19170,Bth_St8,NA,hypothetical protein,11
A,PFAM:GNVR,M,10/1346,4.3e-02,-582,4,4,0,2,CE91St8_26050,Bth_St8,NA,chain-length determining protein,4
A,PFAM:OMP_b-brl,M,13/1346,3.1e-02,-59369,2,7,4,0,BT_RS13950,Bth_VPI,NA,porin family protein,1



■ Table 3: 主たる COG カテゴリごとの集計


scheme,COG,COG名,term数,延べ遺伝子,正味リード,term一覧
<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<chr>
A,C,C: Energy production and conversion,4,28,-1482042,"Ligase_CoA, Pyr_redox_2, Sod_Fe_C, Sod_Fe_N"
A,H,H: Coenzyme transport and metabolism,2,8,-373524,"6PGD, NAD_binding_2"
A,M,M: Cell wall/membrane/envelope biogenesis,3,43,-302143,"GNVR, OMP_b-brl, OmpA"
A,P,P: Inorganic ion transport and metabolism,1,9,-131696,HMA
A,S,S: Function unknown,6,32,-103790,"Acetyltransf_CG, DUF308, Fer4_19, MerR_2, P_gingi_FimA, zf-CDGSH"
A,O,"O: Posttranslational modification, protein turnover, chaperones",3,14,-58961,"PMSR, SelR, UPF0051"
A,T,T: Signal transduction mechanisms,1,4,-55962,PIN_4
A,V,V: Defense mechanisms,1,10,-6076,ABC2_membrane_3
A,L,"L: Replication, recombination and repair",1,5,-161,DUF3987


... (2 outputs left out)

plot without title

plot without title

plot without title

plot without title

### 考察: PFAM の term どうしは遺伝子を共有していない — emapplot は使えない

#### 1. ★★ GO / PFAM の定番である emapplot が本データでは情報を持たない（Table 1）

| scheme | 有意 term | Jaccard > 0 のペア | 連結成分 |
|---|---|---|---|
| (A) down | 22 | **5 / 231（2.2%）** | **17 個**（最大サイズ 2） |
| (B) down | 30 | **9 / 435（2.1%）** | **22 個**（最大サイズ 3） |

**PFAM term どうしはほぼ完全に排他で、ネットワークにならない**（図7-4-1 / 7-4-2）。
→ **emapplot を使わなかった理由を図として残してある。**

#### 2. ★ つながっているペアはほとんど「同じタンパク質の別ドメイン」

`Sod_Fe_N` ＝ `Sod_Fe_C`（SOD の N/C 末）、`6PGD` ＝ `NAD_binding_2`（`gnd` の2ドメイン）、
`PMSR` ＝ `SelR`（`msrA`/`msrB`）、`Biotin_carb_N` ＝ `Biotin_carb_C`（(B)）はいずれも
**Jaccard = 1 = 遺伝子集合が完全一致。**
**生物学的に意味のある連結は `OmpA` — `OMP_b-brl`（外膜タンパク）だけ。**

→ **「22〜30 本の有意 term」は独立な所見が 22〜30 個あるという意味ではない**
（セル6 考察 6 と同じ結論が、ペア単位で定量された）。

★ なお前回「表面クラスターと酸化ストレス/Fe-S クラスター」と書いたのは**遺伝子共有に基づく
クラスターではなく生物学的なグルーピング**だった。**本データに遺伝子共有のクラスターは存在しない。**

#### 3. 代わりに COG カテゴリで束ねると読める（図7-4-3 / 7-4-4）

有意 PFAM term の構成遺伝子には COG が **(A) 97.4% / (B) 100%** 付いている。

| COG | 束になる PFAM term |
|---|---|
| **M: Cell wall/membrane/envelope** | `OmpA`・`OMP_b-brl`・`GNVR`（(B) は `Patatin`・`MS_channel` も） |
| **O: Posttranslational mod., chaperones** | `Clp_N`・`UPF0051`・`DnaJ_C`・`SelR`・`PMSR` |
| **C: Energy production** | `Pyr_redox_2`・`Sod_Fe_N/C`・`Ligase_CoA` |
| **P: Inorganic ion transport** | `HMA`（= `copA`/`actP` 重金属排出） |

#### 4. ★ `S: Function unknown` が 5 term・延べ 62 遺伝子で2番目に多い

**PFAM のシグナルの相当部分が機能未知の遺伝子に乗っている**（`DUF308`・`Acetyltransf_CG`・
`zf-CDGSH`・`MerR_2`・`Fer4_19`）。COG や KEGG では `S` に埋没して見えない部分で、
**PFAM を使う価値はここにある**が、**機能の解釈はできない。**

#### 5. 株の内訳（図7-4-5 / 7-4-6）

`OmpA`（20遺伝子）は4株すべてに分布し、`GNVR` は `Bun_8492` の区分が無い。
**`P_gingi_FimA` は `Odo_St23` 単独（100%）**で、セル6 考察 10 の
「Odo 特異的な表層構造の低下」が棒でも確認できる。


## セル7-5: COG の結果をまとめる（上位分類で束ねる）

セル7-4 は PFAM を**「構成遺伝子の主たる COG カテゴリ」**で束ねた。
**COG 自体は COG で束ねられない**ので **NCBI COG の4大分類**で束ねる。

### セル7・7-2・7-3 との役割分担

| セル | x 軸 | 読むもの |
|---|---|---|
| セル7 | `GeneRatio` | ソース横断の一覧 |
| セル7-2 | `GeneRatio` を株で積み上げ | どの菌の応答か |
| セル7-3 | 正味リード増減 | 効果の大きさ |
| **セル7-5** | **`fold enrichment`** | **上位分類でまとめて「何の系統が動いたか」** |

`fold` を x 軸に使うのはここだけ。`GeneRatio` は term の大きさに引きずられる（`M` は 201 遺伝子
あるので必ず長い）が、`fold` は背景に対する濃縮の強さを表す。

### 描く対象 — 有意（padj<0.05）な 13 行のみ

| 上位分類 | down | up |
|---|---|---|
| Cellular processes and signaling | `M` 細胞壁・膜 / `O` 翻訳後修飾・シャペロン | — |
| Metabolism | `C` エネルギー産生 | `F` ヌクレオチド / `H` 補酵素（(A) のみ） / `I` 脂質 |
| Information storage and processing | — | `J` 翻訳・リボソーム |

**down は「細胞の外側と品質管理」、up は「代謝と翻訳」**と上位分類がきれいに分かれる。

### ★ Table 2 を必ず見る（セル7-4 と同じ理由）

PFAM は1遺伝子に複数ドメインが付くので term が重なった。**COG も `COG_category` が `"CP"` のように
複数文字を持つ遺伝子があるので重なりうる。重なっているカテゴリを独立な所見として数えてはいけない。**

### 図（8枚）

図7-5-1〜4 = dotplot（(A)/(B) × down/up）、図7-5-5〜8 = 株ごとの内訳（同）。
色はセル7・7-2〜7-4 と同じで、**dotplot は濃い色**（`shape 21` の `fill` に当てて灰色の縁を付ける）、
**株内訳の棒は薄い色**（濃いと `Bth_VPI` の青・`Bth_St8` の橙の縁が塗りに埋もれる。セル7-2 で実測）。


In [157]:
# ── セル7-5: COG の結果をまとめる（上位分類で束ねる）──────────────
# セル7-4 は PFAM の term を「構成遺伝子の主たる COG カテゴリ」で束ねた。
# ★ COG 自体は COG で束ねられないので、NCBI COG の4大分類で束ねる。
#
# セル7（dotplot）・セル7-2（株内訳の棒）との役割分担
#   セル7    : x 軸 = GeneRatio。ソース横断で同じ形に並べる一覧
#   セル7-2  : x 軸 = GeneRatio を株で積み上げ。どの菌の応答か
#   ★ セル7-5: x 軸 = fold enrichment。上位分類でグループ化して
#              「down は細胞の外側と品質管理 / up は代謝と翻訳」が読める形にする
library(ggplot2)

options(repr.matrix.max.rows = 500, repr.matrix.max.cols = 50)

# ステージ5・セル7-2〜7-4 と同じ株の色
col_strain = c(Bth_VPI = '#2a78d6', Bth_St8 = '#eb6834', Bun_8492 = '#3f9e5f',
               Odo_St23 = '#9b59b6')

# ── COG の上位分類（NCBI COG の4大分類）─────────────────────────
# ★ 和訳はしない。図に日本語を入れると PNG 保存時に文字化けする（セル5 と同じ理由）
cog_super = c(
  J = 'Information storage and processing', A = 'Information storage and processing',
  K = 'Information storage and processing', L = 'Information storage and processing',
  B = 'Information storage and processing',
  D = 'Cellular processes and signaling',   V = 'Cellular processes and signaling',
  T = 'Cellular processes and signaling',   M = 'Cellular processes and signaling',
  N = 'Cellular processes and signaling',   U = 'Cellular processes and signaling',
  O = 'Cellular processes and signaling',
  C = 'Metabolism', G = 'Metabolism', E = 'Metabolism', F = 'Metabolism',
  H = 'Metabolism', I = 'Metabolism', P = 'Metabolism', Q = 'Metabolism',
  S = 'Poorly characterized')

# ===============================================================
# 1. 有意 COG カテゴリと、その構成遺伝子
# ===============================================================
sig = ora %>% filter(source == 'COG', padj < 0.05) %>%
    mutate(COG = sub('^COG ([A-Z]):.*', '\\1', term),
           上位分類 = cog_super[COG],
           Count = as.integer(sub('/.*', '', GeneRatio)))

sl = sig %>% dplyr::select(scheme, direction, term, COG, 上位分類, geneID) %>%
    tidyr::separate_rows(geneID, sep = '/') %>%
    left_join(res %>% dplyr::select(gene_id, strain, Preferred_name, product,
                                    normA_diff, normB_diff),
              by = c('geneID' = 'gene_id')) %>%
    mutate(diff = ifelse(scheme == 'A', normA_diff, normB_diff))

cat(sprintf("■ 有意 COG カテゴリ: %d 行（(A) down %d / (A) up %d / (B) down %d / (B) up %d）\n",
    nrow(sig),
    sum(sig$scheme=='A' & sig$direction=='down'), sum(sig$scheme=='A' & sig$direction=='up'),
    sum(sig$scheme=='B' & sig$direction=='down'), sum(sig$scheme=='B' & sig$direction=='up')))

cat("\n■ Table 1: 有意 COG カテゴリの一覧（上位分類・正味リード・代表遺伝子つき）\n")
sl %>% group_by(scheme, direction, 上位分類, COG, term) %>%
    summarise(正味リード = round(sum(diff)),
              Bth_VPI = sum(strain == 'Bth_VPI'), Bth_St8 = sum(strain == 'Bth_St8'),
              Bun_8492 = sum(strain == 'Bun_8492'), Odo_St23 = sum(strain == 'Odo_St23'),
              代表遺伝子 = geneID[which.max(abs(diff))],
              代表の株 = as.character(strain[which.max(abs(diff))]),
              代表name = Preferred_name[which.max(abs(diff))],
              代表product = substr(product[which.max(abs(diff))], 1, 32),
              .groups = 'drop') %>%
    left_join(sig %>% dplyr::select(scheme, direction, term, GeneRatio, BgRatio, fold, padj),
              by = c('scheme', 'direction', 'term')) %>%
    transmute(scheme = paste0('(', scheme, ')'), direction, 上位分類, COG,
              term = sub('^COG [A-Z]: ', '', term), GeneRatio, BgRatio,
              fold = round(fold, 2), padj = signif(padj, 2), 正味リード,
              Bth_VPI, Bth_St8, Bun_8492, Odo_St23,
              代表遺伝子, 代表の株, 代表name, 代表product) %>%
    arrange(scheme, direction, 上位分類, COG) %>% as.data.frame()

# ===============================================================
# 2. カテゴリどうしの遺伝子共有（セル7-4 Table 1 と同じ見方）
# ===============================================================
# ★ PFAM は1遺伝子に複数ドメインが付くので term が重なった。
#   COG も COG_category が "CP" のように複数文字を持つ遺伝子があるので重なりうる。
#   重なっていれば、2つのカテゴリを独立な所見として数えてはいけない
cat("\n■ Table 2: 有意 COG カテゴリどうしの遺伝子共有（Jaccard > 0 のペアすべて）\n")
jl = list()
for (sch in c('A', 'B')) {
    for (dr in c('down', 'up')) {
        tb = sig %>% filter(scheme == sch, direction == dr)
        if (nrow(tb) < 2) next
        sets = strsplit(tb$geneID, '/')
        names(sets) = tb$term
        for (i in 1:(length(sets) - 1)) {
            for (j in (i + 1):length(sets)) {
                ic = length(intersect(sets[[i]], sets[[j]]))
                un = length(union(sets[[i]], sets[[j]]))
                if (ic > 0) {
                    jl[[length(jl) + 1]] = data.frame(
                        scheme = sch, direction = dr,
                        term1 = sub('^COG ', '', names(sets)[i]),
                        term2 = sub('^COG ', '', names(sets)[j]),
                        共通 = ic, 和集合 = un, jaccard = round(ic / un, 3))
                }
            }
        }
    }
}
cat(sprintf("  共有ありのペア: %d\n", length(jl)))
bind_rows(jl) %>% arrange(scheme, direction, desc(jaccard)) %>% as.data.frame()

# ===============================================================
# 3. 図7-5-1 〜 7-5-8
# ===============================================================
# 共通の見た目。セル7-4 と同じテーマ（グレー帯の文字は太字・11pt）
thm_dot = theme_bw() +
    theme(panel.grid.minor = element_blank(),
          strip.text.y.right = element_text(angle = 0, hjust = 0, size = 11, face = 'bold'),
          strip.background = element_rect(fill = 'grey92', colour = 'grey60'),
          legend.key.size = unit(0.9, 'lines'))

# dotplot の色はセル7・セル7-4 と同じ「0 = 青(down)/赤(up)、0.05 = 白」。
# ★ 白い点は背景に溶けるので shape = 21 の fill に割り当て、灰色の縁を付ける
fill_dot_dn = scale_fill_gradient(low = 'blue', high = 'white', limits = c(0, 0.05), name = 'p.adjust')
fill_dot_up = scale_fill_gradient(low = 'red',  high = 'white', limits = c(0, 0.05), name = 'p.adjust')
size_dot = scale_size_continuous(range = c(3, 9), name = 'gene count',
                                 guide = guide_legend(override.aes = list(fill = 'grey70')))

# 棒の塗りはセル7-2・7-3・7-4 と同じ薄い色。
# ★ 濃い青/赤で塗ると株の縁（Bth_VPI の青・Bth_St8 の橙）が塗りに埋もれて見えなくなる
fill_bar_dn = scale_fill_gradient(low = '#8fb8f5', high = '#eaf1fd', limits = c(0, 0.05), name = 'p.adjust')
fill_bar_up = scale_fill_gradient(low = '#f08080', high = '#fdeaea', limits = c(0, 0.05), name = 'p.adjust')
col_edge = scale_colour_manual(values = col_strain, name = 'strain',
    guide = guide_legend(override.aes = list(fill = 'white', linewidth = 1)))

# 株ごとの内訳。★ 当たり遺伝子が0本の株も行として残す（0 を明示する）
share = sl %>% dplyr::count(scheme, direction, term, strain, name = 'n_gene') %>%
    tidyr::complete(tidyr::nesting(scheme, direction, term), strain = names(col_strain),
                    fill = list(n_gene = 0)) %>%
    ungroup() %>%
    mutate(strain = factor(strain, levels = names(col_strain))) %>%
    left_join(sig %>% dplyr::select(scheme, direction, term, GeneRatio, padj, fold, 上位分類),
              by = c('scheme', 'direction', 'term')) %>%
    # 棒の長さは GeneRatio。足すとその term の GeneRatio になる（セル7-2 と同じ作り）
    mutate(width = n_gene / as.integer(sub('.*/', '', GeneRatio)))

# ── 図7-5-1: (A) KO で低い × COG（上位分類でグループ化、x = fold enrichment） ──
options(repr.plot.width = 11, repr.plot.height = 4.5)
dd = sig %>% filter(scheme == 'A', direction == 'down') %>% arrange(上位分類, fold) %>%
    mutate(term = factor(sub('^COG ', '', term), levels = sub('^COG ', '', term)))
ggplot(dd, aes(x = fold, y = term, size = Count, fill = padj)) +
    # 破線 = fold 1（濃縮なし）。これより左なら「むしろ少ない」
    geom_vline(xintercept = 1, linetype = 2, colour = 'grey60') +
    geom_point(shape = 21, colour = 'grey30', stroke = 0.5) +
    facet_grid(上位分類 ~ ., scales = 'free_y',
               labeller = labeller(上位分類 = label_wrap_gen(20))) +
    fill_dot_dn + size_dot + thm_dot +
    labs(x = 'fold enrichment', y = NULL,
         title = '(A) pooled - down in KO - COG category, grouped by COG super-category')

# ── 図7-5-2: (A) KO で高い × COG（上位分類でグループ化、x = fold enrichment） ──
options(repr.plot.width = 11, repr.plot.height = 4.5)
dd = sig %>% filter(scheme == 'A', direction == 'up') %>% arrange(上位分類, fold) %>%
    mutate(term = factor(sub('^COG ', '', term), levels = sub('^COG ', '', term)))
ggplot(dd, aes(x = fold, y = term, size = Count, fill = padj)) +
    # 破線 = fold 1（濃縮なし）。これより左なら「むしろ少ない」
    geom_vline(xintercept = 1, linetype = 2, colour = 'grey60') +
    geom_point(shape = 21, colour = 'grey30', stroke = 0.5) +
    facet_grid(上位分類 ~ ., scales = 'free_y',
               labeller = labeller(上位分類 = label_wrap_gen(20))) +
    fill_dot_up + size_dot + thm_dot +
    labs(x = 'fold enrichment', y = NULL,
         title = '(A) pooled - up in KO - COG category, grouped by COG super-category')

# ── 図7-5-3: (B) KO で低い × COG（上位分類でグループ化、x = fold enrichment） ──
options(repr.plot.width = 11, repr.plot.height = 4.5)
dd = sig %>% filter(scheme == 'B', direction == 'down') %>% arrange(上位分類, fold) %>%
    mutate(term = factor(sub('^COG ', '', term), levels = sub('^COG ', '', term)))
ggplot(dd, aes(x = fold, y = term, size = Count, fill = padj)) +
    # 破線 = fold 1（濃縮なし）。これより左なら「むしろ少ない」
    geom_vline(xintercept = 1, linetype = 2, colour = 'grey60') +
    geom_point(shape = 21, colour = 'grey30', stroke = 0.5) +
    facet_grid(上位分類 ~ ., scales = 'free_y',
               labeller = labeller(上位分類 = label_wrap_gen(20))) +
    fill_dot_dn + size_dot + thm_dot +
    labs(x = 'fold enrichment', y = NULL,
         title = '(B) per-strain - down in KO - COG category, grouped by COG super-category')

# ── 図7-5-4: (B) KO で高い × COG（上位分類でグループ化、x = fold enrichment） ──
options(repr.plot.width = 11, repr.plot.height = 4.5)
dd = sig %>% filter(scheme == 'B', direction == 'up') %>% arrange(上位分類, fold) %>%
    mutate(term = factor(sub('^COG ', '', term), levels = sub('^COG ', '', term)))
ggplot(dd, aes(x = fold, y = term, size = Count, fill = padj)) +
    # 破線 = fold 1（濃縮なし）。これより左なら「むしろ少ない」
    geom_vline(xintercept = 1, linetype = 2, colour = 'grey60') +
    geom_point(shape = 21, colour = 'grey30', stroke = 0.5) +
    facet_grid(上位分類 ~ ., scales = 'free_y',
               labeller = labeller(上位分類 = label_wrap_gen(20))) +
    fill_dot_up + size_dot + thm_dot +
    labs(x = 'fold enrichment', y = NULL,
         title = '(B) per-strain - up in KO - COG category, grouped by COG super-category')

# ── 図7-5-5: (A) KO で低い × COG（株ごとの内訳） ──
options(repr.plot.width = 11, repr.plot.height = 4.5)
dd = share %>% filter(scheme == 'A', direction == 'down')
ord = dd %>% distinct(term, 上位分類, fold) %>% arrange(上位分類, fold) %>% pull(term)
dd = dd %>% mutate(term = factor(sub('^COG ', '', term), levels = sub('^COG ', '', ord)))
ggplot(dd, aes(x = width, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.0, width = 0.45, position = position_stack(reverse = TRUE)) +
    facet_grid(上位分類 ~ ., scales = 'free_y',
               labeller = labeller(上位分類 = label_wrap_gen(20))) +
    fill_bar_dn + col_edge +
    # x 軸は 0 の左に余白を作らない（棒の左端を軸に付ける）。セル7-2 と同じ
    scale_x_continuous(expand = expansion(mult = c(0, 0.05))) + thm_dot +
    labs(x = 'GeneRatio (stacked by strain)', y = NULL,
         title = '(A) pooled - down in KO - COG category, stacked by strain within super-category')

# ── 図7-5-6: (A) KO で高い × COG（株ごとの内訳） ──
options(repr.plot.width = 11, repr.plot.height = 4.5)
dd = share %>% filter(scheme == 'A', direction == 'up')
ord = dd %>% distinct(term, 上位分類, fold) %>% arrange(上位分類, fold) %>% pull(term)
dd = dd %>% mutate(term = factor(sub('^COG ', '', term), levels = sub('^COG ', '', ord)))
ggplot(dd, aes(x = width, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.0, width = 0.45, position = position_stack(reverse = TRUE)) +
    facet_grid(上位分類 ~ ., scales = 'free_y',
               labeller = labeller(上位分類 = label_wrap_gen(20))) +
    fill_bar_up + col_edge +
    # x 軸は 0 の左に余白を作らない（棒の左端を軸に付ける）。セル7-2 と同じ
    scale_x_continuous(expand = expansion(mult = c(0, 0.05))) + thm_dot +
    labs(x = 'GeneRatio (stacked by strain)', y = NULL,
         title = '(A) pooled - up in KO - COG category, stacked by strain within super-category')

# ── 図7-5-7: (B) KO で低い × COG（株ごとの内訳） ──
options(repr.plot.width = 11, repr.plot.height = 4.5)
dd = share %>% filter(scheme == 'B', direction == 'down')
ord = dd %>% distinct(term, 上位分類, fold) %>% arrange(上位分類, fold) %>% pull(term)
dd = dd %>% mutate(term = factor(sub('^COG ', '', term), levels = sub('^COG ', '', ord)))
ggplot(dd, aes(x = width, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.0, width = 0.45, position = position_stack(reverse = TRUE)) +
    facet_grid(上位分類 ~ ., scales = 'free_y',
               labeller = labeller(上位分類 = label_wrap_gen(20))) +
    fill_bar_dn + col_edge +
    # x 軸は 0 の左に余白を作らない（棒の左端を軸に付ける）。セル7-2 と同じ
    scale_x_continuous(expand = expansion(mult = c(0, 0.05))) + thm_dot +
    labs(x = 'GeneRatio (stacked by strain)', y = NULL,
         title = '(B) per-strain - down in KO - COG category, stacked by strain within super-category')

# ── 図7-5-8: (B) KO で高い × COG（株ごとの内訳） ──
options(repr.plot.width = 11, repr.plot.height = 4.5)
dd = share %>% filter(scheme == 'B', direction == 'up')
ord = dd %>% distinct(term, 上位分類, fold) %>% arrange(上位分類, fold) %>% pull(term)
dd = dd %>% mutate(term = factor(sub('^COG ', '', term), levels = sub('^COG ', '', ord)))
ggplot(dd, aes(x = width, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.0, width = 0.45, position = position_stack(reverse = TRUE)) +
    facet_grid(上位分類 ~ ., scales = 'free_y',
               labeller = labeller(上位分類 = label_wrap_gen(20))) +
    fill_bar_up + col_edge +
    # x 軸は 0 の左に余白を作らない（棒の左端を軸に付ける）。セル7-2 と同じ
    scale_x_continuous(expand = expansion(mult = c(0, 0.05))) + thm_dot +
    labs(x = 'GeneRatio (stacked by strain)', y = NULL,
         title = '(B) per-strain - up in KO - COG category, stacked by strain within super-category')


■ 有意 COG カテゴリ: 13 行（(A) down 3 / (A) up 4 / (B) down 3 / (B) up 3）

■ Table 1: 有意 COG カテゴリの一覧（上位分類・正味リード・代表遺伝子つき）


scheme,direction,上位分類,COG,term,GeneRatio,BgRatio,fold,padj,正味リード,Bth_VPI,Bth_St8,Bun_8492,Odo_St23,代表遺伝子,代表の株,代表name,代表product
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<chr>,<chr>,<chr>,<chr>
(A),down,Cellular processes and signaling,M,Cell wall/membrane/envelope biogenesis,201/1400,935/8846,1.36,7.4e-06,-558326,56,57,29,59,NQ510_RS01710,Bun_8492,NA,OmpA family protein
(A),down,Cellular processes and signaling,O,"Posttranslational modification, protein turnover, chaperones",75/1400,291/8846,1.63,4.0e-05,-792580,19,17,13,26,NQ510_RS10870,Bun_8492,NA,thioredoxin family protein
(A),down,Metabolism,C,Energy production and conversion,127/1400,528/8846,1.52,5.2e-06,-1334308,31,30,18,48,NQ510_RS12460,Bun_8492,sodB,superoxide dismutase
(A),up,Information storage and processing,J,"Translation, ribosomal structure and biogenesis",123/1539,411/8846,1.72,2.2e-09,341119,6,12,99,6,NQ510_RS06235,Bun_8492,tuf,elongation factor Tu
(A),up,Metabolism,F,Nucleotide transport and metabolism,65/1539,256/8846,1.46,4.6e-03,83314,4,6,47,8,NQ510_RS09940,Bun_8492,pgk,phosphoglycerate kinase
(A),up,Metabolism,H,Coenzyme transport and metabolism,102/1539,388/8846,1.51,4.7e-05,85573,11,9,78,4,NQ510_RS10675,Bun_8492,gcvT,glycine cleavage system aminomet
(A),up,Metabolism,I,Lipid transport and metabolism,47/1539,186/8846,1.45,2.0e-02,71125,1,8,36,2,NQ510_RS05575,Bun_8492,fabF,beta-ketoacyl-ACP synthase II
(B),down,Cellular processes and signaling,M,Cell wall/membrane/envelope biogenesis,199/1438,933/8862,1.31,1.7e-04,-634776,51,54,51,43,NQ510_RS01710,Bun_8492,NA,OmpA family protein
(B),down,Cellular processes and signaling,O,"Posttranslational modification, protein turnover, chaperones",72/1438,290/8862,1.53,5.5e-04,-1084880,17,17,21,17,NQ510_RS16430,Bun_8492,tpx,thiol peroxidase



■ Table 2: 有意 COG カテゴリどうしの遺伝子共有（Jaccard > 0 のペアすべて）
  共有ありのペア: 4


scheme,direction,term1,term2,共通,和集合,jaccard
<chr>,<chr>,<chr>,<chr>,<int>,<int>,<dbl>
A,down,C: Energy production and conversion,"O: Posttranslational modification, protein turnover, chaperones",11,191,0.058
A,up,"J: Translation, ribosomal structure and biogenesis",H: Coenzyme transport and metabolism,2,223,0.009
B,down,C: Energy production and conversion,"O: Posttranslational modification, protein turnover, chaperones",7,184,0.038
B,up,"J: Translation, ribosomal structure and biogenesis",F: Nucleotide transport and metabolism,1,158,0.006


plot without title

plot without title

... (2 outputs left out)

plot without title

plot without title

plot without title

plot without title

### 考察: down は「細胞の外側と品質管理」、up は「代謝と翻訳」

#### 1. 上位分類がきれいに分かれる（図7-5-1 〜 7-5-4）

| 上位分類 | down | up |
|---|---|---|
| **Cellular processes and signaling** | `M` 細胞壁・膜（fold 1.36）/ `O` 翻訳後修飾・シャペロン（**1.63**） | — |
| **Metabolism** | `C` エネルギー産生（1.52） | `F` ヌクレオチド（1.46）/ `H` 補酵素（1.51）/ `I` 脂質（1.45） |
| **Information storage and processing** | — | `J` 翻訳・リボソーム（**1.72**） |

**down 3 本はすべて「外側と品質管理」側、up 4 本はすべて「代謝と翻訳」側**で、
4大分類の境界と向きの境界が一致する。

#### 2. ★ `fold enrichment` で見ると順位が変わる

`GeneRatio` では `M`（201遺伝子）が最大だが、**fold では `O`（1.63）が down 最大、
`J`（1.72）が全体最大。**`M` は term が大きいぶん `GeneRatio` が伸びているだけで、
**背景に対する濃縮の強さでは `O` と `C` のほうが強い。**
→ セル7 の `dotplot`（x = `GeneRatio`）だけ見ていると取り違える。

#### 3. ★ 代表遺伝子が 13 行すべて `Bun_8492`（Table 1）

正味リードを最も動かした遺伝子は 13 行すべてで `Bun_8492`
（`C` = `sodB` / `O` = thioredoxin・`tpx` / `M` = `NQ510_RS01710` OmpA / `J` = `tuf` /
`F` = `pgk` / `H` = `gcvT` / `I` = `fabF`）。
**セル7-3 の「転写量で見ると担い手は `Bun_8492`」がカテゴリ単位でも再現した。**

#### 4. `H: Coenzyme transport` は (A) にしかない

**(B) では padj 0.092 で落ちる唯一のカテゴリ。**菌量変化を含めるかどうかで消えるので、
**`Bun_8492` のシェア増加の寄与が大きい**と読める。

#### 5. ★ `C` と `O` は独立な所見として数えない（Table 2）

| scheme | 組 | 共通 | Jaccard |
|---|---|---|---|
| (A) / (B) | **`C`（エネルギー）— `O`（翻訳後修飾）** | **11 / 7** | 0.058 / 0.038 |

COG も実質ほぼ排他（共有ありは 4 ペアのみ）だが、**`C`—`O` の 11 遺伝子は
`COG_category = "CO"` を持つ 74 遺伝子の一部**で、実体は `sodB`・`tpx`・thioredoxin。
**この2カテゴリが別々に有意になっているのは、同じ酸化ストレス系を2回数えている部分がある。**

#### 6. 株の内訳（図7-5-5 〜 7-5-8）

up 側（`J`・`F`・`H`・`I`）は**どれも `Bun_8492` の区分が棒の大半**を占める
（`J` は 99/123 遺伝子）。一方 down 側の `M`・`C`・`O` は4株が分け合う。
→ **down は群集全体の現象、up は `Bun_8492` の現象**という対比が1枚で読める。


## セル7-6: KEGG module で ORA / GSEA を走らせる（独立したセル）

セル5〜9 は **KEGG pathway / COG / PFAM / CAZy の4ソース**で動いている。
**★ このセルはそこに手を入れず KEGG module だけを独立して扱う。**
TERM2GENE・ORA・GSEA・図を**このセル1つで完結**させる（`geneList` もセル8 に依存せず内部で作る）。

### なぜ module を足すか — pathway は大きすぎて中で向きが混ざる

|- | pathway | module に分けると |
|---|---|---|
| `ko00190` 酸化的リン酸化 | F型ATPase は下がり `Nuo`/`Sdh` は上がる（`Bun` のみ）→ **NES −1.46 で有意にならない** | **`M00157` F型ATPase だけで NES −2.29 / padj 6.8e-05** |
| `ko00020` TCA回路 | 全体で fold 1.85 | **最初の酸化段階 `M00010` は fold 5.03** |
| `ko00520` アミノ糖・ヌクレオチド糖 | 中に全部混ざって消える | **糖ヌクレオチドの種類ごとに向きが分かれる** |

### ★ 代償: 被覆が pathway の半分（`KEGG_ko` が付いた遺伝子しか使えない）

pathway 151 term / 2,333 遺伝子（**23.8%**）に対し、**module 248 term / 1,188 遺伝子（12.1%）**。
主解析の4ソースと対等に並べず別立てにしているのはこのため。
**セル5・6・8 に統合すると下流の図セル（7・7-2・7-3）まで実行済みの出力がやり直しになる**のも理由。

### ★ KEGG の版を日付で固定する（`MOD_VER = '261028'`）

階層 `ko00002.keg` には K番号が載っていないので2ファイル要る
（`ref/ko00002_261028.keg` と `ref/module_ko_261028.tsv`）。
**KEGG は毎日更新される。**実測で **module-K 対応が 10/27 版と 10/28 版で変わり、
ORA の有意 module が 25 → 26 本になった。**→ 版を固定する（セル4 と同じ考え方）。

### ★ ORA では up が1本も出ない

|- | (A) | (B) |
|---|---|---|
| ORA down / up | 12 / **0** | 14 / **0** |
| GSEA down / up | 9 / 3 | 16 / 7 |

**0 であることを出力に明示し、ORA の図は down のみ作る。up を見るには GSEA が要る。**

### 図（8枚）

図7-6-1/2 = ORA dotplot（x = fold、中分類でグループ化）、図7-6-3/4 = ORA 株内訳、
図7-6-5/6 = GSEA dotplot（x = NES、up/down を1枚に）、図7-6-7/8 = GSEA 株内訳。

色の約束はセル7・7-2〜7-5・セル9 と同じ。
★ GSEA の図は**中分類でグループ化するため up/down を1枚に収める必要がある**ので、
セル9 の上下2段ではなく **`ggnewscale` で down と up に別々の `fill` スケール**を当てて同じ色の意味を保つ。


In [158]:
# ── セル7-6: KEGG module で ORA / GSEA を走らせる（独立したセル）──────
# セル5〜9 は KEGG pathway / COG / PFAM / CAZy の4ソースで動いている。
# ★ このセルはそこに手を入れず、KEGG module だけを独立して扱う。
#   TERM2GENE の作成・ORA・GSEA・図を、このセル1つで完結させる。
#
# なぜ module を足すか
#   pathway は大きすぎて中で向きが混在する。実測した例:
#     ko00190 酸化的リン酸化 … F型ATPase は下がり NuoA-I/Sdh は上がる（Bun のみ）
#       → 打ち消し合って GSEA で NES -1.46、有意にならなかった
#   module は「この反応ステップ群」という小さい単位なので、その混在を分けられる。
#   実際 M00157 F型ATPase だけを取り出すと NES -2.29 / padj 6.8e-05 で明確に出る。
#
# ★ 代償: KEGG_ko が付いている遺伝子しか使えない（12.1%。pathway は 23.8%）。
#   主解析の4ソースと並べず別扱いにしているのはこのため。
#
# ★ なぜセル5〜9 に統合しないか
#   統合すると セル5・6・8 とその下流の図セル（7・7-2・7-3）すべてに手が入り、
#   実行済みの出力がすべてやり直しになる。module の被覆が pathway の半分なので、
#   4ソースと対等に並べるより別立てのほうが実態に合う
library(stringr)
suppressMessages(library(KEGGREST))
suppressMessages(library(clusterProfiler))
library(ggplot2)
suppressMessages(library(ggnewscale))

options(repr.matrix.max.rows = 500, repr.matrix.max.cols = 50)

# ステージ5・セル7-2〜7-5 と同じ株の色
col_strain = c(Bth_VPI = '#2a78d6', Bth_St8 = '#eb6834', Bun_8492 = '#3f9e5f',
               Odo_St23 = '#9b59b6')

# ===============================================================
# 1. KEGG から module の情報を取る（取得済みなら読むだけ）
# ===============================================================
# ★ 2つ要る。階層（ko00002.keg）には K番号が載っていないため
#   (1) br:ko00002      … M番号 → 名前 → 分類
#   (2) link/ko/module  … M番号 → K番号（これが TERM2GENE の素）
#
# ★ 版を日付で固定する。KEGG は毎日更新されるので、Sys.Date() で取りに行くと
#   実行日によって結果が変わる。実測: module-K 対応が 2026-10-27 版と
#   2026-10-28 版で変わり、ORA の有意 module が 25 本 → 26 本になった。
#   セル4（ko00001.keg の版を突き合わせる）と同じ考え方
MOD_VER = '261028'
f_mod_keg  = file.path(path, sprintf('ref/ko00002_%s.keg', MOD_VER))
f_mod_link = file.path(path, sprintf('ref/module_ko_%s.tsv', MOD_VER))

if (!file.exists(f_mod_keg)) {
    writeLines(strsplit(keggGet('br:ko00002'), '\n')[[1]], f_mod_keg)
    cat(sprintf("KEGG API から取得して保存: %s\n", basename(f_mod_keg)))
} else cat(sprintf("既に取得済み: %s\n", basename(f_mod_keg)))

if (!file.exists(f_mod_link)) {
    lk = keggLink('ko', 'module')
    write.table(data.frame(module = names(lk), ko = unname(lk)),
                f_mod_link, sep = '\t', row.names = FALSE, quote = FALSE)
    cat(sprintf("KEGG API から取得して保存: %s\n", basename(f_mod_link)))
} else cat(sprintf("既に取得済み: %s\n", basename(f_mod_link)))

# ── 階層を読む（A 大分類 / B 中分類 / C 小分類 / D module）──────
#   ko00001.keg（セル5）と同じ読み方。<b> タグが混ざるので外す
A = ''; B = ''; C = ''; rows = list()
for (l in readLines(f_mod_keg, warn = FALSE)) {
    if (startsWith(l, 'A'))      A = str_trim(str_replace_all(sub('^A', '', l), '</?b>', ''))
    else if (startsWith(l, 'B')) B = str_trim(str_replace_all(sub('^B', '', l), '</?b>', ''))
    else if (startsWith(l, 'C')) C = str_trim(sub('^C\\s*', '', l))
    else if (startsWith(l, 'D')) {
        m = str_match(l, '^D\\s+(M[0-9]+)\\s+(.*)$')
        if (!is.na(m[1]))
            rows[[length(rows) + 1]] = data.frame(
                M = m[2], 大分類 = A, 中分類 = B, 小分類 = C,
                module名 = str_trim(sub('\\s*\\[PATH:[^]]*\\]\\s*$', '', m[3])))
    }
}
mod_map = bind_rows(rows) %>% distinct(M, .keep_all = TRUE)
mod_ko  = read.delim(f_mod_link, stringsAsFactors = FALSE) %>%
          transmute(M = sub('^md:', '', module), K = sub('^ko:', '', ko))

cat(sprintf("\n■ ko00002.keg の module 数: %d / module-K 対応: %d 行（K %d 種）\n",
            nrow(mod_map), nrow(mod_ko), length(unique(mod_ko$K))))

# ===============================================================
# 2. TERM2GENE を作る
# ===============================================================
# res の KEGG_ko は "ko:K00844,ko:K12407" の形。K番号に直して module と結ぶ
t2g_mod = res %>% filter(!is.na(KEGG_ko), KEGG_ko != '-') %>%
    dplyr::select(gene_id, KEGG_ko) %>%
    tidyr::separate_rows(KEGG_ko, sep = ',') %>%
    transmute(gene = gene_id, K = str_trim(sub('^ko:', '', KEGG_ko))) %>%
    filter(K != '') %>%
    inner_join(mod_ko, by = 'K', relationship = 'many-to-many') %>%
    inner_join(mod_map %>% dplyr::select(M, module名), by = 'M') %>%
    transmute(term = paste0(M, '_', module名), gene) %>% distinct()

cat("\n■ Table 1: TERM2GENE の規模（セル5 の4種と並べる）\n")
data.frame(
    source = c('KEGG pathway（セル5）', 'KEGG module（このセル）'),
    terms  = c(length(unique(t2g_kegg$term)), length(unique(t2g_mod$term))),
    対応行 = c(nrow(t2g_kegg), nrow(t2g_mod)),
    遺伝子 = c(length(unique(t2g_kegg$gene)), length(unique(t2g_mod$gene))),
    被覆pct = round(100 * c(length(unique(t2g_kegg$gene)),
                            length(unique(t2g_mod$gene))) / nrow(res), 1),
    check.names = FALSE)

cat("\n■ Table 2: 株ごとに module が付いた遺伝子数\n")
res %>% filter(!marker) %>%
    mutate(mod_hit = gene_id %in% t2g_mod$gene,
           pw_hit  = gene_id %in% t2g_kegg$gene) %>%
    group_by(strain) %>%
    summarise(CDS = dplyr::n(), pathway = sum(pw_hit), module = sum(mod_hit),
              module率 = round(100 * sum(mod_hit) / dplyr::n(), 1), .groups = 'drop') %>%
    as.data.frame()

# ===============================================================
# 3. ORA（セル6 とまったく同じ引数）
# ===============================================================
# universe / pvalueCutoff=1 / qvalueCutoff=1 / minGSSize=1 / maxGSSize=Inf
# （CLAUDE.md §6-4・§6-4b）
A_up_MOD = enricher(upDEGs_A,   universe = universe_A, TERM2GENE = t2g_mod,
                    pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)
A_dn_MOD = enricher(downDEGs_A, universe = universe_A, TERM2GENE = t2g_mod,
                    pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)
B_up_MOD = enricher(upDEGs_B,   universe = universe_B, TERM2GENE = t2g_mod,
                    pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)
B_dn_MOD = enricher(downDEGs_B, universe = universe_B, TERM2GENE = t2g_mod,
                    pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)

ora_mod = rbind(
    as.data.frame(A_up_MOD) %>% mutate(scheme = 'A', direction = 'up'),
    as.data.frame(A_dn_MOD) %>% mutate(scheme = 'A', direction = 'down'),
    as.data.frame(B_up_MOD) %>% mutate(scheme = 'B', direction = 'up'),
    as.data.frame(B_dn_MOD) %>% mutate(scheme = 'B', direction = 'down')) %>%
    transmute(scheme, direction, term = ID, GeneRatio, BgRatio,
              pvalue, padj = p.adjust, Count, geneID) %>%
    # fold enrichment。セル6 の fold と同じ定義（GeneRatio ÷ BgRatio）
    mutate(fold = (as.integer(sub('/.*', '', GeneRatio)) / as.integer(sub('.*/', '', GeneRatio))) /
                  (as.integer(sub('/.*', '', BgRatio))   / as.integer(sub('.*/', '', BgRatio))),
           M = sub('_.*', '', term)) %>%
    left_join(mod_map %>% dplyr::select(M, 中分類, 大分類), by = 'M')

cat(sprintf("\n■ ORA: 全 %d 行、padj<0.05 は %d 行\n", nrow(ora_mod), sum(ora_mod$padj < 0.05)))
cat(sprintf("  ★ 向きの内訳: (A) down %d / (A) up %d / (B) down %d / (B) up %d\n",
    sum(ora_mod$scheme=='A' & ora_mod$direction=='down' & ora_mod$padj<0.05),
    sum(ora_mod$scheme=='A' & ora_mod$direction=='up'   & ora_mod$padj<0.05),
    sum(ora_mod$scheme=='B' & ora_mod$direction=='down' & ora_mod$padj<0.05),
    sum(ora_mod$scheme=='B' & ora_mod$direction=='up'   & ora_mod$padj<0.05)))
cat("  ★ up 側は (A)(B) とも有意 0 本。「無い」ことが結果なので図は down のみ作る。\n")
cat("    up を見るには GSEA が要る（下記 Table 4）\n")

cat("\n■ Table 3: ORA で有意になった module（全行）\n")
ora_mod %>% filter(padj < 0.05) %>%
    transmute(scheme = paste0('(', scheme, ')'), direction, 中分類, term,
              GeneRatio, BgRatio, fold = round(fold, 2), padj = signif(padj, 2)) %>%
    arrange(scheme, direction, 中分類, desc(fold)) %>% as.data.frame()

# ===============================================================
# 4. GSEA（セル8 とまったく同じ引数）
# ===============================================================
# ★ このセルを単独で実行できるよう、geneList もここで作る
#   （セル8 の geneList_A / geneList_B と同じ作り方。Wald 統計量で降順）
gl_A = res %>% filter(!is.na(stat_A), !marker) %>% arrange(desc(stat_A))
geneList_mod_A = gl_A$stat_A; names(geneList_mod_A) = gl_A$gene_id
gl_B = res %>% filter(!is.na(stat_B), !marker) %>% arrange(desc(stat_B))
geneList_mod_B = gl_B$stat_B; names(geneList_mod_B) = gl_B$gene_id

set.seed(1)
gsea_A_MOD = suppressWarnings(GSEA(geneList_mod_A, TERM2GENE = t2g_mod, pvalueCutoff = 1,
                 minGSSize = 1, maxGSSize = Inf, eps = 0, seed = TRUE, verbose = FALSE))
set.seed(1)
gsea_B_MOD = suppressWarnings(GSEA(geneList_mod_B, TERM2GENE = t2g_mod, pvalueCutoff = 1,
                 minGSSize = 1, maxGSSize = Inf, eps = 0, seed = TRUE, verbose = FALSE))

gsea_mod = rbind(
    as.data.frame(gsea_A_MOD) %>% mutate(scheme = 'A'),
    as.data.frame(gsea_B_MOD) %>% mutate(scheme = 'B')) %>%
    transmute(scheme, term = ID, setSize, NES, pvalue, padj = p.adjust,
              direction = ifelse(NES > 0, 'up', 'down'), core = core_enrichment) %>%
    mutate(M = sub('_.*', '', term)) %>%
    left_join(mod_map %>% dplyr::select(M, 中分類, 大分類), by = 'M')

cat(sprintf("\n■ GSEA: 全 %d 行、padj<0.05 は %d 行\n", nrow(gsea_mod), sum(gsea_mod$padj < 0.05)))
cat(sprintf("  向きの内訳: (A) down %d / (A) up %d / (B) down %d / (B) up %d\n",
    sum(gsea_mod$scheme=='A' & gsea_mod$direction=='down' & gsea_mod$padj<0.05),
    sum(gsea_mod$scheme=='A' & gsea_mod$direction=='up'   & gsea_mod$padj<0.05),
    sum(gsea_mod$scheme=='B' & gsea_mod$direction=='down' & gsea_mod$padj<0.05),
    sum(gsea_mod$scheme=='B' & gsea_mod$direction=='up'   & gsea_mod$padj<0.05)))

cat("\n■ Table 4: GSEA で有意になった module（全行）\n")
gsea_mod %>% filter(padj < 0.05) %>%
    transmute(scheme = paste0('(', scheme, ')'), direction, 中分類, term,
              setSize, NES = round(NES, 2), padj = signif(padj, 2)) %>%
    arrange(scheme, 中分類, NES) %>% as.data.frame()

# ===============================================================
# 5. 図のための下ごしらえ
# ===============================================================
# ── ORA: 株ごとの内訳（セル7-2 と同じ作り。棒の長さ = GeneRatio）──
sigm = ora_mod %>% filter(padj < 0.05)
share_mod = sigm %>% dplyr::select(scheme, direction, term, 中分類, geneID, GeneRatio) %>%
    tidyr::separate_rows(geneID, sep = '/') %>%
    left_join(res %>% dplyr::select(gene_id, strain), by = c('geneID' = 'gene_id')) %>%
    dplyr::count(scheme, direction, term, 中分類, GeneRatio, strain, name = 'n_gene') %>%
    # ★ 当たり遺伝子が0本の株も行として残す（0 を明示する）
    tidyr::complete(tidyr::nesting(scheme, direction, term, 中分類, GeneRatio),
                    strain = names(col_strain), fill = list(n_gene = 0)) %>%
    ungroup() %>%
    mutate(strain = factor(strain, levels = names(col_strain)),
           width = n_gene / as.integer(sub('.*/', '', GeneRatio))) %>%
    left_join(sigm %>% dplyr::select(scheme, direction, term, padj, fold),
              by = c('scheme', 'direction', 'term'))

# ── GSEA: 株ごとの内訳（セル9 と同じ作り）──────────────────────
# 棒の長さ = NES。区分 = core enrichment 遺伝子の株ごとの割合 × NES。
# 足すと NES になる。NES が負なら棒は左へ伸びる
gm = gsea_mod %>% filter(padj < 0.05)
core_strain_mod = gm %>% dplyr::select(scheme, term, core) %>%
    tidyr::separate_rows(core, sep = '/') %>%
    left_join(res %>% dplyr::select(gene_id, strain), by = c('core' = 'gene_id')) %>%
    dplyr::count(scheme, term, strain, name = 'n_core') %>%
    tidyr::complete(tidyr::nesting(scheme, term), strain = names(col_strain),
                    fill = list(n_core = 0)) %>% ungroup()
bar_gm = gm %>% left_join(core_strain_mod, by = c('scheme', 'term')) %>%
    group_by(scheme, term) %>% mutate(width = NES * n_core / sum(n_core)) %>% ungroup() %>%
    mutate(strain = factor(strain, levels = names(col_strain)))

cat("\n■ Table 5: GSEA の core enrichment 遺伝子の株ごとの内訳\n")
gm %>% left_join(core_strain_mod %>% tidyr::pivot_wider(names_from = strain, values_from = n_core),
                 by = c('scheme', 'term')) %>%
    transmute(scheme = paste0('(', scheme, ')'), 中分類, term, setSize,
              NES = round(NES, 2), padj = signif(padj, 2),
              Bth_VPI, Bth_St8, Bun_8492, Odo_St23,
              core計 = Bth_VPI + Bth_St8 + Bun_8492 + Odo_St23) %>%
    arrange(scheme, 中分類, NES) %>% as.data.frame()

# ── 見た目の約束（セル7-4・7-5 と同一）────────────────────────
thm_mod = theme_bw() +
    theme(panel.grid.minor = element_blank(),
          strip.text.y.right = element_text(angle = 0, hjust = 0, size = 11, face = 'bold'),
          strip.background = element_rect(fill = 'grey92', colour = 'grey60'),
          legend.key.size = unit(0.9, 'lines'))
thm_mod_bar = thm_mod + theme(panel.grid.major.y = element_blank())

# dotplot は濃い色（セル7 と同じ）。★ 白い点は背景に溶けるので
#   shape = 21 の fill に割り当て、灰色の縁を付ける
fill_dot_dn = scale_fill_gradient(low = 'blue', high = 'white', limits = c(0, 0.05), name = 'p.adjust')
size_dot = scale_size_continuous(range = c(3, 9), name = 'gene count',
                                 guide = guide_legend(override.aes = list(fill = 'grey70')))
# 棒は薄い色（セル7-2・7-3・7-4・7-5 と同じ理由。濃いと株の縁が塗りに埋もれる）
fill_bar_dn = scale_fill_gradient(low = '#8fb8f5', high = '#eaf1fd', limits = c(0, 0.05), name = 'p.adjust')
col_edge = scale_colour_manual(values = col_strain, name = 'strain',
    guide = guide_legend(override.aes = list(fill = 'white', linewidth = 1)))

# ★ module 名は長い。y 軸ラベルは 58 文字で切る（それ以上だと図が名前で埋まる）
TRUNC = 58

# ── 図7-6-1: (A) KO で低い × KEGG module（中分類でグループ化、x = fold enrichment） ──
# ★ up は有意 0 本なので図は作らない（上の cat で 0 であることを出してある）
options(repr.plot.width = 14, repr.plot.height = 5.5)
dd = ora_mod %>% filter(scheme == 'A', direction == 'down', padj < 0.05) %>%
    arrange(中分類, fold) %>%
    mutate(term = factor(substr(term, 1, TRUNC), levels = substr(term, 1, TRUNC)))
ggplot(dd, aes(x = fold, y = term, size = Count, fill = padj)) +
    # 破線 = fold 1（濃縮なし）
    geom_vline(xintercept = 1, linetype = 2, colour = 'grey60') +
    geom_point(shape = 21, colour = 'grey30', stroke = 0.5) +
    facet_grid(中分類 ~ ., scales = 'free_y',
               labeller = labeller(中分類 = label_wrap_gen(18))) +
    fill_dot_dn + size_dot + thm_mod +
    labs(x = 'fold enrichment', y = NULL,
         title = '(A) pooled - down in KO - KEGG module, grouped by module class')

# ── 図7-6-2: (B) KO で低い × KEGG module（中分類でグループ化、x = fold enrichment） ──
# ★ up は有意 0 本なので図は作らない（上の cat で 0 であることを出してある）
options(repr.plot.width = 14, repr.plot.height = 5.5)
dd = ora_mod %>% filter(scheme == 'B', direction == 'down', padj < 0.05) %>%
    arrange(中分類, fold) %>%
    mutate(term = factor(substr(term, 1, TRUNC), levels = substr(term, 1, TRUNC)))
ggplot(dd, aes(x = fold, y = term, size = Count, fill = padj)) +
    # 破線 = fold 1（濃縮なし）
    geom_vline(xintercept = 1, linetype = 2, colour = 'grey60') +
    geom_point(shape = 21, colour = 'grey30', stroke = 0.5) +
    facet_grid(中分類 ~ ., scales = 'free_y',
               labeller = labeller(中分類 = label_wrap_gen(18))) +
    fill_dot_dn + size_dot + thm_mod +
    labs(x = 'fold enrichment', y = NULL,
         title = '(B) per-strain - down in KO - KEGG module, grouped by module class')

# ── 図7-6-3: (A) KO で低い × KEGG module（株ごとの内訳） ──
options(repr.plot.width = 14, repr.plot.height = 5.5)
dd = share_mod %>% filter(scheme == 'A', direction == 'down')
ord = dd %>% distinct(term, 中分類, fold) %>% arrange(中分類, fold) %>% pull(term)
dd = dd %>% mutate(term = factor(substr(term, 1, TRUNC), levels = substr(ord, 1, TRUNC)))
ggplot(dd, aes(x = width, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.0, width = 0.45, position = position_stack(reverse = TRUE)) +
    facet_grid(中分類 ~ ., scales = 'free_y',
               labeller = labeller(中分類 = label_wrap_gen(18))) +
    fill_bar_dn + col_edge +
    # x 軸は 0 の左に余白を作らない（棒の左端を軸に付ける）。セル7-2 と同じ
    scale_x_continuous(expand = expansion(mult = c(0, 0.05))) + thm_mod_bar +
    labs(x = 'GeneRatio (stacked by strain)', y = NULL,
         title = '(A) pooled - down in KO - KEGG module, stacked by strain within module class')

# ── 図7-6-4: (B) KO で低い × KEGG module（株ごとの内訳） ──
options(repr.plot.width = 14, repr.plot.height = 5.5)
dd = share_mod %>% filter(scheme == 'B', direction == 'down')
ord = dd %>% distinct(term, 中分類, fold) %>% arrange(中分類, fold) %>% pull(term)
dd = dd %>% mutate(term = factor(substr(term, 1, TRUNC), levels = substr(ord, 1, TRUNC)))
ggplot(dd, aes(x = width, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.0, width = 0.45, position = position_stack(reverse = TRUE)) +
    facet_grid(中分類 ~ ., scales = 'free_y',
               labeller = labeller(中分類 = label_wrap_gen(18))) +
    fill_bar_dn + col_edge +
    # x 軸は 0 の左に余白を作らない（棒の左端を軸に付ける）。セル7-2 と同じ
    scale_x_continuous(expand = expansion(mult = c(0, 0.05))) + thm_mod_bar +
    labs(x = 'GeneRatio (stacked by strain)', y = NULL,
         title = '(B) per-strain - down in KO - KEGG module, stacked by strain within module class')

# ── 図7-6-5: (A) KEGG module の GSEA（x = NES。up と down を1枚に） ──
# ★ 色の意味はセル9 と同じ「KO で低い = 青 / KO で高い = 赤、0.05 = 白」。
#   セル9 は up と down を上下2段のサブプロットに分けて別々の塗りを当てていたが、
#   ここは 中分類 でグループ化したいので1枚に収める必要がある。
#   → ggnewscale で down と up に別々の fill スケールを当てる
options(repr.plot.width = 14, repr.plot.height = 6.5)
dd = gsea_mod %>% filter(scheme == 'A', padj < 0.05) %>% arrange(中分類, NES) %>%
    mutate(term = factor(substr(term, 1, TRUNC), levels = substr(term, 1, TRUNC)))
lim = max(abs(dd$NES)) * 1.08
ggplot(dd, aes(x = NES, y = term, size = setSize)) +
    geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
    geom_point(data = dd %>% filter(NES < 0), aes(fill = padj),
               shape = 21, colour = 'grey30', stroke = 0.5) +
    scale_fill_gradient(low = 'blue', high = 'white', limits = c(0, 0.05),
                        name = 'p.adjust\n(down in KO)') +
    new_scale_fill() +
    geom_point(data = dd %>% filter(NES > 0), aes(fill = padj),
               shape = 21, colour = 'grey30', stroke = 0.5) +
    scale_fill_gradient(low = 'red', high = 'white', limits = c(0, 0.05),
                        name = 'p.adjust\n(up in KO)') +
    facet_grid(中分類 ~ ., scales = 'free_y', space = 'free_y',
               labeller = labeller(中分類 = label_wrap_gen(18))) +
    scale_x_continuous(limits = c(-lim, lim)) +
    scale_size_continuous(range = c(3, 9), name = 'set size',
                          guide = guide_legend(override.aes = list(fill = 'grey70'))) +
    thm_mod +
    labs(x = 'NES  (negative = down in KO, positive = up in KO)', y = NULL,
         title = '(A) pooled - KEGG module - GSEA')

# ── 図7-6-6: (B) KEGG module の GSEA（x = NES。up と down を1枚に） ──
# ★ 色の意味はセル9 と同じ「KO で低い = 青 / KO で高い = 赤、0.05 = 白」。
#   セル9 は up と down を上下2段のサブプロットに分けて別々の塗りを当てていたが、
#   ここは 中分類 でグループ化したいので1枚に収める必要がある。
#   → ggnewscale で down と up に別々の fill スケールを当てる
options(repr.plot.width = 14, repr.plot.height = 6.5)
dd = gsea_mod %>% filter(scheme == 'B', padj < 0.05) %>% arrange(中分類, NES) %>%
    mutate(term = factor(substr(term, 1, TRUNC), levels = substr(term, 1, TRUNC)))
lim = max(abs(dd$NES)) * 1.08
ggplot(dd, aes(x = NES, y = term, size = setSize)) +
    geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
    geom_point(data = dd %>% filter(NES < 0), aes(fill = padj),
               shape = 21, colour = 'grey30', stroke = 0.5) +
    scale_fill_gradient(low = 'blue', high = 'white', limits = c(0, 0.05),
                        name = 'p.adjust\n(down in KO)') +
    new_scale_fill() +
    geom_point(data = dd %>% filter(NES > 0), aes(fill = padj),
               shape = 21, colour = 'grey30', stroke = 0.5) +
    scale_fill_gradient(low = 'red', high = 'white', limits = c(0, 0.05),
                        name = 'p.adjust\n(up in KO)') +
    facet_grid(中分類 ~ ., scales = 'free_y', space = 'free_y',
               labeller = labeller(中分類 = label_wrap_gen(18))) +
    scale_x_continuous(limits = c(-lim, lim)) +
    scale_size_continuous(range = c(3, 9), name = 'set size',
                          guide = guide_legend(override.aes = list(fill = 'grey70'))) +
    thm_mod +
    labs(x = 'NES  (negative = down in KO, positive = up in KO)', y = NULL,
         title = '(B) per-strain - KEGG module - GSEA')

# ── 図7-6-7: (A) KEGG module の GSEA（株ごとの内訳） ──
# 棒の長さ = NES。区分 = core enrichment 遺伝子の株ごとの割合 × NES。
# 足すと NES になる（セル9 の横棒と同じ作り）
options(repr.plot.width = 14, repr.plot.height = 6.5)
dd = bar_gm %>% filter(scheme == 'A')
ord = dd %>% distinct(term, 中分類, NES) %>% arrange(中分類, NES) %>% pull(term)
dd = dd %>% mutate(term = factor(substr(term, 1, TRUNC), levels = substr(ord, 1, TRUNC)))
lim = max(abs(dd$NES)) * 1.08
ggplot(dd, aes(x = width, y = term, colour = strain)) +
    geom_vline(xintercept = 0, colour = 'grey30') +
    geom_col(data = dd %>% filter(NES < 0), aes(fill = padj),
             linewidth = 1.0, width = 0.45, position = position_stack(reverse = TRUE)) +
    scale_fill_gradient(low = '#8fb8f5', high = '#eaf1fd', limits = c(0, 0.05),
                        name = 'p.adjust\n(down in KO)') +
    new_scale_fill() +
    geom_col(data = dd %>% filter(NES > 0), aes(fill = padj),
             linewidth = 1.0, width = 0.45, position = position_stack(reverse = TRUE)) +
    scale_fill_gradient(low = '#f08080', high = '#fdeaea', limits = c(0, 0.05),
                        name = 'p.adjust\n(up in KO)') +
    col_edge +
    facet_grid(中分類 ~ ., scales = 'free_y', space = 'free_y',
               labeller = labeller(中分類 = label_wrap_gen(18))) +
    scale_x_continuous(limits = c(-lim, lim)) + thm_mod_bar +
    labs(x = 'NES (stacked by strain, from core enrichment genes)', y = NULL,
         title = '(A) pooled - KEGG module - GSEA, stacked by strain within module class')

# ── 図7-6-8: (B) KEGG module の GSEA（株ごとの内訳） ──
# 棒の長さ = NES。区分 = core enrichment 遺伝子の株ごとの割合 × NES。
# 足すと NES になる（セル9 の横棒と同じ作り）
options(repr.plot.width = 14, repr.plot.height = 6.5)
dd = bar_gm %>% filter(scheme == 'B')
ord = dd %>% distinct(term, 中分類, NES) %>% arrange(中分類, NES) %>% pull(term)
dd = dd %>% mutate(term = factor(substr(term, 1, TRUNC), levels = substr(ord, 1, TRUNC)))
lim = max(abs(dd$NES)) * 1.08
ggplot(dd, aes(x = width, y = term, colour = strain)) +
    geom_vline(xintercept = 0, colour = 'grey30') +
    geom_col(data = dd %>% filter(NES < 0), aes(fill = padj),
             linewidth = 1.0, width = 0.45, position = position_stack(reverse = TRUE)) +
    scale_fill_gradient(low = '#8fb8f5', high = '#eaf1fd', limits = c(0, 0.05),
                        name = 'p.adjust\n(down in KO)') +
    new_scale_fill() +
    geom_col(data = dd %>% filter(NES > 0), aes(fill = padj),
             linewidth = 1.0, width = 0.45, position = position_stack(reverse = TRUE)) +
    scale_fill_gradient(low = '#f08080', high = '#fdeaea', limits = c(0, 0.05),
                        name = 'p.adjust\n(up in KO)') +
    col_edge +
    facet_grid(中分類 ~ ., scales = 'free_y', space = 'free_y',
               labeller = labeller(中分類 = label_wrap_gen(18))) +
    scale_x_continuous(limits = c(-lim, lim)) + thm_mod_bar +
    labs(x = 'NES (stacked by strain, from core enrichment genes)', y = NULL,
         title = '(B) per-strain - KEGG module - GSEA, stacked by strain within module class')


既に取得済み: ko00002_261028.keg
既に取得済み: module_ko_261028.tsv

■ ko00002.keg の module 数: 582 / module-K 対応: 4311 行（K 3143 種）

■ Table 1: TERM2GENE の規模（セル5 の4種と並べる）


source,terms,対応行,遺伝子,被覆pct
<chr>,<int>,<int>,<int>,<dbl>
KEGG pathway（セル5）,151,4324,2333,23.8
KEGG module（このセル）,248,2301,1188,12.1



■ Table 2: 株ごとに module が付いた遺伝子数


strain,CDS,pathway,module,module率
<chr>,<int>,<int>,<int>,<dbl>
Bth_St8,1829,362,187,10.2
Bth_VPI,1710,354,188,11.0
Bun_8492,3390,917,465,13.7
Odo_St23,2849,700,348,12.2
pME008,10,0,0,0.0



■ ORA: 全 579 行、padj<0.05 は 26 行
  ★ 向きの内訳: (A) down 12 / (A) up 0 / (B) down 14 / (B) up 0
  ★ up 側は (A)(B) とも有意 0 本。「無い」ことが結果なので図は down のみ作る。
    up を見るには GSEA が要る（下記 Table 4）

■ Table 3: ORA で有意になった module（全行）


scheme,direction,中分類,term,GeneRatio,BgRatio,fold,padj
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
(A),down,Carbohydrate metabolism,"M00010_Citrate cycle, first carbon oxidation, oxaloacetate => 2-oxoglutarate",9/237,9/1175,4.96,2.1e-05
(A),down,Carbohydrate metabolism,"M00006_Pentose phosphate pathway, oxidative phase, glucose-6P => ribulose-5P",10/237,12/1175,4.13,1.4e-04
(A),down,Carbohydrate metabolism,M00004_Pentose phosphate pathway (Pentose phosphate cycle),24/237,33/1175,3.61,6.6e-09
(A),down,Carbohydrate metabolism,"M00008_Entner-Doudoroff pathway, glucose-6P => glyceraldehyde-3P + pyruvate",6/237,9/1175,3.31,3.3e-02
(A),down,Carbohydrate metabolism,M00012_Glyoxylate cycle,6/237,9/1175,3.31,3.3e-02
(A),down,Carbohydrate metabolism,M00740_Methylaspartate cycle,6/237,9/1175,3.31,3.3e-02
(A),down,Carbohydrate metabolism,"M00007_Pentose phosphate pathway, non-oxidative phase, fructose-6P => ribose-5P",11/237,17/1175,3.21,1.8e-03
(A),down,Carbohydrate metabolism,"M00009_Citrate cycle (TCA cycle, Krebs cycle)",19/237,49/1175,1.92,2.4e-02
(A),down,Energy metabolism,"M00157_F-type ATPase, prokaryotes and chloroplasts",15/237,18/1175,4.13,7.5e-07


... (8 outputs left out)

plot without title

plot without title

plot without title

plot without title

### 考察: pathway では打ち消し合っていたものが module で分離できた

#### 1. ★★ 狙いどおりの分離ができた

|- | pathway | module |
|---|---|---|
| 酸化的リン酸化 | `ko00190` GSEA **NES −1.46 で有意にならない** | **`M00157` F型ATPase だけで ORA fold 4.47 / padj 2.6e-08、GSEA NES −2.31** |
| TCA回路 | `ko00020` fold 1.85 | **`M00010` 最初の酸化段階は fold 5.03 / padj 1.8e-05** |

**セル11 で「F型ATPase の低下だけが4株共通」と確定した所見が、module では検定の段階で分離される。**
`M00010`（9/9 遺伝子が当たる）と `M00009`（TCA 全体、18/49）の差も、
**低下しているのが酸化段階の入口であることを示す。**

#### 2. ORA は down のみ、up は GSEA でしか出ない

ORA は **(A) 12 / (B) 14 本すべて down、up は 0 本。**GSEA は (A) up 3 / (B) up 7 本。
**ORA だけでは「KO で下がった」しか言えない。**

ORA down の中身は **PPP（`M00004`/`M00006`/`M00007`、fold 3.2〜4.2）**、
**TCA（`M00009`/`M00010`）**、**`M00157` F型ATPase**、**ED 経路・グリオキシル酸回路**に集中する。

#### 3. ★★ 同じ `Glycan metabolism` の中で糖ヌクレオチドの種類ごとに向きが逆（図7-6-5 / 7-6-6）

| module | scheme | NES | padj |
|---|---|---|---|
| **`M00922` CMP-Neu5Ac**（シアル酸） | (A) / (B) | **+1.87 / +1.73** | 9.7e-04 / 3.1e-02 |
| **`M01011` CMP-Leg5Ac7Ac** | (A) / (B) | **+1.67 / +1.62** | 3.5e-05 / 1.7e-02 |
| **`M01006` GDP-D-Rha4N** | (B) | **+1.91** | 2.7e-02 |
| `M01030` CDP-Abe / `M01031` CDP-Asc | (A) | −1.69 | 4.9e-02 |
| `M01029` CDP-Tyv | (B) | −1.72 | 4.2e-02 |

**`ko00520` アミノ糖・ヌクレオチド糖代謝の中に全部混ざって消えていたものに名前が付いた。**
これは **CLAUDE.md §8 の CPS 座位の入れ替わり**（`Bth_St8` の `29520` 座位が KO で高く、
`26020`・`37520` が W で高い）そのもの。

★ **株内訳（図7-6-7 / 7-6-8）で上がる側と下がる側の株が完全に分かれる。**
`M00922`・`M01011` の core enrichment 遺伝子は **`Bth_St8` 単独（2 / 0 / 0 / 0）**、
`M01029` は **`Bth_VPI` 2 ＋ `Odo_St23` 2 で `Bth_St8` は 0。**

#### 4. (B) でのみ出るもの

**down**: アミノ酸合成4本（Val/Ile・Ile・Cys・Trp、NES −1.77〜−1.80）。
**up**: リボフラビン合成2本（`M00125`/`M00911`、+1.79/+1.84）、
プリンヌクレオチド合成2本（`M00049` アデニン +1.74 / `M00050` グアニン +1.80）。

★ **`M00049`/`M00050` は「ATP 合成」ではなく核酸素材の de novo 生合成。**
**不可逆段階（`purA` +0.73 / `purB` +1.22 / `guaA` +1.19 / `guaB` +1.27、いずれも `Bun_8492`）が
上がっているので「合成能が増えた」とは言える**が、
**`adk`/`gmk` は平衡定数ほぼ1の自由可逆反応なので正味の向きは本データから決まらない**
（F型ATPase と同じ制約。セル11 考察 1）。

#### 5. ★ 注意: 同じ遺伝子集合の module がある

`M01030`（CDP-Abe）と `M01031`（CDP-Asc）は**同じ4遺伝子で Jaccard = 1**、
`M00012`（グリオキシル酸回路）と `M00740`（メチルアスパラギン酸回路）も同値。
`M01011` は `setSize` 3 しかない。**本数を所見の数として数えない**のは PFAM・COG と同じ。

#### 6. 代償は被覆

`KEGG_ko` が付いた **1,188 遺伝子（12.1%）**しか使えず、pathway（2,333 / 23.8%）の半分。
**module で何も出ないことは「その機能が動いていない」ことを意味しない。**


## セル7-7: TCDB（輸送体分類）で ORA / GSEA を走らせる（独立したセル）

セル7-6 と同じく**セル5〜9 に手を入れずに独立して扱う**（`geneList` も内部で作る）。

### なぜ TCDB を足すか — 他のソースに相当する単位が無い輸送体がある

| 単位 | KEGG module | COG | PFAM | TCDB |
|---|---|---|---|---|
| **BAM**（外膜タンパク挿入装置） | なし | M に埋没 | 個別ドメイン | **`TC 1.B.33`** |
| **Lpt**（LPS 排出） | なし | M に埋没 | 個別ドメイン | **`TC 1.B.42`** |
| **TonB-ExbB / Tol-Pal** | なし | M に埋没 | 個別ドメイン | **`TC 2.C.1`** |
| **SusC** / **SusD** | なし | なし | 個別ドメイン | **`TC 1.B.14`** / **`TC 8.A.46`** |

### ★ ただし検定としてはほとんど機能しない

**付与率 8.0%（786 遺伝子 / 85 ファミリー）しかなく、ORA 有意 4 行・GSEA 有意 0 行**
（他ソースは 13〜52 行）。

| scheme | 向き | ファミリー | fold | padj |
|---|---|---|---|---|
| (A) / (B) | down | `TC 3.A.2` F型/V型/A型 ATPase | 2.05 / 2.07 | 0.033 / 0.020 |
| (B) | **up** | `TC 1.B.42` LPS 排出ポーリン | **4.16** | 0.032 |
| (B) | **up** | `TC 3.B.1` Na+ 輸送型カルボン酸デカルボキシラーゼ | **3.78** | 0.032 |

★ **`TC 3.A.2` は F型と V/A型を同じファミリーに入れている**ので `ko00190` と同じ向きの混在が起きる。
**KEGG module なら `M00157`（F型）と `M00159`（V/A型）に分けられるのに、TCDB では分けられない。**
★ **(A) up は有意 0 本**なので出力に明示し、ORA の図は 6 枚。

### ★ 実質の所見は「正味リード増減」の方にある（図7-7-7 / 7-7-8）

| 減少 上位（(A)） | 正味 | | 増加 上位（(A)） | 正味 |
|---|---|---|---|---|
| **`TC 3.A.3` P型 ATPase**（21遺伝子） | **−135,298** | | **`TC 3.A.5` Sec**（18） | **+23,928** |
| `TC 3.A.2` F/V/A型 ATPase（42） | −108,915 | | `TC 1.A.30` 鞭毛モーター（10） | +16,443 |
| `TC 1.A.33` Hsp70（★ 実体は `dnaK`）（6） | −87,967 | | **`TC 1.B.33` BAM**（4） | +12,092 |
| `TC 2.C.1` TonB / Tol-Pal（26） | −33,147 | | **`TC 1.B.42` Lpt**（10） | +4,832 |

**`TC 3.A.3` の減少は銅排出ポンプ5本（`copA` ×3 / `actP` ×2）で 96%**
（`BT_RS05490` −39,043 / `NQ510_RS12055` −35,743 / `CE91St8_34640` −33,504 ほか）。
同じファミリーの K+ ポンプ（`kdpABC`）は n.s. で `Bun_8492` ではむしろ上昇（`kdpA` +2.06）。
→ **「P型 ATPase が下がった」ではなく「重金属排出ポンプだけが下がった」が正確。**
★ この群は **PFAM の `HMA`（9遺伝子、(A) padj 2.7e-03）でも拾えている**ので、
TCDB を入れなくても所見自体は失われない。

### ★ 読むときの注意

- **`TC 1.A.33` "Cation Channel-forming Heat Shock Protein-70" の実体は `dnaK`**（分子シャペロン）で
  **輸送体ではない。**PFAM の `Clp_N`/`DnaJ_C` が down だったのと同じ現象を別ラベルで拾っている
- **`TC 3.A.1` ABC（136遺伝子）は差し引きほぼゼロ**（−17,704、×0.95。有意低下 34 / 上昇 20 が相殺）
- **図7-7-7 / 7-7-8 は検定ではなく記述統計。** p 値を付けていないのは検定していないため

### 図（8枚）

図7-7-1〜3 = ORA dotplot（(A) down / (B) down / (B) up）、図7-7-4〜6 = ORA 株内訳、
**図7-7-7 / 7-7-8 = 全 85 ファミリーの正味リード増減（株で積み上げ、検定なし）。**
図7-7-1・2・4・5 は**有意が 1 行しかないので 1 行だけの図**になる。少なさ自体が結果なのでそのまま出す。


In [159]:
# ── セル7-7: TCDB（輸送体分類）で ORA / GSEA を走らせる（独立したセル）──
# セル7-6（KEGG module）と同じく、セル5〜9 に手を入れずに独立して扱う。
#
# ★ TCDB を足す理由は「他のソースに相当する単位が無い輸送体」があること
#   BAM（外膜タンパク挿入装置）/ Lpt（LPS 排出）/ TonB-ExbB・Tol-Pal /
#   SusC（OMR）・SusD（Glycan-binding）など。KEGG module にも COG にも
#   PFAM にも、これらを1単位でまとめる term が無い。
#
# ★ ただし検定としてはほとんど機能しない（実測）
#   付与率 8.6%（786 遺伝子 / 85 ファミリー）しかなく、
#   ORA 有意 4 行・GSEA 有意 0 行。他ソース（13〜35 行）とは桁違いに少ない。
#   → ORA/GSEA は「出なかった」ことを含めてそのまま出し、
#     ★ 実質の所見は「全 85 ファミリーの正味リード増減」（図7-7-7 / 7-7-8）で示す。
#     こちらは検定ではなく記述統計なので、p 値を付けずに量だけを出す
library(stringr)
suppressMessages(library(clusterProfiler))
library(ggplot2)

options(repr.matrix.max.rows = 500, repr.matrix.max.cols = 50)

col_strain = c(Bth_VPI = '#2a78d6', Bth_St8 = '#eb6834', Bun_8492 = '#3f9e5f',
               Odo_St23 = '#9b59b6')

# ===============================================================
# 1. TC 番号 → ファミリー名
# ===============================================================
# ★ KEGG BRITE の br:ko02000 は使えない。TC 番号が 56 件しか載っておらず、
#   中身は ABC/SLC のファミリー名で整理されていて TC 体系になっていない。
#   → TCDB 本体の公開リストから取る（2,301 ファミリー）
#     https://www.tcdb.org/cgi-bin/projectv/public/families.py （タブ区切り）
f_tc = file.path(path, 'ref/tcdb_families_261027.tsv')
tc_map = read.delim(f_tc, header = FALSE, quote = '', stringsAsFactors = FALSE)
names(tc_map) = c('TC', 'ファミリー名')
tc_map$ファミリー名 = str_trim(tc_map$ファミリー名)
cat(sprintf("■ TCDB ファミリー %d 件を %s から読んだ\n", nrow(tc_map), basename(f_tc)))

# TCDB の第1階層（クラス）。数が少なく変わらないので直接書く。
# ★ 図の facet に出すので英語のみ（日本語を入れると PNG 保存時に文字化けする）
tc_class = c('1' = '1 Channels and pores',
             '2' = '2 Electrochemical potential-driven',
             '3' = '3 Primary active transporters',
             '4' = '4 Group translocators',
             '5' = '5 Transmembrane electron carriers',
             '8' = '8 Accessory factors',
             '9' = '9 Incompletely characterized')

# ===============================================================
# 2. eggNOG の KEGG_TC 列から gene → TC を作る
# ===============================================================
# ★ res には KEGG_TC が入っていないので、eggNOG の生ファイルから読み直す
#   （18 列目が KEGG_TC。ヘッダ行は # で始まるので comment.char で落とす）
ann = read.delim(file.path(path, 'eggnog/all4.emapper.annotations'),
                 comment.char = '#', header = FALSE, stringsAsFactors = FALSE)
names(ann)[c(1, 18)] = c('gene_id', 'KEGG_TC')
cat(sprintf("  eggNOG 行 %d / KEGG_TC あり %d（%.1f%%）\n", nrow(ann),
            sum(ann$KEGG_TC != '-' & ann$KEGG_TC != ''),
            100 * sum(ann$KEGG_TC != '-' & ann$KEGG_TC != '') / nrow(ann)))

# TC は "3.A.1.7" や "8.A.46.1,8.A.46.3" の形。
# ★ ファミリー（上3桁 x.y.z）でまとめる。4桁目まで使うと1遺伝子1 term になって検定にならない
t2g_tc = ann %>% filter(KEGG_TC != '-', KEGG_TC != '') %>%
    dplyr::select(gene_id, KEGG_TC) %>%
    tidyr::separate_rows(KEGG_TC, sep = ',') %>%
    mutate(TC = sapply(strsplit(str_trim(KEGG_TC), '\\.'),
                       function(v) paste(v[1:min(3, length(v))], collapse = '.'))) %>%
    filter(gene_id %in% res$gene_id) %>%
    left_join(tc_map, by = 'TC') %>%
    transmute(term = paste0('TC ', TC, ': ',
                            # ★ 名前が引けないものは term 名が図に出るので英語にする
                            #   （和文を入れると PNG 保存時に mbcsToSbcs 変換が失敗する。
                            #     実測: TC 9.B.25 が TCDB のリストに無く警告が出た）
                            ifelse(is.na(ファミリー名), '(no name in TCDB)', ファミリー名)),
              gene = gene_id) %>% distinct()

cat("\n■ Table 1: TERM2GENE の規模（セル5 の4種・セル7-6 と並べる）\n")
data.frame(
    source = c('KEGG pathway', 'COG', 'PFAM', 'CAZy', 'KEGG module（セル7-6）', 'TCDB（このセル）'),
    terms  = c(length(unique(t2g_kegg$term)), length(unique(t2g_cog$term)),
               length(unique(t2g_pfam$term)), length(unique(t2g_cazy$term)),
               NA, length(unique(t2g_tc$term))),
    遺伝子 = c(length(unique(t2g_kegg$gene)), length(unique(t2g_cog$gene)),
               length(unique(t2g_pfam$gene)), length(unique(t2g_cazy$gene)),
               NA, length(unique(t2g_tc$gene))),
    被覆pct = round(100 * c(length(unique(t2g_kegg$gene)), length(unique(t2g_cog$gene)),
                            length(unique(t2g_pfam$gene)), length(unique(t2g_cazy$gene)),
                            NA, length(unique(t2g_tc$gene))) / nrow(res), 1),
    check.names = FALSE)

cat("\n■ Table 2: 株ごとに TC が付いた遺伝子数\n")
res %>% filter(!marker) %>% mutate(tc = gene_id %in% t2g_tc$gene) %>%
    group_by(strain) %>%
    summarise(CDS = dplyr::n(), TCあり = sum(tc),
              率 = round(100 * sum(tc) / dplyr::n(), 1), .groups = 'drop') %>%
    as.data.frame()

# ===============================================================
# 3. ORA（セル6 とまったく同じ引数）
# ===============================================================
A_up_TC = enricher(upDEGs_A,   universe = universe_A, TERM2GENE = t2g_tc,
                   pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)
A_dn_TC = enricher(downDEGs_A, universe = universe_A, TERM2GENE = t2g_tc,
                   pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)
B_up_TC = enricher(upDEGs_B,   universe = universe_B, TERM2GENE = t2g_tc,
                   pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)
B_dn_TC = enricher(downDEGs_B, universe = universe_B, TERM2GENE = t2g_tc,
                   pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf)

ora_tc = rbind(
    as.data.frame(A_up_TC) %>% mutate(scheme = 'A', direction = 'up'),
    as.data.frame(A_dn_TC) %>% mutate(scheme = 'A', direction = 'down'),
    as.data.frame(B_up_TC) %>% mutate(scheme = 'B', direction = 'up'),
    as.data.frame(B_dn_TC) %>% mutate(scheme = 'B', direction = 'down')) %>%
    transmute(scheme, direction, term = ID, GeneRatio, BgRatio,
              pvalue, padj = p.adjust, Count, geneID) %>%
    mutate(TC = sub('^TC ([0-9]+\\.[A-Z]\\.[0-9]+).*', '\\1', term),
           class = tc_class[substr(TC, 1, 1)],
           fold = (Count / as.integer(sub('.*/', '', GeneRatio))) /
                  (as.integer(sub('/.*', '', BgRatio)) / as.integer(sub('.*/', '', BgRatio))))

cat(sprintf("\n■ ORA: 全 %d 行、padj<0.05 は %d 行\n", nrow(ora_tc), sum(ora_tc$padj < 0.05)))
cat(sprintf("  ★ 向きの内訳: (A) down %d / (A) up %d / (B) down %d / (B) up %d\n",
    sum(ora_tc$scheme=='A' & ora_tc$direction=='down' & ora_tc$padj<0.05),
    sum(ora_tc$scheme=='A' & ora_tc$direction=='up'   & ora_tc$padj<0.05),
    sum(ora_tc$scheme=='B' & ora_tc$direction=='down' & ora_tc$padj<0.05),
    sum(ora_tc$scheme=='B' & ora_tc$direction=='up'   & ora_tc$padj<0.05)))
cat("  ★ (A) up は有意 0 本。「無い」ことが結果なので明示して図は作らない\n")

cat("\n■ Table 3: ORA で有意になった TC ファミリー（全行）\n")
ora_tc %>% filter(padj < 0.05) %>%
    transmute(scheme = paste0('(', scheme, ')'), direction, class, term,
              GeneRatio, BgRatio, fold = round(fold, 2), padj = signif(padj, 2)) %>%
    arrange(scheme, direction, class) %>% as.data.frame()

# ===============================================================
# 4. GSEA（セル8 とまったく同じ引数）
# ===============================================================
# ★ セル7-6 と同じく、このセルを単独で実行できるよう geneList もここで作る
gl_A = res %>% filter(!is.na(stat_A), !marker) %>% arrange(desc(stat_A))
geneList_tc_A = gl_A$stat_A; names(geneList_tc_A) = gl_A$gene_id
gl_B = res %>% filter(!is.na(stat_B), !marker) %>% arrange(desc(stat_B))
geneList_tc_B = gl_B$stat_B; names(geneList_tc_B) = gl_B$gene_id

set.seed(1)
gsea_A_TC = suppressWarnings(GSEA(geneList_tc_A, TERM2GENE = t2g_tc, pvalueCutoff = 1,
                minGSSize = 1, maxGSSize = Inf, eps = 0, seed = TRUE, verbose = FALSE))
set.seed(1)
gsea_B_TC = suppressWarnings(GSEA(geneList_tc_B, TERM2GENE = t2g_tc, pvalueCutoff = 1,
                minGSSize = 1, maxGSSize = Inf, eps = 0, seed = TRUE, verbose = FALSE))
gsea_tc = rbind(as.data.frame(gsea_A_TC) %>% mutate(scheme = 'A'),
                as.data.frame(gsea_B_TC) %>% mutate(scheme = 'B')) %>%
    transmute(scheme, term = ID, setSize, NES, pvalue, padj = p.adjust)

cat(sprintf("\n■ GSEA: 全 %d 行、padj<0.05 は %d 行\n", nrow(gsea_tc), sum(gsea_tc$padj < 0.05)))
cat("  ★ GSEA は 1 本も有意にならない。図は作れない。\n")
cat("    参考までに NES の絶対値が大きい上位 10 行を出す（いずれも padj は有意でない）\n")
gsea_tc %>% arrange(desc(abs(NES))) %>% head(10) %>%
    transmute(scheme = paste0('(', scheme, ')'), term, setSize,
              NES = round(NES, 2), pvalue = signif(pvalue, 2), padj = signif(padj, 2)) %>%
    as.data.frame()

# ===============================================================
# 5. 図のための下ごしらえ
# ===============================================================
sigt = ora_tc %>% filter(padj < 0.05)

# ORA の株ごとの内訳（セル7-2・7-5・7-6 と同じ作り。棒の長さ = GeneRatio）
share_tc = sigt %>% dplyr::select(scheme, direction, term, class, geneID, GeneRatio) %>%
    tidyr::separate_rows(geneID, sep = '/') %>%
    left_join(res %>% dplyr::select(gene_id, strain), by = c('geneID' = 'gene_id')) %>%
    dplyr::count(scheme, direction, term, class, GeneRatio, strain, name = 'n_gene') %>%
    # ★ 当たり遺伝子が0本の株も行として残す（0 を明示する）
    tidyr::complete(tidyr::nesting(scheme, direction, term, class, GeneRatio),
                    strain = names(col_strain), fill = list(n_gene = 0)) %>%
    ungroup() %>%
    mutate(strain = factor(strain, levels = names(col_strain)),
           width = n_gene / as.integer(sub('.*/', '', GeneRatio))) %>%
    left_join(sigt %>% dplyr::select(scheme, direction, term, padj, fold),
              by = c('scheme', 'direction', 'term'))

# ★ 全 85 ファミリーの正味リード増減（検定ではなく記述統計）
#   有意かどうかに関係なく、そのファミリーに属する全遺伝子の norm_diff を足す
net_tc = t2g_tc %>%
    left_join(res %>% dplyr::select(gene_id, strain, marker, padj_A, padj_pooled,
                                    normA_diff, normB_diff), by = c('gene' = 'gene_id')) %>%
    filter(!marker, !is.na(padj_A)) %>%
    tidyr::pivot_longer(c(normA_diff, normB_diff), names_to = 's', values_to = 'diff') %>%
    mutate(scheme = ifelse(s == 'normA_diff', 'A', 'B')) %>%
    group_by(scheme, term, strain) %>%
    summarise(n_gene = dplyr::n(), net = sum(diff), .groups = 'drop') %>%
    mutate(strain = factor(strain, levels = names(col_strain)))

cat(sprintf("\n■ 正味リードを出すファミリー数: %d\n", length(unique(net_tc$term))))
cat("\n■ Table 4: 全 85 ファミリーの正味リード増減（scheme A、ファミリー内の全遺伝子）\n")
net_tc %>% filter(scheme == 'A') %>%
    dplyr::select(term, strain, n_gene, net) %>%
    tidyr::pivot_wider(names_from = strain, values_from = c(n_gene, net)) %>%
    mutate(across(starts_with('net'), round),
           正味合計 = round(rowSums(across(starts_with('net'))))) %>%
    arrange(正味合計) %>% as.data.frame()

# ── 見た目の約束（セル7-4〜7-6 と同一）────────────────────────
thm_tc = theme_bw() +
    theme(panel.grid.minor = element_blank(),
          strip.text.y.right = element_text(angle = 0, hjust = 0, size = 11, face = 'bold'),
          strip.background = element_rect(fill = 'grey92', colour = 'grey60'),
          legend.key.size = unit(0.9, 'lines'))
thm_tc_bar = thm_tc + theme(panel.grid.major.y = element_blank())
size_dot = scale_size_continuous(range = c(3, 9), name = 'gene count',
                                 guide = guide_legend(override.aes = list(fill = 'grey70')))
col_edge = scale_colour_manual(values = col_strain, name = 'strain',
    guide = guide_legend(override.aes = list(fill = 'white', linewidth = 1)))
fill_dot_dn = scale_fill_gradient(low = 'blue', high = 'white', limits = c(0, 0.05), name = 'p.adjust')
fill_dot_up = scale_fill_gradient(low = 'red',  high = 'white', limits = c(0, 0.05), name = 'p.adjust')
fill_bar_dn = scale_fill_gradient(low = '#8fb8f5', high = '#eaf1fd', limits = c(0, 0.05), name = 'p.adjust')
fill_bar_up = scale_fill_gradient(low = '#f08080', high = '#fdeaea', limits = c(0, 0.05), name = 'p.adjust')

# ★ TCDB のファミリー名は長い。y 軸ラベルは 56 文字で切る
TRUNC = 56

# ── 図7-7-1: (A) KO で低い × TCDB ファミリー（TC クラスでグループ化、x = fold enrichment） ──
# ★ 有意なのは TC 3.A.2 の 1 行だけ。少なさ自体が結果なのでそのまま出す
options(repr.plot.width = 13, repr.plot.height = 3.0)
dd = sigt %>% filter(scheme == 'A', direction == 'down') %>% arrange(class, fold) %>%
    mutate(term = factor(substr(term, 1, TRUNC), levels = substr(term, 1, TRUNC)))
ggplot(dd, aes(x = fold, y = term, size = Count, fill = padj)) +
    geom_vline(xintercept = 1, linetype = 2, colour = 'grey60') +
    geom_point(shape = 21, colour = 'grey30', stroke = 0.5) +
    facet_grid(class ~ ., scales = 'free_y', labeller = labeller(class = label_wrap_gen(20))) +
    fill_dot_dn + size_dot + thm_tc +
    labs(x = 'fold enrichment', y = NULL,
         title = '(A) pooled - down in KO - TCDB family, grouped by TC class')

# ── 図7-7-2: (B) KO で低い × TCDB ファミリー（TC クラスでグループ化、x = fold enrichment） ──
# ★ 有意なのは TC 3.A.2 の 1 行だけ。少なさ自体が結果なのでそのまま出す
options(repr.plot.width = 13, repr.plot.height = 3.0)
dd = sigt %>% filter(scheme == 'B', direction == 'down') %>% arrange(class, fold) %>%
    mutate(term = factor(substr(term, 1, TRUNC), levels = substr(term, 1, TRUNC)))
ggplot(dd, aes(x = fold, y = term, size = Count, fill = padj)) +
    geom_vline(xintercept = 1, linetype = 2, colour = 'grey60') +
    geom_point(shape = 21, colour = 'grey30', stroke = 0.5) +
    facet_grid(class ~ ., scales = 'free_y', labeller = labeller(class = label_wrap_gen(20))) +
    fill_dot_dn + size_dot + thm_tc +
    labs(x = 'fold enrichment', y = NULL,
         title = '(B) per-strain - down in KO - TCDB family, grouped by TC class')

# ── 図7-7-3: (B) KO で高い × TCDB ファミリー（TC クラスでグループ化、x = fold enrichment） ──
options(repr.plot.width = 13, repr.plot.height = 3.0)
dd = sigt %>% filter(scheme == 'B', direction == 'up') %>% arrange(class, fold) %>%
    mutate(term = factor(substr(term, 1, TRUNC), levels = substr(term, 1, TRUNC)))
ggplot(dd, aes(x = fold, y = term, size = Count, fill = padj)) +
    geom_vline(xintercept = 1, linetype = 2, colour = 'grey60') +
    geom_point(shape = 21, colour = 'grey30', stroke = 0.5) +
    facet_grid(class ~ ., scales = 'free_y', labeller = labeller(class = label_wrap_gen(20))) +
    fill_dot_up + size_dot + thm_tc +
    labs(x = 'fold enrichment', y = NULL,
         title = '(B) per-strain - up in KO - TCDB family, grouped by TC class')

# ── 図7-7-4: (A) KO で低い × TCDB ファミリー（株ごとの内訳） ──
options(repr.plot.width = 13, repr.plot.height = 3.0)
dd = share_tc %>% filter(scheme == 'A', direction == 'down')
ord = dd %>% distinct(term, class, fold) %>% arrange(class, fold) %>% pull(term)
dd = dd %>% mutate(term = factor(substr(term, 1, TRUNC), levels = substr(ord, 1, TRUNC)))
ggplot(dd, aes(x = width, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.0, width = 0.45, position = position_stack(reverse = TRUE)) +
    facet_grid(class ~ ., scales = 'free_y', labeller = labeller(class = label_wrap_gen(20))) +
    fill_bar_dn + col_edge +
    scale_x_continuous(expand = expansion(mult = c(0, 0.05))) + thm_tc_bar +
    labs(x = 'GeneRatio (stacked by strain)', y = NULL,
         title = '(A) pooled - down in KO - TCDB family, stacked by strain within TC class')

# ── 図7-7-5: (B) KO で低い × TCDB ファミリー（株ごとの内訳） ──
options(repr.plot.width = 13, repr.plot.height = 3.0)
dd = share_tc %>% filter(scheme == 'B', direction == 'down')
ord = dd %>% distinct(term, class, fold) %>% arrange(class, fold) %>% pull(term)
dd = dd %>% mutate(term = factor(substr(term, 1, TRUNC), levels = substr(ord, 1, TRUNC)))
ggplot(dd, aes(x = width, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.0, width = 0.45, position = position_stack(reverse = TRUE)) +
    facet_grid(class ~ ., scales = 'free_y', labeller = labeller(class = label_wrap_gen(20))) +
    fill_bar_dn + col_edge +
    scale_x_continuous(expand = expansion(mult = c(0, 0.05))) + thm_tc_bar +
    labs(x = 'GeneRatio (stacked by strain)', y = NULL,
         title = '(B) per-strain - down in KO - TCDB family, stacked by strain within TC class')

# ── 図7-7-6: (B) KO で高い × TCDB ファミリー（株ごとの内訳） ──
options(repr.plot.width = 13, repr.plot.height = 3.0)
dd = share_tc %>% filter(scheme == 'B', direction == 'up')
ord = dd %>% distinct(term, class, fold) %>% arrange(class, fold) %>% pull(term)
dd = dd %>% mutate(term = factor(substr(term, 1, TRUNC), levels = substr(ord, 1, TRUNC)))
ggplot(dd, aes(x = width, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.0, width = 0.45, position = position_stack(reverse = TRUE)) +
    facet_grid(class ~ ., scales = 'free_y', labeller = labeller(class = label_wrap_gen(20))) +
    fill_bar_up + col_edge +
    scale_x_continuous(expand = expansion(mult = c(0, 0.05))) + thm_tc_bar +
    labs(x = 'GeneRatio (stacked by strain)', y = NULL,
         title = '(B) per-strain - up in KO - TCDB family, stacked by strain within TC class')

# ★ ここからが TCDB の実質の所見。ORA / GSEA がほとんど何も返さないので、
#   検定ではなく「量」で見る。有意かどうかに関係なく全 85 ファミリーを出す。
#   ★ p 値は付けない（検定していないので）。色は株のみ
# ── 図7-7-7: (A) 全 85 TCDB ファミリーの正味リード増減（株で積み上げ） ──
# 並び順は (A) の正味合計の昇順で固定し、2枚で同じ順にする
options(repr.plot.width = 12, repr.plot.height = 13)
ord_net = net_tc %>% filter(scheme == 'A') %>% group_by(term) %>%
    summarise(s = sum(net), .groups = 'drop') %>% arrange(s) %>% pull(term)
dd = net_tc %>% filter(scheme == 'A') %>%
    mutate(term = factor(substr(term, 1, 60), levels = substr(ord_net, 1, 60)))
ggplot(dd, aes(x = net, y = term, fill = strain)) +
    geom_col(width = 0.6, colour = 'grey30', linewidth = 0.2) +
    geom_vline(xintercept = 0, colour = 'grey30') +
    scale_fill_manual(values = col_strain, name = 'strain') +
    scale_x_continuous(labels = scales::comma) +
    theme_bw() +
    theme(panel.grid.minor = element_blank(), panel.grid.major.y = element_blank(),
          axis.text.y = element_text(size = 7)) +
    labs(x = 'net normalized reads (KO - WT), all genes in the family', y = NULL,
         title = '(A) pooled - all 85 TCDB families - net transcript change (no test)')

# ── 図7-7-8: (B) 全 85 TCDB ファミリーの正味リード増減（株で積み上げ） ──
# 並び順は (A) の正味合計の昇順で固定し、2枚で同じ順にする
options(repr.plot.width = 12, repr.plot.height = 13)
ord_net = net_tc %>% filter(scheme == 'A') %>% group_by(term) %>%
    summarise(s = sum(net), .groups = 'drop') %>% arrange(s) %>% pull(term)
dd = net_tc %>% filter(scheme == 'B') %>%
    mutate(term = factor(substr(term, 1, 60), levels = substr(ord_net, 1, 60)))
ggplot(dd, aes(x = net, y = term, fill = strain)) +
    geom_col(width = 0.6, colour = 'grey30', linewidth = 0.2) +
    geom_vline(xintercept = 0, colour = 'grey30') +
    scale_fill_manual(values = col_strain, name = 'strain') +
    scale_x_continuous(labels = scales::comma) +
    theme_bw() +
    theme(panel.grid.minor = element_blank(), panel.grid.major.y = element_blank(),
          axis.text.y = element_text(size = 7)) +
    labs(x = 'net normalized reads (KO - WT), all genes in the family', y = NULL,
         title = '(B) per-strain - all 85 TCDB families - net transcript change (no test)')


■ TCDB ファミリー 2301 件を tcdb_families_261027.tsv から読んだ
  eggNOG 行 17341 / KEGG_TC あり 1496（8.6%）

■ Table 1: TERM2GENE の規模（セル5 の4種・セル7-6 と並べる）


source,terms,遺伝子,被覆pct
<chr>,<int>,<int>,<dbl>
KEGG pathway,151,2333,23.8
COG,21,8986,91.7
PFAM,2477,8576,87.6
CAZy,44,195,2.0
KEGG module（セル7-6）,NA,NA,NA
TCDB（このセル）,85,786,8.0



■ Table 2: 株ごとに TC が付いた遺伝子数


strain,CDS,TCあり,率
<chr>,<int>,<int>,<dbl>
Bth_St8,1829,120,6.6
Bth_VPI,1710,116,6.8
Bun_8492,3390,296,8.7
Odo_St23,2849,254,8.9
pME008,10,0,0.0



■ ORA: 全 158 行、padj<0.05 は 4 行
  ★ 向きの内訳: (A) down 1 / (A) up 0 / (B) down 1 / (B) up 2
  ★ (A) up は有意 0 本。「無い」ことが結果なので明示して図は作らない

■ Table 3: ORA で有意になった TC ファミリー（全行）


,scheme,direction,class,term,GeneRatio,BgRatio,fold,padj
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
"TC 3.A.2: The H+- or Na+-translocating F-type, V-type and A-type ATPase (F-ATPase) Superfamily1",(A),down,3 Primary active transporters,"TC 3.A.2: The H+- or Na+-translocating F-type, V-type and A-type ATPase (F-ATPase) Superfamily",18/162,42/775,2.05,0.033
"TC 3.A.2: The H+- or Na+-translocating F-type, V-type and A-type ATPase (F-ATPase) Superfamily3",(B),down,3 Primary active transporters,"TC 3.A.2: The H+- or Na+-translocating F-type, V-type and A-type ATPase (F-ATPase) Superfamily",19/170,42/777,2.07,0.020
TC 1.B.42: The Outer Membrane Lipopolysaccharide Export Porin (LPS-EP) Family1,(B),up,1 Channels and pores,TC 1.B.42: The Outer Membrane Lipopolysaccharide Export Porin (LPS-EP) Family,6/112,10/777,4.16,0.032
TC 3.B.1: The Na+-transporting Carboxylic Acid Decarboxylase (NaT-DC) Family2,(B),up,3 Primary active transporters,TC 3.B.1: The Na+-transporting Carboxylic Acid Decarboxylase (NaT-DC) Family,6/112,11/777,3.78,0.032


... (8 outputs left out)

plot without title

plot without title

plot without title

plot without title

### 考察: TCDB は検定として機能しない。ただし他では名前が付かない単位がある

#### 1. ★ 検定はほとんど何も返さない

**ORA 有意 4 行・GSEA 有意 0 行**（他ソースは 13〜52 行）。原因は**付与率 8.0%**
（786 遺伝子 / 85 ファミリー）。
**TCDB を「濃縮解析のソース」として他の4つと並べることはできない。**

★ さらに **`TC 3.A.2` が F型と V/A型を同じファミリーに入れている**ため、
`ko00190`（pathway）と同じ向きの混在が起きる。
**KEGG module なら `M00157`（F型、NES −2.31）と `M00159`（V/A型）に分けられるのに
TCDB では分けられない** → **この点では module の下位互換。**

#### 2. ★★ 実質の所見は正味リード（図7-7-7 / 7-7-8）— `TC 3.A.3` の減少は銅排出ポンプ

`TC 3.A.3` P型 ATPase の **−135,298（21遺伝子、×0.46）のうち 96% が銅排出ポンプ5本。**

| 遺伝子 | 株 | name | LFC_A | padj | WT → KO |
|---|---|---|---|---|---|
| `BT_RS05490` | Bth_VPI | **copA** | **−1.83** | 4.6e-15 | 53,636 → 14,593 |
| `NQ510_RS12055` | Bun_8492 | **copA** | −0.79 | 1.5e-03 | **80,155 → 44,412** |
| `CE91St8_34640` | Bth_St8 | **copA** | **−2.30** | 6.4e-09 | 41,152 → 7,648 |
| `NQ510_RS07260` | Bun_8492 | **actP** | −1.33 | 3.7e-08 | 31,426 → 11,975 |
| `CE91St23_13820` | Odo_St23 | **actP** | −1.49 | 3.3e-08 | 2,700 → 919 |

**4株すべてに1本以上あり、全部下がっている。**
一方**同じファミリーの K+ ポンプ（`kdpABC`）は n.s. で、`Bun_8492` ではむしろ上昇**（`kdpA` +2.06）。
→ **「P型 ATPase が下がった」ではなく「重金属排出ポンプだけが下がった」が正確。**

★ **この群は PFAM の `HMA`（9遺伝子、(A) padj 2.7e-03）でも拾えている**ので、
**TCDB を入れなくても所見自体は失われない**（名前が「P型 ATPase」ではなく「HMA ドメイン」になるだけ）。
KEGG では **`ko04016`「植物の MAPK シグナル」**として出てくる（セル6 考察 3）。

#### 3. ★ 外膜では「中身」と「作る装置」で向きが逆

|- | 正味（(A)） | 比 |
|---|---|---|
| `TC 1.B.6` OmpA-OmpF ポーリン | −11,660 | 0.65 |
| **`TC 1.B.33` BAM（外膜タンパクの挿入装置）** | **+12,092** | **2.42** |
| **`TC 1.B.42` Lpt（LPS 排出）** | **+4,832** | **1.72** |
| `TC 3.A.5` Sec（分泌装置） | +23,928 | 1.55 |

**外膜タンパク質そのものは減るのに、外膜に挿入する装置（BamA `yaeT` +1.48 / BamD `yfiO` +1.18）と
LPS を運び出す装置（`lptB` +1.38）は増える。**
`eptA` は lipid A を修飾する酵素なので `Lpt` の上昇は関係しうるが、**本データから因果は言えない。**

#### 4. ★ 読み違えやすいもの

- **`TC 1.A.33` "Cation Channel-forming Heat Shock Protein-70"（−87,967）の実体は `dnaK`**
  （`NQ510_RS17410` −0.92、128,235 → 61,573 ほか）。**輸送体ではない。**
  PFAM の `Clp_N`/`DnaJ_C` が down だったのと同じ現象を別ラベルで拾っている
- **`TC 3.A.1` ABC スーパーファミリー（136遺伝子）は差し引きほぼゼロ**（−17,704、×0.95）。
  **有意に低下 34 本・上昇 20 本が打ち消し合っている**ので、ファミリーで括ると消える

#### 5. SusC / SusD（PUL）は株で向きが逆

`TC 8.A.46` SusD（153遺伝子）は4株合計では ×1.07 だが、
**`Bun_8492` ×1.47（有意高 17 本）に対し `Bth_St8` ×0.69 / `Bth_VPI` ×0.89 / `Odo_St23` ×0.73。**
`TC 1.B.14` SusC も `Bun_8492` ×1.81。
→ **CLAUDE.md §8 の「`Bun_8492` では CPS ではなく PUL が KO で上昇」と一致する。**
ただし `TC 1.B.14` は 21 遺伝子しかなく（*Bacteroides* の SusC は実際にはもっと多い）、
**PUL をきちんと見るには座位単位の検出が要る。TCDB では代用できない。**


## セル8: GSEA（gene set enrichment analysis）

### ★ 過去プロジェクトは GSEA を実行していない

`5_Enrichment_pathway_R.ipynb` セル30 で `geneList`（名前付き数値ベクトルを降順に並べたもの）を
作ってはいるが、**それを使ったのは `enricher` で、`GSEA` / `gseKEGG` / `fgsea` の呼び出しは
全 notebook に1つも無い**（実測: `fgsea` 0 件 / `gseKEGG` 0 件 / `GSEA(` 0 件）。

→ **`geneList` の作り方だけ過去notebook に合わせ、GSEA 自体は新規に書く。**

### なぜ ORA に加えて GSEA をやるか

ORA は **`padj < 0.05` という線引きに依存する。**

- 境界のすぐ下にある遺伝子は全部捨てられる
- 経路全体がわずかに同じ方向へ動いているような**弱く広い変化は拾えない**

GSEA は **全遺伝子を統計量で順位付けし、閾値を切らない。**
**同じ `TERM2GENE` を使う**ので ORA と term 名で直接突き合わせられる。

### ランク付けに使う統計量: Wald 統計量

`LFC` ではなく **`stat`（= `LFC / SE`）** を使う。
`LFC` は低発現遺伝子で大きく振れるが、`stat` は**ばらつきを織り込んだ順位**になる。
符号は KO vs WT なので **上位 = KO で高い、下位 = KO で低い。**

|-| geneList_A | geneList_B |
|---|---|---|
| 遺伝子数 | 9,788 | 9,778 |
| 最大（KO で高い） | 13.89 | 16.81 |
| 最小（KO で低い） | −11.58 | −12.87 |

### 引数は ORA（セル6）と揃える

| 引数 | 値 | 理由 |
|---|---|---|
| `pvalueCutoff` | 1 | 全 term を返させ、有意判定は `padj < 0.05` で自分でやる |
| `minGSSize` / `maxGSSize` | **1 / `Inf`** | セル6 と同じく足切りをしない |
| `eps` | **0** | 既定 `1e-10` だと p 値が頭打ちになる（`ko03010_Ribosome` は 1.8e-10） |
| `seed` + `set.seed(1)` | — | GSEA は乱択を含むので再現性のため固定 |

**足切り無しが GSEA でも安全なことは実測で確認した**: `setSize` 1 の term 12 本・2 の term 8 本は
**すべて有意にならず**、`setSize < 5` で有意になるのは `ko00590_Arachidonic acid metabolism` 1 本だけ。

### ★ タイ（同順位）の警告について

`fgsea` は「順位が同じ遺伝子の並びは任意になる（リストの 5.4%）」という警告を 8 回とも出す。
タイの実態を調べたところ、**2-3 遺伝子の小さな塊が統計量の全域（−5.8 〜 7.2）に散在するだけで、
最も多い値でも 4 遺伝子、0 ちょうどは 0 個。**順位への影響は小さいと判断し、
`suppressWarnings()` で抑えた（他の警告に気づけるよう呼び出しごとに囲んである）。

### 8 回を1回ずつ名前を付けて回す

2スキーム × 4ソース。名前は `gsea_A_KEGG` のように `gsea_<スキーム>_<ソース>`。
所要時間は 8 回で約 44 秒。

In [32]:
# ── GSEA（gene set enrichment analysis）──────────────────────────
# ★ 過去プロジェクトは GSEA を実行していない。
#   5_Enrichment_pathway_R.ipynb セル30 で geneList（名前付き数値ベクトルを
#   降順に並べたもの）を作ってはいるが、それを使ったのは enricher で、
#   GSEA / gseKEGG / fgsea の呼び出しは全 notebook に1つも無い。
#   → geneList の作り方だけ過去notebook に合わせ、GSEA は新規に書く。
#
# なぜ ORA に加えて GSEA をやるか（セル6 考察 10）
#   ORA は padj < 0.05 という線引きに依存する。
#   ・境界のすぐ下にある遺伝子は全部捨てられる
#   ・経路全体がわずかに同じ方向へ動いているような弱く広い変化は拾えない
#   GSEA は ★ 全遺伝子を統計量で順位付けし、閾値を切らない。
#   同じ TERM2GENE を使うので ORA と直接突き合わせられる。
#
# ランク付けに使う統計量: Wald 統計量（CLAUDE.md §7 ステージ6）
#   LFC ではなく stat を使う。LFC は低発現遺伝子で大きく振れるが、
#   stat = LFC / SE なので ばらつきを織り込んだ順位になる。
#   符号は KO vs WT なので、★ 上位 = KO で高い、下位 = KO で低い。
library(clusterProfiler)
options(repr.matrix.max.rows = 500, repr.matrix.max.cols = 50)

# ===============================================================
# 1. geneList を作る（過去notebook セル30 と同じ作法）
# ===============================================================
# 名前付き数値ベクトルにして降順に並べる。マーカーは ORA と同じ理由で外す
gl_A = res %>% filter(!is.na(stat_A), !marker) %>% arrange(desc(stat_A))
geneList_A = gl_A$stat_A
names(geneList_A) = gl_A$gene_id

gl_B = res %>% filter(!is.na(stat_B), !marker) %>% arrange(desc(stat_B))
geneList_B = gl_B$stat_B
names(geneList_B) = gl_B$gene_id

cat("■ Table 1: ランク付けに使う geneList\n")
data.frame(
    scheme = c('(A) 全株プール', '(B) 株ごと'),
    object = c('geneList_A', 'geneList_B'),
    遺伝子数 = c(length(geneList_A), length(geneList_B)),
    最大_KOで高い = round(c(max(geneList_A), max(geneList_B)), 2),
    最小_KOで低い = round(c(min(geneList_A), min(geneList_B)), 2),
    同値の遺伝子数 = c(sum(duplicated(geneList_A) | duplicated(geneList_A, fromLast = TRUE)),
                       sum(duplicated(geneList_B) | duplicated(geneList_B, fromLast = TRUE))),
    check.names = FALSE)

# ★ 同値（タイ）について
#   fgsea は「順位が同じ遺伝子の並びは任意になる」と警告を出す。
#   本データのタイは 2-3 遺伝子の小さな塊が統計量の全域に散っているだけで
#   （最も多い値でも 4 遺伝子）、0 付近に大きな塊は無い。順位への影響は小さい
cat(sprintf("\n  geneList_A の同値: 最も多い値でも %d 遺伝子。0 ちょうどは %d 個\n",
    max(table(geneList_A)), sum(geneList_A == 0)))

# ===============================================================
# 2. GSEA を1回ずつ回す（2スキーム × 4ソース = 8 回）
# ===============================================================
# 引数は ORA（セル6）と揃える
#   pvalueCutoff = 1          全 term を返させ、有意判定は自分でやる
#   minGSSize = 1, maxGSSize = Inf
#     足切りをしない（セル6 と同じ方針）。実測で setSize 1-2 の term は
#     有意 0 件、setSize<5 で有意になるのは 1 本だけなので安全
#   eps = 0                   p 値の下限を切らない（既定 1e-10 だと頭打ちになる）
#   seed = TRUE + set.seed()  GSEA は乱択を含むので再現性のため固定する
#
# ★ suppressWarnings について
#   fgsea は「順位が同じ遺伝子の並びは任意になる（リストの 5.4%）」という
#   警告を 8 回とも同じ内容で出す。タイの実態は Table 1 で確認済み
#   （2-3 遺伝子の小さな塊が全域に散在、最多でも 4 遺伝子）なので、
#   出力が埋まらないよう抑える。他の警告が出たら気づけるよう呼び出しごとに囲む
set.seed(1)
gsea_A_KEGG = suppressWarnings(GSEA(geneList_A, TERM2GENE = t2g_kegg, pvalueCutoff = 1,
                   minGSSize = 1, maxGSSize = Inf, eps = 0, seed = TRUE, verbose = FALSE))
set.seed(1)
gsea_A_COG  = suppressWarnings(GSEA(geneList_A, TERM2GENE = t2g_cog,  pvalueCutoff = 1,
                   minGSSize = 1, maxGSSize = Inf, eps = 0, seed = TRUE, verbose = FALSE))
set.seed(1)
gsea_A_PFAM = suppressWarnings(GSEA(geneList_A, TERM2GENE = t2g_pfam, pvalueCutoff = 1,
                   minGSSize = 1, maxGSSize = Inf, eps = 0, seed = TRUE, verbose = FALSE))
set.seed(1)
gsea_A_CAZy = suppressWarnings(GSEA(geneList_A, TERM2GENE = t2g_cazy, pvalueCutoff = 1,
                   minGSSize = 1, maxGSSize = Inf, eps = 0, seed = TRUE, verbose = FALSE))

set.seed(1)
gsea_B_KEGG = suppressWarnings(GSEA(geneList_B, TERM2GENE = t2g_kegg, pvalueCutoff = 1,
                   minGSSize = 1, maxGSSize = Inf, eps = 0, seed = TRUE, verbose = FALSE))
set.seed(1)
gsea_B_COG  = suppressWarnings(GSEA(geneList_B, TERM2GENE = t2g_cog,  pvalueCutoff = 1,
                   minGSSize = 1, maxGSSize = Inf, eps = 0, seed = TRUE, verbose = FALSE))
set.seed(1)
gsea_B_PFAM = suppressWarnings(GSEA(geneList_B, TERM2GENE = t2g_pfam, pvalueCutoff = 1,
                   minGSSize = 1, maxGSSize = Inf, eps = 0, seed = TRUE, verbose = FALSE))
set.seed(1)
gsea_B_CAZy = suppressWarnings(GSEA(geneList_B, TERM2GENE = t2g_cazy, pvalueCutoff = 1,
                   minGSSize = 1, maxGSSize = Inf, eps = 0, seed = TRUE, verbose = FALSE))

# ===============================================================
# 3. 8 個の結果を1つの表に積む
# ===============================================================
gsea = rbind(
    as.data.frame(gsea_A_KEGG) %>% mutate(scheme = 'A', source = 'KEGG'),
    as.data.frame(gsea_A_COG ) %>% mutate(scheme = 'A', source = 'COG' ),
    as.data.frame(gsea_A_PFAM) %>% mutate(scheme = 'A', source = 'PFAM'),
    as.data.frame(gsea_A_CAZy) %>% mutate(scheme = 'A', source = 'CAZy'),
    as.data.frame(gsea_B_KEGG) %>% mutate(scheme = 'B', source = 'KEGG'),
    as.data.frame(gsea_B_COG ) %>% mutate(scheme = 'B', source = 'COG' ),
    as.data.frame(gsea_B_PFAM) %>% mutate(scheme = 'B', source = 'PFAM'),
    as.data.frame(gsea_B_CAZy) %>% mutate(scheme = 'B', source = 'CAZy')) %>%
    tibble::remove_rownames()

# NES の符号が向きを表す。ORA の up/down に対応づけておく
gsea = gsea %>%
    mutate(direction = ifelse(NES > 0, 'up', 'down')) %>%
    dplyr::select(scheme, direction, source, ID, term = Description, setSize,
                  ES = enrichmentScore, NES, pvalue, padj = p.adjust,
                  core = core_enrichment) %>%
    arrange(scheme, source, padj)

cat(sprintf("\n■ gsea: %d 行（8 回の結果を積んだもの）\n", nrow(gsea)))

# ===============================================================
# 4. 結果を見る
# ===============================================================
cat("\n■ Table 2: 有意 term 数 / 検定した term 数（padj<0.05）\n")
data.frame(
    source = c('KEGG', 'COG', 'PFAM', 'CAZy'),
    A = c(sprintf('%d / %d', sum(gsea_A_KEGG@result$p.adjust < 0.05), nrow(gsea_A_KEGG@result)),
          sprintf('%d / %d', sum(gsea_A_COG @result$p.adjust < 0.05), nrow(gsea_A_COG @result)),
          sprintf('%d / %d', sum(gsea_A_PFAM@result$p.adjust < 0.05), nrow(gsea_A_PFAM@result)),
          sprintf('%d / %d', sum(gsea_A_CAZy@result$p.adjust < 0.05), nrow(gsea_A_CAZy@result))),
    B = c(sprintf('%d / %d', sum(gsea_B_KEGG@result$p.adjust < 0.05), nrow(gsea_B_KEGG@result)),
          sprintf('%d / %d', sum(gsea_B_COG @result$p.adjust < 0.05), nrow(gsea_B_COG @result)),
          sprintf('%d / %d', sum(gsea_B_PFAM@result$p.adjust < 0.05), nrow(gsea_B_PFAM@result)),
          sprintf('%d / %d', sum(gsea_B_CAZy@result$p.adjust < 0.05), nrow(gsea_B_CAZy@result))),
    check.names = FALSE)

# ── 有意 term の core enrichment 遺伝子を株ごとに数える ───────────
# core_enrichment = その term の中で ES を押し上げた遺伝子（leading edge）。
# セル6 Table 4 と同じ要領で株の内訳を出す
core_strain = gsea %>% filter(padj < 0.05) %>%
    dplyr::select(scheme, direction, source, term, core) %>%
    tidyr::separate_rows(core, sep = '/') %>%
    left_join(res %>% dplyr::select(gene_id, strain), by = c('core' = 'gene_id')) %>%
    dplyr::count(scheme, direction, source, term, strain) %>%
    tidyr::pivot_wider(names_from = strain, values_from = n, values_fill = 0)

cat("\n■ Table 3: 有意になった term（padj<0.05）すべて ＋ core enrichment の株ごとの内訳\n")
gsea %>% filter(padj < 0.05) %>%
    left_join(core_strain, by = c('scheme', 'direction', 'source', 'term')) %>%
    mutate(scheme = paste0('(', scheme, ')'), NES = round(NES, 2), padj = signif(padj, 2),
           core_n = Bth_VPI + Bth_St8 + Bun_8492 + Odo_St23) %>%
    dplyr::select(scheme, direction, source, term, setSize, core_n, NES, padj,
                  Bth_VPI, Bth_St8, Bun_8492, Odo_St23) %>%
    arrange(scheme, source, desc(abs(NES))) %>% as.data.frame()

# ===============================================================
# 5. ORA（セル6）と突き合わせる
# ===============================================================
# 同じ TERM2GENE を使っているので term 名で直接つながる
cmp = full_join(
    ora  %>% filter(padj < 0.05) %>% transmute(scheme, source, term,
                   ORA_direction = direction, ORA_padj = padj),
    gsea %>% filter(padj < 0.05) %>% transmute(scheme, source, term,
                   GSEA_direction = direction, GSEA_NES = NES, GSEA_padj = padj),
    by = c('scheme', 'source', 'term'))

cat("\n■ Table 4: ORA と GSEA の一致\n")
cmp %>%
    mutate(判定 = case_when(
        is.na(GSEA_direction) ~ 'ORA のみ',
        is.na(ORA_direction)  ~ 'GSEA のみ',
        ORA_direction == GSEA_direction ~ '両方・向きも一致',
        TRUE ~ '★ 両方だが向きが逆')) %>%
    dplyr::count(scheme, source, 判定) %>%
    tidyr::pivot_wider(names_from = 判定, values_from = n, values_fill = 0) %>%
    as.data.frame()

cat("\n■ Table 5: GSEA でだけ有意になった term（ORA の閾値では拾えなかったもの）\n")
cmp %>% filter(is.na(ORA_direction)) %>%
    mutate(GSEA_NES = round(GSEA_NES, 2), GSEA_padj = signif(GSEA_padj, 2)) %>%
    dplyr::select(scheme, source, term, GSEA_direction, GSEA_NES, GSEA_padj) %>%
    arrange(scheme, source, GSEA_padj) %>% as.data.frame()

cat("\n■ Table 6: ORA でだけ有意になった term\n")
cmp %>% filter(is.na(GSEA_direction)) %>%
    mutate(ORA_padj = signif(ORA_padj, 2)) %>%
    dplyr::select(scheme, source, term, ORA_direction, ORA_padj) %>%
    arrange(scheme, source, ORA_padj) %>% as.data.frame()

# ===============================================================
# 6. 保存
# ===============================================================
write.csv(gsea, file.path(path_pw, 'gsea_all.csv'), row.names = FALSE)
invisible(file.copy(file.path(path_pw, 'gsea_all.csv'),
          file.path(path_project, 'refqc', 'stage6_gsea_all.csv'), overwrite = TRUE))
cat(sprintf("\n保存: %s/gsea_all.csv（%d 行、core 列に leading edge 遺伝子）\n",
            path_pw, nrow(gsea)))
cat("project 複製: refqc/stage6_gsea_all.csv\n")


■ Table 1: ランク付けに使う geneList


scheme,object,遺伝子数,最大_KOで高い,最小_KOで低い,同値の遺伝子数
<chr>,<chr>,<int>,<dbl>,<dbl>,<int>
(A) 全株プール,geneList_A,9788,13.89,-11.58,1019
(B) 株ごと,geneList_B,9778,16.81,-12.87,1043



  geneList_A の同値: 最も多い値でも 4 遺伝子。0 ちょうどは 0 個

■ gsea: 5386 行（8 回の結果を積んだもの）

■ Table 2: 有意 term 数 / 検定した term 数（padj<0.05）


source,A,B
<chr>,<chr>,<chr>
KEGG,21 / 151,25 / 151
COG,8 / 21,5 / 21
PFAM,16 / 2477,26 / 2477
CAZy,0 / 44,0 / 44



■ Table 3: 有意になった term（padj<0.05）すべて ＋ core enrichment の株ごとの内訳


scheme,direction,source,term,setSize,core_n,NES,padj,Bth_VPI,Bth_St8,Bun_8492,Odo_St23
<chr>,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<int>,<int>,<int>,<int>
(A),up,COG,"COG J: Translation, ribosomal structure and biogenesis",411,165,2.16,2.3e-15,11,19,118,17
(A),up,COG,COG H: Coenzyme transport and metabolism,394,150,1.75,7.3e-07,15,14,111,10
(A),down,COG,"COG O: Posttranslational modification, protein turnover, chaperones",295,98,-1.61,1.6e-04,20,18,20,40
(A),down,COG,COG N: Cell motility,53,16,-1.52,2.3e-02,8,7,0,1
(A),up,COG,COG F: Nucleotide transport and metabolism,260,63,1.40,1.5e-02,4,6,45,8
(A),down,COG,COG C: Energy production and conversion,530,135,-1.38,2.4e-03,32,32,21,50
(A),down,COG,COG M: Cell wall/membrane/envelope biogenesis,937,192,-1.37,1.9e-04,56,56,26,54
(A),down,COG,COG P: Inorganic ion transport and metabolism,718,161,-1.28,9.5e-03,41,40,24,56
(A),up,KEGG,ko03010_Ribosome,112,50,2.45,1.8e-10,1,1,39,9


... (3 outputs left out)

scheme,source,term,GSEA_direction,GSEA_NES,GSEA_padj
<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
A,COG,COG P: Inorganic ion transport and metabolism,down,-1.28,9.5e-03
A,COG,COG N: Cell motility,down,-1.52,2.3e-02
A,KEGG,ko00970_Aminoacyl-tRNA biosynthesis,up,2.12,8.5e-05
A,KEGG,ko00983_Drug metabolism - other enzymes,up,1.75,2.8e-02
A,KEGG,ko00230_Purine metabolism,up,1.51,3.1e-02
A,KEGG,ko00511_Other glycan degradation,up,1.60,3.2e-02
A,KEGG,ko00590_Arachidonic acid metabolism,down,-1.56,3.8e-02
A,KEGG,ko04142_Lysosome biogenesis,up,1.69,4.8e-02
A,KEGG,ko04112_Cell cycle - Caulobacter,up,1.67,4.9e-02



■ Table 6: ORA でだけ有意になった term


scheme,source,term,ORA_direction,ORA_padj
<chr>,<chr>,<chr>,<chr>,<dbl>
A,COG,COG I: Lipid transport and metabolism,up,0.0200
A,KEGG,ko00190_Oxidative phosphorylation,down,0.0048
A,KEGG,ko01051_Biosynthesis of ansamycins,down,0.0082
A,KEGG,ko00720_Other carbon fixation pathways,down,0.0200
A,KEGG,ko00710_Carbon fixation by Calvin cycle,down,0.0200
A,KEGG,ko00310_Lysine degradation,down,0.0300
A,PFAM,PFAM:UPF0051,down,0.0027
A,PFAM,PFAM:DUF3987,down,0.0290
A,PFAM,PFAM:P_gingi_FimA,down,0.0290



保存: /mnt/scratch/tanoue/261016_Xiaoxi_eptAoligo_2wFeces_Metatranscriptome/pathway/gsea_all.csv（5386 行、core 列に leading edge 遺伝子）
project 複製: refqc/stage6_gsea_all.csv


### 考察: ORA と1本も矛盾せず、GSEA でしか見えないものが 35 本あった

#### 1. ★ 向きが逆になった term は 0 本

ORA と GSEA の両方で有意な term は **(A) 30 本 / (B) 36 本**あり、**向きが食い違うものは1本も無い。**
原理がまったく違う2つの方法で同じ向きの結論になったので、**セル6 の所見は線引きの産物ではない。**

| scheme | source | 両方一致 | GSEA のみ | ORA のみ |
|---|---|---|---|---|
| (A) | KEGG / COG / PFAM | 14 / 6 / 10 | 7 / 2 / 6 | 5 / 1 / 12 |
| (B) | KEGG / COG / PFAM | 18 / 4 / 14 | 7 / 1 / 12 | 2 / 2 / 16 |

GSEA 上位も ORA と同じ顔ぶれ（`ko03010_Ribosome` NES +2.45、`ko00195` −2.28、
`ko00660` −2.16、`ko00450` −2.14、`ko00970` +2.12、`ko00480` −2.09、`ko00030` −2.08）。
**胆汁酸 `ko00121` も NES −1.92 / padj 1.9e-03 で確認された。**

#### 2. ★★ GSEA でだけ見えた最大の所見 — プロファージ／インテグラーゼの低下

| term | scheme | NES | padj | VPI / St8 / Bun / Odo |
|---|---|---|---|---|
| `PFAM:Phage_integrase` | (A) / (B) | −1.89 / −1.98 | 0.020 / 0.013 | 12 / **13** / 4 / 3 |
| `PFAM:Phage_int_SAM_5` | (A) / (B) | −1.96 / −1.99 | 0.020 / 0.013 | 9 / **12** / 4 / 1 |

中身は **tyrosine recombinase / integrase / transposase**（`Bth_St8` で −0.59〜−2.50）。

★ **CPS locus switching と直結する可能性がある。***Bacteroides* の莢膜多糖座位は
**チロシンリコンビナーゼがプロモーターを反転させて ON/OFF を切り替える**（相変異）。
ステージ5 で PC1（42.96%）の正体とした **`Bth_St8` の CPS locus switching と同じ株で
リコンビナーゼ群が一斉に下がっている。** ORA では個々のドメインが閾値に届かなかった。

関連して **`PFAM:Mfa_like_1`（(B) down、NES −1.90）** も出た。Mfa は *Bacteroidetes* の線毛で、
ORA の `PFAM:P_gingi_FimA`（Odo 特異的）と合わせて**表層構造の低下**という一貫した像になる。

#### 3. GSEA でだけ見えたその他

`ko00970_Aminoacyl-tRNA`（(A) up）、`ko03010_Ribosome`（(B) up、NES +2.44 / padj 3.4e-09 で
**ORA (B) では拾えなかった**）、`ko00230_Purine metabolism`（(A)(B) とも up）、
`ko00740_Riboflavin` ＋ `PFAM:DMRL_synthase`（(B) up、**経路とドメインの両方で一致**）、
`ko00010_Glycolysis`（(B) down）、`PFAM:Citrate_synt`/`Aconitase_C`（(B) down）、
`COG N: Cell motility`（(A) down）。

#### 4. ORA でだけ有意だったもの — 小さい term と `ko00190`

ORA のみの term は**ほとんどが 4-6 遺伝子の小さい PFAM**（`PMSR`・`SelR`・`6PGD`・`Fer4_19`・
`UPF0051`）で、**GSEA の検出力が落ちるため。矛盾ではない。**

★ **例外は `ko00190_Oxidative phosphorylation`**（(A) GSEA padj **0.105** / ORA padj 0.0048）。
**`setSize` 81 に対し down DEG が 27 本しかないので残り 54 本が薄めて NES が伸びない**（−1.46）。
**ATP synthase 本体（`ko00195`、setSize 18）は NES −2.28 と強い** →
**低下しているのは酸化的リン酸化の一部（ATP synthase）であって経路全体ではない。**
（セル11 でマップ上に描いて確定させる）

#### 5. CAZy は GSEA でも 0 本

`setSize` 中央値 3.5、`universe` 側の該当が 192 遺伝子しかなく、**ORA でも GSEA でも検出力が無い。**

#### 保存

`pathway/gsea_all.csv`（5,386 行、`core` 列に leading edge）→ project の `refqc/stage6_gsea_all.csv`


## セル9: GSEA の一覧図

セル7（`dotplot`）とセル7-2（株内訳の横棒）に対応する GSEA 版。**2つの形式を両方作る。**

| 図 | 形式 | 読むもの |
|---|---|---|
| 図9-1 〜 9-6 | `dotplot` | NES の大きさ・`setSize`・`padj` |
| 図9-2-1 〜 9-2-6 | 横棒（株で区分） | **どの株の遺伝子が ES を押し上げたか** |

### ★ 違い1: x 軸は `NES`、size は `setSize` を明示する

`dotplot` は `gseaResult` にも使えるが、**既定は `x = "GeneRatio"` / `size = "Count"`** で、
**ORA と同じ列名なのに中身が違う**（`core_enrichment` から計算される量）。
`Count` は `@result` に無い内部計算列なので**目盛りも合わせられない。**

### ★ 違い2: NES = 0 を境に上下2段に分ける

色は**セル7 と同じ「KO で低い = 青、KO で高い = 赤」**だが、**NES は 0 をまたぐので
1枚に赤と青の2つの `fill` スケールは置けない。**
→ **上段（up、赤）／下段（down、青）に分けて `patchwork` で縦に積む。**

**二重基準にはならない。**どちらの凡例も `limits = c(0, 0.05)` で**数値の範囲は同一**、
**違うのは色相だけで、色相は向き（どちらのパネルか）を表す。**
パネルの高さは term 数に比例させるので上下で行の間隔が揃う。

### ★ x 軸は上下で揃える

同じ `limits` を渡すので **0 の位置と目盛りが上下でぴったり一致する。**
範囲は **|NES| の最大を 0.5 刻みに切り上げ、0 を中心に対称**にする（本データは 6 組み合わせすべて
`-2.5 〜 2.5` に収まった）。0 に破線を引いてあるので上段が右・下段が左に伸びるのが一目で分かる。

| scheme | source | down | up |
|---|---|---|---|
| (A) | KEGG / COG / PFAM | 14 / 5 / 16 | 7 / 3 / **0** |
| (B) | KEGG / COG / PFAM | 19 / 3 / 22 | 6 / 2 / 4 |

**(A) PFAM は up が 0 なので down のみ。CAZy は有意 0 なので作らない。**

### ★ 違い3: `dotplot` は濃い赤/青、横棒は薄い赤/青

セル7-2 で経験したとおり、**濃い塗りの上では株の縁が読めない**
（down の青に `Bth_VPI` の青が同化し、up の赤に `Bth_St8` の橙が埋もれる）。
→ `dotplot` は `blue`/`red`、横棒は `#8fb8f5`/`#f08080`。

### 横棒の区分

棒の長さ = **NES**、区分 = **`core_enrichment`（leading edge）遺伝子の株ごとの割合 × NES**。
足すと NES になり、**NES が負なら棒は左へ伸びる。**
`position_stack(reverse = TRUE)` で 0 に近い側から `Bth_VPI` → `Bth_St8` → `Bun_8492` → `Odo_St23`。


In [82]:
# ── GSEA の一覧図 ───────────────────────────────────────────────
# セル7（ORA の dotplot）とセル7-2（株内訳の横棒）に対応する GSEA 版。
#
# ★ セル7 との違い 1: x 軸は GeneRatio ではなく NES
#   dotplot は gseaResult にも使えるが、既定は x = "GeneRatio", size = "Count" で、
#   これは ORA と同じ列名なのに ★ 中身が違う（GSEA では core_enrichment から
#   計算される量）。誤読を招くので x = 'NES' を明示する。
#
# ★ セル7 との違い 2: up / down を1枚にまとめる
#   NES の符号が向きを表す（正 = KO で高い、負 = KO で低い）ので、
#   0 の右と左に分かれて並ぶ。セル7 は向きごとに図を分けていたが、
#   ここは scheme × source の 6 枚で済む（CAZy は有意 0 なので作らない）。
#
# ★ セル7 との違い 3: 色は padj だけを表す
#   セル7 の赤/青は「向き」を表していた。この図は向きを x 軸で表すので、
#   色に向きの意味を持たせると二重になって紛らわしい。
#   → 灰色系（濃い = padj が小さい）にする。
#     株の色（青/橙/緑/紫）とも衝突しないので、横棒の縁が最も読みやすくなる
library(enrichplot)
library(ggplot2)
library(patchwork)

lab60 = function(x) stringr::str_wrap(x, width = 60)

# 株の色はステージ5 / セル7-2 と同じ
col_strain = c(Bth_VPI = '#2a78d6', Bth_St8 = '#eb6834', Bun_8492 = '#3f9e5f',
               Odo_St23 = '#9b59b6')
col_edge = scale_colour_manual(values = col_strain, name = 'strain',
    guide = guide_legend(override.aes = list(fill = 'white', linewidth = 1)))

thm_bar = theme_bw() +
    theme(panel.grid.major.y = element_blank(),
          panel.grid.minor = element_blank(),
          legend.key.size = unit(0.9, 'lines'))

# ===============================================================
# 0. 横棒グラフ用のデータ（core enrichment の株ごとの内訳）
# ===============================================================
# 棒の長さ = NES。区分 = core enrichment 遺伝子の株ごとの割合 × NES。
# 足すと NES になる。NES が負なら棒は左へ伸びる
bar_gsea = gsea %>% filter(padj < 0.05) %>%
    left_join(core_strain, by = c('scheme', 'direction', 'source', 'term')) %>%
    tidyr::pivot_longer(c(Bth_VPI, Bth_St8, Bun_8492, Odo_St23),
                        names_to = 'strain', values_to = 'n_core') %>%
    group_by(scheme, source, term) %>%
    mutate(width = NES * n_core / sum(n_core)) %>% ungroup() %>%
    mutate(strain = factor(strain, levels = names(col_strain)))

cat(sprintf("\n■ bar_gsea: %d 行（1行 = term × 株）\n", nrow(bar_gsea)))


# ── 色の指定 ───────────────────────────────────────────────────
# ★ セル7 と同じく「KO で低い = 青、KO で高い = 赤」で塗る。
#   ただし NES は 0 をまたぐので、1枚の図に赤と青の2つの fill スケールは置けない。
#   → ★ NES = 0 を境に上下2パネルに分け、patchwork で縦に積む。
#     各パネルに自前の p.adjust 凡例が付く。
#     どちらも limits = c(0, 0.05) で ★ 数値の範囲は同一なので二重基準にはならない。
#     違うのは色相だけで、色相は向き（どちらのパネルか）を表す
#
# dotplot 用（濃い青/赤。セル7 と同じ）
col_dn = scale_fill_gradient(low = 'blue', high = 'white',
                             limits = c(0, 0.05), name = 'p.adjust\n(down in KO)')
col_up = scale_fill_gradient(low = 'red',  high = 'white',
                             limits = c(0, 0.05), name = 'p.adjust\n(up in KO)')

# 横棒用（薄い青/赤。セル7-2 と同じ理由）
#   濃いままだと塗りの上で株の縁が読めない。実測した問題:
#     down（青）: Bth_VPI の青が同化して消える / up（赤）: Bth_St8 の橙が埋もれる
col_dn_bar = scale_fill_gradient(low = '#8fb8f5', high = '#eaf1fd',
                                 limits = c(0, 0.05), name = 'p.adjust\n(down in KO)')
col_up_bar = scale_fill_gradient(low = '#f08080', high = '#fdeaea',
                                 limits = c(0, 0.05), name = 'p.adjust\n(up in KO)')


# ===============================================================
# 1. down / up に分けたコピーを作る
# ===============================================================
# 有意 term だけ残し、さらに NES の符号で分ける。
# A_PFAM は up が 0 本なので down のみ作る
gsea_A_KEGG_dn = gsea_A_KEGG; gsea_A_KEGG_dn@result = gsea_A_KEGG@result %>% filter(p.adjust < 0.05, NES < 0) %>% arrange(NES)
gsea_A_KEGG_up = gsea_A_KEGG; gsea_A_KEGG_up@result = gsea_A_KEGG@result %>% filter(p.adjust < 0.05, NES > 0) %>% arrange(desc(NES))
gsea_A_COG_dn = gsea_A_COG; gsea_A_COG_dn@result = gsea_A_COG@result %>% filter(p.adjust < 0.05, NES < 0) %>% arrange(NES)
gsea_A_COG_up = gsea_A_COG; gsea_A_COG_up@result = gsea_A_COG@result %>% filter(p.adjust < 0.05, NES > 0) %>% arrange(desc(NES))
gsea_A_PFAM_dn = gsea_A_PFAM; gsea_A_PFAM_dn@result = gsea_A_PFAM@result %>% filter(p.adjust < 0.05, NES < 0) %>% arrange(NES)
gsea_B_KEGG_dn = gsea_B_KEGG; gsea_B_KEGG_dn@result = gsea_B_KEGG@result %>% filter(p.adjust < 0.05, NES < 0) %>% arrange(NES)
gsea_B_KEGG_up = gsea_B_KEGG; gsea_B_KEGG_up@result = gsea_B_KEGG@result %>% filter(p.adjust < 0.05, NES > 0) %>% arrange(desc(NES))
gsea_B_COG_dn = gsea_B_COG; gsea_B_COG_dn@result = gsea_B_COG@result %>% filter(p.adjust < 0.05, NES < 0) %>% arrange(NES)
gsea_B_COG_up = gsea_B_COG; gsea_B_COG_up@result = gsea_B_COG@result %>% filter(p.adjust < 0.05, NES > 0) %>% arrange(desc(NES))
gsea_B_PFAM_dn = gsea_B_PFAM; gsea_B_PFAM_dn@result = gsea_B_PFAM@result %>% filter(p.adjust < 0.05, NES < 0) %>% arrange(NES)
gsea_B_PFAM_up = gsea_B_PFAM; gsea_B_PFAM_up@result = gsea_B_PFAM@result %>% filter(p.adjust < 0.05, NES > 0) %>% arrange(desc(NES))

# ── ★ 上下パネルで x 軸を揃えるための範囲 ───────────────────────
# NES の絶対値の最大を 0.5 刻みに切り上げ、0 を中心に対称にする。
# 上段（up）と下段（down）の両方に同じ limits を渡すので、
# 0 の位置と目盛りが上下でぴったり一致する
lim_A_KEGG = ceiling(max(abs(c(gsea_A_KEGG_dn@result$NES, gsea_A_KEGG_up@result$NES))) * 2) / 2
lim_A_COG = ceiling(max(abs(c(gsea_A_COG_dn@result$NES, gsea_A_COG_up@result$NES))) * 2) / 2
lim_A_PFAM = ceiling(max(abs(gsea_A_PFAM_dn@result$NES)) * 2) / 2
lim_B_KEGG = ceiling(max(abs(c(gsea_B_KEGG_dn@result$NES, gsea_B_KEGG_up@result$NES))) * 2) / 2
lim_B_COG = ceiling(max(abs(c(gsea_B_COG_dn@result$NES, gsea_B_COG_up@result$NES))) * 2) / 2
lim_B_PFAM = ceiling(max(abs(c(gsea_B_PFAM_dn@result$NES, gsea_B_PFAM_up@result$NES))) * 2) / 2

cat("■ Table 1: 図にする組み合わせと x 軸の範囲\n")
gsea %>% filter(padj < 0.05) %>% dplyr::count(scheme, source, direction) %>%
    tidyr::pivot_wider(names_from = direction, values_from = n, values_fill = 0) %>%
    mutate(計 = down + up,
           パネル = ifelse(up == 0, 'down のみ', '上下2段'),
           x軸 = sprintf('-%.1f 〜 %.1f',
                c(lim_A_COG, lim_A_KEGG, lim_A_PFAM, lim_B_COG, lim_B_KEGG, lim_B_PFAM),
                c(lim_A_COG, lim_A_KEGG, lim_A_PFAM, lim_B_COG, lim_B_KEGG, lim_B_PFAM))) %>%
    as.data.frame()


# ===============================================================
# 2. NES の dotplot（セル7 に対応）
# ===============================================================
# 点の大きさ = setSize、色 = padj。上段が up（赤）、下段が down（青）
#
# ★ size = 'setSize' を明示する。既定の size = "Count"（core enrichment の
#   遺伝子数）は dotplot が内部で計算する列で @result に無く、目盛りを合わせられない

# ── (A) 全株プール = 宿主が何にさらされているか ──
# ── 図9-1: (A) KEGG（down 14 / up 7） ──
options(repr.plot.width = 10, repr.plot.height = 7.0)
(dotplot(gsea_A_KEGG_up, showCategory = nrow(gsea_A_KEGG_up@result), x = 'NES', size = 'setSize',
         label_format = lab60) +
     ggtitle('(A) pooled - GSEA - KEGG pathway') + col_up + labs(x = NULL) + scale_x_continuous(limits = c(-lim_A_KEGG, lim_A_KEGG)) + geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
     scale_size_continuous(breaks = unique(round(seq(min(gsea_A_KEGG_up@result$setSize),
                                                     max(gsea_A_KEGG_up@result$setSize), length.out = 4))))) /
(dotplot(gsea_A_KEGG_dn, showCategory = nrow(gsea_A_KEGG_dn@result), x = 'NES', size = 'setSize',
         label_format = lab60) +
     col_dn + scale_x_continuous(limits = c(-lim_A_KEGG, lim_A_KEGG)) + geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
     scale_size_continuous(breaks = unique(round(seq(min(gsea_A_KEGG_dn@result$setSize),
                                                     max(gsea_A_KEGG_dn@result$setSize), length.out = 4))))) +
    plot_layout(heights = c(nrow(gsea_A_KEGG_up@result), nrow(gsea_A_KEGG_dn@result)))

# ── 図9-2: (A) COG（down 5 / up 3） ──
options(repr.plot.width = 12, repr.plot.height = 4.2)
(dotplot(gsea_A_COG_up, showCategory = nrow(gsea_A_COG_up@result), x = 'NES', size = 'setSize',
         label_format = lab60) +
     ggtitle('(A) pooled - GSEA - COG category') + col_up + labs(x = NULL) + scale_x_continuous(limits = c(-lim_A_COG, lim_A_COG)) + geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
     scale_size_continuous(breaks = unique(round(seq(min(gsea_A_COG_up@result$setSize),
                                                     max(gsea_A_COG_up@result$setSize), length.out = 4))))) /
(dotplot(gsea_A_COG_dn, showCategory = nrow(gsea_A_COG_dn@result), x = 'NES', size = 'setSize',
         label_format = lab60) +
     col_dn + scale_x_continuous(limits = c(-lim_A_COG, lim_A_COG)) + geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
     scale_size_continuous(breaks = unique(round(seq(min(gsea_A_COG_dn@result$setSize),
                                                     max(gsea_A_COG_dn@result$setSize), length.out = 4))))) +
    plot_layout(heights = c(nrow(gsea_A_COG_up@result), nrow(gsea_A_COG_dn@result)))

# ── 図9-3: (A) PFAM（down 16 / up 0 → down のみ） ──
options(repr.plot.width = 9, repr.plot.height = 5.8)
dotplot(gsea_A_PFAM_dn, showCategory = nrow(gsea_A_PFAM_dn@result), x = 'NES', size = 'setSize',
        label_format = lab60) +
    ggtitle('(A) pooled - GSEA - PFAM domain') + col_dn + scale_x_continuous(limits = c(-lim_A_PFAM, lim_A_PFAM)) + geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
    scale_size_continuous(breaks = unique(round(seq(min(gsea_A_PFAM_dn@result$setSize),
                                                    max(gsea_A_PFAM_dn@result$setSize), length.out = 4))))

# ── (B) 株ごと独立 = 菌内部の制御 ──
# ── 図9-4: (B) KEGG（down 19 / up 6） ──
options(repr.plot.width = 10, repr.plot.height = 8.4)
(dotplot(gsea_B_KEGG_up, showCategory = nrow(gsea_B_KEGG_up@result), x = 'NES', size = 'setSize',
         label_format = lab60) +
     ggtitle('(B) per-strain - GSEA - KEGG pathway') + col_up + labs(x = NULL) + scale_x_continuous(limits = c(-lim_B_KEGG, lim_B_KEGG)) + geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
     scale_size_continuous(breaks = unique(round(seq(min(gsea_B_KEGG_up@result$setSize),
                                                     max(gsea_B_KEGG_up@result$setSize), length.out = 4))))) /
(dotplot(gsea_B_KEGG_dn, showCategory = nrow(gsea_B_KEGG_dn@result), x = 'NES', size = 'setSize',
         label_format = lab60) +
     col_dn + scale_x_continuous(limits = c(-lim_B_KEGG, lim_B_KEGG)) + geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
     scale_size_continuous(breaks = unique(round(seq(min(gsea_B_KEGG_dn@result$setSize),
                                                     max(gsea_B_KEGG_dn@result$setSize), length.out = 4))))) +
    plot_layout(heights = c(nrow(gsea_B_KEGG_up@result), nrow(gsea_B_KEGG_dn@result)))

# ── 図9-5: (B) COG（down 3 / up 2） ──
options(repr.plot.width = 12, repr.plot.height = 3.4)
(dotplot(gsea_B_COG_up, showCategory = nrow(gsea_B_COG_up@result), x = 'NES', size = 'setSize',
         label_format = lab60) +
     ggtitle('(B) per-strain - GSEA - COG category') + col_up + labs(x = NULL) + scale_x_continuous(limits = c(-lim_B_COG, lim_B_COG)) + geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
     scale_size_continuous(breaks = unique(round(seq(min(gsea_B_COG_up@result$setSize),
                                                     max(gsea_B_COG_up@result$setSize), length.out = 4))))) /
(dotplot(gsea_B_COG_dn, showCategory = nrow(gsea_B_COG_dn@result), x = 'NES', size = 'setSize',
         label_format = lab60) +
     col_dn + scale_x_continuous(limits = c(-lim_B_COG, lim_B_COG)) + geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
     scale_size_continuous(breaks = unique(round(seq(min(gsea_B_COG_dn@result$setSize),
                                                     max(gsea_B_COG_dn@result$setSize), length.out = 4))))) +
    plot_layout(heights = c(nrow(gsea_B_COG_up@result), nrow(gsea_B_COG_dn@result)))

# ── 図9-6: (B) PFAM（down 22 / up 4） ──
options(repr.plot.width = 9, repr.plot.height = 8.6)
(dotplot(gsea_B_PFAM_up, showCategory = nrow(gsea_B_PFAM_up@result), x = 'NES', size = 'setSize',
         label_format = lab60) +
     ggtitle('(B) per-strain - GSEA - PFAM domain') + col_up + labs(x = NULL) + scale_x_continuous(limits = c(-lim_B_PFAM, lim_B_PFAM)) + geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
     scale_size_continuous(breaks = unique(round(seq(min(gsea_B_PFAM_up@result$setSize),
                                                     max(gsea_B_PFAM_up@result$setSize), length.out = 4))))) /
(dotplot(gsea_B_PFAM_dn, showCategory = nrow(gsea_B_PFAM_dn@result), x = 'NES', size = 'setSize',
         label_format = lab60) +
     col_dn + scale_x_continuous(limits = c(-lim_B_PFAM, lim_B_PFAM)) + geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
     scale_size_continuous(breaks = unique(round(seq(min(gsea_B_PFAM_dn@result$setSize),
                                                     max(gsea_B_PFAM_dn@result$setSize), length.out = 4))))) +
    plot_layout(heights = c(nrow(gsea_B_PFAM_up@result), nrow(gsea_B_PFAM_dn@result)))

# ===============================================================
# 3. NES の横棒 ＋ core enrichment の株ごとの内訳（セル7-2 に対応）
# ===============================================================
# 棒の長さ = NES。区分 = core enrichment 遺伝子の株ごとの割合 × NES。
# 足すと NES になる。上段が up（赤）、下段が down（青）。
# x 軸は上下で同じ範囲（lim_* を両パネルに渡す）

# ── 図9-2-1: (A) KEGG（down 14 / up 7） ──
bar_dn_A_KEGG = bar_gsea %>% filter(scheme == 'A', source == 'KEGG', direction == 'down')
bar_up_A_KEGG = bar_gsea %>% filter(scheme == 'A', source == 'KEGG', direction == 'up')
options(repr.plot.width = 12, repr.plot.height = 7.5)
(ggplot(bar_up_A_KEGG %>% mutate(term = factor(term, levels = rev(unique(bar_up_A_KEGG$term[order(-bar_up_A_KEGG$NES)])))),
        aes(x = width, y = term, fill = padj, colour = strain)) +
     geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
     col_up_bar + col_edge + thm_bar + scale_x_continuous(limits = c(-lim_A_KEGG, lim_A_KEGG)) + geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
     labs(x = NULL, y = NULL, title = '(A) pooled - GSEA - KEGG pathway')) /
(ggplot(bar_dn_A_KEGG %>% mutate(term = factor(term, levels = rev(unique(bar_dn_A_KEGG$term[order(-bar_dn_A_KEGG$NES)])))),
        aes(x = width, y = term, fill = padj, colour = strain)) +
     geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
     col_dn_bar + col_edge + thm_bar + scale_x_continuous(limits = c(-lim_A_KEGG, lim_A_KEGG)) + geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
     labs(x = 'NES (stacked by strain)', y = NULL)) +
    plot_layout(heights = c(length(unique(bar_up_A_KEGG$term)), length(unique(bar_dn_A_KEGG$term))),
                guides = 'collect')

# ── 図9-2-2: (A) COG（down 5 / up 3） ──
bar_dn_A_COG = bar_gsea %>% filter(scheme == 'A', source == 'COG', direction == 'down')
bar_up_A_COG = bar_gsea %>% filter(scheme == 'A', source == 'COG', direction == 'up')
options(repr.plot.width = 14, repr.plot.height = 4.6)
(ggplot(bar_up_A_COG %>% mutate(term = factor(term, levels = rev(unique(bar_up_A_COG$term[order(-bar_up_A_COG$NES)])))),
        aes(x = width, y = term, fill = padj, colour = strain)) +
     geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
     col_up_bar + col_edge + thm_bar + scale_x_continuous(limits = c(-lim_A_COG, lim_A_COG)) + geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
     labs(x = NULL, y = NULL, title = '(A) pooled - GSEA - COG category')) /
(ggplot(bar_dn_A_COG %>% mutate(term = factor(term, levels = rev(unique(bar_dn_A_COG$term[order(-bar_dn_A_COG$NES)])))),
        aes(x = width, y = term, fill = padj, colour = strain)) +
     geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
     col_dn_bar + col_edge + thm_bar + scale_x_continuous(limits = c(-lim_A_COG, lim_A_COG)) + geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
     labs(x = 'NES (stacked by strain)', y = NULL)) +
    plot_layout(heights = c(length(unique(bar_up_A_COG$term)), length(unique(bar_dn_A_COG$term))),
                guides = 'collect')

# ── 図9-2-3: (A) PFAM（down 16 / up 0 → down のみ） ──
bar_dn_A_PFAM = bar_gsea %>% filter(scheme == 'A', source == 'PFAM', direction == 'down')
options(repr.plot.width = 11, repr.plot.height = 6.2)
ggplot(bar_dn_A_PFAM %>% mutate(term = factor(term, levels = rev(unique(bar_dn_A_PFAM$term[order(-bar_dn_A_PFAM$NES)])))),
       aes(x = width, y = term, fill = padj, colour = strain)) +
    geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
    col_dn_bar + col_edge + thm_bar + scale_x_continuous(limits = c(-lim_A_PFAM, lim_A_PFAM)) + geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
    labs(x = 'NES (stacked by strain)', y = NULL, title = '(A) pooled - GSEA - PFAM domain')

# ── 図9-2-4: (B) KEGG（down 19 / up 6） ──
bar_dn_B_KEGG = bar_gsea %>% filter(scheme == 'B', source == 'KEGG', direction == 'down')
bar_up_B_KEGG = bar_gsea %>% filter(scheme == 'B', source == 'KEGG', direction == 'up')
options(repr.plot.width = 12, repr.plot.height = 8.8)
(ggplot(bar_up_B_KEGG %>% mutate(term = factor(term, levels = rev(unique(bar_up_B_KEGG$term[order(-bar_up_B_KEGG$NES)])))),
        aes(x = width, y = term, fill = padj, colour = strain)) +
     geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
     col_up_bar + col_edge + thm_bar + scale_x_continuous(limits = c(-lim_B_KEGG, lim_B_KEGG)) + geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
     labs(x = NULL, y = NULL, title = '(B) per-strain - GSEA - KEGG pathway')) /
(ggplot(bar_dn_B_KEGG %>% mutate(term = factor(term, levels = rev(unique(bar_dn_B_KEGG$term[order(-bar_dn_B_KEGG$NES)])))),
        aes(x = width, y = term, fill = padj, colour = strain)) +
     geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
     col_dn_bar + col_edge + thm_bar + scale_x_continuous(limits = c(-lim_B_KEGG, lim_B_KEGG)) + geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
     labs(x = 'NES (stacked by strain)', y = NULL)) +
    plot_layout(heights = c(length(unique(bar_up_B_KEGG$term)), length(unique(bar_dn_B_KEGG$term))),
                guides = 'collect')

# ── 図9-2-5: (B) COG（down 3 / up 2） ──
bar_dn_B_COG = bar_gsea %>% filter(scheme == 'B', source == 'COG', direction == 'down')
bar_up_B_COG = bar_gsea %>% filter(scheme == 'B', source == 'COG', direction == 'up')
options(repr.plot.width = 14, repr.plot.height = 3.8)
(ggplot(bar_up_B_COG %>% mutate(term = factor(term, levels = rev(unique(bar_up_B_COG$term[order(-bar_up_B_COG$NES)])))),
        aes(x = width, y = term, fill = padj, colour = strain)) +
     geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
     col_up_bar + col_edge + thm_bar + scale_x_continuous(limits = c(-lim_B_COG, lim_B_COG)) + geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
     labs(x = NULL, y = NULL, title = '(B) per-strain - GSEA - COG category')) /
(ggplot(bar_dn_B_COG %>% mutate(term = factor(term, levels = rev(unique(bar_dn_B_COG$term[order(-bar_dn_B_COG$NES)])))),
        aes(x = width, y = term, fill = padj, colour = strain)) +
     geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
     col_dn_bar + col_edge + thm_bar + scale_x_continuous(limits = c(-lim_B_COG, lim_B_COG)) + geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
     labs(x = 'NES (stacked by strain)', y = NULL)) +
    plot_layout(heights = c(length(unique(bar_up_B_COG$term)), length(unique(bar_dn_B_COG$term))),
                guides = 'collect')

# ── 図9-2-6: (B) PFAM（down 22 / up 4） ──
bar_dn_B_PFAM = bar_gsea %>% filter(scheme == 'B', source == 'PFAM', direction == 'down')
bar_up_B_PFAM = bar_gsea %>% filter(scheme == 'B', source == 'PFAM', direction == 'up')
options(repr.plot.width = 11, repr.plot.height = 9.0)
(ggplot(bar_up_B_PFAM %>% mutate(term = factor(term, levels = rev(unique(bar_up_B_PFAM$term[order(-bar_up_B_PFAM$NES)])))),
        aes(x = width, y = term, fill = padj, colour = strain)) +
     geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
     col_up_bar + col_edge + thm_bar + scale_x_continuous(limits = c(-lim_B_PFAM, lim_B_PFAM)) + geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
     labs(x = NULL, y = NULL, title = '(B) per-strain - GSEA - PFAM domain')) /
(ggplot(bar_dn_B_PFAM %>% mutate(term = factor(term, levels = rev(unique(bar_dn_B_PFAM$term[order(-bar_dn_B_PFAM$NES)])))),
        aes(x = width, y = term, fill = padj, colour = strain)) +
     geom_col(linewidth = 1.2, width = 0.45, position = position_stack(reverse = TRUE)) +
     col_dn_bar + col_edge + thm_bar + scale_x_continuous(limits = c(-lim_B_PFAM, lim_B_PFAM)) + geom_vline(xintercept = 0, linetype = 'dashed', linewidth = 0.3) +
     labs(x = 'NES (stacked by strain)', y = NULL)) +
    plot_layout(heights = c(length(unique(bar_up_B_PFAM$term)), length(unique(bar_dn_B_PFAM$term))),
                guides = 'collect')

options(repr.plot.width = 7, repr.plot.height = 7)   # 既定に戻す


■ bar_gsea: 404 行（1行 = term × 株）
■ Table 1: 図にする組み合わせと x 軸の範囲


scheme,source,down,up,計,パネル,x軸
<chr>,<chr>,<int>,<int>,<int>,<chr>,<chr>
A,COG,5,3,8,上下2段,-2.5 〜 2.5
A,KEGG,14,7,21,上下2段,-2.5 〜 2.5
A,PFAM,16,0,16,down のみ,-2.5 〜 2.5
B,COG,3,2,5,上下2段,-2.5 〜 2.5
B,KEGG,19,6,25,上下2段,-2.5 〜 2.5
B,PFAM,22,4,26,上下2段,-2.5 〜 2.5


Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Scale for size is already present.
Adding another scale for size, which will replace the existing scale.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Scale for size is already present.
Adding another scale for size, which will replace the existing scale.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Scale for size is already present.
Adding another scale for size, which will replace the existing scale.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Scale for size is already present.
Adding another scale for size, which will replace the existing scale.


plot without title

Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Scale for size is already present.
Adding another scale for size, which will replace the existing scale.


plot without title

... (9 outputs left out)

plot without title

plot without title

plot without title

plot without title

### 考察: 1枚で「向き・強さ・どの菌か」が読める

#### 1. (A) KEGG（図9-1 / 9-2-1）— 上下がきれいに分かれる

0 より右（KO で高い）に 7 本、左（KO で低い）に 14 本。
**右端が `ko03010_Ribosome`（NES +2.45、`setSize` 112）、左端が `ko00195_Photosynthesis`（−2.28）。**

x 軸を上下で `-2.5 〜 2.5` に揃えたので、**up の最大（+2.45）と down の最大（−2.28）が
ほぼ同じ長さ**であることが直接比べられる。
**「KO で下がる側のほうが本数は多いが、1本あたりの強さは上下で同程度」**と読める。

横棒（図9-2-1）で株を見ると:

- **右側（up）の `ko03010_Ribosome` と `ko00970_Aminoacyl-tRNA biosynthesis` は区分がほぼ緑一色**
  → **`Bun_8492` が ES を押し上げている。**セル6 Table 4 の内訳（Bun 82.9%）と一致
- **左側（down）はどの term も4色が並ぶ** → 4株共通の応答
- ただし **`ko00120_Primary bile acid biosynthesis` は緑と紫だけ**（`Bth` 2株は `ko00120` に
  アノテーションされた遺伝子を持たない。セル6 考察 1-2 と同じ）

#### 2. `dotplot` と横棒で読むものが違う

- **`dotplot`（図9-1）**: 点の大きさが `setSize` なので、
  **`ko00030_Pentose phosphate pathway`（69）や `ko00020_Citrate cycle`（61）が大きく、
  `ko00590_Arachidonic acid metabolism`（3）が極端に小さい**ことが分かる。
  **`setSize` が小さい term は NES が同程度でも根拠が薄い**という判断材料になる
- **横棒（図9-2-1）**: 株の内訳が分かる代わりに `setSize` が見えない。
  塗りを薄くしてあるので4株すべての縁が読める

**両方を見る必要がある。**

#### 3. 上下に分けたことで色の意味が一意になった

セル7（ORA）の赤/青と**同じ約束**になったので、
**セル7 → セル7-2 → セル9 を通して「青 = KO で低い、赤 = KO で高い」**で読める。

凡例は2つ出るが、**どちらも `0 〜 0.05` の同じ範囲**なので比較できる。
`ko03010_Ribosome`（padj 1.8e-10）は上段で最も濃い赤、
`ko00195_Photosynthesis`（5.0e-05）は下段で最も濃い青になる。

#### 4. (A) と (B) を見比べる（図9-2-1 と 9-2-4）

(B) の up 側でも緑（Bun）は大きいが、**(A) ほど一色ではない。**
**(A) で Bun に偏るのが正規化範囲の違いによる組成シフト**であることが、
セル7-2 と同じく色の面積で見える。

(B) の down は 19 本と (A) の 14 本より多く、**`ko00010_Glycolysis / Gluconeogenesis` や
`ko00500_Starch and sucrose metabolism` が加わる**（セル8 考察 4）。

#### 5. この図で読めないこと

- **ORA との一致/不一致は分からない。**セル8 Table 4-6 で確認する
- **`core_enrichment` の区分は「その株の遺伝子が leading edge に何個入ったか」**であって、
  **その株単独で濃縮が有意かどうかではない**（セル7-2 考察 4 と同じ）
- **enrichment plot（running ES の曲線）はセル10** で出す

## セル10: enrichment plot（running ES の曲線）

GSEA 固有の図。**セル9 は「結果」を並べたものだが、こちらは結果が出た根拠そのものを見せる。**

### 図の3段

**上段** = Running Enrichment Score。遺伝子を統計量の順（左 = KO で高い … 右 = KO で低い）に
たどり、その term に属していれば上へ、属していなければ下へ動かした曲線。
**曲線の頂点が ES**、NES はこれを term の大きさで標準化したもの。
**中段** = その term の遺伝子が順位のどこにあるか（縦棒）。
**下段** = 順位付けに使った **Wald 統計量そのもの**。
**棒が左に固まれば KO で高い、右に固まれば KO で低い。**

### ★ 下段の x 軸に向きを書き足す

既定のラベルは `Rank in Ordered Dataset` だけで**どちら向きの順位か分からない。**
`geneList` は Wald 統計量の**降順**なので、**左端 = KO で最も高い、右端 = WT で最も高い。**
`gseaplot2` が返すのは **3枚の ggplot のリスト**で、**x 軸ラベルを持つのは3枚目だけ**なので
そこだけ差し替える。

### ★ セル8 Table 2 で有意になった term を全部描く（101 本 → 22 枚）

**1枚に重ねられるのは 5-6 本まで**なので、**scheme × source × 向き**で分け、
さらに **NES の強い順**に 5-6 本ずつ区切る。

| scheme | KEGG down / up | COG down / up | PFAM down / up |
|---|---|---|---|
| (A) | 14 / 7 | 5 / 3 | 16 / **0** |
| (B) | 19 / 6 | 3 / 2 | 22 / 4 |

**(A) PFAM の up と CAZy は有意 0 なので図が無い。**

### ★ 色は名前付きベクトルで渡す

`gseaplot2` の `color` は `scale_color_manual` の `values` に渡されるので、
**名前を付けないと `geneSetID` の順ではなく term 名のアルファベット順に割り当てられる。**
実測で `geneSetID = c('ko03010_Ribosome', 'ko00195_Photosynthesis', ...)` に
`color = c('blue','red','darkorange')` を渡すと **`ko03010` に橙が当たった**。
→ `color = setNames(pal_dn[1:5], ids)`。

### 向きの約束はセル7 以降と同じ

**1枚の中の term はすべて同じ向き**なので、**KO で低い = 寒色、KO で高い = 暖色**とし、
1枚の中で見分けるために 6 色ずつ用意する。


In [83]:
# ── enrichment plot（running ES の曲線）──────────────────────────
# GSEA 固有の図。セル9 の一覧図は「結果」を並べたものだが、
# こちらは ★ その結果が出た根拠そのものを見せる。
#
# 図の3段
#   上段 Running Enrichment Score
#        遺伝子を統計量の順（左 = KO で高い … 右 = KO で低い）に1つずつ
#        たどり、その term に属していれば上へ、属していなければ下へ動かした曲線。
#        ★ 曲線の頂点（絶対値が最大の点）が ES。NES はこれを term の大きさで標準化したもの
#   中段 その term の遺伝子が順位のどこにあるか（縦棒）
#        ★ 棒が左に固まれば KO で高い、右に固まれば KO で低い
#   下段 順位付けに使った Wald 統計量そのもの（灰色）
#
# ★ セル8 Table 3 で有意になった term を ★ 全部描く（101 本）。
#   1枚に重ねられるのは 5-6 本までなので、
#   scheme × source × 向き で分け、さらに NES の強い順に 5-6 本ずつ区切る。
library(enrichplot)

# ★ 色は名前付きベクトルで渡す
#   gseaplot2 の color は scale_color_manual の values に渡されるので、
#   名前を付けないと ★ geneSetID の順ではなく term 名のアルファベット順に
#   割り当てられる（実測で確認。Ribosome に blue を当てたつもりが橙になった）。
#
# 向きの約束はセル7 以降と同じ: KO で低い = 寒色、KO で高い = 暖色。
# 1枚の中の term を見分けるため、それぞれ 6 色ずつ用意する
pal_dn = c('#1f78b4', '#00008b', '#00ced1', '#6a3d9a', '#4169e1', '#008080')
pal_up = c('#e41a1c', '#ff7f00', '#a65628', '#f781bf', '#8b0000', '#d95f02')

# ★ 下段の x 軸は「順位」だけでどちら向きか分からないので書き足す。
#   geneList は Wald 統計量の降順なので、左端 = KO で最も高い、右端 = WT で最も高い。
#   gseaplot2 が返すのは3枚の ggplot のリスト（gglist）で、
#   x 軸ラベルを持つのは3枚目だけ。そこだけ差し替える
xlab_rank = 'Rank in Ordered Dataset      <-- higher in KO            higher in WT -->'

# ===============================================================
# 1. 向きごとに term を NES の強い順に並べる
# ===============================================================

ids_A_KEGG_down = gsea %>% filter(scheme == 'A', source == 'KEGG',
                            padj < 0.05, NES < 0) %>%
    arrange(desc(abs(NES))) %>% pull(term)
ids_A_KEGG_up = gsea %>% filter(scheme == 'A', source == 'KEGG',
                            padj < 0.05, NES > 0) %>%
    arrange(desc(abs(NES))) %>% pull(term)
ids_A_COG_down = gsea %>% filter(scheme == 'A', source == 'COG',
                            padj < 0.05, NES < 0) %>%
    arrange(desc(abs(NES))) %>% pull(term)
ids_A_COG_up = gsea %>% filter(scheme == 'A', source == 'COG',
                            padj < 0.05, NES > 0) %>%
    arrange(desc(abs(NES))) %>% pull(term)
ids_A_PFAM_down = gsea %>% filter(scheme == 'A', source == 'PFAM',
                            padj < 0.05, NES < 0) %>%
    arrange(desc(abs(NES))) %>% pull(term)
ids_B_KEGG_down = gsea %>% filter(scheme == 'B', source == 'KEGG',
                            padj < 0.05, NES < 0) %>%
    arrange(desc(abs(NES))) %>% pull(term)
ids_B_KEGG_up = gsea %>% filter(scheme == 'B', source == 'KEGG',
                            padj < 0.05, NES > 0) %>%
    arrange(desc(abs(NES))) %>% pull(term)
ids_B_COG_down = gsea %>% filter(scheme == 'B', source == 'COG',
                            padj < 0.05, NES < 0) %>%
    arrange(desc(abs(NES))) %>% pull(term)
ids_B_COG_up = gsea %>% filter(scheme == 'B', source == 'COG',
                            padj < 0.05, NES > 0) %>%
    arrange(desc(abs(NES))) %>% pull(term)
ids_B_PFAM_down = gsea %>% filter(scheme == 'B', source == 'PFAM',
                            padj < 0.05, NES < 0) %>%
    arrange(desc(abs(NES))) %>% pull(term)
ids_B_PFAM_up = gsea %>% filter(scheme == 'B', source == 'PFAM',
                            padj < 0.05, NES > 0) %>%
    arrange(desc(abs(NES))) %>% pull(term)

cat("■ Table 1: 描く term の数と図の枚数\n")
data.frame(
    scheme = c('A','A','A','A','A','B','B','B','B','B','B'),
    source = c('KEGG','KEGG','COG','COG','PFAM','KEGG','KEGG','COG','COG','PFAM','PFAM'),
    direction = c('down','up','down','up','down','down','up','down','up','down','up'),
    term数 = c(length(ids_A_KEGG_down), length(ids_A_KEGG_up),
               length(ids_A_COG_down),  length(ids_A_COG_up),
               length(ids_A_PFAM_down),
               length(ids_B_KEGG_down), length(ids_B_KEGG_up),
               length(ids_B_COG_down),  length(ids_B_COG_up),
               length(ids_B_PFAM_down), length(ids_B_PFAM_up)),
    check.names = FALSE) %>%
    mutate(図の枚数 = ceiling(term数 / 6), 合計 = sum(term数))


# ===============================================================
# (A) KEGG — down in KO（14 本 / 3 枚）
# ===============================================================
# ── 図10-1: (A) KEGG down in KO 1/3（NES の強い順 1-5 本目）──
ids = ids_A_KEGG_down[1:5]
options(repr.plot.width = 14, repr.plot.height = 6.2)
g = gseaplot2(gsea_A_KEGG, geneSetID = ids, color = setNames(pal_dn[1:5], ids),
          pvalue_table = TRUE,
          title = '(A) KEGG - down in KO (1/3)')
g[[3]] = g[[3]] + xlab(xlab_rank)
g

# ── 図10-2: (A) KEGG down in KO 2/3（NES の強い順 6-10 本目）──
ids = ids_A_KEGG_down[6:10]
options(repr.plot.width = 14, repr.plot.height = 6.2)
g = gseaplot2(gsea_A_KEGG, geneSetID = ids, color = setNames(pal_dn[1:5], ids),
          pvalue_table = TRUE,
          title = '(A) KEGG - down in KO (2/3)')
g[[3]] = g[[3]] + xlab(xlab_rank)
g

# ── 図10-3: (A) KEGG down in KO 3/3（NES の強い順 11-14 本目）──
ids = ids_A_KEGG_down[11:14]
options(repr.plot.width = 14, repr.plot.height = 6.2)
g = gseaplot2(gsea_A_KEGG, geneSetID = ids, color = setNames(pal_dn[1:4], ids),
          pvalue_table = TRUE,
          title = '(A) KEGG - down in KO (3/3)')
g[[3]] = g[[3]] + xlab(xlab_rank)
g


# ===============================================================
# (A) KEGG — up in KO（7 本 / 2 枚）
# ===============================================================
# ── 図10-4: (A) KEGG up in KO 1/2（NES の強い順 1-4 本目）──
ids = ids_A_KEGG_up[1:4]
options(repr.plot.width = 14, repr.plot.height = 6.2)
g = gseaplot2(gsea_A_KEGG, geneSetID = ids, color = setNames(pal_up[1:4], ids),
          pvalue_table = TRUE,
          title = '(A) KEGG - up in KO (1/2)')
g[[3]] = g[[3]] + xlab(xlab_rank)
g

# ── 図10-5: (A) KEGG up in KO 2/2（NES の強い順 5-7 本目）──
ids = ids_A_KEGG_up[5:7]
options(repr.plot.width = 14, repr.plot.height = 6.2)
g = gseaplot2(gsea_A_KEGG, geneSetID = ids, color = setNames(pal_up[1:3], ids),
          pvalue_table = TRUE,
          title = '(A) KEGG - up in KO (2/2)')
g[[3]] = g[[3]] + xlab(xlab_rank)
g


# ===============================================================
# (A) COG — down in KO（5 本 / 1 枚）
# ===============================================================
# ── 図10-6: (A) COG down in KO 1/1（NES の強い順 1-5 本目）──
ids = ids_A_COG_down[1:5]
options(repr.plot.width = 14, repr.plot.height = 6.2)
g = gseaplot2(gsea_A_COG, geneSetID = ids, color = setNames(pal_dn[1:5], ids),
          pvalue_table = TRUE,
          title = '(A) COG - down in KO (1/1)')
g[[3]] = g[[3]] + xlab(xlab_rank)
g


# ===============================================================
# (A) COG — up in KO（3 本 / 1 枚）
# ===============================================================
# ── 図10-7: (A) COG up in KO 1/1（NES の強い順 1-3 本目）──
ids = ids_A_COG_up[1:3]
options(repr.plot.width = 14, repr.plot.height = 6.2)
g = gseaplot2(gsea_A_COG, geneSetID = ids, color = setNames(pal_up[1:3], ids),
          pvalue_table = TRUE,
          title = '(A) COG - up in KO (1/1)')
g[[3]] = g[[3]] + xlab(xlab_rank)
g


# ===============================================================
# (A) PFAM — down in KO（16 本 / 3 枚）
# ===============================================================
# ── 図10-8: (A) PFAM down in KO 1/3（NES の強い順 1-6 本目）──
ids = ids_A_PFAM_down[1:6]
options(repr.plot.width = 14, repr.plot.height = 6.2)
g = gseaplot2(gsea_A_PFAM, geneSetID = ids, color = setNames(pal_dn[1:6], ids),
          pvalue_table = TRUE,
          title = '(A) PFAM - down in KO (1/3)')
g[[3]] = g[[3]] + xlab(xlab_rank)
g

# ── 図10-9: (A) PFAM down in KO 2/3（NES の強い順 7-11 本目）──
ids = ids_A_PFAM_down[7:11]
options(repr.plot.width = 14, repr.plot.height = 6.2)
g = gseaplot2(gsea_A_PFAM, geneSetID = ids, color = setNames(pal_dn[1:5], ids),
          pvalue_table = TRUE,
          title = '(A) PFAM - down in KO (2/3)')
g[[3]] = g[[3]] + xlab(xlab_rank)
g

# ── 図10-10: (A) PFAM down in KO 3/3（NES の強い順 12-16 本目）──
ids = ids_A_PFAM_down[12:16]
options(repr.plot.width = 14, repr.plot.height = 6.2)
g = gseaplot2(gsea_A_PFAM, geneSetID = ids, color = setNames(pal_dn[1:5], ids),
          pvalue_table = TRUE,
          title = '(A) PFAM - down in KO (3/3)')
g[[3]] = g[[3]] + xlab(xlab_rank)
g


# ===============================================================
# (B) KEGG — down in KO（19 本 / 4 枚）
# ===============================================================
# ── 図10-11: (B) KEGG down in KO 1/4（NES の強い順 1-5 本目）──
ids = ids_B_KEGG_down[1:5]
options(repr.plot.width = 14, repr.plot.height = 6.2)
g = gseaplot2(gsea_B_KEGG, geneSetID = ids, color = setNames(pal_dn[1:5], ids),
          pvalue_table = TRUE,
          title = '(B) KEGG - down in KO (1/4)')
g[[3]] = g[[3]] + xlab(xlab_rank)
g

# ── 図10-12: (B) KEGG down in KO 2/4（NES の強い順 6-10 本目）──
ids = ids_B_KEGG_down[6:10]
options(repr.plot.width = 14, repr.plot.height = 6.2)
g = gseaplot2(gsea_B_KEGG, geneSetID = ids, color = setNames(pal_dn[1:5], ids),
          pvalue_table = TRUE,
          title = '(B) KEGG - down in KO (2/4)')
g[[3]] = g[[3]] + xlab(xlab_rank)
g

# ── 図10-13: (B) KEGG down in KO 3/4（NES の強い順 11-15 本目）──
ids = ids_B_KEGG_down[11:15]
options(repr.plot.width = 14, repr.plot.height = 6.2)
g = gseaplot2(gsea_B_KEGG, geneSetID = ids, color = setNames(pal_dn[1:5], ids),
          pvalue_table = TRUE,
          title = '(B) KEGG - down in KO (3/4)')
g[[3]] = g[[3]] + xlab(xlab_rank)
g

# ── 図10-14: (B) KEGG down in KO 4/4（NES の強い順 16-19 本目）──
ids = ids_B_KEGG_down[16:19]
options(repr.plot.width = 14, repr.plot.height = 6.2)
g = gseaplot2(gsea_B_KEGG, geneSetID = ids, color = setNames(pal_dn[1:4], ids),
          pvalue_table = TRUE,
          title = '(B) KEGG - down in KO (4/4)')
g[[3]] = g[[3]] + xlab(xlab_rank)
g


# ===============================================================
# (B) KEGG — up in KO（6 本 / 1 枚）
# ===============================================================
# ── 図10-15: (B) KEGG up in KO 1/1（NES の強い順 1-6 本目）──
ids = ids_B_KEGG_up[1:6]
options(repr.plot.width = 14, repr.plot.height = 6.2)
g = gseaplot2(gsea_B_KEGG, geneSetID = ids, color = setNames(pal_up[1:6], ids),
          pvalue_table = TRUE,
          title = '(B) KEGG - up in KO (1/1)')
g[[3]] = g[[3]] + xlab(xlab_rank)
g


# ===============================================================
# (B) COG — down in KO（3 本 / 1 枚）
# ===============================================================
# ── 図10-16: (B) COG down in KO 1/1（NES の強い順 1-3 本目）──
ids = ids_B_COG_down[1:3]
options(repr.plot.width = 14, repr.plot.height = 6.2)
g = gseaplot2(gsea_B_COG, geneSetID = ids, color = setNames(pal_dn[1:3], ids),
          pvalue_table = TRUE,
          title = '(B) COG - down in KO (1/1)')
g[[3]] = g[[3]] + xlab(xlab_rank)
g


# ===============================================================
# (B) COG — up in KO（2 本 / 1 枚）
# ===============================================================
# ── 図10-17: (B) COG up in KO 1/1（NES の強い順 1-2 本目）──
ids = ids_B_COG_up[1:2]
options(repr.plot.width = 14, repr.plot.height = 6.2)
g = gseaplot2(gsea_B_COG, geneSetID = ids, color = setNames(pal_up[1:2], ids),
          pvalue_table = TRUE,
          title = '(B) COG - up in KO (1/1)')
g[[3]] = g[[3]] + xlab(xlab_rank)
g


# ===============================================================
# (B) PFAM — down in KO（22 本 / 4 枚）
# ===============================================================
# ── 図10-18: (B) PFAM down in KO 1/4（NES の強い順 1-6 本目）──
ids = ids_B_PFAM_down[1:6]
options(repr.plot.width = 14, repr.plot.height = 6.2)
g = gseaplot2(gsea_B_PFAM, geneSetID = ids, color = setNames(pal_dn[1:6], ids),
          pvalue_table = TRUE,
          title = '(B) PFAM - down in KO (1/4)')
g[[3]] = g[[3]] + xlab(xlab_rank)
g

# ── 図10-19: (B) PFAM down in KO 2/4（NES の強い順 7-12 本目）──
ids = ids_B_PFAM_down[7:12]
options(repr.plot.width = 14, repr.plot.height = 6.2)
g = gseaplot2(gsea_B_PFAM, geneSetID = ids, color = setNames(pal_dn[1:6], ids),
          pvalue_table = TRUE,
          title = '(B) PFAM - down in KO (2/4)')
g[[3]] = g[[3]] + xlab(xlab_rank)
g

# ── 図10-20: (B) PFAM down in KO 3/4（NES の強い順 13-17 本目）──
ids = ids_B_PFAM_down[13:17]
options(repr.plot.width = 14, repr.plot.height = 6.2)
g = gseaplot2(gsea_B_PFAM, geneSetID = ids, color = setNames(pal_dn[1:5], ids),
          pvalue_table = TRUE,
          title = '(B) PFAM - down in KO (3/4)')
g[[3]] = g[[3]] + xlab(xlab_rank)
g

# ── 図10-21: (B) PFAM down in KO 4/4（NES の強い順 18-22 本目）──
ids = ids_B_PFAM_down[18:22]
options(repr.plot.width = 14, repr.plot.height = 6.2)
g = gseaplot2(gsea_B_PFAM, geneSetID = ids, color = setNames(pal_dn[1:5], ids),
          pvalue_table = TRUE,
          title = '(B) PFAM - down in KO (4/4)')
g[[3]] = g[[3]] + xlab(xlab_rank)
g


# ===============================================================
# (B) PFAM — up in KO（4 本 / 1 枚）
# ===============================================================
# ── 図10-22: (B) PFAM up in KO 1/1（NES の強い順 1-4 本目）──
ids = ids_B_PFAM_up[1:4]
options(repr.plot.width = 14, repr.plot.height = 6.2)
g = gseaplot2(gsea_B_PFAM, geneSetID = ids, color = setNames(pal_up[1:4], ids),
          pvalue_table = TRUE,
          title = '(B) PFAM - up in KO (1/1)')
g[[3]] = g[[3]] + xlab(xlab_rank)
g

options(repr.plot.width = 7, repr.plot.height = 7)   # 既定に戻す

■ Table 1: 描く term の数と図の枚数


scheme,source,direction,term数,図の枚数,合計
<chr>,<chr>,<chr>,<int>,<dbl>,<int>
A,KEGG,down,14,3,101
A,KEGG,up,7,2,101
A,COG,down,5,1,101
A,COG,up,3,1,101
A,PFAM,down,16,3,101
B,KEGG,down,19,4,101
B,KEGG,up,6,1,101
B,COG,down,3,1,101
B,COG,up,2,1,101


plot without title

plot without title

plot without title

plot without title

... (14 outputs left out)

plot without title

plot without title

plot without title

plot without title

### 考察: 曲線の形で「どういう効き方か」が分かる

NES の数値だけでは分からない**効き方の違い**が、曲線と縦棒の位置に出る。大きく2つの型がある。

| 型 | 曲線 | 縦棒 | 例 |
|---|---|---|---|
| **狭く深い** | 片端で急に落ちる／上がる | 片端に密集 | `ko00195_Photosynthesis`、`ko00121` |
| **広く浅い** | ゆるやかに傾き谷／山が浅い | 全域に散在してやや偏る | `PFAM:Phage_integrase`、`COG C` |

#### 1. 強い term でも意味が違う

`ko00030_Pentose phosphate pathway` は NES −2.08 と強いが**曲線は −0.5 程度までしか下がらず
縦棒も全域に散る**（69 遺伝子）。**「数が多いので全体がやや下へ寄れば NES が出る」型**で、
**18 遺伝子が最下位に密集する `ko00195_Photosynthesis`（ES −0.85）とは意味が違う。**

#### 2. ★ `PFAM:Phage_integrase` は「広く浅い」

**曲線は −0.5 までしか下がらず、縦棒は順位の全域に散らばる。**`setSize` 67 なのでこの浅い ES でも
NES −1.89 / padj 0.020 に達する。**ORA で拾えなかった理由がここに見える**
（`padj < 0.05` を超える遺伝子が少なく「偶然より多いか」では検出できない）。
**全体がわずかに下方へ寄っている**形は閾値を切らない GSEA でないと見えない。

★ **ただし「浅い」＝効果が弱いということでもある。**CPS locus switching との関係
（セル8 考察 2）を主張するなら、**セル11 以降で個々のリコンビナーゼ遺伝子の発現を直接見る必要がある。**

#### 3. (A) と (B) は同じ形

同じ term を見比べると**曲線の形も縦棒の位置もほとんど変わらない**
（`ko03010` NES +2.45 → +2.44、`ko00195` −2.28 → −2.36、`ko00121` −1.92 → −1.85）。
**組成シフトを除いても同じ結論**というセル8 考察 1 の裏づけ。

#### 4. term が大きいときは曲線だけが頼り

`COG J`（937遺伝子）や `COG C`（1,000遺伝子超）は**縦棒が密になって個々の位置が読めない。**
**大きいカテゴリでは enrichment plot の情報量が落ちる**ので、セル9 の一覧図で NES を見るほうがよい。

#### 5. 弱い term ほど後ろの図に来る

各グループは **NES の強い順**に区切ってあるので、**最後の図ほど曲線が浅く偏りも小さい。**
**`padj` が 0.05 のすぐ下にある term は、曲線を見ると根拠が薄いことが分かる。**

#### 6. この図で読めないこと

- **どの株の遺伝子か分からない**（中段の縦棒に株の情報は無い）→ セル9 の横棒を見る
- **core enrichment（leading edge）の範囲が図示されない**（`gseaplot2` は頂点を示さない）
  → `gsea_all.csv` の `core` 列で確認する


## セル11: Pathview — ORA で有意になった KEGG 経路を KEGG マップ上に描く

過去notebook（`4_DESeq2_allmapped_fractionmode_R.ipynb` セル128-135）に対応。

### ★ 違い1: `gene.data` に値を渡す

過去notebook は **K番号の文字ベクトルを渡していた**ので、マップ上の KO が「あるか無いか」で
一律に塗られ**上下も大きさも色に出ない。**本データの主要所見（ATP synthase が下がる等）は
向きが見えないと読めないので、**`LFC` の名前付き数値ベクトルを渡す**（pathview 本来の使い方）。
同じ K 番号に複数株が当たるので **LFC の平均**を取り、(A) と (B) は**別々に描く。**

### ★ 違い2: 色を本notebook の約束に合わせる

pathview の既定は**緑 = 低下 / 赤 = 上昇**なので、`low`/`mid`/`high` で
**青 = KO で低下 / 灰 / 赤 = 上昇**に指定し直す。

### ★ 違い3: 1経路につき 2 枚を続けて出す

|- | 対象 | 意味 |
|---|---|---|
| **(1) ORA の `geneID` のみ** | Table 4 の `geneID` に入った遺伝子だけ | **ORA の結果そのもの。**セル12 の棒グラフと同じ集合 |
| **(2) 経路の全遺伝子** | 有意でないもの・逆向きのものも含む | **経路全体の姿** |

**どちらか一方だけだと読み違える。**(1) だけだと **down の term は必ず全部青、up は全部赤**になり、
色は「どちらに動いたか」ではなく「どの反応が ORA に拾われたか」になる。
(2) だけだと **ORA がどれを拾ったのか分からない。**

**実例**: `ko00190` は (1) では ATP synthase だけが青いが、(2) では `Nuo`/`Sdh`/`Cyd` が赤い
→ **「下がっているのは ATP 合成酵素だけ」は (2) でないと見えない。**
`ko00030` の `2.7.6.1`（PRPP synthetase）と `2.7.1.90`（PPi-PFK）は**有意に上昇**なので
**(A) down の `geneID` には入らず (1) では白**になる。

### 対象: KEGG 有意 term すべて（(A) down 18 / up 1 / (B) down 18 / up 2 = 39 行、経路は 22、× 2 枚 = 78 枚）

### ★ `setwd` が要る

**pathview はカレントディレクトリに PNG と XML を書く仕様**なので出力先へ移動してから呼ぶ
（過去notebook も同じ）。**セル末で必ず元に戻す。**

### ★ 描画が壊れるマップがある — 検品する

実測で **`ko00121_Secondary bile acid biosynthesis` が描けなかった。**
`circle` 型ノードを4個持ち、`some node width is different from others` の警告直後に中身を失う
（`same.layer=FALSE` は巨大な円がマップを覆い、`split.group=TRUE` は `subscript out of bounds`、
`kegg.native=FALSE` は拒否された）。
**描いてみないと判明しない**ので、**「白でない画素の割合」を素のマップと比べて検品する。**
正常な図は **0.58〜0.94**、壊れた `ko00121` は **0.14**。閾値は **0.4**
（pathview はノードの内側を塗り直すので**正常でも素のマップより ink は減る**。
`ko04013` 0.583・`ko04068` 0.594 は正常で目視確認済み）。

### ★ 違い4: セルの出力として表示する

**pathview は PNG をファイルに書くだけで描画デバイスには出さない。**`readPNG` + `grid.raster` で
プロットとして出し、**他の図と同じ「セルの出力」**にして `repr.plot.*` で大きさを揃える。

**大きさは図ごとに変える。**KEGG マップは縦横比が大きく違う
（`ko00310` 0.60 の横長 〜 `ko00120` 1.82 の細長い）。一律だと潰れるか間延びするので、
**幅 15 を基本にし、高さが 18 を超える縦長の図だけ高さ基準に切り替える。**


In [78]:
# ── Pathview: ORA で有意になった KEGG 経路を KEGG マップ上に描く ──
# 過去notebook（4_DESeq2_allmapped_fractionmode_R.ipynb セル128-135）に対応。
#
# ★ 過去notebook との違い 1: gene.data に値を渡す
#   過去notebook は K 番号の文字ベクトルだけを渡していたので、
#   マップ上の KO が「あるか無いか」で一律に塗られ、★ 上下も大きさも色に出なかった。
#   ここは LFC の名前付き数値ベクトルを渡して向きと大きさを色にする（pathview 本来の使い方）。
#
# ★ 違い 2: 色は本notebook の約束どおり KO で低い = 青、KO で高い = 赤。
#   pathview の既定は緑/赤なので low/mid/high で指定し直す。
#
# ★ 違い 3: 1経路につき 2 枚を続けて出す
#   (1) ORA の geneID のみ … セル6 Table 4 の結果そのもの。セル12 の棒グラフと同じ遺伝子集合
#   (2) 経路に属する全遺伝子 … 有意でないものや逆向きのものも含めた経路全体の姿
#   どちらか一方だけだと読み違える。実例:
#     ko00190 を (1) で描くと ATP synthase だけが青くなるが、
#     (2) で描くと NADH dehydrogenase / Sdh / Cyd が赤く、
#     ★「電子伝達系は上がっていて、下がっているのは ATP 合成酵素だけ」が分かる
library(pathview)
library(png)
library(grid)

# ===============================================================
# 1. 描く経路の一覧を作る
# ===============================================================
pv_list = ora %>% filter(source == 'KEGG', padj < 0.05) %>%
    transmute(scheme, direction, term, ko = sub('_.*', '', term), padj) %>%
    arrange(scheme, desc(direction), padj)

cat(sprintf("■ Table 1: 描く経路 %d 行 × 2 枚 = %d 枚（重複なしの経路は %d）\n",
    nrow(pv_list), nrow(pv_list) * 2, length(unique(pv_list$ko))))
pv_list %>% dplyr::count(scheme, direction, name = 'n') %>% as.data.frame()

# ===============================================================
# 2. 出力先を決めて移動する
# ===============================================================
# ★ pathview は「カレントディレクトリ」に PNG/XML を書く仕様なので setwd が要る。
#   過去notebook も setwd(folder_pathway) していた。セル末で元に戻す
dir_pv = file.path(path_pw, 'pathview')
if (!dir.exists(dir_pv)) dir.create(dir_pv, recursive = TRUE)
wd_old = getwd()
setwd(dir_pv)
cat(sprintf("\n出力先: %s\n", dir_pv))

# ===============================================================
# 3. 1経路につき 2 枚ずつ描く
# ===============================================================
# マップによっては描画が壊れる（node の幅が揃わないと中身が消える）ので、
# 出来たファイルの「白でない画素の割合」を素のマップと比べて検品する
pv_list$file_ora = NA_character_; pv_list$file_all = NA_character_
pv_list$stat_ora = NA_character_; pv_list$stat_all = NA_character_
pv_list$nK_ora   = NA_integer_;   pv_list$nK_all   = NA_integer_
pv_list$ink_ora  = NA_real_;      pv_list$ink_all  = NA_real_

for (i in 1:nrow(pv_list)) {
    ko  = pv_list$ko[i]
    sch = pv_list$scheme[i]
    num = sub('^ko', '', ko)

    # (1) ORA の geneID のみ
    g_ora = strsplit(ora$geneID[ora$scheme == sch & ora$source == 'KEGG' &
                                ora$term == pv_list$term[i]], '/')[[1]]
    kv_ora = res %>% filter(gene_id %in% g_ora, !is.na(KEGG_ko)) %>%
        dplyr::select(gene_id, KEGG_ko, LFC_A, LFC_B) %>%
        tidyr::separate_rows(KEGG_ko, sep = ',') %>%
        mutate(K = sub('^ko:', '', KEGG_ko),
               v = if (sch == 'A') LFC_A else LFC_B) %>%
        filter(grepl('^K[0-9]+$', K), !is.na(v)) %>%
        group_by(K) %>% summarise(v = mean(v), .groups = 'drop')

    # (2) 経路に属する全遺伝子（有意でないもの・逆向きのものも含む）
    kv_all = res %>% filter(!is.na(KEGG_Pathway), !is.na(KEGG_ko), !marker) %>%
        dplyr::select(gene_id, KEGG_Pathway, KEGG_ko, LFC_A, LFC_B) %>%
        tidyr::separate_rows(KEGG_Pathway, sep = ',') %>%
        filter(KEGG_Pathway == ko) %>%
        tidyr::separate_rows(KEGG_ko, sep = ',') %>%
        mutate(K = sub('^ko:', '', KEGG_ko),
               v = if (sch == 'A') LFC_A else LFC_B) %>%
        filter(grepl('^K[0-9]+$', K), !is.na(v)) %>%
        group_by(K) %>% summarise(v = mean(v), .groups = 'drop')

    # 2 枚を同じ設定で描く
    for (kind in c('ORA', 'all')) {
        kv  = if (kind == 'ORA') kv_ora else kv_all
        vec = setNames(kv$v, kv$K)
        suf = paste0(sch, '_', kind)
        f   = paste0(ko, '.', suf, '.png')

        ok = tryCatch({
            invisible(pathview(gene.data = vec, pathway.id = num, species = 'ko',
                               out.suffix = suf, limit = list(gene = 2.5),
                               low  = list(gene = 'blue'),
                               mid  = list(gene = 'gray90'),
                               high = list(gene = 'red')))
            TRUE }, error = function(e) FALSE)

        st = 'ERROR'; ink = NA_real_
        if (ok && file.exists(f)) {
            # 検品: 素のマップと比べて中身が残っているか
            ikf = function(p) { im = readPNG(p)
                                mean(im[,,1] < 0.98 | im[,,2] < 0.98 | im[,,3] < 0.98) }
            ink = round(ikf(f) / ikf(paste0(ko, '.png')), 3)
            # 閾値 0.4: 正常な図は 0.58〜0.94、壊れた ko00121 は 0.14 だった。
            # pathview はノードの内側を塗り直すので正常でも素のマップより ink は減る
            st = if (ink > 0.4) 'OK' else '★ 描画が壊れた'
        }
        if (kind == 'ORA') {
            pv_list$file_ora[i] = f; pv_list$stat_ora[i] = st
            pv_list$nK_ora[i] = length(vec); pv_list$ink_ora[i] = ink
        } else {
            pv_list$file_all[i] = f; pv_list$stat_all[i] = st
            pv_list$nK_all[i] = length(vec); pv_list$ink_all[i] = ink
        }
    }
}

cat("\n■ Table 2: 描画の結果（ORA = geneID のみ / all = 経路の全遺伝子）\n")
pv_list %>% transmute(scheme, direction, term,
                      K_ORA = nK_ora, 判定_ORA = stat_ora,
                      K_all = nK_all, 判定_all = stat_all) %>% as.data.frame()

# ===============================================================
# 4. セルの出力として、1経路ごとに 2 枚続けて表示する
# ===============================================================
# ★ pathview は PNG をファイルに書くだけで、描画デバイスには出さない。
#   readPNG で読み、grid.raster でプロットとして出す。
#
# ★ 大きさは図ごとに変える。KEGG マップは縦横比が大きく違い
#   （ko00310 は 0.60 の横長、ko00120 は 1.82 の縦長）、一律だと潰れるか間延びする。
#   幅 15 を基本にし、高さが 18 を超える縦長の図だけ高さ基準に切り替える
for (i in 1:nrow(pv_list)) {
    cat(sprintf("\n\n■ 図11-%d: (%s) %s  [%s]\n", i, pv_list$scheme[i],
                pv_list$term[i], pv_list$direction[i]))

    for (kind in c('ORA', 'all')) {
        f  = if (kind == 'ORA') pv_list$file_ora[i] else pv_list$file_all[i]
        st = if (kind == 'ORA') pv_list$stat_ora[i] else pv_list$stat_all[i]
        nK = if (kind == 'ORA') pv_list$nK_ora[i]   else pv_list$nK_all[i]
        tag = if (kind == 'ORA') '(1) ORA の geneID のみ' else '(2) 経路に属する全遺伝子'

        if (st != 'OK') { cat(sprintf("   %s → %s（図なし）\n", tag, st)); next }
        im = readPNG(f)
        h_px = dim(im)[1]; w_px = dim(im)[2]
        W = 15; H = W * h_px / w_px
        if (H > 18) { H = 18; W = H * w_px / h_px }
        cat(sprintf("   %s  K %d 個\n", tag, nK))
        options(repr.plot.width = W, repr.plot.height = H)
        grid.newpage()
        grid.raster(im)
    }
}

options(repr.plot.width = 7, repr.plot.height = 7)   # 既定に戻す

# ===============================================================
# 5. 後始末と保存
# ===============================================================
setwd(wd_old)   # ★ pathview のために移動したので必ず戻す
write.csv(pv_list %>% dplyr::select(-file_ora, -file_all),
          file.path(path_pw, 'pathview_index.csv'), row.names = FALSE)
invisible(file.copy(file.path(path_pw, 'pathview_index.csv'),
          file.path(path_project, 'refqc', 'stage6_pathview_index.csv'), overwrite = TRUE))
cat(sprintf("\n図: %s （ORA 版 %d 枚 + 全遺伝子版 %d 枚）\n", dir_pv,
            sum(pv_list$stat_ora == 'OK'), sum(pv_list$stat_all == 'OK')))
cat(sprintf("索引: %s/pathview_index.csv → project に refqc/stage6_pathview_index.csv\n", path_pw))
cat(sprintf("作業ディレクトリを戻した: %s\n", getwd()))


■ Table 1: 描く経路 39 行 × 2 枚 = 78 枚（重複なしの経路は 22）


scheme,direction,n
<chr>,<chr>,<int>
A,down,18
A,up,1
B,down,18
B,up,2



出力先: /mnt/scratch/tanoue/261016_Xiaoxi_eptAoligo_2wFeces_Metatranscriptome/pathway/pathview


Note: Only KEGG ortholog gene ID is supported, make sure it looks like "K01488"!

Note: Mapping via KEGG gene ID (not Entrez) is supported for this species,
it looks like "K01488"!

Info: Working in directory /mnt/scratch/tanoue/261016_Xiaoxi_eptAoligo_2wFeces_Metatranscriptome/pathway/pathview

Info: Writing image file ko03010.A_ORA.png

Note: Only KEGG ortholog gene ID is supported, make sure it looks like "K01488"!

Note: Mapping via KEGG gene ID (not Entrez) is supported for this species,
it looks like "K01488"!

Info: Working in directory /mnt/scratch/tanoue/261016_Xiaoxi_eptAoligo_2wFeces_Metatranscriptome/pathway/pathview

Info: Writing image file ko03010.A_all.png

Note: Only KEGG ortholog gene ID is supported, make sure it looks like "K01488"!

Note: Mapping via KEGG gene ID (not Entrez) is supported for this species,
it looks like "K01488"!

Info: Working in directory /mnt/scratch/tanoue/261016_Xiaoxi_eptAoligo_2wFeces_Metatranscriptome/pathway/pathview

Info: Writing image f


■ Table 2: 描画の結果（ORA = geneID のみ / all = 経路の全遺伝子）


scheme,direction,term,K_ORA,判定_ORA,K_all,判定_all
<chr>,<chr>,<chr>,<int>,<chr>,<int>,<chr>
A,up,ko03010_Ribosome,2,OK,54,OK
A,down,ko00195_Photosynthesis,8,OK,8,OK
A,down,ko00030_Pentose phosphate pathway,15,OK,26,OK
A,down,ko00480_Glutathione metabolism,5,OK,5,OK
A,down,ko04016_MAPK signaling pathway - plant,3,OK,3,OK
A,down,ko04068_FoxO signaling pathway,2,OK,2,OK
A,down,ko00020_Citrate cycle (TCA cycle),12,OK,26,OK
A,down,ko00450_Selenocompound metabolism,3,OK,9,OK
A,down,ko00121_Secondary bile acid biosynthesis,3,ERROR,3,ERROR


... (144 outputs left out)

   (2) 経路に属する全遺伝子  K 8 個


plot without title


図: /mnt/scratch/tanoue/261016_Xiaoxi_eptAoligo_2wFeces_Metatranscriptome/pathway/pathview （ORA 版 37 枚 + 全遺伝子版 37 枚）
索引: /mnt/scratch/tanoue/261016_Xiaoxi_eptAoligo_2wFeces_Metatranscriptome/pathway/pathview_index.csv → project に refqc/stage6_pathview_index.csv
作業ディレクトリを戻した: /home


plot without title

### 考察: 経路名からは見えなかった「中での向きの混在」が見える

#### 1. ★★ `ko00190_Oxidative phosphorylation` は「低下」ではない

★ **pathview は K 番号ごとに4株の LFC を平均する**（下記 6）ので、**色だけで向きを決めてはいけない。**
遺伝子ごとの実測は次のとおり。

| 複合体 | 遺伝子 | 実測（`LFC_A`） | 読み |
|---|---|---|---|
| **F-type ATPase** `atpA`〜`atpH` | 18 | **18本すべて負。4株すべて**（−0.43〜−1.79） | **低下。4株共通** |
| **NADH dehydrogenase** `nuoA`〜`nuoN` | 17 | 有意は **`Bun_8492` の `nuoA` +1.25 / `nuoB` +0.89 / `nuoC` +0.75 だけ** | **上昇は `Bun_8492` 限定** |
| **Sdh / Frd** | 6 | **`Bun_8492` の3本のみ** +1.03〜+1.42（padj 1e-6〜1e-8） | **上昇は `Bun_8492` 限定** |
| **Cytochrome bd** `cydA`/`cydB` | 4 | −0.24 / 0.00 / −0.27 / −0.64 | **★ 上昇していない** |

**4株そろって起きているのは F-type ATPase の低下だけ。**図で Nuo と Sdh が赤いのは
**4株平均が `Bun_8492` に引っ張られている**ため。
これは **GSEA で `ko00190` の NES が −1.46 にとどまった**（セル8 考察 4）理由をそのまま説明する。

→ **ORA の「酸化的リン酸化が低下」は誤り。**正しくは
「**F-type ATPase の構成遺伝子が4株すべてで低下し、呼吸鎖側は `Bun_8492` でのみ上昇**」。

★ **ただし「ATP 合成が低下した」とは書けない。** F型ATPase は**合成と加水分解の両方向に回る
可逆機械**で、向きを決めるのは **PMF と ATP の位相化学ポテンシャルの綱引き（熱力学）**であって
転写量ではない。向きの制御に関わる **ε サブユニット（`atpC`）**も、`Bun_8492` で8本中**最も下がらず**
（−0.43）、`Odo_St23` で**最も下がる**（−1.71）と**株で順位が逆**で一貫しない。
`Bth` 2株は **γ（`atpG`）1本しか載っていない**（mappability の制約、§3）。
**書けるのは「F型ATPase の構成遺伝子の転写が低下した」までで、向きは本データでは決まらない。**

#### 2. `ko00020_Citrate cycle` も混在 — ただし単一の読みは取り下げる

**酸化方向の低下は3株で一貫している。**`prpC`（St8 −2.25 / VPI −2.01 / Bun −1.12）、
`acnA`（−1.84 / −1.49 / −0.68）、`icd`（−1.52 / −1.85 / −1.05）、いずれも padj < 1e-3。
※ `Odo_St23` はこの3酵素がアノテーションに載っていない。

**一方、赤（上昇）側は株ごとに別の現象。**

| 酵素 | 実測 | 注意 |
|---|---|---|
| `1.3.5.x` Frd/Sdh | **`Bun_8492` の3本のみ** | `Odo_St23` の3本は n.s. |
| `4.2.1.2` fumarase | **`Odo_St23` の `CE91St23_17090` +3.52 / `17100` +3.27 のみ** | ★ **CLAUDE.md §8 の Odo 座位 `17090`–`17280`（KO 高）の先頭そのもの。**しかも K 番号が `K01677`（fumarase）と `K03779`（L-tartrate dehydratase）の**両方**でフマラーゼと確定していない |

→ **「酸化的 TCA を縮小し還元的分岐に寄せた」という単一の読みは取り下げる。**
還元方向の「上昇」は `Bun_8492` の Frd/Sdh と `Odo_St23` の座位内遺伝子という**別々の事象**で、
後者は §8 の座位の問題そのもの。**座位の定義が決まるまで経路の所見として扱わない。**

#### 3. `ko00030_Pentose phosphate pathway` — PRPP への接続

酸化的 PPP の入口から非酸化的分岐までほぼ全部が青（`3.1.1.31` が最も濃い）。
**上昇は `2.7.1.90`（PPi-PFK）と `2.7.6.1`（PRPP synthetase）の2つだけ。**
PRPP はプリン・ピリミジン合成の出発物質で、**GSEA で `ko00230_Purine metabolism` が
(A)(B) 両方で up だった**ことと整合 → **「NADPH 産生は下げ、核酸合成へ炭素を回している」。**
一覧表からは見えなかった経路間の接続。

#### 4. 真核経路の正体が図で確定する

`ko04068_FoxO` は **`Catalase` と `MnSOD` の2つだけ**、`ko04146_Peroxisome` は `sodB`/`katA`/`icd`、
`ko00195_Photosynthesis` は **F-type ATPase の行のみ**、`ko01051_ansamycins` は transketolase 1個のみ。
**マップの大部分が無色**なので、**「この経路が動いた」ではなく「この経路の中の数個の酵素が動いた」**
であることが一目で分かる。

#### 5. 胆汁酸は経路名が誤解を招く

`ko00120` で色がつくのは **`3.5.1.24`（BSH）の4箇所だけ**で、**マップ上流はすべて宿主（肝臓）の酵素。
本データはマップ上 18 KO のうち 1 種しか持たない。細菌は一次胆汁酸を合成しない。**
実体は**宿主が作った胆汁酸の脱抱合が減っていること。**
`ko00121` は描画に失敗したが、**15 KO のうち本データが持つのは 2 種**（`K01442` BSH と
`K00076` 7α-HSDH）のみで、**`bai` オペロンは1つも持たない。**描けてもほぼ空白になる。

#### 6. この図で読めないこと / (1)(2) を並べる意味

- **どの株の遺伝子か分からない。**K 番号ごとに4株平均なので**株で向きが逆だと相殺されて灰色**になる
  → 株の内訳はセル9 の横棒とセル6 Table 4 を見る
- **有意かどうかは色に入っていない**（有意でない遺伝子も同じように塗られる）
- **(1) ORA の `geneID` のみ**は「ORA がどの反応を拾ったか」が分かるが**逆向きの反応が見えない**、
  **(2) 全遺伝子**は**向きの混在が見える**が ORA がどれを拾ったかは分からない。
  `ko00030` で (2) が赤かった `prs` +1.29 / `pfp` +1.34 は、**(A) down の `geneID` には
  原理的に入らない**ので (1) では白になる

#### 7. (A) と (B) の図はほぼ同じ

色のつき方がほとんど変わらない = **組成シフトを除いても同じ結論**（セル8 考察 1 の裏づけ）。

#### 保存

`pathway/pathview/`（ORA 版 37枚 + 全遺伝子版 37枚 + KEGG の XML/PNG）／
索引 `pathway/pathview_index.csv` → project の `refqc/stage6_pathview_index.csv`


## セル12: 経路ごとの遺伝子発現 棒グラフ（株ごと・有意差つき）

過去notebook（`4_DESeq2_allmapped_fractionmode_R.ipynb` セル150-153、175-188）の
`ggbarplot` / `geom_bar` + `geom_quasirandom` に対応。

### ★ 過去notebook との違い

|- | 過去notebook | 本notebook |
|---|---|---|
| 単位 | 株ごとに全遺伝子をゲノム順 | **経路ごとに絞り、遺伝子1つを1パネル** |
| ラベル | 上位40個に `geom_label_repel` | **全パネルに名前・`gene_id`・K番号・EC** |
| 区別 | DEG かどうかを棒の色で | **パネル名の文字色で3分類** |
| 統計 | なし | **SD のエラーバー ＋ Welch の t 検定** |

### パネル名の文字色で3分類

**ORA hit**（橙 `#d95f02`、セル6 が拾った）／**DEG only**（紫 `#7570b3`、`padj < 0.05` だが
ORA には拾われなかった）／**not DEG**（黒）。赤・青は棒の群分け（WT/KO）に使うので別の色を当て、
`strip` 背景は橙・紫が沈まないよう **`grey97`** まで薄くする。

★ **なぜ3分類が要るか。** ORA は**向きごとに別の検定**なので、**down の term では
「有意に上昇した遺伝子」が `geneID` に入らない。**`ko00030` の `prs`（2.7.6.1）と `pfp`（2.7.1.90）が
その例で、**どちらも `Bun_8492` で有意に上昇しているのに ORA (A) down では見えない。**

### パネル名は3行（名前+`gene_id` / K番号 / EC）

**K番号と EC を同じ行に置くと、EC が複数ある遺伝子で左右が切れる**（`gnd` は `EC 1.1.1.343,1.1.1.44`）。

### ★ 検定は Welch の t 検定

R の `t.test()` は**既定が `var.equal = FALSE`** なので Student の t 検定ではない
（n = 6 vs 5 で分散も揃わないためこれが適切）。
**DESeq2 の padj とは別の検定**（負の二項分布 vs 正規分布）で、**多重比較の補正はしていない。**

### ★ (A) と (B) の両方を作る

(A)（図12-1〜12-7）は4株まとめて1組の size factor = **菌量の変化を含む**、
(B)（図12-8〜12-14）は株ごとに別々 = **含まない。**
`Bun_8492` は WT 54.8% → KO 65.1% なので **(A) では `Bun` の全遺伝子が一律 1.26 倍 底上げされる。**
並べると**どこまでが菌量でどこからが発現制御か**が分かる。
**DEG の判定もスキームに合わせる**（(A) は `padj_A`、(B) は `padj_pooled`）。

★ **`ko03010_Ribosome` は (B) の ORA では有意でない**ので図12-14 は **ORA hit が 0 個**になる。
これ自体が「リボソームの上昇は (A) でしか出ない」という情報。

### 1経路 = 1図 / 1セル = 1図

`patchwork` で4株を縦に積み、**高さを遺伝子数に比例**させる（1行 12 パネル、凡例は集約）。

★ **Jupyter は `cat()` の文字出力をひとまとめにしてから図を出す**ので、1セルで7経路を回すと
**「見出しが7行並んだ後に図が7枚続く」形**になり対応が取れない。→ **1セル = 1図**にする。
このセルは共通の設定だけを作り、**描画は図12-1 〜 12-14。**

### ★ 図中に日本語を入れない

**PNG 保存時に `mbcsToSbcs` 変換が失敗して文字化けする**（実測）。日本語の経路名は `cat()` 側に置く。
副題の `Welch's` も `'`（U+2019）が同じ警告を出すので **`Welch two-sample t-test`** と書く。

### 描く図（7経路 × 2スキーム = 14 枚）

`ko00030` ペントースリン酸 / `ko00020` TCA / `ko00710` 炭素固定(Calvin) / `ko00720` その他の炭素固定 /
`ko00120` 一次胆汁酸 / `ko00121` 二次胆汁酸 / `ko03010` リボソーム。


In [56]:
# ── 経路ごとの遺伝子発現 棒グラフ（準備）──────────────────────
# 過去notebook（4_DESeq2_allmapped_fractionmode_R.ipynb セル150-153、175-188）の
# ggbarplot / geom_bar + geom_quasirandom に対応。
#
# ★ 経路ごと・スキームごとにセルを分ける理由
#   Jupyter は cat() の文字出力をひとまとめにしてから図を出すので、
#   1セルで複数回すと「見出しが並んだ後に図が続く」形になり対応が取れない。
#   → 1セル = 1図 にして、見出しと図が必ず隣り合うようにする
library(ggplot2); library(ggbeeswarm); library(ggtext); library(patchwork)

# ★ (A) と (B) の正規化カウントを両方読む
#   (A) 4株をまとめて1組の size factor  … 群集としての出力。菌量の変化を含む
#   (B) 株ごとに別々の size factor      … 株内の発現制御。菌量の変化を含まない
nc_A = read.csv(file.path(path_de, 'A_normalized_counts.csv'), row.names = 1, check.names = FALSE)
nc_B = read.csv(file.path(path_de, 'B_normalized_counts.csv'), row.names = 1, check.names = FALSE)
cat(sprintf("A_normalized_counts: %d 遺伝子 / B_normalized_counts: %d 遺伝子\n",
            nrow(nc_A), nrow(nc_B)))

# パネル名の文字色。★ 赤・青は棒の群分け（WT/KO）に使うので別の色にする
col_cat  = c('ORA hit' = '#d95f02', 'DEG only' = '#7570b3', 'not DEG' = '#000000')
strains4 = c('Bth_VPI', 'Bth_St8', 'Bun_8492', 'Odo_St23')
NCOL     = 12   # 1行に並べるパネル数

dir_gb = file.path(path_pw, 'genebar')
if (!dir.exists(dir_gb)) dir.create(dir_gb, recursive = TRUE)

cat("\n■ Table 1: これから描く図（7経路 × 2スキーム = 14 枚）\n")
data.frame(
    ko   = rep(c('ko00030','ko00020','ko00710','ko00720','ko00120','ko00121','ko03010'), 2),
    経路 = rep(c('ペントースリン酸経路','TCA回路','炭素固定(Calvin)','その他の炭素固定',
                 '一次胆汁酸','二次胆汁酸','リボソーム'), 2),
    scheme = rep(c('A','B'), each = 7),
    セル = c(paste0('12-', 1:7), paste0('12-', 8:14)),
    check.names = FALSE) %>%
    left_join(ora %>% filter(source == 'KEGG', padj < 0.05) %>%
                  transmute(ko = sub('_.*', '', term), scheme,
                            ORAで有意 = sprintf('%s (padj %.1e)', direction, padj)),
              by = c('ko', 'scheme')) %>%
    mutate(ORAで有意 = ifelse(is.na(ORAで有意), '★ 有意でない', ORAで有意))


A_normalized_counts: 9795 遺伝子 / B_normalized_counts: 9785 遺伝子

■ Table 1: これから描く図（7経路 × 2スキーム = 14 枚）


ko,経路,scheme,セル,ORAで有意
<chr>,<chr>,<chr>,<chr>,<chr>
ko00030,ペントースリン酸経路,A,12-1,down (padj 4.1e-06)
ko00020,TCA回路,A,12-2,down (padj 9.6e-04)
ko00710,炭素固定(Calvin),A,12-3,down (padj 2.0e-02)
ko00720,その他の炭素固定,A,12-4,down (padj 2.0e-02)
ko00120,一次胆汁酸,A,12-5,down (padj 3.0e-02)
ko00121,二次胆汁酸,A,12-6,down (padj 2.4e-03)
ko03010,リボソーム,A,12-7,up (padj 8.9e-03)
ko00030,ペントースリン酸経路,B,12-8,down (padj 2.2e-07)
ko00020,TCA回路,B,12-9,down (padj 6.2e-04)


In [57]:
# ── 図12-1: ko00030（ペントースリン酸経路）  scheme A ───────────────────
# セル12（準備）を先に実行しておくこと
#   （nc_A / nc_B / col_cat / strains4 / NCOL / dir_gb を使う）
ko = 'ko00030'; ko_jp = 'ペントースリン酸経路'; SCHEME = 'A'

# スキームに応じて使うものを切り替える
nc       = if (SCHEME == 'A') nc_A else nc_B
y_lab = ifelse(SCHEME == 'A',
               'normalized count  (A: pooled across all 4 strains)',
               'normalized count  (B: per-strain size factors)')
sub_norm = ifelse(SCHEME == 'A',
    paste0("**one size factor set computed from all 4 strains pooled** ",
           "(= scheme A, the community-level output; includes the shift in strain abundance)"),
    paste0("**size factors computed separately within each strain** ",
           "(= scheme B, within-strain regulation; excludes the shift in strain abundance)"))

tgt = ora$term[ora$source == 'KEGG' & grepl(paste0('^', ko, '_'), ora$term)][1]

# ORA で拾われた遺伝子（down/up のどちらでも拾われていれば ORA hit）
g_ora = ora %>% filter(scheme == SCHEME, source == 'KEGG', term == tgt, padj < 0.05) %>%
    pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()

# 経路に属する全遺伝子
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
    dplyr::select(gene_id, KEGG_Pathway) %>%
    tidyr::separate_rows(KEGG_Pathway, sep = ',') %>%
    filter(KEGG_Pathway == ko) %>% pull(gene_id) %>% unique()

cat(sprintf("%s（%s）  scheme %s  全 %d 遺伝子 / ORA hit %d%s\n",
            tgt, ko_jp, SCHEME, length(g_all), length(g_ora),
            ifelse(length(g_ora) == 0, '  ★ この経路はこのスキームの ORA では有意でない', '')))

# ★ DEG の判定もスキームに合わせる（(A) は padj_A、(B) は padj_pooled）
d = nc[intersect(g_all, rownames(nc)), , drop = FALSE] %>%
    tibble::rownames_to_column('gene_id') %>%
    tidyr::pivot_longer(-gene_id, names_to = 'sample', values_to = 'norm') %>%
    mutate(group = factor(ifelse(grepl('W', sample), 'WT', 'KO'), levels = c('WT','KO'))) %>%
    left_join(res %>% dplyr::select(gene_id, strain, Preferred_name, product,
                                    KEGG_ko, EC, padj_A, padj_pooled), by = 'gene_id') %>%
    mutate(padj_use = if (SCHEME == 'A') padj_A else padj_pooled,
           cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                           !is.na(padj_use) & padj_use < 0.05 ~ 'DEG only',
                           TRUE ~ 'not DEG'),
           cat = factor(cat, levels = names(col_cat)),
           label = ifelse(is.na(Preferred_name) | Preferred_name == '-',
                          substr(product, 1, 14), Preferred_name),
           K   = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           EC2 = sapply(strsplit(ifelse(is.na(EC), '-', EC), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           # ★ 3行にする（名前+gene_id / K番号 / EC）。
           #   1行にまとめると EC が複数ある遺伝子で左右が切れる
           panel = sprintf("<span style='color:%s'>**%s  %s<br>%s<br>EC %s**</span>",
                           col_cat[as.character(cat)], label, gene_id, K, EC2))

# ★ 検定は Welch の t 検定（R の t.test は既定が var.equal = FALSE）。
#   等分散を仮定する Student の t 検定ではない。n = 6 vs 5 で分散も揃わないため適切。
#   ★ DESeq2 の padj とは別の検定（負の二項分布 vs 正規分布）。多重比較の補正はしていない
tt = d %>% group_by(gene_id, strain, panel) %>%
    summarise(p = tryCatch(t.test(norm ~ group)$p.value, error = function(e) NA_real_),
              ymax = max(max(norm), max(tapply(norm, group, function(v) mean(v) + sd(v)))),
              .groups = 'drop') %>%
    mutate(star = cut(p, c(-Inf, .001, .01, .05, Inf), labels = c('***','**','*','n.s.')),
           star = ifelse(is.na(star), 'n.s.', as.character(star)))

# 株ごとのサブプロットを作り、patchwork で1枚にまとめる
subs = list(); hts = c()
for (s in strains4) {
    ds = d %>% filter(strain == s) %>% arrange(cat, label)
    if (nrow(ds) == 0) next
    ds$panel = factor(ds$panel, levels = unique(ds$panel))   # 分類順に並べる
    ts = tt %>% filter(strain == s) %>% mutate(panel = factor(panel, levels = levels(ds$panel)))
    n_gene = length(unique(ds$gene_id)); n_row = ceiling(n_gene / NCOL)
    subs[[s]] = ggplot(ds, aes(x = group, y = norm, fill = group)) +
        geom_bar(stat = 'summary', fun = 'mean', width = 0.65, colour = 'grey30', linewidth = 0.3) +
        # ★ SD のエラーバー（平均 ± SD の両側）。上側だけにすると errorbar が両端に
        #   キャップを描き、下のキャップが棒の上辺（＝平均）に乗って紛らわしい
        stat_summary(fun = mean,
                     fun.min = function(v) mean(v) - sd(v),
                     fun.max = function(v) mean(v) + sd(v),
                     geom = 'errorbar', width = 0.25, linewidth = 0.4, colour = 'grey20') +
        geom_quasirandom(width = 0.15, size = 1.3, alpha = 0.85, colour = 'grey20') +
        geom_text(data = ts, aes(x = 1.5, y = ymax * 1.18, label = star),
                  inherit.aes = FALSE, size = 3.6) +
        facet_wrap(~ panel, scales = 'free_y', ncol = NCOL) +
        scale_fill_manual(values = c(WT = 'grey75', KO = '#e05252'), name = NULL) +
        scale_y_continuous(expand = expansion(mult = c(0, 0.25))) +
        labs(title = sprintf('%s   (%d genes)', s, n_gene), x = NULL, y = y_lab) +
        theme_bw() +
        # ★ strip の灰色が濃いと橙・紫の文字が沈むので、ほぼ白まで薄くする
        theme(strip.background = element_rect(fill='grey97', colour='grey75', linewidth=0.3),
              strip.text = element_markdown(size = 9),
              plot.title = element_text(size = 13, face = 'bold'),
              axis.text = element_text(size = 7.5),
              axis.title.y = element_text(size = 9),
              panel.grid.major.x = element_blank())
    hts = c(hts, n_row + 0.45)   # 株の見出し分を足す
}

P = wrap_plots(subs, ncol = 1, heights = hts) +
    plot_layout(guides = 'collect') +
    # ★ 図中に日本語を入れない（PNG 保存時に mbcsToSbcs 変換が失敗して文字化けする）。
    #   日本語の経路名は上の cat() でテキストとして出す
    plot_annotation(title = sprintf('%s   [scheme %s]', tgt, SCHEME),
      subtitle = paste0(
        "panel name:  <span style='color:#d95f02'>**ORA hit**</span>  /  ",
        "<span style='color:#7570b3'>**DEG only**</span>  /  ",
        "<span style='color:#000000'>**not DEG**</span><br>",
        "y axis = **normalized count** - DESeq2 median-of-ratios with ", sub_norm, "<br>",
        "bar = mean, error bar = \u00b1SD, points = each mouse  |  ",
        "significance = **Welch two-sample t-test** on normalized counts ",
        "(*** p<0.001, ** p<0.01, * p<0.05; not DESeq2 padj, no multiple-testing correction)"),
      theme = theme(plot.title = element_text(size = 15, face = 'bold'),
                    plot.subtitle = element_markdown(size = 10, lineheight = 1.3)))

suppressMessages(ggsave(file.path(dir_gb, sprintf('genebar_%s_%s.png', ko, SCHEME)), P,
                        width = 24, height = 2.1 * sum(hts) + 1.4, dpi = 100, limitsize = FALSE))
options(repr.plot.width = 24, repr.plot.height = 2.1 * sum(hts) + 1.4)
P


ko00030_Pentose phosphate pathway（ペントースリン酸経路）  scheme A  全 69 遺伝子 / ORA hit 31


plot without title

In [58]:
# ── 図12-2: ko00020（TCA回路）  scheme A ───────────────────
# セル12（準備）を先に実行しておくこと
#   （nc_A / nc_B / col_cat / strains4 / NCOL / dir_gb を使う）
ko = 'ko00020'; ko_jp = 'TCA回路'; SCHEME = 'A'

# スキームに応じて使うものを切り替える
nc       = if (SCHEME == 'A') nc_A else nc_B
y_lab = ifelse(SCHEME == 'A',
               'normalized count  (A: pooled across all 4 strains)',
               'normalized count  (B: per-strain size factors)')
sub_norm = ifelse(SCHEME == 'A',
    paste0("**one size factor set computed from all 4 strains pooled** ",
           "(= scheme A, the community-level output; includes the shift in strain abundance)"),
    paste0("**size factors computed separately within each strain** ",
           "(= scheme B, within-strain regulation; excludes the shift in strain abundance)"))

tgt = ora$term[ora$source == 'KEGG' & grepl(paste0('^', ko, '_'), ora$term)][1]

# ORA で拾われた遺伝子（down/up のどちらでも拾われていれば ORA hit）
g_ora = ora %>% filter(scheme == SCHEME, source == 'KEGG', term == tgt, padj < 0.05) %>%
    pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()

# 経路に属する全遺伝子
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
    dplyr::select(gene_id, KEGG_Pathway) %>%
    tidyr::separate_rows(KEGG_Pathway, sep = ',') %>%
    filter(KEGG_Pathway == ko) %>% pull(gene_id) %>% unique()

cat(sprintf("%s（%s）  scheme %s  全 %d 遺伝子 / ORA hit %d%s\n",
            tgt, ko_jp, SCHEME, length(g_all), length(g_ora),
            ifelse(length(g_ora) == 0, '  ★ この経路はこのスキームの ORA では有意でない', '')))

# ★ DEG の判定もスキームに合わせる（(A) は padj_A、(B) は padj_pooled）
d = nc[intersect(g_all, rownames(nc)), , drop = FALSE] %>%
    tibble::rownames_to_column('gene_id') %>%
    tidyr::pivot_longer(-gene_id, names_to = 'sample', values_to = 'norm') %>%
    mutate(group = factor(ifelse(grepl('W', sample), 'WT', 'KO'), levels = c('WT','KO'))) %>%
    left_join(res %>% dplyr::select(gene_id, strain, Preferred_name, product,
                                    KEGG_ko, EC, padj_A, padj_pooled), by = 'gene_id') %>%
    mutate(padj_use = if (SCHEME == 'A') padj_A else padj_pooled,
           cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                           !is.na(padj_use) & padj_use < 0.05 ~ 'DEG only',
                           TRUE ~ 'not DEG'),
           cat = factor(cat, levels = names(col_cat)),
           label = ifelse(is.na(Preferred_name) | Preferred_name == '-',
                          substr(product, 1, 14), Preferred_name),
           K   = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           EC2 = sapply(strsplit(ifelse(is.na(EC), '-', EC), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           # ★ 3行にする（名前+gene_id / K番号 / EC）。
           #   1行にまとめると EC が複数ある遺伝子で左右が切れる
           panel = sprintf("<span style='color:%s'>**%s  %s<br>%s<br>EC %s**</span>",
                           col_cat[as.character(cat)], label, gene_id, K, EC2))

# ★ 検定は Welch の t 検定（R の t.test は既定が var.equal = FALSE）。
#   等分散を仮定する Student の t 検定ではない。n = 6 vs 5 で分散も揃わないため適切。
#   ★ DESeq2 の padj とは別の検定（負の二項分布 vs 正規分布）。多重比較の補正はしていない
tt = d %>% group_by(gene_id, strain, panel) %>%
    summarise(p = tryCatch(t.test(norm ~ group)$p.value, error = function(e) NA_real_),
              ymax = max(max(norm), max(tapply(norm, group, function(v) mean(v) + sd(v)))),
              .groups = 'drop') %>%
    mutate(star = cut(p, c(-Inf, .001, .01, .05, Inf), labels = c('***','**','*','n.s.')),
           star = ifelse(is.na(star), 'n.s.', as.character(star)))

# 株ごとのサブプロットを作り、patchwork で1枚にまとめる
subs = list(); hts = c()
for (s in strains4) {
    ds = d %>% filter(strain == s) %>% arrange(cat, label)
    if (nrow(ds) == 0) next
    ds$panel = factor(ds$panel, levels = unique(ds$panel))   # 分類順に並べる
    ts = tt %>% filter(strain == s) %>% mutate(panel = factor(panel, levels = levels(ds$panel)))
    n_gene = length(unique(ds$gene_id)); n_row = ceiling(n_gene / NCOL)
    subs[[s]] = ggplot(ds, aes(x = group, y = norm, fill = group)) +
        geom_bar(stat = 'summary', fun = 'mean', width = 0.65, colour = 'grey30', linewidth = 0.3) +
        # ★ SD のエラーバー（平均 ± SD の両側）。上側だけにすると errorbar が両端に
        #   キャップを描き、下のキャップが棒の上辺（＝平均）に乗って紛らわしい
        stat_summary(fun = mean,
                     fun.min = function(v) mean(v) - sd(v),
                     fun.max = function(v) mean(v) + sd(v),
                     geom = 'errorbar', width = 0.25, linewidth = 0.4, colour = 'grey20') +
        geom_quasirandom(width = 0.15, size = 1.3, alpha = 0.85, colour = 'grey20') +
        geom_text(data = ts, aes(x = 1.5, y = ymax * 1.18, label = star),
                  inherit.aes = FALSE, size = 3.6) +
        facet_wrap(~ panel, scales = 'free_y', ncol = NCOL) +
        scale_fill_manual(values = c(WT = 'grey75', KO = '#e05252'), name = NULL) +
        scale_y_continuous(expand = expansion(mult = c(0, 0.25))) +
        labs(title = sprintf('%s   (%d genes)', s, n_gene), x = NULL, y = y_lab) +
        theme_bw() +
        # ★ strip の灰色が濃いと橙・紫の文字が沈むので、ほぼ白まで薄くする
        theme(strip.background = element_rect(fill='grey97', colour='grey75', linewidth=0.3),
              strip.text = element_markdown(size = 9),
              plot.title = element_text(size = 13, face = 'bold'),
              axis.text = element_text(size = 7.5),
              axis.title.y = element_text(size = 9),
              panel.grid.major.x = element_blank())
    hts = c(hts, n_row + 0.45)   # 株の見出し分を足す
}

P = wrap_plots(subs, ncol = 1, heights = hts) +
    plot_layout(guides = 'collect') +
    # ★ 図中に日本語を入れない（PNG 保存時に mbcsToSbcs 変換が失敗して文字化けする）。
    #   日本語の経路名は上の cat() でテキストとして出す
    plot_annotation(title = sprintf('%s   [scheme %s]', tgt, SCHEME),
      subtitle = paste0(
        "panel name:  <span style='color:#d95f02'>**ORA hit**</span>  /  ",
        "<span style='color:#7570b3'>**DEG only**</span>  /  ",
        "<span style='color:#000000'>**not DEG**</span><br>",
        "y axis = **normalized count** - DESeq2 median-of-ratios with ", sub_norm, "<br>",
        "bar = mean, error bar = \u00b1SD, points = each mouse  |  ",
        "significance = **Welch two-sample t-test** on normalized counts ",
        "(*** p<0.001, ** p<0.01, * p<0.05; not DESeq2 padj, no multiple-testing correction)"),
      theme = theme(plot.title = element_text(size = 15, face = 'bold'),
                    plot.subtitle = element_markdown(size = 10, lineheight = 1.3)))

suppressMessages(ggsave(file.path(dir_gb, sprintf('genebar_%s_%s.png', ko, SCHEME)), P,
                        width = 24, height = 2.1 * sum(hts) + 1.4, dpi = 100, limitsize = FALSE))
options(repr.plot.width = 24, repr.plot.height = 2.1 * sum(hts) + 1.4)
P


ko00020_Citrate cycle (TCA cycle)（TCA回路）  scheme A  全 61 遺伝子 / ORA hit 24


plot without title

In [59]:
# ── 図12-3: ko00710（炭素固定(Calvin)）  scheme A ───────────────────
# セル12（準備）を先に実行しておくこと
#   （nc_A / nc_B / col_cat / strains4 / NCOL / dir_gb を使う）
ko = 'ko00710'; ko_jp = '炭素固定(Calvin)'; SCHEME = 'A'

# スキームに応じて使うものを切り替える
nc       = if (SCHEME == 'A') nc_A else nc_B
y_lab = ifelse(SCHEME == 'A',
               'normalized count  (A: pooled across all 4 strains)',
               'normalized count  (B: per-strain size factors)')
sub_norm = ifelse(SCHEME == 'A',
    paste0("**one size factor set computed from all 4 strains pooled** ",
           "(= scheme A, the community-level output; includes the shift in strain abundance)"),
    paste0("**size factors computed separately within each strain** ",
           "(= scheme B, within-strain regulation; excludes the shift in strain abundance)"))

tgt = ora$term[ora$source == 'KEGG' & grepl(paste0('^', ko, '_'), ora$term)][1]

# ORA で拾われた遺伝子（down/up のどちらでも拾われていれば ORA hit）
g_ora = ora %>% filter(scheme == SCHEME, source == 'KEGG', term == tgt, padj < 0.05) %>%
    pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()

# 経路に属する全遺伝子
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
    dplyr::select(gene_id, KEGG_Pathway) %>%
    tidyr::separate_rows(KEGG_Pathway, sep = ',') %>%
    filter(KEGG_Pathway == ko) %>% pull(gene_id) %>% unique()

cat(sprintf("%s（%s）  scheme %s  全 %d 遺伝子 / ORA hit %d%s\n",
            tgt, ko_jp, SCHEME, length(g_all), length(g_ora),
            ifelse(length(g_ora) == 0, '  ★ この経路はこのスキームの ORA では有意でない', '')))

# ★ DEG の判定もスキームに合わせる（(A) は padj_A、(B) は padj_pooled）
d = nc[intersect(g_all, rownames(nc)), , drop = FALSE] %>%
    tibble::rownames_to_column('gene_id') %>%
    tidyr::pivot_longer(-gene_id, names_to = 'sample', values_to = 'norm') %>%
    mutate(group = factor(ifelse(grepl('W', sample), 'WT', 'KO'), levels = c('WT','KO'))) %>%
    left_join(res %>% dplyr::select(gene_id, strain, Preferred_name, product,
                                    KEGG_ko, EC, padj_A, padj_pooled), by = 'gene_id') %>%
    mutate(padj_use = if (SCHEME == 'A') padj_A else padj_pooled,
           cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                           !is.na(padj_use) & padj_use < 0.05 ~ 'DEG only',
                           TRUE ~ 'not DEG'),
           cat = factor(cat, levels = names(col_cat)),
           label = ifelse(is.na(Preferred_name) | Preferred_name == '-',
                          substr(product, 1, 14), Preferred_name),
           K   = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           EC2 = sapply(strsplit(ifelse(is.na(EC), '-', EC), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           # ★ 3行にする（名前+gene_id / K番号 / EC）。
           #   1行にまとめると EC が複数ある遺伝子で左右が切れる
           panel = sprintf("<span style='color:%s'>**%s  %s<br>%s<br>EC %s**</span>",
                           col_cat[as.character(cat)], label, gene_id, K, EC2))

# ★ 検定は Welch の t 検定（R の t.test は既定が var.equal = FALSE）。
#   等分散を仮定する Student の t 検定ではない。n = 6 vs 5 で分散も揃わないため適切。
#   ★ DESeq2 の padj とは別の検定（負の二項分布 vs 正規分布）。多重比較の補正はしていない
tt = d %>% group_by(gene_id, strain, panel) %>%
    summarise(p = tryCatch(t.test(norm ~ group)$p.value, error = function(e) NA_real_),
              ymax = max(max(norm), max(tapply(norm, group, function(v) mean(v) + sd(v)))),
              .groups = 'drop') %>%
    mutate(star = cut(p, c(-Inf, .001, .01, .05, Inf), labels = c('***','**','*','n.s.')),
           star = ifelse(is.na(star), 'n.s.', as.character(star)))

# 株ごとのサブプロットを作り、patchwork で1枚にまとめる
subs = list(); hts = c()
for (s in strains4) {
    ds = d %>% filter(strain == s) %>% arrange(cat, label)
    if (nrow(ds) == 0) next
    ds$panel = factor(ds$panel, levels = unique(ds$panel))   # 分類順に並べる
    ts = tt %>% filter(strain == s) %>% mutate(panel = factor(panel, levels = levels(ds$panel)))
    n_gene = length(unique(ds$gene_id)); n_row = ceiling(n_gene / NCOL)
    subs[[s]] = ggplot(ds, aes(x = group, y = norm, fill = group)) +
        geom_bar(stat = 'summary', fun = 'mean', width = 0.65, colour = 'grey30', linewidth = 0.3) +
        # ★ SD のエラーバー（平均 ± SD の両側）。上側だけにすると errorbar が両端に
        #   キャップを描き、下のキャップが棒の上辺（＝平均）に乗って紛らわしい
        stat_summary(fun = mean,
                     fun.min = function(v) mean(v) - sd(v),
                     fun.max = function(v) mean(v) + sd(v),
                     geom = 'errorbar', width = 0.25, linewidth = 0.4, colour = 'grey20') +
        geom_quasirandom(width = 0.15, size = 1.3, alpha = 0.85, colour = 'grey20') +
        geom_text(data = ts, aes(x = 1.5, y = ymax * 1.18, label = star),
                  inherit.aes = FALSE, size = 3.6) +
        facet_wrap(~ panel, scales = 'free_y', ncol = NCOL) +
        scale_fill_manual(values = c(WT = 'grey75', KO = '#e05252'), name = NULL) +
        scale_y_continuous(expand = expansion(mult = c(0, 0.25))) +
        labs(title = sprintf('%s   (%d genes)', s, n_gene), x = NULL, y = y_lab) +
        theme_bw() +
        # ★ strip の灰色が濃いと橙・紫の文字が沈むので、ほぼ白まで薄くする
        theme(strip.background = element_rect(fill='grey97', colour='grey75', linewidth=0.3),
              strip.text = element_markdown(size = 9),
              plot.title = element_text(size = 13, face = 'bold'),
              axis.text = element_text(size = 7.5),
              axis.title.y = element_text(size = 9),
              panel.grid.major.x = element_blank())
    hts = c(hts, n_row + 0.45)   # 株の見出し分を足す
}

P = wrap_plots(subs, ncol = 1, heights = hts) +
    plot_layout(guides = 'collect') +
    # ★ 図中に日本語を入れない（PNG 保存時に mbcsToSbcs 変換が失敗して文字化けする）。
    #   日本語の経路名は上の cat() でテキストとして出す
    plot_annotation(title = sprintf('%s   [scheme %s]', tgt, SCHEME),
      subtitle = paste0(
        "panel name:  <span style='color:#d95f02'>**ORA hit**</span>  /  ",
        "<span style='color:#7570b3'>**DEG only**</span>  /  ",
        "<span style='color:#000000'>**not DEG**</span><br>",
        "y axis = **normalized count** - DESeq2 median-of-ratios with ", sub_norm, "<br>",
        "bar = mean, error bar = \u00b1SD, points = each mouse  |  ",
        "significance = **Welch two-sample t-test** on normalized counts ",
        "(*** p<0.001, ** p<0.01, * p<0.05; not DESeq2 padj, no multiple-testing correction)"),
      theme = theme(plot.title = element_text(size = 15, face = 'bold'),
                    plot.subtitle = element_markdown(size = 10, lineheight = 1.3)))

suppressMessages(ggsave(file.path(dir_gb, sprintf('genebar_%s_%s.png', ko, SCHEME)), P,
                        width = 24, height = 2.1 * sum(hts) + 1.4, dpi = 100, limitsize = FALSE))
options(repr.plot.width = 24, repr.plot.height = 2.1 * sum(hts) + 1.4)
P


ko00710_Carbon fixation by Calvin cycle（炭素固定(Calvin)）  scheme A  全 40 遺伝子 / ORA hit 15


plot without title

In [60]:
# ── 図12-4: ko00720（その他の炭素固定）  scheme A ───────────────────
# セル12（準備）を先に実行しておくこと
#   （nc_A / nc_B / col_cat / strains4 / NCOL / dir_gb を使う）
ko = 'ko00720'; ko_jp = 'その他の炭素固定'; SCHEME = 'A'

# スキームに応じて使うものを切り替える
nc       = if (SCHEME == 'A') nc_A else nc_B
y_lab = ifelse(SCHEME == 'A',
               'normalized count  (A: pooled across all 4 strains)',
               'normalized count  (B: per-strain size factors)')
sub_norm = ifelse(SCHEME == 'A',
    paste0("**one size factor set computed from all 4 strains pooled** ",
           "(= scheme A, the community-level output; includes the shift in strain abundance)"),
    paste0("**size factors computed separately within each strain** ",
           "(= scheme B, within-strain regulation; excludes the shift in strain abundance)"))

tgt = ora$term[ora$source == 'KEGG' & grepl(paste0('^', ko, '_'), ora$term)][1]

# ORA で拾われた遺伝子（down/up のどちらでも拾われていれば ORA hit）
g_ora = ora %>% filter(scheme == SCHEME, source == 'KEGG', term == tgt, padj < 0.05) %>%
    pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()

# 経路に属する全遺伝子
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
    dplyr::select(gene_id, KEGG_Pathway) %>%
    tidyr::separate_rows(KEGG_Pathway, sep = ',') %>%
    filter(KEGG_Pathway == ko) %>% pull(gene_id) %>% unique()

cat(sprintf("%s（%s）  scheme %s  全 %d 遺伝子 / ORA hit %d%s\n",
            tgt, ko_jp, SCHEME, length(g_all), length(g_ora),
            ifelse(length(g_ora) == 0, '  ★ この経路はこのスキームの ORA では有意でない', '')))

# ★ DEG の判定もスキームに合わせる（(A) は padj_A、(B) は padj_pooled）
d = nc[intersect(g_all, rownames(nc)), , drop = FALSE] %>%
    tibble::rownames_to_column('gene_id') %>%
    tidyr::pivot_longer(-gene_id, names_to = 'sample', values_to = 'norm') %>%
    mutate(group = factor(ifelse(grepl('W', sample), 'WT', 'KO'), levels = c('WT','KO'))) %>%
    left_join(res %>% dplyr::select(gene_id, strain, Preferred_name, product,
                                    KEGG_ko, EC, padj_A, padj_pooled), by = 'gene_id') %>%
    mutate(padj_use = if (SCHEME == 'A') padj_A else padj_pooled,
           cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                           !is.na(padj_use) & padj_use < 0.05 ~ 'DEG only',
                           TRUE ~ 'not DEG'),
           cat = factor(cat, levels = names(col_cat)),
           label = ifelse(is.na(Preferred_name) | Preferred_name == '-',
                          substr(product, 1, 14), Preferred_name),
           K   = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           EC2 = sapply(strsplit(ifelse(is.na(EC), '-', EC), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           # ★ 3行にする（名前+gene_id / K番号 / EC）。
           #   1行にまとめると EC が複数ある遺伝子で左右が切れる
           panel = sprintf("<span style='color:%s'>**%s  %s<br>%s<br>EC %s**</span>",
                           col_cat[as.character(cat)], label, gene_id, K, EC2))

# ★ 検定は Welch の t 検定（R の t.test は既定が var.equal = FALSE）。
#   等分散を仮定する Student の t 検定ではない。n = 6 vs 5 で分散も揃わないため適切。
#   ★ DESeq2 の padj とは別の検定（負の二項分布 vs 正規分布）。多重比較の補正はしていない
tt = d %>% group_by(gene_id, strain, panel) %>%
    summarise(p = tryCatch(t.test(norm ~ group)$p.value, error = function(e) NA_real_),
              ymax = max(max(norm), max(tapply(norm, group, function(v) mean(v) + sd(v)))),
              .groups = 'drop') %>%
    mutate(star = cut(p, c(-Inf, .001, .01, .05, Inf), labels = c('***','**','*','n.s.')),
           star = ifelse(is.na(star), 'n.s.', as.character(star)))

# 株ごとのサブプロットを作り、patchwork で1枚にまとめる
subs = list(); hts = c()
for (s in strains4) {
    ds = d %>% filter(strain == s) %>% arrange(cat, label)
    if (nrow(ds) == 0) next
    ds$panel = factor(ds$panel, levels = unique(ds$panel))   # 分類順に並べる
    ts = tt %>% filter(strain == s) %>% mutate(panel = factor(panel, levels = levels(ds$panel)))
    n_gene = length(unique(ds$gene_id)); n_row = ceiling(n_gene / NCOL)
    subs[[s]] = ggplot(ds, aes(x = group, y = norm, fill = group)) +
        geom_bar(stat = 'summary', fun = 'mean', width = 0.65, colour = 'grey30', linewidth = 0.3) +
        # ★ SD のエラーバー（平均 ± SD の両側）。上側だけにすると errorbar が両端に
        #   キャップを描き、下のキャップが棒の上辺（＝平均）に乗って紛らわしい
        stat_summary(fun = mean,
                     fun.min = function(v) mean(v) - sd(v),
                     fun.max = function(v) mean(v) + sd(v),
                     geom = 'errorbar', width = 0.25, linewidth = 0.4, colour = 'grey20') +
        geom_quasirandom(width = 0.15, size = 1.3, alpha = 0.85, colour = 'grey20') +
        geom_text(data = ts, aes(x = 1.5, y = ymax * 1.18, label = star),
                  inherit.aes = FALSE, size = 3.6) +
        facet_wrap(~ panel, scales = 'free_y', ncol = NCOL) +
        scale_fill_manual(values = c(WT = 'grey75', KO = '#e05252'), name = NULL) +
        scale_y_continuous(expand = expansion(mult = c(0, 0.25))) +
        labs(title = sprintf('%s   (%d genes)', s, n_gene), x = NULL, y = y_lab) +
        theme_bw() +
        # ★ strip の灰色が濃いと橙・紫の文字が沈むので、ほぼ白まで薄くする
        theme(strip.background = element_rect(fill='grey97', colour='grey75', linewidth=0.3),
              strip.text = element_markdown(size = 9),
              plot.title = element_text(size = 13, face = 'bold'),
              axis.text = element_text(size = 7.5),
              axis.title.y = element_text(size = 9),
              panel.grid.major.x = element_blank())
    hts = c(hts, n_row + 0.45)   # 株の見出し分を足す
}

P = wrap_plots(subs, ncol = 1, heights = hts) +
    plot_layout(guides = 'collect') +
    # ★ 図中に日本語を入れない（PNG 保存時に mbcsToSbcs 変換が失敗して文字化けする）。
    #   日本語の経路名は上の cat() でテキストとして出す
    plot_annotation(title = sprintf('%s   [scheme %s]', tgt, SCHEME),
      subtitle = paste0(
        "panel name:  <span style='color:#d95f02'>**ORA hit**</span>  /  ",
        "<span style='color:#7570b3'>**DEG only**</span>  /  ",
        "<span style='color:#000000'>**not DEG**</span><br>",
        "y axis = **normalized count** - DESeq2 median-of-ratios with ", sub_norm, "<br>",
        "bar = mean, error bar = \u00b1SD, points = each mouse  |  ",
        "significance = **Welch two-sample t-test** on normalized counts ",
        "(*** p<0.001, ** p<0.01, * p<0.05; not DESeq2 padj, no multiple-testing correction)"),
      theme = theme(plot.title = element_text(size = 15, face = 'bold'),
                    plot.subtitle = element_markdown(size = 10, lineheight = 1.3)))

suppressMessages(ggsave(file.path(dir_gb, sprintf('genebar_%s_%s.png', ko, SCHEME)), P,
                        width = 24, height = 2.1 * sum(hts) + 1.4, dpi = 100, limitsize = FALSE))
options(repr.plot.width = 24, repr.plot.height = 2.1 * sum(hts) + 1.4)
P


ko00720_Other carbon fixation pathways（その他の炭素固定）  scheme A  全 81 遺伝子 / ORA hit 25


plot without title

In [61]:
# ── 図12-5: ko00120（一次胆汁酸）  scheme A ───────────────────
# セル12（準備）を先に実行しておくこと
#   （nc_A / nc_B / col_cat / strains4 / NCOL / dir_gb を使う）
ko = 'ko00120'; ko_jp = '一次胆汁酸'; SCHEME = 'A'

# スキームに応じて使うものを切り替える
nc       = if (SCHEME == 'A') nc_A else nc_B
y_lab = ifelse(SCHEME == 'A',
               'normalized count  (A: pooled across all 4 strains)',
               'normalized count  (B: per-strain size factors)')
sub_norm = ifelse(SCHEME == 'A',
    paste0("**one size factor set computed from all 4 strains pooled** ",
           "(= scheme A, the community-level output; includes the shift in strain abundance)"),
    paste0("**size factors computed separately within each strain** ",
           "(= scheme B, within-strain regulation; excludes the shift in strain abundance)"))

tgt = ora$term[ora$source == 'KEGG' & grepl(paste0('^', ko, '_'), ora$term)][1]

# ORA で拾われた遺伝子（down/up のどちらでも拾われていれば ORA hit）
g_ora = ora %>% filter(scheme == SCHEME, source == 'KEGG', term == tgt, padj < 0.05) %>%
    pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()

# 経路に属する全遺伝子
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
    dplyr::select(gene_id, KEGG_Pathway) %>%
    tidyr::separate_rows(KEGG_Pathway, sep = ',') %>%
    filter(KEGG_Pathway == ko) %>% pull(gene_id) %>% unique()

cat(sprintf("%s（%s）  scheme %s  全 %d 遺伝子 / ORA hit %d%s\n",
            tgt, ko_jp, SCHEME, length(g_all), length(g_ora),
            ifelse(length(g_ora) == 0, '  ★ この経路はこのスキームの ORA では有意でない', '')))

# ★ DEG の判定もスキームに合わせる（(A) は padj_A、(B) は padj_pooled）
d = nc[intersect(g_all, rownames(nc)), , drop = FALSE] %>%
    tibble::rownames_to_column('gene_id') %>%
    tidyr::pivot_longer(-gene_id, names_to = 'sample', values_to = 'norm') %>%
    mutate(group = factor(ifelse(grepl('W', sample), 'WT', 'KO'), levels = c('WT','KO'))) %>%
    left_join(res %>% dplyr::select(gene_id, strain, Preferred_name, product,
                                    KEGG_ko, EC, padj_A, padj_pooled), by = 'gene_id') %>%
    mutate(padj_use = if (SCHEME == 'A') padj_A else padj_pooled,
           cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                           !is.na(padj_use) & padj_use < 0.05 ~ 'DEG only',
                           TRUE ~ 'not DEG'),
           cat = factor(cat, levels = names(col_cat)),
           label = ifelse(is.na(Preferred_name) | Preferred_name == '-',
                          substr(product, 1, 14), Preferred_name),
           K   = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           EC2 = sapply(strsplit(ifelse(is.na(EC), '-', EC), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           # ★ 3行にする（名前+gene_id / K番号 / EC）。
           #   1行にまとめると EC が複数ある遺伝子で左右が切れる
           panel = sprintf("<span style='color:%s'>**%s  %s<br>%s<br>EC %s**</span>",
                           col_cat[as.character(cat)], label, gene_id, K, EC2))

# ★ 検定は Welch の t 検定（R の t.test は既定が var.equal = FALSE）。
#   等分散を仮定する Student の t 検定ではない。n = 6 vs 5 で分散も揃わないため適切。
#   ★ DESeq2 の padj とは別の検定（負の二項分布 vs 正規分布）。多重比較の補正はしていない
tt = d %>% group_by(gene_id, strain, panel) %>%
    summarise(p = tryCatch(t.test(norm ~ group)$p.value, error = function(e) NA_real_),
              ymax = max(max(norm), max(tapply(norm, group, function(v) mean(v) + sd(v)))),
              .groups = 'drop') %>%
    mutate(star = cut(p, c(-Inf, .001, .01, .05, Inf), labels = c('***','**','*','n.s.')),
           star = ifelse(is.na(star), 'n.s.', as.character(star)))

# 株ごとのサブプロットを作り、patchwork で1枚にまとめる
subs = list(); hts = c()
for (s in strains4) {
    ds = d %>% filter(strain == s) %>% arrange(cat, label)
    if (nrow(ds) == 0) next
    ds$panel = factor(ds$panel, levels = unique(ds$panel))   # 分類順に並べる
    ts = tt %>% filter(strain == s) %>% mutate(panel = factor(panel, levels = levels(ds$panel)))
    n_gene = length(unique(ds$gene_id)); n_row = ceiling(n_gene / NCOL)
    subs[[s]] = ggplot(ds, aes(x = group, y = norm, fill = group)) +
        geom_bar(stat = 'summary', fun = 'mean', width = 0.65, colour = 'grey30', linewidth = 0.3) +
        # ★ SD のエラーバー（平均 ± SD の両側）。上側だけにすると errorbar が両端に
        #   キャップを描き、下のキャップが棒の上辺（＝平均）に乗って紛らわしい
        stat_summary(fun = mean,
                     fun.min = function(v) mean(v) - sd(v),
                     fun.max = function(v) mean(v) + sd(v),
                     geom = 'errorbar', width = 0.25, linewidth = 0.4, colour = 'grey20') +
        geom_quasirandom(width = 0.15, size = 1.3, alpha = 0.85, colour = 'grey20') +
        geom_text(data = ts, aes(x = 1.5, y = ymax * 1.18, label = star),
                  inherit.aes = FALSE, size = 3.6) +
        facet_wrap(~ panel, scales = 'free_y', ncol = NCOL) +
        scale_fill_manual(values = c(WT = 'grey75', KO = '#e05252'), name = NULL) +
        scale_y_continuous(expand = expansion(mult = c(0, 0.25))) +
        labs(title = sprintf('%s   (%d genes)', s, n_gene), x = NULL, y = y_lab) +
        theme_bw() +
        # ★ strip の灰色が濃いと橙・紫の文字が沈むので、ほぼ白まで薄くする
        theme(strip.background = element_rect(fill='grey97', colour='grey75', linewidth=0.3),
              strip.text = element_markdown(size = 9),
              plot.title = element_text(size = 13, face = 'bold'),
              axis.text = element_text(size = 7.5),
              axis.title.y = element_text(size = 9),
              panel.grid.major.x = element_blank())
    hts = c(hts, n_row + 0.45)   # 株の見出し分を足す
}

P = wrap_plots(subs, ncol = 1, heights = hts) +
    plot_layout(guides = 'collect') +
    # ★ 図中に日本語を入れない（PNG 保存時に mbcsToSbcs 変換が失敗して文字化けする）。
    #   日本語の経路名は上の cat() でテキストとして出す
    plot_annotation(title = sprintf('%s   [scheme %s]', tgt, SCHEME),
      subtitle = paste0(
        "panel name:  <span style='color:#d95f02'>**ORA hit**</span>  /  ",
        "<span style='color:#7570b3'>**DEG only**</span>  /  ",
        "<span style='color:#000000'>**not DEG**</span><br>",
        "y axis = **normalized count** - DESeq2 median-of-ratios with ", sub_norm, "<br>",
        "bar = mean, error bar = \u00b1SD, points = each mouse  |  ",
        "significance = **Welch two-sample t-test** on normalized counts ",
        "(*** p<0.001, ** p<0.01, * p<0.05; not DESeq2 padj, no multiple-testing correction)"),
      theme = theme(plot.title = element_text(size = 15, face = 'bold'),
                    plot.subtitle = element_markdown(size = 10, lineheight = 1.3)))

suppressMessages(ggsave(file.path(dir_gb, sprintf('genebar_%s_%s.png', ko, SCHEME)), P,
                        width = 24, height = 2.1 * sum(hts) + 1.4, dpi = 100, limitsize = FALSE))
options(repr.plot.width = 24, repr.plot.height = 2.1 * sum(hts) + 1.4)
P


ko00120_Primary bile acid biosynthesis（一次胆汁酸）  scheme A  全 5 遺伝子 / ORA hit 4


plot without title

In [62]:
# ── 図12-6: ko00121（二次胆汁酸）  scheme A ───────────────────
# セル12（準備）を先に実行しておくこと
#   （nc_A / nc_B / col_cat / strains4 / NCOL / dir_gb を使う）
ko = 'ko00121'; ko_jp = '二次胆汁酸'; SCHEME = 'A'

# スキームに応じて使うものを切り替える
nc       = if (SCHEME == 'A') nc_A else nc_B
y_lab = ifelse(SCHEME == 'A',
               'normalized count  (A: pooled across all 4 strains)',
               'normalized count  (B: per-strain size factors)')
sub_norm = ifelse(SCHEME == 'A',
    paste0("**one size factor set computed from all 4 strains pooled** ",
           "(= scheme A, the community-level output; includes the shift in strain abundance)"),
    paste0("**size factors computed separately within each strain** ",
           "(= scheme B, within-strain regulation; excludes the shift in strain abundance)"))

tgt = ora$term[ora$source == 'KEGG' & grepl(paste0('^', ko, '_'), ora$term)][1]

# ORA で拾われた遺伝子（down/up のどちらでも拾われていれば ORA hit）
g_ora = ora %>% filter(scheme == SCHEME, source == 'KEGG', term == tgt, padj < 0.05) %>%
    pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()

# 経路に属する全遺伝子
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
    dplyr::select(gene_id, KEGG_Pathway) %>%
    tidyr::separate_rows(KEGG_Pathway, sep = ',') %>%
    filter(KEGG_Pathway == ko) %>% pull(gene_id) %>% unique()

cat(sprintf("%s（%s）  scheme %s  全 %d 遺伝子 / ORA hit %d%s\n",
            tgt, ko_jp, SCHEME, length(g_all), length(g_ora),
            ifelse(length(g_ora) == 0, '  ★ この経路はこのスキームの ORA では有意でない', '')))

# ★ DEG の判定もスキームに合わせる（(A) は padj_A、(B) は padj_pooled）
d = nc[intersect(g_all, rownames(nc)), , drop = FALSE] %>%
    tibble::rownames_to_column('gene_id') %>%
    tidyr::pivot_longer(-gene_id, names_to = 'sample', values_to = 'norm') %>%
    mutate(group = factor(ifelse(grepl('W', sample), 'WT', 'KO'), levels = c('WT','KO'))) %>%
    left_join(res %>% dplyr::select(gene_id, strain, Preferred_name, product,
                                    KEGG_ko, EC, padj_A, padj_pooled), by = 'gene_id') %>%
    mutate(padj_use = if (SCHEME == 'A') padj_A else padj_pooled,
           cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                           !is.na(padj_use) & padj_use < 0.05 ~ 'DEG only',
                           TRUE ~ 'not DEG'),
           cat = factor(cat, levels = names(col_cat)),
           label = ifelse(is.na(Preferred_name) | Preferred_name == '-',
                          substr(product, 1, 14), Preferred_name),
           K   = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           EC2 = sapply(strsplit(ifelse(is.na(EC), '-', EC), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           # ★ 3行にする（名前+gene_id / K番号 / EC）。
           #   1行にまとめると EC が複数ある遺伝子で左右が切れる
           panel = sprintf("<span style='color:%s'>**%s  %s<br>%s<br>EC %s**</span>",
                           col_cat[as.character(cat)], label, gene_id, K, EC2))

# ★ 検定は Welch の t 検定（R の t.test は既定が var.equal = FALSE）。
#   等分散を仮定する Student の t 検定ではない。n = 6 vs 5 で分散も揃わないため適切。
#   ★ DESeq2 の padj とは別の検定（負の二項分布 vs 正規分布）。多重比較の補正はしていない
tt = d %>% group_by(gene_id, strain, panel) %>%
    summarise(p = tryCatch(t.test(norm ~ group)$p.value, error = function(e) NA_real_),
              ymax = max(max(norm), max(tapply(norm, group, function(v) mean(v) + sd(v)))),
              .groups = 'drop') %>%
    mutate(star = cut(p, c(-Inf, .001, .01, .05, Inf), labels = c('***','**','*','n.s.')),
           star = ifelse(is.na(star), 'n.s.', as.character(star)))

# 株ごとのサブプロットを作り、patchwork で1枚にまとめる
subs = list(); hts = c()
for (s in strains4) {
    ds = d %>% filter(strain == s) %>% arrange(cat, label)
    if (nrow(ds) == 0) next
    ds$panel = factor(ds$panel, levels = unique(ds$panel))   # 分類順に並べる
    ts = tt %>% filter(strain == s) %>% mutate(panel = factor(panel, levels = levels(ds$panel)))
    n_gene = length(unique(ds$gene_id)); n_row = ceiling(n_gene / NCOL)
    subs[[s]] = ggplot(ds, aes(x = group, y = norm, fill = group)) +
        geom_bar(stat = 'summary', fun = 'mean', width = 0.65, colour = 'grey30', linewidth = 0.3) +
        # ★ SD のエラーバー（平均 ± SD の両側）。上側だけにすると errorbar が両端に
        #   キャップを描き、下のキャップが棒の上辺（＝平均）に乗って紛らわしい
        stat_summary(fun = mean,
                     fun.min = function(v) mean(v) - sd(v),
                     fun.max = function(v) mean(v) + sd(v),
                     geom = 'errorbar', width = 0.25, linewidth = 0.4, colour = 'grey20') +
        geom_quasirandom(width = 0.15, size = 1.3, alpha = 0.85, colour = 'grey20') +
        geom_text(data = ts, aes(x = 1.5, y = ymax * 1.18, label = star),
                  inherit.aes = FALSE, size = 3.6) +
        facet_wrap(~ panel, scales = 'free_y', ncol = NCOL) +
        scale_fill_manual(values = c(WT = 'grey75', KO = '#e05252'), name = NULL) +
        scale_y_continuous(expand = expansion(mult = c(0, 0.25))) +
        labs(title = sprintf('%s   (%d genes)', s, n_gene), x = NULL, y = y_lab) +
        theme_bw() +
        # ★ strip の灰色が濃いと橙・紫の文字が沈むので、ほぼ白まで薄くする
        theme(strip.background = element_rect(fill='grey97', colour='grey75', linewidth=0.3),
              strip.text = element_markdown(size = 9),
              plot.title = element_text(size = 13, face = 'bold'),
              axis.text = element_text(size = 7.5),
              axis.title.y = element_text(size = 9),
              panel.grid.major.x = element_blank())
    hts = c(hts, n_row + 0.45)   # 株の見出し分を足す
}

P = wrap_plots(subs, ncol = 1, heights = hts) +
    plot_layout(guides = 'collect') +
    # ★ 図中に日本語を入れない（PNG 保存時に mbcsToSbcs 変換が失敗して文字化けする）。
    #   日本語の経路名は上の cat() でテキストとして出す
    plot_annotation(title = sprintf('%s   [scheme %s]', tgt, SCHEME),
      subtitle = paste0(
        "panel name:  <span style='color:#d95f02'>**ORA hit**</span>  /  ",
        "<span style='color:#7570b3'>**DEG only**</span>  /  ",
        "<span style='color:#000000'>**not DEG**</span><br>",
        "y axis = **normalized count** - DESeq2 median-of-ratios with ", sub_norm, "<br>",
        "bar = mean, error bar = \u00b1SD, points = each mouse  |  ",
        "significance = **Welch two-sample t-test** on normalized counts ",
        "(*** p<0.001, ** p<0.01, * p<0.05; not DESeq2 padj, no multiple-testing correction)"),
      theme = theme(plot.title = element_text(size = 15, face = 'bold'),
                    plot.subtitle = element_markdown(size = 10, lineheight = 1.3)))

suppressMessages(ggsave(file.path(dir_gb, sprintf('genebar_%s_%s.png', ko, SCHEME)), P,
                        width = 24, height = 2.1 * sum(hts) + 1.4, dpi = 100, limitsize = FALSE))
options(repr.plot.width = 24, repr.plot.height = 2.1 * sum(hts) + 1.4)
P


ko00121_Secondary bile acid biosynthesis（二次胆汁酸）  scheme A  全 7 遺伝子 / ORA hit 6


plot without title

In [63]:
# ── 図12-7: ko03010（リボソーム）  scheme A ───────────────────
# セル12（準備）を先に実行しておくこと
#   （nc_A / nc_B / col_cat / strains4 / NCOL / dir_gb を使う）
ko = 'ko03010'; ko_jp = 'リボソーム'; SCHEME = 'A'

# スキームに応じて使うものを切り替える
nc       = if (SCHEME == 'A') nc_A else nc_B
y_lab = ifelse(SCHEME == 'A',
               'normalized count  (A: pooled across all 4 strains)',
               'normalized count  (B: per-strain size factors)')
sub_norm = ifelse(SCHEME == 'A',
    paste0("**one size factor set computed from all 4 strains pooled** ",
           "(= scheme A, the community-level output; includes the shift in strain abundance)"),
    paste0("**size factors computed separately within each strain** ",
           "(= scheme B, within-strain regulation; excludes the shift in strain abundance)"))

tgt = ora$term[ora$source == 'KEGG' & grepl(paste0('^', ko, '_'), ora$term)][1]

# ORA で拾われた遺伝子（down/up のどちらでも拾われていれば ORA hit）
g_ora = ora %>% filter(scheme == SCHEME, source == 'KEGG', term == tgt, padj < 0.05) %>%
    pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()

# 経路に属する全遺伝子
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
    dplyr::select(gene_id, KEGG_Pathway) %>%
    tidyr::separate_rows(KEGG_Pathway, sep = ',') %>%
    filter(KEGG_Pathway == ko) %>% pull(gene_id) %>% unique()

cat(sprintf("%s（%s）  scheme %s  全 %d 遺伝子 / ORA hit %d%s\n",
            tgt, ko_jp, SCHEME, length(g_all), length(g_ora),
            ifelse(length(g_ora) == 0, '  ★ この経路はこのスキームの ORA では有意でない', '')))

# ★ DEG の判定もスキームに合わせる（(A) は padj_A、(B) は padj_pooled）
d = nc[intersect(g_all, rownames(nc)), , drop = FALSE] %>%
    tibble::rownames_to_column('gene_id') %>%
    tidyr::pivot_longer(-gene_id, names_to = 'sample', values_to = 'norm') %>%
    mutate(group = factor(ifelse(grepl('W', sample), 'WT', 'KO'), levels = c('WT','KO'))) %>%
    left_join(res %>% dplyr::select(gene_id, strain, Preferred_name, product,
                                    KEGG_ko, EC, padj_A, padj_pooled), by = 'gene_id') %>%
    mutate(padj_use = if (SCHEME == 'A') padj_A else padj_pooled,
           cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                           !is.na(padj_use) & padj_use < 0.05 ~ 'DEG only',
                           TRUE ~ 'not DEG'),
           cat = factor(cat, levels = names(col_cat)),
           label = ifelse(is.na(Preferred_name) | Preferred_name == '-',
                          substr(product, 1, 14), Preferred_name),
           K   = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           EC2 = sapply(strsplit(ifelse(is.na(EC), '-', EC), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           # ★ 3行にする（名前+gene_id / K番号 / EC）。
           #   1行にまとめると EC が複数ある遺伝子で左右が切れる
           panel = sprintf("<span style='color:%s'>**%s  %s<br>%s<br>EC %s**</span>",
                           col_cat[as.character(cat)], label, gene_id, K, EC2))

# ★ 検定は Welch の t 検定（R の t.test は既定が var.equal = FALSE）。
#   等分散を仮定する Student の t 検定ではない。n = 6 vs 5 で分散も揃わないため適切。
#   ★ DESeq2 の padj とは別の検定（負の二項分布 vs 正規分布）。多重比較の補正はしていない
tt = d %>% group_by(gene_id, strain, panel) %>%
    summarise(p = tryCatch(t.test(norm ~ group)$p.value, error = function(e) NA_real_),
              ymax = max(max(norm), max(tapply(norm, group, function(v) mean(v) + sd(v)))),
              .groups = 'drop') %>%
    mutate(star = cut(p, c(-Inf, .001, .01, .05, Inf), labels = c('***','**','*','n.s.')),
           star = ifelse(is.na(star), 'n.s.', as.character(star)))

# 株ごとのサブプロットを作り、patchwork で1枚にまとめる
subs = list(); hts = c()
for (s in strains4) {
    ds = d %>% filter(strain == s) %>% arrange(cat, label)
    if (nrow(ds) == 0) next
    ds$panel = factor(ds$panel, levels = unique(ds$panel))   # 分類順に並べる
    ts = tt %>% filter(strain == s) %>% mutate(panel = factor(panel, levels = levels(ds$panel)))
    n_gene = length(unique(ds$gene_id)); n_row = ceiling(n_gene / NCOL)
    subs[[s]] = ggplot(ds, aes(x = group, y = norm, fill = group)) +
        geom_bar(stat = 'summary', fun = 'mean', width = 0.65, colour = 'grey30', linewidth = 0.3) +
        # ★ SD のエラーバー（平均 ± SD の両側）。上側だけにすると errorbar が両端に
        #   キャップを描き、下のキャップが棒の上辺（＝平均）に乗って紛らわしい
        stat_summary(fun = mean,
                     fun.min = function(v) mean(v) - sd(v),
                     fun.max = function(v) mean(v) + sd(v),
                     geom = 'errorbar', width = 0.25, linewidth = 0.4, colour = 'grey20') +
        geom_quasirandom(width = 0.15, size = 1.3, alpha = 0.85, colour = 'grey20') +
        geom_text(data = ts, aes(x = 1.5, y = ymax * 1.18, label = star),
                  inherit.aes = FALSE, size = 3.6) +
        facet_wrap(~ panel, scales = 'free_y', ncol = NCOL) +
        scale_fill_manual(values = c(WT = 'grey75', KO = '#e05252'), name = NULL) +
        scale_y_continuous(expand = expansion(mult = c(0, 0.25))) +
        labs(title = sprintf('%s   (%d genes)', s, n_gene), x = NULL, y = y_lab) +
        theme_bw() +
        # ★ strip の灰色が濃いと橙・紫の文字が沈むので、ほぼ白まで薄くする
        theme(strip.background = element_rect(fill='grey97', colour='grey75', linewidth=0.3),
              strip.text = element_markdown(size = 9),
              plot.title = element_text(size = 13, face = 'bold'),
              axis.text = element_text(size = 7.5),
              axis.title.y = element_text(size = 9),
              panel.grid.major.x = element_blank())
    hts = c(hts, n_row + 0.45)   # 株の見出し分を足す
}

P = wrap_plots(subs, ncol = 1, heights = hts) +
    plot_layout(guides = 'collect') +
    # ★ 図中に日本語を入れない（PNG 保存時に mbcsToSbcs 変換が失敗して文字化けする）。
    #   日本語の経路名は上の cat() でテキストとして出す
    plot_annotation(title = sprintf('%s   [scheme %s]', tgt, SCHEME),
      subtitle = paste0(
        "panel name:  <span style='color:#d95f02'>**ORA hit**</span>  /  ",
        "<span style='color:#7570b3'>**DEG only**</span>  /  ",
        "<span style='color:#000000'>**not DEG**</span><br>",
        "y axis = **normalized count** - DESeq2 median-of-ratios with ", sub_norm, "<br>",
        "bar = mean, error bar = \u00b1SD, points = each mouse  |  ",
        "significance = **Welch two-sample t-test** on normalized counts ",
        "(*** p<0.001, ** p<0.01, * p<0.05; not DESeq2 padj, no multiple-testing correction)"),
      theme = theme(plot.title = element_text(size = 15, face = 'bold'),
                    plot.subtitle = element_markdown(size = 10, lineheight = 1.3)))

suppressMessages(ggsave(file.path(dir_gb, sprintf('genebar_%s_%s.png', ko, SCHEME)), P,
                        width = 24, height = 2.1 * sum(hts) + 1.4, dpi = 100, limitsize = FALSE))
options(repr.plot.width = 24, repr.plot.height = 2.1 * sum(hts) + 1.4)
P


ko03010_Ribosome（リボソーム）  scheme A  全 112 遺伝子 / ORA hit 41


plot without title

In [64]:
# ── 図12-8: ko00030（ペントースリン酸経路）  scheme B ───────────────────
# セル12（準備）を先に実行しておくこと
#   （nc_A / nc_B / col_cat / strains4 / NCOL / dir_gb を使う）
ko = 'ko00030'; ko_jp = 'ペントースリン酸経路'; SCHEME = 'B'

# スキームに応じて使うものを切り替える
nc       = if (SCHEME == 'A') nc_A else nc_B
y_lab = ifelse(SCHEME == 'A',
               'normalized count  (A: pooled across all 4 strains)',
               'normalized count  (B: per-strain size factors)')
sub_norm = ifelse(SCHEME == 'A',
    paste0("**one size factor set computed from all 4 strains pooled** ",
           "(= scheme A, the community-level output; includes the shift in strain abundance)"),
    paste0("**size factors computed separately within each strain** ",
           "(= scheme B, within-strain regulation; excludes the shift in strain abundance)"))

tgt = ora$term[ora$source == 'KEGG' & grepl(paste0('^', ko, '_'), ora$term)][1]

# ORA で拾われた遺伝子（down/up のどちらでも拾われていれば ORA hit）
g_ora = ora %>% filter(scheme == SCHEME, source == 'KEGG', term == tgt, padj < 0.05) %>%
    pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()

# 経路に属する全遺伝子
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
    dplyr::select(gene_id, KEGG_Pathway) %>%
    tidyr::separate_rows(KEGG_Pathway, sep = ',') %>%
    filter(KEGG_Pathway == ko) %>% pull(gene_id) %>% unique()

cat(sprintf("%s（%s）  scheme %s  全 %d 遺伝子 / ORA hit %d%s\n",
            tgt, ko_jp, SCHEME, length(g_all), length(g_ora),
            ifelse(length(g_ora) == 0, '  ★ この経路はこのスキームの ORA では有意でない', '')))

# ★ DEG の判定もスキームに合わせる（(A) は padj_A、(B) は padj_pooled）
d = nc[intersect(g_all, rownames(nc)), , drop = FALSE] %>%
    tibble::rownames_to_column('gene_id') %>%
    tidyr::pivot_longer(-gene_id, names_to = 'sample', values_to = 'norm') %>%
    mutate(group = factor(ifelse(grepl('W', sample), 'WT', 'KO'), levels = c('WT','KO'))) %>%
    left_join(res %>% dplyr::select(gene_id, strain, Preferred_name, product,
                                    KEGG_ko, EC, padj_A, padj_pooled), by = 'gene_id') %>%
    mutate(padj_use = if (SCHEME == 'A') padj_A else padj_pooled,
           cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                           !is.na(padj_use) & padj_use < 0.05 ~ 'DEG only',
                           TRUE ~ 'not DEG'),
           cat = factor(cat, levels = names(col_cat)),
           label = ifelse(is.na(Preferred_name) | Preferred_name == '-',
                          substr(product, 1, 14), Preferred_name),
           K   = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           EC2 = sapply(strsplit(ifelse(is.na(EC), '-', EC), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           # ★ 3行にする（名前+gene_id / K番号 / EC）。
           #   1行にまとめると EC が複数ある遺伝子で左右が切れる
           panel = sprintf("<span style='color:%s'>**%s  %s<br>%s<br>EC %s**</span>",
                           col_cat[as.character(cat)], label, gene_id, K, EC2))

# ★ 検定は Welch の t 検定（R の t.test は既定が var.equal = FALSE）。
#   等分散を仮定する Student の t 検定ではない。n = 6 vs 5 で分散も揃わないため適切。
#   ★ DESeq2 の padj とは別の検定（負の二項分布 vs 正規分布）。多重比較の補正はしていない
tt = d %>% group_by(gene_id, strain, panel) %>%
    summarise(p = tryCatch(t.test(norm ~ group)$p.value, error = function(e) NA_real_),
              ymax = max(max(norm), max(tapply(norm, group, function(v) mean(v) + sd(v)))),
              .groups = 'drop') %>%
    mutate(star = cut(p, c(-Inf, .001, .01, .05, Inf), labels = c('***','**','*','n.s.')),
           star = ifelse(is.na(star), 'n.s.', as.character(star)))

# 株ごとのサブプロットを作り、patchwork で1枚にまとめる
subs = list(); hts = c()
for (s in strains4) {
    ds = d %>% filter(strain == s) %>% arrange(cat, label)
    if (nrow(ds) == 0) next
    ds$panel = factor(ds$panel, levels = unique(ds$panel))   # 分類順に並べる
    ts = tt %>% filter(strain == s) %>% mutate(panel = factor(panel, levels = levels(ds$panel)))
    n_gene = length(unique(ds$gene_id)); n_row = ceiling(n_gene / NCOL)
    subs[[s]] = ggplot(ds, aes(x = group, y = norm, fill = group)) +
        geom_bar(stat = 'summary', fun = 'mean', width = 0.65, colour = 'grey30', linewidth = 0.3) +
        # ★ SD のエラーバー（平均 ± SD の両側）。上側だけにすると errorbar が両端に
        #   キャップを描き、下のキャップが棒の上辺（＝平均）に乗って紛らわしい
        stat_summary(fun = mean,
                     fun.min = function(v) mean(v) - sd(v),
                     fun.max = function(v) mean(v) + sd(v),
                     geom = 'errorbar', width = 0.25, linewidth = 0.4, colour = 'grey20') +
        geom_quasirandom(width = 0.15, size = 1.3, alpha = 0.85, colour = 'grey20') +
        geom_text(data = ts, aes(x = 1.5, y = ymax * 1.18, label = star),
                  inherit.aes = FALSE, size = 3.6) +
        facet_wrap(~ panel, scales = 'free_y', ncol = NCOL) +
        scale_fill_manual(values = c(WT = 'grey75', KO = '#e05252'), name = NULL) +
        scale_y_continuous(expand = expansion(mult = c(0, 0.25))) +
        labs(title = sprintf('%s   (%d genes)', s, n_gene), x = NULL, y = y_lab) +
        theme_bw() +
        # ★ strip の灰色が濃いと橙・紫の文字が沈むので、ほぼ白まで薄くする
        theme(strip.background = element_rect(fill='grey97', colour='grey75', linewidth=0.3),
              strip.text = element_markdown(size = 9),
              plot.title = element_text(size = 13, face = 'bold'),
              axis.text = element_text(size = 7.5),
              axis.title.y = element_text(size = 9),
              panel.grid.major.x = element_blank())
    hts = c(hts, n_row + 0.45)   # 株の見出し分を足す
}

P = wrap_plots(subs, ncol = 1, heights = hts) +
    plot_layout(guides = 'collect') +
    # ★ 図中に日本語を入れない（PNG 保存時に mbcsToSbcs 変換が失敗して文字化けする）。
    #   日本語の経路名は上の cat() でテキストとして出す
    plot_annotation(title = sprintf('%s   [scheme %s]', tgt, SCHEME),
      subtitle = paste0(
        "panel name:  <span style='color:#d95f02'>**ORA hit**</span>  /  ",
        "<span style='color:#7570b3'>**DEG only**</span>  /  ",
        "<span style='color:#000000'>**not DEG**</span><br>",
        "y axis = **normalized count** - DESeq2 median-of-ratios with ", sub_norm, "<br>",
        "bar = mean, error bar = \u00b1SD, points = each mouse  |  ",
        "significance = **Welch two-sample t-test** on normalized counts ",
        "(*** p<0.001, ** p<0.01, * p<0.05; not DESeq2 padj, no multiple-testing correction)"),
      theme = theme(plot.title = element_text(size = 15, face = 'bold'),
                    plot.subtitle = element_markdown(size = 10, lineheight = 1.3)))

suppressMessages(ggsave(file.path(dir_gb, sprintf('genebar_%s_%s.png', ko, SCHEME)), P,
                        width = 24, height = 2.1 * sum(hts) + 1.4, dpi = 100, limitsize = FALSE))
options(repr.plot.width = 24, repr.plot.height = 2.1 * sum(hts) + 1.4)
P


ko00030_Pentose phosphate pathway（ペントースリン酸経路）  scheme B  全 69 遺伝子 / ORA hit 33


plot without title

In [65]:
# ── 図12-9: ko00020（TCA回路）  scheme B ───────────────────
# セル12（準備）を先に実行しておくこと
#   （nc_A / nc_B / col_cat / strains4 / NCOL / dir_gb を使う）
ko = 'ko00020'; ko_jp = 'TCA回路'; SCHEME = 'B'

# スキームに応じて使うものを切り替える
nc       = if (SCHEME == 'A') nc_A else nc_B
y_lab = ifelse(SCHEME == 'A',
               'normalized count  (A: pooled across all 4 strains)',
               'normalized count  (B: per-strain size factors)')
sub_norm = ifelse(SCHEME == 'A',
    paste0("**one size factor set computed from all 4 strains pooled** ",
           "(= scheme A, the community-level output; includes the shift in strain abundance)"),
    paste0("**size factors computed separately within each strain** ",
           "(= scheme B, within-strain regulation; excludes the shift in strain abundance)"))

tgt = ora$term[ora$source == 'KEGG' & grepl(paste0('^', ko, '_'), ora$term)][1]

# ORA で拾われた遺伝子（down/up のどちらでも拾われていれば ORA hit）
g_ora = ora %>% filter(scheme == SCHEME, source == 'KEGG', term == tgt, padj < 0.05) %>%
    pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()

# 経路に属する全遺伝子
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
    dplyr::select(gene_id, KEGG_Pathway) %>%
    tidyr::separate_rows(KEGG_Pathway, sep = ',') %>%
    filter(KEGG_Pathway == ko) %>% pull(gene_id) %>% unique()

cat(sprintf("%s（%s）  scheme %s  全 %d 遺伝子 / ORA hit %d%s\n",
            tgt, ko_jp, SCHEME, length(g_all), length(g_ora),
            ifelse(length(g_ora) == 0, '  ★ この経路はこのスキームの ORA では有意でない', '')))

# ★ DEG の判定もスキームに合わせる（(A) は padj_A、(B) は padj_pooled）
d = nc[intersect(g_all, rownames(nc)), , drop = FALSE] %>%
    tibble::rownames_to_column('gene_id') %>%
    tidyr::pivot_longer(-gene_id, names_to = 'sample', values_to = 'norm') %>%
    mutate(group = factor(ifelse(grepl('W', sample), 'WT', 'KO'), levels = c('WT','KO'))) %>%
    left_join(res %>% dplyr::select(gene_id, strain, Preferred_name, product,
                                    KEGG_ko, EC, padj_A, padj_pooled), by = 'gene_id') %>%
    mutate(padj_use = if (SCHEME == 'A') padj_A else padj_pooled,
           cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                           !is.na(padj_use) & padj_use < 0.05 ~ 'DEG only',
                           TRUE ~ 'not DEG'),
           cat = factor(cat, levels = names(col_cat)),
           label = ifelse(is.na(Preferred_name) | Preferred_name == '-',
                          substr(product, 1, 14), Preferred_name),
           K   = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           EC2 = sapply(strsplit(ifelse(is.na(EC), '-', EC), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           # ★ 3行にする（名前+gene_id / K番号 / EC）。
           #   1行にまとめると EC が複数ある遺伝子で左右が切れる
           panel = sprintf("<span style='color:%s'>**%s  %s<br>%s<br>EC %s**</span>",
                           col_cat[as.character(cat)], label, gene_id, K, EC2))

# ★ 検定は Welch の t 検定（R の t.test は既定が var.equal = FALSE）。
#   等分散を仮定する Student の t 検定ではない。n = 6 vs 5 で分散も揃わないため適切。
#   ★ DESeq2 の padj とは別の検定（負の二項分布 vs 正規分布）。多重比較の補正はしていない
tt = d %>% group_by(gene_id, strain, panel) %>%
    summarise(p = tryCatch(t.test(norm ~ group)$p.value, error = function(e) NA_real_),
              ymax = max(max(norm), max(tapply(norm, group, function(v) mean(v) + sd(v)))),
              .groups = 'drop') %>%
    mutate(star = cut(p, c(-Inf, .001, .01, .05, Inf), labels = c('***','**','*','n.s.')),
           star = ifelse(is.na(star), 'n.s.', as.character(star)))

# 株ごとのサブプロットを作り、patchwork で1枚にまとめる
subs = list(); hts = c()
for (s in strains4) {
    ds = d %>% filter(strain == s) %>% arrange(cat, label)
    if (nrow(ds) == 0) next
    ds$panel = factor(ds$panel, levels = unique(ds$panel))   # 分類順に並べる
    ts = tt %>% filter(strain == s) %>% mutate(panel = factor(panel, levels = levels(ds$panel)))
    n_gene = length(unique(ds$gene_id)); n_row = ceiling(n_gene / NCOL)
    subs[[s]] = ggplot(ds, aes(x = group, y = norm, fill = group)) +
        geom_bar(stat = 'summary', fun = 'mean', width = 0.65, colour = 'grey30', linewidth = 0.3) +
        # ★ SD のエラーバー（平均 ± SD の両側）。上側だけにすると errorbar が両端に
        #   キャップを描き、下のキャップが棒の上辺（＝平均）に乗って紛らわしい
        stat_summary(fun = mean,
                     fun.min = function(v) mean(v) - sd(v),
                     fun.max = function(v) mean(v) + sd(v),
                     geom = 'errorbar', width = 0.25, linewidth = 0.4, colour = 'grey20') +
        geom_quasirandom(width = 0.15, size = 1.3, alpha = 0.85, colour = 'grey20') +
        geom_text(data = ts, aes(x = 1.5, y = ymax * 1.18, label = star),
                  inherit.aes = FALSE, size = 3.6) +
        facet_wrap(~ panel, scales = 'free_y', ncol = NCOL) +
        scale_fill_manual(values = c(WT = 'grey75', KO = '#e05252'), name = NULL) +
        scale_y_continuous(expand = expansion(mult = c(0, 0.25))) +
        labs(title = sprintf('%s   (%d genes)', s, n_gene), x = NULL, y = y_lab) +
        theme_bw() +
        # ★ strip の灰色が濃いと橙・紫の文字が沈むので、ほぼ白まで薄くする
        theme(strip.background = element_rect(fill='grey97', colour='grey75', linewidth=0.3),
              strip.text = element_markdown(size = 9),
              plot.title = element_text(size = 13, face = 'bold'),
              axis.text = element_text(size = 7.5),
              axis.title.y = element_text(size = 9),
              panel.grid.major.x = element_blank())
    hts = c(hts, n_row + 0.45)   # 株の見出し分を足す
}

P = wrap_plots(subs, ncol = 1, heights = hts) +
    plot_layout(guides = 'collect') +
    # ★ 図中に日本語を入れない（PNG 保存時に mbcsToSbcs 変換が失敗して文字化けする）。
    #   日本語の経路名は上の cat() でテキストとして出す
    plot_annotation(title = sprintf('%s   [scheme %s]', tgt, SCHEME),
      subtitle = paste0(
        "panel name:  <span style='color:#d95f02'>**ORA hit**</span>  /  ",
        "<span style='color:#7570b3'>**DEG only**</span>  /  ",
        "<span style='color:#000000'>**not DEG**</span><br>",
        "y axis = **normalized count** - DESeq2 median-of-ratios with ", sub_norm, "<br>",
        "bar = mean, error bar = \u00b1SD, points = each mouse  |  ",
        "significance = **Welch two-sample t-test** on normalized counts ",
        "(*** p<0.001, ** p<0.01, * p<0.05; not DESeq2 padj, no multiple-testing correction)"),
      theme = theme(plot.title = element_text(size = 15, face = 'bold'),
                    plot.subtitle = element_markdown(size = 10, lineheight = 1.3)))

suppressMessages(ggsave(file.path(dir_gb, sprintf('genebar_%s_%s.png', ko, SCHEME)), P,
                        width = 24, height = 2.1 * sum(hts) + 1.4, dpi = 100, limitsize = FALSE))
options(repr.plot.width = 24, repr.plot.height = 2.1 * sum(hts) + 1.4)
P


ko00020_Citrate cycle (TCA cycle)（TCA回路）  scheme B  全 61 遺伝子 / ORA hit 24


plot without title

In [66]:
# ── 図12-10: ko00710（炭素固定(Calvin)）  scheme B ───────────────────
# セル12（準備）を先に実行しておくこと
#   （nc_A / nc_B / col_cat / strains4 / NCOL / dir_gb を使う）
ko = 'ko00710'; ko_jp = '炭素固定(Calvin)'; SCHEME = 'B'

# スキームに応じて使うものを切り替える
nc       = if (SCHEME == 'A') nc_A else nc_B
y_lab = ifelse(SCHEME == 'A',
               'normalized count  (A: pooled across all 4 strains)',
               'normalized count  (B: per-strain size factors)')
sub_norm = ifelse(SCHEME == 'A',
    paste0("**one size factor set computed from all 4 strains pooled** ",
           "(= scheme A, the community-level output; includes the shift in strain abundance)"),
    paste0("**size factors computed separately within each strain** ",
           "(= scheme B, within-strain regulation; excludes the shift in strain abundance)"))

tgt = ora$term[ora$source == 'KEGG' & grepl(paste0('^', ko, '_'), ora$term)][1]

# ORA で拾われた遺伝子（down/up のどちらでも拾われていれば ORA hit）
g_ora = ora %>% filter(scheme == SCHEME, source == 'KEGG', term == tgt, padj < 0.05) %>%
    pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()

# 経路に属する全遺伝子
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
    dplyr::select(gene_id, KEGG_Pathway) %>%
    tidyr::separate_rows(KEGG_Pathway, sep = ',') %>%
    filter(KEGG_Pathway == ko) %>% pull(gene_id) %>% unique()

cat(sprintf("%s（%s）  scheme %s  全 %d 遺伝子 / ORA hit %d%s\n",
            tgt, ko_jp, SCHEME, length(g_all), length(g_ora),
            ifelse(length(g_ora) == 0, '  ★ この経路はこのスキームの ORA では有意でない', '')))

# ★ DEG の判定もスキームに合わせる（(A) は padj_A、(B) は padj_pooled）
d = nc[intersect(g_all, rownames(nc)), , drop = FALSE] %>%
    tibble::rownames_to_column('gene_id') %>%
    tidyr::pivot_longer(-gene_id, names_to = 'sample', values_to = 'norm') %>%
    mutate(group = factor(ifelse(grepl('W', sample), 'WT', 'KO'), levels = c('WT','KO'))) %>%
    left_join(res %>% dplyr::select(gene_id, strain, Preferred_name, product,
                                    KEGG_ko, EC, padj_A, padj_pooled), by = 'gene_id') %>%
    mutate(padj_use = if (SCHEME == 'A') padj_A else padj_pooled,
           cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                           !is.na(padj_use) & padj_use < 0.05 ~ 'DEG only',
                           TRUE ~ 'not DEG'),
           cat = factor(cat, levels = names(col_cat)),
           label = ifelse(is.na(Preferred_name) | Preferred_name == '-',
                          substr(product, 1, 14), Preferred_name),
           K   = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           EC2 = sapply(strsplit(ifelse(is.na(EC), '-', EC), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           # ★ 3行にする（名前+gene_id / K番号 / EC）。
           #   1行にまとめると EC が複数ある遺伝子で左右が切れる
           panel = sprintf("<span style='color:%s'>**%s  %s<br>%s<br>EC %s**</span>",
                           col_cat[as.character(cat)], label, gene_id, K, EC2))

# ★ 検定は Welch の t 検定（R の t.test は既定が var.equal = FALSE）。
#   等分散を仮定する Student の t 検定ではない。n = 6 vs 5 で分散も揃わないため適切。
#   ★ DESeq2 の padj とは別の検定（負の二項分布 vs 正規分布）。多重比較の補正はしていない
tt = d %>% group_by(gene_id, strain, panel) %>%
    summarise(p = tryCatch(t.test(norm ~ group)$p.value, error = function(e) NA_real_),
              ymax = max(max(norm), max(tapply(norm, group, function(v) mean(v) + sd(v)))),
              .groups = 'drop') %>%
    mutate(star = cut(p, c(-Inf, .001, .01, .05, Inf), labels = c('***','**','*','n.s.')),
           star = ifelse(is.na(star), 'n.s.', as.character(star)))

# 株ごとのサブプロットを作り、patchwork で1枚にまとめる
subs = list(); hts = c()
for (s in strains4) {
    ds = d %>% filter(strain == s) %>% arrange(cat, label)
    if (nrow(ds) == 0) next
    ds$panel = factor(ds$panel, levels = unique(ds$panel))   # 分類順に並べる
    ts = tt %>% filter(strain == s) %>% mutate(panel = factor(panel, levels = levels(ds$panel)))
    n_gene = length(unique(ds$gene_id)); n_row = ceiling(n_gene / NCOL)
    subs[[s]] = ggplot(ds, aes(x = group, y = norm, fill = group)) +
        geom_bar(stat = 'summary', fun = 'mean', width = 0.65, colour = 'grey30', linewidth = 0.3) +
        # ★ SD のエラーバー（平均 ± SD の両側）。上側だけにすると errorbar が両端に
        #   キャップを描き、下のキャップが棒の上辺（＝平均）に乗って紛らわしい
        stat_summary(fun = mean,
                     fun.min = function(v) mean(v) - sd(v),
                     fun.max = function(v) mean(v) + sd(v),
                     geom = 'errorbar', width = 0.25, linewidth = 0.4, colour = 'grey20') +
        geom_quasirandom(width = 0.15, size = 1.3, alpha = 0.85, colour = 'grey20') +
        geom_text(data = ts, aes(x = 1.5, y = ymax * 1.18, label = star),
                  inherit.aes = FALSE, size = 3.6) +
        facet_wrap(~ panel, scales = 'free_y', ncol = NCOL) +
        scale_fill_manual(values = c(WT = 'grey75', KO = '#e05252'), name = NULL) +
        scale_y_continuous(expand = expansion(mult = c(0, 0.25))) +
        labs(title = sprintf('%s   (%d genes)', s, n_gene), x = NULL, y = y_lab) +
        theme_bw() +
        # ★ strip の灰色が濃いと橙・紫の文字が沈むので、ほぼ白まで薄くする
        theme(strip.background = element_rect(fill='grey97', colour='grey75', linewidth=0.3),
              strip.text = element_markdown(size = 9),
              plot.title = element_text(size = 13, face = 'bold'),
              axis.text = element_text(size = 7.5),
              axis.title.y = element_text(size = 9),
              panel.grid.major.x = element_blank())
    hts = c(hts, n_row + 0.45)   # 株の見出し分を足す
}

P = wrap_plots(subs, ncol = 1, heights = hts) +
    plot_layout(guides = 'collect') +
    # ★ 図中に日本語を入れない（PNG 保存時に mbcsToSbcs 変換が失敗して文字化けする）。
    #   日本語の経路名は上の cat() でテキストとして出す
    plot_annotation(title = sprintf('%s   [scheme %s]', tgt, SCHEME),
      subtitle = paste0(
        "panel name:  <span style='color:#d95f02'>**ORA hit**</span>  /  ",
        "<span style='color:#7570b3'>**DEG only**</span>  /  ",
        "<span style='color:#000000'>**not DEG**</span><br>",
        "y axis = **normalized count** - DESeq2 median-of-ratios with ", sub_norm, "<br>",
        "bar = mean, error bar = \u00b1SD, points = each mouse  |  ",
        "significance = **Welch two-sample t-test** on normalized counts ",
        "(*** p<0.001, ** p<0.01, * p<0.05; not DESeq2 padj, no multiple-testing correction)"),
      theme = theme(plot.title = element_text(size = 15, face = 'bold'),
                    plot.subtitle = element_markdown(size = 10, lineheight = 1.3)))

suppressMessages(ggsave(file.path(dir_gb, sprintf('genebar_%s_%s.png', ko, SCHEME)), P,
                        width = 24, height = 2.1 * sum(hts) + 1.4, dpi = 100, limitsize = FALSE))
options(repr.plot.width = 24, repr.plot.height = 2.1 * sum(hts) + 1.4)
P


ko00710_Carbon fixation by Calvin cycle（炭素固定(Calvin)）  scheme B  全 40 遺伝子 / ORA hit 17


plot without title

In [67]:
# ── 図12-11: ko00720（その他の炭素固定）  scheme B ───────────────────
# セル12（準備）を先に実行しておくこと
#   （nc_A / nc_B / col_cat / strains4 / NCOL / dir_gb を使う）
ko = 'ko00720'; ko_jp = 'その他の炭素固定'; SCHEME = 'B'

# スキームに応じて使うものを切り替える
nc       = if (SCHEME == 'A') nc_A else nc_B
y_lab = ifelse(SCHEME == 'A',
               'normalized count  (A: pooled across all 4 strains)',
               'normalized count  (B: per-strain size factors)')
sub_norm = ifelse(SCHEME == 'A',
    paste0("**one size factor set computed from all 4 strains pooled** ",
           "(= scheme A, the community-level output; includes the shift in strain abundance)"),
    paste0("**size factors computed separately within each strain** ",
           "(= scheme B, within-strain regulation; excludes the shift in strain abundance)"))

tgt = ora$term[ora$source == 'KEGG' & grepl(paste0('^', ko, '_'), ora$term)][1]

# ORA で拾われた遺伝子（down/up のどちらでも拾われていれば ORA hit）
g_ora = ora %>% filter(scheme == SCHEME, source == 'KEGG', term == tgt, padj < 0.05) %>%
    pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()

# 経路に属する全遺伝子
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
    dplyr::select(gene_id, KEGG_Pathway) %>%
    tidyr::separate_rows(KEGG_Pathway, sep = ',') %>%
    filter(KEGG_Pathway == ko) %>% pull(gene_id) %>% unique()

cat(sprintf("%s（%s）  scheme %s  全 %d 遺伝子 / ORA hit %d%s\n",
            tgt, ko_jp, SCHEME, length(g_all), length(g_ora),
            ifelse(length(g_ora) == 0, '  ★ この経路はこのスキームの ORA では有意でない', '')))

# ★ DEG の判定もスキームに合わせる（(A) は padj_A、(B) は padj_pooled）
d = nc[intersect(g_all, rownames(nc)), , drop = FALSE] %>%
    tibble::rownames_to_column('gene_id') %>%
    tidyr::pivot_longer(-gene_id, names_to = 'sample', values_to = 'norm') %>%
    mutate(group = factor(ifelse(grepl('W', sample), 'WT', 'KO'), levels = c('WT','KO'))) %>%
    left_join(res %>% dplyr::select(gene_id, strain, Preferred_name, product,
                                    KEGG_ko, EC, padj_A, padj_pooled), by = 'gene_id') %>%
    mutate(padj_use = if (SCHEME == 'A') padj_A else padj_pooled,
           cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                           !is.na(padj_use) & padj_use < 0.05 ~ 'DEG only',
                           TRUE ~ 'not DEG'),
           cat = factor(cat, levels = names(col_cat)),
           label = ifelse(is.na(Preferred_name) | Preferred_name == '-',
                          substr(product, 1, 14), Preferred_name),
           K   = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           EC2 = sapply(strsplit(ifelse(is.na(EC), '-', EC), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           # ★ 3行にする（名前+gene_id / K番号 / EC）。
           #   1行にまとめると EC が複数ある遺伝子で左右が切れる
           panel = sprintf("<span style='color:%s'>**%s  %s<br>%s<br>EC %s**</span>",
                           col_cat[as.character(cat)], label, gene_id, K, EC2))

# ★ 検定は Welch の t 検定（R の t.test は既定が var.equal = FALSE）。
#   等分散を仮定する Student の t 検定ではない。n = 6 vs 5 で分散も揃わないため適切。
#   ★ DESeq2 の padj とは別の検定（負の二項分布 vs 正規分布）。多重比較の補正はしていない
tt = d %>% group_by(gene_id, strain, panel) %>%
    summarise(p = tryCatch(t.test(norm ~ group)$p.value, error = function(e) NA_real_),
              ymax = max(max(norm), max(tapply(norm, group, function(v) mean(v) + sd(v)))),
              .groups = 'drop') %>%
    mutate(star = cut(p, c(-Inf, .001, .01, .05, Inf), labels = c('***','**','*','n.s.')),
           star = ifelse(is.na(star), 'n.s.', as.character(star)))

# 株ごとのサブプロットを作り、patchwork で1枚にまとめる
subs = list(); hts = c()
for (s in strains4) {
    ds = d %>% filter(strain == s) %>% arrange(cat, label)
    if (nrow(ds) == 0) next
    ds$panel = factor(ds$panel, levels = unique(ds$panel))   # 分類順に並べる
    ts = tt %>% filter(strain == s) %>% mutate(panel = factor(panel, levels = levels(ds$panel)))
    n_gene = length(unique(ds$gene_id)); n_row = ceiling(n_gene / NCOL)
    subs[[s]] = ggplot(ds, aes(x = group, y = norm, fill = group)) +
        geom_bar(stat = 'summary', fun = 'mean', width = 0.65, colour = 'grey30', linewidth = 0.3) +
        # ★ SD のエラーバー（平均 ± SD の両側）。上側だけにすると errorbar が両端に
        #   キャップを描き、下のキャップが棒の上辺（＝平均）に乗って紛らわしい
        stat_summary(fun = mean,
                     fun.min = function(v) mean(v) - sd(v),
                     fun.max = function(v) mean(v) + sd(v),
                     geom = 'errorbar', width = 0.25, linewidth = 0.4, colour = 'grey20') +
        geom_quasirandom(width = 0.15, size = 1.3, alpha = 0.85, colour = 'grey20') +
        geom_text(data = ts, aes(x = 1.5, y = ymax * 1.18, label = star),
                  inherit.aes = FALSE, size = 3.6) +
        facet_wrap(~ panel, scales = 'free_y', ncol = NCOL) +
        scale_fill_manual(values = c(WT = 'grey75', KO = '#e05252'), name = NULL) +
        scale_y_continuous(expand = expansion(mult = c(0, 0.25))) +
        labs(title = sprintf('%s   (%d genes)', s, n_gene), x = NULL, y = y_lab) +
        theme_bw() +
        # ★ strip の灰色が濃いと橙・紫の文字が沈むので、ほぼ白まで薄くする
        theme(strip.background = element_rect(fill='grey97', colour='grey75', linewidth=0.3),
              strip.text = element_markdown(size = 9),
              plot.title = element_text(size = 13, face = 'bold'),
              axis.text = element_text(size = 7.5),
              axis.title.y = element_text(size = 9),
              panel.grid.major.x = element_blank())
    hts = c(hts, n_row + 0.45)   # 株の見出し分を足す
}

P = wrap_plots(subs, ncol = 1, heights = hts) +
    plot_layout(guides = 'collect') +
    # ★ 図中に日本語を入れない（PNG 保存時に mbcsToSbcs 変換が失敗して文字化けする）。
    #   日本語の経路名は上の cat() でテキストとして出す
    plot_annotation(title = sprintf('%s   [scheme %s]', tgt, SCHEME),
      subtitle = paste0(
        "panel name:  <span style='color:#d95f02'>**ORA hit**</span>  /  ",
        "<span style='color:#7570b3'>**DEG only**</span>  /  ",
        "<span style='color:#000000'>**not DEG**</span><br>",
        "y axis = **normalized count** - DESeq2 median-of-ratios with ", sub_norm, "<br>",
        "bar = mean, error bar = \u00b1SD, points = each mouse  |  ",
        "significance = **Welch two-sample t-test** on normalized counts ",
        "(*** p<0.001, ** p<0.01, * p<0.05; not DESeq2 padj, no multiple-testing correction)"),
      theme = theme(plot.title = element_text(size = 15, face = 'bold'),
                    plot.subtitle = element_markdown(size = 10, lineheight = 1.3)))

suppressMessages(ggsave(file.path(dir_gb, sprintf('genebar_%s_%s.png', ko, SCHEME)), P,
                        width = 24, height = 2.1 * sum(hts) + 1.4, dpi = 100, limitsize = FALSE))
options(repr.plot.width = 24, repr.plot.height = 2.1 * sum(hts) + 1.4)
P


ko00720_Other carbon fixation pathways（その他の炭素固定）  scheme B  全 81 遺伝子 / ORA hit 26


plot without title

In [68]:
# ── 図12-12: ko00120（一次胆汁酸）  scheme B ───────────────────
# セル12（準備）を先に実行しておくこと
#   （nc_A / nc_B / col_cat / strains4 / NCOL / dir_gb を使う）
ko = 'ko00120'; ko_jp = '一次胆汁酸'; SCHEME = 'B'

# スキームに応じて使うものを切り替える
nc       = if (SCHEME == 'A') nc_A else nc_B
y_lab = ifelse(SCHEME == 'A',
               'normalized count  (A: pooled across all 4 strains)',
               'normalized count  (B: per-strain size factors)')
sub_norm = ifelse(SCHEME == 'A',
    paste0("**one size factor set computed from all 4 strains pooled** ",
           "(= scheme A, the community-level output; includes the shift in strain abundance)"),
    paste0("**size factors computed separately within each strain** ",
           "(= scheme B, within-strain regulation; excludes the shift in strain abundance)"))

tgt = ora$term[ora$source == 'KEGG' & grepl(paste0('^', ko, '_'), ora$term)][1]

# ORA で拾われた遺伝子（down/up のどちらでも拾われていれば ORA hit）
g_ora = ora %>% filter(scheme == SCHEME, source == 'KEGG', term == tgt, padj < 0.05) %>%
    pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()

# 経路に属する全遺伝子
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
    dplyr::select(gene_id, KEGG_Pathway) %>%
    tidyr::separate_rows(KEGG_Pathway, sep = ',') %>%
    filter(KEGG_Pathway == ko) %>% pull(gene_id) %>% unique()

cat(sprintf("%s（%s）  scheme %s  全 %d 遺伝子 / ORA hit %d%s\n",
            tgt, ko_jp, SCHEME, length(g_all), length(g_ora),
            ifelse(length(g_ora) == 0, '  ★ この経路はこのスキームの ORA では有意でない', '')))

# ★ DEG の判定もスキームに合わせる（(A) は padj_A、(B) は padj_pooled）
d = nc[intersect(g_all, rownames(nc)), , drop = FALSE] %>%
    tibble::rownames_to_column('gene_id') %>%
    tidyr::pivot_longer(-gene_id, names_to = 'sample', values_to = 'norm') %>%
    mutate(group = factor(ifelse(grepl('W', sample), 'WT', 'KO'), levels = c('WT','KO'))) %>%
    left_join(res %>% dplyr::select(gene_id, strain, Preferred_name, product,
                                    KEGG_ko, EC, padj_A, padj_pooled), by = 'gene_id') %>%
    mutate(padj_use = if (SCHEME == 'A') padj_A else padj_pooled,
           cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                           !is.na(padj_use) & padj_use < 0.05 ~ 'DEG only',
                           TRUE ~ 'not DEG'),
           cat = factor(cat, levels = names(col_cat)),
           label = ifelse(is.na(Preferred_name) | Preferred_name == '-',
                          substr(product, 1, 14), Preferred_name),
           K   = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           EC2 = sapply(strsplit(ifelse(is.na(EC), '-', EC), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           # ★ 3行にする（名前+gene_id / K番号 / EC）。
           #   1行にまとめると EC が複数ある遺伝子で左右が切れる
           panel = sprintf("<span style='color:%s'>**%s  %s<br>%s<br>EC %s**</span>",
                           col_cat[as.character(cat)], label, gene_id, K, EC2))

# ★ 検定は Welch の t 検定（R の t.test は既定が var.equal = FALSE）。
#   等分散を仮定する Student の t 検定ではない。n = 6 vs 5 で分散も揃わないため適切。
#   ★ DESeq2 の padj とは別の検定（負の二項分布 vs 正規分布）。多重比較の補正はしていない
tt = d %>% group_by(gene_id, strain, panel) %>%
    summarise(p = tryCatch(t.test(norm ~ group)$p.value, error = function(e) NA_real_),
              ymax = max(max(norm), max(tapply(norm, group, function(v) mean(v) + sd(v)))),
              .groups = 'drop') %>%
    mutate(star = cut(p, c(-Inf, .001, .01, .05, Inf), labels = c('***','**','*','n.s.')),
           star = ifelse(is.na(star), 'n.s.', as.character(star)))

# 株ごとのサブプロットを作り、patchwork で1枚にまとめる
subs = list(); hts = c()
for (s in strains4) {
    ds = d %>% filter(strain == s) %>% arrange(cat, label)
    if (nrow(ds) == 0) next
    ds$panel = factor(ds$panel, levels = unique(ds$panel))   # 分類順に並べる
    ts = tt %>% filter(strain == s) %>% mutate(panel = factor(panel, levels = levels(ds$panel)))
    n_gene = length(unique(ds$gene_id)); n_row = ceiling(n_gene / NCOL)
    subs[[s]] = ggplot(ds, aes(x = group, y = norm, fill = group)) +
        geom_bar(stat = 'summary', fun = 'mean', width = 0.65, colour = 'grey30', linewidth = 0.3) +
        # ★ SD のエラーバー（平均 ± SD の両側）。上側だけにすると errorbar が両端に
        #   キャップを描き、下のキャップが棒の上辺（＝平均）に乗って紛らわしい
        stat_summary(fun = mean,
                     fun.min = function(v) mean(v) - sd(v),
                     fun.max = function(v) mean(v) + sd(v),
                     geom = 'errorbar', width = 0.25, linewidth = 0.4, colour = 'grey20') +
        geom_quasirandom(width = 0.15, size = 1.3, alpha = 0.85, colour = 'grey20') +
        geom_text(data = ts, aes(x = 1.5, y = ymax * 1.18, label = star),
                  inherit.aes = FALSE, size = 3.6) +
        facet_wrap(~ panel, scales = 'free_y', ncol = NCOL) +
        scale_fill_manual(values = c(WT = 'grey75', KO = '#e05252'), name = NULL) +
        scale_y_continuous(expand = expansion(mult = c(0, 0.25))) +
        labs(title = sprintf('%s   (%d genes)', s, n_gene), x = NULL, y = y_lab) +
        theme_bw() +
        # ★ strip の灰色が濃いと橙・紫の文字が沈むので、ほぼ白まで薄くする
        theme(strip.background = element_rect(fill='grey97', colour='grey75', linewidth=0.3),
              strip.text = element_markdown(size = 9),
              plot.title = element_text(size = 13, face = 'bold'),
              axis.text = element_text(size = 7.5),
              axis.title.y = element_text(size = 9),
              panel.grid.major.x = element_blank())
    hts = c(hts, n_row + 0.45)   # 株の見出し分を足す
}

P = wrap_plots(subs, ncol = 1, heights = hts) +
    plot_layout(guides = 'collect') +
    # ★ 図中に日本語を入れない（PNG 保存時に mbcsToSbcs 変換が失敗して文字化けする）。
    #   日本語の経路名は上の cat() でテキストとして出す
    plot_annotation(title = sprintf('%s   [scheme %s]', tgt, SCHEME),
      subtitle = paste0(
        "panel name:  <span style='color:#d95f02'>**ORA hit**</span>  /  ",
        "<span style='color:#7570b3'>**DEG only**</span>  /  ",
        "<span style='color:#000000'>**not DEG**</span><br>",
        "y axis = **normalized count** - DESeq2 median-of-ratios with ", sub_norm, "<br>",
        "bar = mean, error bar = \u00b1SD, points = each mouse  |  ",
        "significance = **Welch two-sample t-test** on normalized counts ",
        "(*** p<0.001, ** p<0.01, * p<0.05; not DESeq2 padj, no multiple-testing correction)"),
      theme = theme(plot.title = element_text(size = 15, face = 'bold'),
                    plot.subtitle = element_markdown(size = 10, lineheight = 1.3)))

suppressMessages(ggsave(file.path(dir_gb, sprintf('genebar_%s_%s.png', ko, SCHEME)), P,
                        width = 24, height = 2.1 * sum(hts) + 1.4, dpi = 100, limitsize = FALSE))
options(repr.plot.width = 24, repr.plot.height = 2.1 * sum(hts) + 1.4)
P


ko00120_Primary bile acid biosynthesis（一次胆汁酸）  scheme B  全 5 遺伝子 / ORA hit 4


plot without title

In [69]:
# ── 図12-13: ko00121（二次胆汁酸）  scheme B ───────────────────
# セル12（準備）を先に実行しておくこと
#   （nc_A / nc_B / col_cat / strains4 / NCOL / dir_gb を使う）
ko = 'ko00121'; ko_jp = '二次胆汁酸'; SCHEME = 'B'

# スキームに応じて使うものを切り替える
nc       = if (SCHEME == 'A') nc_A else nc_B
y_lab = ifelse(SCHEME == 'A',
               'normalized count  (A: pooled across all 4 strains)',
               'normalized count  (B: per-strain size factors)')
sub_norm = ifelse(SCHEME == 'A',
    paste0("**one size factor set computed from all 4 strains pooled** ",
           "(= scheme A, the community-level output; includes the shift in strain abundance)"),
    paste0("**size factors computed separately within each strain** ",
           "(= scheme B, within-strain regulation; excludes the shift in strain abundance)"))

tgt = ora$term[ora$source == 'KEGG' & grepl(paste0('^', ko, '_'), ora$term)][1]

# ORA で拾われた遺伝子（down/up のどちらでも拾われていれば ORA hit）
g_ora = ora %>% filter(scheme == SCHEME, source == 'KEGG', term == tgt, padj < 0.05) %>%
    pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()

# 経路に属する全遺伝子
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
    dplyr::select(gene_id, KEGG_Pathway) %>%
    tidyr::separate_rows(KEGG_Pathway, sep = ',') %>%
    filter(KEGG_Pathway == ko) %>% pull(gene_id) %>% unique()

cat(sprintf("%s（%s）  scheme %s  全 %d 遺伝子 / ORA hit %d%s\n",
            tgt, ko_jp, SCHEME, length(g_all), length(g_ora),
            ifelse(length(g_ora) == 0, '  ★ この経路はこのスキームの ORA では有意でない', '')))

# ★ DEG の判定もスキームに合わせる（(A) は padj_A、(B) は padj_pooled）
d = nc[intersect(g_all, rownames(nc)), , drop = FALSE] %>%
    tibble::rownames_to_column('gene_id') %>%
    tidyr::pivot_longer(-gene_id, names_to = 'sample', values_to = 'norm') %>%
    mutate(group = factor(ifelse(grepl('W', sample), 'WT', 'KO'), levels = c('WT','KO'))) %>%
    left_join(res %>% dplyr::select(gene_id, strain, Preferred_name, product,
                                    KEGG_ko, EC, padj_A, padj_pooled), by = 'gene_id') %>%
    mutate(padj_use = if (SCHEME == 'A') padj_A else padj_pooled,
           cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                           !is.na(padj_use) & padj_use < 0.05 ~ 'DEG only',
                           TRUE ~ 'not DEG'),
           cat = factor(cat, levels = names(col_cat)),
           label = ifelse(is.na(Preferred_name) | Preferred_name == '-',
                          substr(product, 1, 14), Preferred_name),
           K   = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           EC2 = sapply(strsplit(ifelse(is.na(EC), '-', EC), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           # ★ 3行にする（名前+gene_id / K番号 / EC）。
           #   1行にまとめると EC が複数ある遺伝子で左右が切れる
           panel = sprintf("<span style='color:%s'>**%s  %s<br>%s<br>EC %s**</span>",
                           col_cat[as.character(cat)], label, gene_id, K, EC2))

# ★ 検定は Welch の t 検定（R の t.test は既定が var.equal = FALSE）。
#   等分散を仮定する Student の t 検定ではない。n = 6 vs 5 で分散も揃わないため適切。
#   ★ DESeq2 の padj とは別の検定（負の二項分布 vs 正規分布）。多重比較の補正はしていない
tt = d %>% group_by(gene_id, strain, panel) %>%
    summarise(p = tryCatch(t.test(norm ~ group)$p.value, error = function(e) NA_real_),
              ymax = max(max(norm), max(tapply(norm, group, function(v) mean(v) + sd(v)))),
              .groups = 'drop') %>%
    mutate(star = cut(p, c(-Inf, .001, .01, .05, Inf), labels = c('***','**','*','n.s.')),
           star = ifelse(is.na(star), 'n.s.', as.character(star)))

# 株ごとのサブプロットを作り、patchwork で1枚にまとめる
subs = list(); hts = c()
for (s in strains4) {
    ds = d %>% filter(strain == s) %>% arrange(cat, label)
    if (nrow(ds) == 0) next
    ds$panel = factor(ds$panel, levels = unique(ds$panel))   # 分類順に並べる
    ts = tt %>% filter(strain == s) %>% mutate(panel = factor(panel, levels = levels(ds$panel)))
    n_gene = length(unique(ds$gene_id)); n_row = ceiling(n_gene / NCOL)
    subs[[s]] = ggplot(ds, aes(x = group, y = norm, fill = group)) +
        geom_bar(stat = 'summary', fun = 'mean', width = 0.65, colour = 'grey30', linewidth = 0.3) +
        # ★ SD のエラーバー（平均 ± SD の両側）。上側だけにすると errorbar が両端に
        #   キャップを描き、下のキャップが棒の上辺（＝平均）に乗って紛らわしい
        stat_summary(fun = mean,
                     fun.min = function(v) mean(v) - sd(v),
                     fun.max = function(v) mean(v) + sd(v),
                     geom = 'errorbar', width = 0.25, linewidth = 0.4, colour = 'grey20') +
        geom_quasirandom(width = 0.15, size = 1.3, alpha = 0.85, colour = 'grey20') +
        geom_text(data = ts, aes(x = 1.5, y = ymax * 1.18, label = star),
                  inherit.aes = FALSE, size = 3.6) +
        facet_wrap(~ panel, scales = 'free_y', ncol = NCOL) +
        scale_fill_manual(values = c(WT = 'grey75', KO = '#e05252'), name = NULL) +
        scale_y_continuous(expand = expansion(mult = c(0, 0.25))) +
        labs(title = sprintf('%s   (%d genes)', s, n_gene), x = NULL, y = y_lab) +
        theme_bw() +
        # ★ strip の灰色が濃いと橙・紫の文字が沈むので、ほぼ白まで薄くする
        theme(strip.background = element_rect(fill='grey97', colour='grey75', linewidth=0.3),
              strip.text = element_markdown(size = 9),
              plot.title = element_text(size = 13, face = 'bold'),
              axis.text = element_text(size = 7.5),
              axis.title.y = element_text(size = 9),
              panel.grid.major.x = element_blank())
    hts = c(hts, n_row + 0.45)   # 株の見出し分を足す
}

P = wrap_plots(subs, ncol = 1, heights = hts) +
    plot_layout(guides = 'collect') +
    # ★ 図中に日本語を入れない（PNG 保存時に mbcsToSbcs 変換が失敗して文字化けする）。
    #   日本語の経路名は上の cat() でテキストとして出す
    plot_annotation(title = sprintf('%s   [scheme %s]', tgt, SCHEME),
      subtitle = paste0(
        "panel name:  <span style='color:#d95f02'>**ORA hit**</span>  /  ",
        "<span style='color:#7570b3'>**DEG only**</span>  /  ",
        "<span style='color:#000000'>**not DEG**</span><br>",
        "y axis = **normalized count** - DESeq2 median-of-ratios with ", sub_norm, "<br>",
        "bar = mean, error bar = \u00b1SD, points = each mouse  |  ",
        "significance = **Welch two-sample t-test** on normalized counts ",
        "(*** p<0.001, ** p<0.01, * p<0.05; not DESeq2 padj, no multiple-testing correction)"),
      theme = theme(plot.title = element_text(size = 15, face = 'bold'),
                    plot.subtitle = element_markdown(size = 10, lineheight = 1.3)))

suppressMessages(ggsave(file.path(dir_gb, sprintf('genebar_%s_%s.png', ko, SCHEME)), P,
                        width = 24, height = 2.1 * sum(hts) + 1.4, dpi = 100, limitsize = FALSE))
options(repr.plot.width = 24, repr.plot.height = 2.1 * sum(hts) + 1.4)
P


ko00121_Secondary bile acid biosynthesis（二次胆汁酸）  scheme B  全 7 遺伝子 / ORA hit 6


plot without title

In [70]:
# ── 図12-14: ko03010（リボソーム）  scheme B ───────────────────
# セル12（準備）を先に実行しておくこと
#   （nc_A / nc_B / col_cat / strains4 / NCOL / dir_gb を使う）
ko = 'ko03010'; ko_jp = 'リボソーム'; SCHEME = 'B'

# スキームに応じて使うものを切り替える
nc       = if (SCHEME == 'A') nc_A else nc_B
y_lab = ifelse(SCHEME == 'A',
               'normalized count  (A: pooled across all 4 strains)',
               'normalized count  (B: per-strain size factors)')
sub_norm = ifelse(SCHEME == 'A',
    paste0("**one size factor set computed from all 4 strains pooled** ",
           "(= scheme A, the community-level output; includes the shift in strain abundance)"),
    paste0("**size factors computed separately within each strain** ",
           "(= scheme B, within-strain regulation; excludes the shift in strain abundance)"))

tgt = ora$term[ora$source == 'KEGG' & grepl(paste0('^', ko, '_'), ora$term)][1]

# ORA で拾われた遺伝子（down/up のどちらでも拾われていれば ORA hit）
g_ora = ora %>% filter(scheme == SCHEME, source == 'KEGG', term == tgt, padj < 0.05) %>%
    pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()

# 経路に属する全遺伝子
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
    dplyr::select(gene_id, KEGG_Pathway) %>%
    tidyr::separate_rows(KEGG_Pathway, sep = ',') %>%
    filter(KEGG_Pathway == ko) %>% pull(gene_id) %>% unique()

cat(sprintf("%s（%s）  scheme %s  全 %d 遺伝子 / ORA hit %d%s\n",
            tgt, ko_jp, SCHEME, length(g_all), length(g_ora),
            ifelse(length(g_ora) == 0, '  ★ この経路はこのスキームの ORA では有意でない', '')))

# ★ DEG の判定もスキームに合わせる（(A) は padj_A、(B) は padj_pooled）
d = nc[intersect(g_all, rownames(nc)), , drop = FALSE] %>%
    tibble::rownames_to_column('gene_id') %>%
    tidyr::pivot_longer(-gene_id, names_to = 'sample', values_to = 'norm') %>%
    mutate(group = factor(ifelse(grepl('W', sample), 'WT', 'KO'), levels = c('WT','KO'))) %>%
    left_join(res %>% dplyr::select(gene_id, strain, Preferred_name, product,
                                    KEGG_ko, EC, padj_A, padj_pooled), by = 'gene_id') %>%
    mutate(padj_use = if (SCHEME == 'A') padj_A else padj_pooled,
           cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                           !is.na(padj_use) & padj_use < 0.05 ~ 'DEG only',
                           TRUE ~ 'not DEG'),
           cat = factor(cat, levels = names(col_cat)),
           label = ifelse(is.na(Preferred_name) | Preferred_name == '-',
                          substr(product, 1, 14), Preferred_name),
           K   = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           EC2 = sapply(strsplit(ifelse(is.na(EC), '-', EC), ','),
                        function(v) paste(head(v, 2), collapse = ',')),
           # ★ 3行にする（名前+gene_id / K番号 / EC）。
           #   1行にまとめると EC が複数ある遺伝子で左右が切れる
           panel = sprintf("<span style='color:%s'>**%s  %s<br>%s<br>EC %s**</span>",
                           col_cat[as.character(cat)], label, gene_id, K, EC2))

# ★ 検定は Welch の t 検定（R の t.test は既定が var.equal = FALSE）。
#   等分散を仮定する Student の t 検定ではない。n = 6 vs 5 で分散も揃わないため適切。
#   ★ DESeq2 の padj とは別の検定（負の二項分布 vs 正規分布）。多重比較の補正はしていない
tt = d %>% group_by(gene_id, strain, panel) %>%
    summarise(p = tryCatch(t.test(norm ~ group)$p.value, error = function(e) NA_real_),
              ymax = max(max(norm), max(tapply(norm, group, function(v) mean(v) + sd(v)))),
              .groups = 'drop') %>%
    mutate(star = cut(p, c(-Inf, .001, .01, .05, Inf), labels = c('***','**','*','n.s.')),
           star = ifelse(is.na(star), 'n.s.', as.character(star)))

# 株ごとのサブプロットを作り、patchwork で1枚にまとめる
subs = list(); hts = c()
for (s in strains4) {
    ds = d %>% filter(strain == s) %>% arrange(cat, label)
    if (nrow(ds) == 0) next
    ds$panel = factor(ds$panel, levels = unique(ds$panel))   # 分類順に並べる
    ts = tt %>% filter(strain == s) %>% mutate(panel = factor(panel, levels = levels(ds$panel)))
    n_gene = length(unique(ds$gene_id)); n_row = ceiling(n_gene / NCOL)
    subs[[s]] = ggplot(ds, aes(x = group, y = norm, fill = group)) +
        geom_bar(stat = 'summary', fun = 'mean', width = 0.65, colour = 'grey30', linewidth = 0.3) +
        # ★ SD のエラーバー（平均 ± SD の両側）。上側だけにすると errorbar が両端に
        #   キャップを描き、下のキャップが棒の上辺（＝平均）に乗って紛らわしい
        stat_summary(fun = mean,
                     fun.min = function(v) mean(v) - sd(v),
                     fun.max = function(v) mean(v) + sd(v),
                     geom = 'errorbar', width = 0.25, linewidth = 0.4, colour = 'grey20') +
        geom_quasirandom(width = 0.15, size = 1.3, alpha = 0.85, colour = 'grey20') +
        geom_text(data = ts, aes(x = 1.5, y = ymax * 1.18, label = star),
                  inherit.aes = FALSE, size = 3.6) +
        facet_wrap(~ panel, scales = 'free_y', ncol = NCOL) +
        scale_fill_manual(values = c(WT = 'grey75', KO = '#e05252'), name = NULL) +
        scale_y_continuous(expand = expansion(mult = c(0, 0.25))) +
        labs(title = sprintf('%s   (%d genes)', s, n_gene), x = NULL, y = y_lab) +
        theme_bw() +
        # ★ strip の灰色が濃いと橙・紫の文字が沈むので、ほぼ白まで薄くする
        theme(strip.background = element_rect(fill='grey97', colour='grey75', linewidth=0.3),
              strip.text = element_markdown(size = 9),
              plot.title = element_text(size = 13, face = 'bold'),
              axis.text = element_text(size = 7.5),
              axis.title.y = element_text(size = 9),
              panel.grid.major.x = element_blank())
    hts = c(hts, n_row + 0.45)   # 株の見出し分を足す
}

P = wrap_plots(subs, ncol = 1, heights = hts) +
    plot_layout(guides = 'collect') +
    # ★ 図中に日本語を入れない（PNG 保存時に mbcsToSbcs 変換が失敗して文字化けする）。
    #   日本語の経路名は上の cat() でテキストとして出す
    plot_annotation(title = sprintf('%s   [scheme %s]', tgt, SCHEME),
      subtitle = paste0(
        "panel name:  <span style='color:#d95f02'>**ORA hit**</span>  /  ",
        "<span style='color:#7570b3'>**DEG only**</span>  /  ",
        "<span style='color:#000000'>**not DEG**</span><br>",
        "y axis = **normalized count** - DESeq2 median-of-ratios with ", sub_norm, "<br>",
        "bar = mean, error bar = \u00b1SD, points = each mouse  |  ",
        "significance = **Welch two-sample t-test** on normalized counts ",
        "(*** p<0.001, ** p<0.01, * p<0.05; not DESeq2 padj, no multiple-testing correction)"),
      theme = theme(plot.title = element_text(size = 15, face = 'bold'),
                    plot.subtitle = element_markdown(size = 10, lineheight = 1.3)))

suppressMessages(ggsave(file.path(dir_gb, sprintf('genebar_%s_%s.png', ko, SCHEME)), P,
                        width = 24, height = 2.1 * sum(hts) + 1.4, dpi = 100, limitsize = FALSE))
options(repr.plot.width = 24, repr.plot.height = 2.1 * sum(hts) + 1.4)
P


ko03010_Ribosome（リボソーム）  scheme B  全 112 遺伝子 / ORA hit 0  ★ この経路はこのスキームの ORA では有意でない


plot without title

### 考察: 「経路が下がった」の中身は株ごとにまったく違う

#### 1. ★ `Bun_8492` だけが逆向きの遺伝子を多数持つ（Table 1）

| ko | Bth_VPI | Bth_St8 | **Bun_8492** | Odo_St23 |
|---|---|---|---|---|
| `ko00020` ORA hit / DEG only | 9 / 0 | 9 / 0 | **3 / 12** | 3 / 2 |
| `ko00720` ORA hit / DEG only | 9 / 1 | 9 / 1 | **3 / 18** | 4 / 2 |

**`Bun_8492` は「DEG only」が突出して多く、中身はほとんどが上昇。**
→ **同じ経路の中で `Bth` 2株は下げ、`Bun_8492` は上げている。**
ORA は4株をまとめて検定するので、**多数派の `Bth` の向きだけが「経路の向き」として報告される。**

#### 2. ★★ ただし `Bun_8492` の上昇の約半分は組成シフト

この図の縦軸は **(A) の正規化カウント**で**菌の転写産物シェアの変化を含む。**
`Bun_8492` は **WT 54.8% → KO 65.1%（p = 0.0043）**（CLAUDE.md §7 ステージ4）。
LFC 中央値は **(A) +0.335 / (B) 0.000**（他3株は逆に (A) で −0.07〜−0.18）＝
**(A) では `Bun` の全遺伝子が一律 1.26 倍 底上げされている。**

| ko | Bun の DEG only | (A) LFC 中央値 | (B) LFC 中央値 | (B) でも有意 up |
|---|---|---|---|---|
| `ko00030` | 6 | +0.90 | **+0.50** | 4 / 6 |
| `ko00020` | 12 | +0.80 | **+0.39** | 6 / 12 |
| `ko00720` | 18 | +0.89 | **+0.47** | 10 / 18 |

**ゼロにはならない。**一律の底上げ（+0.335）を超えた分が残る。
→ **正しい言い方は「上昇のうち約半分は菌量の増加、残り半分は `Bun` 内部での実際の上方制御」。
「`Bun` が発現を上げた」と単純に書いてはいけない**（CLAUDE.md §7 ステージ5）。

#### 3. `ko00030` — 酸化的 PPP は下げ、解糖と PRPP 合成は上げる

`Bun_8492` の「DEG only」6個（`fbp`・`pfkA`・**`pfp`(2.7.1.90)**・**`prs`(2.7.6.1)**・`deoC`・
glycosyl hydrolase）は**すべて上昇**。一方 ORA hit 10個（`zwf`・`pgl`・`gnd`・`tal`・`tkt`・`fba`）は
すべて低下。**`prs`（PRPP synthetase）の上昇は、セル11 の「NADPH 産生を下げ核酸合成へ炭素を回す」と整合。**

#### 4. 同じ遺伝子名でもコピーによって挙動が違う

`Bth_VPI` の `fbaB` は `BT_RS08410`（ORA hit、低下 `***`、発現 2.4万）と `BT_RS05575`（n.s.、2,500）、
`pfkA` は3コピーのうち `BT_RS05555` だけが上昇 `**`。
**pathview では K 番号ごとに平均されるためこの違いは見えない。**

#### 5. `ko03010_Ribosome` は `Bun_8492` 以外ほぼ動いていない

`Odo_St23` は 52 遺伝子のうち **47 が not DEG**、`Bun_8492` は 54 のうち 34 が ORA hit。
**`Bth` 2株が3遺伝子しかないのは、リボソームタンパク質が保存的で `mappability ≥ 0.5` の
フィルタがほぼ全部落とすため**（ANI 98% の2株では区別できない）。
→ **「KO で翻訳が上がった」は `Bun_8492` の現象であり、`Bth` 2株については判定できていない。**

#### 6. 胆汁酸は役割が株で分かれている

`ko00120` に **`Bth` 2株の遺伝子は無い**（BSH を持たない。Bun 3 / Odo 2）。
`ko00121` で `Bth` に1個ずつあるのは `7α-HSDH`（`K00076`）。
→ **脱抱合は `Bun`/`Odo`、7α 酸化は `Bth`。**

#### 7. この図で読めないこと

- **`not DEG` の `n.s.` は「差が無い」ではなく「この検出力では差を示せない」**
  （n = 6 vs 5、`Odo_St23` は dispersion の収縮で保守的になる）
- **星は Welch の t 検定で DESeq2 の padj とは別。**分類（橙/紫/黒）は padj、星は t 検定なので
  **両者が食い違うパネルがありうる**
- **多重比較の補正をしていない。**経路内に数十遺伝子あるので `*` 1個は偶然でも出る

#### 8. (A) と (B) を並べて見る

`ko00720` の「DEG only」18 個は (A) で中央値 +0.89 → **(B) +0.47**、有意も **18 → 10** に減る。
**`ko03010` の (B)（図12-14）は ORA hit が 0 個**で全パネルが紫か黒になり、
**「リボソームの上昇は (A) でしか ORA に出ない」**こと自体が所見。
ただし **(B) でも `Bun` の 54 遺伝子中 34 が `padj_pooled < 0.05`** なのでゼロではない
（セル6 で (B) up KEGG が 0 本だったのは BH 補正後の話）。

#### 保存

`pathway/genebar/genebar_<ko>_<scheme>.png`（7経路 × 2スキーム = 14 枚）


## セル12-2: PFAM グループごとの遺伝子発現 棒グラフ（株ごと・有意差つき）

セル12 を **KEGG 経路 → PFAM の COG グループ**に置き換えたもの。図の作り（株ごとのパネル／
WT・KO の棒／各マウスの点／±SD／Welch の t 検定）は**セル12 とまったく同じ**で、
**セル12（準備）の `nc_A` / `nc_B` / `col_cat` / `strains4` / `NCOL` / `dir_gb` をそのまま使う。**

### なぜ PFAM をグループで描くか

セル7-4 で実測したとおり、**PFAM の有意 term どうしは遺伝子をほとんど共有していない**
（Jaccard > 0 のペアは (A) 2.2% / (B) 2.1%）。term 単位で描くと 22〜30 枚並ぶだけで、
しかも多くは同じタンパク質の別ドメイン（`Sod_Fe_N` ＝ `Sod_Fe_C` など）の重複になる。
→ セル7-4 と同じ **「構成遺伝子の主たる COG カテゴリ」**で束ねた単位で描く。

### 描く対象

| 図 | scheme | COG グループ | 含む PFAM term | 全遺伝子 / ORA hit |
|---|---|---|---|---|
| 図12-2-1 | (A) | M: Cell wall/membrane/envelope | `OmpA`,`OMP_b-brl`,`GNVR` | 82 / 38 |
| 図12-2-2 | (B) | 同 | `OmpA`,`OMP_b-brl`,`Patatin`,`MS_channel` | 87 / 43 |
| 図12-2-3 | (B) | I: Lipid transport | `CPSase_L_D2`,`Biotin_carb_C/N` | 10 / 7 |
| **図12-2-4 / 5** | (A) / (B) | **I カテゴリ全体**（PFAM ではなく COG） | — | 186 / 187 |

★ **`I: Lipid transport` は (A) の有意 PFAM には存在しない。**「無い」ことも結果なので (B) のみ描く。
★ **ORA に当たっていない遺伝子も全部描く**（橙 = ORA hit / 紫 = DEG only / 黒 = not DEG）。
「そのドメインを持つが動いていない遺伝子」が何本あるかを同じ図で読むため。

### ★ セル12 から変えた点（いずれも実際に描いて確認した）

| 変更 | 理由 |
|---|---|
| パネル名を **`gene_id` / 名前 / PFAM ドメイン**の3行に（セル12 は「名前+gene_id」/ K番号 / EC） | PFAM の遺伝子は `Preferred_name` も K/EC も無いものが多く、`product` の先頭を名前に使うため1行目が長くなり **パネル幅から左右がはみ出て切れた** |
| ドメイン名が 24 文字超ならコンマごとに改行 | `Biotin_carb_C,Biotin_carb_N,CPSase_L_D2` が切れたため |
| 列数を `min(NCOL, 最も遺伝子が多い株の数)`、幅もそれに合わせる（下限 16 inch） | I グループは最大4遺伝子で `ncol = 12` だとパネルが横に引き伸ばされた。下限 16 inch はこれより狭いと subtitle が右で切れるため |

### 読み方の注意（セル12 と同じ）

星は **Welch の t 検定**で **DESeq2 の `padj` とは別**（パネル名の色のほうが DESeq2 の判定）。
y 軸は**パネルごとに自由スケール**なので**パネル間で高さを比べてはいけない。**
(A) は菌量の変化を含み、(B) は含まない。


In [160]:
# ── 図12-2-1: PFAM の COG M グループ（細胞壁・膜・エンベロープ）  scheme A ──
# セル12（準備）を先に実行しておくこと
#   （nc_A / nc_B / col_cat / strains4 / NCOL / dir_gb を使う）
#
# ★ セル12 は KEGG 経路ごとに描いた。ここは PFAM の有意 term を
#   セル7-4 と同じ「主たる COG カテゴリ」で束ねたグループごとに描く。
#   ゲノム位置マップ（セル13 相当）は作らない。
library(ggplot2); library(ggbeeswarm); library(ggtext); library(patchwork)

SCHEME = 'A'; COG_LETTER = 'M'
COG_NAME = 'M: Cell wall/membrane/envelope biogenesis'
PF_TERMS = c('PFAM:OmpA', 'PFAM:OMP_b-brl', 'PFAM:GNVR')

nc = if (SCHEME == 'A') nc_A else nc_B
y_lab = ifelse(SCHEME == 'A',
               'normalized count  (A: pooled across all 4 strains)',
               'normalized count  (B: per-strain size factors)')
sub_norm = ifelse(SCHEME == 'A',
    paste0("**one size factor set computed from all 4 strains pooled** ",
           "(= scheme A, the community-level output; includes the shift in strain abundance)"),
    paste0("**size factors computed separately within each strain** ",
           "(= scheme B, within-strain regulation; excludes the shift in strain abundance)"))

pf_names = sub('^PFAM:', '', PF_TERMS)

# ORA で拾われた遺伝子（このグループのどれかの term に入っていれば ORA hit）
g_ora = ora %>% filter(scheme == SCHEME, source == 'PFAM',
                       term %in% PF_TERMS, padj < 0.05) %>%
    pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()

# このグループの PFAM ドメインを持つ全遺伝子。
# ★ ORA に当たっていない遺伝子も落とさず全部描く（「動いていない」ことも結果）
pf_gene = res %>% filter(!is.na(PFAMs), !marker) %>%
    dplyr::select(gene_id, PFAMs) %>% tidyr::separate_rows(PFAMs, sep = ',') %>%
    mutate(PFAMs = str_trim(PFAMs)) %>% filter(PFAMs %in% pf_names)
g_all = unique(pf_gene$gene_id)

# 各遺伝子が持つ「このグループの」ドメイン名（パネルの3行目に出す）
pf_lab = pf_gene %>% group_by(gene_id) %>%
    summarise(pf = paste(sort(unique(PFAMs)), collapse = ','), .groups = 'drop') %>%
    # ★ ドメイン名を並べるとパネル幅（2 inch ≒ 26 文字）を超えて左右が切れる。
    #   24 文字を超えるものはコンマごとに改行する
    mutate(pf = ifelse(nchar(pf) > 24, gsub(',', '<br>', pf), pf))

cat(sprintf("COG %s  scheme %s\n  PFAM: %s\n  全 %d 遺伝子 / ORA hit %d\n",
            COG_NAME, SCHEME, paste(pf_names, collapse = ', '),
            length(g_all), length(g_ora)))

# ★ DEG の判定もスキームに合わせる（(A) は padj_A、(B) は padj_pooled）
d = nc[intersect(g_all, rownames(nc)), , drop = FALSE] %>%
    tibble::rownames_to_column('gene_id') %>%
    tidyr::pivot_longer(-gene_id, names_to = 'sample', values_to = 'norm') %>%
    mutate(group = factor(ifelse(grepl('W', sample), 'WT', 'KO'), levels = c('WT','KO'))) %>%
    left_join(res %>% dplyr::select(gene_id, strain, Preferred_name, product,
                                    padj_A, padj_pooled), by = 'gene_id') %>%
    left_join(pf_lab, by = 'gene_id') %>%
    mutate(padj_use = if (SCHEME == 'A') padj_A else padj_pooled,
           cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                           !is.na(padj_use) & padj_use < 0.05 ~ 'DEG only',
                           TRUE ~ 'not DEG'),
           cat = factor(cat, levels = names(col_cat)),
           label = ifelse(is.na(Preferred_name) | Preferred_name == '-',
                          substr(product, 1, 18), Preferred_name),
           # ★ セル12 は「名前＋gene_id」/ K番号 / EC の3行だった。PFAM の遺伝子は
           #   Preferred_name も K/EC も付いていないものが多く、product の先頭を
           #   名前に使うので1行目が長くなり ★ パネル幅から左右がはみ出て切れた。
           #   gene_id を独立した1行目にし、3行目は K/EC の代わりに
           #   「このグループのどのドメインを持つか」を出す
           panel = sprintf("<span style='color:%s'>**%s<br>%s<br>%s**</span>",
                           col_cat[as.character(cat)], gene_id, label, pf))

# ★ 検定は Welch の t 検定（R の t.test は既定が var.equal = FALSE）。
#   ★ DESeq2 の padj とは別の検定（負の二項分布 vs 正規分布）。多重比較の補正はしていない
tt = d %>% group_by(gene_id, strain, panel) %>%
    summarise(p = tryCatch(t.test(norm ~ group)$p.value, error = function(e) NA_real_),
              ymax = max(max(norm), max(tapply(norm, group, function(v) mean(v) + sd(v)))),
              .groups = 'drop') %>%
    mutate(star = cut(p, c(-Inf, .001, .01, .05, Inf), labels = c('***','**','*','n.s.')),
           star = ifelse(is.na(star), 'n.s.', as.character(star)))

# ★ 1行の列数は「最も遺伝子が多い株の数」を上限にする。遺伝子が少ないグループで
#   ncol = NCOL のままだと、パネルが横に引き伸ばされて間延びする
n_max = max(sapply(strains4, function(s) length(unique(d$gene_id[d$strain == s]))))

subs = list(); hts = c()
for (s in strains4) {
    ds = d %>% filter(strain == s) %>% arrange(cat, label)
    if (nrow(ds) == 0) next
    ds$panel = factor(ds$panel, levels = unique(ds$panel))   # 分類順に並べる
    ts = tt %>% filter(strain == s) %>% mutate(panel = factor(panel, levels = levels(ds$panel)))
    n_gene = length(unique(ds$gene_id)); n_row = ceiling(n_gene / min(NCOL, n_max))
    subs[[s]] = ggplot(ds, aes(x = group, y = norm, fill = group)) +
        geom_bar(stat = 'summary', fun = 'mean', width = 0.65, colour = 'grey30', linewidth = 0.3) +
        # ★ SD のエラーバー（平均 ± SD の両側）。上側だけにすると errorbar が両端に
        #   キャップを描き、下のキャップが棒の上辺（＝平均）に乗って紛らわしい
        stat_summary(fun = mean,
                     fun.min = function(v) mean(v) - sd(v),
                     fun.max = function(v) mean(v) + sd(v),
                     geom = 'errorbar', width = 0.25, linewidth = 0.4, colour = 'grey20') +
        geom_quasirandom(width = 0.15, size = 1.3, alpha = 0.85, colour = 'grey20') +
        geom_text(data = ts, aes(x = 1.5, y = ymax * 1.18, label = star),
                  inherit.aes = FALSE, size = 3.6) +
        facet_wrap(~ panel, scales = 'free_y', ncol = min(NCOL, n_max)) +
        scale_fill_manual(values = c(WT = 'grey75', KO = '#e05252'), name = NULL) +
        scale_y_continuous(expand = expansion(mult = c(0, 0.25))) +
        labs(title = sprintf('%s   (%d genes)', s, n_gene), x = NULL, y = y_lab) +
        theme_bw() +
        # ★ strip の灰色が濃いと橙・紫の文字が沈むので、ほぼ白まで薄くする
        theme(strip.background = element_rect(fill='grey97', colour='grey75', linewidth=0.3),
              strip.text = element_markdown(size = 9),
              plot.title = element_text(size = 13, face = 'bold'),
              axis.text = element_text(size = 7.5),
              axis.title.y = element_text(size = 9),
              panel.grid.major.x = element_blank())
    hts = c(hts, n_row + 0.45)   # 株の見出し分を足す
}

P = wrap_plots(subs, ncol = 1, heights = hts) +
    plot_layout(guides = 'collect') +
    # ★ 図中に日本語を入れない（PNG 保存時に mbcsToSbcs 変換が失敗して文字化けする）
    plot_annotation(title = sprintf('COG %s   [scheme %s]', COG_NAME, SCHEME),
      subtitle = paste0(
        "PFAM domains in this group: **", paste(pf_names, collapse = ', '), "**<br>",
        "panel name:  <span style='color:#d95f02'>**ORA hit**</span>  /  ",
        "<span style='color:#7570b3'>**DEG only**</span>  /  ",
        "<span style='color:#000000'>**not DEG**</span><br>",
        "y axis = **normalized count** - DESeq2 median-of-ratios with ", sub_norm, "<br>",
        "bar = mean, error bar = ±SD, points = each mouse  |  ",
        "significance = **Welch two-sample t-test** on normalized counts ",
        "(*** p<0.001, ** p<0.01, * p<0.05; not DESeq2 padj, no multiple-testing correction)"),
      theme = theme(plot.title = element_text(size = 15, face = 'bold'),
                    plot.subtitle = element_markdown(size = 10, lineheight = 1.3)))

# ★ 幅も遺伝子数に合わせる（パネル1枚 24/NCOL = 2 inch を保つ）。
#   ただし下限は 16 inch。これより狭いと上の subtitle の行が右で切れる
W_FIG = max(16, 24 * min(NCOL, n_max) / NCOL)
suppressMessages(ggsave(file.path(dir_gb, sprintf('genebar_PFAM_%s_%s.png', COG_LETTER, SCHEME)), P,
                        width = W_FIG, height = 2.1 * sum(hts) + 1.4, dpi = 100, limitsize = FALSE))
options(repr.plot.width = W_FIG, repr.plot.height = 2.1 * sum(hts) + 1.4)
P


COG M: Cell wall/membrane/envelope biogenesis  scheme A
  PFAM: OmpA, OMP_b-brl, GNVR
  全 82 遺伝子 / ORA hit 38


plot without title

In [161]:
# ── 図12-2-2: PFAM の COG M グループ（細胞壁・膜・エンベロープ）  scheme B ──
# セル12（準備）を先に実行しておくこと
#   （nc_A / nc_B / col_cat / strains4 / NCOL / dir_gb を使う）
#
# ★ セル12 は KEGG 経路ごとに描いた。ここは PFAM の有意 term を
#   セル7-4 と同じ「主たる COG カテゴリ」で束ねたグループごとに描く。
#   ゲノム位置マップ（セル13 相当）は作らない。
library(ggplot2); library(ggbeeswarm); library(ggtext); library(patchwork)

SCHEME = 'B'; COG_LETTER = 'M'
COG_NAME = 'M: Cell wall/membrane/envelope biogenesis'
PF_TERMS = c('PFAM:OmpA', 'PFAM:OMP_b-brl', 'PFAM:Patatin', 'PFAM:MS_channel')

nc = if (SCHEME == 'A') nc_A else nc_B
y_lab = ifelse(SCHEME == 'A',
               'normalized count  (A: pooled across all 4 strains)',
               'normalized count  (B: per-strain size factors)')
sub_norm = ifelse(SCHEME == 'A',
    paste0("**one size factor set computed from all 4 strains pooled** ",
           "(= scheme A, the community-level output; includes the shift in strain abundance)"),
    paste0("**size factors computed separately within each strain** ",
           "(= scheme B, within-strain regulation; excludes the shift in strain abundance)"))

pf_names = sub('^PFAM:', '', PF_TERMS)

# ORA で拾われた遺伝子（このグループのどれかの term に入っていれば ORA hit）
g_ora = ora %>% filter(scheme == SCHEME, source == 'PFAM',
                       term %in% PF_TERMS, padj < 0.05) %>%
    pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()

# このグループの PFAM ドメインを持つ全遺伝子。
# ★ ORA に当たっていない遺伝子も落とさず全部描く（「動いていない」ことも結果）
pf_gene = res %>% filter(!is.na(PFAMs), !marker) %>%
    dplyr::select(gene_id, PFAMs) %>% tidyr::separate_rows(PFAMs, sep = ',') %>%
    mutate(PFAMs = str_trim(PFAMs)) %>% filter(PFAMs %in% pf_names)
g_all = unique(pf_gene$gene_id)

# 各遺伝子が持つ「このグループの」ドメイン名（パネルの3行目に出す）
pf_lab = pf_gene %>% group_by(gene_id) %>%
    summarise(pf = paste(sort(unique(PFAMs)), collapse = ','), .groups = 'drop') %>%
    # ★ ドメイン名を並べるとパネル幅（2 inch ≒ 26 文字）を超えて左右が切れる。
    #   24 文字を超えるものはコンマごとに改行する
    mutate(pf = ifelse(nchar(pf) > 24, gsub(',', '<br>', pf), pf))

cat(sprintf("COG %s  scheme %s\n  PFAM: %s\n  全 %d 遺伝子 / ORA hit %d\n",
            COG_NAME, SCHEME, paste(pf_names, collapse = ', '),
            length(g_all), length(g_ora)))

# ★ DEG の判定もスキームに合わせる（(A) は padj_A、(B) は padj_pooled）
d = nc[intersect(g_all, rownames(nc)), , drop = FALSE] %>%
    tibble::rownames_to_column('gene_id') %>%
    tidyr::pivot_longer(-gene_id, names_to = 'sample', values_to = 'norm') %>%
    mutate(group = factor(ifelse(grepl('W', sample), 'WT', 'KO'), levels = c('WT','KO'))) %>%
    left_join(res %>% dplyr::select(gene_id, strain, Preferred_name, product,
                                    padj_A, padj_pooled), by = 'gene_id') %>%
    left_join(pf_lab, by = 'gene_id') %>%
    mutate(padj_use = if (SCHEME == 'A') padj_A else padj_pooled,
           cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                           !is.na(padj_use) & padj_use < 0.05 ~ 'DEG only',
                           TRUE ~ 'not DEG'),
           cat = factor(cat, levels = names(col_cat)),
           label = ifelse(is.na(Preferred_name) | Preferred_name == '-',
                          substr(product, 1, 18), Preferred_name),
           # ★ セル12 は「名前＋gene_id」/ K番号 / EC の3行だった。PFAM の遺伝子は
           #   Preferred_name も K/EC も付いていないものが多く、product の先頭を
           #   名前に使うので1行目が長くなり ★ パネル幅から左右がはみ出て切れた。
           #   gene_id を独立した1行目にし、3行目は K/EC の代わりに
           #   「このグループのどのドメインを持つか」を出す
           panel = sprintf("<span style='color:%s'>**%s<br>%s<br>%s**</span>",
                           col_cat[as.character(cat)], gene_id, label, pf))

# ★ 検定は Welch の t 検定（R の t.test は既定が var.equal = FALSE）。
#   ★ DESeq2 の padj とは別の検定（負の二項分布 vs 正規分布）。多重比較の補正はしていない
tt = d %>% group_by(gene_id, strain, panel) %>%
    summarise(p = tryCatch(t.test(norm ~ group)$p.value, error = function(e) NA_real_),
              ymax = max(max(norm), max(tapply(norm, group, function(v) mean(v) + sd(v)))),
              .groups = 'drop') %>%
    mutate(star = cut(p, c(-Inf, .001, .01, .05, Inf), labels = c('***','**','*','n.s.')),
           star = ifelse(is.na(star), 'n.s.', as.character(star)))

# ★ 1行の列数は「最も遺伝子が多い株の数」を上限にする。遺伝子が少ないグループで
#   ncol = NCOL のままだと、パネルが横に引き伸ばされて間延びする
n_max = max(sapply(strains4, function(s) length(unique(d$gene_id[d$strain == s]))))

subs = list(); hts = c()
for (s in strains4) {
    ds = d %>% filter(strain == s) %>% arrange(cat, label)
    if (nrow(ds) == 0) next
    ds$panel = factor(ds$panel, levels = unique(ds$panel))   # 分類順に並べる
    ts = tt %>% filter(strain == s) %>% mutate(panel = factor(panel, levels = levels(ds$panel)))
    n_gene = length(unique(ds$gene_id)); n_row = ceiling(n_gene / min(NCOL, n_max))
    subs[[s]] = ggplot(ds, aes(x = group, y = norm, fill = group)) +
        geom_bar(stat = 'summary', fun = 'mean', width = 0.65, colour = 'grey30', linewidth = 0.3) +
        # ★ SD のエラーバー（平均 ± SD の両側）。上側だけにすると errorbar が両端に
        #   キャップを描き、下のキャップが棒の上辺（＝平均）に乗って紛らわしい
        stat_summary(fun = mean,
                     fun.min = function(v) mean(v) - sd(v),
                     fun.max = function(v) mean(v) + sd(v),
                     geom = 'errorbar', width = 0.25, linewidth = 0.4, colour = 'grey20') +
        geom_quasirandom(width = 0.15, size = 1.3, alpha = 0.85, colour = 'grey20') +
        geom_text(data = ts, aes(x = 1.5, y = ymax * 1.18, label = star),
                  inherit.aes = FALSE, size = 3.6) +
        facet_wrap(~ panel, scales = 'free_y', ncol = min(NCOL, n_max)) +
        scale_fill_manual(values = c(WT = 'grey75', KO = '#e05252'), name = NULL) +
        scale_y_continuous(expand = expansion(mult = c(0, 0.25))) +
        labs(title = sprintf('%s   (%d genes)', s, n_gene), x = NULL, y = y_lab) +
        theme_bw() +
        # ★ strip の灰色が濃いと橙・紫の文字が沈むので、ほぼ白まで薄くする
        theme(strip.background = element_rect(fill='grey97', colour='grey75', linewidth=0.3),
              strip.text = element_markdown(size = 9),
              plot.title = element_text(size = 13, face = 'bold'),
              axis.text = element_text(size = 7.5),
              axis.title.y = element_text(size = 9),
              panel.grid.major.x = element_blank())
    hts = c(hts, n_row + 0.45)   # 株の見出し分を足す
}

P = wrap_plots(subs, ncol = 1, heights = hts) +
    plot_layout(guides = 'collect') +
    # ★ 図中に日本語を入れない（PNG 保存時に mbcsToSbcs 変換が失敗して文字化けする）
    plot_annotation(title = sprintf('COG %s   [scheme %s]', COG_NAME, SCHEME),
      subtitle = paste0(
        "PFAM domains in this group: **", paste(pf_names, collapse = ', '), "**<br>",
        "panel name:  <span style='color:#d95f02'>**ORA hit**</span>  /  ",
        "<span style='color:#7570b3'>**DEG only**</span>  /  ",
        "<span style='color:#000000'>**not DEG**</span><br>",
        "y axis = **normalized count** - DESeq2 median-of-ratios with ", sub_norm, "<br>",
        "bar = mean, error bar = ±SD, points = each mouse  |  ",
        "significance = **Welch two-sample t-test** on normalized counts ",
        "(*** p<0.001, ** p<0.01, * p<0.05; not DESeq2 padj, no multiple-testing correction)"),
      theme = theme(plot.title = element_text(size = 15, face = 'bold'),
                    plot.subtitle = element_markdown(size = 10, lineheight = 1.3)))

# ★ 幅も遺伝子数に合わせる（パネル1枚 24/NCOL = 2 inch を保つ）。
#   ただし下限は 16 inch。これより狭いと上の subtitle の行が右で切れる
W_FIG = max(16, 24 * min(NCOL, n_max) / NCOL)
suppressMessages(ggsave(file.path(dir_gb, sprintf('genebar_PFAM_%s_%s.png', COG_LETTER, SCHEME)), P,
                        width = W_FIG, height = 2.1 * sum(hts) + 1.4, dpi = 100, limitsize = FALSE))
options(repr.plot.width = W_FIG, repr.plot.height = 2.1 * sum(hts) + 1.4)
P


COG M: Cell wall/membrane/envelope biogenesis  scheme B
  PFAM: OmpA, OMP_b-brl, Patatin, MS_channel
  全 87 遺伝子 / ORA hit 43


plot without title

In [162]:
# ── 図12-2-3: PFAM の COG I グループ（脂質の輸送と代謝）  scheme B ──
# セル12（準備）を先に実行しておくこと
#   （nc_A / nc_B / col_cat / strains4 / NCOL / dir_gb を使う）
#
# ★ セル12 は KEGG 経路ごとに描いた。ここは PFAM の有意 term を
#   セル7-4 と同じ「主たる COG カテゴリ」で束ねたグループごとに描く。
#   ゲノム位置マップ（セル13 相当）は作らない。
library(ggplot2); library(ggbeeswarm); library(ggtext); library(patchwork)

SCHEME = 'B'; COG_LETTER = 'I'
COG_NAME = 'I: Lipid transport and metabolism'
PF_TERMS = c('PFAM:CPSase_L_D2', 'PFAM:Biotin_carb_C', 'PFAM:Biotin_carb_N')

nc = if (SCHEME == 'A') nc_A else nc_B
y_lab = ifelse(SCHEME == 'A',
               'normalized count  (A: pooled across all 4 strains)',
               'normalized count  (B: per-strain size factors)')
sub_norm = ifelse(SCHEME == 'A',
    paste0("**one size factor set computed from all 4 strains pooled** ",
           "(= scheme A, the community-level output; includes the shift in strain abundance)"),
    paste0("**size factors computed separately within each strain** ",
           "(= scheme B, within-strain regulation; excludes the shift in strain abundance)"))

pf_names = sub('^PFAM:', '', PF_TERMS)

# ORA で拾われた遺伝子（このグループのどれかの term に入っていれば ORA hit）
g_ora = ora %>% filter(scheme == SCHEME, source == 'PFAM',
                       term %in% PF_TERMS, padj < 0.05) %>%
    pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()

# このグループの PFAM ドメインを持つ全遺伝子。
# ★ ORA に当たっていない遺伝子も落とさず全部描く（「動いていない」ことも結果）
pf_gene = res %>% filter(!is.na(PFAMs), !marker) %>%
    dplyr::select(gene_id, PFAMs) %>% tidyr::separate_rows(PFAMs, sep = ',') %>%
    mutate(PFAMs = str_trim(PFAMs)) %>% filter(PFAMs %in% pf_names)
g_all = unique(pf_gene$gene_id)

# 各遺伝子が持つ「このグループの」ドメイン名（パネルの3行目に出す）
pf_lab = pf_gene %>% group_by(gene_id) %>%
    summarise(pf = paste(sort(unique(PFAMs)), collapse = ','), .groups = 'drop') %>%
    # ★ ドメイン名を並べるとパネル幅（2 inch ≒ 26 文字）を超えて左右が切れる。
    #   24 文字を超えるものはコンマごとに改行する
    mutate(pf = ifelse(nchar(pf) > 24, gsub(',', '<br>', pf), pf))

cat(sprintf("COG %s  scheme %s\n  PFAM: %s\n  全 %d 遺伝子 / ORA hit %d\n",
            COG_NAME, SCHEME, paste(pf_names, collapse = ', '),
            length(g_all), length(g_ora)))

# ★ DEG の判定もスキームに合わせる（(A) は padj_A、(B) は padj_pooled）
d = nc[intersect(g_all, rownames(nc)), , drop = FALSE] %>%
    tibble::rownames_to_column('gene_id') %>%
    tidyr::pivot_longer(-gene_id, names_to = 'sample', values_to = 'norm') %>%
    mutate(group = factor(ifelse(grepl('W', sample), 'WT', 'KO'), levels = c('WT','KO'))) %>%
    left_join(res %>% dplyr::select(gene_id, strain, Preferred_name, product,
                                    padj_A, padj_pooled), by = 'gene_id') %>%
    left_join(pf_lab, by = 'gene_id') %>%
    mutate(padj_use = if (SCHEME == 'A') padj_A else padj_pooled,
           cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                           !is.na(padj_use) & padj_use < 0.05 ~ 'DEG only',
                           TRUE ~ 'not DEG'),
           cat = factor(cat, levels = names(col_cat)),
           label = ifelse(is.na(Preferred_name) | Preferred_name == '-',
                          substr(product, 1, 18), Preferred_name),
           # ★ セル12 は「名前＋gene_id」/ K番号 / EC の3行だった。PFAM の遺伝子は
           #   Preferred_name も K/EC も付いていないものが多く、product の先頭を
           #   名前に使うので1行目が長くなり ★ パネル幅から左右がはみ出て切れた。
           #   gene_id を独立した1行目にし、3行目は K/EC の代わりに
           #   「このグループのどのドメインを持つか」を出す
           panel = sprintf("<span style='color:%s'>**%s<br>%s<br>%s**</span>",
                           col_cat[as.character(cat)], gene_id, label, pf))

# ★ 検定は Welch の t 検定（R の t.test は既定が var.equal = FALSE）。
#   ★ DESeq2 の padj とは別の検定（負の二項分布 vs 正規分布）。多重比較の補正はしていない
tt = d %>% group_by(gene_id, strain, panel) %>%
    summarise(p = tryCatch(t.test(norm ~ group)$p.value, error = function(e) NA_real_),
              ymax = max(max(norm), max(tapply(norm, group, function(v) mean(v) + sd(v)))),
              .groups = 'drop') %>%
    mutate(star = cut(p, c(-Inf, .001, .01, .05, Inf), labels = c('***','**','*','n.s.')),
           star = ifelse(is.na(star), 'n.s.', as.character(star)))

# ★ 1行の列数は「最も遺伝子が多い株の数」を上限にする。遺伝子が少ないグループで
#   ncol = NCOL のままだと、パネルが横に引き伸ばされて間延びする
n_max = max(sapply(strains4, function(s) length(unique(d$gene_id[d$strain == s]))))

subs = list(); hts = c()
for (s in strains4) {
    ds = d %>% filter(strain == s) %>% arrange(cat, label)
    if (nrow(ds) == 0) next
    ds$panel = factor(ds$panel, levels = unique(ds$panel))   # 分類順に並べる
    ts = tt %>% filter(strain == s) %>% mutate(panel = factor(panel, levels = levels(ds$panel)))
    n_gene = length(unique(ds$gene_id)); n_row = ceiling(n_gene / min(NCOL, n_max))
    subs[[s]] = ggplot(ds, aes(x = group, y = norm, fill = group)) +
        geom_bar(stat = 'summary', fun = 'mean', width = 0.65, colour = 'grey30', linewidth = 0.3) +
        # ★ SD のエラーバー（平均 ± SD の両側）。上側だけにすると errorbar が両端に
        #   キャップを描き、下のキャップが棒の上辺（＝平均）に乗って紛らわしい
        stat_summary(fun = mean,
                     fun.min = function(v) mean(v) - sd(v),
                     fun.max = function(v) mean(v) + sd(v),
                     geom = 'errorbar', width = 0.25, linewidth = 0.4, colour = 'grey20') +
        geom_quasirandom(width = 0.15, size = 1.3, alpha = 0.85, colour = 'grey20') +
        geom_text(data = ts, aes(x = 1.5, y = ymax * 1.18, label = star),
                  inherit.aes = FALSE, size = 3.6) +
        facet_wrap(~ panel, scales = 'free_y', ncol = min(NCOL, n_max)) +
        scale_fill_manual(values = c(WT = 'grey75', KO = '#e05252'), name = NULL) +
        scale_y_continuous(expand = expansion(mult = c(0, 0.25))) +
        labs(title = sprintf('%s   (%d genes)', s, n_gene), x = NULL, y = y_lab) +
        theme_bw() +
        # ★ strip の灰色が濃いと橙・紫の文字が沈むので、ほぼ白まで薄くする
        theme(strip.background = element_rect(fill='grey97', colour='grey75', linewidth=0.3),
              strip.text = element_markdown(size = 9),
              plot.title = element_text(size = 13, face = 'bold'),
              axis.text = element_text(size = 7.5),
              axis.title.y = element_text(size = 9),
              panel.grid.major.x = element_blank())
    hts = c(hts, n_row + 0.45)   # 株の見出し分を足す
}

P = wrap_plots(subs, ncol = 1, heights = hts) +
    plot_layout(guides = 'collect') +
    # ★ 図中に日本語を入れない（PNG 保存時に mbcsToSbcs 変換が失敗して文字化けする）
    plot_annotation(title = sprintf('COG %s   [scheme %s]', COG_NAME, SCHEME),
      subtitle = paste0(
        "PFAM domains in this group: **", paste(pf_names, collapse = ', '), "**<br>",
        "panel name:  <span style='color:#d95f02'>**ORA hit**</span>  /  ",
        "<span style='color:#7570b3'>**DEG only**</span>  /  ",
        "<span style='color:#000000'>**not DEG**</span><br>",
        "y axis = **normalized count** - DESeq2 median-of-ratios with ", sub_norm, "<br>",
        "bar = mean, error bar = ±SD, points = each mouse  |  ",
        "significance = **Welch two-sample t-test** on normalized counts ",
        "(*** p<0.001, ** p<0.01, * p<0.05; not DESeq2 padj, no multiple-testing correction)"),
      theme = theme(plot.title = element_text(size = 15, face = 'bold'),
                    plot.subtitle = element_markdown(size = 10, lineheight = 1.3)))

# ★ 幅も遺伝子数に合わせる（パネル1枚 24/NCOL = 2 inch を保つ）。
#   ただし下限は 16 inch。これより狭いと上の subtitle の行が右で切れる
W_FIG = max(16, 24 * min(NCOL, n_max) / NCOL)
suppressMessages(ggsave(file.path(dir_gb, sprintf('genebar_PFAM_%s_%s.png', COG_LETTER, SCHEME)), P,
                        width = W_FIG, height = 2.1 * sum(hts) + 1.4, dpi = 100, limitsize = FALSE))
options(repr.plot.width = W_FIG, repr.plot.height = 2.1 * sum(hts) + 1.4)
P


COG I: Lipid transport and metabolism  scheme B
  PFAM: CPSase_L_D2, Biotin_carb_C, Biotin_carb_N
  全 10 遺伝子 / ORA hit 7


plot without title

In [163]:
# ── 図12-2-4: COG I カテゴリ全体（186 遺伝子）  scheme A ──
#   ★ 図12-2-3 とは対象が違う。図12-2-3 は PFAM 3ドメイン（CPSase_L_D2 /
#     Biotin_carb_C / Biotin_carb_N）を持つ 10 遺伝子だけ。こちらは
#     COG I カテゴリに属する全遺伝子
# セル12（準備）を先に実行しておくこと
#   （nc_A / nc_B / col_cat / strains4 / NCOL / dir_gb を使う）
#
# ★ なぜ COG I だけ遺伝子ごとに描くか
#   ORA では COG I は「KO で上昇」に有意（(A) fold 1.45 padj 2.0e-02 /
#   (B) fold 1.65 padj 2.5e-03）。ところが PFAM 側（図12-2-3）で見た
#   accC / アセチル CoA カルボキシラーゼ群は Bth 2株で強く低下していた。
#   ★ 矛盾ではなく、同じカテゴリの中で株ごとに向きが逆になっている。
#     ORA は「down DEG リストに過剰か」を見るだけで、カテゴリの平均の向きは見ない。
#   遺伝子ごとに出さないとこれが読めないので、I だけは棒グラフを作る。
#   （M・C・J などは 400〜940 遺伝子あり、同じ作りだと図の高さが 80〜170 inch に
#     なるため作らない）
library(ggplot2); library(ggbeeswarm); library(ggtext); library(patchwork)

SCHEME = 'A'; COG_LETTER = 'I'
COG_TERM = 'COG I: Lipid transport and metabolism'

nc = if (SCHEME == 'A') nc_A else nc_B
y_lab = ifelse(SCHEME == 'A',
               'normalized count  (A: pooled across all 4 strains)',
               'normalized count  (B: per-strain size factors)')
sub_norm = ifelse(SCHEME == 'A',
    paste0("**one size factor set computed from all 4 strains pooled** ",
           "(= scheme A, the community-level output; includes the shift in strain abundance)"),
    paste0("**size factors computed separately within each strain** ",
           "(= scheme B, within-strain regulation; excludes the shift in strain abundance)"))

# ORA で「KO で上昇」に拾われた遺伝子
g_ora = ora %>% filter(scheme == SCHEME, source == 'COG', term == COG_TERM, padj < 0.05) %>%
    pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()

# ★ カテゴリに属する全遺伝子を描く。ORA が拾わなかった遺伝子（下がっている方を含む）を
#   落とすと、ここで見たいことがちょうど消える
g_all = T2G$COG %>% filter(term == COG_TERM) %>% pull(gene)
padj_col = if (SCHEME == 'A') res$padj_A else res$padj_pooled
g_all = intersect(g_all, res$gene_id[!res$marker & !is.na(padj_col)])

cat(sprintf("%s  scheme %s\n  全 %d 遺伝子 / ORA hit（KO で上昇）%d\n",
            COG_TERM, SCHEME, length(g_all), length(g_ora)))

d = nc[intersect(g_all, rownames(nc)), , drop = FALSE] %>%
    tibble::rownames_to_column('gene_id') %>%
    tidyr::pivot_longer(-gene_id, names_to = 'sample', values_to = 'norm') %>%
    mutate(group = factor(ifelse(grepl('W', sample), 'WT', 'KO'), levels = c('WT','KO'))) %>%
    left_join(res %>% dplyr::select(gene_id, strain, Preferred_name, product, PFAMs,
                                    padj_A, padj_pooled), by = 'gene_id') %>%
    mutate(padj_use = if (SCHEME == 'A') padj_A else padj_pooled,
           cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                           !is.na(padj_use) & padj_use < 0.05 ~ 'DEG only',
                           TRUE ~ 'not DEG'),
           cat = factor(cat, levels = names(col_cat)),
           label = ifelse(is.na(Preferred_name) | Preferred_name == '-',
                          substr(product, 1, 18), Preferred_name),
           # ★ パネル名は element_markdown（ggtext → gridtext）で描く。
           #   gridtext は commonmark の smart punctuation を有効にしているので、
           #   ★ ハイフン2つ '--' を en ダッシュ（–, U+2013）に勝手に変換する。
           #   'succinyl-CoA--3-ketoacid-CoA transferase' がこれに当たり、
           #   PNG 保存時に mbcsToSbcs の変換警告が出る（データ側は ASCII のまま）。
           #   連続するハイフンを1つにまとめて回避する
           label = gsub('-{2,}', '-', gsub('[^ -~]', '-', label)),
           # ★ 3行目は PFAM ドメイン（COG は1文字なのでパネルに出す意味がない）。
           #   先頭1つだけにする。並べるとパネル幅から左右がはみ出て切れる
           pf = sapply(strsplit(ifelse(is.na(PFAMs), '-', PFAMs), ','),
                       function(v) substr(v[1], 1, 20)),
           panel = sprintf("<span style='color:%s'>**%s<br>%s<br>%s**</span>",
                           col_cat[as.character(cat)], gene_id, label, pf))

# 株ごとの向きの内訳を先に表で出す。★ 図の読み方がこの表で決まる
cat("\n■ Table 1: 株ごとの向きの内訳（DESeq2 の padj で判定）\n")
d %>% distinct(gene_id, strain, padj_use, cat) %>%
    left_join(res %>% dplyr::select(gene_id, LFC_A, LFC_B), by = 'gene_id') %>%
    mutate(LFC = if (SCHEME == 'A') LFC_A else LFC_B,
           判定 = case_when(!is.na(padj_use) & padj_use < 0.05 & LFC > 0 ~ 'up',
                            !is.na(padj_use) & padj_use < 0.05 & LFC < 0 ~ 'down',
                            TRUE ~ 'n.s.')) %>%
    dplyr::count(strain, 判定) %>%
    tidyr::pivot_wider(names_from = 判定, values_from = n, values_fill = 0) %>%
    mutate(計 = up + down + n.s.) %>% as.data.frame()

cat("\n■ Table 2: 株ごとの正味リード増減（向き別）\n")
d %>% distinct(gene_id, strain, padj_use) %>%
    left_join(res %>% dplyr::select(gene_id, LFC_A, LFC_B, normA_diff, normB_diff), by = 'gene_id') %>%
    mutate(LFC  = if (SCHEME == 'A') LFC_A else LFC_B,
           diff = if (SCHEME == 'A') normA_diff else normB_diff,
           判定 = case_when(!is.na(padj_use) & padj_use < 0.05 & LFC > 0 ~ 'up',
                            !is.na(padj_use) & padj_use < 0.05 & LFC < 0 ~ 'down',
                            TRUE ~ 'n.s.')) %>%
    group_by(strain, 判定) %>% summarise(n = n(), net = round(sum(diff)), .groups = 'drop') %>%
    tidyr::pivot_wider(names_from = 判定, values_from = c(n, net), values_fill = 0) %>%
    mutate(正味合計 = net_up + net_down + net_n.s.) %>% as.data.frame()

tt = d %>% group_by(gene_id, strain, panel) %>%
    summarise(p = tryCatch(t.test(norm ~ group)$p.value, error = function(e) NA_real_),
              ymax = max(max(norm), max(tapply(norm, group, function(v) mean(v) + sd(v)))),
              .groups = 'drop') %>%
    mutate(star = cut(p, c(-Inf, .001, .01, .05, Inf), labels = c('***','**','*','n.s.')),
           star = ifelse(is.na(star), 'n.s.', as.character(star)))

n_max = max(sapply(strains4, function(s) length(unique(d$gene_id[d$strain == s]))))

subs = list(); hts = c()
for (s in strains4) {
    ds = d %>% filter(strain == s) %>% arrange(cat, label)
    if (nrow(ds) == 0) next
    ds$panel = factor(ds$panel, levels = unique(ds$panel))
    ts = tt %>% filter(strain == s) %>% mutate(panel = factor(panel, levels = levels(ds$panel)))
    n_gene = length(unique(ds$gene_id)); n_row = ceiling(n_gene / min(NCOL, n_max))
    subs[[s]] = ggplot(ds, aes(x = group, y = norm, fill = group)) +
        geom_bar(stat = 'summary', fun = 'mean', width = 0.65, colour = 'grey30', linewidth = 0.3) +
        stat_summary(fun = mean,
                     fun.min = function(v) mean(v) - sd(v),
                     fun.max = function(v) mean(v) + sd(v),
                     geom = 'errorbar', width = 0.25, linewidth = 0.4, colour = 'grey20') +
        geom_quasirandom(width = 0.15, size = 1.3, alpha = 0.85, colour = 'grey20') +
        geom_text(data = ts, aes(x = 1.5, y = ymax * 1.18, label = star),
                  inherit.aes = FALSE, size = 3.6) +
        facet_wrap(~ panel, scales = 'free_y', ncol = min(NCOL, n_max)) +
        scale_fill_manual(values = c(WT = 'grey75', KO = '#e05252'), name = NULL) +
        scale_y_continuous(expand = expansion(mult = c(0, 0.25))) +
        labs(title = sprintf('%s   (%d genes)', s, n_gene), x = NULL, y = y_lab) +
        theme_bw() +
        theme(strip.background = element_rect(fill='grey97', colour='grey75', linewidth=0.3),
              strip.text = element_markdown(size = 9),
              plot.title = element_text(size = 13, face = 'bold'),
              axis.text = element_text(size = 7.5),
              axis.title.y = element_text(size = 9),
              panel.grid.major.x = element_blank())
    hts = c(hts, n_row + 0.45)
}

P = wrap_plots(subs, ncol = 1, heights = hts) +
    plot_layout(guides = 'collect') +
    plot_annotation(
      title = sprintf('(%s) %s - COG I: Lipid transport and metabolism',
                      SCHEME, ifelse(SCHEME == 'A', 'pooled', 'per-strain')),
      subtitle = paste0(
        "ORA found this category **enriched among genes UP in KO**, but the direction is ",
        "**opposite between strains** - see the panel colours and Table 1<br>",
        "panel name:  <span style='color:#d95f02'>**ORA hit (up in KO)**</span>  /  ",
        "<span style='color:#7570b3'>**DEG only**</span>  /  ",
        "<span style='color:#000000'>**not DEG**</span><br>",
        "y axis = **normalized count** - DESeq2 median-of-ratios with ", sub_norm, "<br>",
        "bar = mean, error bar = ±SD, points = each mouse  |  ",
        "significance = **Welch two-sample t-test** on normalized counts ",
        "(*** p<0.001, ** p<0.01, * p<0.05; not DESeq2 padj, no multiple-testing correction)"),
      theme = theme(plot.title = element_text(size = 15, face = 'bold'),
                    plot.subtitle = element_markdown(size = 10, lineheight = 1.3)))

W_FIG = max(16, 24 * min(NCOL, n_max) / NCOL)
suppressMessages(ggsave(file.path(dir_gb, sprintf('genebar_COG_%s_%s.png', COG_LETTER, SCHEME)), P,
                        width = W_FIG, height = 2.1 * sum(hts) + 1.4, dpi = 100, limitsize = FALSE))
options(repr.plot.width = W_FIG, repr.plot.height = 2.1 * sum(hts) + 1.4)
P


COG I: Lipid transport and metabolism  scheme A
  全 186 遺伝子 / ORA hit（KO で上昇）47

■ Table 1: 株ごとの向きの内訳（DESeq2 の padj で判定）


strain,down,n.s.,up,計
<chr>,<int>,<int>,<int>,<int>
Bth_St8,8,22,8,38
Bth_VPI,8,16,1,25
Bun_8492,4,25,36,65
Odo_St23,7,49,2,58



■ Table 2: 株ごとの正味リード増減（向き別）


strain,n_down,n_n.s.,n_up,net_down,net_n.s.,net_up,正味合計
<chr>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
Bth_St8,8,22,8,-58138,-1232,920,-58450
Bth_VPI,8,16,1,-29309,-428,20,-29717
Bun_8492,4,25,36,-15574,-42487,70135,12074
Odo_St23,7,49,2,-837,-1420,50,-2207


plot without title

In [164]:
# ── 図12-2-5: COG I カテゴリ全体（187 遺伝子）  scheme B ──
#   ★ 図12-2-3 とは対象が違う（図12-2-4 のコメント参照）
# セル12（準備）を先に実行しておくこと
#   （nc_A / nc_B / col_cat / strains4 / NCOL / dir_gb を使う）
#
# ★ なぜ COG I だけ遺伝子ごとに描くか
#   ORA では COG I は「KO で上昇」に有意（(A) fold 1.45 padj 2.0e-02 /
#   (B) fold 1.65 padj 2.5e-03）。ところが PFAM 側（図12-2-3）で見た
#   accC / アセチル CoA カルボキシラーゼ群は Bth 2株で強く低下していた。
#   ★ 矛盾ではなく、同じカテゴリの中で株ごとに向きが逆になっている。
#     ORA は「down DEG リストに過剰か」を見るだけで、カテゴリの平均の向きは見ない。
#   遺伝子ごとに出さないとこれが読めないので、I だけは棒グラフを作る。
#   （M・C・J などは 400〜940 遺伝子あり、同じ作りだと図の高さが 80〜170 inch に
#     なるため作らない）
library(ggplot2); library(ggbeeswarm); library(ggtext); library(patchwork)

SCHEME = 'B'; COG_LETTER = 'I'
COG_TERM = 'COG I: Lipid transport and metabolism'

nc = if (SCHEME == 'A') nc_A else nc_B
y_lab = ifelse(SCHEME == 'A',
               'normalized count  (A: pooled across all 4 strains)',
               'normalized count  (B: per-strain size factors)')
sub_norm = ifelse(SCHEME == 'A',
    paste0("**one size factor set computed from all 4 strains pooled** ",
           "(= scheme A, the community-level output; includes the shift in strain abundance)"),
    paste0("**size factors computed separately within each strain** ",
           "(= scheme B, within-strain regulation; excludes the shift in strain abundance)"))

# ORA で「KO で上昇」に拾われた遺伝子
g_ora = ora %>% filter(scheme == SCHEME, source == 'COG', term == COG_TERM, padj < 0.05) %>%
    pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()

# ★ カテゴリに属する全遺伝子を描く。ORA が拾わなかった遺伝子（下がっている方を含む）を
#   落とすと、ここで見たいことがちょうど消える
g_all = T2G$COG %>% filter(term == COG_TERM) %>% pull(gene)
padj_col = if (SCHEME == 'A') res$padj_A else res$padj_pooled
g_all = intersect(g_all, res$gene_id[!res$marker & !is.na(padj_col)])

cat(sprintf("%s  scheme %s\n  全 %d 遺伝子 / ORA hit（KO で上昇）%d\n",
            COG_TERM, SCHEME, length(g_all), length(g_ora)))

d = nc[intersect(g_all, rownames(nc)), , drop = FALSE] %>%
    tibble::rownames_to_column('gene_id') %>%
    tidyr::pivot_longer(-gene_id, names_to = 'sample', values_to = 'norm') %>%
    mutate(group = factor(ifelse(grepl('W', sample), 'WT', 'KO'), levels = c('WT','KO'))) %>%
    left_join(res %>% dplyr::select(gene_id, strain, Preferred_name, product, PFAMs,
                                    padj_A, padj_pooled), by = 'gene_id') %>%
    mutate(padj_use = if (SCHEME == 'A') padj_A else padj_pooled,
           cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                           !is.na(padj_use) & padj_use < 0.05 ~ 'DEG only',
                           TRUE ~ 'not DEG'),
           cat = factor(cat, levels = names(col_cat)),
           label = ifelse(is.na(Preferred_name) | Preferred_name == '-',
                          substr(product, 1, 18), Preferred_name),
           # ★ パネル名は element_markdown（ggtext → gridtext）で描く。
           #   gridtext は commonmark の smart punctuation を有効にしているので、
           #   ★ ハイフン2つ '--' を en ダッシュ（–, U+2013）に勝手に変換する。
           #   'succinyl-CoA--3-ketoacid-CoA transferase' がこれに当たり、
           #   PNG 保存時に mbcsToSbcs の変換警告が出る（データ側は ASCII のまま）。
           #   連続するハイフンを1つにまとめて回避する
           label = gsub('-{2,}', '-', gsub('[^ -~]', '-', label)),
           # ★ 3行目は PFAM ドメイン（COG は1文字なのでパネルに出す意味がない）。
           #   先頭1つだけにする。並べるとパネル幅から左右がはみ出て切れる
           pf = sapply(strsplit(ifelse(is.na(PFAMs), '-', PFAMs), ','),
                       function(v) substr(v[1], 1, 20)),
           panel = sprintf("<span style='color:%s'>**%s<br>%s<br>%s**</span>",
                           col_cat[as.character(cat)], gene_id, label, pf))

# 株ごとの向きの内訳を先に表で出す。★ 図の読み方がこの表で決まる
cat("\n■ Table 1: 株ごとの向きの内訳（DESeq2 の padj で判定）\n")
d %>% distinct(gene_id, strain, padj_use, cat) %>%
    left_join(res %>% dplyr::select(gene_id, LFC_A, LFC_B), by = 'gene_id') %>%
    mutate(LFC = if (SCHEME == 'A') LFC_A else LFC_B,
           判定 = case_when(!is.na(padj_use) & padj_use < 0.05 & LFC > 0 ~ 'up',
                            !is.na(padj_use) & padj_use < 0.05 & LFC < 0 ~ 'down',
                            TRUE ~ 'n.s.')) %>%
    dplyr::count(strain, 判定) %>%
    tidyr::pivot_wider(names_from = 判定, values_from = n, values_fill = 0) %>%
    mutate(計 = up + down + n.s.) %>% as.data.frame()

cat("\n■ Table 2: 株ごとの正味リード増減（向き別）\n")
d %>% distinct(gene_id, strain, padj_use) %>%
    left_join(res %>% dplyr::select(gene_id, LFC_A, LFC_B, normA_diff, normB_diff), by = 'gene_id') %>%
    mutate(LFC  = if (SCHEME == 'A') LFC_A else LFC_B,
           diff = if (SCHEME == 'A') normA_diff else normB_diff,
           判定 = case_when(!is.na(padj_use) & padj_use < 0.05 & LFC > 0 ~ 'up',
                            !is.na(padj_use) & padj_use < 0.05 & LFC < 0 ~ 'down',
                            TRUE ~ 'n.s.')) %>%
    group_by(strain, 判定) %>% summarise(n = n(), net = round(sum(diff)), .groups = 'drop') %>%
    tidyr::pivot_wider(names_from = 判定, values_from = c(n, net), values_fill = 0) %>%
    mutate(正味合計 = net_up + net_down + net_n.s.) %>% as.data.frame()

tt = d %>% group_by(gene_id, strain, panel) %>%
    summarise(p = tryCatch(t.test(norm ~ group)$p.value, error = function(e) NA_real_),
              ymax = max(max(norm), max(tapply(norm, group, function(v) mean(v) + sd(v)))),
              .groups = 'drop') %>%
    mutate(star = cut(p, c(-Inf, .001, .01, .05, Inf), labels = c('***','**','*','n.s.')),
           star = ifelse(is.na(star), 'n.s.', as.character(star)))

n_max = max(sapply(strains4, function(s) length(unique(d$gene_id[d$strain == s]))))

subs = list(); hts = c()
for (s in strains4) {
    ds = d %>% filter(strain == s) %>% arrange(cat, label)
    if (nrow(ds) == 0) next
    ds$panel = factor(ds$panel, levels = unique(ds$panel))
    ts = tt %>% filter(strain == s) %>% mutate(panel = factor(panel, levels = levels(ds$panel)))
    n_gene = length(unique(ds$gene_id)); n_row = ceiling(n_gene / min(NCOL, n_max))
    subs[[s]] = ggplot(ds, aes(x = group, y = norm, fill = group)) +
        geom_bar(stat = 'summary', fun = 'mean', width = 0.65, colour = 'grey30', linewidth = 0.3) +
        stat_summary(fun = mean,
                     fun.min = function(v) mean(v) - sd(v),
                     fun.max = function(v) mean(v) + sd(v),
                     geom = 'errorbar', width = 0.25, linewidth = 0.4, colour = 'grey20') +
        geom_quasirandom(width = 0.15, size = 1.3, alpha = 0.85, colour = 'grey20') +
        geom_text(data = ts, aes(x = 1.5, y = ymax * 1.18, label = star),
                  inherit.aes = FALSE, size = 3.6) +
        facet_wrap(~ panel, scales = 'free_y', ncol = min(NCOL, n_max)) +
        scale_fill_manual(values = c(WT = 'grey75', KO = '#e05252'), name = NULL) +
        scale_y_continuous(expand = expansion(mult = c(0, 0.25))) +
        labs(title = sprintf('%s   (%d genes)', s, n_gene), x = NULL, y = y_lab) +
        theme_bw() +
        theme(strip.background = element_rect(fill='grey97', colour='grey75', linewidth=0.3),
              strip.text = element_markdown(size = 9),
              plot.title = element_text(size = 13, face = 'bold'),
              axis.text = element_text(size = 7.5),
              axis.title.y = element_text(size = 9),
              panel.grid.major.x = element_blank())
    hts = c(hts, n_row + 0.45)
}

P = wrap_plots(subs, ncol = 1, heights = hts) +
    plot_layout(guides = 'collect') +
    plot_annotation(
      title = sprintf('(%s) %s - COG I: Lipid transport and metabolism',
                      SCHEME, ifelse(SCHEME == 'A', 'pooled', 'per-strain')),
      subtitle = paste0(
        "ORA found this category **enriched among genes UP in KO**, but the direction is ",
        "**opposite between strains** - see the panel colours and Table 1<br>",
        "panel name:  <span style='color:#d95f02'>**ORA hit (up in KO)**</span>  /  ",
        "<span style='color:#7570b3'>**DEG only**</span>  /  ",
        "<span style='color:#000000'>**not DEG**</span><br>",
        "y axis = **normalized count** - DESeq2 median-of-ratios with ", sub_norm, "<br>",
        "bar = mean, error bar = ±SD, points = each mouse  |  ",
        "significance = **Welch two-sample t-test** on normalized counts ",
        "(*** p<0.001, ** p<0.01, * p<0.05; not DESeq2 padj, no multiple-testing correction)"),
      theme = theme(plot.title = element_text(size = 15, face = 'bold'),
                    plot.subtitle = element_markdown(size = 10, lineheight = 1.3)))

W_FIG = max(16, 24 * min(NCOL, n_max) / NCOL)
suppressMessages(ggsave(file.path(dir_gb, sprintf('genebar_COG_%s_%s.png', COG_LETTER, SCHEME)), P,
                        width = W_FIG, height = 2.1 * sum(hts) + 1.4, dpi = 100, limitsize = FALSE))
options(repr.plot.width = W_FIG, repr.plot.height = 2.1 * sum(hts) + 1.4)
P


COG I: Lipid transport and metabolism  scheme B
  全 187 遺伝子 / ORA hit（KO で上昇）47

■ Table 1: 株ごとの向きの内訳（DESeq2 の padj で判定）


strain,down,n.s.,up,計
<chr>,<int>,<int>,<int>,<int>
Bth_St8,8,17,13,38
Bth_VPI,7,14,5,26
Bun_8492,10,35,20,65
Odo_St23,5,44,9,58



■ Table 2: 株ごとの正味リード増減（向き別）


strain,n_down,n_n.s.,n_up,net_down,net_n.s.,net_up,正味合計
<chr>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
Bth_St8,8,17,13,-54666,-617,1796,-53487
Bth_VPI,7,14,5,-27001,58,641,-26302
Bun_8492,10,35,20,-128022,13620,27840,-86562
Odo_St23,5,44,9,-430,117,1642,1329


plot without title

### 考察: 同じ COG カテゴリの中で株ごとに向きが逆

#### 1. `M: Cell wall`（図12-2-1 / 12-2-2）— 低下が主だが上昇も混ざる

| 株 | 遺伝子 | 有意に低下 | 有意に上昇 |
|---|---|---|---|
| Bth_St8 | 24 | **18** | 2（`CE91St8_15930`・`CE91St8_31990`） |
| Bth_VPI | 17 | 10 | 2（`BT_RS03875` patatin ***・`BT_RS04505`） |
| Bun_8492 | 26 | 11 | 1（`NQ510_RS14520` patatin） |
| Odo_St23 | 20 | 7 | 1（`CE91St23_10690`） |

量では **`NQ510_RS01710`（OmpA family、WT 約12万）が桁違いに大きく**、
セル7-3 の「転写量で見ると担い手は `Bun_8492`」と整合する。

★ **`Patatin` は向きが株でバラバラ**（`Bth_VPI` は上昇、`Bth_St8` は低下）、
`MS_channel` は `Bun_8492` の `mscM`/`mscS` が低下、`Odo_St23` の `mscS` は n.s.。
**ドメイン単位でまとめると向きが混ざる例。**

#### 2. `I: Lipid transport`（図12-2-3）— 実体は脂肪酸合成の起点

`accC`（アセチル CoA カルボキシラーゼ）が **`Bth_VPI` ** / `Bth_St8` ** / `Bun_8492` * で
3株そろって低下**。一方 `CPSase_L_D2` だけを持つ `carB`（カルバモイルリン酸合成酵素、
ピリミジン合成）は **`Bun_8492`・`Odo_St23` とも n.s.**
→ **COG のラベル「脂質輸送・代謝」どおりではなく、動いているのは `accC` 系だけ。**
同じ `CPSase_L_D2` ドメインを共有しているせいで1つの term にまとまっているだけ。

#### 3. ★★ COG I カテゴリ全体（図12-2-4 / 12-2-5）— 3つの見方で結論が割れる

| 見方 | 結論 |
|---|---|
| **ORA**（遺伝子数、up/down 別々） | **(A)(B) とも有意に up**（fold 1.45 / 1.65） |
| **GSEA**（Wald 統計量の順位、1本の軸） | **(A)(B) とも有意でない**（padj 0.061 / 0.087） |
| **正味リード**（転写量） | **(A)(B) とも減少**（−78,299 / −165,023） |

**ORA が up になるのは遺伝子を「数える」から。**1遺伝子あたりの正味は
**(A) で up +1,513 / down −3,847（2.5倍）、(B) で +679 / −7,004（10倍）**と down が大きい。
さらに **ORA は up と down を別々に背景と比べる**ので、down 側が背景とほぼ同じ
（fold 0.92 / 0.99）なら「down には富んでいない」と「up には富んでいる」が両立する。

★ **n.s. の遺伝子が検定に入らないことも効いている。**`NQ510_RS11040`（OmpP1/FadL、
長鎖脂肪酸の取り込み口）は (A) で padj 0.11 の n.s. だが**単独で −58,867**、
(B) では **−100,258（LFC_B −0.90、padj 1.5e-03）**でカテゴリ最大の減少を作る。

→ **「COG I が KO で上がった」と単独で書くのは誤り。**正しくは
**「遺伝子の本数としては上昇側に偏るが、転写量としてはカテゴリ全体で減少している」。**

#### 4. 株の内訳 — 開始段階は4株、伸長段階は `Bun_8492` だけ

**(A) では `Bun_8492` だけ正味プラス（+12,074）だが、(B) では −86,562 と符号が逆転する**
（菌量増加 WT 54.8% → KO 65.1% を取り除くため）。
**(B) では `Bun_8492` の `accC` も有意に低下（−0.62、padj 1.4e-02）** するので、
**4株すべてで脂肪酸合成の開始段階が下がっている。**
一方 `fabF`（+1.43、8,100 → 23,054）・`fabH`・`fabZ` の**伸長段階は `Bun_8492` だけが上昇**し、
これは (A)(B) とも変わらない。


## セル13: 経路ごとのゲノム位置マップ ＋ 遺伝子表

過去notebook（`6_DEGs_genome_location_R.ipynb`）と `4_reads.ipynb` セル167 に対応。
**CLAUDE.md §7 ステージ6 の「有意経路はゲノム座標順に LFC をプロットし、1オペロンが
効いているだけでないか確認」**にあたる。

### 1経路 = 1セル。中で3つを続けて出す

**(1)** pathview（経路の全遺伝子、セル11 の再掲）→ **(2)** pathview（ORA の `geneID` のみ、再掲）
→ **(3)** ゲノム位置マップ ＋ 遺伝子表（このセルで作る）。

**pathview を先に置く理由**: **化合物名と反応の向きは pathview にしか無い**
（eggNOG に基質・産物の情報が無く、KEGG REST から引くには KGML の座標まで要る）。
**「どの反応か」を押さえてから「ゲノムのどこにあるか」を見る**順にする。

### (3) の作り

**A: ゲノム位置マップ** — x 軸 = 各 contig 内の位置（Mb）／y 軸 = LFC／灰色の小点 = その contig の
全 CDS（背景）／**点の縁 = 分類**（橙 ORA hit / 紫 DEG only / 黒 not DEG）／
**点の塗り = `log10(正規化カウント + 1)` の WT 群平均**（mako: 淡青 → 濃紺）／
ラベル = `Preferred_name`（無ければ `product` の先頭10文字）。
縁と塗りを分けるため **`shape = 21`**、発現量は3〜4桁にわたるので `log10`（0 があるので `+1`）。

**この図（地図＋表）だけ縦横 1.125 倍**（24.75 × 10.8 inch ※高さは遺伝子数で変わる）。
pathview の再掲は等倍。縮めた分**ラベルの文字は `3.4` → `4.4` pt**にした。

**B: 遺伝子表** — `strain`/`contig`/`Mb`/`gene_id`/`name`/`K`/`EC`/`product`/`class` の 9 列。
**K番号と EC は省略しない。行の文字色が A の分類と一致する。**
縦に長くなるので**行を横に分割**（1ブロック最大 24 行）。

**LFC と padj は列に入れない**（LFC は**上の地図の y 軸そのもの**、padj は**分類の色**で示している）。

★ **フォントは経路ごとに自動で決める。**`ko03010`（112 遺伝子）のように行が多いとブロック数が増え、
1ブロックに割ける幅が狭くなるため、固定サイズだと**右端の `class` 列が切れる**（実際に切れていた）。
→ **実際に表を組んで幅を測り、収まる一番大きいフォントを選ぶ**（7.0 pt から 0.1 pt ずつ下げる）。
実測で `ko00120` 7.0 pt / `ko00720` 5.4 pt / `ko03010` 5.6 pt。

### ★ 全 contig を常に描く

**経路の遺伝子が無い contig も空パネルとして残す。**「副 contig に無い」ことを示すのが目的なので、
**経路によって描く contig を変えない**（変えると図どうしで見ている範囲が違って比較できない）。
副 contig は主の 1% 未満（0.033〜0.065 Mb 対 4.6〜6.2 Mb）なので、
**描画幅は `MINW = 0.6 Mb` 相当で底上げしつつ x 軸は実際の長さに合わせる**
（副 contig は bp あたりの縮尺が主より大きくなる）。

### 実装でつまずいた点

| 現象 | 原因と対処 |
|---|---|
| A のタイトルが出ない | patchwork を入れ子にすると**内側の `plot_annotation` が描画されない**。`wrap_elements(full = P)` で包む |
| 凡例が2つ並ぶ | `drop = FALSE` でも**データが無い水準はキーの点が描かれない**。**3分類が最もそろう contig 1枚からだけ**凡例を出す |
| 凡例が右で地図を狭める | **収集した凡例の位置は patchwork 側の theme で決まる**（サブプロットの `legend.position` は効かない） |
| 表の文字色がずれる | `ggtexttable` は色を**列方向（column-major）に展開**する。`rep(col, ncol)` が正しい |
| 表の上に空白ができる | `ggtexttable` は**パネルの中央**に置かれる。行数から詰めて見積もる |
| `ko00121` の再掲で `readPNG` がエラー | セル11 の検品で落ちた図は**中身が書かれず 0 バイトで残る**。`file.exists()` は `TRUE` になるので **`file.size(f) == 0` も見て飛ばす** |
| 測った表の幅より実際が広い | 文字幅は **device のフォント metrics** で決まる（同じ表・6.2 pt で pdf 4.74 / cairo png 4.93 / **ragg 5.26** inch）。`ggsave` は **ragg** を使うので**測るのも ragg device 上で**行う |
| フォントを計算で逆算すると合わない | 文字幅は**フォントサイズに比例しない**（hinting で段階的に変わる。5.5 pt で 4.75、5.9 pt で 5.04 inch）。**0.1 pt ずつ下げて実測**する |


In [114]:
# ── 経路ごとのゲノム位置マップ（準備）────────────────────────
# 過去notebook（6_DEGs_genome_location_R.ipynb）と
# 4_reads.ipynb セル167（ゲノム上の位置と PC1 への効き方）に対応。
#
# ★ 1セル = 1図 にする（セル12 と同じ理由）。
#   Jupyter は cat() の文字出力をひとまとめにしてから図を出すので、
#   1セルで複数回すと見出しと図の対応が取れない
#
# セル13-1 〜 13-14 で、経路ごとに次の3つを続けて出す
#   (1) pathview（経路の全遺伝子）    セル11 で作ったものを再掲
#   (2) pathview（ORA の geneID のみ）セル11 で作ったものを再掲
#   (3) ゲノム位置マップ ＋ 遺伝子表
library(ggplot2); library(ggtext); library(ggrepel); library(patchwork); library(ggpubr)
library(png); library(grid)

# パネル名・点の縁・表の文字色に使う分類の色（セル12 と同じ）
col_cat  = c('ORA hit' = '#d95f02', 'DEG only' = '#7570b3', 'not DEG' = '#000000')
strains4 = c('Bth_VPI', 'Bth_St8', 'Bun_8492', 'Odo_St23')

# ★ 副 contig に割り当てる最小の「描画幅」（Mb 換算）。
#   副 contig は主の 1% 未満なので、比例配分だけだと見えない細片になる。
#   幅は底上げしつつ x 軸は実際の長さに合わせるので、bp あたりの縮尺が主より大きくなる
MINW = 0.6

dir_pv = file.path(path_pw, 'pathview')   # セル11 の出力（再掲に使う）
dir_gp = file.path(path_pw, 'genomepos')
if (!dir.exists(dir_gp)) dir.create(dir_gp, recursive = TRUE)

cat(sprintf("pathview の出力先: %s（%d ファイル）\n", dir_pv,
            length(list.files(dir_pv, pattern = '\\.png$'))))

# ★ 全 contig を常に描く。経路の遺伝子が無い contig も空パネルとして残す。
#   「副 contig に無い」ことを示すのが目的なので、経路によって描く contig を変えない
cat("\n■ Table 1: 描画対象の contig（経路によらず常にこの10本を描く）\n")
res %>% filter(biotype == 'CDS', strain %in% strains4, !is.na(end)) %>%
    group_by(strain, contig) %>%
    summarise(CDS = dplyr::n(), 長さMb = round(max(end)/1e6, 3), .groups = 'drop') %>%
    arrange(factor(strain, levels = strains4), desc(CDS)) %>%
    mutate(種別 = ifelse(CDS == ave(CDS, strain, FUN = max), '主', '副'),
           描画幅Mb = round(pmax(長さMb, MINW), 2)) %>%
    as.data.frame()

cat("\n■ Table 2: これから描く図（7経路 × 2スキーム = 14 枚）\n")
data.frame(
    セル = paste0('13-', 1:14),
    ko   = rep(c('ko00030','ko00020','ko00710','ko00720','ko00120','ko00121','ko03010'), 2),
    経路 = rep(c('ペントースリン酸経路','TCA回路','炭素固定(Calvin)','その他の炭素固定',
                 '一次胆汁酸','二次胆汁酸','リボソーム'), 2),
    scheme = rep(c('A','B'), each = 7),
    check.names = FALSE) %>%
    left_join(ora %>% filter(source == 'KEGG', padj < 0.05) %>%
                  transmute(ko = sub('_.*', '', term), scheme,
                            ORAで有意 = sprintf('%s (padj %.1e)', direction, padj)),
              by = c('ko', 'scheme')) %>%
    mutate(ORAで有意 = ifelse(is.na(ORAで有意), '★ 有意でない（pathview も無い）', ORAで有意))


pathview の出力先: /mnt/scratch/tanoue/261016_Xiaoxi_eptAoligo_2wFeces_Metatranscriptome/pathway/pathview（139 ファイル）

■ Table 1: 描画対象の contig（経路によらず常にこの10本を描く）


strain,contig,CDS,長さMb,種別,描画幅Mb
<chr>,<chr>,<int>,<dbl>,<chr>,<dbl>
Bth_VPI,Bth_VPI~NC_004663.1,1676,6.243,主,6.24
Bth_VPI,Bth_VPI~NC_004703.1,36,0.033,副,0.60
Bth_St8,Bth_St8~BQNN01000001.1,1780,6.132,主,6.13
Bth_St8,Bth_St8~BQNN01000002.1,41,0.045,副,0.60
Bth_St8,Bth_St8~BQNN01000005.1,7,0.004,副,0.60
Bth_St8,Bth_St8~BQNN01000004.1,2,0.004,副,0.60
Bun_8492,Bun_8492~NZ_CP102263.1,3392,4.629,主,4.63
Odo_St23,Odo_St23~BQOE01000001.1,2801,5.647,主,5.65
Odo_St23,Odo_St23~BQOE01000002.1,38,0.065,副,0.60



■ Table 2: これから描く図（7経路 × 2スキーム = 14 枚）


セル,ko,経路,scheme,ORAで有意
<chr>,<chr>,<chr>,<chr>,<chr>
13-1,ko00030,ペントースリン酸経路,A,down (padj 4.1e-06)
13-2,ko00020,TCA回路,A,down (padj 9.6e-04)
13-3,ko00710,炭素固定(Calvin),A,down (padj 2.0e-02)
13-4,ko00720,その他の炭素固定,A,down (padj 2.0e-02)
13-5,ko00120,一次胆汁酸,A,down (padj 3.0e-02)
13-6,ko00121,二次胆汁酸,A,down (padj 2.4e-03)
13-7,ko03010,リボソーム,A,up (padj 8.9e-03)
13-8,ko00030,ペントースリン酸経路,B,down (padj 2.2e-07)
13-9,ko00020,TCA回路,B,down (padj 6.2e-04)


In [115]:
# ── 図13-1: ko00030（ペントースリン酸経路）  scheme A ───────────────────
# セル13（準備）を先に実行しておくこと（col_cat / strains4 / MINW / dir_pv / dir_gp を使う）
#
# ★ 3つを続けて出す
#   (1) pathview（経路の全遺伝子）… 反応と化合物名。経路の中での位置づけ
#   (2) pathview（ORA の geneID のみ）… ORA が拾った反応だけ
#   (3) ゲノム位置マップ ＋ 遺伝子表 … どこにあるか、何の遺伝子か
library(ggplot2); library(ggtext); library(ggrepel); library(patchwork); library(ggpubr)
library(png); library(grid)

ko = 'ko00030'; ko_jp = 'ペントースリン酸経路'; SCHEME = 'A'

# ── (1)(2) セル11 で作った pathview を再掲する ────────────────
# ★ 化合物名や反応の向きは pathview にしか無いので、地図の前に置く。
#   ORA で有意でない組み合わせ（例: ko03010 の scheme B）はファイルが無いので飛ばす
for (kind in c('all', 'ORA')) {
    f = file.path(dir_pv, sprintf('%s.%s_%s.png', ko, SCHEME, kind))
    tag = if (kind == 'all') '(1) pathview - all genes of the pathway' else
                             '(2) pathview - ORA geneID only'
    if (!file.exists(f)) {
        cat(sprintf("%s → ファイルなし（セル11 未実行、またはこのスキームの ORA で有意でない）\n", tag))
        next
    }
    # ★ セル11 の検品で落ちた図（ko00121）は中身が書かれず 0 バイトで残る。
    #   file.exists() は TRUE になるので、大きさも見ないと readPNG がエラーで止まる
    if (file.size(f) == 0) {
        cat(sprintf("%s → pathview が描けなかった（セル11 の検品で「★ 描画が壊れた」）\n", tag))
        next
    }
    im = readPNG(f); h_px = dim(im)[1]; w_px = dim(im)[2]
    W = 15; H = W * h_px / w_px
    if (H > 18) { H = 18; W = H * w_px / h_px }
    cat(sprintf("%s  [%s]\n", tag, basename(f)))
    options(repr.plot.width = W, repr.plot.height = H)
    grid.newpage(); grid.raster(im)
}

# ── (3) ゲノム位置マップ ＋ 遺伝子表 ──────────────────────────
tgt = ora$term[ora$source=='KEGG' & grepl(paste0('^',ko,'_'), ora$term)][1]
g_ora = ora %>% filter(scheme==SCHEME, source=='KEGG', term==tgt, padj<0.05) %>%
        pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
        dplyr::select(gene_id, KEGG_Pathway) %>% tidyr::separate_rows(KEGG_Pathway, sep=',') %>%
        filter(KEGG_Pathway==ko) %>% pull(gene_id) %>% unique()

# contig を CDS 数の多い順に並べる（主 → 副）
ctg = res %>% filter(biotype=='CDS', strain %in% strains4, !is.na(end)) %>%
    group_by(strain, contig) %>% summarise(CDS=dplyr::n(), 長さMb=max(end)/1e6, .groups='drop') %>%
    arrange(factor(strain, levels=strains4), desc(CDS)) %>%
    # ★ 描画幅は最小値で底上げするが、x 軸は contig の実際の長さに合わせる。
    #   → 副 contig は bp あたりの縮尺が主より大きくなり、パネルを目一杯使う
    mutate(幅 = pmax(長さMb, MINW),
           短名 = sub('^[^~]+~', '', contig),
           # ★ contig 名は空白が無いので str_wrap では折れない。
           #   パネルが狭いと重なるので、8文字ごとに改行を入れて縦に積む（文字は削らない）
           名表示 = gsub('(.{10})', '\\1\n', 短名))

cat(sprintf("%s（%s）  contig %d 本（主 4 / 副 %d）\n", tgt, ko_jp, nrow(ctg), nrow(ctg)-4))
print(ctg %>% transmute(strain, contig=短名, CDS, 長さMb=round(長さMb,3), 描画幅=round(幅,2)) %>%
      as.data.frame(), row.names=FALSE)

bg = res %>% filter(biotype=='CDS', !marker, !is.na(start)) %>%
     mutate(位置Mb = start/1e6)
fg = bg %>% filter(gene_id %in% g_all) %>%
     mutate(cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                            !is.na(padj_A) & padj_A < 0.05 ~ 'DEG only', TRUE ~ 'not DEG'),
            cat = factor(cat, levels=names(col_cat)),
            # ★ 丸の塗りに発現強度（WT 群の正規化カウント平均）を入れる。
            #   発現量は3〜4桁にわたるので log10 にする（0 があるので +1）
            発現 = log10(normA_WT + 1),
            label = ifelse(is.na(Preferred_name)|Preferred_name=='-', substr(product,1,10), Preferred_name))
cat(sprintf("\n経路の全遺伝子 %d / 図に出る %d（padj が NA で検定されなかったものも含めて表示）\n",
    length(g_all), nrow(fg)))

ylim = range(c(bg$LFC_A[bg$strain %in% strains4]), na.rm=TRUE)

# ★ 凡例は「3分類が最もそろって現れる contig」1枚からだけ出す。
#   drop=FALSE を付けても ★ データが無い水準はキーの点が描かれないため、
#   パネルごとに凡例の見た目が変わり、guides='collect' が別物と判定して2つ並ぶ
leg_i = which.max(sapply(ctg$contig, function(cc)
            length(unique(fg$cat[fg$contig == cc]))))
cat(sprintf("凡例を出すパネル: %s（%d 水準）\n",
            ctg$短名[leg_i], length(unique(fg$cat[fg$contig == ctg$contig[leg_i]]))))

subs = list()
for (i in 1:nrow(ctg)) {
    b = bg %>% filter(contig == ctg$contig[i])
    f = fg %>% filter(contig == ctg$contig[i])
    is_main = ctg$CDS[i] == max(ctg$CDS[ctg$strain == ctg$strain[i]])
    p = ggplot(b, aes(x=位置Mb, y=LFC_A)) +
        geom_hline(yintercept=0, colour='grey20', linewidth=0.3) +
        geom_hline(yintercept=c(-1,1), linetype='dashed', colour='grey65', linewidth=0.3) +
        geom_point(colour='#b9c6d4', size=0.5, alpha=0.6) +
        coord_cartesian(xlim=c(0, ctg$長さMb[i]), ylim=ylim) +
        labs(title = sprintf('%s\n%s\n%d CDS, %.3f Mb',
                             ifelse(is_main, as.character(ctg$strain[i]), ''),
                             ctg$名表示[i], ctg$CDS[i], ctg$長さMb[i]),
             x=NULL, y=if (i==1) 'log2 fold change (KO / WT), scheme A' else NULL) +
        theme_bw(base_size=10) +
        theme(plot.title=element_text(size=ifelse(is_main, 9, 5.5),
                                    face=ifelse(is_main,'bold','plain'), lineheight=1.15),
              panel.grid.minor=element_blank(),
              axis.text.y=if (i==1) element_text() else element_blank(),
              axis.ticks.y=if (i==1) element_line() else element_blank(),
              legend.position='bottom', legend.box='horizontal',
              legend.title=element_text(size=8), legend.text=element_text(size=7.5),
              legend.margin=margin(1,1,1,1),
              plot.margin=margin(2,2,0,2))
    # ★ 経路の遺伝子が無い contig（0 行）では colour の水準が1つも現れないため、
    #   scale_colour_manual が「No shared levels found」の警告を出す。
    #   図は正しく描かれる（空パネルに色が無いだけ）ので、描画時に警告だけ抑える
    # ★ shape 21（縁と塗りを別々に指定できる丸）を使う。
    #   縁 = 分類（ORA hit / DEG only / not DEG）、塗り = 発現強度
    p = p + geom_point(data=f, aes(colour=cat, fill=発現), shape=21,
                       size=3.2, stroke=1.1, show.legend = (i == leg_i))
    if (nrow(f) > 0)
        p = p +
                # ★ 遺伝子名は小さいと読めないので大きめに。
                #   重なりは ggrepel の force と box.padding で逃がす
                geom_text_repel(data=f, aes(label=label, colour=cat), size=4.4,
                                fontface='bold', max.overlaps=60, seed=1,
                                force=3, min.segment.length=0,
                                segment.size=0.3, box.padding=0.45, show.legend=FALSE)
    # ★ 色の scale は「各パネルに」付ける。
    #   patchwork の guides='collect' は収集時点のパネルごとの凡例を比べるので、
    #   後から & で当てると収集に間に合わず凡例が2つ残る。
    #   全パネルが colour の aes を持ち、drop=FALSE で水準が揃うので1つにまとまる
    subs[[i]] = p + scale_colour_manual(values=col_cat, name=NULL, drop=FALSE,
                                        guide = guide_legend(override.aes = list(
                                            fill='white', size=3.2, stroke=1.1))) +
        # ★ 凡例を下に横向きで出す。右に置くと凡例の文字幅ぶん地図が狭くなる
        scale_fill_viridis_c(option='mako', direction=-1,
                             limits=range(fg$発現, na.rm=TRUE),
                             name='log10(DESeq2 normalized count + 1), WT mean',
                             guide = guide_colourbar(direction='horizontal',
                                                     title.position='top',
                                                     barwidth=unit(5,'cm'),
                                                     barheight=unit(0.35,'cm')))
}
P = wrap_plots(subs, nrow=1, widths=ctg$幅) + plot_layout(guides='collect') +
    plot_annotation(title = sprintf('A.  %s   [scheme %s]  - position on the genome', tgt, SCHEME),
      subtitle = paste0("grey = all CDS  |  coloured = genes of this pathway  ",
                        "(<span style='color:#d95f02'>**ORA hit**</span> / ",
                        "<span style='color:#7570b3'>**DEG only**</span> / ",
                        "<span style='color:#000000'>**not DEG**</span>)<br>",
                        "**gene labels = Preferred_name** (eggNOG; the first 10 characters of *product* ",
                        "are shown when Preferred_name is absent)<br>",
                        "point border = class, **point fill = log10(DESeq2 normalized count + 1), ",
                        "WT group mean** (scheme A: one size factor set for all 4 strains pooled)<br>",
                        "all contigs shown; panel width ∝ contig length, ",
                        sprintf("small contigs get a minimum width (%.1f Mb equivalent) and are drawn at a larger bp scale  |  x axis = position within each contig (Mb)", MINW)),

      # ★ 収集した凡例の位置は patchwork 側の theme で決まる。
      #   サブプロットの theme に legend.position を書いても効かない
      theme = theme(plot.title=element_text(size=14, face='bold'),
                    plot.subtitle=element_markdown(size=9, lineheight=1.3),
                    legend.position='bottom', legend.box='horizontal',
                    legend.title=element_text(size=8.5),
                    legend.text=element_text(size=8)))
# ── 表を同じ図の中に入れる（panel B）──────────────────────────
# ★ 図だけでは locus_tag / K番号 / EC が分からないので表を下に並べる。
#   ggpubr::ggtexttable は表を ggplot として返すので patchwork に載せられる
library(ggpubr)

tbl = fg %>%
    mutate(K  = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                       function(v) paste(v, collapse=',')),
           EC2 = ifelse(is.na(EC), '-', EC)) %>%
    transmute(strain, contig = sub('^[^~]+~', '', contig),
              Mb = round(位置Mb, 3), gene_id, name = label,
              K = K, EC = EC2, product = substr(product, 1, 26),
              # ★ LFC と padj は表に入れない。
              #   LFC は上の地図の y 軸そのもの、padj は分類（class）の色で示している。
              #   列を減らしたぶん1列あたりの幅が広がり、文字が重ならない
              class = as.character(cat)) %>%
    arrange(strain, Mb)

# ★ 縦に長くなりすぎるので、行を横に分割して並べる（1ブロック最大 24 行）
NROW_BLK = 24
nblk = ceiling(nrow(tbl) / NROW_BLK)
blks = split(1:nrow(tbl), ceiling(seq_along(1:nrow(tbl)) / ceiling(nrow(tbl)/nblk)))
cat(sprintf("\n表: %d 行 → %d ブロックに分けて横に並べる\n", nrow(tbl), length(blks)))

# ★ この図（地図＋表）だけ縦横 1.125 倍で出す（pathview の再掲は等倍のまま）
SCALE = 1.125
W_FIG = 22 * SCALE

# ★ 表のフォントは経路ごとに自動で決める。
#   列の幅は中身の文字数で決まるので、ko03010（112 遺伝子）のように行が多い経路では
#   ブロック数が増え、1ブロックに割ける幅が狭くなる。
#   固定の 6.2 pt のままだと表が割り当て幅をはみ出し、右端の class 列が切れてしまう。
#   → 実際に表を組んで幅を測り、収まる一番大きいフォントを選ぶ
#
# ★ 文字の幅は「フォントサイズに比例」しない。
#   小さいサイズでは hinting でカクカク変わる（実測: 5.5 pt で 4.75 inch、
#   5.9 pt で 5.04 inch）ので、計算で逆算せず 0.1 pt ずつ下げて実測で確かめる。
# ★ 測るときの device を間違えないこと。文字幅は device のフォント metrics で決まる。
#   実測（同じ表・同じ 5.9 pt）: pdf 4.70 / cairo png 4.81 / ragg 5.04 inch。
#   ggplot2 3.5 の ggsave は ragg があれば ragg を使うので、一番広く出る ragg で測る。
#   （Jupyter の画面表示は cairo png なので、ragg に合わせておけばどちらでも溢れない）
W_BLK = W_FIG / length(blks)             # 1ブロックに割ける幅 (inch)
ragg::agg_png(tempfile(fileext = '.png'), width = 1000, height = 1000, res = 100)
for (fs_try in seq(7.0, 3.4, by = -0.1)) {
    w_max = 0
    for (b in seq_along(blks)) {
        tg0 = gridExtra::tableGrob(
                  tbl[blks[[b]], ], rows = NULL,
                  theme = ttheme(base_size = fs_try, padding = unit(c(2.2, 1.6), 'mm'),
                                 colnames.style = colnames_style(size = fs_try + 0.4),
                                 tbody.style = tbody_style(size = fs_try)))
        w_max = max(w_max, sum(convertWidth(tg0$widths, 'in', valueOnly = TRUE)))
    }
    FS = fs_try
    if (w_max <= W_BLK * 0.98) break     # 0.98 は plot.margin のぶんの余裕
}
invisible(dev.off())
cat(sprintf("表のフォント: %.1f pt（最も広いブロック %.2f inch / 1ブロックの割り当て %.2f inch）\n",
            FS, w_max, W_BLK))

# ★ 表の文字色を地図と揃える（ORA hit = 橙 / DEG only = 紫 / not DEG = 黒）。
#   ggtexttable（gridExtra の tableGrob）は色を ★ 列方向（column-major）に展開する ので、
#   行ごとの色ベクトルを列数ぶん「繰り返して」渡す（rep(col, ncol) であって each= ではない。
#   each= にすると列ごとに1色になる。実測で確認）
tg = lapply(seq_along(blks), function(b) {
    ix  = blks[[b]]
    col = unname(col_cat[tbl$class[ix]])
    g = ggtexttable(tbl[ix, ], rows = NULL,
                theme = ttheme(base_size = FS, padding = unit(c(2.2, 1.6), 'mm'),
                               colnames.style = colnames_style(size = FS + 0.4, fill = 'grey92'),
                               tbody.style = tbody_style(size = FS, fill = 'white',
                                                         color = rep(col, ncol(tbl)))))
    g = g + theme(plot.margin = margin(0, 2, 2, 2))
    if (b == 1) g = g + labs(title = 'B.  gene list') +
        theme(plot.title = element_text(size = 13, face = 'bold', hjust = 0,
                                        margin = margin(b = 2)))
    g
})
TB = wrap_plots(tg, nrow = 1)

# ★ 表（ggtexttable）はパネルの中央に置かれるので、割り当てた高さが余ると
#   上下に空白ができる。行数から必要な高さを詰めて見積もる
H_map = 6.4
H_tab = 0.125 * max(sapply(blks, length)) + 0.35
# ★ tag_levels は使わない。patchwork の全サブプロット（contig パネル10個 + 表3枚）に
#   A〜M と振られてしまうため。見出しは A の図タイトルと B の表タイトルで示す。
#
# ★ wrap_elements() で包む。
#   patchwork を入れ子（P / TB）にすると ★ 内側の plot_annotation が描画されず、
#   A のタイトルが消える。包むと内側が1つの要素として確定し、タイトルが残る
PB = (wrap_elements(full = P) / TB) + plot_layout(heights = c(H_map, H_tab))

# SCALE / W_FIG は表のフォントを決めるときに上で定義してある
suppressWarnings(suppressMessages(
    ggsave(file.path(dir_gp, sprintf('genomepos_%s_%s.png', ko, SCHEME)), PB,
           width = W_FIG, height = (H_map + H_tab) * SCALE,
           dpi = 100, limitsize = FALSE)))
options(repr.plot.width = W_FIG, repr.plot.height = (H_map + H_tab) * SCALE)
suppressWarnings(print(PB))



(1) pathview - all genes of the pathway  [ko00030.A_all.png]
(2) pathview - ORA geneID only  [ko00030.A_ORA.png]


plot without title

ko00030_Pentose phosphate pathway（ペントースリン酸経路）  contig 10 本（主 4 / 副 6）
   strain         contig  CDS 長さMb 描画幅
  Bth_VPI    NC_004663.1 1676  6.243   6.24
  Bth_VPI    NC_004703.1   36  0.033   0.60
  Bth_St8 BQNN01000001.1 1780  6.132   6.13
  Bth_St8 BQNN01000002.1   41  0.045   0.60
  Bth_St8 BQNN01000005.1    7  0.004   0.60
  Bth_St8 BQNN01000004.1    2  0.004   0.60
 Bun_8492  NZ_CP102263.1 3392  4.629   4.63
 Odo_St23 BQOE01000001.1 2801  5.647   5.65
 Odo_St23 BQOE01000002.1   38  0.065   0.60
 Odo_St23 BQOE01000003.1   12  0.032   0.60

経路の全遺伝子 69 / 図に出る 69（padj が NA で検定されなかったものも含めて表示）
凡例を出すパネル: NC_004663.1（3 水準）

表: 69 行 → 3 ブロックに分けて横に並べる
表のフォント: 7.0 pt（最も広いブロック 7.34 inch / 1ブロックの割り当て 8.25 inch）


plot without title

plot without title

In [116]:
# ── 図13-2: ko00020（TCA回路）  scheme A ───────────────────
# セル13（準備）を先に実行しておくこと（col_cat / strains4 / MINW / dir_pv / dir_gp を使う）
#
# ★ 3つを続けて出す
#   (1) pathview（経路の全遺伝子）… 反応と化合物名。経路の中での位置づけ
#   (2) pathview（ORA の geneID のみ）… ORA が拾った反応だけ
#   (3) ゲノム位置マップ ＋ 遺伝子表 … どこにあるか、何の遺伝子か
library(ggplot2); library(ggtext); library(ggrepel); library(patchwork); library(ggpubr)
library(png); library(grid)

ko = 'ko00020'; ko_jp = 'TCA回路'; SCHEME = 'A'

# ── (1)(2) セル11 で作った pathview を再掲する ────────────────
# ★ 化合物名や反応の向きは pathview にしか無いので、地図の前に置く。
#   ORA で有意でない組み合わせ（例: ko03010 の scheme B）はファイルが無いので飛ばす
for (kind in c('all', 'ORA')) {
    f = file.path(dir_pv, sprintf('%s.%s_%s.png', ko, SCHEME, kind))
    tag = if (kind == 'all') '(1) pathview - all genes of the pathway' else
                             '(2) pathview - ORA geneID only'
    if (!file.exists(f)) {
        cat(sprintf("%s → ファイルなし（セル11 未実行、またはこのスキームの ORA で有意でない）\n", tag))
        next
    }
    # ★ セル11 の検品で落ちた図（ko00121）は中身が書かれず 0 バイトで残る。
    #   file.exists() は TRUE になるので、大きさも見ないと readPNG がエラーで止まる
    if (file.size(f) == 0) {
        cat(sprintf("%s → pathview が描けなかった（セル11 の検品で「★ 描画が壊れた」）\n", tag))
        next
    }
    im = readPNG(f); h_px = dim(im)[1]; w_px = dim(im)[2]
    W = 15; H = W * h_px / w_px
    if (H > 18) { H = 18; W = H * w_px / h_px }
    cat(sprintf("%s  [%s]\n", tag, basename(f)))
    options(repr.plot.width = W, repr.plot.height = H)
    grid.newpage(); grid.raster(im)
}

# ── (3) ゲノム位置マップ ＋ 遺伝子表 ──────────────────────────
tgt = ora$term[ora$source=='KEGG' & grepl(paste0('^',ko,'_'), ora$term)][1]
g_ora = ora %>% filter(scheme==SCHEME, source=='KEGG', term==tgt, padj<0.05) %>%
        pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
        dplyr::select(gene_id, KEGG_Pathway) %>% tidyr::separate_rows(KEGG_Pathway, sep=',') %>%
        filter(KEGG_Pathway==ko) %>% pull(gene_id) %>% unique()

# contig を CDS 数の多い順に並べる（主 → 副）
ctg = res %>% filter(biotype=='CDS', strain %in% strains4, !is.na(end)) %>%
    group_by(strain, contig) %>% summarise(CDS=dplyr::n(), 長さMb=max(end)/1e6, .groups='drop') %>%
    arrange(factor(strain, levels=strains4), desc(CDS)) %>%
    # ★ 描画幅は最小値で底上げするが、x 軸は contig の実際の長さに合わせる。
    #   → 副 contig は bp あたりの縮尺が主より大きくなり、パネルを目一杯使う
    mutate(幅 = pmax(長さMb, MINW),
           短名 = sub('^[^~]+~', '', contig),
           # ★ contig 名は空白が無いので str_wrap では折れない。
           #   パネルが狭いと重なるので、8文字ごとに改行を入れて縦に積む（文字は削らない）
           名表示 = gsub('(.{10})', '\\1\n', 短名))

cat(sprintf("%s（%s）  contig %d 本（主 4 / 副 %d）\n", tgt, ko_jp, nrow(ctg), nrow(ctg)-4))
print(ctg %>% transmute(strain, contig=短名, CDS, 長さMb=round(長さMb,3), 描画幅=round(幅,2)) %>%
      as.data.frame(), row.names=FALSE)

bg = res %>% filter(biotype=='CDS', !marker, !is.na(start)) %>%
     mutate(位置Mb = start/1e6)
fg = bg %>% filter(gene_id %in% g_all) %>%
     mutate(cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                            !is.na(padj_A) & padj_A < 0.05 ~ 'DEG only', TRUE ~ 'not DEG'),
            cat = factor(cat, levels=names(col_cat)),
            # ★ 丸の塗りに発現強度（WT 群の正規化カウント平均）を入れる。
            #   発現量は3〜4桁にわたるので log10 にする（0 があるので +1）
            発現 = log10(normA_WT + 1),
            label = ifelse(is.na(Preferred_name)|Preferred_name=='-', substr(product,1,10), Preferred_name))
cat(sprintf("\n経路の全遺伝子 %d / 図に出る %d（padj が NA で検定されなかったものも含めて表示）\n",
    length(g_all), nrow(fg)))

ylim = range(c(bg$LFC_A[bg$strain %in% strains4]), na.rm=TRUE)

# ★ 凡例は「3分類が最もそろって現れる contig」1枚からだけ出す。
#   drop=FALSE を付けても ★ データが無い水準はキーの点が描かれないため、
#   パネルごとに凡例の見た目が変わり、guides='collect' が別物と判定して2つ並ぶ
leg_i = which.max(sapply(ctg$contig, function(cc)
            length(unique(fg$cat[fg$contig == cc]))))
cat(sprintf("凡例を出すパネル: %s（%d 水準）\n",
            ctg$短名[leg_i], length(unique(fg$cat[fg$contig == ctg$contig[leg_i]]))))

subs = list()
for (i in 1:nrow(ctg)) {
    b = bg %>% filter(contig == ctg$contig[i])
    f = fg %>% filter(contig == ctg$contig[i])
    is_main = ctg$CDS[i] == max(ctg$CDS[ctg$strain == ctg$strain[i]])
    p = ggplot(b, aes(x=位置Mb, y=LFC_A)) +
        geom_hline(yintercept=0, colour='grey20', linewidth=0.3) +
        geom_hline(yintercept=c(-1,1), linetype='dashed', colour='grey65', linewidth=0.3) +
        geom_point(colour='#b9c6d4', size=0.5, alpha=0.6) +
        coord_cartesian(xlim=c(0, ctg$長さMb[i]), ylim=ylim) +
        labs(title = sprintf('%s\n%s\n%d CDS, %.3f Mb',
                             ifelse(is_main, as.character(ctg$strain[i]), ''),
                             ctg$名表示[i], ctg$CDS[i], ctg$長さMb[i]),
             x=NULL, y=if (i==1) 'log2 fold change (KO / WT), scheme A' else NULL) +
        theme_bw(base_size=10) +
        theme(plot.title=element_text(size=ifelse(is_main, 9, 5.5),
                                    face=ifelse(is_main,'bold','plain'), lineheight=1.15),
              panel.grid.minor=element_blank(),
              axis.text.y=if (i==1) element_text() else element_blank(),
              axis.ticks.y=if (i==1) element_line() else element_blank(),
              legend.position='bottom', legend.box='horizontal',
              legend.title=element_text(size=8), legend.text=element_text(size=7.5),
              legend.margin=margin(1,1,1,1),
              plot.margin=margin(2,2,0,2))
    # ★ 経路の遺伝子が無い contig（0 行）では colour の水準が1つも現れないため、
    #   scale_colour_manual が「No shared levels found」の警告を出す。
    #   図は正しく描かれる（空パネルに色が無いだけ）ので、描画時に警告だけ抑える
    # ★ shape 21（縁と塗りを別々に指定できる丸）を使う。
    #   縁 = 分類（ORA hit / DEG only / not DEG）、塗り = 発現強度
    p = p + geom_point(data=f, aes(colour=cat, fill=発現), shape=21,
                       size=3.2, stroke=1.1, show.legend = (i == leg_i))
    if (nrow(f) > 0)
        p = p +
                # ★ 遺伝子名は小さいと読めないので大きめに。
                #   重なりは ggrepel の force と box.padding で逃がす
                geom_text_repel(data=f, aes(label=label, colour=cat), size=4.4,
                                fontface='bold', max.overlaps=60, seed=1,
                                force=3, min.segment.length=0,
                                segment.size=0.3, box.padding=0.45, show.legend=FALSE)
    # ★ 色の scale は「各パネルに」付ける。
    #   patchwork の guides='collect' は収集時点のパネルごとの凡例を比べるので、
    #   後から & で当てると収集に間に合わず凡例が2つ残る。
    #   全パネルが colour の aes を持ち、drop=FALSE で水準が揃うので1つにまとまる
    subs[[i]] = p + scale_colour_manual(values=col_cat, name=NULL, drop=FALSE,
                                        guide = guide_legend(override.aes = list(
                                            fill='white', size=3.2, stroke=1.1))) +
        # ★ 凡例を下に横向きで出す。右に置くと凡例の文字幅ぶん地図が狭くなる
        scale_fill_viridis_c(option='mako', direction=-1,
                             limits=range(fg$発現, na.rm=TRUE),
                             name='log10(DESeq2 normalized count + 1), WT mean',
                             guide = guide_colourbar(direction='horizontal',
                                                     title.position='top',
                                                     barwidth=unit(5,'cm'),
                                                     barheight=unit(0.35,'cm')))
}
P = wrap_plots(subs, nrow=1, widths=ctg$幅) + plot_layout(guides='collect') +
    plot_annotation(title = sprintf('A.  %s   [scheme %s]  - position on the genome', tgt, SCHEME),
      subtitle = paste0("grey = all CDS  |  coloured = genes of this pathway  ",
                        "(<span style='color:#d95f02'>**ORA hit**</span> / ",
                        "<span style='color:#7570b3'>**DEG only**</span> / ",
                        "<span style='color:#000000'>**not DEG**</span>)<br>",
                        "**gene labels = Preferred_name** (eggNOG; the first 10 characters of *product* ",
                        "are shown when Preferred_name is absent)<br>",
                        "point border = class, **point fill = log10(DESeq2 normalized count + 1), ",
                        "WT group mean** (scheme A: one size factor set for all 4 strains pooled)<br>",
                        "all contigs shown; panel width ∝ contig length, ",
                        sprintf("small contigs get a minimum width (%.1f Mb equivalent) and are drawn at a larger bp scale  |  x axis = position within each contig (Mb)", MINW)),

      # ★ 収集した凡例の位置は patchwork 側の theme で決まる。
      #   サブプロットの theme に legend.position を書いても効かない
      theme = theme(plot.title=element_text(size=14, face='bold'),
                    plot.subtitle=element_markdown(size=9, lineheight=1.3),
                    legend.position='bottom', legend.box='horizontal',
                    legend.title=element_text(size=8.5),
                    legend.text=element_text(size=8)))
# ── 表を同じ図の中に入れる（panel B）──────────────────────────
# ★ 図だけでは locus_tag / K番号 / EC が分からないので表を下に並べる。
#   ggpubr::ggtexttable は表を ggplot として返すので patchwork に載せられる
library(ggpubr)

tbl = fg %>%
    mutate(K  = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                       function(v) paste(v, collapse=',')),
           EC2 = ifelse(is.na(EC), '-', EC)) %>%
    transmute(strain, contig = sub('^[^~]+~', '', contig),
              Mb = round(位置Mb, 3), gene_id, name = label,
              K = K, EC = EC2, product = substr(product, 1, 26),
              # ★ LFC と padj は表に入れない。
              #   LFC は上の地図の y 軸そのもの、padj は分類（class）の色で示している。
              #   列を減らしたぶん1列あたりの幅が広がり、文字が重ならない
              class = as.character(cat)) %>%
    arrange(strain, Mb)

# ★ 縦に長くなりすぎるので、行を横に分割して並べる（1ブロック最大 24 行）
NROW_BLK = 24
nblk = ceiling(nrow(tbl) / NROW_BLK)
blks = split(1:nrow(tbl), ceiling(seq_along(1:nrow(tbl)) / ceiling(nrow(tbl)/nblk)))
cat(sprintf("\n表: %d 行 → %d ブロックに分けて横に並べる\n", nrow(tbl), length(blks)))

# ★ この図（地図＋表）だけ縦横 1.125 倍で出す（pathview の再掲は等倍のまま）
SCALE = 1.125
W_FIG = 22 * SCALE

# ★ 表のフォントは経路ごとに自動で決める。
#   列の幅は中身の文字数で決まるので、ko03010（112 遺伝子）のように行が多い経路では
#   ブロック数が増え、1ブロックに割ける幅が狭くなる。
#   固定の 6.2 pt のままだと表が割り当て幅をはみ出し、右端の class 列が切れてしまう。
#   → 実際に表を組んで幅を測り、収まる一番大きいフォントを選ぶ
#
# ★ 文字の幅は「フォントサイズに比例」しない。
#   小さいサイズでは hinting でカクカク変わる（実測: 5.5 pt で 4.75 inch、
#   5.9 pt で 5.04 inch）ので、計算で逆算せず 0.1 pt ずつ下げて実測で確かめる。
# ★ 測るときの device を間違えないこと。文字幅は device のフォント metrics で決まる。
#   実測（同じ表・同じ 5.9 pt）: pdf 4.70 / cairo png 4.81 / ragg 5.04 inch。
#   ggplot2 3.5 の ggsave は ragg があれば ragg を使うので、一番広く出る ragg で測る。
#   （Jupyter の画面表示は cairo png なので、ragg に合わせておけばどちらでも溢れない）
W_BLK = W_FIG / length(blks)             # 1ブロックに割ける幅 (inch)
ragg::agg_png(tempfile(fileext = '.png'), width = 1000, height = 1000, res = 100)
for (fs_try in seq(7.0, 3.4, by = -0.1)) {
    w_max = 0
    for (b in seq_along(blks)) {
        tg0 = gridExtra::tableGrob(
                  tbl[blks[[b]], ], rows = NULL,
                  theme = ttheme(base_size = fs_try, padding = unit(c(2.2, 1.6), 'mm'),
                                 colnames.style = colnames_style(size = fs_try + 0.4),
                                 tbody.style = tbody_style(size = fs_try)))
        w_max = max(w_max, sum(convertWidth(tg0$widths, 'in', valueOnly = TRUE)))
    }
    FS = fs_try
    if (w_max <= W_BLK * 0.98) break     # 0.98 は plot.margin のぶんの余裕
}
invisible(dev.off())
cat(sprintf("表のフォント: %.1f pt（最も広いブロック %.2f inch / 1ブロックの割り当て %.2f inch）\n",
            FS, w_max, W_BLK))

# ★ 表の文字色を地図と揃える（ORA hit = 橙 / DEG only = 紫 / not DEG = 黒）。
#   ggtexttable（gridExtra の tableGrob）は色を ★ 列方向（column-major）に展開する ので、
#   行ごとの色ベクトルを列数ぶん「繰り返して」渡す（rep(col, ncol) であって each= ではない。
#   each= にすると列ごとに1色になる。実測で確認）
tg = lapply(seq_along(blks), function(b) {
    ix  = blks[[b]]
    col = unname(col_cat[tbl$class[ix]])
    g = ggtexttable(tbl[ix, ], rows = NULL,
                theme = ttheme(base_size = FS, padding = unit(c(2.2, 1.6), 'mm'),
                               colnames.style = colnames_style(size = FS + 0.4, fill = 'grey92'),
                               tbody.style = tbody_style(size = FS, fill = 'white',
                                                         color = rep(col, ncol(tbl)))))
    g = g + theme(plot.margin = margin(0, 2, 2, 2))
    if (b == 1) g = g + labs(title = 'B.  gene list') +
        theme(plot.title = element_text(size = 13, face = 'bold', hjust = 0,
                                        margin = margin(b = 2)))
    g
})
TB = wrap_plots(tg, nrow = 1)

# ★ 表（ggtexttable）はパネルの中央に置かれるので、割り当てた高さが余ると
#   上下に空白ができる。行数から必要な高さを詰めて見積もる
H_map = 6.4
H_tab = 0.125 * max(sapply(blks, length)) + 0.35
# ★ tag_levels は使わない。patchwork の全サブプロット（contig パネル10個 + 表3枚）に
#   A〜M と振られてしまうため。見出しは A の図タイトルと B の表タイトルで示す。
#
# ★ wrap_elements() で包む。
#   patchwork を入れ子（P / TB）にすると ★ 内側の plot_annotation が描画されず、
#   A のタイトルが消える。包むと内側が1つの要素として確定し、タイトルが残る
PB = (wrap_elements(full = P) / TB) + plot_layout(heights = c(H_map, H_tab))

# SCALE / W_FIG は表のフォントを決めるときに上で定義してある
suppressWarnings(suppressMessages(
    ggsave(file.path(dir_gp, sprintf('genomepos_%s_%s.png', ko, SCHEME)), PB,
           width = W_FIG, height = (H_map + H_tab) * SCALE,
           dpi = 100, limitsize = FALSE)))
options(repr.plot.width = W_FIG, repr.plot.height = (H_map + H_tab) * SCALE)
suppressWarnings(print(PB))



(1) pathview - all genes of the pathway  [ko00020.A_all.png]
(2) pathview - ORA geneID only  [ko00020.A_ORA.png]


plot without title

ko00020_Citrate cycle (TCA cycle)（TCA回路）  contig 10 本（主 4 / 副 6）
   strain         contig  CDS 長さMb 描画幅
  Bth_VPI    NC_004663.1 1676  6.243   6.24
  Bth_VPI    NC_004703.1   36  0.033   0.60
  Bth_St8 BQNN01000001.1 1780  6.132   6.13
  Bth_St8 BQNN01000002.1   41  0.045   0.60
  Bth_St8 BQNN01000005.1    7  0.004   0.60
  Bth_St8 BQNN01000004.1    2  0.004   0.60
 Bun_8492  NZ_CP102263.1 3392  4.629   4.63
 Odo_St23 BQOE01000001.1 2801  5.647   5.65
 Odo_St23 BQOE01000002.1   38  0.065   0.60
 Odo_St23 BQOE01000003.1   12  0.032   0.60

経路の全遺伝子 61 / 図に出る 61（padj が NA で検定されなかったものも含めて表示）
凡例を出すパネル: NZ_CP102263.1（3 水準）

表: 61 行 → 3 ブロックに分けて横に並べる
表のフォント: 7.0 pt（最も広いブロック 7.19 inch / 1ブロックの割り当て 8.25 inch）


plot without title

plot without title

In [117]:
# ── 図13-3: ko00710（炭素固定(Calvin)）  scheme A ───────────────────
# セル13（準備）を先に実行しておくこと（col_cat / strains4 / MINW / dir_pv / dir_gp を使う）
#
# ★ 3つを続けて出す
#   (1) pathview（経路の全遺伝子）… 反応と化合物名。経路の中での位置づけ
#   (2) pathview（ORA の geneID のみ）… ORA が拾った反応だけ
#   (3) ゲノム位置マップ ＋ 遺伝子表 … どこにあるか、何の遺伝子か
library(ggplot2); library(ggtext); library(ggrepel); library(patchwork); library(ggpubr)
library(png); library(grid)

ko = 'ko00710'; ko_jp = '炭素固定(Calvin)'; SCHEME = 'A'

# ── (1)(2) セル11 で作った pathview を再掲する ────────────────
# ★ 化合物名や反応の向きは pathview にしか無いので、地図の前に置く。
#   ORA で有意でない組み合わせ（例: ko03010 の scheme B）はファイルが無いので飛ばす
for (kind in c('all', 'ORA')) {
    f = file.path(dir_pv, sprintf('%s.%s_%s.png', ko, SCHEME, kind))
    tag = if (kind == 'all') '(1) pathview - all genes of the pathway' else
                             '(2) pathview - ORA geneID only'
    if (!file.exists(f)) {
        cat(sprintf("%s → ファイルなし（セル11 未実行、またはこのスキームの ORA で有意でない）\n", tag))
        next
    }
    # ★ セル11 の検品で落ちた図（ko00121）は中身が書かれず 0 バイトで残る。
    #   file.exists() は TRUE になるので、大きさも見ないと readPNG がエラーで止まる
    if (file.size(f) == 0) {
        cat(sprintf("%s → pathview が描けなかった（セル11 の検品で「★ 描画が壊れた」）\n", tag))
        next
    }
    im = readPNG(f); h_px = dim(im)[1]; w_px = dim(im)[2]
    W = 15; H = W * h_px / w_px
    if (H > 18) { H = 18; W = H * w_px / h_px }
    cat(sprintf("%s  [%s]\n", tag, basename(f)))
    options(repr.plot.width = W, repr.plot.height = H)
    grid.newpage(); grid.raster(im)
}

# ── (3) ゲノム位置マップ ＋ 遺伝子表 ──────────────────────────
tgt = ora$term[ora$source=='KEGG' & grepl(paste0('^',ko,'_'), ora$term)][1]
g_ora = ora %>% filter(scheme==SCHEME, source=='KEGG', term==tgt, padj<0.05) %>%
        pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
        dplyr::select(gene_id, KEGG_Pathway) %>% tidyr::separate_rows(KEGG_Pathway, sep=',') %>%
        filter(KEGG_Pathway==ko) %>% pull(gene_id) %>% unique()

# contig を CDS 数の多い順に並べる（主 → 副）
ctg = res %>% filter(biotype=='CDS', strain %in% strains4, !is.na(end)) %>%
    group_by(strain, contig) %>% summarise(CDS=dplyr::n(), 長さMb=max(end)/1e6, .groups='drop') %>%
    arrange(factor(strain, levels=strains4), desc(CDS)) %>%
    # ★ 描画幅は最小値で底上げするが、x 軸は contig の実際の長さに合わせる。
    #   → 副 contig は bp あたりの縮尺が主より大きくなり、パネルを目一杯使う
    mutate(幅 = pmax(長さMb, MINW),
           短名 = sub('^[^~]+~', '', contig),
           # ★ contig 名は空白が無いので str_wrap では折れない。
           #   パネルが狭いと重なるので、8文字ごとに改行を入れて縦に積む（文字は削らない）
           名表示 = gsub('(.{10})', '\\1\n', 短名))

cat(sprintf("%s（%s）  contig %d 本（主 4 / 副 %d）\n", tgt, ko_jp, nrow(ctg), nrow(ctg)-4))
print(ctg %>% transmute(strain, contig=短名, CDS, 長さMb=round(長さMb,3), 描画幅=round(幅,2)) %>%
      as.data.frame(), row.names=FALSE)

bg = res %>% filter(biotype=='CDS', !marker, !is.na(start)) %>%
     mutate(位置Mb = start/1e6)
fg = bg %>% filter(gene_id %in% g_all) %>%
     mutate(cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                            !is.na(padj_A) & padj_A < 0.05 ~ 'DEG only', TRUE ~ 'not DEG'),
            cat = factor(cat, levels=names(col_cat)),
            # ★ 丸の塗りに発現強度（WT 群の正規化カウント平均）を入れる。
            #   発現量は3〜4桁にわたるので log10 にする（0 があるので +1）
            発現 = log10(normA_WT + 1),
            label = ifelse(is.na(Preferred_name)|Preferred_name=='-', substr(product,1,10), Preferred_name))
cat(sprintf("\n経路の全遺伝子 %d / 図に出る %d（padj が NA で検定されなかったものも含めて表示）\n",
    length(g_all), nrow(fg)))

ylim = range(c(bg$LFC_A[bg$strain %in% strains4]), na.rm=TRUE)

# ★ 凡例は「3分類が最もそろって現れる contig」1枚からだけ出す。
#   drop=FALSE を付けても ★ データが無い水準はキーの点が描かれないため、
#   パネルごとに凡例の見た目が変わり、guides='collect' が別物と判定して2つ並ぶ
leg_i = which.max(sapply(ctg$contig, function(cc)
            length(unique(fg$cat[fg$contig == cc]))))
cat(sprintf("凡例を出すパネル: %s（%d 水準）\n",
            ctg$短名[leg_i], length(unique(fg$cat[fg$contig == ctg$contig[leg_i]]))))

subs = list()
for (i in 1:nrow(ctg)) {
    b = bg %>% filter(contig == ctg$contig[i])
    f = fg %>% filter(contig == ctg$contig[i])
    is_main = ctg$CDS[i] == max(ctg$CDS[ctg$strain == ctg$strain[i]])
    p = ggplot(b, aes(x=位置Mb, y=LFC_A)) +
        geom_hline(yintercept=0, colour='grey20', linewidth=0.3) +
        geom_hline(yintercept=c(-1,1), linetype='dashed', colour='grey65', linewidth=0.3) +
        geom_point(colour='#b9c6d4', size=0.5, alpha=0.6) +
        coord_cartesian(xlim=c(0, ctg$長さMb[i]), ylim=ylim) +
        labs(title = sprintf('%s\n%s\n%d CDS, %.3f Mb',
                             ifelse(is_main, as.character(ctg$strain[i]), ''),
                             ctg$名表示[i], ctg$CDS[i], ctg$長さMb[i]),
             x=NULL, y=if (i==1) 'log2 fold change (KO / WT), scheme A' else NULL) +
        theme_bw(base_size=10) +
        theme(plot.title=element_text(size=ifelse(is_main, 9, 5.5),
                                    face=ifelse(is_main,'bold','plain'), lineheight=1.15),
              panel.grid.minor=element_blank(),
              axis.text.y=if (i==1) element_text() else element_blank(),
              axis.ticks.y=if (i==1) element_line() else element_blank(),
              legend.position='bottom', legend.box='horizontal',
              legend.title=element_text(size=8), legend.text=element_text(size=7.5),
              legend.margin=margin(1,1,1,1),
              plot.margin=margin(2,2,0,2))
    # ★ 経路の遺伝子が無い contig（0 行）では colour の水準が1つも現れないため、
    #   scale_colour_manual が「No shared levels found」の警告を出す。
    #   図は正しく描かれる（空パネルに色が無いだけ）ので、描画時に警告だけ抑える
    # ★ shape 21（縁と塗りを別々に指定できる丸）を使う。
    #   縁 = 分類（ORA hit / DEG only / not DEG）、塗り = 発現強度
    p = p + geom_point(data=f, aes(colour=cat, fill=発現), shape=21,
                       size=3.2, stroke=1.1, show.legend = (i == leg_i))
    if (nrow(f) > 0)
        p = p +
                # ★ 遺伝子名は小さいと読めないので大きめに。
                #   重なりは ggrepel の force と box.padding で逃がす
                geom_text_repel(data=f, aes(label=label, colour=cat), size=4.4,
                                fontface='bold', max.overlaps=60, seed=1,
                                force=3, min.segment.length=0,
                                segment.size=0.3, box.padding=0.45, show.legend=FALSE)
    # ★ 色の scale は「各パネルに」付ける。
    #   patchwork の guides='collect' は収集時点のパネルごとの凡例を比べるので、
    #   後から & で当てると収集に間に合わず凡例が2つ残る。
    #   全パネルが colour の aes を持ち、drop=FALSE で水準が揃うので1つにまとまる
    subs[[i]] = p + scale_colour_manual(values=col_cat, name=NULL, drop=FALSE,
                                        guide = guide_legend(override.aes = list(
                                            fill='white', size=3.2, stroke=1.1))) +
        # ★ 凡例を下に横向きで出す。右に置くと凡例の文字幅ぶん地図が狭くなる
        scale_fill_viridis_c(option='mako', direction=-1,
                             limits=range(fg$発現, na.rm=TRUE),
                             name='log10(DESeq2 normalized count + 1), WT mean',
                             guide = guide_colourbar(direction='horizontal',
                                                     title.position='top',
                                                     barwidth=unit(5,'cm'),
                                                     barheight=unit(0.35,'cm')))
}
P = wrap_plots(subs, nrow=1, widths=ctg$幅) + plot_layout(guides='collect') +
    plot_annotation(title = sprintf('A.  %s   [scheme %s]  - position on the genome', tgt, SCHEME),
      subtitle = paste0("grey = all CDS  |  coloured = genes of this pathway  ",
                        "(<span style='color:#d95f02'>**ORA hit**</span> / ",
                        "<span style='color:#7570b3'>**DEG only**</span> / ",
                        "<span style='color:#000000'>**not DEG**</span>)<br>",
                        "**gene labels = Preferred_name** (eggNOG; the first 10 characters of *product* ",
                        "are shown when Preferred_name is absent)<br>",
                        "point border = class, **point fill = log10(DESeq2 normalized count + 1), ",
                        "WT group mean** (scheme A: one size factor set for all 4 strains pooled)<br>",
                        "all contigs shown; panel width ∝ contig length, ",
                        sprintf("small contigs get a minimum width (%.1f Mb equivalent) and are drawn at a larger bp scale  |  x axis = position within each contig (Mb)", MINW)),

      # ★ 収集した凡例の位置は patchwork 側の theme で決まる。
      #   サブプロットの theme に legend.position を書いても効かない
      theme = theme(plot.title=element_text(size=14, face='bold'),
                    plot.subtitle=element_markdown(size=9, lineheight=1.3),
                    legend.position='bottom', legend.box='horizontal',
                    legend.title=element_text(size=8.5),
                    legend.text=element_text(size=8)))
# ── 表を同じ図の中に入れる（panel B）──────────────────────────
# ★ 図だけでは locus_tag / K番号 / EC が分からないので表を下に並べる。
#   ggpubr::ggtexttable は表を ggplot として返すので patchwork に載せられる
library(ggpubr)

tbl = fg %>%
    mutate(K  = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                       function(v) paste(v, collapse=',')),
           EC2 = ifelse(is.na(EC), '-', EC)) %>%
    transmute(strain, contig = sub('^[^~]+~', '', contig),
              Mb = round(位置Mb, 3), gene_id, name = label,
              K = K, EC = EC2, product = substr(product, 1, 26),
              # ★ LFC と padj は表に入れない。
              #   LFC は上の地図の y 軸そのもの、padj は分類（class）の色で示している。
              #   列を減らしたぶん1列あたりの幅が広がり、文字が重ならない
              class = as.character(cat)) %>%
    arrange(strain, Mb)

# ★ 縦に長くなりすぎるので、行を横に分割して並べる（1ブロック最大 24 行）
NROW_BLK = 24
nblk = ceiling(nrow(tbl) / NROW_BLK)
blks = split(1:nrow(tbl), ceiling(seq_along(1:nrow(tbl)) / ceiling(nrow(tbl)/nblk)))
cat(sprintf("\n表: %d 行 → %d ブロックに分けて横に並べる\n", nrow(tbl), length(blks)))

# ★ この図（地図＋表）だけ縦横 1.125 倍で出す（pathview の再掲は等倍のまま）
SCALE = 1.125
W_FIG = 22 * SCALE

# ★ 表のフォントは経路ごとに自動で決める。
#   列の幅は中身の文字数で決まるので、ko03010（112 遺伝子）のように行が多い経路では
#   ブロック数が増え、1ブロックに割ける幅が狭くなる。
#   固定の 6.2 pt のままだと表が割り当て幅をはみ出し、右端の class 列が切れてしまう。
#   → 実際に表を組んで幅を測り、収まる一番大きいフォントを選ぶ
#
# ★ 文字の幅は「フォントサイズに比例」しない。
#   小さいサイズでは hinting でカクカク変わる（実測: 5.5 pt で 4.75 inch、
#   5.9 pt で 5.04 inch）ので、計算で逆算せず 0.1 pt ずつ下げて実測で確かめる。
# ★ 測るときの device を間違えないこと。文字幅は device のフォント metrics で決まる。
#   実測（同じ表・同じ 5.9 pt）: pdf 4.70 / cairo png 4.81 / ragg 5.04 inch。
#   ggplot2 3.5 の ggsave は ragg があれば ragg を使うので、一番広く出る ragg で測る。
#   （Jupyter の画面表示は cairo png なので、ragg に合わせておけばどちらでも溢れない）
W_BLK = W_FIG / length(blks)             # 1ブロックに割ける幅 (inch)
ragg::agg_png(tempfile(fileext = '.png'), width = 1000, height = 1000, res = 100)
for (fs_try in seq(7.0, 3.4, by = -0.1)) {
    w_max = 0
    for (b in seq_along(blks)) {
        tg0 = gridExtra::tableGrob(
                  tbl[blks[[b]], ], rows = NULL,
                  theme = ttheme(base_size = fs_try, padding = unit(c(2.2, 1.6), 'mm'),
                                 colnames.style = colnames_style(size = fs_try + 0.4),
                                 tbody.style = tbody_style(size = fs_try)))
        w_max = max(w_max, sum(convertWidth(tg0$widths, 'in', valueOnly = TRUE)))
    }
    FS = fs_try
    if (w_max <= W_BLK * 0.98) break     # 0.98 は plot.margin のぶんの余裕
}
invisible(dev.off())
cat(sprintf("表のフォント: %.1f pt（最も広いブロック %.2f inch / 1ブロックの割り当て %.2f inch）\n",
            FS, w_max, W_BLK))

# ★ 表の文字色を地図と揃える（ORA hit = 橙 / DEG only = 紫 / not DEG = 黒）。
#   ggtexttable（gridExtra の tableGrob）は色を ★ 列方向（column-major）に展開する ので、
#   行ごとの色ベクトルを列数ぶん「繰り返して」渡す（rep(col, ncol) であって each= ではない。
#   each= にすると列ごとに1色になる。実測で確認）
tg = lapply(seq_along(blks), function(b) {
    ix  = blks[[b]]
    col = unname(col_cat[tbl$class[ix]])
    g = ggtexttable(tbl[ix, ], rows = NULL,
                theme = ttheme(base_size = FS, padding = unit(c(2.2, 1.6), 'mm'),
                               colnames.style = colnames_style(size = FS + 0.4, fill = 'grey92'),
                               tbody.style = tbody_style(size = FS, fill = 'white',
                                                         color = rep(col, ncol(tbl)))))
    g = g + theme(plot.margin = margin(0, 2, 2, 2))
    if (b == 1) g = g + labs(title = 'B.  gene list') +
        theme(plot.title = element_text(size = 13, face = 'bold', hjust = 0,
                                        margin = margin(b = 2)))
    g
})
TB = wrap_plots(tg, nrow = 1)

# ★ 表（ggtexttable）はパネルの中央に置かれるので、割り当てた高さが余ると
#   上下に空白ができる。行数から必要な高さを詰めて見積もる
H_map = 6.4
H_tab = 0.125 * max(sapply(blks, length)) + 0.35
# ★ tag_levels は使わない。patchwork の全サブプロット（contig パネル10個 + 表3枚）に
#   A〜M と振られてしまうため。見出しは A の図タイトルと B の表タイトルで示す。
#
# ★ wrap_elements() で包む。
#   patchwork を入れ子（P / TB）にすると ★ 内側の plot_annotation が描画されず、
#   A のタイトルが消える。包むと内側が1つの要素として確定し、タイトルが残る
PB = (wrap_elements(full = P) / TB) + plot_layout(heights = c(H_map, H_tab))

# SCALE / W_FIG は表のフォントを決めるときに上で定義してある
suppressWarnings(suppressMessages(
    ggsave(file.path(dir_gp, sprintf('genomepos_%s_%s.png', ko, SCHEME)), PB,
           width = W_FIG, height = (H_map + H_tab) * SCALE,
           dpi = 100, limitsize = FALSE)))
options(repr.plot.width = W_FIG, repr.plot.height = (H_map + H_tab) * SCALE)
suppressWarnings(print(PB))



(1) pathview - all genes of the pathway  [ko00710.A_all.png]
(2) pathview - ORA geneID only  [ko00710.A_ORA.png]


plot without title

ko00710_Carbon fixation by Calvin cycle（炭素固定(Calvin)）  contig 10 本（主 4 / 副 6）
   strain         contig  CDS 長さMb 描画幅
  Bth_VPI    NC_004663.1 1676  6.243   6.24
  Bth_VPI    NC_004703.1   36  0.033   0.60
  Bth_St8 BQNN01000001.1 1780  6.132   6.13
  Bth_St8 BQNN01000002.1   41  0.045   0.60
  Bth_St8 BQNN01000005.1    7  0.004   0.60
  Bth_St8 BQNN01000004.1    2  0.004   0.60
 Bun_8492  NZ_CP102263.1 3392  4.629   4.63
 Odo_St23 BQOE01000001.1 2801  5.647   5.65
 Odo_St23 BQOE01000002.1   38  0.065   0.60
 Odo_St23 BQOE01000003.1   12  0.032   0.60

経路の全遺伝子 40 / 図に出る 40（padj が NA で検定されなかったものも含めて表示）
凡例を出すパネル: NZ_CP102263.1（3 水準）

表: 40 行 → 2 ブロックに分けて横に並べる
表のフォント: 7.0 pt（最も広いブロック 7.27 inch / 1ブロックの割り当て 12.38 inch）


plot without title

plot without title

In [118]:
# ── 図13-4: ko00720（その他の炭素固定）  scheme A ───────────────────
# セル13（準備）を先に実行しておくこと（col_cat / strains4 / MINW / dir_pv / dir_gp を使う）
#
# ★ 3つを続けて出す
#   (1) pathview（経路の全遺伝子）… 反応と化合物名。経路の中での位置づけ
#   (2) pathview（ORA の geneID のみ）… ORA が拾った反応だけ
#   (3) ゲノム位置マップ ＋ 遺伝子表 … どこにあるか、何の遺伝子か
library(ggplot2); library(ggtext); library(ggrepel); library(patchwork); library(ggpubr)
library(png); library(grid)

ko = 'ko00720'; ko_jp = 'その他の炭素固定'; SCHEME = 'A'

# ── (1)(2) セル11 で作った pathview を再掲する ────────────────
# ★ 化合物名や反応の向きは pathview にしか無いので、地図の前に置く。
#   ORA で有意でない組み合わせ（例: ko03010 の scheme B）はファイルが無いので飛ばす
for (kind in c('all', 'ORA')) {
    f = file.path(dir_pv, sprintf('%s.%s_%s.png', ko, SCHEME, kind))
    tag = if (kind == 'all') '(1) pathview - all genes of the pathway' else
                             '(2) pathview - ORA geneID only'
    if (!file.exists(f)) {
        cat(sprintf("%s → ファイルなし（セル11 未実行、またはこのスキームの ORA で有意でない）\n", tag))
        next
    }
    # ★ セル11 の検品で落ちた図（ko00121）は中身が書かれず 0 バイトで残る。
    #   file.exists() は TRUE になるので、大きさも見ないと readPNG がエラーで止まる
    if (file.size(f) == 0) {
        cat(sprintf("%s → pathview が描けなかった（セル11 の検品で「★ 描画が壊れた」）\n", tag))
        next
    }
    im = readPNG(f); h_px = dim(im)[1]; w_px = dim(im)[2]
    W = 15; H = W * h_px / w_px
    if (H > 18) { H = 18; W = H * w_px / h_px }
    cat(sprintf("%s  [%s]\n", tag, basename(f)))
    options(repr.plot.width = W, repr.plot.height = H)
    grid.newpage(); grid.raster(im)
}

# ── (3) ゲノム位置マップ ＋ 遺伝子表 ──────────────────────────
tgt = ora$term[ora$source=='KEGG' & grepl(paste0('^',ko,'_'), ora$term)][1]
g_ora = ora %>% filter(scheme==SCHEME, source=='KEGG', term==tgt, padj<0.05) %>%
        pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
        dplyr::select(gene_id, KEGG_Pathway) %>% tidyr::separate_rows(KEGG_Pathway, sep=',') %>%
        filter(KEGG_Pathway==ko) %>% pull(gene_id) %>% unique()

# contig を CDS 数の多い順に並べる（主 → 副）
ctg = res %>% filter(biotype=='CDS', strain %in% strains4, !is.na(end)) %>%
    group_by(strain, contig) %>% summarise(CDS=dplyr::n(), 長さMb=max(end)/1e6, .groups='drop') %>%
    arrange(factor(strain, levels=strains4), desc(CDS)) %>%
    # ★ 描画幅は最小値で底上げするが、x 軸は contig の実際の長さに合わせる。
    #   → 副 contig は bp あたりの縮尺が主より大きくなり、パネルを目一杯使う
    mutate(幅 = pmax(長さMb, MINW),
           短名 = sub('^[^~]+~', '', contig),
           # ★ contig 名は空白が無いので str_wrap では折れない。
           #   パネルが狭いと重なるので、8文字ごとに改行を入れて縦に積む（文字は削らない）
           名表示 = gsub('(.{10})', '\\1\n', 短名))

cat(sprintf("%s（%s）  contig %d 本（主 4 / 副 %d）\n", tgt, ko_jp, nrow(ctg), nrow(ctg)-4))
print(ctg %>% transmute(strain, contig=短名, CDS, 長さMb=round(長さMb,3), 描画幅=round(幅,2)) %>%
      as.data.frame(), row.names=FALSE)

bg = res %>% filter(biotype=='CDS', !marker, !is.na(start)) %>%
     mutate(位置Mb = start/1e6)
fg = bg %>% filter(gene_id %in% g_all) %>%
     mutate(cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                            !is.na(padj_A) & padj_A < 0.05 ~ 'DEG only', TRUE ~ 'not DEG'),
            cat = factor(cat, levels=names(col_cat)),
            # ★ 丸の塗りに発現強度（WT 群の正規化カウント平均）を入れる。
            #   発現量は3〜4桁にわたるので log10 にする（0 があるので +1）
            発現 = log10(normA_WT + 1),
            label = ifelse(is.na(Preferred_name)|Preferred_name=='-', substr(product,1,10), Preferred_name))
cat(sprintf("\n経路の全遺伝子 %d / 図に出る %d（padj が NA で検定されなかったものも含めて表示）\n",
    length(g_all), nrow(fg)))

ylim = range(c(bg$LFC_A[bg$strain %in% strains4]), na.rm=TRUE)

# ★ 凡例は「3分類が最もそろって現れる contig」1枚からだけ出す。
#   drop=FALSE を付けても ★ データが無い水準はキーの点が描かれないため、
#   パネルごとに凡例の見た目が変わり、guides='collect' が別物と判定して2つ並ぶ
leg_i = which.max(sapply(ctg$contig, function(cc)
            length(unique(fg$cat[fg$contig == cc]))))
cat(sprintf("凡例を出すパネル: %s（%d 水準）\n",
            ctg$短名[leg_i], length(unique(fg$cat[fg$contig == ctg$contig[leg_i]]))))

subs = list()
for (i in 1:nrow(ctg)) {
    b = bg %>% filter(contig == ctg$contig[i])
    f = fg %>% filter(contig == ctg$contig[i])
    is_main = ctg$CDS[i] == max(ctg$CDS[ctg$strain == ctg$strain[i]])
    p = ggplot(b, aes(x=位置Mb, y=LFC_A)) +
        geom_hline(yintercept=0, colour='grey20', linewidth=0.3) +
        geom_hline(yintercept=c(-1,1), linetype='dashed', colour='grey65', linewidth=0.3) +
        geom_point(colour='#b9c6d4', size=0.5, alpha=0.6) +
        coord_cartesian(xlim=c(0, ctg$長さMb[i]), ylim=ylim) +
        labs(title = sprintf('%s\n%s\n%d CDS, %.3f Mb',
                             ifelse(is_main, as.character(ctg$strain[i]), ''),
                             ctg$名表示[i], ctg$CDS[i], ctg$長さMb[i]),
             x=NULL, y=if (i==1) 'log2 fold change (KO / WT), scheme A' else NULL) +
        theme_bw(base_size=10) +
        theme(plot.title=element_text(size=ifelse(is_main, 9, 5.5),
                                    face=ifelse(is_main,'bold','plain'), lineheight=1.15),
              panel.grid.minor=element_blank(),
              axis.text.y=if (i==1) element_text() else element_blank(),
              axis.ticks.y=if (i==1) element_line() else element_blank(),
              legend.position='bottom', legend.box='horizontal',
              legend.title=element_text(size=8), legend.text=element_text(size=7.5),
              legend.margin=margin(1,1,1,1),
              plot.margin=margin(2,2,0,2))
    # ★ 経路の遺伝子が無い contig（0 行）では colour の水準が1つも現れないため、
    #   scale_colour_manual が「No shared levels found」の警告を出す。
    #   図は正しく描かれる（空パネルに色が無いだけ）ので、描画時に警告だけ抑える
    # ★ shape 21（縁と塗りを別々に指定できる丸）を使う。
    #   縁 = 分類（ORA hit / DEG only / not DEG）、塗り = 発現強度
    p = p + geom_point(data=f, aes(colour=cat, fill=発現), shape=21,
                       size=3.2, stroke=1.1, show.legend = (i == leg_i))
    if (nrow(f) > 0)
        p = p +
                # ★ 遺伝子名は小さいと読めないので大きめに。
                #   重なりは ggrepel の force と box.padding で逃がす
                geom_text_repel(data=f, aes(label=label, colour=cat), size=4.4,
                                fontface='bold', max.overlaps=60, seed=1,
                                force=3, min.segment.length=0,
                                segment.size=0.3, box.padding=0.45, show.legend=FALSE)
    # ★ 色の scale は「各パネルに」付ける。
    #   patchwork の guides='collect' は収集時点のパネルごとの凡例を比べるので、
    #   後から & で当てると収集に間に合わず凡例が2つ残る。
    #   全パネルが colour の aes を持ち、drop=FALSE で水準が揃うので1つにまとまる
    subs[[i]] = p + scale_colour_manual(values=col_cat, name=NULL, drop=FALSE,
                                        guide = guide_legend(override.aes = list(
                                            fill='white', size=3.2, stroke=1.1))) +
        # ★ 凡例を下に横向きで出す。右に置くと凡例の文字幅ぶん地図が狭くなる
        scale_fill_viridis_c(option='mako', direction=-1,
                             limits=range(fg$発現, na.rm=TRUE),
                             name='log10(DESeq2 normalized count + 1), WT mean',
                             guide = guide_colourbar(direction='horizontal',
                                                     title.position='top',
                                                     barwidth=unit(5,'cm'),
                                                     barheight=unit(0.35,'cm')))
}
P = wrap_plots(subs, nrow=1, widths=ctg$幅) + plot_layout(guides='collect') +
    plot_annotation(title = sprintf('A.  %s   [scheme %s]  - position on the genome', tgt, SCHEME),
      subtitle = paste0("grey = all CDS  |  coloured = genes of this pathway  ",
                        "(<span style='color:#d95f02'>**ORA hit**</span> / ",
                        "<span style='color:#7570b3'>**DEG only**</span> / ",
                        "<span style='color:#000000'>**not DEG**</span>)<br>",
                        "**gene labels = Preferred_name** (eggNOG; the first 10 characters of *product* ",
                        "are shown when Preferred_name is absent)<br>",
                        "point border = class, **point fill = log10(DESeq2 normalized count + 1), ",
                        "WT group mean** (scheme A: one size factor set for all 4 strains pooled)<br>",
                        "all contigs shown; panel width ∝ contig length, ",
                        sprintf("small contigs get a minimum width (%.1f Mb equivalent) and are drawn at a larger bp scale  |  x axis = position within each contig (Mb)", MINW)),

      # ★ 収集した凡例の位置は patchwork 側の theme で決まる。
      #   サブプロットの theme に legend.position を書いても効かない
      theme = theme(plot.title=element_text(size=14, face='bold'),
                    plot.subtitle=element_markdown(size=9, lineheight=1.3),
                    legend.position='bottom', legend.box='horizontal',
                    legend.title=element_text(size=8.5),
                    legend.text=element_text(size=8)))
# ── 表を同じ図の中に入れる（panel B）──────────────────────────
# ★ 図だけでは locus_tag / K番号 / EC が分からないので表を下に並べる。
#   ggpubr::ggtexttable は表を ggplot として返すので patchwork に載せられる
library(ggpubr)

tbl = fg %>%
    mutate(K  = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                       function(v) paste(v, collapse=',')),
           EC2 = ifelse(is.na(EC), '-', EC)) %>%
    transmute(strain, contig = sub('^[^~]+~', '', contig),
              Mb = round(位置Mb, 3), gene_id, name = label,
              K = K, EC = EC2, product = substr(product, 1, 26),
              # ★ LFC と padj は表に入れない。
              #   LFC は上の地図の y 軸そのもの、padj は分類（class）の色で示している。
              #   列を減らしたぶん1列あたりの幅が広がり、文字が重ならない
              class = as.character(cat)) %>%
    arrange(strain, Mb)

# ★ 縦に長くなりすぎるので、行を横に分割して並べる（1ブロック最大 24 行）
NROW_BLK = 24
nblk = ceiling(nrow(tbl) / NROW_BLK)
blks = split(1:nrow(tbl), ceiling(seq_along(1:nrow(tbl)) / ceiling(nrow(tbl)/nblk)))
cat(sprintf("\n表: %d 行 → %d ブロックに分けて横に並べる\n", nrow(tbl), length(blks)))

# ★ この図（地図＋表）だけ縦横 1.125 倍で出す（pathview の再掲は等倍のまま）
SCALE = 1.125
W_FIG = 22 * SCALE

# ★ 表のフォントは経路ごとに自動で決める。
#   列の幅は中身の文字数で決まるので、ko03010（112 遺伝子）のように行が多い経路では
#   ブロック数が増え、1ブロックに割ける幅が狭くなる。
#   固定の 6.2 pt のままだと表が割り当て幅をはみ出し、右端の class 列が切れてしまう。
#   → 実際に表を組んで幅を測り、収まる一番大きいフォントを選ぶ
#
# ★ 文字の幅は「フォントサイズに比例」しない。
#   小さいサイズでは hinting でカクカク変わる（実測: 5.5 pt で 4.75 inch、
#   5.9 pt で 5.04 inch）ので、計算で逆算せず 0.1 pt ずつ下げて実測で確かめる。
# ★ 測るときの device を間違えないこと。文字幅は device のフォント metrics で決まる。
#   実測（同じ表・同じ 5.9 pt）: pdf 4.70 / cairo png 4.81 / ragg 5.04 inch。
#   ggplot2 3.5 の ggsave は ragg があれば ragg を使うので、一番広く出る ragg で測る。
#   （Jupyter の画面表示は cairo png なので、ragg に合わせておけばどちらでも溢れない）
W_BLK = W_FIG / length(blks)             # 1ブロックに割ける幅 (inch)
ragg::agg_png(tempfile(fileext = '.png'), width = 1000, height = 1000, res = 100)
for (fs_try in seq(7.0, 3.4, by = -0.1)) {
    w_max = 0
    for (b in seq_along(blks)) {
        tg0 = gridExtra::tableGrob(
                  tbl[blks[[b]], ], rows = NULL,
                  theme = ttheme(base_size = fs_try, padding = unit(c(2.2, 1.6), 'mm'),
                                 colnames.style = colnames_style(size = fs_try + 0.4),
                                 tbody.style = tbody_style(size = fs_try)))
        w_max = max(w_max, sum(convertWidth(tg0$widths, 'in', valueOnly = TRUE)))
    }
    FS = fs_try
    if (w_max <= W_BLK * 0.98) break     # 0.98 は plot.margin のぶんの余裕
}
invisible(dev.off())
cat(sprintf("表のフォント: %.1f pt（最も広いブロック %.2f inch / 1ブロックの割り当て %.2f inch）\n",
            FS, w_max, W_BLK))

# ★ 表の文字色を地図と揃える（ORA hit = 橙 / DEG only = 紫 / not DEG = 黒）。
#   ggtexttable（gridExtra の tableGrob）は色を ★ 列方向（column-major）に展開する ので、
#   行ごとの色ベクトルを列数ぶん「繰り返して」渡す（rep(col, ncol) であって each= ではない。
#   each= にすると列ごとに1色になる。実測で確認）
tg = lapply(seq_along(blks), function(b) {
    ix  = blks[[b]]
    col = unname(col_cat[tbl$class[ix]])
    g = ggtexttable(tbl[ix, ], rows = NULL,
                theme = ttheme(base_size = FS, padding = unit(c(2.2, 1.6), 'mm'),
                               colnames.style = colnames_style(size = FS + 0.4, fill = 'grey92'),
                               tbody.style = tbody_style(size = FS, fill = 'white',
                                                         color = rep(col, ncol(tbl)))))
    g = g + theme(plot.margin = margin(0, 2, 2, 2))
    if (b == 1) g = g + labs(title = 'B.  gene list') +
        theme(plot.title = element_text(size = 13, face = 'bold', hjust = 0,
                                        margin = margin(b = 2)))
    g
})
TB = wrap_plots(tg, nrow = 1)

# ★ 表（ggtexttable）はパネルの中央に置かれるので、割り当てた高さが余ると
#   上下に空白ができる。行数から必要な高さを詰めて見積もる
H_map = 6.4
H_tab = 0.125 * max(sapply(blks, length)) + 0.35
# ★ tag_levels は使わない。patchwork の全サブプロット（contig パネル10個 + 表3枚）に
#   A〜M と振られてしまうため。見出しは A の図タイトルと B の表タイトルで示す。
#
# ★ wrap_elements() で包む。
#   patchwork を入れ子（P / TB）にすると ★ 内側の plot_annotation が描画されず、
#   A のタイトルが消える。包むと内側が1つの要素として確定し、タイトルが残る
PB = (wrap_elements(full = P) / TB) + plot_layout(heights = c(H_map, H_tab))

# SCALE / W_FIG は表のフォントを決めるときに上で定義してある
suppressWarnings(suppressMessages(
    ggsave(file.path(dir_gp, sprintf('genomepos_%s_%s.png', ko, SCHEME)), PB,
           width = W_FIG, height = (H_map + H_tab) * SCALE,
           dpi = 100, limitsize = FALSE)))
options(repr.plot.width = W_FIG, repr.plot.height = (H_map + H_tab) * SCALE)
suppressWarnings(print(PB))



(1) pathview - all genes of the pathway  [ko00720.A_all.png]
(2) pathview - ORA geneID only  [ko00720.A_ORA.png]


plot without title

ko00720_Other carbon fixation pathways（その他の炭素固定）  contig 10 本（主 4 / 副 6）
   strain         contig  CDS 長さMb 描画幅
  Bth_VPI    NC_004663.1 1676  6.243   6.24
  Bth_VPI    NC_004703.1   36  0.033   0.60
  Bth_St8 BQNN01000001.1 1780  6.132   6.13
  Bth_St8 BQNN01000002.1   41  0.045   0.60
  Bth_St8 BQNN01000005.1    7  0.004   0.60
  Bth_St8 BQNN01000004.1    2  0.004   0.60
 Bun_8492  NZ_CP102263.1 3392  4.629   4.63
 Odo_St23 BQOE01000001.1 2801  5.647   5.65
 Odo_St23 BQOE01000002.1   38  0.065   0.60
 Odo_St23 BQOE01000003.1   12  0.032   0.60

経路の全遺伝子 81 / 図に出る 81（padj が NA で検定されなかったものも含めて表示）
凡例を出すパネル: NC_004663.1（3 水準）

表: 81 行 → 4 ブロックに分けて横に並べる
表のフォント: 5.4 pt（最も広いブロック 6.02 inch / 1ブロックの割り当て 6.19 inch）


plot without title

plot without title

In [119]:
# ── 図13-5: ko00120（一次胆汁酸）  scheme A ───────────────────
# セル13（準備）を先に実行しておくこと（col_cat / strains4 / MINW / dir_pv / dir_gp を使う）
#
# ★ 3つを続けて出す
#   (1) pathview（経路の全遺伝子）… 反応と化合物名。経路の中での位置づけ
#   (2) pathview（ORA の geneID のみ）… ORA が拾った反応だけ
#   (3) ゲノム位置マップ ＋ 遺伝子表 … どこにあるか、何の遺伝子か
library(ggplot2); library(ggtext); library(ggrepel); library(patchwork); library(ggpubr)
library(png); library(grid)

ko = 'ko00120'; ko_jp = '一次胆汁酸'; SCHEME = 'A'

# ── (1)(2) セル11 で作った pathview を再掲する ────────────────
# ★ 化合物名や反応の向きは pathview にしか無いので、地図の前に置く。
#   ORA で有意でない組み合わせ（例: ko03010 の scheme B）はファイルが無いので飛ばす
for (kind in c('all', 'ORA')) {
    f = file.path(dir_pv, sprintf('%s.%s_%s.png', ko, SCHEME, kind))
    tag = if (kind == 'all') '(1) pathview - all genes of the pathway' else
                             '(2) pathview - ORA geneID only'
    if (!file.exists(f)) {
        cat(sprintf("%s → ファイルなし（セル11 未実行、またはこのスキームの ORA で有意でない）\n", tag))
        next
    }
    # ★ セル11 の検品で落ちた図（ko00121）は中身が書かれず 0 バイトで残る。
    #   file.exists() は TRUE になるので、大きさも見ないと readPNG がエラーで止まる
    if (file.size(f) == 0) {
        cat(sprintf("%s → pathview が描けなかった（セル11 の検品で「★ 描画が壊れた」）\n", tag))
        next
    }
    im = readPNG(f); h_px = dim(im)[1]; w_px = dim(im)[2]
    W = 15; H = W * h_px / w_px
    if (H > 18) { H = 18; W = H * w_px / h_px }
    cat(sprintf("%s  [%s]\n", tag, basename(f)))
    options(repr.plot.width = W, repr.plot.height = H)
    grid.newpage(); grid.raster(im)
}

# ── (3) ゲノム位置マップ ＋ 遺伝子表 ──────────────────────────
tgt = ora$term[ora$source=='KEGG' & grepl(paste0('^',ko,'_'), ora$term)][1]
g_ora = ora %>% filter(scheme==SCHEME, source=='KEGG', term==tgt, padj<0.05) %>%
        pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
        dplyr::select(gene_id, KEGG_Pathway) %>% tidyr::separate_rows(KEGG_Pathway, sep=',') %>%
        filter(KEGG_Pathway==ko) %>% pull(gene_id) %>% unique()

# contig を CDS 数の多い順に並べる（主 → 副）
ctg = res %>% filter(biotype=='CDS', strain %in% strains4, !is.na(end)) %>%
    group_by(strain, contig) %>% summarise(CDS=dplyr::n(), 長さMb=max(end)/1e6, .groups='drop') %>%
    arrange(factor(strain, levels=strains4), desc(CDS)) %>%
    # ★ 描画幅は最小値で底上げするが、x 軸は contig の実際の長さに合わせる。
    #   → 副 contig は bp あたりの縮尺が主より大きくなり、パネルを目一杯使う
    mutate(幅 = pmax(長さMb, MINW),
           短名 = sub('^[^~]+~', '', contig),
           # ★ contig 名は空白が無いので str_wrap では折れない。
           #   パネルが狭いと重なるので、8文字ごとに改行を入れて縦に積む（文字は削らない）
           名表示 = gsub('(.{10})', '\\1\n', 短名))

cat(sprintf("%s（%s）  contig %d 本（主 4 / 副 %d）\n", tgt, ko_jp, nrow(ctg), nrow(ctg)-4))
print(ctg %>% transmute(strain, contig=短名, CDS, 長さMb=round(長さMb,3), 描画幅=round(幅,2)) %>%
      as.data.frame(), row.names=FALSE)

bg = res %>% filter(biotype=='CDS', !marker, !is.na(start)) %>%
     mutate(位置Mb = start/1e6)
fg = bg %>% filter(gene_id %in% g_all) %>%
     mutate(cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                            !is.na(padj_A) & padj_A < 0.05 ~ 'DEG only', TRUE ~ 'not DEG'),
            cat = factor(cat, levels=names(col_cat)),
            # ★ 丸の塗りに発現強度（WT 群の正規化カウント平均）を入れる。
            #   発現量は3〜4桁にわたるので log10 にする（0 があるので +1）
            発現 = log10(normA_WT + 1),
            label = ifelse(is.na(Preferred_name)|Preferred_name=='-', substr(product,1,10), Preferred_name))
cat(sprintf("\n経路の全遺伝子 %d / 図に出る %d（padj が NA で検定されなかったものも含めて表示）\n",
    length(g_all), nrow(fg)))

ylim = range(c(bg$LFC_A[bg$strain %in% strains4]), na.rm=TRUE)

# ★ 凡例は「3分類が最もそろって現れる contig」1枚からだけ出す。
#   drop=FALSE を付けても ★ データが無い水準はキーの点が描かれないため、
#   パネルごとに凡例の見た目が変わり、guides='collect' が別物と判定して2つ並ぶ
leg_i = which.max(sapply(ctg$contig, function(cc)
            length(unique(fg$cat[fg$contig == cc]))))
cat(sprintf("凡例を出すパネル: %s（%d 水準）\n",
            ctg$短名[leg_i], length(unique(fg$cat[fg$contig == ctg$contig[leg_i]]))))

subs = list()
for (i in 1:nrow(ctg)) {
    b = bg %>% filter(contig == ctg$contig[i])
    f = fg %>% filter(contig == ctg$contig[i])
    is_main = ctg$CDS[i] == max(ctg$CDS[ctg$strain == ctg$strain[i]])
    p = ggplot(b, aes(x=位置Mb, y=LFC_A)) +
        geom_hline(yintercept=0, colour='grey20', linewidth=0.3) +
        geom_hline(yintercept=c(-1,1), linetype='dashed', colour='grey65', linewidth=0.3) +
        geom_point(colour='#b9c6d4', size=0.5, alpha=0.6) +
        coord_cartesian(xlim=c(0, ctg$長さMb[i]), ylim=ylim) +
        labs(title = sprintf('%s\n%s\n%d CDS, %.3f Mb',
                             ifelse(is_main, as.character(ctg$strain[i]), ''),
                             ctg$名表示[i], ctg$CDS[i], ctg$長さMb[i]),
             x=NULL, y=if (i==1) 'log2 fold change (KO / WT), scheme A' else NULL) +
        theme_bw(base_size=10) +
        theme(plot.title=element_text(size=ifelse(is_main, 9, 5.5),
                                    face=ifelse(is_main,'bold','plain'), lineheight=1.15),
              panel.grid.minor=element_blank(),
              axis.text.y=if (i==1) element_text() else element_blank(),
              axis.ticks.y=if (i==1) element_line() else element_blank(),
              legend.position='bottom', legend.box='horizontal',
              legend.title=element_text(size=8), legend.text=element_text(size=7.5),
              legend.margin=margin(1,1,1,1),
              plot.margin=margin(2,2,0,2))
    # ★ 経路の遺伝子が無い contig（0 行）では colour の水準が1つも現れないため、
    #   scale_colour_manual が「No shared levels found」の警告を出す。
    #   図は正しく描かれる（空パネルに色が無いだけ）ので、描画時に警告だけ抑える
    # ★ shape 21（縁と塗りを別々に指定できる丸）を使う。
    #   縁 = 分類（ORA hit / DEG only / not DEG）、塗り = 発現強度
    p = p + geom_point(data=f, aes(colour=cat, fill=発現), shape=21,
                       size=3.2, stroke=1.1, show.legend = (i == leg_i))
    if (nrow(f) > 0)
        p = p +
                # ★ 遺伝子名は小さいと読めないので大きめに。
                #   重なりは ggrepel の force と box.padding で逃がす
                geom_text_repel(data=f, aes(label=label, colour=cat), size=4.4,
                                fontface='bold', max.overlaps=60, seed=1,
                                force=3, min.segment.length=0,
                                segment.size=0.3, box.padding=0.45, show.legend=FALSE)
    # ★ 色の scale は「各パネルに」付ける。
    #   patchwork の guides='collect' は収集時点のパネルごとの凡例を比べるので、
    #   後から & で当てると収集に間に合わず凡例が2つ残る。
    #   全パネルが colour の aes を持ち、drop=FALSE で水準が揃うので1つにまとまる
    subs[[i]] = p + scale_colour_manual(values=col_cat, name=NULL, drop=FALSE,
                                        guide = guide_legend(override.aes = list(
                                            fill='white', size=3.2, stroke=1.1))) +
        # ★ 凡例を下に横向きで出す。右に置くと凡例の文字幅ぶん地図が狭くなる
        scale_fill_viridis_c(option='mako', direction=-1,
                             limits=range(fg$発現, na.rm=TRUE),
                             name='log10(DESeq2 normalized count + 1), WT mean',
                             guide = guide_colourbar(direction='horizontal',
                                                     title.position='top',
                                                     barwidth=unit(5,'cm'),
                                                     barheight=unit(0.35,'cm')))
}
P = wrap_plots(subs, nrow=1, widths=ctg$幅) + plot_layout(guides='collect') +
    plot_annotation(title = sprintf('A.  %s   [scheme %s]  - position on the genome', tgt, SCHEME),
      subtitle = paste0("grey = all CDS  |  coloured = genes of this pathway  ",
                        "(<span style='color:#d95f02'>**ORA hit**</span> / ",
                        "<span style='color:#7570b3'>**DEG only**</span> / ",
                        "<span style='color:#000000'>**not DEG**</span>)<br>",
                        "**gene labels = Preferred_name** (eggNOG; the first 10 characters of *product* ",
                        "are shown when Preferred_name is absent)<br>",
                        "point border = class, **point fill = log10(DESeq2 normalized count + 1), ",
                        "WT group mean** (scheme A: one size factor set for all 4 strains pooled)<br>",
                        "all contigs shown; panel width ∝ contig length, ",
                        sprintf("small contigs get a minimum width (%.1f Mb equivalent) and are drawn at a larger bp scale  |  x axis = position within each contig (Mb)", MINW)),

      # ★ 収集した凡例の位置は patchwork 側の theme で決まる。
      #   サブプロットの theme に legend.position を書いても効かない
      theme = theme(plot.title=element_text(size=14, face='bold'),
                    plot.subtitle=element_markdown(size=9, lineheight=1.3),
                    legend.position='bottom', legend.box='horizontal',
                    legend.title=element_text(size=8.5),
                    legend.text=element_text(size=8)))
# ── 表を同じ図の中に入れる（panel B）──────────────────────────
# ★ 図だけでは locus_tag / K番号 / EC が分からないので表を下に並べる。
#   ggpubr::ggtexttable は表を ggplot として返すので patchwork に載せられる
library(ggpubr)

tbl = fg %>%
    mutate(K  = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                       function(v) paste(v, collapse=',')),
           EC2 = ifelse(is.na(EC), '-', EC)) %>%
    transmute(strain, contig = sub('^[^~]+~', '', contig),
              Mb = round(位置Mb, 3), gene_id, name = label,
              K = K, EC = EC2, product = substr(product, 1, 26),
              # ★ LFC と padj は表に入れない。
              #   LFC は上の地図の y 軸そのもの、padj は分類（class）の色で示している。
              #   列を減らしたぶん1列あたりの幅が広がり、文字が重ならない
              class = as.character(cat)) %>%
    arrange(strain, Mb)

# ★ 縦に長くなりすぎるので、行を横に分割して並べる（1ブロック最大 24 行）
NROW_BLK = 24
nblk = ceiling(nrow(tbl) / NROW_BLK)
blks = split(1:nrow(tbl), ceiling(seq_along(1:nrow(tbl)) / ceiling(nrow(tbl)/nblk)))
cat(sprintf("\n表: %d 行 → %d ブロックに分けて横に並べる\n", nrow(tbl), length(blks)))

# ★ この図（地図＋表）だけ縦横 1.125 倍で出す（pathview の再掲は等倍のまま）
SCALE = 1.125
W_FIG = 22 * SCALE

# ★ 表のフォントは経路ごとに自動で決める。
#   列の幅は中身の文字数で決まるので、ko03010（112 遺伝子）のように行が多い経路では
#   ブロック数が増え、1ブロックに割ける幅が狭くなる。
#   固定の 6.2 pt のままだと表が割り当て幅をはみ出し、右端の class 列が切れてしまう。
#   → 実際に表を組んで幅を測り、収まる一番大きいフォントを選ぶ
#
# ★ 文字の幅は「フォントサイズに比例」しない。
#   小さいサイズでは hinting でカクカク変わる（実測: 5.5 pt で 4.75 inch、
#   5.9 pt で 5.04 inch）ので、計算で逆算せず 0.1 pt ずつ下げて実測で確かめる。
# ★ 測るときの device を間違えないこと。文字幅は device のフォント metrics で決まる。
#   実測（同じ表・同じ 5.9 pt）: pdf 4.70 / cairo png 4.81 / ragg 5.04 inch。
#   ggplot2 3.5 の ggsave は ragg があれば ragg を使うので、一番広く出る ragg で測る。
#   （Jupyter の画面表示は cairo png なので、ragg に合わせておけばどちらでも溢れない）
W_BLK = W_FIG / length(blks)             # 1ブロックに割ける幅 (inch)
ragg::agg_png(tempfile(fileext = '.png'), width = 1000, height = 1000, res = 100)
for (fs_try in seq(7.0, 3.4, by = -0.1)) {
    w_max = 0
    for (b in seq_along(blks)) {
        tg0 = gridExtra::tableGrob(
                  tbl[blks[[b]], ], rows = NULL,
                  theme = ttheme(base_size = fs_try, padding = unit(c(2.2, 1.6), 'mm'),
                                 colnames.style = colnames_style(size = fs_try + 0.4),
                                 tbody.style = tbody_style(size = fs_try)))
        w_max = max(w_max, sum(convertWidth(tg0$widths, 'in', valueOnly = TRUE)))
    }
    FS = fs_try
    if (w_max <= W_BLK * 0.98) break     # 0.98 は plot.margin のぶんの余裕
}
invisible(dev.off())
cat(sprintf("表のフォント: %.1f pt（最も広いブロック %.2f inch / 1ブロックの割り当て %.2f inch）\n",
            FS, w_max, W_BLK))

# ★ 表の文字色を地図と揃える（ORA hit = 橙 / DEG only = 紫 / not DEG = 黒）。
#   ggtexttable（gridExtra の tableGrob）は色を ★ 列方向（column-major）に展開する ので、
#   行ごとの色ベクトルを列数ぶん「繰り返して」渡す（rep(col, ncol) であって each= ではない。
#   each= にすると列ごとに1色になる。実測で確認）
tg = lapply(seq_along(blks), function(b) {
    ix  = blks[[b]]
    col = unname(col_cat[tbl$class[ix]])
    g = ggtexttable(tbl[ix, ], rows = NULL,
                theme = ttheme(base_size = FS, padding = unit(c(2.2, 1.6), 'mm'),
                               colnames.style = colnames_style(size = FS + 0.4, fill = 'grey92'),
                               tbody.style = tbody_style(size = FS, fill = 'white',
                                                         color = rep(col, ncol(tbl)))))
    g = g + theme(plot.margin = margin(0, 2, 2, 2))
    if (b == 1) g = g + labs(title = 'B.  gene list') +
        theme(plot.title = element_text(size = 13, face = 'bold', hjust = 0,
                                        margin = margin(b = 2)))
    g
})
TB = wrap_plots(tg, nrow = 1)

# ★ 表（ggtexttable）はパネルの中央に置かれるので、割り当てた高さが余ると
#   上下に空白ができる。行数から必要な高さを詰めて見積もる
H_map = 6.4
H_tab = 0.125 * max(sapply(blks, length)) + 0.35
# ★ tag_levels は使わない。patchwork の全サブプロット（contig パネル10個 + 表3枚）に
#   A〜M と振られてしまうため。見出しは A の図タイトルと B の表タイトルで示す。
#
# ★ wrap_elements() で包む。
#   patchwork を入れ子（P / TB）にすると ★ 内側の plot_annotation が描画されず、
#   A のタイトルが消える。包むと内側が1つの要素として確定し、タイトルが残る
PB = (wrap_elements(full = P) / TB) + plot_layout(heights = c(H_map, H_tab))

# SCALE / W_FIG は表のフォントを決めるときに上で定義してある
suppressWarnings(suppressMessages(
    ggsave(file.path(dir_gp, sprintf('genomepos_%s_%s.png', ko, SCHEME)), PB,
           width = W_FIG, height = (H_map + H_tab) * SCALE,
           dpi = 100, limitsize = FALSE)))
options(repr.plot.width = W_FIG, repr.plot.height = (H_map + H_tab) * SCALE)
suppressWarnings(print(PB))



(1) pathview - all genes of the pathway  [ko00120.A_all.png]
(2) pathview - ORA geneID only  [ko00120.A_ORA.png]


plot without title

ko00120_Primary bile acid biosynthesis（一次胆汁酸）  contig 10 本（主 4 / 副 6）
   strain         contig  CDS 長さMb 描画幅
  Bth_VPI    NC_004663.1 1676  6.243   6.24
  Bth_VPI    NC_004703.1   36  0.033   0.60
  Bth_St8 BQNN01000001.1 1780  6.132   6.13
  Bth_St8 BQNN01000002.1   41  0.045   0.60
  Bth_St8 BQNN01000005.1    7  0.004   0.60
  Bth_St8 BQNN01000004.1    2  0.004   0.60
 Bun_8492  NZ_CP102263.1 3392  4.629   4.63
 Odo_St23 BQOE01000001.1 2801  5.647   5.65
 Odo_St23 BQOE01000002.1   38  0.065   0.60
 Odo_St23 BQOE01000003.1   12  0.032   0.60

経路の全遺伝子 5 / 図に出る 5（padj が NA で検定されなかったものも含めて表示）
凡例を出すパネル: NZ_CP102263.1（2 水準）

表: 5 行 → 1 ブロックに分けて横に並べる
表のフォント: 7.0 pt（最も広いブロック 6.26 inch / 1ブロックの割り当て 24.75 inch）


plot without title

plot without title

In [120]:
# ── 図13-6: ko00121（二次胆汁酸）  scheme A ───────────────────
# セル13（準備）を先に実行しておくこと（col_cat / strains4 / MINW / dir_pv / dir_gp を使う）
#
# ★ 3つを続けて出す
#   (1) pathview（経路の全遺伝子）… 反応と化合物名。経路の中での位置づけ
#   (2) pathview（ORA の geneID のみ）… ORA が拾った反応だけ
#   (3) ゲノム位置マップ ＋ 遺伝子表 … どこにあるか、何の遺伝子か
library(ggplot2); library(ggtext); library(ggrepel); library(patchwork); library(ggpubr)
library(png); library(grid)

ko = 'ko00121'; ko_jp = '二次胆汁酸'; SCHEME = 'A'

# ── (1)(2) セル11 で作った pathview を再掲する ────────────────
# ★ 化合物名や反応の向きは pathview にしか無いので、地図の前に置く。
#   ORA で有意でない組み合わせ（例: ko03010 の scheme B）はファイルが無いので飛ばす
for (kind in c('all', 'ORA')) {
    f = file.path(dir_pv, sprintf('%s.%s_%s.png', ko, SCHEME, kind))
    tag = if (kind == 'all') '(1) pathview - all genes of the pathway' else
                             '(2) pathview - ORA geneID only'
    if (!file.exists(f)) {
        cat(sprintf("%s → ファイルなし（セル11 未実行、またはこのスキームの ORA で有意でない）\n", tag))
        next
    }
    # ★ セル11 の検品で落ちた図（ko00121）は中身が書かれず 0 バイトで残る。
    #   file.exists() は TRUE になるので、大きさも見ないと readPNG がエラーで止まる
    if (file.size(f) == 0) {
        cat(sprintf("%s → pathview が描けなかった（セル11 の検品で「★ 描画が壊れた」）\n", tag))
        next
    }
    im = readPNG(f); h_px = dim(im)[1]; w_px = dim(im)[2]
    W = 15; H = W * h_px / w_px
    if (H > 18) { H = 18; W = H * w_px / h_px }
    cat(sprintf("%s  [%s]\n", tag, basename(f)))
    options(repr.plot.width = W, repr.plot.height = H)
    grid.newpage(); grid.raster(im)
}

# ── (3) ゲノム位置マップ ＋ 遺伝子表 ──────────────────────────
tgt = ora$term[ora$source=='KEGG' & grepl(paste0('^',ko,'_'), ora$term)][1]
g_ora = ora %>% filter(scheme==SCHEME, source=='KEGG', term==tgt, padj<0.05) %>%
        pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
        dplyr::select(gene_id, KEGG_Pathway) %>% tidyr::separate_rows(KEGG_Pathway, sep=',') %>%
        filter(KEGG_Pathway==ko) %>% pull(gene_id) %>% unique()

# contig を CDS 数の多い順に並べる（主 → 副）
ctg = res %>% filter(biotype=='CDS', strain %in% strains4, !is.na(end)) %>%
    group_by(strain, contig) %>% summarise(CDS=dplyr::n(), 長さMb=max(end)/1e6, .groups='drop') %>%
    arrange(factor(strain, levels=strains4), desc(CDS)) %>%
    # ★ 描画幅は最小値で底上げするが、x 軸は contig の実際の長さに合わせる。
    #   → 副 contig は bp あたりの縮尺が主より大きくなり、パネルを目一杯使う
    mutate(幅 = pmax(長さMb, MINW),
           短名 = sub('^[^~]+~', '', contig),
           # ★ contig 名は空白が無いので str_wrap では折れない。
           #   パネルが狭いと重なるので、8文字ごとに改行を入れて縦に積む（文字は削らない）
           名表示 = gsub('(.{10})', '\\1\n', 短名))

cat(sprintf("%s（%s）  contig %d 本（主 4 / 副 %d）\n", tgt, ko_jp, nrow(ctg), nrow(ctg)-4))
print(ctg %>% transmute(strain, contig=短名, CDS, 長さMb=round(長さMb,3), 描画幅=round(幅,2)) %>%
      as.data.frame(), row.names=FALSE)

bg = res %>% filter(biotype=='CDS', !marker, !is.na(start)) %>%
     mutate(位置Mb = start/1e6)
fg = bg %>% filter(gene_id %in% g_all) %>%
     mutate(cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                            !is.na(padj_A) & padj_A < 0.05 ~ 'DEG only', TRUE ~ 'not DEG'),
            cat = factor(cat, levels=names(col_cat)),
            # ★ 丸の塗りに発現強度（WT 群の正規化カウント平均）を入れる。
            #   発現量は3〜4桁にわたるので log10 にする（0 があるので +1）
            発現 = log10(normA_WT + 1),
            label = ifelse(is.na(Preferred_name)|Preferred_name=='-', substr(product,1,10), Preferred_name))
cat(sprintf("\n経路の全遺伝子 %d / 図に出る %d（padj が NA で検定されなかったものも含めて表示）\n",
    length(g_all), nrow(fg)))

ylim = range(c(bg$LFC_A[bg$strain %in% strains4]), na.rm=TRUE)

# ★ 凡例は「3分類が最もそろって現れる contig」1枚からだけ出す。
#   drop=FALSE を付けても ★ データが無い水準はキーの点が描かれないため、
#   パネルごとに凡例の見た目が変わり、guides='collect' が別物と判定して2つ並ぶ
leg_i = which.max(sapply(ctg$contig, function(cc)
            length(unique(fg$cat[fg$contig == cc]))))
cat(sprintf("凡例を出すパネル: %s（%d 水準）\n",
            ctg$短名[leg_i], length(unique(fg$cat[fg$contig == ctg$contig[leg_i]]))))

subs = list()
for (i in 1:nrow(ctg)) {
    b = bg %>% filter(contig == ctg$contig[i])
    f = fg %>% filter(contig == ctg$contig[i])
    is_main = ctg$CDS[i] == max(ctg$CDS[ctg$strain == ctg$strain[i]])
    p = ggplot(b, aes(x=位置Mb, y=LFC_A)) +
        geom_hline(yintercept=0, colour='grey20', linewidth=0.3) +
        geom_hline(yintercept=c(-1,1), linetype='dashed', colour='grey65', linewidth=0.3) +
        geom_point(colour='#b9c6d4', size=0.5, alpha=0.6) +
        coord_cartesian(xlim=c(0, ctg$長さMb[i]), ylim=ylim) +
        labs(title = sprintf('%s\n%s\n%d CDS, %.3f Mb',
                             ifelse(is_main, as.character(ctg$strain[i]), ''),
                             ctg$名表示[i], ctg$CDS[i], ctg$長さMb[i]),
             x=NULL, y=if (i==1) 'log2 fold change (KO / WT), scheme A' else NULL) +
        theme_bw(base_size=10) +
        theme(plot.title=element_text(size=ifelse(is_main, 9, 5.5),
                                    face=ifelse(is_main,'bold','plain'), lineheight=1.15),
              panel.grid.minor=element_blank(),
              axis.text.y=if (i==1) element_text() else element_blank(),
              axis.ticks.y=if (i==1) element_line() else element_blank(),
              legend.position='bottom', legend.box='horizontal',
              legend.title=element_text(size=8), legend.text=element_text(size=7.5),
              legend.margin=margin(1,1,1,1),
              plot.margin=margin(2,2,0,2))
    # ★ 経路の遺伝子が無い contig（0 行）では colour の水準が1つも現れないため、
    #   scale_colour_manual が「No shared levels found」の警告を出す。
    #   図は正しく描かれる（空パネルに色が無いだけ）ので、描画時に警告だけ抑える
    # ★ shape 21（縁と塗りを別々に指定できる丸）を使う。
    #   縁 = 分類（ORA hit / DEG only / not DEG）、塗り = 発現強度
    p = p + geom_point(data=f, aes(colour=cat, fill=発現), shape=21,
                       size=3.2, stroke=1.1, show.legend = (i == leg_i))
    if (nrow(f) > 0)
        p = p +
                # ★ 遺伝子名は小さいと読めないので大きめに。
                #   重なりは ggrepel の force と box.padding で逃がす
                geom_text_repel(data=f, aes(label=label, colour=cat), size=4.4,
                                fontface='bold', max.overlaps=60, seed=1,
                                force=3, min.segment.length=0,
                                segment.size=0.3, box.padding=0.45, show.legend=FALSE)
    # ★ 色の scale は「各パネルに」付ける。
    #   patchwork の guides='collect' は収集時点のパネルごとの凡例を比べるので、
    #   後から & で当てると収集に間に合わず凡例が2つ残る。
    #   全パネルが colour の aes を持ち、drop=FALSE で水準が揃うので1つにまとまる
    subs[[i]] = p + scale_colour_manual(values=col_cat, name=NULL, drop=FALSE,
                                        guide = guide_legend(override.aes = list(
                                            fill='white', size=3.2, stroke=1.1))) +
        # ★ 凡例を下に横向きで出す。右に置くと凡例の文字幅ぶん地図が狭くなる
        scale_fill_viridis_c(option='mako', direction=-1,
                             limits=range(fg$発現, na.rm=TRUE),
                             name='log10(DESeq2 normalized count + 1), WT mean',
                             guide = guide_colourbar(direction='horizontal',
                                                     title.position='top',
                                                     barwidth=unit(5,'cm'),
                                                     barheight=unit(0.35,'cm')))
}
P = wrap_plots(subs, nrow=1, widths=ctg$幅) + plot_layout(guides='collect') +
    plot_annotation(title = sprintf('A.  %s   [scheme %s]  - position on the genome', tgt, SCHEME),
      subtitle = paste0("grey = all CDS  |  coloured = genes of this pathway  ",
                        "(<span style='color:#d95f02'>**ORA hit**</span> / ",
                        "<span style='color:#7570b3'>**DEG only**</span> / ",
                        "<span style='color:#000000'>**not DEG**</span>)<br>",
                        "**gene labels = Preferred_name** (eggNOG; the first 10 characters of *product* ",
                        "are shown when Preferred_name is absent)<br>",
                        "point border = class, **point fill = log10(DESeq2 normalized count + 1), ",
                        "WT group mean** (scheme A: one size factor set for all 4 strains pooled)<br>",
                        "all contigs shown; panel width ∝ contig length, ",
                        sprintf("small contigs get a minimum width (%.1f Mb equivalent) and are drawn at a larger bp scale  |  x axis = position within each contig (Mb)", MINW)),

      # ★ 収集した凡例の位置は patchwork 側の theme で決まる。
      #   サブプロットの theme に legend.position を書いても効かない
      theme = theme(plot.title=element_text(size=14, face='bold'),
                    plot.subtitle=element_markdown(size=9, lineheight=1.3),
                    legend.position='bottom', legend.box='horizontal',
                    legend.title=element_text(size=8.5),
                    legend.text=element_text(size=8)))
# ── 表を同じ図の中に入れる（panel B）──────────────────────────
# ★ 図だけでは locus_tag / K番号 / EC が分からないので表を下に並べる。
#   ggpubr::ggtexttable は表を ggplot として返すので patchwork に載せられる
library(ggpubr)

tbl = fg %>%
    mutate(K  = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                       function(v) paste(v, collapse=',')),
           EC2 = ifelse(is.na(EC), '-', EC)) %>%
    transmute(strain, contig = sub('^[^~]+~', '', contig),
              Mb = round(位置Mb, 3), gene_id, name = label,
              K = K, EC = EC2, product = substr(product, 1, 26),
              # ★ LFC と padj は表に入れない。
              #   LFC は上の地図の y 軸そのもの、padj は分類（class）の色で示している。
              #   列を減らしたぶん1列あたりの幅が広がり、文字が重ならない
              class = as.character(cat)) %>%
    arrange(strain, Mb)

# ★ 縦に長くなりすぎるので、行を横に分割して並べる（1ブロック最大 24 行）
NROW_BLK = 24
nblk = ceiling(nrow(tbl) / NROW_BLK)
blks = split(1:nrow(tbl), ceiling(seq_along(1:nrow(tbl)) / ceiling(nrow(tbl)/nblk)))
cat(sprintf("\n表: %d 行 → %d ブロックに分けて横に並べる\n", nrow(tbl), length(blks)))

# ★ この図（地図＋表）だけ縦横 1.125 倍で出す（pathview の再掲は等倍のまま）
SCALE = 1.125
W_FIG = 22 * SCALE

# ★ 表のフォントは経路ごとに自動で決める。
#   列の幅は中身の文字数で決まるので、ko03010（112 遺伝子）のように行が多い経路では
#   ブロック数が増え、1ブロックに割ける幅が狭くなる。
#   固定の 6.2 pt のままだと表が割り当て幅をはみ出し、右端の class 列が切れてしまう。
#   → 実際に表を組んで幅を測り、収まる一番大きいフォントを選ぶ
#
# ★ 文字の幅は「フォントサイズに比例」しない。
#   小さいサイズでは hinting でカクカク変わる（実測: 5.5 pt で 4.75 inch、
#   5.9 pt で 5.04 inch）ので、計算で逆算せず 0.1 pt ずつ下げて実測で確かめる。
# ★ 測るときの device を間違えないこと。文字幅は device のフォント metrics で決まる。
#   実測（同じ表・同じ 5.9 pt）: pdf 4.70 / cairo png 4.81 / ragg 5.04 inch。
#   ggplot2 3.5 の ggsave は ragg があれば ragg を使うので、一番広く出る ragg で測る。
#   （Jupyter の画面表示は cairo png なので、ragg に合わせておけばどちらでも溢れない）
W_BLK = W_FIG / length(blks)             # 1ブロックに割ける幅 (inch)
ragg::agg_png(tempfile(fileext = '.png'), width = 1000, height = 1000, res = 100)
for (fs_try in seq(7.0, 3.4, by = -0.1)) {
    w_max = 0
    for (b in seq_along(blks)) {
        tg0 = gridExtra::tableGrob(
                  tbl[blks[[b]], ], rows = NULL,
                  theme = ttheme(base_size = fs_try, padding = unit(c(2.2, 1.6), 'mm'),
                                 colnames.style = colnames_style(size = fs_try + 0.4),
                                 tbody.style = tbody_style(size = fs_try)))
        w_max = max(w_max, sum(convertWidth(tg0$widths, 'in', valueOnly = TRUE)))
    }
    FS = fs_try
    if (w_max <= W_BLK * 0.98) break     # 0.98 は plot.margin のぶんの余裕
}
invisible(dev.off())
cat(sprintf("表のフォント: %.1f pt（最も広いブロック %.2f inch / 1ブロックの割り当て %.2f inch）\n",
            FS, w_max, W_BLK))

# ★ 表の文字色を地図と揃える（ORA hit = 橙 / DEG only = 紫 / not DEG = 黒）。
#   ggtexttable（gridExtra の tableGrob）は色を ★ 列方向（column-major）に展開する ので、
#   行ごとの色ベクトルを列数ぶん「繰り返して」渡す（rep(col, ncol) であって each= ではない。
#   each= にすると列ごとに1色になる。実測で確認）
tg = lapply(seq_along(blks), function(b) {
    ix  = blks[[b]]
    col = unname(col_cat[tbl$class[ix]])
    g = ggtexttable(tbl[ix, ], rows = NULL,
                theme = ttheme(base_size = FS, padding = unit(c(2.2, 1.6), 'mm'),
                               colnames.style = colnames_style(size = FS + 0.4, fill = 'grey92'),
                               tbody.style = tbody_style(size = FS, fill = 'white',
                                                         color = rep(col, ncol(tbl)))))
    g = g + theme(plot.margin = margin(0, 2, 2, 2))
    if (b == 1) g = g + labs(title = 'B.  gene list') +
        theme(plot.title = element_text(size = 13, face = 'bold', hjust = 0,
                                        margin = margin(b = 2)))
    g
})
TB = wrap_plots(tg, nrow = 1)

# ★ 表（ggtexttable）はパネルの中央に置かれるので、割り当てた高さが余ると
#   上下に空白ができる。行数から必要な高さを詰めて見積もる
H_map = 6.4
H_tab = 0.125 * max(sapply(blks, length)) + 0.35
# ★ tag_levels は使わない。patchwork の全サブプロット（contig パネル10個 + 表3枚）に
#   A〜M と振られてしまうため。見出しは A の図タイトルと B の表タイトルで示す。
#
# ★ wrap_elements() で包む。
#   patchwork を入れ子（P / TB）にすると ★ 内側の plot_annotation が描画されず、
#   A のタイトルが消える。包むと内側が1つの要素として確定し、タイトルが残る
PB = (wrap_elements(full = P) / TB) + plot_layout(heights = c(H_map, H_tab))

# SCALE / W_FIG は表のフォントを決めるときに上で定義してある
suppressWarnings(suppressMessages(
    ggsave(file.path(dir_gp, sprintf('genomepos_%s_%s.png', ko, SCHEME)), PB,
           width = W_FIG, height = (H_map + H_tab) * SCALE,
           dpi = 100, limitsize = FALSE)))
options(repr.plot.width = W_FIG, repr.plot.height = (H_map + H_tab) * SCALE)
suppressWarnings(print(PB))



(1) pathview - all genes of the pathway → pathview が描けなかった（セル11 の検品で「★ 描画が壊れた」）
(2) pathview - ORA geneID only → pathview が描けなかった（セル11 の検品で「★ 描画が壊れた」）
ko00121_Secondary bile acid biosynthesis（二次胆汁酸）  contig 10 本（主 4 / 副 6）
   strain         contig  CDS 長さMb 描画幅
  Bth_VPI    NC_004663.1 1676  6.243   6.24
  Bth_VPI    NC_004703.1   36  0.033   0.60
  Bth_St8 BQNN01000001.1 1780  6.132   6.13
  Bth_St8 BQNN01000002.1   41  0.045   0.60
  Bth_St8 BQNN01000005.1    7  0.004   0.60
  Bth_St8 BQNN01000004.1    2  0.004   0.60
 Bun_8492  NZ_CP102263.1 3392  4.629   4.63
 Odo_St23 BQOE01000001.1 2801  5.647   5.65
 Odo_St23 BQOE01000002.1   38  0.065   0.60
 Odo_St23 BQOE01000003.1   12  0.032   0.60

経路の全遺伝子 7 / 図に出る 7（padj が NA で検定されなかったものも含めて表示）
凡例を出すパネル: NZ_CP102263.1（2 水準）

表: 7 行 → 1 ブロックに分けて横に並べる
表のフォント: 7.0 pt（最も広いブロック 7.30 inch / 1ブロックの割り当て 24.75 inch）


plot without title

In [121]:
# ── 図13-7: ko03010（リボソーム）  scheme A ───────────────────
# セル13（準備）を先に実行しておくこと（col_cat / strains4 / MINW / dir_pv / dir_gp を使う）
#
# ★ 3つを続けて出す
#   (1) pathview（経路の全遺伝子）… 反応と化合物名。経路の中での位置づけ
#   (2) pathview（ORA の geneID のみ）… ORA が拾った反応だけ
#   (3) ゲノム位置マップ ＋ 遺伝子表 … どこにあるか、何の遺伝子か
library(ggplot2); library(ggtext); library(ggrepel); library(patchwork); library(ggpubr)
library(png); library(grid)

ko = 'ko03010'; ko_jp = 'リボソーム'; SCHEME = 'A'

# ── (1)(2) セル11 で作った pathview を再掲する ────────────────
# ★ 化合物名や反応の向きは pathview にしか無いので、地図の前に置く。
#   ORA で有意でない組み合わせ（例: ko03010 の scheme B）はファイルが無いので飛ばす
for (kind in c('all', 'ORA')) {
    f = file.path(dir_pv, sprintf('%s.%s_%s.png', ko, SCHEME, kind))
    tag = if (kind == 'all') '(1) pathview - all genes of the pathway' else
                             '(2) pathview - ORA geneID only'
    if (!file.exists(f)) {
        cat(sprintf("%s → ファイルなし（セル11 未実行、またはこのスキームの ORA で有意でない）\n", tag))
        next
    }
    # ★ セル11 の検品で落ちた図（ko00121）は中身が書かれず 0 バイトで残る。
    #   file.exists() は TRUE になるので、大きさも見ないと readPNG がエラーで止まる
    if (file.size(f) == 0) {
        cat(sprintf("%s → pathview が描けなかった（セル11 の検品で「★ 描画が壊れた」）\n", tag))
        next
    }
    im = readPNG(f); h_px = dim(im)[1]; w_px = dim(im)[2]
    W = 15; H = W * h_px / w_px
    if (H > 18) { H = 18; W = H * w_px / h_px }
    cat(sprintf("%s  [%s]\n", tag, basename(f)))
    options(repr.plot.width = W, repr.plot.height = H)
    grid.newpage(); grid.raster(im)
}

# ── (3) ゲノム位置マップ ＋ 遺伝子表 ──────────────────────────
tgt = ora$term[ora$source=='KEGG' & grepl(paste0('^',ko,'_'), ora$term)][1]
g_ora = ora %>% filter(scheme==SCHEME, source=='KEGG', term==tgt, padj<0.05) %>%
        pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
        dplyr::select(gene_id, KEGG_Pathway) %>% tidyr::separate_rows(KEGG_Pathway, sep=',') %>%
        filter(KEGG_Pathway==ko) %>% pull(gene_id) %>% unique()

# contig を CDS 数の多い順に並べる（主 → 副）
ctg = res %>% filter(biotype=='CDS', strain %in% strains4, !is.na(end)) %>%
    group_by(strain, contig) %>% summarise(CDS=dplyr::n(), 長さMb=max(end)/1e6, .groups='drop') %>%
    arrange(factor(strain, levels=strains4), desc(CDS)) %>%
    # ★ 描画幅は最小値で底上げするが、x 軸は contig の実際の長さに合わせる。
    #   → 副 contig は bp あたりの縮尺が主より大きくなり、パネルを目一杯使う
    mutate(幅 = pmax(長さMb, MINW),
           短名 = sub('^[^~]+~', '', contig),
           # ★ contig 名は空白が無いので str_wrap では折れない。
           #   パネルが狭いと重なるので、8文字ごとに改行を入れて縦に積む（文字は削らない）
           名表示 = gsub('(.{10})', '\\1\n', 短名))

cat(sprintf("%s（%s）  contig %d 本（主 4 / 副 %d）\n", tgt, ko_jp, nrow(ctg), nrow(ctg)-4))
print(ctg %>% transmute(strain, contig=短名, CDS, 長さMb=round(長さMb,3), 描画幅=round(幅,2)) %>%
      as.data.frame(), row.names=FALSE)

bg = res %>% filter(biotype=='CDS', !marker, !is.na(start)) %>%
     mutate(位置Mb = start/1e6)
fg = bg %>% filter(gene_id %in% g_all) %>%
     mutate(cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                            !is.na(padj_A) & padj_A < 0.05 ~ 'DEG only', TRUE ~ 'not DEG'),
            cat = factor(cat, levels=names(col_cat)),
            # ★ 丸の塗りに発現強度（WT 群の正規化カウント平均）を入れる。
            #   発現量は3〜4桁にわたるので log10 にする（0 があるので +1）
            発現 = log10(normA_WT + 1),
            label = ifelse(is.na(Preferred_name)|Preferred_name=='-', substr(product,1,10), Preferred_name))
cat(sprintf("\n経路の全遺伝子 %d / 図に出る %d（padj が NA で検定されなかったものも含めて表示）\n",
    length(g_all), nrow(fg)))

ylim = range(c(bg$LFC_A[bg$strain %in% strains4]), na.rm=TRUE)

# ★ 凡例は「3分類が最もそろって現れる contig」1枚からだけ出す。
#   drop=FALSE を付けても ★ データが無い水準はキーの点が描かれないため、
#   パネルごとに凡例の見た目が変わり、guides='collect' が別物と判定して2つ並ぶ
leg_i = which.max(sapply(ctg$contig, function(cc)
            length(unique(fg$cat[fg$contig == cc]))))
cat(sprintf("凡例を出すパネル: %s（%d 水準）\n",
            ctg$短名[leg_i], length(unique(fg$cat[fg$contig == ctg$contig[leg_i]]))))

subs = list()
for (i in 1:nrow(ctg)) {
    b = bg %>% filter(contig == ctg$contig[i])
    f = fg %>% filter(contig == ctg$contig[i])
    is_main = ctg$CDS[i] == max(ctg$CDS[ctg$strain == ctg$strain[i]])
    p = ggplot(b, aes(x=位置Mb, y=LFC_A)) +
        geom_hline(yintercept=0, colour='grey20', linewidth=0.3) +
        geom_hline(yintercept=c(-1,1), linetype='dashed', colour='grey65', linewidth=0.3) +
        geom_point(colour='#b9c6d4', size=0.5, alpha=0.6) +
        coord_cartesian(xlim=c(0, ctg$長さMb[i]), ylim=ylim) +
        labs(title = sprintf('%s\n%s\n%d CDS, %.3f Mb',
                             ifelse(is_main, as.character(ctg$strain[i]), ''),
                             ctg$名表示[i], ctg$CDS[i], ctg$長さMb[i]),
             x=NULL, y=if (i==1) 'log2 fold change (KO / WT), scheme A' else NULL) +
        theme_bw(base_size=10) +
        theme(plot.title=element_text(size=ifelse(is_main, 9, 5.5),
                                    face=ifelse(is_main,'bold','plain'), lineheight=1.15),
              panel.grid.minor=element_blank(),
              axis.text.y=if (i==1) element_text() else element_blank(),
              axis.ticks.y=if (i==1) element_line() else element_blank(),
              legend.position='bottom', legend.box='horizontal',
              legend.title=element_text(size=8), legend.text=element_text(size=7.5),
              legend.margin=margin(1,1,1,1),
              plot.margin=margin(2,2,0,2))
    # ★ 経路の遺伝子が無い contig（0 行）では colour の水準が1つも現れないため、
    #   scale_colour_manual が「No shared levels found」の警告を出す。
    #   図は正しく描かれる（空パネルに色が無いだけ）ので、描画時に警告だけ抑える
    # ★ shape 21（縁と塗りを別々に指定できる丸）を使う。
    #   縁 = 分類（ORA hit / DEG only / not DEG）、塗り = 発現強度
    p = p + geom_point(data=f, aes(colour=cat, fill=発現), shape=21,
                       size=3.2, stroke=1.1, show.legend = (i == leg_i))
    if (nrow(f) > 0)
        p = p +
                # ★ 遺伝子名は小さいと読めないので大きめに。
                #   重なりは ggrepel の force と box.padding で逃がす
                geom_text_repel(data=f, aes(label=label, colour=cat), size=4.4,
                                fontface='bold', max.overlaps=60, seed=1,
                                force=3, min.segment.length=0,
                                segment.size=0.3, box.padding=0.45, show.legend=FALSE)
    # ★ 色の scale は「各パネルに」付ける。
    #   patchwork の guides='collect' は収集時点のパネルごとの凡例を比べるので、
    #   後から & で当てると収集に間に合わず凡例が2つ残る。
    #   全パネルが colour の aes を持ち、drop=FALSE で水準が揃うので1つにまとまる
    subs[[i]] = p + scale_colour_manual(values=col_cat, name=NULL, drop=FALSE,
                                        guide = guide_legend(override.aes = list(
                                            fill='white', size=3.2, stroke=1.1))) +
        # ★ 凡例を下に横向きで出す。右に置くと凡例の文字幅ぶん地図が狭くなる
        scale_fill_viridis_c(option='mako', direction=-1,
                             limits=range(fg$発現, na.rm=TRUE),
                             name='log10(DESeq2 normalized count + 1), WT mean',
                             guide = guide_colourbar(direction='horizontal',
                                                     title.position='top',
                                                     barwidth=unit(5,'cm'),
                                                     barheight=unit(0.35,'cm')))
}
P = wrap_plots(subs, nrow=1, widths=ctg$幅) + plot_layout(guides='collect') +
    plot_annotation(title = sprintf('A.  %s   [scheme %s]  - position on the genome', tgt, SCHEME),
      subtitle = paste0("grey = all CDS  |  coloured = genes of this pathway  ",
                        "(<span style='color:#d95f02'>**ORA hit**</span> / ",
                        "<span style='color:#7570b3'>**DEG only**</span> / ",
                        "<span style='color:#000000'>**not DEG**</span>)<br>",
                        "**gene labels = Preferred_name** (eggNOG; the first 10 characters of *product* ",
                        "are shown when Preferred_name is absent)<br>",
                        "point border = class, **point fill = log10(DESeq2 normalized count + 1), ",
                        "WT group mean** (scheme A: one size factor set for all 4 strains pooled)<br>",
                        "all contigs shown; panel width ∝ contig length, ",
                        sprintf("small contigs get a minimum width (%.1f Mb equivalent) and are drawn at a larger bp scale  |  x axis = position within each contig (Mb)", MINW)),

      # ★ 収集した凡例の位置は patchwork 側の theme で決まる。
      #   サブプロットの theme に legend.position を書いても効かない
      theme = theme(plot.title=element_text(size=14, face='bold'),
                    plot.subtitle=element_markdown(size=9, lineheight=1.3),
                    legend.position='bottom', legend.box='horizontal',
                    legend.title=element_text(size=8.5),
                    legend.text=element_text(size=8)))
# ── 表を同じ図の中に入れる（panel B）──────────────────────────
# ★ 図だけでは locus_tag / K番号 / EC が分からないので表を下に並べる。
#   ggpubr::ggtexttable は表を ggplot として返すので patchwork に載せられる
library(ggpubr)

tbl = fg %>%
    mutate(K  = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                       function(v) paste(v, collapse=',')),
           EC2 = ifelse(is.na(EC), '-', EC)) %>%
    transmute(strain, contig = sub('^[^~]+~', '', contig),
              Mb = round(位置Mb, 3), gene_id, name = label,
              K = K, EC = EC2, product = substr(product, 1, 26),
              # ★ LFC と padj は表に入れない。
              #   LFC は上の地図の y 軸そのもの、padj は分類（class）の色で示している。
              #   列を減らしたぶん1列あたりの幅が広がり、文字が重ならない
              class = as.character(cat)) %>%
    arrange(strain, Mb)

# ★ 縦に長くなりすぎるので、行を横に分割して並べる（1ブロック最大 24 行）
NROW_BLK = 24
nblk = ceiling(nrow(tbl) / NROW_BLK)
blks = split(1:nrow(tbl), ceiling(seq_along(1:nrow(tbl)) / ceiling(nrow(tbl)/nblk)))
cat(sprintf("\n表: %d 行 → %d ブロックに分けて横に並べる\n", nrow(tbl), length(blks)))

# ★ この図（地図＋表）だけ縦横 1.125 倍で出す（pathview の再掲は等倍のまま）
SCALE = 1.125
W_FIG = 22 * SCALE

# ★ 表のフォントは経路ごとに自動で決める。
#   列の幅は中身の文字数で決まるので、ko03010（112 遺伝子）のように行が多い経路では
#   ブロック数が増え、1ブロックに割ける幅が狭くなる。
#   固定の 6.2 pt のままだと表が割り当て幅をはみ出し、右端の class 列が切れてしまう。
#   → 実際に表を組んで幅を測り、収まる一番大きいフォントを選ぶ
#
# ★ 文字の幅は「フォントサイズに比例」しない。
#   小さいサイズでは hinting でカクカク変わる（実測: 5.5 pt で 4.75 inch、
#   5.9 pt で 5.04 inch）ので、計算で逆算せず 0.1 pt ずつ下げて実測で確かめる。
# ★ 測るときの device を間違えないこと。文字幅は device のフォント metrics で決まる。
#   実測（同じ表・同じ 5.9 pt）: pdf 4.70 / cairo png 4.81 / ragg 5.04 inch。
#   ggplot2 3.5 の ggsave は ragg があれば ragg を使うので、一番広く出る ragg で測る。
#   （Jupyter の画面表示は cairo png なので、ragg に合わせておけばどちらでも溢れない）
W_BLK = W_FIG / length(blks)             # 1ブロックに割ける幅 (inch)
ragg::agg_png(tempfile(fileext = '.png'), width = 1000, height = 1000, res = 100)
for (fs_try in seq(7.0, 3.4, by = -0.1)) {
    w_max = 0
    for (b in seq_along(blks)) {
        tg0 = gridExtra::tableGrob(
                  tbl[blks[[b]], ], rows = NULL,
                  theme = ttheme(base_size = fs_try, padding = unit(c(2.2, 1.6), 'mm'),
                                 colnames.style = colnames_style(size = fs_try + 0.4),
                                 tbody.style = tbody_style(size = fs_try)))
        w_max = max(w_max, sum(convertWidth(tg0$widths, 'in', valueOnly = TRUE)))
    }
    FS = fs_try
    if (w_max <= W_BLK * 0.98) break     # 0.98 は plot.margin のぶんの余裕
}
invisible(dev.off())
cat(sprintf("表のフォント: %.1f pt（最も広いブロック %.2f inch / 1ブロックの割り当て %.2f inch）\n",
            FS, w_max, W_BLK))

# ★ 表の文字色を地図と揃える（ORA hit = 橙 / DEG only = 紫 / not DEG = 黒）。
#   ggtexttable（gridExtra の tableGrob）は色を ★ 列方向（column-major）に展開する ので、
#   行ごとの色ベクトルを列数ぶん「繰り返して」渡す（rep(col, ncol) であって each= ではない。
#   each= にすると列ごとに1色になる。実測で確認）
tg = lapply(seq_along(blks), function(b) {
    ix  = blks[[b]]
    col = unname(col_cat[tbl$class[ix]])
    g = ggtexttable(tbl[ix, ], rows = NULL,
                theme = ttheme(base_size = FS, padding = unit(c(2.2, 1.6), 'mm'),
                               colnames.style = colnames_style(size = FS + 0.4, fill = 'grey92'),
                               tbody.style = tbody_style(size = FS, fill = 'white',
                                                         color = rep(col, ncol(tbl)))))
    g = g + theme(plot.margin = margin(0, 2, 2, 2))
    if (b == 1) g = g + labs(title = 'B.  gene list') +
        theme(plot.title = element_text(size = 13, face = 'bold', hjust = 0,
                                        margin = margin(b = 2)))
    g
})
TB = wrap_plots(tg, nrow = 1)

# ★ 表（ggtexttable）はパネルの中央に置かれるので、割り当てた高さが余ると
#   上下に空白ができる。行数から必要な高さを詰めて見積もる
H_map = 6.4
H_tab = 0.125 * max(sapply(blks, length)) + 0.35
# ★ tag_levels は使わない。patchwork の全サブプロット（contig パネル10個 + 表3枚）に
#   A〜M と振られてしまうため。見出しは A の図タイトルと B の表タイトルで示す。
#
# ★ wrap_elements() で包む。
#   patchwork を入れ子（P / TB）にすると ★ 内側の plot_annotation が描画されず、
#   A のタイトルが消える。包むと内側が1つの要素として確定し、タイトルが残る
PB = (wrap_elements(full = P) / TB) + plot_layout(heights = c(H_map, H_tab))

# SCALE / W_FIG は表のフォントを決めるときに上で定義してある
suppressWarnings(suppressMessages(
    ggsave(file.path(dir_gp, sprintf('genomepos_%s_%s.png', ko, SCHEME)), PB,
           width = W_FIG, height = (H_map + H_tab) * SCALE,
           dpi = 100, limitsize = FALSE)))
options(repr.plot.width = W_FIG, repr.plot.height = (H_map + H_tab) * SCALE)
suppressWarnings(print(PB))



(1) pathview - all genes of the pathway  [ko03010.A_all.png]
(2) pathview - ORA geneID only  [ko03010.A_ORA.png]


plot without title

ko03010_Ribosome（リボソーム）  contig 10 本（主 4 / 副 6）
   strain         contig  CDS 長さMb 描画幅
  Bth_VPI    NC_004663.1 1676  6.243   6.24
  Bth_VPI    NC_004703.1   36  0.033   0.60
  Bth_St8 BQNN01000001.1 1780  6.132   6.13
  Bth_St8 BQNN01000002.1   41  0.045   0.60
  Bth_St8 BQNN01000005.1    7  0.004   0.60
  Bth_St8 BQNN01000004.1    2  0.004   0.60
 Bun_8492  NZ_CP102263.1 3392  4.629   4.63
 Odo_St23 BQOE01000001.1 2801  5.647   5.65
 Odo_St23 BQOE01000002.1   38  0.065   0.60
 Odo_St23 BQOE01000003.1   12  0.032   0.60

経路の全遺伝子 112 / 図に出る 112（padj が NA で検定されなかったものも含めて表示）
凡例を出すパネル: NC_004663.1（2 水準）

表: 112 行 → 5 ブロックに分けて横に並べる
表のフォント: 5.6 pt（最も広いブロック 4.82 inch / 1ブロックの割り当て 4.95 inch）


plot without title

plot without title

In [122]:
# ── 図13-8: ko00030（ペントースリン酸経路）  scheme B ───────────────────
# セル13（準備）を先に実行しておくこと（col_cat / strains4 / MINW / dir_pv / dir_gp を使う）
#
# ★ 3つを続けて出す
#   (1) pathview（経路の全遺伝子）… 反応と化合物名。経路の中での位置づけ
#   (2) pathview（ORA の geneID のみ）… ORA が拾った反応だけ
#   (3) ゲノム位置マップ ＋ 遺伝子表 … どこにあるか、何の遺伝子か
library(ggplot2); library(ggtext); library(ggrepel); library(patchwork); library(ggpubr)
library(png); library(grid)

ko = 'ko00030'; ko_jp = 'ペントースリン酸経路'; SCHEME = 'B'

# ── (1)(2) セル11 で作った pathview を再掲する ────────────────
# ★ 化合物名や反応の向きは pathview にしか無いので、地図の前に置く。
#   ORA で有意でない組み合わせ（例: ko03010 の scheme B）はファイルが無いので飛ばす
for (kind in c('all', 'ORA')) {
    f = file.path(dir_pv, sprintf('%s.%s_%s.png', ko, SCHEME, kind))
    tag = if (kind == 'all') '(1) pathview - all genes of the pathway' else
                             '(2) pathview - ORA geneID only'
    if (!file.exists(f)) {
        cat(sprintf("%s → ファイルなし（セル11 未実行、またはこのスキームの ORA で有意でない）\n", tag))
        next
    }
    # ★ セル11 の検品で落ちた図（ko00121）は中身が書かれず 0 バイトで残る。
    #   file.exists() は TRUE になるので、大きさも見ないと readPNG がエラーで止まる
    if (file.size(f) == 0) {
        cat(sprintf("%s → pathview が描けなかった（セル11 の検品で「★ 描画が壊れた」）\n", tag))
        next
    }
    im = readPNG(f); h_px = dim(im)[1]; w_px = dim(im)[2]
    W = 15; H = W * h_px / w_px
    if (H > 18) { H = 18; W = H * w_px / h_px }
    cat(sprintf("%s  [%s]\n", tag, basename(f)))
    options(repr.plot.width = W, repr.plot.height = H)
    grid.newpage(); grid.raster(im)
}

# ── (3) ゲノム位置マップ ＋ 遺伝子表 ──────────────────────────
tgt = ora$term[ora$source=='KEGG' & grepl(paste0('^',ko,'_'), ora$term)][1]
g_ora = ora %>% filter(scheme==SCHEME, source=='KEGG', term==tgt, padj<0.05) %>%
        pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
        dplyr::select(gene_id, KEGG_Pathway) %>% tidyr::separate_rows(KEGG_Pathway, sep=',') %>%
        filter(KEGG_Pathway==ko) %>% pull(gene_id) %>% unique()

# contig を CDS 数の多い順に並べる（主 → 副）
ctg = res %>% filter(biotype=='CDS', strain %in% strains4, !is.na(end)) %>%
    group_by(strain, contig) %>% summarise(CDS=dplyr::n(), 長さMb=max(end)/1e6, .groups='drop') %>%
    arrange(factor(strain, levels=strains4), desc(CDS)) %>%
    # ★ 描画幅は最小値で底上げするが、x 軸は contig の実際の長さに合わせる。
    #   → 副 contig は bp あたりの縮尺が主より大きくなり、パネルを目一杯使う
    mutate(幅 = pmax(長さMb, MINW),
           短名 = sub('^[^~]+~', '', contig),
           # ★ contig 名は空白が無いので str_wrap では折れない。
           #   パネルが狭いと重なるので、8文字ごとに改行を入れて縦に積む（文字は削らない）
           名表示 = gsub('(.{10})', '\\1\n', 短名))

cat(sprintf("%s（%s）  contig %d 本（主 4 / 副 %d）\n", tgt, ko_jp, nrow(ctg), nrow(ctg)-4))
print(ctg %>% transmute(strain, contig=短名, CDS, 長さMb=round(長さMb,3), 描画幅=round(幅,2)) %>%
      as.data.frame(), row.names=FALSE)

bg = res %>% filter(biotype=='CDS', !marker, !is.na(start)) %>%
     mutate(位置Mb = start/1e6)
fg = bg %>% filter(gene_id %in% g_all) %>%
     mutate(cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                            !is.na(padj_pooled) & padj_pooled < 0.05 ~ 'DEG only', TRUE ~ 'not DEG'),
            cat = factor(cat, levels=names(col_cat)),
            # ★ 丸の塗りに発現強度（WT 群の正規化カウント平均）を入れる。
            #   発現量は3〜4桁にわたるので log10 にする（0 があるので +1）
            発現 = log10(normB_WT + 1),
            label = ifelse(is.na(Preferred_name)|Preferred_name=='-', substr(product,1,10), Preferred_name))
cat(sprintf("\n経路の全遺伝子 %d / 図に出る %d（padj が NA で検定されなかったものも含めて表示）\n",
    length(g_all), nrow(fg)))

ylim = range(c(bg$LFC_B[bg$strain %in% strains4]), na.rm=TRUE)

# ★ 凡例は「3分類が最もそろって現れる contig」1枚からだけ出す。
#   drop=FALSE を付けても ★ データが無い水準はキーの点が描かれないため、
#   パネルごとに凡例の見た目が変わり、guides='collect' が別物と判定して2つ並ぶ
leg_i = which.max(sapply(ctg$contig, function(cc)
            length(unique(fg$cat[fg$contig == cc]))))
cat(sprintf("凡例を出すパネル: %s（%d 水準）\n",
            ctg$短名[leg_i], length(unique(fg$cat[fg$contig == ctg$contig[leg_i]]))))

subs = list()
for (i in 1:nrow(ctg)) {
    b = bg %>% filter(contig == ctg$contig[i])
    f = fg %>% filter(contig == ctg$contig[i])
    is_main = ctg$CDS[i] == max(ctg$CDS[ctg$strain == ctg$strain[i]])
    p = ggplot(b, aes(x=位置Mb, y=LFC_B)) +
        geom_hline(yintercept=0, colour='grey20', linewidth=0.3) +
        geom_hline(yintercept=c(-1,1), linetype='dashed', colour='grey65', linewidth=0.3) +
        geom_point(colour='#b9c6d4', size=0.5, alpha=0.6) +
        coord_cartesian(xlim=c(0, ctg$長さMb[i]), ylim=ylim) +
        labs(title = sprintf('%s\n%s\n%d CDS, %.3f Mb',
                             ifelse(is_main, as.character(ctg$strain[i]), ''),
                             ctg$名表示[i], ctg$CDS[i], ctg$長さMb[i]),
             x=NULL, y=if (i==1) 'log2 fold change (KO / WT), scheme B' else NULL) +
        theme_bw(base_size=10) +
        theme(plot.title=element_text(size=ifelse(is_main, 9, 5.5),
                                    face=ifelse(is_main,'bold','plain'), lineheight=1.15),
              panel.grid.minor=element_blank(),
              axis.text.y=if (i==1) element_text() else element_blank(),
              axis.ticks.y=if (i==1) element_line() else element_blank(),
              legend.position='bottom', legend.box='horizontal',
              legend.title=element_text(size=8), legend.text=element_text(size=7.5),
              legend.margin=margin(1,1,1,1),
              plot.margin=margin(2,2,0,2))
    # ★ 経路の遺伝子が無い contig（0 行）では colour の水準が1つも現れないため、
    #   scale_colour_manual が「No shared levels found」の警告を出す。
    #   図は正しく描かれる（空パネルに色が無いだけ）ので、描画時に警告だけ抑える
    # ★ shape 21（縁と塗りを別々に指定できる丸）を使う。
    #   縁 = 分類（ORA hit / DEG only / not DEG）、塗り = 発現強度
    p = p + geom_point(data=f, aes(colour=cat, fill=発現), shape=21,
                       size=3.2, stroke=1.1, show.legend = (i == leg_i))
    if (nrow(f) > 0)
        p = p +
                # ★ 遺伝子名は小さいと読めないので大きめに。
                #   重なりは ggrepel の force と box.padding で逃がす
                geom_text_repel(data=f, aes(label=label, colour=cat), size=4.4,
                                fontface='bold', max.overlaps=60, seed=1,
                                force=3, min.segment.length=0,
                                segment.size=0.3, box.padding=0.45, show.legend=FALSE)
    # ★ 色の scale は「各パネルに」付ける。
    #   patchwork の guides='collect' は収集時点のパネルごとの凡例を比べるので、
    #   後から & で当てると収集に間に合わず凡例が2つ残る。
    #   全パネルが colour の aes を持ち、drop=FALSE で水準が揃うので1つにまとまる
    subs[[i]] = p + scale_colour_manual(values=col_cat, name=NULL, drop=FALSE,
                                        guide = guide_legend(override.aes = list(
                                            fill='white', size=3.2, stroke=1.1))) +
        # ★ 凡例を下に横向きで出す。右に置くと凡例の文字幅ぶん地図が狭くなる
        scale_fill_viridis_c(option='mako', direction=-1,
                             limits=range(fg$発現, na.rm=TRUE),
                             name='log10(DESeq2 normalized count + 1), WT mean',
                             guide = guide_colourbar(direction='horizontal',
                                                     title.position='top',
                                                     barwidth=unit(5,'cm'),
                                                     barheight=unit(0.35,'cm')))
}
P = wrap_plots(subs, nrow=1, widths=ctg$幅) + plot_layout(guides='collect') +
    plot_annotation(title = sprintf('A.  %s   [scheme %s]  - position on the genome', tgt, SCHEME),
      subtitle = paste0("grey = all CDS  |  coloured = genes of this pathway  ",
                        "(<span style='color:#d95f02'>**ORA hit**</span> / ",
                        "<span style='color:#7570b3'>**DEG only**</span> / ",
                        "<span style='color:#000000'>**not DEG**</span>)<br>",
                        "**gene labels = Preferred_name** (eggNOG; the first 10 characters of *product* ",
                        "are shown when Preferred_name is absent)<br>",
                        "point border = class, **point fill = log10(DESeq2 normalized count + 1), ",
                        "WT group mean** (scheme B: one size factor set for all 4 strains pooled)<br>",
                        "all contigs shown; panel width ∝ contig length, ",
                        sprintf("small contigs get a minimum width (%.1f Mb equivalent) and are drawn at a larger bp scale  |  x axis = position within each contig (Mb)", MINW)),

      # ★ 収集した凡例の位置は patchwork 側の theme で決まる。
      #   サブプロットの theme に legend.position を書いても効かない
      theme = theme(plot.title=element_text(size=14, face='bold'),
                    plot.subtitle=element_markdown(size=9, lineheight=1.3),
                    legend.position='bottom', legend.box='horizontal',
                    legend.title=element_text(size=8.5),
                    legend.text=element_text(size=8)))
# ── 表を同じ図の中に入れる（panel B）──────────────────────────
# ★ 図だけでは locus_tag / K番号 / EC が分からないので表を下に並べる。
#   ggpubr::ggtexttable は表を ggplot として返すので patchwork に載せられる
library(ggpubr)

tbl = fg %>%
    mutate(K  = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                       function(v) paste(v, collapse=',')),
           EC2 = ifelse(is.na(EC), '-', EC)) %>%
    transmute(strain, contig = sub('^[^~]+~', '', contig),
              Mb = round(位置Mb, 3), gene_id, name = label,
              K = K, EC = EC2, product = substr(product, 1, 26),
              # ★ LFC と padj は表に入れない。
              #   LFC は上の地図の y 軸そのもの、padj は分類（class）の色で示している。
              #   列を減らしたぶん1列あたりの幅が広がり、文字が重ならない
              class = as.character(cat)) %>%
    arrange(strain, Mb)

# ★ 縦に長くなりすぎるので、行を横に分割して並べる（1ブロック最大 24 行）
NROW_BLK = 24
nblk = ceiling(nrow(tbl) / NROW_BLK)
blks = split(1:nrow(tbl), ceiling(seq_along(1:nrow(tbl)) / ceiling(nrow(tbl)/nblk)))
cat(sprintf("\n表: %d 行 → %d ブロックに分けて横に並べる\n", nrow(tbl), length(blks)))

# ★ この図（地図＋表）だけ縦横 1.125 倍で出す（pathview の再掲は等倍のまま）
SCALE = 1.125
W_FIG = 22 * SCALE

# ★ 表のフォントは経路ごとに自動で決める。
#   列の幅は中身の文字数で決まるので、ko03010（112 遺伝子）のように行が多い経路では
#   ブロック数が増え、1ブロックに割ける幅が狭くなる。
#   固定の 6.2 pt のままだと表が割り当て幅をはみ出し、右端の class 列が切れてしまう。
#   → 実際に表を組んで幅を測り、収まる一番大きいフォントを選ぶ
#
# ★ 文字の幅は「フォントサイズに比例」しない。
#   小さいサイズでは hinting でカクカク変わる（実測: 5.5 pt で 4.75 inch、
#   5.9 pt で 5.04 inch）ので、計算で逆算せず 0.1 pt ずつ下げて実測で確かめる。
# ★ 測るときの device を間違えないこと。文字幅は device のフォント metrics で決まる。
#   実測（同じ表・同じ 5.9 pt）: pdf 4.70 / cairo png 4.81 / ragg 5.04 inch。
#   ggplot2 3.5 の ggsave は ragg があれば ragg を使うので、一番広く出る ragg で測る。
#   （Jupyter の画面表示は cairo png なので、ragg に合わせておけばどちらでも溢れない）
W_BLK = W_FIG / length(blks)             # 1ブロックに割ける幅 (inch)
ragg::agg_png(tempfile(fileext = '.png'), width = 1000, height = 1000, res = 100)
for (fs_try in seq(7.0, 3.4, by = -0.1)) {
    w_max = 0
    for (b in seq_along(blks)) {
        tg0 = gridExtra::tableGrob(
                  tbl[blks[[b]], ], rows = NULL,
                  theme = ttheme(base_size = fs_try, padding = unit(c(2.2, 1.6), 'mm'),
                                 colnames.style = colnames_style(size = fs_try + 0.4),
                                 tbody.style = tbody_style(size = fs_try)))
        w_max = max(w_max, sum(convertWidth(tg0$widths, 'in', valueOnly = TRUE)))
    }
    FS = fs_try
    if (w_max <= W_BLK * 0.98) break     # 0.98 は plot.margin のぶんの余裕
}
invisible(dev.off())
cat(sprintf("表のフォント: %.1f pt（最も広いブロック %.2f inch / 1ブロックの割り当て %.2f inch）\n",
            FS, w_max, W_BLK))

# ★ 表の文字色を地図と揃える（ORA hit = 橙 / DEG only = 紫 / not DEG = 黒）。
#   ggtexttable（gridExtra の tableGrob）は色を ★ 列方向（column-major）に展開する ので、
#   行ごとの色ベクトルを列数ぶん「繰り返して」渡す（rep(col, ncol) であって each= ではない。
#   each= にすると列ごとに1色になる。実測で確認）
tg = lapply(seq_along(blks), function(b) {
    ix  = blks[[b]]
    col = unname(col_cat[tbl$class[ix]])
    g = ggtexttable(tbl[ix, ], rows = NULL,
                theme = ttheme(base_size = FS, padding = unit(c(2.2, 1.6), 'mm'),
                               colnames.style = colnames_style(size = FS + 0.4, fill = 'grey92'),
                               tbody.style = tbody_style(size = FS, fill = 'white',
                                                         color = rep(col, ncol(tbl)))))
    g = g + theme(plot.margin = margin(0, 2, 2, 2))
    if (b == 1) g = g + labs(title = 'B.  gene list') +
        theme(plot.title = element_text(size = 13, face = 'bold', hjust = 0,
                                        margin = margin(b = 2)))
    g
})
TB = wrap_plots(tg, nrow = 1)

# ★ 表（ggtexttable）はパネルの中央に置かれるので、割り当てた高さが余ると
#   上下に空白ができる。行数から必要な高さを詰めて見積もる
H_map = 6.4
H_tab = 0.125 * max(sapply(blks, length)) + 0.35
# ★ tag_levels は使わない。patchwork の全サブプロット（contig パネル10個 + 表3枚）に
#   A〜M と振られてしまうため。見出しは A の図タイトルと B の表タイトルで示す。
#
# ★ wrap_elements() で包む。
#   patchwork を入れ子（P / TB）にすると ★ 内側の plot_annotation が描画されず、
#   A のタイトルが消える。包むと内側が1つの要素として確定し、タイトルが残る
PB = (wrap_elements(full = P) / TB) + plot_layout(heights = c(H_map, H_tab))

# SCALE / W_FIG は表のフォントを決めるときに上で定義してある
suppressWarnings(suppressMessages(
    ggsave(file.path(dir_gp, sprintf('genomepos_%s_%s.png', ko, SCHEME)), PB,
           width = W_FIG, height = (H_map + H_tab) * SCALE,
           dpi = 100, limitsize = FALSE)))
options(repr.plot.width = W_FIG, repr.plot.height = (H_map + H_tab) * SCALE)
suppressWarnings(print(PB))



(1) pathview - all genes of the pathway  [ko00030.B_all.png]
(2) pathview - ORA geneID only  [ko00030.B_ORA.png]


plot without title

ko00030_Pentose phosphate pathway（ペントースリン酸経路）  contig 10 本（主 4 / 副 6）
   strain         contig  CDS 長さMb 描画幅
  Bth_VPI    NC_004663.1 1676  6.243   6.24
  Bth_VPI    NC_004703.1   36  0.033   0.60
  Bth_St8 BQNN01000001.1 1780  6.132   6.13
  Bth_St8 BQNN01000002.1   41  0.045   0.60
  Bth_St8 BQNN01000005.1    7  0.004   0.60
  Bth_St8 BQNN01000004.1    2  0.004   0.60
 Bun_8492  NZ_CP102263.1 3392  4.629   4.63
 Odo_St23 BQOE01000001.1 2801  5.647   5.65
 Odo_St23 BQOE01000002.1   38  0.065   0.60
 Odo_St23 BQOE01000003.1   12  0.032   0.60

経路の全遺伝子 69 / 図に出る 69（padj が NA で検定されなかったものも含めて表示）
凡例を出すパネル: NC_004663.1（3 水準）

表: 69 行 → 3 ブロックに分けて横に並べる
表のフォント: 7.0 pt（最も広いブロック 7.34 inch / 1ブロックの割り当て 8.25 inch）


plot without title

plot without title

In [123]:
# ── 図13-9: ko00020（TCA回路）  scheme B ───────────────────
# セル13（準備）を先に実行しておくこと（col_cat / strains4 / MINW / dir_pv / dir_gp を使う）
#
# ★ 3つを続けて出す
#   (1) pathview（経路の全遺伝子）… 反応と化合物名。経路の中での位置づけ
#   (2) pathview（ORA の geneID のみ）… ORA が拾った反応だけ
#   (3) ゲノム位置マップ ＋ 遺伝子表 … どこにあるか、何の遺伝子か
library(ggplot2); library(ggtext); library(ggrepel); library(patchwork); library(ggpubr)
library(png); library(grid)

ko = 'ko00020'; ko_jp = 'TCA回路'; SCHEME = 'B'

# ── (1)(2) セル11 で作った pathview を再掲する ────────────────
# ★ 化合物名や反応の向きは pathview にしか無いので、地図の前に置く。
#   ORA で有意でない組み合わせ（例: ko03010 の scheme B）はファイルが無いので飛ばす
for (kind in c('all', 'ORA')) {
    f = file.path(dir_pv, sprintf('%s.%s_%s.png', ko, SCHEME, kind))
    tag = if (kind == 'all') '(1) pathview - all genes of the pathway' else
                             '(2) pathview - ORA geneID only'
    if (!file.exists(f)) {
        cat(sprintf("%s → ファイルなし（セル11 未実行、またはこのスキームの ORA で有意でない）\n", tag))
        next
    }
    # ★ セル11 の検品で落ちた図（ko00121）は中身が書かれず 0 バイトで残る。
    #   file.exists() は TRUE になるので、大きさも見ないと readPNG がエラーで止まる
    if (file.size(f) == 0) {
        cat(sprintf("%s → pathview が描けなかった（セル11 の検品で「★ 描画が壊れた」）\n", tag))
        next
    }
    im = readPNG(f); h_px = dim(im)[1]; w_px = dim(im)[2]
    W = 15; H = W * h_px / w_px
    if (H > 18) { H = 18; W = H * w_px / h_px }
    cat(sprintf("%s  [%s]\n", tag, basename(f)))
    options(repr.plot.width = W, repr.plot.height = H)
    grid.newpage(); grid.raster(im)
}

# ── (3) ゲノム位置マップ ＋ 遺伝子表 ──────────────────────────
tgt = ora$term[ora$source=='KEGG' & grepl(paste0('^',ko,'_'), ora$term)][1]
g_ora = ora %>% filter(scheme==SCHEME, source=='KEGG', term==tgt, padj<0.05) %>%
        pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
        dplyr::select(gene_id, KEGG_Pathway) %>% tidyr::separate_rows(KEGG_Pathway, sep=',') %>%
        filter(KEGG_Pathway==ko) %>% pull(gene_id) %>% unique()

# contig を CDS 数の多い順に並べる（主 → 副）
ctg = res %>% filter(biotype=='CDS', strain %in% strains4, !is.na(end)) %>%
    group_by(strain, contig) %>% summarise(CDS=dplyr::n(), 長さMb=max(end)/1e6, .groups='drop') %>%
    arrange(factor(strain, levels=strains4), desc(CDS)) %>%
    # ★ 描画幅は最小値で底上げするが、x 軸は contig の実際の長さに合わせる。
    #   → 副 contig は bp あたりの縮尺が主より大きくなり、パネルを目一杯使う
    mutate(幅 = pmax(長さMb, MINW),
           短名 = sub('^[^~]+~', '', contig),
           # ★ contig 名は空白が無いので str_wrap では折れない。
           #   パネルが狭いと重なるので、8文字ごとに改行を入れて縦に積む（文字は削らない）
           名表示 = gsub('(.{10})', '\\1\n', 短名))

cat(sprintf("%s（%s）  contig %d 本（主 4 / 副 %d）\n", tgt, ko_jp, nrow(ctg), nrow(ctg)-4))
print(ctg %>% transmute(strain, contig=短名, CDS, 長さMb=round(長さMb,3), 描画幅=round(幅,2)) %>%
      as.data.frame(), row.names=FALSE)

bg = res %>% filter(biotype=='CDS', !marker, !is.na(start)) %>%
     mutate(位置Mb = start/1e6)
fg = bg %>% filter(gene_id %in% g_all) %>%
     mutate(cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                            !is.na(padj_pooled) & padj_pooled < 0.05 ~ 'DEG only', TRUE ~ 'not DEG'),
            cat = factor(cat, levels=names(col_cat)),
            # ★ 丸の塗りに発現強度（WT 群の正規化カウント平均）を入れる。
            #   発現量は3〜4桁にわたるので log10 にする（0 があるので +1）
            発現 = log10(normB_WT + 1),
            label = ifelse(is.na(Preferred_name)|Preferred_name=='-', substr(product,1,10), Preferred_name))
cat(sprintf("\n経路の全遺伝子 %d / 図に出る %d（padj が NA で検定されなかったものも含めて表示）\n",
    length(g_all), nrow(fg)))

ylim = range(c(bg$LFC_B[bg$strain %in% strains4]), na.rm=TRUE)

# ★ 凡例は「3分類が最もそろって現れる contig」1枚からだけ出す。
#   drop=FALSE を付けても ★ データが無い水準はキーの点が描かれないため、
#   パネルごとに凡例の見た目が変わり、guides='collect' が別物と判定して2つ並ぶ
leg_i = which.max(sapply(ctg$contig, function(cc)
            length(unique(fg$cat[fg$contig == cc]))))
cat(sprintf("凡例を出すパネル: %s（%d 水準）\n",
            ctg$短名[leg_i], length(unique(fg$cat[fg$contig == ctg$contig[leg_i]]))))

subs = list()
for (i in 1:nrow(ctg)) {
    b = bg %>% filter(contig == ctg$contig[i])
    f = fg %>% filter(contig == ctg$contig[i])
    is_main = ctg$CDS[i] == max(ctg$CDS[ctg$strain == ctg$strain[i]])
    p = ggplot(b, aes(x=位置Mb, y=LFC_B)) +
        geom_hline(yintercept=0, colour='grey20', linewidth=0.3) +
        geom_hline(yintercept=c(-1,1), linetype='dashed', colour='grey65', linewidth=0.3) +
        geom_point(colour='#b9c6d4', size=0.5, alpha=0.6) +
        coord_cartesian(xlim=c(0, ctg$長さMb[i]), ylim=ylim) +
        labs(title = sprintf('%s\n%s\n%d CDS, %.3f Mb',
                             ifelse(is_main, as.character(ctg$strain[i]), ''),
                             ctg$名表示[i], ctg$CDS[i], ctg$長さMb[i]),
             x=NULL, y=if (i==1) 'log2 fold change (KO / WT), scheme B' else NULL) +
        theme_bw(base_size=10) +
        theme(plot.title=element_text(size=ifelse(is_main, 9, 5.5),
                                    face=ifelse(is_main,'bold','plain'), lineheight=1.15),
              panel.grid.minor=element_blank(),
              axis.text.y=if (i==1) element_text() else element_blank(),
              axis.ticks.y=if (i==1) element_line() else element_blank(),
              legend.position='bottom', legend.box='horizontal',
              legend.title=element_text(size=8), legend.text=element_text(size=7.5),
              legend.margin=margin(1,1,1,1),
              plot.margin=margin(2,2,0,2))
    # ★ 経路の遺伝子が無い contig（0 行）では colour の水準が1つも現れないため、
    #   scale_colour_manual が「No shared levels found」の警告を出す。
    #   図は正しく描かれる（空パネルに色が無いだけ）ので、描画時に警告だけ抑える
    # ★ shape 21（縁と塗りを別々に指定できる丸）を使う。
    #   縁 = 分類（ORA hit / DEG only / not DEG）、塗り = 発現強度
    p = p + geom_point(data=f, aes(colour=cat, fill=発現), shape=21,
                       size=3.2, stroke=1.1, show.legend = (i == leg_i))
    if (nrow(f) > 0)
        p = p +
                # ★ 遺伝子名は小さいと読めないので大きめに。
                #   重なりは ggrepel の force と box.padding で逃がす
                geom_text_repel(data=f, aes(label=label, colour=cat), size=4.4,
                                fontface='bold', max.overlaps=60, seed=1,
                                force=3, min.segment.length=0,
                                segment.size=0.3, box.padding=0.45, show.legend=FALSE)
    # ★ 色の scale は「各パネルに」付ける。
    #   patchwork の guides='collect' は収集時点のパネルごとの凡例を比べるので、
    #   後から & で当てると収集に間に合わず凡例が2つ残る。
    #   全パネルが colour の aes を持ち、drop=FALSE で水準が揃うので1つにまとまる
    subs[[i]] = p + scale_colour_manual(values=col_cat, name=NULL, drop=FALSE,
                                        guide = guide_legend(override.aes = list(
                                            fill='white', size=3.2, stroke=1.1))) +
        # ★ 凡例を下に横向きで出す。右に置くと凡例の文字幅ぶん地図が狭くなる
        scale_fill_viridis_c(option='mako', direction=-1,
                             limits=range(fg$発現, na.rm=TRUE),
                             name='log10(DESeq2 normalized count + 1), WT mean',
                             guide = guide_colourbar(direction='horizontal',
                                                     title.position='top',
                                                     barwidth=unit(5,'cm'),
                                                     barheight=unit(0.35,'cm')))
}
P = wrap_plots(subs, nrow=1, widths=ctg$幅) + plot_layout(guides='collect') +
    plot_annotation(title = sprintf('A.  %s   [scheme %s]  - position on the genome', tgt, SCHEME),
      subtitle = paste0("grey = all CDS  |  coloured = genes of this pathway  ",
                        "(<span style='color:#d95f02'>**ORA hit**</span> / ",
                        "<span style='color:#7570b3'>**DEG only**</span> / ",
                        "<span style='color:#000000'>**not DEG**</span>)<br>",
                        "**gene labels = Preferred_name** (eggNOG; the first 10 characters of *product* ",
                        "are shown when Preferred_name is absent)<br>",
                        "point border = class, **point fill = log10(DESeq2 normalized count + 1), ",
                        "WT group mean** (scheme B: one size factor set for all 4 strains pooled)<br>",
                        "all contigs shown; panel width ∝ contig length, ",
                        sprintf("small contigs get a minimum width (%.1f Mb equivalent) and are drawn at a larger bp scale  |  x axis = position within each contig (Mb)", MINW)),

      # ★ 収集した凡例の位置は patchwork 側の theme で決まる。
      #   サブプロットの theme に legend.position を書いても効かない
      theme = theme(plot.title=element_text(size=14, face='bold'),
                    plot.subtitle=element_markdown(size=9, lineheight=1.3),
                    legend.position='bottom', legend.box='horizontal',
                    legend.title=element_text(size=8.5),
                    legend.text=element_text(size=8)))
# ── 表を同じ図の中に入れる（panel B）──────────────────────────
# ★ 図だけでは locus_tag / K番号 / EC が分からないので表を下に並べる。
#   ggpubr::ggtexttable は表を ggplot として返すので patchwork に載せられる
library(ggpubr)

tbl = fg %>%
    mutate(K  = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                       function(v) paste(v, collapse=',')),
           EC2 = ifelse(is.na(EC), '-', EC)) %>%
    transmute(strain, contig = sub('^[^~]+~', '', contig),
              Mb = round(位置Mb, 3), gene_id, name = label,
              K = K, EC = EC2, product = substr(product, 1, 26),
              # ★ LFC と padj は表に入れない。
              #   LFC は上の地図の y 軸そのもの、padj は分類（class）の色で示している。
              #   列を減らしたぶん1列あたりの幅が広がり、文字が重ならない
              class = as.character(cat)) %>%
    arrange(strain, Mb)

# ★ 縦に長くなりすぎるので、行を横に分割して並べる（1ブロック最大 24 行）
NROW_BLK = 24
nblk = ceiling(nrow(tbl) / NROW_BLK)
blks = split(1:nrow(tbl), ceiling(seq_along(1:nrow(tbl)) / ceiling(nrow(tbl)/nblk)))
cat(sprintf("\n表: %d 行 → %d ブロックに分けて横に並べる\n", nrow(tbl), length(blks)))

# ★ この図（地図＋表）だけ縦横 1.125 倍で出す（pathview の再掲は等倍のまま）
SCALE = 1.125
W_FIG = 22 * SCALE

# ★ 表のフォントは経路ごとに自動で決める。
#   列の幅は中身の文字数で決まるので、ko03010（112 遺伝子）のように行が多い経路では
#   ブロック数が増え、1ブロックに割ける幅が狭くなる。
#   固定の 6.2 pt のままだと表が割り当て幅をはみ出し、右端の class 列が切れてしまう。
#   → 実際に表を組んで幅を測り、収まる一番大きいフォントを選ぶ
#
# ★ 文字の幅は「フォントサイズに比例」しない。
#   小さいサイズでは hinting でカクカク変わる（実測: 5.5 pt で 4.75 inch、
#   5.9 pt で 5.04 inch）ので、計算で逆算せず 0.1 pt ずつ下げて実測で確かめる。
# ★ 測るときの device を間違えないこと。文字幅は device のフォント metrics で決まる。
#   実測（同じ表・同じ 5.9 pt）: pdf 4.70 / cairo png 4.81 / ragg 5.04 inch。
#   ggplot2 3.5 の ggsave は ragg があれば ragg を使うので、一番広く出る ragg で測る。
#   （Jupyter の画面表示は cairo png なので、ragg に合わせておけばどちらでも溢れない）
W_BLK = W_FIG / length(blks)             # 1ブロックに割ける幅 (inch)
ragg::agg_png(tempfile(fileext = '.png'), width = 1000, height = 1000, res = 100)
for (fs_try in seq(7.0, 3.4, by = -0.1)) {
    w_max = 0
    for (b in seq_along(blks)) {
        tg0 = gridExtra::tableGrob(
                  tbl[blks[[b]], ], rows = NULL,
                  theme = ttheme(base_size = fs_try, padding = unit(c(2.2, 1.6), 'mm'),
                                 colnames.style = colnames_style(size = fs_try + 0.4),
                                 tbody.style = tbody_style(size = fs_try)))
        w_max = max(w_max, sum(convertWidth(tg0$widths, 'in', valueOnly = TRUE)))
    }
    FS = fs_try
    if (w_max <= W_BLK * 0.98) break     # 0.98 は plot.margin のぶんの余裕
}
invisible(dev.off())
cat(sprintf("表のフォント: %.1f pt（最も広いブロック %.2f inch / 1ブロックの割り当て %.2f inch）\n",
            FS, w_max, W_BLK))

# ★ 表の文字色を地図と揃える（ORA hit = 橙 / DEG only = 紫 / not DEG = 黒）。
#   ggtexttable（gridExtra の tableGrob）は色を ★ 列方向（column-major）に展開する ので、
#   行ごとの色ベクトルを列数ぶん「繰り返して」渡す（rep(col, ncol) であって each= ではない。
#   each= にすると列ごとに1色になる。実測で確認）
tg = lapply(seq_along(blks), function(b) {
    ix  = blks[[b]]
    col = unname(col_cat[tbl$class[ix]])
    g = ggtexttable(tbl[ix, ], rows = NULL,
                theme = ttheme(base_size = FS, padding = unit(c(2.2, 1.6), 'mm'),
                               colnames.style = colnames_style(size = FS + 0.4, fill = 'grey92'),
                               tbody.style = tbody_style(size = FS, fill = 'white',
                                                         color = rep(col, ncol(tbl)))))
    g = g + theme(plot.margin = margin(0, 2, 2, 2))
    if (b == 1) g = g + labs(title = 'B.  gene list') +
        theme(plot.title = element_text(size = 13, face = 'bold', hjust = 0,
                                        margin = margin(b = 2)))
    g
})
TB = wrap_plots(tg, nrow = 1)

# ★ 表（ggtexttable）はパネルの中央に置かれるので、割り当てた高さが余ると
#   上下に空白ができる。行数から必要な高さを詰めて見積もる
H_map = 6.4
H_tab = 0.125 * max(sapply(blks, length)) + 0.35
# ★ tag_levels は使わない。patchwork の全サブプロット（contig パネル10個 + 表3枚）に
#   A〜M と振られてしまうため。見出しは A の図タイトルと B の表タイトルで示す。
#
# ★ wrap_elements() で包む。
#   patchwork を入れ子（P / TB）にすると ★ 内側の plot_annotation が描画されず、
#   A のタイトルが消える。包むと内側が1つの要素として確定し、タイトルが残る
PB = (wrap_elements(full = P) / TB) + plot_layout(heights = c(H_map, H_tab))

# SCALE / W_FIG は表のフォントを決めるときに上で定義してある
suppressWarnings(suppressMessages(
    ggsave(file.path(dir_gp, sprintf('genomepos_%s_%s.png', ko, SCHEME)), PB,
           width = W_FIG, height = (H_map + H_tab) * SCALE,
           dpi = 100, limitsize = FALSE)))
options(repr.plot.width = W_FIG, repr.plot.height = (H_map + H_tab) * SCALE)
suppressWarnings(print(PB))



(1) pathview - all genes of the pathway  [ko00020.B_all.png]
(2) pathview - ORA geneID only  [ko00020.B_ORA.png]


plot without title

ko00020_Citrate cycle (TCA cycle)（TCA回路）  contig 10 本（主 4 / 副 6）
   strain         contig  CDS 長さMb 描画幅
  Bth_VPI    NC_004663.1 1676  6.243   6.24
  Bth_VPI    NC_004703.1   36  0.033   0.60
  Bth_St8 BQNN01000001.1 1780  6.132   6.13
  Bth_St8 BQNN01000002.1   41  0.045   0.60
  Bth_St8 BQNN01000005.1    7  0.004   0.60
  Bth_St8 BQNN01000004.1    2  0.004   0.60
 Bun_8492  NZ_CP102263.1 3392  4.629   4.63
 Odo_St23 BQOE01000001.1 2801  5.647   5.65
 Odo_St23 BQOE01000002.1   38  0.065   0.60
 Odo_St23 BQOE01000003.1   12  0.032   0.60

経路の全遺伝子 61 / 図に出る 61（padj が NA で検定されなかったものも含めて表示）
凡例を出すパネル: NZ_CP102263.1（3 水準）

表: 61 行 → 3 ブロックに分けて横に並べる
表のフォント: 7.0 pt（最も広いブロック 7.19 inch / 1ブロックの割り当て 8.25 inch）


plot without title

plot without title

In [124]:
# ── 図13-10: ko00710（炭素固定(Calvin)）  scheme B ───────────────────
# セル13（準備）を先に実行しておくこと（col_cat / strains4 / MINW / dir_pv / dir_gp を使う）
#
# ★ 3つを続けて出す
#   (1) pathview（経路の全遺伝子）… 反応と化合物名。経路の中での位置づけ
#   (2) pathview（ORA の geneID のみ）… ORA が拾った反応だけ
#   (3) ゲノム位置マップ ＋ 遺伝子表 … どこにあるか、何の遺伝子か
library(ggplot2); library(ggtext); library(ggrepel); library(patchwork); library(ggpubr)
library(png); library(grid)

ko = 'ko00710'; ko_jp = '炭素固定(Calvin)'; SCHEME = 'B'

# ── (1)(2) セル11 で作った pathview を再掲する ────────────────
# ★ 化合物名や反応の向きは pathview にしか無いので、地図の前に置く。
#   ORA で有意でない組み合わせ（例: ko03010 の scheme B）はファイルが無いので飛ばす
for (kind in c('all', 'ORA')) {
    f = file.path(dir_pv, sprintf('%s.%s_%s.png', ko, SCHEME, kind))
    tag = if (kind == 'all') '(1) pathview - all genes of the pathway' else
                             '(2) pathview - ORA geneID only'
    if (!file.exists(f)) {
        cat(sprintf("%s → ファイルなし（セル11 未実行、またはこのスキームの ORA で有意でない）\n", tag))
        next
    }
    # ★ セル11 の検品で落ちた図（ko00121）は中身が書かれず 0 バイトで残る。
    #   file.exists() は TRUE になるので、大きさも見ないと readPNG がエラーで止まる
    if (file.size(f) == 0) {
        cat(sprintf("%s → pathview が描けなかった（セル11 の検品で「★ 描画が壊れた」）\n", tag))
        next
    }
    im = readPNG(f); h_px = dim(im)[1]; w_px = dim(im)[2]
    W = 15; H = W * h_px / w_px
    if (H > 18) { H = 18; W = H * w_px / h_px }
    cat(sprintf("%s  [%s]\n", tag, basename(f)))
    options(repr.plot.width = W, repr.plot.height = H)
    grid.newpage(); grid.raster(im)
}

# ── (3) ゲノム位置マップ ＋ 遺伝子表 ──────────────────────────
tgt = ora$term[ora$source=='KEGG' & grepl(paste0('^',ko,'_'), ora$term)][1]
g_ora = ora %>% filter(scheme==SCHEME, source=='KEGG', term==tgt, padj<0.05) %>%
        pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
        dplyr::select(gene_id, KEGG_Pathway) %>% tidyr::separate_rows(KEGG_Pathway, sep=',') %>%
        filter(KEGG_Pathway==ko) %>% pull(gene_id) %>% unique()

# contig を CDS 数の多い順に並べる（主 → 副）
ctg = res %>% filter(biotype=='CDS', strain %in% strains4, !is.na(end)) %>%
    group_by(strain, contig) %>% summarise(CDS=dplyr::n(), 長さMb=max(end)/1e6, .groups='drop') %>%
    arrange(factor(strain, levels=strains4), desc(CDS)) %>%
    # ★ 描画幅は最小値で底上げするが、x 軸は contig の実際の長さに合わせる。
    #   → 副 contig は bp あたりの縮尺が主より大きくなり、パネルを目一杯使う
    mutate(幅 = pmax(長さMb, MINW),
           短名 = sub('^[^~]+~', '', contig),
           # ★ contig 名は空白が無いので str_wrap では折れない。
           #   パネルが狭いと重なるので、8文字ごとに改行を入れて縦に積む（文字は削らない）
           名表示 = gsub('(.{10})', '\\1\n', 短名))

cat(sprintf("%s（%s）  contig %d 本（主 4 / 副 %d）\n", tgt, ko_jp, nrow(ctg), nrow(ctg)-4))
print(ctg %>% transmute(strain, contig=短名, CDS, 長さMb=round(長さMb,3), 描画幅=round(幅,2)) %>%
      as.data.frame(), row.names=FALSE)

bg = res %>% filter(biotype=='CDS', !marker, !is.na(start)) %>%
     mutate(位置Mb = start/1e6)
fg = bg %>% filter(gene_id %in% g_all) %>%
     mutate(cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                            !is.na(padj_pooled) & padj_pooled < 0.05 ~ 'DEG only', TRUE ~ 'not DEG'),
            cat = factor(cat, levels=names(col_cat)),
            # ★ 丸の塗りに発現強度（WT 群の正規化カウント平均）を入れる。
            #   発現量は3〜4桁にわたるので log10 にする（0 があるので +1）
            発現 = log10(normB_WT + 1),
            label = ifelse(is.na(Preferred_name)|Preferred_name=='-', substr(product,1,10), Preferred_name))
cat(sprintf("\n経路の全遺伝子 %d / 図に出る %d（padj が NA で検定されなかったものも含めて表示）\n",
    length(g_all), nrow(fg)))

ylim = range(c(bg$LFC_B[bg$strain %in% strains4]), na.rm=TRUE)

# ★ 凡例は「3分類が最もそろって現れる contig」1枚からだけ出す。
#   drop=FALSE を付けても ★ データが無い水準はキーの点が描かれないため、
#   パネルごとに凡例の見た目が変わり、guides='collect' が別物と判定して2つ並ぶ
leg_i = which.max(sapply(ctg$contig, function(cc)
            length(unique(fg$cat[fg$contig == cc]))))
cat(sprintf("凡例を出すパネル: %s（%d 水準）\n",
            ctg$短名[leg_i], length(unique(fg$cat[fg$contig == ctg$contig[leg_i]]))))

subs = list()
for (i in 1:nrow(ctg)) {
    b = bg %>% filter(contig == ctg$contig[i])
    f = fg %>% filter(contig == ctg$contig[i])
    is_main = ctg$CDS[i] == max(ctg$CDS[ctg$strain == ctg$strain[i]])
    p = ggplot(b, aes(x=位置Mb, y=LFC_B)) +
        geom_hline(yintercept=0, colour='grey20', linewidth=0.3) +
        geom_hline(yintercept=c(-1,1), linetype='dashed', colour='grey65', linewidth=0.3) +
        geom_point(colour='#b9c6d4', size=0.5, alpha=0.6) +
        coord_cartesian(xlim=c(0, ctg$長さMb[i]), ylim=ylim) +
        labs(title = sprintf('%s\n%s\n%d CDS, %.3f Mb',
                             ifelse(is_main, as.character(ctg$strain[i]), ''),
                             ctg$名表示[i], ctg$CDS[i], ctg$長さMb[i]),
             x=NULL, y=if (i==1) 'log2 fold change (KO / WT), scheme B' else NULL) +
        theme_bw(base_size=10) +
        theme(plot.title=element_text(size=ifelse(is_main, 9, 5.5),
                                    face=ifelse(is_main,'bold','plain'), lineheight=1.15),
              panel.grid.minor=element_blank(),
              axis.text.y=if (i==1) element_text() else element_blank(),
              axis.ticks.y=if (i==1) element_line() else element_blank(),
              legend.position='bottom', legend.box='horizontal',
              legend.title=element_text(size=8), legend.text=element_text(size=7.5),
              legend.margin=margin(1,1,1,1),
              plot.margin=margin(2,2,0,2))
    # ★ 経路の遺伝子が無い contig（0 行）では colour の水準が1つも現れないため、
    #   scale_colour_manual が「No shared levels found」の警告を出す。
    #   図は正しく描かれる（空パネルに色が無いだけ）ので、描画時に警告だけ抑える
    # ★ shape 21（縁と塗りを別々に指定できる丸）を使う。
    #   縁 = 分類（ORA hit / DEG only / not DEG）、塗り = 発現強度
    p = p + geom_point(data=f, aes(colour=cat, fill=発現), shape=21,
                       size=3.2, stroke=1.1, show.legend = (i == leg_i))
    if (nrow(f) > 0)
        p = p +
                # ★ 遺伝子名は小さいと読めないので大きめに。
                #   重なりは ggrepel の force と box.padding で逃がす
                geom_text_repel(data=f, aes(label=label, colour=cat), size=4.4,
                                fontface='bold', max.overlaps=60, seed=1,
                                force=3, min.segment.length=0,
                                segment.size=0.3, box.padding=0.45, show.legend=FALSE)
    # ★ 色の scale は「各パネルに」付ける。
    #   patchwork の guides='collect' は収集時点のパネルごとの凡例を比べるので、
    #   後から & で当てると収集に間に合わず凡例が2つ残る。
    #   全パネルが colour の aes を持ち、drop=FALSE で水準が揃うので1つにまとまる
    subs[[i]] = p + scale_colour_manual(values=col_cat, name=NULL, drop=FALSE,
                                        guide = guide_legend(override.aes = list(
                                            fill='white', size=3.2, stroke=1.1))) +
        # ★ 凡例を下に横向きで出す。右に置くと凡例の文字幅ぶん地図が狭くなる
        scale_fill_viridis_c(option='mako', direction=-1,
                             limits=range(fg$発現, na.rm=TRUE),
                             name='log10(DESeq2 normalized count + 1), WT mean',
                             guide = guide_colourbar(direction='horizontal',
                                                     title.position='top',
                                                     barwidth=unit(5,'cm'),
                                                     barheight=unit(0.35,'cm')))
}
P = wrap_plots(subs, nrow=1, widths=ctg$幅) + plot_layout(guides='collect') +
    plot_annotation(title = sprintf('A.  %s   [scheme %s]  - position on the genome', tgt, SCHEME),
      subtitle = paste0("grey = all CDS  |  coloured = genes of this pathway  ",
                        "(<span style='color:#d95f02'>**ORA hit**</span> / ",
                        "<span style='color:#7570b3'>**DEG only**</span> / ",
                        "<span style='color:#000000'>**not DEG**</span>)<br>",
                        "**gene labels = Preferred_name** (eggNOG; the first 10 characters of *product* ",
                        "are shown when Preferred_name is absent)<br>",
                        "point border = class, **point fill = log10(DESeq2 normalized count + 1), ",
                        "WT group mean** (scheme B: one size factor set for all 4 strains pooled)<br>",
                        "all contigs shown; panel width ∝ contig length, ",
                        sprintf("small contigs get a minimum width (%.1f Mb equivalent) and are drawn at a larger bp scale  |  x axis = position within each contig (Mb)", MINW)),

      # ★ 収集した凡例の位置は patchwork 側の theme で決まる。
      #   サブプロットの theme に legend.position を書いても効かない
      theme = theme(plot.title=element_text(size=14, face='bold'),
                    plot.subtitle=element_markdown(size=9, lineheight=1.3),
                    legend.position='bottom', legend.box='horizontal',
                    legend.title=element_text(size=8.5),
                    legend.text=element_text(size=8)))
# ── 表を同じ図の中に入れる（panel B）──────────────────────────
# ★ 図だけでは locus_tag / K番号 / EC が分からないので表を下に並べる。
#   ggpubr::ggtexttable は表を ggplot として返すので patchwork に載せられる
library(ggpubr)

tbl = fg %>%
    mutate(K  = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                       function(v) paste(v, collapse=',')),
           EC2 = ifelse(is.na(EC), '-', EC)) %>%
    transmute(strain, contig = sub('^[^~]+~', '', contig),
              Mb = round(位置Mb, 3), gene_id, name = label,
              K = K, EC = EC2, product = substr(product, 1, 26),
              # ★ LFC と padj は表に入れない。
              #   LFC は上の地図の y 軸そのもの、padj は分類（class）の色で示している。
              #   列を減らしたぶん1列あたりの幅が広がり、文字が重ならない
              class = as.character(cat)) %>%
    arrange(strain, Mb)

# ★ 縦に長くなりすぎるので、行を横に分割して並べる（1ブロック最大 24 行）
NROW_BLK = 24
nblk = ceiling(nrow(tbl) / NROW_BLK)
blks = split(1:nrow(tbl), ceiling(seq_along(1:nrow(tbl)) / ceiling(nrow(tbl)/nblk)))
cat(sprintf("\n表: %d 行 → %d ブロックに分けて横に並べる\n", nrow(tbl), length(blks)))

# ★ この図（地図＋表）だけ縦横 1.125 倍で出す（pathview の再掲は等倍のまま）
SCALE = 1.125
W_FIG = 22 * SCALE

# ★ 表のフォントは経路ごとに自動で決める。
#   列の幅は中身の文字数で決まるので、ko03010（112 遺伝子）のように行が多い経路では
#   ブロック数が増え、1ブロックに割ける幅が狭くなる。
#   固定の 6.2 pt のままだと表が割り当て幅をはみ出し、右端の class 列が切れてしまう。
#   → 実際に表を組んで幅を測り、収まる一番大きいフォントを選ぶ
#
# ★ 文字の幅は「フォントサイズに比例」しない。
#   小さいサイズでは hinting でカクカク変わる（実測: 5.5 pt で 4.75 inch、
#   5.9 pt で 5.04 inch）ので、計算で逆算せず 0.1 pt ずつ下げて実測で確かめる。
# ★ 測るときの device を間違えないこと。文字幅は device のフォント metrics で決まる。
#   実測（同じ表・同じ 5.9 pt）: pdf 4.70 / cairo png 4.81 / ragg 5.04 inch。
#   ggplot2 3.5 の ggsave は ragg があれば ragg を使うので、一番広く出る ragg で測る。
#   （Jupyter の画面表示は cairo png なので、ragg に合わせておけばどちらでも溢れない）
W_BLK = W_FIG / length(blks)             # 1ブロックに割ける幅 (inch)
ragg::agg_png(tempfile(fileext = '.png'), width = 1000, height = 1000, res = 100)
for (fs_try in seq(7.0, 3.4, by = -0.1)) {
    w_max = 0
    for (b in seq_along(blks)) {
        tg0 = gridExtra::tableGrob(
                  tbl[blks[[b]], ], rows = NULL,
                  theme = ttheme(base_size = fs_try, padding = unit(c(2.2, 1.6), 'mm'),
                                 colnames.style = colnames_style(size = fs_try + 0.4),
                                 tbody.style = tbody_style(size = fs_try)))
        w_max = max(w_max, sum(convertWidth(tg0$widths, 'in', valueOnly = TRUE)))
    }
    FS = fs_try
    if (w_max <= W_BLK * 0.98) break     # 0.98 は plot.margin のぶんの余裕
}
invisible(dev.off())
cat(sprintf("表のフォント: %.1f pt（最も広いブロック %.2f inch / 1ブロックの割り当て %.2f inch）\n",
            FS, w_max, W_BLK))

# ★ 表の文字色を地図と揃える（ORA hit = 橙 / DEG only = 紫 / not DEG = 黒）。
#   ggtexttable（gridExtra の tableGrob）は色を ★ 列方向（column-major）に展開する ので、
#   行ごとの色ベクトルを列数ぶん「繰り返して」渡す（rep(col, ncol) であって each= ではない。
#   each= にすると列ごとに1色になる。実測で確認）
tg = lapply(seq_along(blks), function(b) {
    ix  = blks[[b]]
    col = unname(col_cat[tbl$class[ix]])
    g = ggtexttable(tbl[ix, ], rows = NULL,
                theme = ttheme(base_size = FS, padding = unit(c(2.2, 1.6), 'mm'),
                               colnames.style = colnames_style(size = FS + 0.4, fill = 'grey92'),
                               tbody.style = tbody_style(size = FS, fill = 'white',
                                                         color = rep(col, ncol(tbl)))))
    g = g + theme(plot.margin = margin(0, 2, 2, 2))
    if (b == 1) g = g + labs(title = 'B.  gene list') +
        theme(plot.title = element_text(size = 13, face = 'bold', hjust = 0,
                                        margin = margin(b = 2)))
    g
})
TB = wrap_plots(tg, nrow = 1)

# ★ 表（ggtexttable）はパネルの中央に置かれるので、割り当てた高さが余ると
#   上下に空白ができる。行数から必要な高さを詰めて見積もる
H_map = 6.4
H_tab = 0.125 * max(sapply(blks, length)) + 0.35
# ★ tag_levels は使わない。patchwork の全サブプロット（contig パネル10個 + 表3枚）に
#   A〜M と振られてしまうため。見出しは A の図タイトルと B の表タイトルで示す。
#
# ★ wrap_elements() で包む。
#   patchwork を入れ子（P / TB）にすると ★ 内側の plot_annotation が描画されず、
#   A のタイトルが消える。包むと内側が1つの要素として確定し、タイトルが残る
PB = (wrap_elements(full = P) / TB) + plot_layout(heights = c(H_map, H_tab))

# SCALE / W_FIG は表のフォントを決めるときに上で定義してある
suppressWarnings(suppressMessages(
    ggsave(file.path(dir_gp, sprintf('genomepos_%s_%s.png', ko, SCHEME)), PB,
           width = W_FIG, height = (H_map + H_tab) * SCALE,
           dpi = 100, limitsize = FALSE)))
options(repr.plot.width = W_FIG, repr.plot.height = (H_map + H_tab) * SCALE)
suppressWarnings(print(PB))



(1) pathview - all genes of the pathway  [ko00710.B_all.png]
(2) pathview - ORA geneID only  [ko00710.B_ORA.png]


plot without title

ko00710_Carbon fixation by Calvin cycle（炭素固定(Calvin)）  contig 10 本（主 4 / 副 6）
   strain         contig  CDS 長さMb 描画幅
  Bth_VPI    NC_004663.1 1676  6.243   6.24
  Bth_VPI    NC_004703.1   36  0.033   0.60
  Bth_St8 BQNN01000001.1 1780  6.132   6.13
  Bth_St8 BQNN01000002.1   41  0.045   0.60
  Bth_St8 BQNN01000005.1    7  0.004   0.60
  Bth_St8 BQNN01000004.1    2  0.004   0.60
 Bun_8492  NZ_CP102263.1 3392  4.629   4.63
 Odo_St23 BQOE01000001.1 2801  5.647   5.65
 Odo_St23 BQOE01000002.1   38  0.065   0.60
 Odo_St23 BQOE01000003.1   12  0.032   0.60

経路の全遺伝子 40 / 図に出る 40（padj が NA で検定されなかったものも含めて表示）
凡例を出すパネル: NZ_CP102263.1（3 水準）

表: 40 行 → 2 ブロックに分けて横に並べる
表のフォント: 7.0 pt（最も広いブロック 7.27 inch / 1ブロックの割り当て 12.38 inch）


plot without title

plot without title

In [125]:
# ── 図13-11: ko00720（その他の炭素固定）  scheme B ───────────────────
# セル13（準備）を先に実行しておくこと（col_cat / strains4 / MINW / dir_pv / dir_gp を使う）
#
# ★ 3つを続けて出す
#   (1) pathview（経路の全遺伝子）… 反応と化合物名。経路の中での位置づけ
#   (2) pathview（ORA の geneID のみ）… ORA が拾った反応だけ
#   (3) ゲノム位置マップ ＋ 遺伝子表 … どこにあるか、何の遺伝子か
library(ggplot2); library(ggtext); library(ggrepel); library(patchwork); library(ggpubr)
library(png); library(grid)

ko = 'ko00720'; ko_jp = 'その他の炭素固定'; SCHEME = 'B'

# ── (1)(2) セル11 で作った pathview を再掲する ────────────────
# ★ 化合物名や反応の向きは pathview にしか無いので、地図の前に置く。
#   ORA で有意でない組み合わせ（例: ko03010 の scheme B）はファイルが無いので飛ばす
for (kind in c('all', 'ORA')) {
    f = file.path(dir_pv, sprintf('%s.%s_%s.png', ko, SCHEME, kind))
    tag = if (kind == 'all') '(1) pathview - all genes of the pathway' else
                             '(2) pathview - ORA geneID only'
    if (!file.exists(f)) {
        cat(sprintf("%s → ファイルなし（セル11 未実行、またはこのスキームの ORA で有意でない）\n", tag))
        next
    }
    # ★ セル11 の検品で落ちた図（ko00121）は中身が書かれず 0 バイトで残る。
    #   file.exists() は TRUE になるので、大きさも見ないと readPNG がエラーで止まる
    if (file.size(f) == 0) {
        cat(sprintf("%s → pathview が描けなかった（セル11 の検品で「★ 描画が壊れた」）\n", tag))
        next
    }
    im = readPNG(f); h_px = dim(im)[1]; w_px = dim(im)[2]
    W = 15; H = W * h_px / w_px
    if (H > 18) { H = 18; W = H * w_px / h_px }
    cat(sprintf("%s  [%s]\n", tag, basename(f)))
    options(repr.plot.width = W, repr.plot.height = H)
    grid.newpage(); grid.raster(im)
}

# ── (3) ゲノム位置マップ ＋ 遺伝子表 ──────────────────────────
tgt = ora$term[ora$source=='KEGG' & grepl(paste0('^',ko,'_'), ora$term)][1]
g_ora = ora %>% filter(scheme==SCHEME, source=='KEGG', term==tgt, padj<0.05) %>%
        pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
        dplyr::select(gene_id, KEGG_Pathway) %>% tidyr::separate_rows(KEGG_Pathway, sep=',') %>%
        filter(KEGG_Pathway==ko) %>% pull(gene_id) %>% unique()

# contig を CDS 数の多い順に並べる（主 → 副）
ctg = res %>% filter(biotype=='CDS', strain %in% strains4, !is.na(end)) %>%
    group_by(strain, contig) %>% summarise(CDS=dplyr::n(), 長さMb=max(end)/1e6, .groups='drop') %>%
    arrange(factor(strain, levels=strains4), desc(CDS)) %>%
    # ★ 描画幅は最小値で底上げするが、x 軸は contig の実際の長さに合わせる。
    #   → 副 contig は bp あたりの縮尺が主より大きくなり、パネルを目一杯使う
    mutate(幅 = pmax(長さMb, MINW),
           短名 = sub('^[^~]+~', '', contig),
           # ★ contig 名は空白が無いので str_wrap では折れない。
           #   パネルが狭いと重なるので、8文字ごとに改行を入れて縦に積む（文字は削らない）
           名表示 = gsub('(.{10})', '\\1\n', 短名))

cat(sprintf("%s（%s）  contig %d 本（主 4 / 副 %d）\n", tgt, ko_jp, nrow(ctg), nrow(ctg)-4))
print(ctg %>% transmute(strain, contig=短名, CDS, 長さMb=round(長さMb,3), 描画幅=round(幅,2)) %>%
      as.data.frame(), row.names=FALSE)

bg = res %>% filter(biotype=='CDS', !marker, !is.na(start)) %>%
     mutate(位置Mb = start/1e6)
fg = bg %>% filter(gene_id %in% g_all) %>%
     mutate(cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                            !is.na(padj_pooled) & padj_pooled < 0.05 ~ 'DEG only', TRUE ~ 'not DEG'),
            cat = factor(cat, levels=names(col_cat)),
            # ★ 丸の塗りに発現強度（WT 群の正規化カウント平均）を入れる。
            #   発現量は3〜4桁にわたるので log10 にする（0 があるので +1）
            発現 = log10(normB_WT + 1),
            label = ifelse(is.na(Preferred_name)|Preferred_name=='-', substr(product,1,10), Preferred_name))
cat(sprintf("\n経路の全遺伝子 %d / 図に出る %d（padj が NA で検定されなかったものも含めて表示）\n",
    length(g_all), nrow(fg)))

ylim = range(c(bg$LFC_B[bg$strain %in% strains4]), na.rm=TRUE)

# ★ 凡例は「3分類が最もそろって現れる contig」1枚からだけ出す。
#   drop=FALSE を付けても ★ データが無い水準はキーの点が描かれないため、
#   パネルごとに凡例の見た目が変わり、guides='collect' が別物と判定して2つ並ぶ
leg_i = which.max(sapply(ctg$contig, function(cc)
            length(unique(fg$cat[fg$contig == cc]))))
cat(sprintf("凡例を出すパネル: %s（%d 水準）\n",
            ctg$短名[leg_i], length(unique(fg$cat[fg$contig == ctg$contig[leg_i]]))))

subs = list()
for (i in 1:nrow(ctg)) {
    b = bg %>% filter(contig == ctg$contig[i])
    f = fg %>% filter(contig == ctg$contig[i])
    is_main = ctg$CDS[i] == max(ctg$CDS[ctg$strain == ctg$strain[i]])
    p = ggplot(b, aes(x=位置Mb, y=LFC_B)) +
        geom_hline(yintercept=0, colour='grey20', linewidth=0.3) +
        geom_hline(yintercept=c(-1,1), linetype='dashed', colour='grey65', linewidth=0.3) +
        geom_point(colour='#b9c6d4', size=0.5, alpha=0.6) +
        coord_cartesian(xlim=c(0, ctg$長さMb[i]), ylim=ylim) +
        labs(title = sprintf('%s\n%s\n%d CDS, %.3f Mb',
                             ifelse(is_main, as.character(ctg$strain[i]), ''),
                             ctg$名表示[i], ctg$CDS[i], ctg$長さMb[i]),
             x=NULL, y=if (i==1) 'log2 fold change (KO / WT), scheme B' else NULL) +
        theme_bw(base_size=10) +
        theme(plot.title=element_text(size=ifelse(is_main, 9, 5.5),
                                    face=ifelse(is_main,'bold','plain'), lineheight=1.15),
              panel.grid.minor=element_blank(),
              axis.text.y=if (i==1) element_text() else element_blank(),
              axis.ticks.y=if (i==1) element_line() else element_blank(),
              legend.position='bottom', legend.box='horizontal',
              legend.title=element_text(size=8), legend.text=element_text(size=7.5),
              legend.margin=margin(1,1,1,1),
              plot.margin=margin(2,2,0,2))
    # ★ 経路の遺伝子が無い contig（0 行）では colour の水準が1つも現れないため、
    #   scale_colour_manual が「No shared levels found」の警告を出す。
    #   図は正しく描かれる（空パネルに色が無いだけ）ので、描画時に警告だけ抑える
    # ★ shape 21（縁と塗りを別々に指定できる丸）を使う。
    #   縁 = 分類（ORA hit / DEG only / not DEG）、塗り = 発現強度
    p = p + geom_point(data=f, aes(colour=cat, fill=発現), shape=21,
                       size=3.2, stroke=1.1, show.legend = (i == leg_i))
    if (nrow(f) > 0)
        p = p +
                # ★ 遺伝子名は小さいと読めないので大きめに。
                #   重なりは ggrepel の force と box.padding で逃がす
                geom_text_repel(data=f, aes(label=label, colour=cat), size=4.4,
                                fontface='bold', max.overlaps=60, seed=1,
                                force=3, min.segment.length=0,
                                segment.size=0.3, box.padding=0.45, show.legend=FALSE)
    # ★ 色の scale は「各パネルに」付ける。
    #   patchwork の guides='collect' は収集時点のパネルごとの凡例を比べるので、
    #   後から & で当てると収集に間に合わず凡例が2つ残る。
    #   全パネルが colour の aes を持ち、drop=FALSE で水準が揃うので1つにまとまる
    subs[[i]] = p + scale_colour_manual(values=col_cat, name=NULL, drop=FALSE,
                                        guide = guide_legend(override.aes = list(
                                            fill='white', size=3.2, stroke=1.1))) +
        # ★ 凡例を下に横向きで出す。右に置くと凡例の文字幅ぶん地図が狭くなる
        scale_fill_viridis_c(option='mako', direction=-1,
                             limits=range(fg$発現, na.rm=TRUE),
                             name='log10(DESeq2 normalized count + 1), WT mean',
                             guide = guide_colourbar(direction='horizontal',
                                                     title.position='top',
                                                     barwidth=unit(5,'cm'),
                                                     barheight=unit(0.35,'cm')))
}
P = wrap_plots(subs, nrow=1, widths=ctg$幅) + plot_layout(guides='collect') +
    plot_annotation(title = sprintf('A.  %s   [scheme %s]  - position on the genome', tgt, SCHEME),
      subtitle = paste0("grey = all CDS  |  coloured = genes of this pathway  ",
                        "(<span style='color:#d95f02'>**ORA hit**</span> / ",
                        "<span style='color:#7570b3'>**DEG only**</span> / ",
                        "<span style='color:#000000'>**not DEG**</span>)<br>",
                        "**gene labels = Preferred_name** (eggNOG; the first 10 characters of *product* ",
                        "are shown when Preferred_name is absent)<br>",
                        "point border = class, **point fill = log10(DESeq2 normalized count + 1), ",
                        "WT group mean** (scheme B: one size factor set for all 4 strains pooled)<br>",
                        "all contigs shown; panel width ∝ contig length, ",
                        sprintf("small contigs get a minimum width (%.1f Mb equivalent) and are drawn at a larger bp scale  |  x axis = position within each contig (Mb)", MINW)),

      # ★ 収集した凡例の位置は patchwork 側の theme で決まる。
      #   サブプロットの theme に legend.position を書いても効かない
      theme = theme(plot.title=element_text(size=14, face='bold'),
                    plot.subtitle=element_markdown(size=9, lineheight=1.3),
                    legend.position='bottom', legend.box='horizontal',
                    legend.title=element_text(size=8.5),
                    legend.text=element_text(size=8)))
# ── 表を同じ図の中に入れる（panel B）──────────────────────────
# ★ 図だけでは locus_tag / K番号 / EC が分からないので表を下に並べる。
#   ggpubr::ggtexttable は表を ggplot として返すので patchwork に載せられる
library(ggpubr)

tbl = fg %>%
    mutate(K  = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                       function(v) paste(v, collapse=',')),
           EC2 = ifelse(is.na(EC), '-', EC)) %>%
    transmute(strain, contig = sub('^[^~]+~', '', contig),
              Mb = round(位置Mb, 3), gene_id, name = label,
              K = K, EC = EC2, product = substr(product, 1, 26),
              # ★ LFC と padj は表に入れない。
              #   LFC は上の地図の y 軸そのもの、padj は分類（class）の色で示している。
              #   列を減らしたぶん1列あたりの幅が広がり、文字が重ならない
              class = as.character(cat)) %>%
    arrange(strain, Mb)

# ★ 縦に長くなりすぎるので、行を横に分割して並べる（1ブロック最大 24 行）
NROW_BLK = 24
nblk = ceiling(nrow(tbl) / NROW_BLK)
blks = split(1:nrow(tbl), ceiling(seq_along(1:nrow(tbl)) / ceiling(nrow(tbl)/nblk)))
cat(sprintf("\n表: %d 行 → %d ブロックに分けて横に並べる\n", nrow(tbl), length(blks)))

# ★ この図（地図＋表）だけ縦横 1.125 倍で出す（pathview の再掲は等倍のまま）
SCALE = 1.125
W_FIG = 22 * SCALE

# ★ 表のフォントは経路ごとに自動で決める。
#   列の幅は中身の文字数で決まるので、ko03010（112 遺伝子）のように行が多い経路では
#   ブロック数が増え、1ブロックに割ける幅が狭くなる。
#   固定の 6.2 pt のままだと表が割り当て幅をはみ出し、右端の class 列が切れてしまう。
#   → 実際に表を組んで幅を測り、収まる一番大きいフォントを選ぶ
#
# ★ 文字の幅は「フォントサイズに比例」しない。
#   小さいサイズでは hinting でカクカク変わる（実測: 5.5 pt で 4.75 inch、
#   5.9 pt で 5.04 inch）ので、計算で逆算せず 0.1 pt ずつ下げて実測で確かめる。
# ★ 測るときの device を間違えないこと。文字幅は device のフォント metrics で決まる。
#   実測（同じ表・同じ 5.9 pt）: pdf 4.70 / cairo png 4.81 / ragg 5.04 inch。
#   ggplot2 3.5 の ggsave は ragg があれば ragg を使うので、一番広く出る ragg で測る。
#   （Jupyter の画面表示は cairo png なので、ragg に合わせておけばどちらでも溢れない）
W_BLK = W_FIG / length(blks)             # 1ブロックに割ける幅 (inch)
ragg::agg_png(tempfile(fileext = '.png'), width = 1000, height = 1000, res = 100)
for (fs_try in seq(7.0, 3.4, by = -0.1)) {
    w_max = 0
    for (b in seq_along(blks)) {
        tg0 = gridExtra::tableGrob(
                  tbl[blks[[b]], ], rows = NULL,
                  theme = ttheme(base_size = fs_try, padding = unit(c(2.2, 1.6), 'mm'),
                                 colnames.style = colnames_style(size = fs_try + 0.4),
                                 tbody.style = tbody_style(size = fs_try)))
        w_max = max(w_max, sum(convertWidth(tg0$widths, 'in', valueOnly = TRUE)))
    }
    FS = fs_try
    if (w_max <= W_BLK * 0.98) break     # 0.98 は plot.margin のぶんの余裕
}
invisible(dev.off())
cat(sprintf("表のフォント: %.1f pt（最も広いブロック %.2f inch / 1ブロックの割り当て %.2f inch）\n",
            FS, w_max, W_BLK))

# ★ 表の文字色を地図と揃える（ORA hit = 橙 / DEG only = 紫 / not DEG = 黒）。
#   ggtexttable（gridExtra の tableGrob）は色を ★ 列方向（column-major）に展開する ので、
#   行ごとの色ベクトルを列数ぶん「繰り返して」渡す（rep(col, ncol) であって each= ではない。
#   each= にすると列ごとに1色になる。実測で確認）
tg = lapply(seq_along(blks), function(b) {
    ix  = blks[[b]]
    col = unname(col_cat[tbl$class[ix]])
    g = ggtexttable(tbl[ix, ], rows = NULL,
                theme = ttheme(base_size = FS, padding = unit(c(2.2, 1.6), 'mm'),
                               colnames.style = colnames_style(size = FS + 0.4, fill = 'grey92'),
                               tbody.style = tbody_style(size = FS, fill = 'white',
                                                         color = rep(col, ncol(tbl)))))
    g = g + theme(plot.margin = margin(0, 2, 2, 2))
    if (b == 1) g = g + labs(title = 'B.  gene list') +
        theme(plot.title = element_text(size = 13, face = 'bold', hjust = 0,
                                        margin = margin(b = 2)))
    g
})
TB = wrap_plots(tg, nrow = 1)

# ★ 表（ggtexttable）はパネルの中央に置かれるので、割り当てた高さが余ると
#   上下に空白ができる。行数から必要な高さを詰めて見積もる
H_map = 6.4
H_tab = 0.125 * max(sapply(blks, length)) + 0.35
# ★ tag_levels は使わない。patchwork の全サブプロット（contig パネル10個 + 表3枚）に
#   A〜M と振られてしまうため。見出しは A の図タイトルと B の表タイトルで示す。
#
# ★ wrap_elements() で包む。
#   patchwork を入れ子（P / TB）にすると ★ 内側の plot_annotation が描画されず、
#   A のタイトルが消える。包むと内側が1つの要素として確定し、タイトルが残る
PB = (wrap_elements(full = P) / TB) + plot_layout(heights = c(H_map, H_tab))

# SCALE / W_FIG は表のフォントを決めるときに上で定義してある
suppressWarnings(suppressMessages(
    ggsave(file.path(dir_gp, sprintf('genomepos_%s_%s.png', ko, SCHEME)), PB,
           width = W_FIG, height = (H_map + H_tab) * SCALE,
           dpi = 100, limitsize = FALSE)))
options(repr.plot.width = W_FIG, repr.plot.height = (H_map + H_tab) * SCALE)
suppressWarnings(print(PB))



(1) pathview - all genes of the pathway  [ko00720.B_all.png]
(2) pathview - ORA geneID only  [ko00720.B_ORA.png]


plot without title

ko00720_Other carbon fixation pathways（その他の炭素固定）  contig 10 本（主 4 / 副 6）
   strain         contig  CDS 長さMb 描画幅
  Bth_VPI    NC_004663.1 1676  6.243   6.24
  Bth_VPI    NC_004703.1   36  0.033   0.60
  Bth_St8 BQNN01000001.1 1780  6.132   6.13
  Bth_St8 BQNN01000002.1   41  0.045   0.60
  Bth_St8 BQNN01000005.1    7  0.004   0.60
  Bth_St8 BQNN01000004.1    2  0.004   0.60
 Bun_8492  NZ_CP102263.1 3392  4.629   4.63
 Odo_St23 BQOE01000001.1 2801  5.647   5.65
 Odo_St23 BQOE01000002.1   38  0.065   0.60
 Odo_St23 BQOE01000003.1   12  0.032   0.60

経路の全遺伝子 81 / 図に出る 81（padj が NA で検定されなかったものも含めて表示）
凡例を出すパネル: NC_004663.1（3 水準）

表: 81 行 → 4 ブロックに分けて横に並べる
表のフォント: 5.4 pt（最も広いブロック 6.02 inch / 1ブロックの割り当て 6.19 inch）


plot without title

plot without title

In [126]:
# ── 図13-12: ko00120（一次胆汁酸）  scheme B ───────────────────
# セル13（準備）を先に実行しておくこと（col_cat / strains4 / MINW / dir_pv / dir_gp を使う）
#
# ★ 3つを続けて出す
#   (1) pathview（経路の全遺伝子）… 反応と化合物名。経路の中での位置づけ
#   (2) pathview（ORA の geneID のみ）… ORA が拾った反応だけ
#   (3) ゲノム位置マップ ＋ 遺伝子表 … どこにあるか、何の遺伝子か
library(ggplot2); library(ggtext); library(ggrepel); library(patchwork); library(ggpubr)
library(png); library(grid)

ko = 'ko00120'; ko_jp = '一次胆汁酸'; SCHEME = 'B'

# ── (1)(2) セル11 で作った pathview を再掲する ────────────────
# ★ 化合物名や反応の向きは pathview にしか無いので、地図の前に置く。
#   ORA で有意でない組み合わせ（例: ko03010 の scheme B）はファイルが無いので飛ばす
for (kind in c('all', 'ORA')) {
    f = file.path(dir_pv, sprintf('%s.%s_%s.png', ko, SCHEME, kind))
    tag = if (kind == 'all') '(1) pathview - all genes of the pathway' else
                             '(2) pathview - ORA geneID only'
    if (!file.exists(f)) {
        cat(sprintf("%s → ファイルなし（セル11 未実行、またはこのスキームの ORA で有意でない）\n", tag))
        next
    }
    # ★ セル11 の検品で落ちた図（ko00121）は中身が書かれず 0 バイトで残る。
    #   file.exists() は TRUE になるので、大きさも見ないと readPNG がエラーで止まる
    if (file.size(f) == 0) {
        cat(sprintf("%s → pathview が描けなかった（セル11 の検品で「★ 描画が壊れた」）\n", tag))
        next
    }
    im = readPNG(f); h_px = dim(im)[1]; w_px = dim(im)[2]
    W = 15; H = W * h_px / w_px
    if (H > 18) { H = 18; W = H * w_px / h_px }
    cat(sprintf("%s  [%s]\n", tag, basename(f)))
    options(repr.plot.width = W, repr.plot.height = H)
    grid.newpage(); grid.raster(im)
}

# ── (3) ゲノム位置マップ ＋ 遺伝子表 ──────────────────────────
tgt = ora$term[ora$source=='KEGG' & grepl(paste0('^',ko,'_'), ora$term)][1]
g_ora = ora %>% filter(scheme==SCHEME, source=='KEGG', term==tgt, padj<0.05) %>%
        pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
        dplyr::select(gene_id, KEGG_Pathway) %>% tidyr::separate_rows(KEGG_Pathway, sep=',') %>%
        filter(KEGG_Pathway==ko) %>% pull(gene_id) %>% unique()

# contig を CDS 数の多い順に並べる（主 → 副）
ctg = res %>% filter(biotype=='CDS', strain %in% strains4, !is.na(end)) %>%
    group_by(strain, contig) %>% summarise(CDS=dplyr::n(), 長さMb=max(end)/1e6, .groups='drop') %>%
    arrange(factor(strain, levels=strains4), desc(CDS)) %>%
    # ★ 描画幅は最小値で底上げするが、x 軸は contig の実際の長さに合わせる。
    #   → 副 contig は bp あたりの縮尺が主より大きくなり、パネルを目一杯使う
    mutate(幅 = pmax(長さMb, MINW),
           短名 = sub('^[^~]+~', '', contig),
           # ★ contig 名は空白が無いので str_wrap では折れない。
           #   パネルが狭いと重なるので、8文字ごとに改行を入れて縦に積む（文字は削らない）
           名表示 = gsub('(.{10})', '\\1\n', 短名))

cat(sprintf("%s（%s）  contig %d 本（主 4 / 副 %d）\n", tgt, ko_jp, nrow(ctg), nrow(ctg)-4))
print(ctg %>% transmute(strain, contig=短名, CDS, 長さMb=round(長さMb,3), 描画幅=round(幅,2)) %>%
      as.data.frame(), row.names=FALSE)

bg = res %>% filter(biotype=='CDS', !marker, !is.na(start)) %>%
     mutate(位置Mb = start/1e6)
fg = bg %>% filter(gene_id %in% g_all) %>%
     mutate(cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                            !is.na(padj_pooled) & padj_pooled < 0.05 ~ 'DEG only', TRUE ~ 'not DEG'),
            cat = factor(cat, levels=names(col_cat)),
            # ★ 丸の塗りに発現強度（WT 群の正規化カウント平均）を入れる。
            #   発現量は3〜4桁にわたるので log10 にする（0 があるので +1）
            発現 = log10(normB_WT + 1),
            label = ifelse(is.na(Preferred_name)|Preferred_name=='-', substr(product,1,10), Preferred_name))
cat(sprintf("\n経路の全遺伝子 %d / 図に出る %d（padj が NA で検定されなかったものも含めて表示）\n",
    length(g_all), nrow(fg)))

ylim = range(c(bg$LFC_B[bg$strain %in% strains4]), na.rm=TRUE)

# ★ 凡例は「3分類が最もそろって現れる contig」1枚からだけ出す。
#   drop=FALSE を付けても ★ データが無い水準はキーの点が描かれないため、
#   パネルごとに凡例の見た目が変わり、guides='collect' が別物と判定して2つ並ぶ
leg_i = which.max(sapply(ctg$contig, function(cc)
            length(unique(fg$cat[fg$contig == cc]))))
cat(sprintf("凡例を出すパネル: %s（%d 水準）\n",
            ctg$短名[leg_i], length(unique(fg$cat[fg$contig == ctg$contig[leg_i]]))))

subs = list()
for (i in 1:nrow(ctg)) {
    b = bg %>% filter(contig == ctg$contig[i])
    f = fg %>% filter(contig == ctg$contig[i])
    is_main = ctg$CDS[i] == max(ctg$CDS[ctg$strain == ctg$strain[i]])
    p = ggplot(b, aes(x=位置Mb, y=LFC_B)) +
        geom_hline(yintercept=0, colour='grey20', linewidth=0.3) +
        geom_hline(yintercept=c(-1,1), linetype='dashed', colour='grey65', linewidth=0.3) +
        geom_point(colour='#b9c6d4', size=0.5, alpha=0.6) +
        coord_cartesian(xlim=c(0, ctg$長さMb[i]), ylim=ylim) +
        labs(title = sprintf('%s\n%s\n%d CDS, %.3f Mb',
                             ifelse(is_main, as.character(ctg$strain[i]), ''),
                             ctg$名表示[i], ctg$CDS[i], ctg$長さMb[i]),
             x=NULL, y=if (i==1) 'log2 fold change (KO / WT), scheme B' else NULL) +
        theme_bw(base_size=10) +
        theme(plot.title=element_text(size=ifelse(is_main, 9, 5.5),
                                    face=ifelse(is_main,'bold','plain'), lineheight=1.15),
              panel.grid.minor=element_blank(),
              axis.text.y=if (i==1) element_text() else element_blank(),
              axis.ticks.y=if (i==1) element_line() else element_blank(),
              legend.position='bottom', legend.box='horizontal',
              legend.title=element_text(size=8), legend.text=element_text(size=7.5),
              legend.margin=margin(1,1,1,1),
              plot.margin=margin(2,2,0,2))
    # ★ 経路の遺伝子が無い contig（0 行）では colour の水準が1つも現れないため、
    #   scale_colour_manual が「No shared levels found」の警告を出す。
    #   図は正しく描かれる（空パネルに色が無いだけ）ので、描画時に警告だけ抑える
    # ★ shape 21（縁と塗りを別々に指定できる丸）を使う。
    #   縁 = 分類（ORA hit / DEG only / not DEG）、塗り = 発現強度
    p = p + geom_point(data=f, aes(colour=cat, fill=発現), shape=21,
                       size=3.2, stroke=1.1, show.legend = (i == leg_i))
    if (nrow(f) > 0)
        p = p +
                # ★ 遺伝子名は小さいと読めないので大きめに。
                #   重なりは ggrepel の force と box.padding で逃がす
                geom_text_repel(data=f, aes(label=label, colour=cat), size=4.4,
                                fontface='bold', max.overlaps=60, seed=1,
                                force=3, min.segment.length=0,
                                segment.size=0.3, box.padding=0.45, show.legend=FALSE)
    # ★ 色の scale は「各パネルに」付ける。
    #   patchwork の guides='collect' は収集時点のパネルごとの凡例を比べるので、
    #   後から & で当てると収集に間に合わず凡例が2つ残る。
    #   全パネルが colour の aes を持ち、drop=FALSE で水準が揃うので1つにまとまる
    subs[[i]] = p + scale_colour_manual(values=col_cat, name=NULL, drop=FALSE,
                                        guide = guide_legend(override.aes = list(
                                            fill='white', size=3.2, stroke=1.1))) +
        # ★ 凡例を下に横向きで出す。右に置くと凡例の文字幅ぶん地図が狭くなる
        scale_fill_viridis_c(option='mako', direction=-1,
                             limits=range(fg$発現, na.rm=TRUE),
                             name='log10(DESeq2 normalized count + 1), WT mean',
                             guide = guide_colourbar(direction='horizontal',
                                                     title.position='top',
                                                     barwidth=unit(5,'cm'),
                                                     barheight=unit(0.35,'cm')))
}
P = wrap_plots(subs, nrow=1, widths=ctg$幅) + plot_layout(guides='collect') +
    plot_annotation(title = sprintf('A.  %s   [scheme %s]  - position on the genome', tgt, SCHEME),
      subtitle = paste0("grey = all CDS  |  coloured = genes of this pathway  ",
                        "(<span style='color:#d95f02'>**ORA hit**</span> / ",
                        "<span style='color:#7570b3'>**DEG only**</span> / ",
                        "<span style='color:#000000'>**not DEG**</span>)<br>",
                        "**gene labels = Preferred_name** (eggNOG; the first 10 characters of *product* ",
                        "are shown when Preferred_name is absent)<br>",
                        "point border = class, **point fill = log10(DESeq2 normalized count + 1), ",
                        "WT group mean** (scheme B: one size factor set for all 4 strains pooled)<br>",
                        "all contigs shown; panel width ∝ contig length, ",
                        sprintf("small contigs get a minimum width (%.1f Mb equivalent) and are drawn at a larger bp scale  |  x axis = position within each contig (Mb)", MINW)),

      # ★ 収集した凡例の位置は patchwork 側の theme で決まる。
      #   サブプロットの theme に legend.position を書いても効かない
      theme = theme(plot.title=element_text(size=14, face='bold'),
                    plot.subtitle=element_markdown(size=9, lineheight=1.3),
                    legend.position='bottom', legend.box='horizontal',
                    legend.title=element_text(size=8.5),
                    legend.text=element_text(size=8)))
# ── 表を同じ図の中に入れる（panel B）──────────────────────────
# ★ 図だけでは locus_tag / K番号 / EC が分からないので表を下に並べる。
#   ggpubr::ggtexttable は表を ggplot として返すので patchwork に載せられる
library(ggpubr)

tbl = fg %>%
    mutate(K  = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                       function(v) paste(v, collapse=',')),
           EC2 = ifelse(is.na(EC), '-', EC)) %>%
    transmute(strain, contig = sub('^[^~]+~', '', contig),
              Mb = round(位置Mb, 3), gene_id, name = label,
              K = K, EC = EC2, product = substr(product, 1, 26),
              # ★ LFC と padj は表に入れない。
              #   LFC は上の地図の y 軸そのもの、padj は分類（class）の色で示している。
              #   列を減らしたぶん1列あたりの幅が広がり、文字が重ならない
              class = as.character(cat)) %>%
    arrange(strain, Mb)

# ★ 縦に長くなりすぎるので、行を横に分割して並べる（1ブロック最大 24 行）
NROW_BLK = 24
nblk = ceiling(nrow(tbl) / NROW_BLK)
blks = split(1:nrow(tbl), ceiling(seq_along(1:nrow(tbl)) / ceiling(nrow(tbl)/nblk)))
cat(sprintf("\n表: %d 行 → %d ブロックに分けて横に並べる\n", nrow(tbl), length(blks)))

# ★ この図（地図＋表）だけ縦横 1.125 倍で出す（pathview の再掲は等倍のまま）
SCALE = 1.125
W_FIG = 22 * SCALE

# ★ 表のフォントは経路ごとに自動で決める。
#   列の幅は中身の文字数で決まるので、ko03010（112 遺伝子）のように行が多い経路では
#   ブロック数が増え、1ブロックに割ける幅が狭くなる。
#   固定の 6.2 pt のままだと表が割り当て幅をはみ出し、右端の class 列が切れてしまう。
#   → 実際に表を組んで幅を測り、収まる一番大きいフォントを選ぶ
#
# ★ 文字の幅は「フォントサイズに比例」しない。
#   小さいサイズでは hinting でカクカク変わる（実測: 5.5 pt で 4.75 inch、
#   5.9 pt で 5.04 inch）ので、計算で逆算せず 0.1 pt ずつ下げて実測で確かめる。
# ★ 測るときの device を間違えないこと。文字幅は device のフォント metrics で決まる。
#   実測（同じ表・同じ 5.9 pt）: pdf 4.70 / cairo png 4.81 / ragg 5.04 inch。
#   ggplot2 3.5 の ggsave は ragg があれば ragg を使うので、一番広く出る ragg で測る。
#   （Jupyter の画面表示は cairo png なので、ragg に合わせておけばどちらでも溢れない）
W_BLK = W_FIG / length(blks)             # 1ブロックに割ける幅 (inch)
ragg::agg_png(tempfile(fileext = '.png'), width = 1000, height = 1000, res = 100)
for (fs_try in seq(7.0, 3.4, by = -0.1)) {
    w_max = 0
    for (b in seq_along(blks)) {
        tg0 = gridExtra::tableGrob(
                  tbl[blks[[b]], ], rows = NULL,
                  theme = ttheme(base_size = fs_try, padding = unit(c(2.2, 1.6), 'mm'),
                                 colnames.style = colnames_style(size = fs_try + 0.4),
                                 tbody.style = tbody_style(size = fs_try)))
        w_max = max(w_max, sum(convertWidth(tg0$widths, 'in', valueOnly = TRUE)))
    }
    FS = fs_try
    if (w_max <= W_BLK * 0.98) break     # 0.98 は plot.margin のぶんの余裕
}
invisible(dev.off())
cat(sprintf("表のフォント: %.1f pt（最も広いブロック %.2f inch / 1ブロックの割り当て %.2f inch）\n",
            FS, w_max, W_BLK))

# ★ 表の文字色を地図と揃える（ORA hit = 橙 / DEG only = 紫 / not DEG = 黒）。
#   ggtexttable（gridExtra の tableGrob）は色を ★ 列方向（column-major）に展開する ので、
#   行ごとの色ベクトルを列数ぶん「繰り返して」渡す（rep(col, ncol) であって each= ではない。
#   each= にすると列ごとに1色になる。実測で確認）
tg = lapply(seq_along(blks), function(b) {
    ix  = blks[[b]]
    col = unname(col_cat[tbl$class[ix]])
    g = ggtexttable(tbl[ix, ], rows = NULL,
                theme = ttheme(base_size = FS, padding = unit(c(2.2, 1.6), 'mm'),
                               colnames.style = colnames_style(size = FS + 0.4, fill = 'grey92'),
                               tbody.style = tbody_style(size = FS, fill = 'white',
                                                         color = rep(col, ncol(tbl)))))
    g = g + theme(plot.margin = margin(0, 2, 2, 2))
    if (b == 1) g = g + labs(title = 'B.  gene list') +
        theme(plot.title = element_text(size = 13, face = 'bold', hjust = 0,
                                        margin = margin(b = 2)))
    g
})
TB = wrap_plots(tg, nrow = 1)

# ★ 表（ggtexttable）はパネルの中央に置かれるので、割り当てた高さが余ると
#   上下に空白ができる。行数から必要な高さを詰めて見積もる
H_map = 6.4
H_tab = 0.125 * max(sapply(blks, length)) + 0.35
# ★ tag_levels は使わない。patchwork の全サブプロット（contig パネル10個 + 表3枚）に
#   A〜M と振られてしまうため。見出しは A の図タイトルと B の表タイトルで示す。
#
# ★ wrap_elements() で包む。
#   patchwork を入れ子（P / TB）にすると ★ 内側の plot_annotation が描画されず、
#   A のタイトルが消える。包むと内側が1つの要素として確定し、タイトルが残る
PB = (wrap_elements(full = P) / TB) + plot_layout(heights = c(H_map, H_tab))

# SCALE / W_FIG は表のフォントを決めるときに上で定義してある
suppressWarnings(suppressMessages(
    ggsave(file.path(dir_gp, sprintf('genomepos_%s_%s.png', ko, SCHEME)), PB,
           width = W_FIG, height = (H_map + H_tab) * SCALE,
           dpi = 100, limitsize = FALSE)))
options(repr.plot.width = W_FIG, repr.plot.height = (H_map + H_tab) * SCALE)
suppressWarnings(print(PB))



(1) pathview - all genes of the pathway  [ko00120.B_all.png]
(2) pathview - ORA geneID only  [ko00120.B_ORA.png]


plot without title

ko00120_Primary bile acid biosynthesis（一次胆汁酸）  contig 10 本（主 4 / 副 6）
   strain         contig  CDS 長さMb 描画幅
  Bth_VPI    NC_004663.1 1676  6.243   6.24
  Bth_VPI    NC_004703.1   36  0.033   0.60
  Bth_St8 BQNN01000001.1 1780  6.132   6.13
  Bth_St8 BQNN01000002.1   41  0.045   0.60
  Bth_St8 BQNN01000005.1    7  0.004   0.60
  Bth_St8 BQNN01000004.1    2  0.004   0.60
 Bun_8492  NZ_CP102263.1 3392  4.629   4.63
 Odo_St23 BQOE01000001.1 2801  5.647   5.65
 Odo_St23 BQOE01000002.1   38  0.065   0.60
 Odo_St23 BQOE01000003.1   12  0.032   0.60

経路の全遺伝子 5 / 図に出る 5（padj が NA で検定されなかったものも含めて表示）
凡例を出すパネル: NZ_CP102263.1（2 水準）

表: 5 行 → 1 ブロックに分けて横に並べる
表のフォント: 7.0 pt（最も広いブロック 6.26 inch / 1ブロックの割り当て 24.75 inch）


plot without title

plot without title

In [127]:
# ── 図13-13: ko00121（二次胆汁酸）  scheme B ───────────────────
# セル13（準備）を先に実行しておくこと（col_cat / strains4 / MINW / dir_pv / dir_gp を使う）
#
# ★ 3つを続けて出す
#   (1) pathview（経路の全遺伝子）… 反応と化合物名。経路の中での位置づけ
#   (2) pathview（ORA の geneID のみ）… ORA が拾った反応だけ
#   (3) ゲノム位置マップ ＋ 遺伝子表 … どこにあるか、何の遺伝子か
library(ggplot2); library(ggtext); library(ggrepel); library(patchwork); library(ggpubr)
library(png); library(grid)

ko = 'ko00121'; ko_jp = '二次胆汁酸'; SCHEME = 'B'

# ── (1)(2) セル11 で作った pathview を再掲する ────────────────
# ★ 化合物名や反応の向きは pathview にしか無いので、地図の前に置く。
#   ORA で有意でない組み合わせ（例: ko03010 の scheme B）はファイルが無いので飛ばす
for (kind in c('all', 'ORA')) {
    f = file.path(dir_pv, sprintf('%s.%s_%s.png', ko, SCHEME, kind))
    tag = if (kind == 'all') '(1) pathview - all genes of the pathway' else
                             '(2) pathview - ORA geneID only'
    if (!file.exists(f)) {
        cat(sprintf("%s → ファイルなし（セル11 未実行、またはこのスキームの ORA で有意でない）\n", tag))
        next
    }
    # ★ セル11 の検品で落ちた図（ko00121）は中身が書かれず 0 バイトで残る。
    #   file.exists() は TRUE になるので、大きさも見ないと readPNG がエラーで止まる
    if (file.size(f) == 0) {
        cat(sprintf("%s → pathview が描けなかった（セル11 の検品で「★ 描画が壊れた」）\n", tag))
        next
    }
    im = readPNG(f); h_px = dim(im)[1]; w_px = dim(im)[2]
    W = 15; H = W * h_px / w_px
    if (H > 18) { H = 18; W = H * w_px / h_px }
    cat(sprintf("%s  [%s]\n", tag, basename(f)))
    options(repr.plot.width = W, repr.plot.height = H)
    grid.newpage(); grid.raster(im)
}

# ── (3) ゲノム位置マップ ＋ 遺伝子表 ──────────────────────────
tgt = ora$term[ora$source=='KEGG' & grepl(paste0('^',ko,'_'), ora$term)][1]
g_ora = ora %>% filter(scheme==SCHEME, source=='KEGG', term==tgt, padj<0.05) %>%
        pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
        dplyr::select(gene_id, KEGG_Pathway) %>% tidyr::separate_rows(KEGG_Pathway, sep=',') %>%
        filter(KEGG_Pathway==ko) %>% pull(gene_id) %>% unique()

# contig を CDS 数の多い順に並べる（主 → 副）
ctg = res %>% filter(biotype=='CDS', strain %in% strains4, !is.na(end)) %>%
    group_by(strain, contig) %>% summarise(CDS=dplyr::n(), 長さMb=max(end)/1e6, .groups='drop') %>%
    arrange(factor(strain, levels=strains4), desc(CDS)) %>%
    # ★ 描画幅は最小値で底上げするが、x 軸は contig の実際の長さに合わせる。
    #   → 副 contig は bp あたりの縮尺が主より大きくなり、パネルを目一杯使う
    mutate(幅 = pmax(長さMb, MINW),
           短名 = sub('^[^~]+~', '', contig),
           # ★ contig 名は空白が無いので str_wrap では折れない。
           #   パネルが狭いと重なるので、8文字ごとに改行を入れて縦に積む（文字は削らない）
           名表示 = gsub('(.{10})', '\\1\n', 短名))

cat(sprintf("%s（%s）  contig %d 本（主 4 / 副 %d）\n", tgt, ko_jp, nrow(ctg), nrow(ctg)-4))
print(ctg %>% transmute(strain, contig=短名, CDS, 長さMb=round(長さMb,3), 描画幅=round(幅,2)) %>%
      as.data.frame(), row.names=FALSE)

bg = res %>% filter(biotype=='CDS', !marker, !is.na(start)) %>%
     mutate(位置Mb = start/1e6)
fg = bg %>% filter(gene_id %in% g_all) %>%
     mutate(cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                            !is.na(padj_pooled) & padj_pooled < 0.05 ~ 'DEG only', TRUE ~ 'not DEG'),
            cat = factor(cat, levels=names(col_cat)),
            # ★ 丸の塗りに発現強度（WT 群の正規化カウント平均）を入れる。
            #   発現量は3〜4桁にわたるので log10 にする（0 があるので +1）
            発現 = log10(normB_WT + 1),
            label = ifelse(is.na(Preferred_name)|Preferred_name=='-', substr(product,1,10), Preferred_name))
cat(sprintf("\n経路の全遺伝子 %d / 図に出る %d（padj が NA で検定されなかったものも含めて表示）\n",
    length(g_all), nrow(fg)))

ylim = range(c(bg$LFC_B[bg$strain %in% strains4]), na.rm=TRUE)

# ★ 凡例は「3分類が最もそろって現れる contig」1枚からだけ出す。
#   drop=FALSE を付けても ★ データが無い水準はキーの点が描かれないため、
#   パネルごとに凡例の見た目が変わり、guides='collect' が別物と判定して2つ並ぶ
leg_i = which.max(sapply(ctg$contig, function(cc)
            length(unique(fg$cat[fg$contig == cc]))))
cat(sprintf("凡例を出すパネル: %s（%d 水準）\n",
            ctg$短名[leg_i], length(unique(fg$cat[fg$contig == ctg$contig[leg_i]]))))

subs = list()
for (i in 1:nrow(ctg)) {
    b = bg %>% filter(contig == ctg$contig[i])
    f = fg %>% filter(contig == ctg$contig[i])
    is_main = ctg$CDS[i] == max(ctg$CDS[ctg$strain == ctg$strain[i]])
    p = ggplot(b, aes(x=位置Mb, y=LFC_B)) +
        geom_hline(yintercept=0, colour='grey20', linewidth=0.3) +
        geom_hline(yintercept=c(-1,1), linetype='dashed', colour='grey65', linewidth=0.3) +
        geom_point(colour='#b9c6d4', size=0.5, alpha=0.6) +
        coord_cartesian(xlim=c(0, ctg$長さMb[i]), ylim=ylim) +
        labs(title = sprintf('%s\n%s\n%d CDS, %.3f Mb',
                             ifelse(is_main, as.character(ctg$strain[i]), ''),
                             ctg$名表示[i], ctg$CDS[i], ctg$長さMb[i]),
             x=NULL, y=if (i==1) 'log2 fold change (KO / WT), scheme B' else NULL) +
        theme_bw(base_size=10) +
        theme(plot.title=element_text(size=ifelse(is_main, 9, 5.5),
                                    face=ifelse(is_main,'bold','plain'), lineheight=1.15),
              panel.grid.minor=element_blank(),
              axis.text.y=if (i==1) element_text() else element_blank(),
              axis.ticks.y=if (i==1) element_line() else element_blank(),
              legend.position='bottom', legend.box='horizontal',
              legend.title=element_text(size=8), legend.text=element_text(size=7.5),
              legend.margin=margin(1,1,1,1),
              plot.margin=margin(2,2,0,2))
    # ★ 経路の遺伝子が無い contig（0 行）では colour の水準が1つも現れないため、
    #   scale_colour_manual が「No shared levels found」の警告を出す。
    #   図は正しく描かれる（空パネルに色が無いだけ）ので、描画時に警告だけ抑える
    # ★ shape 21（縁と塗りを別々に指定できる丸）を使う。
    #   縁 = 分類（ORA hit / DEG only / not DEG）、塗り = 発現強度
    p = p + geom_point(data=f, aes(colour=cat, fill=発現), shape=21,
                       size=3.2, stroke=1.1, show.legend = (i == leg_i))
    if (nrow(f) > 0)
        p = p +
                # ★ 遺伝子名は小さいと読めないので大きめに。
                #   重なりは ggrepel の force と box.padding で逃がす
                geom_text_repel(data=f, aes(label=label, colour=cat), size=4.4,
                                fontface='bold', max.overlaps=60, seed=1,
                                force=3, min.segment.length=0,
                                segment.size=0.3, box.padding=0.45, show.legend=FALSE)
    # ★ 色の scale は「各パネルに」付ける。
    #   patchwork の guides='collect' は収集時点のパネルごとの凡例を比べるので、
    #   後から & で当てると収集に間に合わず凡例が2つ残る。
    #   全パネルが colour の aes を持ち、drop=FALSE で水準が揃うので1つにまとまる
    subs[[i]] = p + scale_colour_manual(values=col_cat, name=NULL, drop=FALSE,
                                        guide = guide_legend(override.aes = list(
                                            fill='white', size=3.2, stroke=1.1))) +
        # ★ 凡例を下に横向きで出す。右に置くと凡例の文字幅ぶん地図が狭くなる
        scale_fill_viridis_c(option='mako', direction=-1,
                             limits=range(fg$発現, na.rm=TRUE),
                             name='log10(DESeq2 normalized count + 1), WT mean',
                             guide = guide_colourbar(direction='horizontal',
                                                     title.position='top',
                                                     barwidth=unit(5,'cm'),
                                                     barheight=unit(0.35,'cm')))
}
P = wrap_plots(subs, nrow=1, widths=ctg$幅) + plot_layout(guides='collect') +
    plot_annotation(title = sprintf('A.  %s   [scheme %s]  - position on the genome', tgt, SCHEME),
      subtitle = paste0("grey = all CDS  |  coloured = genes of this pathway  ",
                        "(<span style='color:#d95f02'>**ORA hit**</span> / ",
                        "<span style='color:#7570b3'>**DEG only**</span> / ",
                        "<span style='color:#000000'>**not DEG**</span>)<br>",
                        "**gene labels = Preferred_name** (eggNOG; the first 10 characters of *product* ",
                        "are shown when Preferred_name is absent)<br>",
                        "point border = class, **point fill = log10(DESeq2 normalized count + 1), ",
                        "WT group mean** (scheme B: one size factor set for all 4 strains pooled)<br>",
                        "all contigs shown; panel width ∝ contig length, ",
                        sprintf("small contigs get a minimum width (%.1f Mb equivalent) and are drawn at a larger bp scale  |  x axis = position within each contig (Mb)", MINW)),

      # ★ 収集した凡例の位置は patchwork 側の theme で決まる。
      #   サブプロットの theme に legend.position を書いても効かない
      theme = theme(plot.title=element_text(size=14, face='bold'),
                    plot.subtitle=element_markdown(size=9, lineheight=1.3),
                    legend.position='bottom', legend.box='horizontal',
                    legend.title=element_text(size=8.5),
                    legend.text=element_text(size=8)))
# ── 表を同じ図の中に入れる（panel B）──────────────────────────
# ★ 図だけでは locus_tag / K番号 / EC が分からないので表を下に並べる。
#   ggpubr::ggtexttable は表を ggplot として返すので patchwork に載せられる
library(ggpubr)

tbl = fg %>%
    mutate(K  = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                       function(v) paste(v, collapse=',')),
           EC2 = ifelse(is.na(EC), '-', EC)) %>%
    transmute(strain, contig = sub('^[^~]+~', '', contig),
              Mb = round(位置Mb, 3), gene_id, name = label,
              K = K, EC = EC2, product = substr(product, 1, 26),
              # ★ LFC と padj は表に入れない。
              #   LFC は上の地図の y 軸そのもの、padj は分類（class）の色で示している。
              #   列を減らしたぶん1列あたりの幅が広がり、文字が重ならない
              class = as.character(cat)) %>%
    arrange(strain, Mb)

# ★ 縦に長くなりすぎるので、行を横に分割して並べる（1ブロック最大 24 行）
NROW_BLK = 24
nblk = ceiling(nrow(tbl) / NROW_BLK)
blks = split(1:nrow(tbl), ceiling(seq_along(1:nrow(tbl)) / ceiling(nrow(tbl)/nblk)))
cat(sprintf("\n表: %d 行 → %d ブロックに分けて横に並べる\n", nrow(tbl), length(blks)))

# ★ この図（地図＋表）だけ縦横 1.125 倍で出す（pathview の再掲は等倍のまま）
SCALE = 1.125
W_FIG = 22 * SCALE

# ★ 表のフォントは経路ごとに自動で決める。
#   列の幅は中身の文字数で決まるので、ko03010（112 遺伝子）のように行が多い経路では
#   ブロック数が増え、1ブロックに割ける幅が狭くなる。
#   固定の 6.2 pt のままだと表が割り当て幅をはみ出し、右端の class 列が切れてしまう。
#   → 実際に表を組んで幅を測り、収まる一番大きいフォントを選ぶ
#
# ★ 文字の幅は「フォントサイズに比例」しない。
#   小さいサイズでは hinting でカクカク変わる（実測: 5.5 pt で 4.75 inch、
#   5.9 pt で 5.04 inch）ので、計算で逆算せず 0.1 pt ずつ下げて実測で確かめる。
# ★ 測るときの device を間違えないこと。文字幅は device のフォント metrics で決まる。
#   実測（同じ表・同じ 5.9 pt）: pdf 4.70 / cairo png 4.81 / ragg 5.04 inch。
#   ggplot2 3.5 の ggsave は ragg があれば ragg を使うので、一番広く出る ragg で測る。
#   （Jupyter の画面表示は cairo png なので、ragg に合わせておけばどちらでも溢れない）
W_BLK = W_FIG / length(blks)             # 1ブロックに割ける幅 (inch)
ragg::agg_png(tempfile(fileext = '.png'), width = 1000, height = 1000, res = 100)
for (fs_try in seq(7.0, 3.4, by = -0.1)) {
    w_max = 0
    for (b in seq_along(blks)) {
        tg0 = gridExtra::tableGrob(
                  tbl[blks[[b]], ], rows = NULL,
                  theme = ttheme(base_size = fs_try, padding = unit(c(2.2, 1.6), 'mm'),
                                 colnames.style = colnames_style(size = fs_try + 0.4),
                                 tbody.style = tbody_style(size = fs_try)))
        w_max = max(w_max, sum(convertWidth(tg0$widths, 'in', valueOnly = TRUE)))
    }
    FS = fs_try
    if (w_max <= W_BLK * 0.98) break     # 0.98 は plot.margin のぶんの余裕
}
invisible(dev.off())
cat(sprintf("表のフォント: %.1f pt（最も広いブロック %.2f inch / 1ブロックの割り当て %.2f inch）\n",
            FS, w_max, W_BLK))

# ★ 表の文字色を地図と揃える（ORA hit = 橙 / DEG only = 紫 / not DEG = 黒）。
#   ggtexttable（gridExtra の tableGrob）は色を ★ 列方向（column-major）に展開する ので、
#   行ごとの色ベクトルを列数ぶん「繰り返して」渡す（rep(col, ncol) であって each= ではない。
#   each= にすると列ごとに1色になる。実測で確認）
tg = lapply(seq_along(blks), function(b) {
    ix  = blks[[b]]
    col = unname(col_cat[tbl$class[ix]])
    g = ggtexttable(tbl[ix, ], rows = NULL,
                theme = ttheme(base_size = FS, padding = unit(c(2.2, 1.6), 'mm'),
                               colnames.style = colnames_style(size = FS + 0.4, fill = 'grey92'),
                               tbody.style = tbody_style(size = FS, fill = 'white',
                                                         color = rep(col, ncol(tbl)))))
    g = g + theme(plot.margin = margin(0, 2, 2, 2))
    if (b == 1) g = g + labs(title = 'B.  gene list') +
        theme(plot.title = element_text(size = 13, face = 'bold', hjust = 0,
                                        margin = margin(b = 2)))
    g
})
TB = wrap_plots(tg, nrow = 1)

# ★ 表（ggtexttable）はパネルの中央に置かれるので、割り当てた高さが余ると
#   上下に空白ができる。行数から必要な高さを詰めて見積もる
H_map = 6.4
H_tab = 0.125 * max(sapply(blks, length)) + 0.35
# ★ tag_levels は使わない。patchwork の全サブプロット（contig パネル10個 + 表3枚）に
#   A〜M と振られてしまうため。見出しは A の図タイトルと B の表タイトルで示す。
#
# ★ wrap_elements() で包む。
#   patchwork を入れ子（P / TB）にすると ★ 内側の plot_annotation が描画されず、
#   A のタイトルが消える。包むと内側が1つの要素として確定し、タイトルが残る
PB = (wrap_elements(full = P) / TB) + plot_layout(heights = c(H_map, H_tab))

# SCALE / W_FIG は表のフォントを決めるときに上で定義してある
suppressWarnings(suppressMessages(
    ggsave(file.path(dir_gp, sprintf('genomepos_%s_%s.png', ko, SCHEME)), PB,
           width = W_FIG, height = (H_map + H_tab) * SCALE,
           dpi = 100, limitsize = FALSE)))
options(repr.plot.width = W_FIG, repr.plot.height = (H_map + H_tab) * SCALE)
suppressWarnings(print(PB))



(1) pathview - all genes of the pathway → pathview が描けなかった（セル11 の検品で「★ 描画が壊れた」）
(2) pathview - ORA geneID only → pathview が描けなかった（セル11 の検品で「★ 描画が壊れた」）
ko00121_Secondary bile acid biosynthesis（二次胆汁酸）  contig 10 本（主 4 / 副 6）
   strain         contig  CDS 長さMb 描画幅
  Bth_VPI    NC_004663.1 1676  6.243   6.24
  Bth_VPI    NC_004703.1   36  0.033   0.60
  Bth_St8 BQNN01000001.1 1780  6.132   6.13
  Bth_St8 BQNN01000002.1   41  0.045   0.60
  Bth_St8 BQNN01000005.1    7  0.004   0.60
  Bth_St8 BQNN01000004.1    2  0.004   0.60
 Bun_8492  NZ_CP102263.1 3392  4.629   4.63
 Odo_St23 BQOE01000001.1 2801  5.647   5.65
 Odo_St23 BQOE01000002.1   38  0.065   0.60
 Odo_St23 BQOE01000003.1   12  0.032   0.60

経路の全遺伝子 7 / 図に出る 7（padj が NA で検定されなかったものも含めて表示）
凡例を出すパネル: NZ_CP102263.1（2 水準）

表: 7 行 → 1 ブロックに分けて横に並べる
表のフォント: 7.0 pt（最も広いブロック 7.30 inch / 1ブロックの割り当て 24.75 inch）


plot without title

In [128]:
# ── 図13-14: ko03010（リボソーム）  scheme B ───────────────────
# セル13（準備）を先に実行しておくこと（col_cat / strains4 / MINW / dir_pv / dir_gp を使う）
#
# ★ 3つを続けて出す
#   (1) pathview（経路の全遺伝子）… 反応と化合物名。経路の中での位置づけ
#   (2) pathview（ORA の geneID のみ）… ORA が拾った反応だけ
#   (3) ゲノム位置マップ ＋ 遺伝子表 … どこにあるか、何の遺伝子か
library(ggplot2); library(ggtext); library(ggrepel); library(patchwork); library(ggpubr)
library(png); library(grid)

ko = 'ko03010'; ko_jp = 'リボソーム'; SCHEME = 'B'

# ── (1)(2) セル11 で作った pathview を再掲する ────────────────
# ★ 化合物名や反応の向きは pathview にしか無いので、地図の前に置く。
#   ORA で有意でない組み合わせ（例: ko03010 の scheme B）はファイルが無いので飛ばす
for (kind in c('all', 'ORA')) {
    f = file.path(dir_pv, sprintf('%s.%s_%s.png', ko, SCHEME, kind))
    tag = if (kind == 'all') '(1) pathview - all genes of the pathway' else
                             '(2) pathview - ORA geneID only'
    if (!file.exists(f)) {
        cat(sprintf("%s → ファイルなし（セル11 未実行、またはこのスキームの ORA で有意でない）\n", tag))
        next
    }
    # ★ セル11 の検品で落ちた図（ko00121）は中身が書かれず 0 バイトで残る。
    #   file.exists() は TRUE になるので、大きさも見ないと readPNG がエラーで止まる
    if (file.size(f) == 0) {
        cat(sprintf("%s → pathview が描けなかった（セル11 の検品で「★ 描画が壊れた」）\n", tag))
        next
    }
    im = readPNG(f); h_px = dim(im)[1]; w_px = dim(im)[2]
    W = 15; H = W * h_px / w_px
    if (H > 18) { H = 18; W = H * w_px / h_px }
    cat(sprintf("%s  [%s]\n", tag, basename(f)))
    options(repr.plot.width = W, repr.plot.height = H)
    grid.newpage(); grid.raster(im)
}

# ── (3) ゲノム位置マップ ＋ 遺伝子表 ──────────────────────────
tgt = ora$term[ora$source=='KEGG' & grepl(paste0('^',ko,'_'), ora$term)][1]
g_ora = ora %>% filter(scheme==SCHEME, source=='KEGG', term==tgt, padj<0.05) %>%
        pull(geneID) %>% strsplit('/') %>% unlist() %>% unique()
g_all = res %>% filter(!is.na(KEGG_Pathway), !marker) %>%
        dplyr::select(gene_id, KEGG_Pathway) %>% tidyr::separate_rows(KEGG_Pathway, sep=',') %>%
        filter(KEGG_Pathway==ko) %>% pull(gene_id) %>% unique()

# contig を CDS 数の多い順に並べる（主 → 副）
ctg = res %>% filter(biotype=='CDS', strain %in% strains4, !is.na(end)) %>%
    group_by(strain, contig) %>% summarise(CDS=dplyr::n(), 長さMb=max(end)/1e6, .groups='drop') %>%
    arrange(factor(strain, levels=strains4), desc(CDS)) %>%
    # ★ 描画幅は最小値で底上げするが、x 軸は contig の実際の長さに合わせる。
    #   → 副 contig は bp あたりの縮尺が主より大きくなり、パネルを目一杯使う
    mutate(幅 = pmax(長さMb, MINW),
           短名 = sub('^[^~]+~', '', contig),
           # ★ contig 名は空白が無いので str_wrap では折れない。
           #   パネルが狭いと重なるので、8文字ごとに改行を入れて縦に積む（文字は削らない）
           名表示 = gsub('(.{10})', '\\1\n', 短名))

cat(sprintf("%s（%s）  contig %d 本（主 4 / 副 %d）\n", tgt, ko_jp, nrow(ctg), nrow(ctg)-4))
print(ctg %>% transmute(strain, contig=短名, CDS, 長さMb=round(長さMb,3), 描画幅=round(幅,2)) %>%
      as.data.frame(), row.names=FALSE)

bg = res %>% filter(biotype=='CDS', !marker, !is.na(start)) %>%
     mutate(位置Mb = start/1e6)
fg = bg %>% filter(gene_id %in% g_all) %>%
     mutate(cat = case_when(gene_id %in% g_ora ~ 'ORA hit',
                            !is.na(padj_pooled) & padj_pooled < 0.05 ~ 'DEG only', TRUE ~ 'not DEG'),
            cat = factor(cat, levels=names(col_cat)),
            # ★ 丸の塗りに発現強度（WT 群の正規化カウント平均）を入れる。
            #   発現量は3〜4桁にわたるので log10 にする（0 があるので +1）
            発現 = log10(normB_WT + 1),
            label = ifelse(is.na(Preferred_name)|Preferred_name=='-', substr(product,1,10), Preferred_name))
cat(sprintf("\n経路の全遺伝子 %d / 図に出る %d（padj が NA で検定されなかったものも含めて表示）\n",
    length(g_all), nrow(fg)))

ylim = range(c(bg$LFC_B[bg$strain %in% strains4]), na.rm=TRUE)

# ★ 凡例は「3分類が最もそろって現れる contig」1枚からだけ出す。
#   drop=FALSE を付けても ★ データが無い水準はキーの点が描かれないため、
#   パネルごとに凡例の見た目が変わり、guides='collect' が別物と判定して2つ並ぶ
leg_i = which.max(sapply(ctg$contig, function(cc)
            length(unique(fg$cat[fg$contig == cc]))))
cat(sprintf("凡例を出すパネル: %s（%d 水準）\n",
            ctg$短名[leg_i], length(unique(fg$cat[fg$contig == ctg$contig[leg_i]]))))

subs = list()
for (i in 1:nrow(ctg)) {
    b = bg %>% filter(contig == ctg$contig[i])
    f = fg %>% filter(contig == ctg$contig[i])
    is_main = ctg$CDS[i] == max(ctg$CDS[ctg$strain == ctg$strain[i]])
    p = ggplot(b, aes(x=位置Mb, y=LFC_B)) +
        geom_hline(yintercept=0, colour='grey20', linewidth=0.3) +
        geom_hline(yintercept=c(-1,1), linetype='dashed', colour='grey65', linewidth=0.3) +
        geom_point(colour='#b9c6d4', size=0.5, alpha=0.6) +
        coord_cartesian(xlim=c(0, ctg$長さMb[i]), ylim=ylim) +
        labs(title = sprintf('%s\n%s\n%d CDS, %.3f Mb',
                             ifelse(is_main, as.character(ctg$strain[i]), ''),
                             ctg$名表示[i], ctg$CDS[i], ctg$長さMb[i]),
             x=NULL, y=if (i==1) 'log2 fold change (KO / WT), scheme B' else NULL) +
        theme_bw(base_size=10) +
        theme(plot.title=element_text(size=ifelse(is_main, 9, 5.5),
                                    face=ifelse(is_main,'bold','plain'), lineheight=1.15),
              panel.grid.minor=element_blank(),
              axis.text.y=if (i==1) element_text() else element_blank(),
              axis.ticks.y=if (i==1) element_line() else element_blank(),
              legend.position='bottom', legend.box='horizontal',
              legend.title=element_text(size=8), legend.text=element_text(size=7.5),
              legend.margin=margin(1,1,1,1),
              plot.margin=margin(2,2,0,2))
    # ★ 経路の遺伝子が無い contig（0 行）では colour の水準が1つも現れないため、
    #   scale_colour_manual が「No shared levels found」の警告を出す。
    #   図は正しく描かれる（空パネルに色が無いだけ）ので、描画時に警告だけ抑える
    # ★ shape 21（縁と塗りを別々に指定できる丸）を使う。
    #   縁 = 分類（ORA hit / DEG only / not DEG）、塗り = 発現強度
    p = p + geom_point(data=f, aes(colour=cat, fill=発現), shape=21,
                       size=3.2, stroke=1.1, show.legend = (i == leg_i))
    if (nrow(f) > 0)
        p = p +
                # ★ 遺伝子名は小さいと読めないので大きめに。
                #   重なりは ggrepel の force と box.padding で逃がす
                geom_text_repel(data=f, aes(label=label, colour=cat), size=4.4,
                                fontface='bold', max.overlaps=60, seed=1,
                                force=3, min.segment.length=0,
                                segment.size=0.3, box.padding=0.45, show.legend=FALSE)
    # ★ 色の scale は「各パネルに」付ける。
    #   patchwork の guides='collect' は収集時点のパネルごとの凡例を比べるので、
    #   後から & で当てると収集に間に合わず凡例が2つ残る。
    #   全パネルが colour の aes を持ち、drop=FALSE で水準が揃うので1つにまとまる
    subs[[i]] = p + scale_colour_manual(values=col_cat, name=NULL, drop=FALSE,
                                        guide = guide_legend(override.aes = list(
                                            fill='white', size=3.2, stroke=1.1))) +
        # ★ 凡例を下に横向きで出す。右に置くと凡例の文字幅ぶん地図が狭くなる
        scale_fill_viridis_c(option='mako', direction=-1,
                             limits=range(fg$発現, na.rm=TRUE),
                             name='log10(DESeq2 normalized count + 1), WT mean',
                             guide = guide_colourbar(direction='horizontal',
                                                     title.position='top',
                                                     barwidth=unit(5,'cm'),
                                                     barheight=unit(0.35,'cm')))
}
P = wrap_plots(subs, nrow=1, widths=ctg$幅) + plot_layout(guides='collect') +
    plot_annotation(title = sprintf('A.  %s   [scheme %s]  - position on the genome', tgt, SCHEME),
      subtitle = paste0("grey = all CDS  |  coloured = genes of this pathway  ",
                        "(<span style='color:#d95f02'>**ORA hit**</span> / ",
                        "<span style='color:#7570b3'>**DEG only**</span> / ",
                        "<span style='color:#000000'>**not DEG**</span>)<br>",
                        "**gene labels = Preferred_name** (eggNOG; the first 10 characters of *product* ",
                        "are shown when Preferred_name is absent)<br>",
                        "point border = class, **point fill = log10(DESeq2 normalized count + 1), ",
                        "WT group mean** (scheme B: one size factor set for all 4 strains pooled)<br>",
                        "all contigs shown; panel width ∝ contig length, ",
                        sprintf("small contigs get a minimum width (%.1f Mb equivalent) and are drawn at a larger bp scale  |  x axis = position within each contig (Mb)", MINW)),

      # ★ 収集した凡例の位置は patchwork 側の theme で決まる。
      #   サブプロットの theme に legend.position を書いても効かない
      theme = theme(plot.title=element_text(size=14, face='bold'),
                    plot.subtitle=element_markdown(size=9, lineheight=1.3),
                    legend.position='bottom', legend.box='horizontal',
                    legend.title=element_text(size=8.5),
                    legend.text=element_text(size=8)))
# ── 表を同じ図の中に入れる（panel B）──────────────────────────
# ★ 図だけでは locus_tag / K番号 / EC が分からないので表を下に並べる。
#   ggpubr::ggtexttable は表を ggplot として返すので patchwork に載せられる
library(ggpubr)

tbl = fg %>%
    mutate(K  = sapply(strsplit(gsub('ko:', '', ifelse(is.na(KEGG_ko), '-', KEGG_ko)), ','),
                       function(v) paste(v, collapse=',')),
           EC2 = ifelse(is.na(EC), '-', EC)) %>%
    transmute(strain, contig = sub('^[^~]+~', '', contig),
              Mb = round(位置Mb, 3), gene_id, name = label,
              K = K, EC = EC2, product = substr(product, 1, 26),
              # ★ LFC と padj は表に入れない。
              #   LFC は上の地図の y 軸そのもの、padj は分類（class）の色で示している。
              #   列を減らしたぶん1列あたりの幅が広がり、文字が重ならない
              class = as.character(cat)) %>%
    arrange(strain, Mb)

# ★ 縦に長くなりすぎるので、行を横に分割して並べる（1ブロック最大 24 行）
NROW_BLK = 24
nblk = ceiling(nrow(tbl) / NROW_BLK)
blks = split(1:nrow(tbl), ceiling(seq_along(1:nrow(tbl)) / ceiling(nrow(tbl)/nblk)))
cat(sprintf("\n表: %d 行 → %d ブロックに分けて横に並べる\n", nrow(tbl), length(blks)))

# ★ この図（地図＋表）だけ縦横 1.125 倍で出す（pathview の再掲は等倍のまま）
SCALE = 1.125
W_FIG = 22 * SCALE

# ★ 表のフォントは経路ごとに自動で決める。
#   列の幅は中身の文字数で決まるので、ko03010（112 遺伝子）のように行が多い経路では
#   ブロック数が増え、1ブロックに割ける幅が狭くなる。
#   固定の 6.2 pt のままだと表が割り当て幅をはみ出し、右端の class 列が切れてしまう。
#   → 実際に表を組んで幅を測り、収まる一番大きいフォントを選ぶ
#
# ★ 文字の幅は「フォントサイズに比例」しない。
#   小さいサイズでは hinting でカクカク変わる（実測: 5.5 pt で 4.75 inch、
#   5.9 pt で 5.04 inch）ので、計算で逆算せず 0.1 pt ずつ下げて実測で確かめる。
# ★ 測るときの device を間違えないこと。文字幅は device のフォント metrics で決まる。
#   実測（同じ表・同じ 5.9 pt）: pdf 4.70 / cairo png 4.81 / ragg 5.04 inch。
#   ggplot2 3.5 の ggsave は ragg があれば ragg を使うので、一番広く出る ragg で測る。
#   （Jupyter の画面表示は cairo png なので、ragg に合わせておけばどちらでも溢れない）
W_BLK = W_FIG / length(blks)             # 1ブロックに割ける幅 (inch)
ragg::agg_png(tempfile(fileext = '.png'), width = 1000, height = 1000, res = 100)
for (fs_try in seq(7.0, 3.4, by = -0.1)) {
    w_max = 0
    for (b in seq_along(blks)) {
        tg0 = gridExtra::tableGrob(
                  tbl[blks[[b]], ], rows = NULL,
                  theme = ttheme(base_size = fs_try, padding = unit(c(2.2, 1.6), 'mm'),
                                 colnames.style = colnames_style(size = fs_try + 0.4),
                                 tbody.style = tbody_style(size = fs_try)))
        w_max = max(w_max, sum(convertWidth(tg0$widths, 'in', valueOnly = TRUE)))
    }
    FS = fs_try
    if (w_max <= W_BLK * 0.98) break     # 0.98 は plot.margin のぶんの余裕
}
invisible(dev.off())
cat(sprintf("表のフォント: %.1f pt（最も広いブロック %.2f inch / 1ブロックの割り当て %.2f inch）\n",
            FS, w_max, W_BLK))

# ★ 表の文字色を地図と揃える（ORA hit = 橙 / DEG only = 紫 / not DEG = 黒）。
#   ggtexttable（gridExtra の tableGrob）は色を ★ 列方向（column-major）に展開する ので、
#   行ごとの色ベクトルを列数ぶん「繰り返して」渡す（rep(col, ncol) であって each= ではない。
#   each= にすると列ごとに1色になる。実測で確認）
tg = lapply(seq_along(blks), function(b) {
    ix  = blks[[b]]
    col = unname(col_cat[tbl$class[ix]])
    g = ggtexttable(tbl[ix, ], rows = NULL,
                theme = ttheme(base_size = FS, padding = unit(c(2.2, 1.6), 'mm'),
                               colnames.style = colnames_style(size = FS + 0.4, fill = 'grey92'),
                               tbody.style = tbody_style(size = FS, fill = 'white',
                                                         color = rep(col, ncol(tbl)))))
    g = g + theme(plot.margin = margin(0, 2, 2, 2))
    if (b == 1) g = g + labs(title = 'B.  gene list') +
        theme(plot.title = element_text(size = 13, face = 'bold', hjust = 0,
                                        margin = margin(b = 2)))
    g
})
TB = wrap_plots(tg, nrow = 1)

# ★ 表（ggtexttable）はパネルの中央に置かれるので、割り当てた高さが余ると
#   上下に空白ができる。行数から必要な高さを詰めて見積もる
H_map = 6.4
H_tab = 0.125 * max(sapply(blks, length)) + 0.35
# ★ tag_levels は使わない。patchwork の全サブプロット（contig パネル10個 + 表3枚）に
#   A〜M と振られてしまうため。見出しは A の図タイトルと B の表タイトルで示す。
#
# ★ wrap_elements() で包む。
#   patchwork を入れ子（P / TB）にすると ★ 内側の plot_annotation が描画されず、
#   A のタイトルが消える。包むと内側が1つの要素として確定し、タイトルが残る
PB = (wrap_elements(full = P) / TB) + plot_layout(heights = c(H_map, H_tab))

# SCALE / W_FIG は表のフォントを決めるときに上で定義してある
suppressWarnings(suppressMessages(
    ggsave(file.path(dir_gp, sprintf('genomepos_%s_%s.png', ko, SCHEME)), PB,
           width = W_FIG, height = (H_map + H_tab) * SCALE,
           dpi = 100, limitsize = FALSE)))
options(repr.plot.width = W_FIG, repr.plot.height = (H_map + H_tab) * SCALE)
suppressWarnings(print(PB))



(1) pathview - all genes of the pathway → ファイルなし（セル11 未実行、またはこのスキームの ORA で有意でない）
(2) pathview - ORA geneID only → ファイルなし（セル11 未実行、またはこのスキームの ORA で有意でない）
ko03010_Ribosome（リボソーム）  contig 10 本（主 4 / 副 6）
   strain         contig  CDS 長さMb 描画幅
  Bth_VPI    NC_004663.1 1676  6.243   6.24
  Bth_VPI    NC_004703.1   36  0.033   0.60
  Bth_St8 BQNN01000001.1 1780  6.132   6.13
  Bth_St8 BQNN01000002.1   41  0.045   0.60
  Bth_St8 BQNN01000005.1    7  0.004   0.60
  Bth_St8 BQNN01000004.1    2  0.004   0.60
 Bun_8492  NZ_CP102263.1 3392  4.629   4.63
 Odo_St23 BQOE01000001.1 2801  5.647   5.65
 Odo_St23 BQOE01000002.1   38  0.065   0.60
 Odo_St23 BQOE01000003.1   12  0.032   0.60

経路の全遺伝子 112 / 図に出る 112（padj が NA で検定されなかったものも含めて表示）
凡例を出すパネル: BQNN01000001.1（2 水準）

表: 112 行 → 5 ブロックに分けて横に並べる
表のフォント: 5.6 pt（最も広いブロック 4.84 inch / 1ブロックの割り当て 4.95 inch）


plot without title

### 考察: 経路の遺伝子はゲノム上に散在する（1オペロンではない）

#### 1. ★ CLAUDE.md の確認項目「1オペロンが効いているだけではないか」は満たされる

`ko00030` を見ると、**ORA hit（橙）はゲノム全体に散らばっている。**
`4_reads.ipynb` で見た CPS 座位（`CE91St8_29520〜` が 35.4 kb に固まる）のような
**まとまった塊にはなっていない。**代謝経路なので当然だが、
**「経路が下がった」が1オペロンの産物ではないことを図で示せる。**

ただし**近接ペアはある。**
- `Bth_VPI` の 1.3〜1.5 Mb に `gnd`・`fbp`・`pgl` が集まる
- `Bun_8492` の 2.8〜3.0 Mb に `tal`・`tkt`・`gnd`・`zwf`・`pgl` が並び、**橙が密集**

**この領域はオペロン単位で動いている可能性がある。**

#### 2. ★ 発現量を塗りに入れたことで分かること

`Bth_VPI` の `pgl` は**2コピーあり、LFC はどちらも −1.6〜−1.8 と似ている**が、
**塗りはまったく違う。**

| gene_id | K番号 | WT 平均 | 塗り |
|---|---|---|---|
| `BT_RS06160` | K01057 | **12,800** | 濃い |
| `BT_RS05660` | K07404 | **30** | ほぼ白 |

**3桁違う。**位置と向きだけでは区別できなかった情報で、
**「どちらのコピーが実際に効いているか」**が一目で分かる。

`deoC` や not DEG の `pfkA` も塗りが薄く、**動いていないうえに発現量も低い。**
一方 `tal`・`fbaB`（橙縁・濃い塗り）は**高発現かつ大きく低下**している。

#### 3. 副 contig には7経路の遺伝子が1つも無い

| strain | 副 contig | CDS | 長さ | 経路の遺伝子 |
|---|---|---|---|---|
| Bth_VPI | NC_004703.1（**プラスミド**） | 36 | 0.033 Mb | **0** |
| Bth_St8 | BQNN01000002.1 | 41 | 0.045 Mb | **0** |
| Bth_St8 | BQNN01000005.1 | 7 | 0.004 Mb | **0** |
| Bth_St8 | BQNN01000004.1 | 2 | 0.004 Mb | **0** |
| Odo_St23 | BQOE01000002.1 | 38 | 0.065 Mb | **0** |
| Odo_St23 | BQOE01000003.1 | 12 | 0.032 Mb | **0** |
| Bun_8492 | （副 contig なし） | — | — | — |

**空パネルとして残してある**ので、「調べていない」のではなく「無い」ことが読み取れる。

#### 4. この図で読めないこと

- **オペロン構造そのものは分からない。**近接は見えるが、同一転写単位かどうかは
  鎖の向きと遺伝子間距離を見る必要がある（`res` に `strand` はある）
- **塗りは WT 群の平均**なので、KO で発現が上がった遺伝子でも塗りは WT の値
- **背景の灰色点には発現情報を入れていない**（点が多すぎて潰れるため）

#### 保存

`pathway/genomepos/genomepos_<ko>_<scheme>.png`（7経路 × 2スキーム = 14 枚）

## セル14: 最終成果物のデータを作る（2パネル図）

CLAUDE.md §7 ステージ6「最終成果物の形」にあたる。過去プロジェクト T19 の
`LP食での経路シフト.html` と同形式の **(A) / (B) 2パネル図**を、
**5ソース × ORA / GSEA の4図**に分けて作る。

### ★ 先に実行しておくセル

セル5（TERM2GENE）／セル6（ORA）／セル8（GSEA）／**セル7-6（module）／セル7-7（TCDB）**。
使うもの: `t2g_kegg` `t2g_cog` `t2g_pfam` `t2g_mod` `t2g_tc` `ora` `gsea`
`ora_mod` `gsea_mod` `ora_tc` `gsea_tc` `res`

### なぜ4図に分けるか

| 図 | 中身 | 行数 |
|---|---|---|
| **図A** | ORA / KEGG pathway + module | 36 |
| **図B** | ORA / COG + PFAM + TCDB | 43 |
| **図C** | GSEA / KEGG pathway + module | 55 |
| **図D** | GSEA / COG + PFAM（**TCDB は 0**） | 38 |

★ **ORA と GSEA で有意になる term は半分しか重ならない**（和集合 122 本に対し両方で有意なのは 50 本）。
1枚にまとめると「どちらの検定で有意か」が読めなくなる。
★ **1 term = 1行。(A)/(B) は左右のパネル**なので、片方だけで有意な term も同じ1行。

### ★ スコア = Cliff's δ

`δ = 2(AUC − 0.5) = P(経路内 > 経路外) − P(経路内 < 経路外)`

**Wilcoxon 順位和検定の効果量。+1 で経路内が完全に上側、−1 で完全に下側、0 で差なし。**
比べているのは **term 内の遺伝子の Wald 統計量**と**それ以外の遺伝子の Wald 統計量**で、
(A) は `stat_A`、(B) は `stat_B`（株ごとの Wald をプールしたもの）を使う。マーカーは除外。
AUC は Mann-Whitney の `U / (n1·n2)` なので順位から直接出せる（ties は平均順位）。

順位検定の p 値も BH 補正する（**図 × scheme ごと**）。**ORA / GSEA の padj とは別物。**

### ★ 株ごとの δ も計算する（棒は株で割れない）

**δ は順位の統計量で加法的ではない**ので、セル7-2・7-3 のように**棒を株の寄与に割ることは
原理的にできない**（長さは δ、区分は正味リードという別物を1つの図形に混ぜることになる）。

代わりに **「その株の term 内遺伝子 vs ★ その株の term 外遺伝子」で別々に δ を計算する。**
独立な統計量なので**全体の δ より外側に出ることもあり、それが正しい挙動。**

★ **株あたり 2 本未満だと計算できない**（172 行中、4株そろうのは **87 行 = 51%**）。
描けない株は図に目盛りを置かないことで「2 本未満」であることを示す。

### ホバーに入れるもの（3種類）

| 欄 | 選び方 | 目的 |
|---|---|---|
| **実体** | **正味リード（転写量）**の大きい順に4本 | ★ **行ラベルは term の名前のままにする**ので、KEGG のラベルと中身が食い違う場合（`ko00195 Photosynthesis` の実体が ATP synthase など）をここで示す |
| **該当遺伝子** | **効果量（\|LFC\|）**の大きい順。term が **12 本以下なら全部**、多ければ**株ごとに3本** | `locus_tag` + `Preferred_name` を引けるようにする。PFAM・COG は1 term に数十〜数百あるので全部は出せない |
| **担い手の株** | **正味リード増減**の上位3株（(A)/(B) × up/down） | CLAUDE.md §7「順位は遺伝子数ではなく転写量」。セル7-3 で **104 term 中 50 本で1位が入れ替わる**ことを実測済み |

### 書き出すもの

| ファイル | 内容 |
|---|---|
| `pathway/panel2/panel2_data.json` | セル15 が読む（図の全データ） |
| `pathway/panel2/panel2_scores.csv` | 344 行（行 × scheme）。δ・順位検定の p・ORA padj・GSEA padj |
| `pathway/panel2/panel2_strain_delta.csv` | 1,376 行（行 × scheme × 株）。株ごとの δ と n |


In [173]:
# ── セル14: 最終成果物のデータを作る（2パネル図）──────────────────
# CLAUDE.md §7 ステージ6「最終成果物の形」にあたる。
# 過去プロジェクト T19 の `LP食での経路シフト` と同形式の2パネル図を、
# ★ 5ソース（KEGG pathway / KEGG module / COG / PFAM / TCDB）× ORA / GSEA
#   の4図に分けて作る。
#
# ★ 先に実行しておくセル
#   セル5（TERM2GENE）/ セル6（ORA）/ セル8（GSEA）/ セル7-6（module）/ セル7-7（TCDB）
#   使うもの: t2g_kegg, t2g_cog, t2g_pfam, t2g_mod, t2g_tc, ora, gsea,
#             ora_mod, gsea_mod, ora_tc, gsea_tc, res
suppressMessages(library(jsonlite))
options(repr.matrix.max.rows = 500, repr.matrix.max.cols = 50)

# ===============================================================
# 1. 5ソースを1つに揃える
# ===============================================================
T2G5 = bind_rows(
  t2g_kegg %>% mutate(source = 'KEGG'),   t2g_cog  %>% mutate(source = 'COG'),
  t2g_pfam %>% mutate(source = 'PFAM'),   t2g_mod  %>% mutate(source = 'module'),
  t2g_tc   %>% mutate(source = 'TCDB'))
ORA5 = bind_rows(
  ora     %>% transmute(source, scheme, direction, term, padj),
  ora_mod %>% transmute(source = 'module', scheme, direction, term, padj),
  ora_tc  %>% transmute(source = 'TCDB',   scheme, direction, term, padj))
GSEA5 = bind_rows(
  gsea     %>% transmute(source, scheme, direction, term, padj),
  gsea_mod %>% transmute(source = 'module', scheme, direction, term, padj),
  gsea_tc  %>% transmute(source = 'TCDB', scheme,
                         direction = ifelse(NES > 0, 'up', 'down'), term, padj))

# ── 4図の定義 ──────────────────────────────────────────────────
# ★ ORA と GSEA を分けるのは、両者で有意になる term が半分しか重ならないため
#   （和集合 122 本に対し、両方で有意なのは 50 本）。1枚にまとめると
#   「どちらの検定で有意か」が読めなくなる
FIG = list(
  A = list(test = 'ORA',  src = c('KEGG','module')),
  B = list(test = 'ORA',  src = c('COG','PFAM','TCDB')),
  C = list(test = 'GSEA', src = c('KEGG','module')),
  D = list(test = 'GSEA', src = c('COG','PFAM','TCDB')))

rows = bind_rows(lapply(names(FIG), function(f) {
  z = FIG[[f]]
  (if (z$test == 'ORA') ORA5 else GSEA5) %>%
    filter(padj < 0.05, source %in% z$src) %>% distinct(source, term) %>% mutate(fig = f)
}))

cat("■ Table 1: 図ごとの行数（1行 = term。(A)/(B) は左右のパネルなので重複させない）\n")
rows %>% dplyr::count(fig, source) %>%
  tidyr::pivot_wider(names_from = source, values_from = n, values_fill = 0) %>%
  mutate(計 = rowSums(across(where(is.numeric)))) %>% as.data.frame()
cat(sprintf("\n  合計 %d 行\n", nrow(rows)))
cat("  ★ 図D に TCDB の行は立たない（GSEA で有意 0 本）\n")

# ===============================================================
# 2. スコア = Cliff's δ を計算する
# ===============================================================
# ★ δ = 2(AUC − 0.5) = P(経路内 > 経路外) − P(経路内 < 経路外)
#   Wilcoxon 順位和検定の効果量。+1 で経路内が完全に上側、−1 で完全に下側、0 で差なし。
#   比べているのは term 内の遺伝子の Wald 統計量と、それ以外の遺伝子の Wald 統計量。
#   AUC は Mann-Whitney の U / (n1*n2) なので、順位から直接出せる（ties は平均順位）
cliff_delta = function(stat_all, in_set) {
    n1 = sum(in_set); n2 = sum(!in_set)
    if (n1 < 2 || n2 < 2) return(c(NA, NA, n1))
    r = rank(stat_all)
    U = sum(r[in_set]) - n1 * (n1 + 1) / 2
    p = suppressWarnings(wilcox.test(stat_all[in_set], stat_all[!in_set], exact = FALSE)$p.value)
    c(2 * (U / (n1 * n2) - 0.5), p, n1)
}

# ★ (A) は stat_A、(B) は stat_B（株ごとの Wald をプールしたもの）。マーカーは除外
st = list(A = res %>% filter(!is.na(stat_A), !marker) %>% dplyr::select(gene_id, s = stat_A),
          B = res %>% filter(!is.na(stat_B), !marker) %>% dplyr::select(gene_id, s = stat_B))

sc = list()
for (sch in c('A','B')) {
    v = st[[sch]]$s; names(v) = st[[sch]]$gene_id
    for (k in 1:nrow(rows)) {
        g = T2G5$gene[T2G5$source == rows$source[k] & T2G5$term == rows$term[k]]
        z = cliff_delta(v, names(v) %in% g)
        sc[[length(sc) + 1]] = data.frame(
            fig = rows$fig[k], source = rows$source[k], term = rows$term[k],
            scheme = sch, score = z[1], p_auc = z[2], n = z[3])
    }
}
# ★ 順位検定の p も BH 補正する（図 × scheme ごと）。ORA/GSEA の padj とは別物
sc = bind_rows(sc) %>% group_by(fig, scheme) %>%
     mutate(padj_auc = p.adjust(p_auc, method = 'BH')) %>% ungroup()

cat("\n■ Table 2: Cliff's δ の分布（図 × scheme）\n")
sc %>% group_by(fig, scheme) %>%
  summarise(n = dplyr::n(), 最小 = round(min(score), 3), 中央値 = round(median(score), 3),
            最大 = round(max(score), 3), 順位検定で有意 = sum(padj_auc < 0.05), .groups = 'drop') %>%
  as.data.frame()

# ===============================================================
# 3. ホバーの中身
# ===============================================================
# term に属する遺伝子を引き直す（マーカー除外、検定されたもののみ）
gene_long = rows %>%
    left_join(T2G5, by = c('source','term'), relationship = 'many-to-many') %>%
    left_join(res %>% dplyr::select(gene_id, strain, marker, Preferred_name, product,
                                    padj_A, padj_pooled, LFC_A, LFC_B,
                                    normA_diff, normB_diff), by = c('gene' = 'gene_id')) %>%
    filter(!marker, !is.na(padj_A)) %>% distinct(fig, source, term, gene, .keep_all = TRUE)

# ── (1) 実体 = 正味リード（転写量）を最も動かした遺伝子 上位4本 ──
#   ★ 行ラベルは term の名前のままにするので、KEGG のラベルと実体が食い違う場合
#     （ko00195 Photosynthesis の中身が ATP synthase など）はここで示す
jitsutai = gene_long %>% group_by(fig, source, term) %>%
    slice_max(abs(normA_diff), n = 4, with_ties = FALSE) %>%
    summarise(ent = list(lapply(1:dplyr::n(), function(i) list(
        g = gene[i], st = as.character(strain)[i],
        nm = ifelse(is.na(Preferred_name[i]) | Preferred_name[i] == '-', '', Preferred_name[i]),
        pr = substr(product[i], 1, 44),
        lfc = round(LFC_A[i], 2), net = round(normA_diff[i])))), .groups = 'drop')

# ── (2) 該当遺伝子 = locus_tag + Preferred_name ────────────────
#   ★ term が小さければ全部、多ければ株ごとに効果量（|LFC|）の大きいものを3本。
#     PFAM・COG は1 term に数十〜数百遺伝子あるので全部は出せない
NSHOW_ALL = 12   # これ以下なら全遺伝子を出す
NTOP      = 3    # 株ごとに出す本数

mk_genes = function(f, s, t, n_total) {
    g = gene_long %>% filter(fig == f, source == s, term == t) %>%
        mutate(nm = ifelse(is.na(Preferred_name) | Preferred_name == '-', '', Preferred_name),
               sg = !is.na(padj_A) & padj_A < 0.05)
    unname(lapply(split(g, droplevels(factor(g$strain))), function(z) {
        z = z %>% arrange(desc(abs(LFC_A)))
        k = if (n_total <= NSHOW_ALL) nrow(z) else min(NTOP, nrow(z))
        list(st = as.character(z$strain)[1], n = nrow(z), nsig = sum(z$sg),
             items = lapply(seq_len(k), function(i) list(
                 g = z$gene[i], nm = z$nm[i], lfc = round(z$LFC_A[i], 2), sig = z$sg[i])))
    }))
}

# ── (3) 担い手の株 = 正味リード増減の上位3株 ───────────────────
#   ★ CLAUDE.md §7「順位は遺伝子数ではなく転写量」。セル7-3 で
#     遺伝子数と転写量では 104 term 中 50 本で1位が入れ替わることを実測している
drv = bind_rows(lapply(c('A','B'), function(sch)
    gene_long %>%
        mutate(pj = if (sch == 'A') padj_A else padj_pooled,
               lf = if (sch == 'A') LFC_A  else LFC_B,
               nd = if (sch == 'A') normA_diff else normB_diff) %>%
        filter(!is.na(pj), pj < 0.05) %>%
        mutate(dir = ifelse(lf > 0, 'up', 'down')) %>%
        group_by(fig, source, term, dir, strain) %>%
        summarise(n = dplyr::n(), net = sum(nd), .groups = 'drop_last') %>%
        mutate(pct = round(100 * net / sum(net))) %>% ungroup() %>%
        mutate(scheme = sch)))

mk_drv = function(f, s, t, sch, dr) {
    z = drv %>% filter(fig == f, source == s, term == t, scheme == sch, dir == dr) %>%
        arrange(desc(abs(net))) %>% head(3)
    list(n_all = sum(drv$n[drv$fig == f & drv$source == s & drv$term == t &
                           drv$scheme == sch & drv$dir == dr]),
         top = lapply(seq_len(nrow(z)), function(i)
             list(st = as.character(z$strain)[i], n = z$n[i], pct = z$pct[i])))
}

# ── (4) ★ 株ごとの Cliff's δ ───────────────────────────────────
#   ★ δ は加法的でないので、棒を株の寄与に割ることは原理的にできない。
#     代わりに「その株の term 内遺伝子 vs ★ その株の term 外遺伝子」で
#     別々に δ を計算する。独立な統計量なので、全体の δ より外側に出ることもある。
#   ★ 株あたり 2 本未満だと計算できない（172 行中、4株そろうのは 87 行 = 51%）
strains4 = c('Bth_VPI', 'Bth_St8', 'Bun_8492', 'Odo_St23')
res_nm = res %>% filter(!marker)
sd_all = list()
for (sch in c('A','B')) {
    for (s in strains4) {
        z = res_nm %>% filter(strain == s) %>%
            mutate(stt = if (sch == 'A') stat_A else stat_B) %>% filter(!is.na(stt))
        v = z$stt; names(v) = z$gene_id
        for (k in 1:nrow(rows)) {
            g = T2G5$gene[T2G5$source == rows$source[k] & T2G5$term == rows$term[k]]
            ins = names(v) %in% g
            dd = cliff_delta(v, ins)
            sd_all[[length(sd_all) + 1]] = data.frame(
                fig = rows$fig[k], source = rows$source[k], term = rows$term[k],
                scheme = sch, strain = s, n_in = sum(ins), d = dd[1])
        }
    }
}
sd_all = bind_rows(sd_all)

cat("\n■ Table 3: 株ごとの δ を計算できた行数（株あたり 2 本以上が必要）\n")
sd_all %>% filter(scheme == 'A') %>% group_by(strain) %>%
    summarise(全行 = dplyr::n(), 計算可 = sum(!is.na(d)),
              率pct = round(100 * sum(!is.na(d)) / dplyr::n()), .groups = 'drop') %>%
    as.data.frame()
cat("\n  1行あたり何株ぶん描けるか（scheme A）\n")
sd_all %>% filter(scheme == 'A') %>% group_by(fig, source, term) %>%
    summarise(k = sum(!is.na(d)), .groups = 'drop') %>% dplyr::count(fig, k) %>%
    tidyr::pivot_wider(names_from = k, values_from = n, values_fill = 0, names_prefix = '株') %>%
    as.data.frame()

mk_sd = function(f, s, t, sch) {
    z = sd_all %>% filter(fig == f, source == s, term == t, scheme == sch)
    z = z[match(strains4, z$strain), ]
    lapply(seq_along(strains4), function(i) list(
        st = strains4[i],
        n  = ifelse(is.na(z$n_in[i]), 0L, as.integer(z$n_in[i])),
        d  = if (is.na(z$d[i])) NULL else round(z$d[i], 3)))
}

# ===============================================================
# 4. 1行ぶんのレコードに組み立てる
# ===============================================================
panel2 = list()
for (k in 1:nrow(rows)) {
    f = rows$fig[k]; s = rows$source[k]; t = rows$term[k]
    sA = sc %>% filter(fig == f, source == s, term == t, scheme == 'A')
    sB = sc %>% filter(fig == f, source == s, term == t, scheme == 'B')
    oa = ORA5  %>% filter(source == s, term == t)
    ga = GSEA5 %>% filter(source == s, term == t)
    gp = function(d, sch) { v = d$padj[d$scheme == sch]; if (length(v)) min(v) else NA }
    dirv = c(ORA5$direction[ORA5$source == s & ORA5$term == t & ORA5$padj < 0.05],
             GSEA5$direction[GSEA5$source == s & GSEA5$term == t & GSEA5$padj < 0.05])
    panel2[[k]] = list(
        fig = f, src = s, term = t, n = sA$n,
        dir = if (length(dirv)) names(sort(table(dirv), decreasing = TRUE))[1] else 'down',
        A = list(score = round(sA$score, 4), p_auc = signif(sA$p_auc, 3),
                 padj_auc = signif(sA$padj_auc, 3), ora = signif(gp(oa,'A'), 3),
                 gsea = signif(gp(ga,'A'), 3), sd = mk_sd(f, s, t, 'A')),
        B = list(score = round(sB$score, 4), p_auc = signif(sB$p_auc, 3),
                 padj_auc = signif(sB$padj_auc, 3), ora = signif(gp(oa,'B'), 3),
                 gsea = signif(gp(ga,'B'), 3), sd = mk_sd(f, s, t, 'B')),
        ent    = jitsutai$ent[jitsutai$fig == f & jitsutai$source == s & jitsutai$term == t][[1]],
        genes  = mk_genes(f, s, t, sA$n),
        all_sh = sA$n <= NSHOW_ALL,
        dA = list(up = mk_drv(f,s,t,'A','up'), dn = mk_drv(f,s,t,'A','down')),
        dB = list(up = mk_drv(f,s,t,'B','up'), dn = mk_drv(f,s,t,'B','down')))
}
names(panel2) = NULL

dir_p2 = file.path(path_pw, 'panel2')
if (!dir.exists(dir_p2)) dir.create(dir_p2, recursive = TRUE)
f_json = file.path(dir_p2, 'panel2_data.json')
# ★ digits = NA が必須。jsonlite::toJSON の digits 既定は 4 で、しかも
#   「有効数字」ではなく「小数点以下4桁」に丸める。上の signif(, 3) で
#   決めた精度がここで上書きされ、p値が壊れていた（実測: padj_auc 10 件・
#   ora 12 件・gsea 13 件が 0 になり、p_auc は 281/344 が 1e-4 の整数倍。
#   ko00030 の padj_auc は真値 1.67e-05 が 0 と表示されていた）。
#   ややこしいのは 3.82e-06 は残るのに 1.23e-05 が 0 になる点で、
#   「小さい値は無事、1e-5〜1e-3 だけ壊れる」という気づきにくい形で出る。
#   CSV（write.csv）は jsonlite を通らないので無影響だった
write(toJSON(panel2, auto_unbox = TRUE, null = 'null', na = 'null',
             digits = NA), f_json)

# 数値は CSV でも残す（図を使わずに表で参照できるように）
f_csv = file.path(dir_p2, 'panel2_scores.csv')
# ★ ORA5 / GSEA5 は up と down で2行あるので、term ごとに小さいほうの padj を取ってから繋ぐ
#   （そのまま left_join すると many-to-many の警告が出て行が増える）
sc %>% left_join(rows, by = c('fig','source','term')) %>%
    left_join(ORA5  %>% group_by(source, scheme, term) %>%
                  summarise(ora_padj = min(padj), .groups = 'drop'),
              by = c('source','scheme','term')) %>%
    left_join(GSEA5 %>% group_by(source, scheme, term) %>%
                  summarise(gsea_padj = min(padj), .groups = 'drop'),
              by = c('source','scheme','term')) %>%
    arrange(fig, source, desc(score)) %>% write.csv(f_csv, row.names = FALSE)

f_csv2 = file.path(dir_p2, 'panel2_strain_delta.csv')
write.csv(sd_all, f_csv2, row.names = FALSE)

cat(sprintf("\n■ 書き出し\n  %s（%.0f KB）\n  %s（%d 行）\n  %s（%d 行）\n",
            f_json, file.size(f_json)/1024, f_csv, nrow(sc), f_csv2, nrow(sd_all)))

cat("\n■ Table 4: ★ 検定の結論とスコアが食い違う行（その図の検定では有意だが、順位検定では有意でない）\n")
cat("  ORA は「閾値を超えた遺伝子が偶然より多いか」しか見ないので、\n")
cat("  term 全体としてはほとんど偏っていなくても有意になりうる\n")
bind_rows(lapply(1:nrow(rows), function(k) {
    r = panel2[[k]]; tst = if (FIG[[r$fig]]$test == 'ORA') 'ora' else 'gsea'
    s_fig  = any(c(r$A[[tst]], r$B[[tst]]) < 0.05, na.rm = TRUE)
    s_rank = any(c(r$A$padj_auc, r$B$padj_auc) < 0.05, na.rm = TRUE)
    if (s_fig && !s_rank) data.frame(
        fig = r$fig, source = r$src, term = substr(r$term, 1, 50), n = r$n,
        A_score = r$A$score, A_padj_rank = r$A$padj_auc,
        A_検定 = r$A[[tst]], B_検定 = r$B[[tst]]) else NULL
})) %>% as.data.frame()


■ Table 1: 図ごとの行数（1行 = term。(A)/(B) は左右のパネルなので重複させない）


fig,KEGG,module,COG,PFAM,TCDB,計
<chr>,<int>,<int>,<int>,<int>,<int>,<dbl>
A,22,14,0,0,0,36
B,0,0,7,33,3,43
C,30,25,0,0,0,55
D,0,0,8,30,0,38



  合計 172 行
  ★ 図D に TCDB の行は立たない（GSEA で有意 0 本）

■ Table 2: Cliff's δ の分布（図 × scheme）


fig,scheme,n,最小,中央値,最大,順位検定で有意
<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<int>
A,A,36,-0.947,-0.536,0.745,29
A,B,36,-0.960,-0.604,0.866,30
B,A,43,-0.954,-0.642,0.447,40
B,B,43,-0.948,-0.672,0.534,39
C,A,55,-0.947,-0.278,0.745,38
C,B,55,-0.960,-0.333,0.866,47
D,A,38,-0.961,-0.431,0.989,33
D,B,38,-0.973,-0.435,0.994,31



■ Table 3: 株ごとの δ を計算できた行数（株あたり 2 本以上が必要）


strain,全行,計算可,率pct
<chr>,<int>,<int>,<dbl>
Bth_St8,172,111,65
Bth_VPI,172,115,67
Bun_8492,172,128,74
Odo_St23,172,133,77



  1行あたり何株ぶん描けるか（scheme A）


fig,株0,株1,株2,株3,株4
<chr>,<int>,<int>,<int>,<int>,<int>
A,1,2,8,4,21
B,7,4,11,2,19
C,2,3,14,6,30
D,7,5,4,5,17



■ 書き出し
  /mnt/scratch/tanoue/261016_Xiaoxi_eptAoligo_2wFeces_Metatranscriptome/pathway/panel2/panel2_data.json（337 KB）
  /mnt/scratch/tanoue/261016_Xiaoxi_eptAoligo_2wFeces_Metatranscriptome/pathway/panel2/panel2_scores.csv（344 行）
  /mnt/scratch/tanoue/261016_Xiaoxi_eptAoligo_2wFeces_Metatranscriptome/pathway/panel2/panel2_strain_delta.csv（1376 行）

■ Table 4: ★ 検定の結論とスコアが食い違う行（その図の検定では有意だが、順位検定では有意でない）
  ORA は「閾値を超えた遺伝子が偶然より多いか」しか見ないので、
  term 全体としてはほとんど偏っていなくても有意になりうる


fig,source,term,n,A_score,A_padj_rank,A_検定,B_検定
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
A,KEGG,ko00720_Other carbon fixation pathways,81,-0.0254,0.7130,1.98e-02,0.006630
A,KEGG,ko00620_Pyruvate metabolism,84,-0.0163,0.7970,1.62e-01,0.011400
A,module,"M00009_Citrate cycle (TCA cycle, Krebs cycle)",49,-0.1039,0.2350,2.41e-02,0.035000
A,module,"M00741_Propanoyl-CoA metabolism, propanoyl-CoA =>",19,-0.1598,0.2490,6.23e-02,0.017700
A,module,M00373_Ethylmalonyl pathway,21,-0.1313,0.3150,1.15e-01,0.035000
B,COG,COG C: Energy production and conversion,530,-0.0404,0.1200,5.15e-06,0.000531
B,PFAM,PFAM:Patatin,19,-0.2313,0.0850,5.11e-02,0.047600
B,TCDB,TC 3.B.1: The Na+-transporting Carboxylic Acid Dec,11,0.2683,0.1240,6.03e-01,0.031600
C,KEGG,ko00590_Arachidonic acid metabolism,3,-0.6810,0.0580,3.81e-02,0.051500


## セル15: 最終成果物の HTML を書き出す（日本語版・英語版）

セル14 の `panel2_data.json` を読み込んだ**単一 HTML**（4図をタブで切り替え）を、
**日本語版 `panel2.html` と英語版 `panel2_en.html` の2本**作る。

★ **テンプレートを1つにして文字列を差し替える方式は採らない。**テンプレートを直したときに
置換が静かに外れるため。**2本のテンプレートを並べて持ち、両方とも書き出す。**
英語版には**日本語が残っていないかを検品**する（`grep('[ぁ-ヿ一-龯]', ...)` の行数が 0 か）。

### ★ なぜ ggiraph ではなく手書き HTML なのか（環境を確認したうえでの判断）

| 理由 | 実測 |
|---|---|
| **pandoc が無い** | `htmlwidgets::saveWidget(selfcontained = TRUE)` が使えない → ggiraph だと **`_files/` フォルダ付き**の出力になり1ファイルで渡せない |
| **`ggiraph` が未導入** | 依存の `gdtools` も無い。入れると CLAUDE.md §10 の環境記録の更新が要る |
| ホバーの中身 | 「実体」「該当遺伝子（株ごと）」「担い手の株」を入れるには**自由な HTML** が要る |
| **★ 日本語が使える** | HTML は**ブラウザが文字を描く**ので、PNG の `mbcsToSbcs` 文字化け問題が無い |

→ 過去プロジェクト T19 の成果物（`LP食での経路シフト.html`）と同じ方式にする。
**`jsonlite` 以外の依存は無い。**

### 図の読み方

| 要素 | 意味 |
|---|---|
| **大きい丸** | **term 全体の δ。** 塗り = **その図の検定（ORA または GSEA）で padj < 0.05**、白抜き = 有意でない |
| 棒（0 から丸まで） | 向きの色（赤 = KO で上昇 / 青 = KO で低下） |
| **★ 下段の細い目盛り 4本** | **株ごとの δ。** 株の色（`Bth_VPI` 青 / `Bth_St8` 橙 / `Bun_8492` 緑 / `Odo_St23` 紫）。**目盛りが無い株 = 2 本未満で計算できない** |
| **橙の行** | **片方のパネルでのみ有意**（左右で結論が違う = 菌量の寄与。74 / 172 行） |
| **⚑** | **その図の検定では有意なのに、順位検定（δ）では有意でない**（16 / 172 行） |

★ **点の位置（δ）と塗り（padj）は別の検定由来。**ORA は「閾値を超えた遺伝子が偶然より多いか」
しか見ないので、**term 全体としてはほとんど偏っていなくても有意になりうる**
（`COG C` は ORA padj 5.1e-06 なのに δ −0.040。`Bun_8492` だけ +0.11 で逆向きだから打ち消されている）。

### 実装の注記

- HTML テンプレートは **R の raw string（`r"---( ... )---"`）**で持つ。中に `"` も `'` も `${}` も
  入るので、通常の文字列だとエスケープだらけになる
- **Jupyter の出力としては表示しない。**IRkernel では htmlwidgets でない生 HTML を
  安定して描画できないので、**ファイルに書き出してブラウザで開く**
- scratch 側（`pathway/panel2/panel2.html`）と **project 直下（`panel2.html`）の両方**に書く


In [174]:
# ── セル15: 最終成果物の HTML を書き出す（日本語版・英語版）──────
# セル14 の panel2_data.json を読み込んだ単一 HTML を作る。4図をタブで切り替える。
#
# ★ なぜ ggiraph ではなく手書き HTML なのか（環境を確認したうえでの判断）
#   ・pandoc が無いので htmlwidgets::saveWidget(selfcontained = TRUE) が使えない
#     → ggiraph だと `_files/` フォルダ付きの出力になり、1ファイルで渡せない
#   ・ggiraph 自体が未導入（依存の gdtools も無い）。入れると CLAUDE.md §10 の更新が要る
#   ・ホバーに「実体の遺伝子」「該当遺伝子（株ごと）」「担い手の株」を入れるには
#     自由な HTML が要る
#   ・★ HTML はブラウザが文字を描くので日本語が使える（PNG の mbcsToSbcs 問題が無い）
#   → 過去プロジェクト T19 の成果物（LP食での経路シフト.html）と同じ方式にする。
#     jsonlite 以外の依存は無い
#
# ★ 日本語版と英語版を別ファイルで出す
#   投稿・共有用に英語版が要る。テンプレートを1つにして文字列を差し替える方式は、
#   テンプレートを直したときに置換が静かに外れるので採らない。
#   ★ 2本のテンプレートを並べて持ち、両方とも書き出す（中身の対応は目視で保つ）
#
# ★ 先にセル14 を実行しておくこと（panel2_data.json を読む）

# ── テンプレート（日本語版）────────────────────────────────────
# ★ R の raw string（r"---( ... )---"）で入れる。中に " も ' も ${} も入るので
#   通常の文字列だとエスケープだらけになる
tpl_ja = r"---(<!doctype html><html lang="ja"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1,viewport-fit=cover">
<title>ステージ6: 経路シフト（ORA / GSEA × 5ソース）</title>
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=IBM+Plex+Sans:wght@400;500;600&family=IBM+Plex+Mono:wght@400;500&family=Source+Serif+4:opsz,wght@8..60,600&display=swap">
<style>
:root{
  --paper:#faf9f7; --surface:#fff; --ink:#15150f; --ink-2:#55544d; --ink-3:#8a897f;
  --rule:#e3e1d8; --rule-soft:#efeee8; --axis:#c2c0b4;
  --up:#e34948; --down:#2a78d6; --flip:#eda100; --disagree:#8e44ad;
  --s-VPI:#2a78d6; --s-St8:#eb6834; --s-8492:#3f9e5f; --s-St23:#9b59b6;
  --ff-display:"Source Serif 4",Georgia,serif;
  --ff-body:"IBM Plex Sans",system-ui,-apple-system,sans-serif;
  --ff-mono:"IBM Plex Mono",ui-monospace,Menlo,monospace;
}
@media (prefers-color-scheme:dark){:root:not([data-theme="light"]){
  --paper:#131312; --surface:#1b1b19; --ink:#f0efe9; --ink-2:#bdbcb2; --ink-3:#8a897f;
  --rule:#302f2b; --rule-soft:#262523; --axis:#3d3c37;
  --up:#e66767; --down:#3987e5; --flip:#c98500; --disagree:#a569bd; color-scheme:dark;}}
:root[data-theme="dark"]{--paper:#131312;--surface:#1b1b19;--ink:#f0efe9;--ink-2:#bdbcb2;
  --ink-3:#8a897f;--rule:#302f2b;--rule-soft:#262523;--axis:#3d3c37;
  --up:#e66767;--down:#3987e5;--flip:#c98500;--disagree:#a569bd;color-scheme:dark;}
*{box-sizing:border-box}
body{background:var(--paper);color:var(--ink);font-family:var(--ff-body);font-size:14px;line-height:1.5;
  margin:0;padding:26px max(16px,env(safe-area-inset-left,0px)) 48px}
.wrap{max-width:1180px;margin:0 auto;display:flex;flex-direction:column;gap:20px}
header{border-bottom:1px solid var(--rule);padding-bottom:16px;display:flex;flex-direction:column;gap:8px}
.eyebrow{font-family:var(--ff-mono);font-size:11px;letter-spacing:.09em;text-transform:uppercase;color:var(--ink-3)}
h1{font-family:var(--ff-display);font-weight:600;font-size:clamp(22px,4vw,30px);line-height:1.2;margin:0}
.sub{color:var(--ink-2);max-width:74ch;margin:0}
.meta{font-family:var(--ff-mono);font-size:11.5px;color:var(--ink-3);display:flex;flex-wrap:wrap;gap:4px 16px}
.tabs{display:flex;gap:6px;flex-wrap:wrap}
.tab{font:inherit;font-size:12.5px;padding:6px 13px;border:1px solid var(--rule);border-radius:3px;
  background:var(--surface);color:var(--ink-2);cursor:pointer}
.tab[aria-selected=true]{border-color:var(--ink);color:var(--ink);font-weight:600}
.tab small{display:block;font-family:var(--ff-mono);font-size:10px;color:var(--ink-3);font-weight:400}
.legend{display:flex;flex-wrap:wrap;gap:9px 20px;align-items:center;font-size:12.5px;color:var(--ink-2)}
.lg{display:flex;align-items:center;gap:6px}
.sw{width:11px;height:11px;border-radius:50%;flex:none}
.sw.open{background:var(--paper);box-shadow:inset 0 0 0 2px currentColor}
.sw.sq{border-radius:2px}
.chart{background:var(--surface);border:1px solid var(--rule);border-radius:4px;overflow:hidden}
.hdr,.row{display:grid;grid-template-columns:minmax(300px,1.5fr) minmax(150px,1fr) minmax(150px,1fr);
  column-gap:14px;align-items:center}
.hdr{padding:11px 16px 7px;border-bottom:1px solid var(--rule)}
.hdr .ph{font-size:12px;font-weight:600;line-height:1.3}
.hdr .ph small{display:block;font-weight:400;color:var(--ink-3);font-size:11px;font-family:var(--ff-mono)}
.rows{padding:5px 16px 3px}
.row{min-height:29px;border-bottom:1px solid var(--rule-soft)}
.row:last-child{border-bottom:0}
.row:hover{background:color-mix(in srgb,var(--ink) 4%,transparent)}
.row.flip{background:color-mix(in srgb,var(--flip) 10%,transparent)}
.grp{grid-column:1/-1;display:flex;align-items:baseline;gap:9px;margin:9px 0 2px;
  padding-bottom:3px;border-bottom:1px solid var(--rule)}
.grp b{font-size:11px;letter-spacing:.07em;text-transform:uppercase;font-family:var(--ff-mono)}
.grp span{font-family:var(--ff-mono);font-size:10px;color:var(--ink-3)}
.rows .grp:first-child{margin-top:2px}
.lbl{min-width:0;padding:3px 0}
.lbl b{font-weight:500;font-size:12px;line-height:1.3;overflow-wrap:anywhere;font-family:var(--ff-mono)}
.lbl span{display:block;font-family:var(--ff-mono);font-size:10px;color:var(--ink-3)}
.cell{min-width:0}
svg{display:block;width:100%;height:auto;overflow:visible}
.axisrow{padding:2px 16px 10px;border-top:1px solid var(--rule)}
.tip{position:fixed;z-index:50;pointer-events:none;opacity:0;transition:opacity .1s;
  background:var(--surface);border:1px solid var(--rule);border-radius:4px;padding:10px 12px;
  box-shadow:0 6px 22px rgba(0,0,0,.16);max-width:360px;font-size:12px;line-height:1.45}
.tip.on{opacity:1}
.tip h4{margin:0 0 2px;font-size:12px;font-weight:600;font-family:var(--ff-mono);overflow-wrap:anywhere}
.tip .k{font-family:var(--ff-mono);font-size:10.5px;color:var(--ink-3)}
.tip table{border-collapse:collapse;margin-top:7px;font-size:11px;width:100%}
.tip th,.tip td{padding:1px 6px 1px 0;text-align:left}
.tip th{color:var(--ink-3);font-weight:400;font-size:10.5px}
.tip td{font-family:var(--ff-mono);font-variant-numeric:tabular-nums}
.tip .pan{margin-top:8px;padding-top:6px;border-top:1px solid var(--rule-soft)}
.tip .pt{font-size:10.5px;letter-spacing:.05em;text-transform:uppercase;color:var(--ink-3);
  font-family:var(--ff-mono);margin-bottom:2px}
.tip ul{margin:1px 0 0;padding-left:14px;color:var(--ink-2);font-size:11px}
.tip .ent{font-family:var(--ff-mono);font-size:10.5px;color:var(--ink-2);margin:1px 0 0;padding-left:0;list-style:none}
.tip .ent li{margin-bottom:1px}
.tip .none{margin:1px 0 0;color:var(--ink-3);font-size:11px}
.tip .gstr{margin-top:4px}
.tip .sn{font-family:var(--ff-mono);font-size:10.5px;font-weight:600}
.tip .ent b{font-weight:600;color:var(--ink)}
.tools{display:flex;gap:9px;align-items:center;flex-wrap:wrap}
button.ctl{font:inherit;font-size:12.5px;padding:5px 12px;border-radius:3px;border:1px solid var(--rule);
  background:var(--surface);color:var(--ink);cursor:pointer}
.tblwrap{overflow-x:auto;border:1px solid var(--rule);border-radius:4px;background:var(--surface)}
table.num{border-collapse:collapse;width:100%;font-size:12px;min-width:900px}
table.num th,table.num td{padding:5px 10px;text-align:left;border-bottom:1px solid var(--rule-soft);white-space:nowrap}
table.num th{font-size:10.5px;text-transform:uppercase;letter-spacing:.05em;color:var(--ink-3)}
table.num td.n{font-family:var(--ff-mono);text-align:right;font-variant-numeric:tabular-nums}
.notes{display:grid;grid-template-columns:repeat(auto-fit,minmax(270px,1fr));gap:13px}
.note{border-left:2px solid var(--rule);padding:2px 0 2px 12px}
.note h3{margin:0 0 3px;font-size:12.5px;font-weight:600}
.note p{margin:0;font-size:12.5px;color:var(--ink-2)}
.note code{font-family:var(--ff-mono);font-size:11.5px}
footer{font-size:11.5px;color:var(--ink-3);border-top:1px solid var(--rule);padding-top:13px}
footer p{margin:0 0 5px}
@media (max-width:680px){.hdr,.row{grid-template-columns:1fr 1fr;column-gap:10px}
  .lbl{grid-column:1/-1;padding:6px 0 1px}.row{min-height:auto;padding-bottom:5px}}
</style></head><body>
<div class="wrap">
<header>
  <div class="eyebrow">eptA oligo-colonization gnotobiotic metatranscriptome — stage 6 (株ごとの δ つき)</div>
  <h1>菌量の入れ替わりと、株内の発現制御を分ける</h1>
  <p class="sub">WT コンソーシアム（n=6）に対する eptA 変異コンソーシアム（n=5）の機能レベル変化。
  左は群集全体の機能出力で菌量の増減を含みます（scheme A）。右は株ごとに別々に DESeq2 を回したもので菌量変化を含みません（scheme B）。
  ★ <b>行ラベルは term の名前のまま。実体・該当遺伝子・株ごとの δ はカーソルを合わせると出ます。</b><br>
  ★ <b>大きい丸が term 全体の δ、その下の細い目盛りが株ごとに別々に計算した δ です。</b></p>
  <div class="meta"><span>スコア = Cliff's δ = 2×(AUC − 0.5)</span><span>padj: Benjamini–Hochberg</span><span id="dt"></span></div>
</header>
<div class="tabs" id="tabs" role="tablist"></div>
<div class="legend">
  <span class="lg"><span class="sw" style="background:var(--up)"></span>KO で上昇</span>
  <span class="lg"><span class="sw" style="background:var(--down)"></span>KO で低下</span>
  <span class="lg" style="color:var(--ink-2)"><span class="sw open" style="color:var(--axis)"></span>その図の検定で padj ≥ 0.05</span>
  <span class="lg"><span class="sw sq" style="background:var(--flip)"></span>片方のパネルでのみ有意</span>
  <span class="lg"><b style="font-family:var(--ff-mono)">[!]</b> <span style="color:var(--ink-2)">スコア（順位検定）と結論が食い違う</span></span>
  <span class="lg"><b style="font-family:var(--ff-mono)">*</b> <span style="color:var(--ink-2)">片方のパネルでのみ有意</span></span>
</div>
<div class="legend" style="border-top:1px solid var(--rule-soft);padding-top:9px">
  <span style="font-size:11.5px;color:var(--ink-3);font-family:var(--ff-mono)">株ごとの δ（下段の目盛り）:</span>
  <span class="lg"><span class="sw sq" style="background:var(--s-VPI);width:3px;height:13px;border-radius:1px"></span>Bth_VPI</span>
  <span class="lg"><span class="sw sq" style="background:var(--s-St8);width:3px;height:13px;border-radius:1px"></span>Bth_St8</span>
  <span class="lg"><span class="sw sq" style="background:var(--s-8492);width:3px;height:13px;border-radius:1px"></span>Bun_8492</span>
  <span class="lg"><span class="sw sq" style="background:var(--s-St23);width:3px;height:13px;border-radius:1px"></span>Odo_St23</span>
  <span style="font-size:11.5px;color:var(--ink-3)">目盛りが無い株 = その株に 2 本未満しかなく δ を計算できない</span>
</div>
<div class="chart">
  <div class="hdr"><div></div>
    <div class="ph">(A) 群集としての機能出力<small>全4株プール — 菌量変化を含む</small></div>
    <div class="ph">(B) 株内の発現制御<small>株ごとに DESeq2 — 菌量変化を除く</small></div></div>
  <div class="rows" id="rows"></div>
  <div class="axisrow"><div class="hdr" style="padding:0;border:0">
    <div class="lbl" style="padding:0"><span>スコア (Cliff's δ) = 2×(AUC − 0.5)</span></div>
    <div class="cell" id="axA"></div><div class="cell" id="axB"></div></div></div>
</div>
<div class="tools">
  <button class="ctl" id="srt">並び: ソース順</button>
  <button class="ctl" id="tgl" aria-expanded="false">数値テーブルを表示</button>
  <span style="font-size:12px;color:var(--ink-3)">行にカーソルを合わせると、実体の遺伝子と担い手の株が出ます</span>
</div>
<div class="tblwrap" id="tblwrap" hidden></div>
<div class="notes" id="notes"></div>
<footer id="foot"></footer>
</div>
<div class="tip" id="tip" role="status" aria-live="polite"></div>
<script id="pwdata" type="application/json">__DATA__</script>
<script>
(function(){
const D=JSON.parse(document.getElementById('pwdata').textContent);
document.getElementById('dt').textContent=__DATE__;
const FIG={A:{t:'図A',s:'ORA / KEGG pathway + module',test:'ora'},
           B:{t:'図B',s:'ORA / COG + PFAM + TCDB',test:'ora'},
           C:{t:'図C',s:'GSEA / KEGG pathway + module',test:'gsea'},
           D:{t:'図D',s:'GSEA / COG + PFAM + TCDB',test:'gsea'}};
const X0=-1.02,X1=1.02,W=200,H=27,CY=9;      // ★ 下段に株の目盛りを置くぶん行を高くした
const STY0=16,STY1=24;                       // 株の目盛りの上下
const SCOL={Bth_VPI:'var(--s-VPI)',Bth_St8:'var(--s-St8)',Bun_8492:'var(--s-8492)',Odo_St23:'var(--s-St23)'};
const sx=v=>(v-X0)/(X1-X0)*W, ZERO=sx(0);
const esc=s=>String(s).replace(/[&<>"]/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));
const fmt=q=>q==null?'—':(q<1e-4?q.toExponential(1).replace('e-','×10⁻'):(+q).toPrecision(2));
let cur='A', byScore=false;   // ★ 既定はソースごとに塊にする

const sig=(r,p,test)=>{const v=r[p][test];return v!=null&&v<0.05;};
function dot(r,p,test){
  const v=r[p].score,x=sx(v),s=sig(r,p,test),col=v>0?'var(--up)':'var(--down)';
  // ★ 株ごとの δ は「加法的に分解したもの」ではなく、その株の中で別々に計算した独立な統計量。
  //   だから棒を区分に割らず、下段の目盛りとして置く（全体の丸より外に出ることもある）
  const ticks=r[p].sd.filter(z=>z.d!=null).map(z=>
    `<line x1="${sx(z.d)}" y1="${STY0}" x2="${sx(z.d)}" y2="${STY1}" stroke="${SCOL[z.st]}"
      stroke-width="2.4" vector-effect="non-scaling-stroke" opacity=".85"/>`).join('');
  return `<svg viewBox="0 0 ${W} ${H}" preserveAspectRatio="none" aria-hidden="true">
   <line x1="${ZERO}" y1="2" x2="${ZERO}" y2="${H-2}" stroke="var(--axis)" stroke-width="1" vector-effect="non-scaling-stroke"/>
   <line x1="0" y1="${(STY0+STY1)/2}" x2="${W}" y2="${(STY0+STY1)/2}" stroke="var(--rule-soft)" stroke-width="1" vector-effect="non-scaling-stroke"/>
   ${ticks}
   <line x1="${ZERO}" y1="${CY}" x2="${x}" y2="${CY}" stroke="${col}" stroke-width="2" vector-effect="non-scaling-stroke" opacity="${s?.9:.35}"/>
   <circle cx="${x}" cy="${CY}" r="4.6" fill="${s?col:'var(--surface)'}" stroke="${col}" stroke-width="${s?0:2}" vector-effect="non-scaling-stroke"/></svg>`;
}
function axis(){const t=[-1,-.5,0,.5,1];
  return `<svg viewBox="0 0 ${W} 20" preserveAspectRatio="none" aria-hidden="true">
   <line x1="0" y1="1" x2="${W}" y2="1" stroke="var(--axis)" stroke-width="1" vector-effect="non-scaling-stroke"/>
   ${t.map(v=>`<line x1="${sx(v)}" y1="1" x2="${sx(v)}" y2="5" stroke="var(--axis)" stroke-width="1" vector-effect="non-scaling-stroke"/><text x="${sx(v)}" y="15" font-size="9" fill="var(--ink-3)" text-anchor="middle">${v}</text>`).join('')}</svg>`;}

const SRCORD={KEGG:0,module:1,COG:0,PFAM:1,TCDB:2};
const SRCLAB={KEGG:'KEGG pathway',module:'KEGG module',COG:'COG category',PFAM:'PFAM domain',TCDB:'TCDB family'};
function rowsOf(f){
  const test=FIG[f].test;
  let d=D.filter(r=>r.fig===f);
  // ★ 既定はソースごとに塊にし、塊の中は
  //   「両パネルで有意／非有意が一致する行」を上、「片方のパネルでのみ有意な行（橙）」を下に
  //   まとめ、それぞれの中はスコア降順
  const fl=r=>sig(r,'A',test)!==sig(r,'B',test);
  d.sort(byScore ? (a,b)=>b.A.score-a.A.score
                 : (a,b)=> a.src!==b.src ? SRCORD[a.src]-SRCORD[b.src]
                         : (fl(a)!==fl(b) ? (fl(a)?1:-1) : b.A.score-a.A.score));
  let last=null;
  return d.map(r=>{
    let head='';
    if(!byScore && r.src!==last){
      last=r.src;
      const n=d.filter(x=>x.src===r.src).length;
      head=`<div class="grp"><b>${SRCLAB[r.src]}</b><span>${n} 行</span></div>`;
    }
    const sA=sig(r,'A',test), sB=sig(r,'B',test);
    // ★ 順位検定（スコア）と、その図の検定の結論が食い違う行に印を付ける
    const dis=(sA||sB) && !((r.A.padj_auc!=null&&r.A.padj_auc<0.05)||(r.B.padj_auc!=null&&r.B.padj_auc<0.05));
    return head+`<div class="row${sA!==sB?' flip':''}${dis?' dis':''}" data-k="${D.indexOf(r)}" tabindex="0">
      <div class="lbl"><b>${esc(r.term)}${dis?' [!]':''}${sA!==sB?' *':''}</b><span>${r.src} · n=${r.n}</span></div>
      <div class="cell">${dot(r,'A',test)}</div><div class="cell">${dot(r,'B',test)}</div></div>`;
  }).join('');
}
function draw(){
  document.getElementById('rows').innerHTML=rowsOf(cur);
  document.getElementById('axA').innerHTML=axis();
  document.getElementById('axB').innerHTML=axis();
  document.querySelectorAll('.tab').forEach(b=>b.setAttribute('aria-selected',String(b.dataset.f===cur)));
  const test=FIG[cur].test, d=D.filter(r=>r.fig===cur);
  document.getElementById('tblwrap').innerHTML=
   `<table class="num"><thead><tr><th>term</th><th>source</th><th>n</th>
    <th>(A) score</th><th>(A) p_auc</th><th>(A) ORA</th><th>(A) GSEA</th>
    <th>(B) score</th><th>(B) p_auc</th><th>(B) ORA</th><th>(B) GSEA</th></tr></thead><tbody>${
    d.map(r=>`<tr><td>${esc(r.term)}</td><td>${r.src}</td><td class="n">${r.n}</td>
     <td class="n" style="color:${r.A.score>0?'var(--up)':'var(--down)'}">${r.A.score>0?'+':''}${r.A.score.toFixed(3)}</td>
     <td class="n">${fmt(r.A.p_auc)}</td><td class="n">${fmt(r.A.ora)}</td><td class="n">${fmt(r.A.gsea)}</td>
     <td class="n" style="color:${r.B.score>0?'var(--up)':'var(--down)'}">${r.B.score>0?'+':''}${r.B.score.toFixed(3)}</td>
     <td class="n">${fmt(r.B.p_auc)}</td><td class="n">${fmt(r.B.ora)}</td><td class="n">${fmt(r.B.gsea)}</td></tr>`).join('')}</tbody></table>`;
}
const tip=document.getElementById('tip');
function show(k,x,y){
  const r=D[k];
  const dl=o=>o.top.length?`<ul>${o.top.map(t=>`<li>${esc(t.st)} <span style="color:var(--ink-3)">${t.n}本 · ${t.pct}%</span></li>`).join('')}</ul>`:'<p class="none">有意な遺伝子なし</p>';
  const pan=(lab,o,d)=>`<div class="pan"><div class="pt">${lab} — 担い手（転写量順）</div>
    <div style="font-size:10.5px;color:var(--ink-3);margin-top:3px">↑ 上昇 (${o.up.n_all})</div>${dl(o.up)}
    <div style="font-size:10.5px;color:var(--ink-3);margin-top:3px">↓ 低下 (${o.dn.n_all})</div>${dl(o.dn)}</div>`;
  tip.innerHTML=`<h4>${esc(r.term)}</h4><div class="k">${r.src} · n=${r.n}</div>
   <table><tr><th></th><th>score</th><th>p(順位)</th><th>ORA</th><th>GSEA</th></tr>
   <tr><th>(A)</th><td>${r.A.score>0?'+':''}${r.A.score.toFixed(3)}</td><td>${fmt(r.A.p_auc)}</td><td>${fmt(r.A.ora)}</td><td>${fmt(r.A.gsea)}</td></tr>
   <tr><th>(B)</th><td>${r.B.score>0?'+':''}${r.B.score.toFixed(3)}</td><td>${fmt(r.B.p_auc)}</td><td>${fmt(r.B.ora)}</td><td>${fmt(r.B.gsea)}</td></tr></table>
   <div class="pan"><div class="pt">実体 — 正味リードを動かした遺伝子</div>
     <ul class="ent">${r.ent.map(e=>`<li>${esc(e.g)} <b>${esc(e.nm)}</b> ${esc(e.pr)}<br><span style="color:var(--ink-3)">${esc(e.st)} · LFC ${e.lfc>0?'+':''}${e.lfc} · 正味 ${e.net>0?'+':''}${e.net.toLocaleString()}</span></li>`).join('')}</ul></div>
   <div class="pan"><div class="pt">株ごとの δ（その株の中で別々に計算）</div>
     <table><tr><th></th>${r.A.sd.map(z=>`<th style="color:${SCOL[z.st]}">${esc(z.st.replace('Bth_','B.').replace('Bun_','U.').replace('Odo_','O.'))}</th>`).join('')}</tr>
     <tr><th>(A)</th>${r.A.sd.map(z=>`<td>${z.d==null?'—':(z.d>0?'+':'')+z.d}</td>`).join('')}</tr>
     <tr><th>(B)</th>${r.B.sd.map(z=>`<td>${z.d==null?'—':(z.d>0?'+':'')+z.d}</td>`).join('')}</tr>
     <tr><th>n</th>${r.A.sd.map(z=>`<td style="color:var(--ink-3)">${z.n}</td>`).join('')}</tr></table></div>
   <div class="pan"><div class="pt">該当遺伝子${r.all_sh?'（全 '+r.n+' 本）':'（株ごとに |LFC| 上位3本 / 全 '+r.n+' 本）'}</div>
     ${r.genes.map(z=>`<div class="gstr"><span class="sn" style="color:var(--s-${z.st.split('_')[1]||z.st})">${esc(z.st)}</span>
       <span style="color:var(--ink-3)">${z.n}本・有意 ${z.nsig}</span>
       <ul class="ent">${z.items.map(e=>`<li>${esc(e.g)}${e.nm?' <b>'+esc(e.nm)+'</b>':''} <span style="color:${e.lfc>0?'var(--up)':'var(--down)'}">${e.lfc>0?'+':''}${e.lfc}</span>${e.sig?'<span style="color:var(--ink-3)"> *</span>':''}</li>`).join('')}</ul></div>`).join('')}</div>
   ${pan('(A) 群集出力',r.dA)}${pan('(B) 株内制御',r.dB)}`;
  tip.classList.add('on');
  const w=tip.offsetWidth,h=tip.offsetHeight;
  tip.style.left=Math.min(Math.max(8,x+14),innerWidth-w-8)+'px';
  tip.style.top=Math.min(Math.max(8,y+14),innerHeight-h-8)+'px';
}
document.getElementById('tabs').innerHTML=Object.keys(FIG).map(f=>{
  const n=D.filter(r=>r.fig===f).length;
  return `<button class="tab" data-f="${f}" role="tab"><b>${FIG[f].t}</b><small>${FIG[f].s} · ${n}行</small></button>`;}).join('');
document.getElementById('tabs').addEventListener('click',e=>{const b=e.target.closest('.tab');if(b){cur=b.dataset.f;draw();}});
const rowsEl=document.getElementById('rows');
rowsEl.addEventListener('pointermove',e=>{const el=e.target.closest('.row');if(el)show(+el.dataset.k,e.clientX,e.clientY);});
rowsEl.addEventListener('pointerleave',()=>tip.classList.remove('on'));
rowsEl.addEventListener('focusin',e=>{const el=e.target.closest('.row');if(el){const b=el.getBoundingClientRect();show(+el.dataset.k,b.left+40,b.top);}});
rowsEl.addEventListener('focusout',()=>tip.classList.remove('on'));
document.getElementById('srt').addEventListener('click',e=>{byScore=!byScore;e.target.textContent='並び: '+(byScore?'スコア順':'ソース順');draw();});
const tw=document.getElementById('tblwrap'),btn=document.getElementById('tgl');
btn.addEventListener('click',()=>{const o=tw.hidden;tw.hidden=!o;btn.setAttribute('aria-expanded',String(o));btn.textContent=o?'数値テーブルを隠す':'数値テーブルを表示';});
document.getElementById('notes').innerHTML=`
 <div class="note"><h3>行ラベルは term のまま</h3><p>KEGG のラベルは実体と一致しないことがあります（<code>ko00195 Photosynthesis</code> の中身は F0F1 ATP 合成酵素）。ラベルは引き直せるよう原文のままにし、<b>実体はホバーの「実体」欄</b>に正味リードを動かした遺伝子として出しています。</p></div>
 <div class="note"><h3>スコアと色は別の検定</h3><p>点の位置は <code>2×(AUC−0.5)</code>（Wald 統計量の順位）、塗りつぶしは<b>その図の検定（ORA または GSEA）の padj</b> です。両者が食い違う行に <code>[!]</code> を付けています。<code>ko00720</code> は ORA padj 0.02 ですがスコアは −0.025（p 0.69）で、ORA が拾ったのは閾値を超えた一部の遺伝子だけです。</p></div>
 <div class="note"><h3>片方のパネルでのみ有意</h3><p>橙の行（<code>*</code>）。左右で結論が違う＝菌量の増減が効いている可能性があります。<b>(A) の上昇を「その菌が発現を上げた」と書いてはいけません。</b>★ <b>橙の行はソースの塊の下側にまとめてあります。</b></p></div>
 <div class="note"><h3>本数を所見の数として数えない</h3><p>172 行には重複が含まれます（<code>PFAM:Sod_Fe_N</code> と <code>Sod_Fe_C</code> は同じ <code>sodB</code> 5遺伝子、<code>M00157</code> は <code>ko00190</code> の部分集合）。</p></div>
 <div class="note"><h3>図D に TCDB は立たない</h3><p>TCDB は GSEA で有意 0 本のため。ORA でも 4 行だけで、付与率は 8.0%（786 遺伝子 / 85 ファミリー）しかありません。</p></div>
 <div class="note"><h3>株ごとの δ は積み上げではない</h3><p>δ は順位の統計量で<b>加法的ではない</b>ため、棒を株の寄与に割ることは原理的にできません。下段の目盛りは<b>その株の term 内遺伝子 vs その株の term 外遺伝子</b>で別々に計算した独立な δ です。<b>全体の丸より外側に出ることもあり、それが正しい挙動です。</b>4株そろうのは 172 行中 87 行（51%）で、残りは株あたり 2 本未満の株があるためです。</p></div>
 <div class="note"><h3>株ごとの δ で読めること</h3><p><code>ko00190_Oxidative phosphorylation</code> は全体 −0.10 と弱いのに <b>Bth 2株は +0.13 / +0.15 で逆向き</b>、<code>COG C</code> は ⚑ の行ですが <b>Bun_8492 だけ +0.11</b> で打ち消されています。<code>ko03010_Ribosome</code> は Bun +0.48 / Odo +0.44 に対し <b>Bth_VPI −0.11</b>（ただし n=3）。</p></div>
 <div class="note"><h3>該当遺伝子の出し方</h3><p>term 内が <b>12 本以下なら全部</b>、多ければ<b>株ごとに |LFC| の大きい3本</b>を出しています。<code>*</code> は DESeq2 の <code>padj_A &lt; 0.05</code>。「実体」欄が<b>転写量</b>で選んだ代表、「該当遺伝子」欄が<b>効果量</b>で選んだ代表です。</p></div>
 <div class="note"><h3>担い手は転写量で順位づけ</h3><p>遺伝子数ではなく正味リード増減（KO − WT）で並べています。括弧内はその向きの有意遺伝子数、% はその向きの正味変化に占める割合。<b>遺伝子数で数えると順位が 48% の term で入れ替わります</b>（セル7-3）。</p></div>`;
document.getElementById('foot').innerHTML=`
 <p><b>スコア = Cliff's δ</b>。<code>δ = 2(AUC − 0.5) = P(経路内 &gt; 経路外) − P(経路内 &lt; 経路外)</code><br>
 Wilcoxon 順位和検定の効果量。<b>+1 で経路内が完全に上側、−1 で完全に下側、0 で差なし。</b>
 比べているのは term 内の遺伝子の Wald 統計量と、それ以外の遺伝子の Wald 統計量。
 n = term 内で検定された遺伝子数（scheme A）。</p>
 <p>(A) は4株をプールした1回の DESeq2（CDS のみ、global size factor）。(B) は株ごとに独立した DESeq2 を4株で回し、p 値をプールして BH 補正したもの。<b>アイソレータ差は考慮していません</b>（異なる菌叢を定着させる以上アイソレータを分けざるを得ないため。CLAUDE.md §1）。</p>
 <p>ORA は <code>clusterProfiler::enricher</code>（<code>universe</code> 指定、<code>minGSSize=1</code> / <code>maxGSSize=Inf</code>）、GSEA は <code>clusterProfiler::GSEA</code>（Wald 統計量でランク付け）。KEGG global map と真核・疾患カテゴリは除外。<code>eptA</code> のマーカー feature は全解析から除外。</p>
 <p>オペロンによる遺伝子間の非独立性は未補正のため、p 値は出やすめに出ます。<b>★ これは試作版です。</b></p>`;
draw();
})();
</script></body></html>
)---"

# ── テンプレート（英語版）──────────────────────────────────────
tpl_en = r"---(<!doctype html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1,viewport-fit=cover">
<title>Stage 6: functional shift (ORA / GSEA across 5 sources)</title>
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=IBM+Plex+Sans:wght@400;500;600&family=IBM+Plex+Mono:wght@400;500&family=Source+Serif+4:opsz,wght@8..60,600&display=swap">
<style>
:root{
  --paper:#faf9f7; --surface:#fff; --ink:#15150f; --ink-2:#55544d; --ink-3:#8a897f;
  --rule:#e3e1d8; --rule-soft:#efeee8; --axis:#c2c0b4;
  --up:#e34948; --down:#2a78d6; --flip:#eda100; --disagree:#8e44ad;
  --s-VPI:#2a78d6; --s-St8:#eb6834; --s-8492:#3f9e5f; --s-St23:#9b59b6;
  --ff-display:"Source Serif 4",Georgia,serif;
  --ff-body:"IBM Plex Sans",system-ui,-apple-system,sans-serif;
  --ff-mono:"IBM Plex Mono",ui-monospace,Menlo,monospace;
}
@media (prefers-color-scheme:dark){:root:not([data-theme="light"]){
  --paper:#131312; --surface:#1b1b19; --ink:#f0efe9; --ink-2:#bdbcb2; --ink-3:#8a897f;
  --rule:#302f2b; --rule-soft:#262523; --axis:#3d3c37;
  --up:#e66767; --down:#3987e5; --flip:#c98500; --disagree:#a569bd; color-scheme:dark;}}
:root[data-theme="dark"]{--paper:#131312;--surface:#1b1b19;--ink:#f0efe9;--ink-2:#bdbcb2;
  --ink-3:#8a897f;--rule:#302f2b;--rule-soft:#262523;--axis:#3d3c37;
  --up:#e66767;--down:#3987e5;--flip:#c98500;--disagree:#a569bd;color-scheme:dark;}
*{box-sizing:border-box}
body{background:var(--paper);color:var(--ink);font-family:var(--ff-body);font-size:14px;line-height:1.5;
  margin:0;padding:26px max(16px,env(safe-area-inset-left,0px)) 48px}
.wrap{max-width:1180px;margin:0 auto;display:flex;flex-direction:column;gap:20px}
header{border-bottom:1px solid var(--rule);padding-bottom:16px;display:flex;flex-direction:column;gap:8px}
.eyebrow{font-family:var(--ff-mono);font-size:11px;letter-spacing:.09em;text-transform:uppercase;color:var(--ink-3)}
h1{font-family:var(--ff-display);font-weight:600;font-size:clamp(22px,4vw,30px);line-height:1.2;margin:0}
.sub{color:var(--ink-2);max-width:74ch;margin:0}
.meta{font-family:var(--ff-mono);font-size:11.5px;color:var(--ink-3);display:flex;flex-wrap:wrap;gap:4px 16px}
.tabs{display:flex;gap:6px;flex-wrap:wrap}
.tab{font:inherit;font-size:12.5px;padding:6px 13px;border:1px solid var(--rule);border-radius:3px;
  background:var(--surface);color:var(--ink-2);cursor:pointer}
.tab[aria-selected=true]{border-color:var(--ink);color:var(--ink);font-weight:600}
.tab small{display:block;font-family:var(--ff-mono);font-size:10px;color:var(--ink-3);font-weight:400}
.legend{display:flex;flex-wrap:wrap;gap:9px 20px;align-items:center;font-size:12.5px;color:var(--ink-2)}
.lg{display:flex;align-items:center;gap:6px}
.sw{width:11px;height:11px;border-radius:50%;flex:none}
.sw.open{background:var(--paper);box-shadow:inset 0 0 0 2px currentColor}
.sw.sq{border-radius:2px}
.chart{background:var(--surface);border:1px solid var(--rule);border-radius:4px;overflow:hidden}
.hdr,.row{display:grid;grid-template-columns:minmax(300px,1.5fr) minmax(150px,1fr) minmax(150px,1fr);
  column-gap:14px;align-items:center}
.hdr{padding:11px 16px 7px;border-bottom:1px solid var(--rule)}
.hdr .ph{font-size:12px;font-weight:600;line-height:1.3}
.hdr .ph small{display:block;font-weight:400;color:var(--ink-3);font-size:11px;font-family:var(--ff-mono)}
.rows{padding:5px 16px 3px}
.row{min-height:29px;border-bottom:1px solid var(--rule-soft)}
.row:last-child{border-bottom:0}
.row:hover{background:color-mix(in srgb,var(--ink) 4%,transparent)}
.row.flip{background:color-mix(in srgb,var(--flip) 10%,transparent)}
.grp{grid-column:1/-1;display:flex;align-items:baseline;gap:9px;margin:9px 0 2px;
  padding-bottom:3px;border-bottom:1px solid var(--rule)}
.grp b{font-size:11px;letter-spacing:.07em;text-transform:uppercase;font-family:var(--ff-mono)}
.grp span{font-family:var(--ff-mono);font-size:10px;color:var(--ink-3)}
.rows .grp:first-child{margin-top:2px}
.lbl{min-width:0;padding:3px 0}
.lbl b{font-weight:500;font-size:12px;line-height:1.3;overflow-wrap:anywhere;font-family:var(--ff-mono)}
.lbl span{display:block;font-family:var(--ff-mono);font-size:10px;color:var(--ink-3)}
.cell{min-width:0}
svg{display:block;width:100%;height:auto;overflow:visible}
.axisrow{padding:2px 16px 10px;border-top:1px solid var(--rule)}
.tip{position:fixed;z-index:50;pointer-events:none;opacity:0;transition:opacity .1s;
  background:var(--surface);border:1px solid var(--rule);border-radius:4px;padding:10px 12px;
  box-shadow:0 6px 22px rgba(0,0,0,.16);max-width:360px;font-size:12px;line-height:1.45}
.tip.on{opacity:1}
.tip h4{margin:0 0 2px;font-size:12px;font-weight:600;font-family:var(--ff-mono);overflow-wrap:anywhere}
.tip .k{font-family:var(--ff-mono);font-size:10.5px;color:var(--ink-3)}
.tip table{border-collapse:collapse;margin-top:7px;font-size:11px;width:100%}
.tip th,.tip td{padding:1px 6px 1px 0;text-align:left}
.tip th{color:var(--ink-3);font-weight:400;font-size:10.5px}
.tip td{font-family:var(--ff-mono);font-variant-numeric:tabular-nums}
.tip .pan{margin-top:8px;padding-top:6px;border-top:1px solid var(--rule-soft)}
.tip .pt{font-size:10.5px;letter-spacing:.05em;text-transform:uppercase;color:var(--ink-3);
  font-family:var(--ff-mono);margin-bottom:2px}
.tip ul{margin:1px 0 0;padding-left:14px;color:var(--ink-2);font-size:11px}
.tip .ent{font-family:var(--ff-mono);font-size:10.5px;color:var(--ink-2);margin:1px 0 0;padding-left:0;list-style:none}
.tip .ent li{margin-bottom:1px}
.tip .none{margin:1px 0 0;color:var(--ink-3);font-size:11px}
.tip .gstr{margin-top:4px}
.tip .sn{font-family:var(--ff-mono);font-size:10.5px;font-weight:600}
.tip .ent b{font-weight:600;color:var(--ink)}
.tools{display:flex;gap:9px;align-items:center;flex-wrap:wrap}
button.ctl{font:inherit;font-size:12.5px;padding:5px 12px;border-radius:3px;border:1px solid var(--rule);
  background:var(--surface);color:var(--ink);cursor:pointer}
.tblwrap{overflow-x:auto;border:1px solid var(--rule);border-radius:4px;background:var(--surface)}
table.num{border-collapse:collapse;width:100%;font-size:12px;min-width:900px}
table.num th,table.num td{padding:5px 10px;text-align:left;border-bottom:1px solid var(--rule-soft);white-space:nowrap}
table.num th{font-size:10.5px;text-transform:uppercase;letter-spacing:.05em;color:var(--ink-3)}
table.num td.n{font-family:var(--ff-mono);text-align:right;font-variant-numeric:tabular-nums}
.notes{display:grid;grid-template-columns:repeat(auto-fit,minmax(270px,1fr));gap:13px}
.note{border-left:2px solid var(--rule);padding:2px 0 2px 12px}
.note h3{margin:0 0 3px;font-size:12.5px;font-weight:600}
.note p{margin:0;font-size:12.5px;color:var(--ink-2)}
.note code{font-family:var(--ff-mono);font-size:11.5px}
footer{font-size:11.5px;color:var(--ink-3);border-top:1px solid var(--rule);padding-top:13px}
footer p{margin:0 0 5px}
@media (max-width:680px){.hdr,.row{grid-template-columns:1fr 1fr;column-gap:10px}
  .lbl{grid-column:1/-1;padding:6px 0 1px}.row{min-height:auto;padding-bottom:5px}}
</style></head><body>
<div class="wrap">
<header>
  <div class="eyebrow">eptA oligo-colonization gnotobiotic metatranscriptome — stage 6 (with per-strain delta)</div>
  <h1>Separating the shift in strain abundance from within-strain regulation</h1>
  <p class="sub">Functional-level change in the eptA-mutant consortium (n=5) relative to the wild-type consortium (n=6).
  The left panel is the community-level output and <b>includes</b> the shift in strain abundance (scheme A).
  The right panel runs DESeq2 separately within each strain and therefore <b>excludes</b> it (scheme B).
  <b>Row labels are the raw term names.</b> The genes behind each term, the member genes and the per-strain delta appear on hover.<br>
  <b>The large dot is the term-level delta; the thin ticks below it are deltas computed separately within each strain.</b></p>
  <div class="meta"><span>score = Cliff's delta = 2(AUC - 0.5)</span><span>padj: Benjamini–Hochberg</span><span id="dt"></span></div>
</header>
<div class="tabs" id="tabs" role="tablist"></div>
<div class="legend">
  <span class="lg"><span class="sw" style="background:var(--up)"></span>up in KO</span>
  <span class="lg"><span class="sw" style="background:var(--down)"></span>down in KO</span>
  <span class="lg" style="color:var(--ink-2)"><span class="sw open" style="color:var(--axis)"></span>padj &ge; 0.05 in this figure's test</span>
  <span class="lg"><span class="sw sq" style="background:var(--flip)"></span>significant in one panel only</span>
  <span class="lg"><b style="font-family:var(--ff-mono)">[!]</b> <span style="color:var(--ink-2)">significant in this test but not by the rank test on delta</span></span>
  <span class="lg"><b style="font-family:var(--ff-mono)">*</b> <span style="color:var(--ink-2)">significant in one panel only</span></span>
</div>
<div class="legend" style="border-top:1px solid var(--rule-soft);padding-top:9px">
  <span style="font-size:11.5px;color:var(--ink-3);font-family:var(--ff-mono)">per-strain delta (ticks below):</span>
  <span class="lg"><span class="sw sq" style="background:var(--s-VPI);width:3px;height:13px;border-radius:1px"></span>Bth_VPI</span>
  <span class="lg"><span class="sw sq" style="background:var(--s-St8);width:3px;height:13px;border-radius:1px"></span>Bth_St8</span>
  <span class="lg"><span class="sw sq" style="background:var(--s-8492);width:3px;height:13px;border-radius:1px"></span>Bun_8492</span>
  <span class="lg"><span class="sw sq" style="background:var(--s-St23);width:3px;height:13px;border-radius:1px"></span>Odo_St23</span>
  <span style="font-size:11.5px;color:var(--ink-3)">no tick for a strain = fewer than 2 genes in that strain, so delta cannot be computed</span>
</div>
<div class="chart">
  <div class="hdr"><div></div>
    <div class="ph">(A) community-level output<small>all 4 strains pooled &mdash; includes abundance shift</small></div>
    <div class="ph">(B) within-strain regulation<small>DESeq2 per strain &mdash; excludes abundance shift</small></div></div>
  <div class="rows" id="rows"></div>
  <div class="axisrow"><div class="hdr" style="padding:0;border:0">
    <div class="lbl" style="padding:0"><span>score (Cliff's delta) = 2(AUC - 0.5)</span></div>
    <div class="cell" id="axA"></div><div class="cell" id="axB"></div></div></div>
</div>
<div class="tools">
  <button class="ctl" id="srt">sort: by source</button>
  <button class="ctl" id="tgl" aria-expanded="false">show numeric table</button>
  <span style="font-size:12px;color:var(--ink-3)">hover a row to see the genes behind the term and which strains drive it</span>
</div>
<div class="tblwrap" id="tblwrap" hidden></div>
<div class="notes" id="notes"></div>
<footer id="foot"></footer>
</div>
<div class="tip" id="tip" role="status" aria-live="polite"></div>
<script id="pwdata" type="application/json">__DATA__</script>
<script>
(function(){
const D=JSON.parse(document.getElementById('pwdata').textContent);
document.getElementById('dt').textContent=__DATE__;
const FIG={A:{t:'Fig A',s:'ORA / KEGG pathway + module',test:'ora'},
           B:{t:'Fig B',s:'ORA / COG + PFAM + TCDB',test:'ora'},
           C:{t:'Fig C',s:'GSEA / KEGG pathway + module',test:'gsea'},
           D:{t:'Fig D',s:'GSEA / COG + PFAM + TCDB',test:'gsea'}};
const X0=-1.02,X1=1.02,W=200,H=27,CY=9;      // taller rows to make room for the per-strain ticks below
const STY0=16,STY1=24;                       // top/bottom of the per-strain ticks
const SCOL={Bth_VPI:'var(--s-VPI)',Bth_St8:'var(--s-St8)',Bun_8492:'var(--s-8492)',Odo_St23:'var(--s-St23)'};
const sx=v=>(v-X0)/(X1-X0)*W, ZERO=sx(0);
const esc=s=>String(s).replace(/[&<>"]/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));
const fmt=q=>q==null?'—':(q<1e-4?q.toExponential(1).replace('e-','×10⁻'):(+q).toPrecision(2));
let cur='A', byScore=false;   // default: group rows by source

const sig=(r,p,test)=>{const v=r[p][test];return v!=null&&v<0.05;};
function dot(r,p,test){
  const v=r[p].score,x=sx(v),s=sig(r,p,test),col=v>0?'var(--up)':'var(--down)';
  // Per-strain delta is NOT an additive decomposition: it is an independent statistic
  // computed within each strain. The bar is therefore not split into segments; the values
  // are drawn as ticks below, and they can fall outside the overall dot.
  const ticks=r[p].sd.filter(z=>z.d!=null).map(z=>
    `<line x1="${sx(z.d)}" y1="${STY0}" x2="${sx(z.d)}" y2="${STY1}" stroke="${SCOL[z.st]}"
      stroke-width="2.4" vector-effect="non-scaling-stroke" opacity=".85"/>`).join('');
  return `<svg viewBox="0 0 ${W} ${H}" preserveAspectRatio="none" aria-hidden="true">
   <line x1="${ZERO}" y1="2" x2="${ZERO}" y2="${H-2}" stroke="var(--axis)" stroke-width="1" vector-effect="non-scaling-stroke"/>
   <line x1="0" y1="${(STY0+STY1)/2}" x2="${W}" y2="${(STY0+STY1)/2}" stroke="var(--rule-soft)" stroke-width="1" vector-effect="non-scaling-stroke"/>
   ${ticks}
   <line x1="${ZERO}" y1="${CY}" x2="${x}" y2="${CY}" stroke="${col}" stroke-width="2" vector-effect="non-scaling-stroke" opacity="${s?.9:.35}"/>
   <circle cx="${x}" cy="${CY}" r="4.6" fill="${s?col:'var(--surface)'}" stroke="${col}" stroke-width="${s?0:2}" vector-effect="non-scaling-stroke"/></svg>`;
}
function axis(){const t=[-1,-.5,0,.5,1];
  return `<svg viewBox="0 0 ${W} 20" preserveAspectRatio="none" aria-hidden="true">
   <line x1="0" y1="1" x2="${W}" y2="1" stroke="var(--axis)" stroke-width="1" vector-effect="non-scaling-stroke"/>
   ${t.map(v=>`<line x1="${sx(v)}" y1="1" x2="${sx(v)}" y2="5" stroke="var(--axis)" stroke-width="1" vector-effect="non-scaling-stroke"/><text x="${sx(v)}" y="15" font-size="9" fill="var(--ink-3)" text-anchor="middle">${v}</text>`).join('')}</svg>`;}

const SRCORD={KEGG:0,module:1,COG:0,PFAM:1,TCDB:2};
const SRCLAB={KEGG:'KEGG pathway',module:'KEGG module',COG:'COG category',PFAM:'PFAM domain',TCDB:'TCDB family'};
function rowsOf(f){
  const test=FIG[f].test;
  let d=D.filter(r=>r.fig===f);
  // Default order: grouped by source; within a group, rows that agree between the two
  // panels come first and rows significant in one panel only (orange) are collected at the
  // bottom; within each of those, sorted by descending score.
  const fl=r=>sig(r,'A',test)!==sig(r,'B',test);
  d.sort(byScore ? (a,b)=>b.A.score-a.A.score
                 : (a,b)=> a.src!==b.src ? SRCORD[a.src]-SRCORD[b.src]
                         : (fl(a)!==fl(b) ? (fl(a)?1:-1) : b.A.score-a.A.score));
  let last=null;
  return d.map(r=>{
    let head='';
    if(!byScore && r.src!==last){
      last=r.src;
      const n=d.filter(x=>x.src===r.src).length;
      head=`<div class="grp"><b>${SRCLAB[r.src]}</b><span>${n} rows</span></div>`;
    }
    const sA=sig(r,'A',test), sB=sig(r,'B',test);
    // flag rows where this figure's test and the rank test on delta disagree
    const dis=(sA||sB) && !((r.A.padj_auc!=null&&r.A.padj_auc<0.05)||(r.B.padj_auc!=null&&r.B.padj_auc<0.05));
    return head+`<div class="row${sA!==sB?' flip':''}${dis?' dis':''}" data-k="${D.indexOf(r)}" tabindex="0">
      <div class="lbl"><b>${esc(r.term)}${dis?' [!]':''}${sA!==sB?' *':''}</b><span>${r.src} · n=${r.n}</span></div>
      <div class="cell">${dot(r,'A',test)}</div><div class="cell">${dot(r,'B',test)}</div></div>`;
  }).join('');
}
function draw(){
  document.getElementById('rows').innerHTML=rowsOf(cur);
  document.getElementById('axA').innerHTML=axis();
  document.getElementById('axB').innerHTML=axis();
  document.querySelectorAll('.tab').forEach(b=>b.setAttribute('aria-selected',String(b.dataset.f===cur)));
  const test=FIG[cur].test, d=D.filter(r=>r.fig===cur);
  document.getElementById('tblwrap').innerHTML=
   `<table class="num"><thead><tr><th>term</th><th>source</th><th>n</th>
    <th>(A) score</th><th>(A) p_auc</th><th>(A) ORA</th><th>(A) GSEA</th>
    <th>(B) score</th><th>(B) p_auc</th><th>(B) ORA</th><th>(B) GSEA</th></tr></thead><tbody>${
    d.map(r=>`<tr><td>${esc(r.term)}</td><td>${r.src}</td><td class="n">${r.n}</td>
     <td class="n" style="color:${r.A.score>0?'var(--up)':'var(--down)'}">${r.A.score>0?'+':''}${r.A.score.toFixed(3)}</td>
     <td class="n">${fmt(r.A.p_auc)}</td><td class="n">${fmt(r.A.ora)}</td><td class="n">${fmt(r.A.gsea)}</td>
     <td class="n" style="color:${r.B.score>0?'var(--up)':'var(--down)'}">${r.B.score>0?'+':''}${r.B.score.toFixed(3)}</td>
     <td class="n">${fmt(r.B.p_auc)}</td><td class="n">${fmt(r.B.ora)}</td><td class="n">${fmt(r.B.gsea)}</td></tr>`).join('')}</tbody></table>`;
}
const tip=document.getElementById('tip');
function show(k,x,y){
  const r=D[k];
  const dl=o=>o.top.length?`<ul>${o.top.map(t=>`<li>${esc(t.st)} <span style="color:var(--ink-3)">${t.n} genes &middot; ${t.pct}%</span></li>`).join('')}</ul>`:'<p class="none">no significant gene</p>';
  const pan=(lab,o,d)=>`<div class="pan"><div class="pt">${lab} &mdash; drivers (ranked by net reads)</div>
    <div style="font-size:10.5px;color:var(--ink-3);margin-top:3px">&uarr; up (${o.up.n_all})</div>${dl(o.up)}
    <div style="font-size:10.5px;color:var(--ink-3);margin-top:3px">&darr; down (${o.dn.n_all})</div>${dl(o.dn)}</div>`;
  tip.innerHTML=`<h4>${esc(r.term)}</h4><div class="k">${r.src} · n=${r.n}</div>
   <table><tr><th></th><th>score</th><th>p(rank)</th><th>ORA</th><th>GSEA</th></tr>
   <tr><th>(A)</th><td>${r.A.score>0?'+':''}${r.A.score.toFixed(3)}</td><td>${fmt(r.A.p_auc)}</td><td>${fmt(r.A.ora)}</td><td>${fmt(r.A.gsea)}</td></tr>
   <tr><th>(B)</th><td>${r.B.score>0?'+':''}${r.B.score.toFixed(3)}</td><td>${fmt(r.B.p_auc)}</td><td>${fmt(r.B.ora)}</td><td>${fmt(r.B.gsea)}</td></tr></table>
   <div class="pan"><div class="pt">genes behind the term &mdash; largest net read change</div>
     <ul class="ent">${r.ent.map(e=>`<li>${esc(e.g)} <b>${esc(e.nm)}</b> ${esc(e.pr)}<br><span style="color:var(--ink-3)">${esc(e.st)} &middot; LFC ${e.lfc>0?'+':''}${e.lfc} &middot; net ${e.net>0?'+':''}${e.net.toLocaleString()}</span></li>`).join('')}</ul></div>
   <div class="pan"><div class="pt">per-strain delta (computed within each strain)</div>
     <table><tr><th></th>${r.A.sd.map(z=>`<th style="color:${SCOL[z.st]}">${esc(z.st.replace('Bth_','B.').replace('Bun_','U.').replace('Odo_','O.'))}</th>`).join('')}</tr>
     <tr><th>(A)</th>${r.A.sd.map(z=>`<td>${z.d==null?'—':(z.d>0?'+':'')+z.d}</td>`).join('')}</tr>
     <tr><th>(B)</th>${r.B.sd.map(z=>`<td>${z.d==null?'—':(z.d>0?'+':'')+z.d}</td>`).join('')}</tr>
     <tr><th>n</th>${r.A.sd.map(z=>`<td style="color:var(--ink-3)">${z.n}</td>`).join('')}</tr></table></div>
   <div class="pan"><div class="pt">member genes${r.all_sh?' (all '+r.n+')':' (top 3 by |LFC| per strain / '+r.n+' total)'}</div>
     ${r.genes.map(z=>`<div class="gstr"><span class="sn" style="color:var(--s-${z.st.split('_')[1]||z.st})">${esc(z.st)}</span>
       <span style="color:var(--ink-3)">${z.n} genes, ${z.nsig} significant</span>
       <ul class="ent">${z.items.map(e=>`<li>${esc(e.g)}${e.nm?' <b>'+esc(e.nm)+'</b>':''} <span style="color:${e.lfc>0?'var(--up)':'var(--down)'}">${e.lfc>0?'+':''}${e.lfc}</span>${e.sig?'<span style="color:var(--ink-3)"> *</span>':''}</li>`).join('')}</ul></div>`).join('')}</div>
   ${pan('(A) community output',r.dA)}${pan('(B) within-strain',r.dB)}`;
  tip.classList.add('on');
  const w=tip.offsetWidth,h=tip.offsetHeight;
  tip.style.left=Math.min(Math.max(8,x+14),innerWidth-w-8)+'px';
  tip.style.top=Math.min(Math.max(8,y+14),innerHeight-h-8)+'px';
}
document.getElementById('tabs').innerHTML=Object.keys(FIG).map(f=>{
  const n=D.filter(r=>r.fig===f).length;
  return `<button class="tab" data-f="${f}" role="tab"><b>${FIG[f].t}</b><small>${FIG[f].s} &middot; ${n} rows</small></button>`;}).join('');
document.getElementById('tabs').addEventListener('click',e=>{const b=e.target.closest('.tab');if(b){cur=b.dataset.f;draw();}});
const rowsEl=document.getElementById('rows');
rowsEl.addEventListener('pointermove',e=>{const el=e.target.closest('.row');if(el)show(+el.dataset.k,e.clientX,e.clientY);});
rowsEl.addEventListener('pointerleave',()=>tip.classList.remove('on'));
rowsEl.addEventListener('focusin',e=>{const el=e.target.closest('.row');if(el){const b=el.getBoundingClientRect();show(+el.dataset.k,b.left+40,b.top);}});
rowsEl.addEventListener('focusout',()=>tip.classList.remove('on'));
document.getElementById('srt').addEventListener('click',e=>{byScore=!byScore;e.target.textContent='sort: '+(byScore?'by score':'by source');draw();});
const tw=document.getElementById('tblwrap'),btn=document.getElementById('tgl');
btn.addEventListener('click',()=>{const o=tw.hidden;tw.hidden=!o;btn.setAttribute('aria-expanded',String(o));btn.textContent=o?'hide numeric table':'show numeric table';});
document.getElementById('notes').innerHTML=`
 <div class="note"><h3>Row labels are the raw term names</h3><p>KEGG labels do not always match what is actually there (<code>ko00195 Photosynthesis</code> is in fact the F0F1 ATP synthase). Labels are kept verbatim so they can be looked up, and <b>what the term actually contains is shown on hover</b> as the genes with the largest net read change.</p></div>
 <div class="note"><h3>Dot position and fill come from different tests</h3><p>The position is <code>2(AUC - 0.5)</code> on the Wald-statistic ranking; the fill is <b>the padj of this figure's test (ORA or GSEA)</b>. Rows where the two disagree are marked <code>[!]</code>. <code>ko00720</code> has ORA padj 0.02 but a score of -0.025 (p 0.69): ORA only picked up the genes that crossed the threshold.</p></div>
 <div class="note"><h3>Significant in one panel only</h3><p>Orange rows (<code>*</code>). A different conclusion between the panels means the shift in strain abundance may be responsible. <b>An increase in (A) must not be written as "that strain upregulated the gene".</b> <b>Orange rows are collected at the bottom of each source block.</b></p></div>
 <div class="note"><h3>Do not count rows as findings</h3><p>The 172 rows contain redundancy: <code>PFAM:Sod_Fe_N</code> and <code>Sod_Fe_C</code> are the same five <code>sodB</code> genes, and <code>M00157</code> is a subset of <code>ko00190</code>.</p></div>
 <div class="note"><h3>No TCDB rows in Fig D</h3><p>No TCDB family is significant by GSEA. Even by ORA there are only 4 rows, and TCDB is assigned to just 8.0% of genes (786 genes / 85 families).</p></div>
 <div class="note"><h3>Per-strain delta is not a stacked decomposition</h3><p>Delta is a rank statistic and is <b>not additive</b>, so the bar cannot be split into per-strain contributions. Each tick is an independent delta computed from <b>that strain's in-term genes versus that strain's out-of-term genes</b>. <b>A tick can lie outside the overall dot, and that is correct behaviour.</b> All four strains are drawn for 87 of 172 rows (51%); elsewhere some strain has fewer than 2 genes.</p></div>
 <div class="note"><h3>What the per-strain delta shows</h3><p><code>ko00190_Oxidative phosphorylation</code> is only -0.10 overall, yet <b>the two Bth strains run the other way at +0.13 / +0.15</b>. <code>COG C</code> is a <code>[!]</code> row and is cancelled out because <b>Bun_8492 alone is +0.11</b>. For <code>ko03010_Ribosome</code>, Bun is +0.48 and Odo +0.44 while <b>Bth_VPI is -0.11</b> (on only n=3 genes).</p></div>
 <div class="note"><h3>How member genes are chosen</h3><p><b>All genes if the term has 12 or fewer</b>; otherwise the <b>top 3 by |LFC| within each strain</b>. <code>*</code> marks <code>padj_A &lt; 0.05</code> from DESeq2. The first hover block picks representatives by <b>transcript amount</b>, the member-gene block by <b>effect size</b>.</p></div>
 <div class="note"><h3>Drivers are ranked by transcript amount</h3><p>Strains are ordered by net read change (KO - WT), not by gene count. The number in parentheses is the count of significant genes in that direction, and % is the share of the net change in that direction. <b>Ranking by gene count changes which strain comes first in 48% of terms.</b></p></div>`;
document.getElementById('foot').innerHTML=`
 <p><b>Score = Cliff's delta.</b> <code>delta = 2(AUC - 0.5) = P(in-term &gt; out-of-term) - P(in-term &lt; out-of-term)</code><br>
 The effect size of the Wilcoxon rank-sum test. <b>+1 means the in-term genes are entirely above, -1 entirely below, 0 no difference.</b>
 The comparison is between the Wald statistics of the genes in the term and those of all other genes.
 n = number of genes in the term that were tested (scheme A).</p>
 <p>(A) is a single DESeq2 run on all four strains pooled (CDS only, global size factors). (B) runs DESeq2 independently within each of the four strains and pools the p values before a single BH correction. <b>Isolator effects are not modelled</b>: colonising two different consortia requires separate isolators, so the group and the isolator are inseparable by design.</p>
 <p>ORA: <code>clusterProfiler::enricher</code> with an explicit <code>universe</code> and <code>minGSSize=1</code> / <code>maxGSSize=Inf</code>. GSEA: <code>clusterProfiler::GSEA</code> ranked by the Wald statistic. KEGG global maps and the eukaryote/disease categories are excluded, as are the <code>eptA</code> marker features.</p>
 <p>Non-independence between genes in the same operon is not corrected for, so p values are anti-conservative.</p>`;
draw();
})();
</script></body></html>
)---"

dir_p2 = file.path(path_pw, 'panel2')
f_json = file.path(dir_p2, 'panel2_data.json')
dat    = paste(readLines(f_json, warn = FALSE), collapse = '')
today  = sprintf('"%s"', format(Sys.Date(), '%Y-%m-%d'))

for (z in list(list(tpl = tpl_ja, f = 'panel2.html',    lab = '日本語版'),
               list(tpl = tpl_en, f = 'panel2_en.html', lab = '英語版'))) {
    html = sub('__DATA__', dat,   z$tpl, fixed = TRUE)
    html = sub('__DATE__', today, html,  fixed = TRUE)
    f_out = file.path(dir_p2, z$f)
    writeLines(html, f_out, useBytes = TRUE)
    # ★ project 側にも複製する（scratch は共有しないので）。
    #   ★ getwd() は使わない。Jupyter のカーネルの作業ディレクトリは notebook の
    #     置き場とは限らず、実測で /home になっていた（Permission denied）。
    #     セル2 で定義した path_project を使う
    f_proj = file.path(path_project, z$f)
    invisible(file.copy(f_out, f_proj, overwrite = TRUE))
    cat(sprintf("■ %s: %s（%.0f KB）→ %s\n", z$lab, f_out, file.size(f_out)/1024, f_proj))
    cat(sprintf("    JSON が埋まったか: %s / __DATA__ が残っていないか: %s\n",
        ifelse(grepl('"fig"', html, fixed = TRUE), 'OK', '★ 失敗'),
        ifelse(grepl('__DATA__', html, fixed = TRUE), '★ 残っている', 'OK')))
    # ★ 英語版に日本語が残っていないか検品する（置換もれの検出）
    if (z$lab == '英語版') {
        n_cjk = length(grep('[぀-ヿ一-鿿]', strsplit(html, '\n')[[1]]))
        cat(sprintf("    英語版に残った日本語の行: %d（0 なら OK）\n", n_cjk))
    }
}

cat("\n■ 検品: 図ごとの行数\n")
data.frame(
    図 = c('A','B','C','D'),
    内容 = c('ORA / KEGG pathway + module', 'ORA / COG + PFAM + TCDB',
             'GSEA / KEGG pathway + module', 'GSEA / COG + PFAM + TCDB'),
    行数 = sapply(c('A','B','C','D'), function(f) sum(sapply(panel2, function(r) r$fig == f))),
    check.names = FALSE)

cat("\n  ★ ブラウザで開くこと。Jupyter の出力としては表示しない\n")
cat("    （IRkernel では htmlwidgets でない生 HTML を安定して描画できない。\n")
cat("      また 361 KB を出力に埋めると notebook が肥大し、ツールチップは\n")
cat("      position:fixed なのでセル出力の矩形から出られずに切れる）\n")
cat("    → notebook 内で読むための静的版はセル15-2 で作る\n")


■ 日本語版: /mnt/scratch/tanoue/261016_Xiaoxi_eptAoligo_2wFeces_Metatranscriptome/pathway/panel2/panel2.html（362 KB）→ /home/tanoue/261016_Xiaoxi_eptAoligo_2wFeces_Metatranscriptome/panel2.html
    JSON が埋まったか: OK / __DATA__ が残っていないか: OK
■ 英語版: /mnt/scratch/tanoue/261016_Xiaoxi_eptAoligo_2wFeces_Metatranscriptome/pathway/panel2/panel2_en.html（362 KB）→ /home/tanoue/261016_Xiaoxi_eptAoligo_2wFeces_Metatranscriptome/panel2_en.html
    JSON が埋まったか: OK / __DATA__ が残っていないか: OK
    英語版に残った日本語の行: 0（0 なら OK）

■ 検品: 図ごとの行数


,図,内容,行数
,<chr>,<chr>,<int>
A,A,ORA / KEGG pathway + module,36
B,B,ORA / COG + PFAM + TCDB,43
C,C,GSEA / KEGG pathway + module,55
D,D,GSEA / COG + PFAM + TCDB,38



  ★ ブラウザで開くこと。Jupyter の出力としては表示しない
    （IRkernel では htmlwidgets でない生 HTML を安定して描画できない。
      また 361 KB を出力に埋めると notebook が肥大し、ツールチップは
      position:fixed なのでセル出力の矩形から出られずに切れる）
    → notebook 内で読むための静的版はセル15-2 で作る


## セル15-2: 2パネル図の静的版（notebook の中で読むための図）

セル15 の HTML は**ブラウザで開く前提**なので、**notebook 内で読める静的版**も作る。

|- | セル15（HTML） | **セル15-2（静的版）** |
|---|---|---|
| 大きい丸（全体の δ）・棒・向きの色・有意性 | ○ | ○ |
| 株ごとの δ の目盛り | ○ | ○ |
| 橙の行（片方のパネルのみ有意）・`[!]` | ○ | ○ |
| **ホバー**（実体・該当遺伝子・担い手） | ○ | **×** → セル14 の Table と `panel2_scores.csv` で参照 |
| notebook のセル出力として読める | **×** | ○ |
| 単一ファイルで渡せる | ○ | △（PNG 4枚） |

### ★ 静的版で変えたこと

|- | 理由 |
|---|---|
| 記号を **`*` と `[!]`（ASCII）**にした | `⚑` などの記号は **PNG 保存時に `mbcsToSbcs` 変換が失敗**する。term 名はもともと ASCII なので図全体が英数字だけになる |
| **y を離散軸（`scale_y_discrete`）**にした | 数値 y ＋ 全体の `breaks` だと、**facet の境界で隣のソースの行ラベルが1行混入する**（実測: KEGG のパネルに module の先頭行が出て 22 行のはずが 23 行表示になった）。`geom_tile(height = 0.9)` が y の範囲を広げ、`expand` と合わせて隣の break を引き込んでいた |
| 株の目盛りを **`shape = 124`（縦棒）の点**＋`position_nudge` にした | 離散軸では `geom_segment` で縦棒を引けないため |
| ソース名を **`angle = -90`（縦書き）**にした | `angle = 0` だと右の灰色帯がラベルの長さぶん太くなって描画領域を圧迫する |

### 並び（HTML と同一）

**ソースごとの塊 → 塊の中は「両パネルで一致する行」を上、「片方のパネルでのみ有意な行（橙）」を下**に
まとめ、それぞれの中は δ 降順（上が正、下が負）。
★ ggplot の離散軸は **level 1 を一番下に描く**ので、`arrange(src_lab, desc(flip), score)` と
**橙の行を先に並べる**形にしてある。

### 書き出し

`pathway/panel2/panel2_static_{A,B,C,D}.png`（15 × 10.2〜14.1 inch）。セルの出力としても4図を出す。


In [172]:
# ── セル15-2: 2パネル図の静的版（notebook の中で読むための図）──────
# セル15 の HTML はブラウザで開く前提なので、notebook 内で読める静的版も作る。
#
# ★ HTML でしか得られないもの: ホバー（実体の遺伝子・該当遺伝子・担い手の株）
# ★ 静的版で得られるもの: notebook のセル出力として他の図と並べて読めること
#   → 役割を分ける。数値はセル14 の Table と panel2_scores.csv で参照できる
#
# ★ 先にセル14 を実行しておくこと（sc / rows / sd_all / ORA5 / GSEA5 / FIG を使う）
library(ggplot2)
options(repr.matrix.max.rows = 500)
# ステージ5・セル7-2〜7-7 と同じ株の色
col_strain = c(Bth_VPI = '#2a78d6', Bth_St8 = '#eb6834', Bun_8492 = '#3f9e5f', Odo_St23 = '#9b59b6')
# ★ 図のタイトル。日本語を入れない（PNG 保存時に mbcsToSbcs 変換が失敗する）
FIGT = list(A = 'ORA  -  KEGG pathway + module',  B = 'ORA  -  COG + PFAM + TCDB',
            C = 'GSEA  -  KEGG pathway + module', D = 'GSEA  -  COG + PFAM + TCDB')
SRCLAB = c(KEGG='KEGG pathway', module='KEGG module', COG='COG category',
           PFAM='PFAM domain', TCDB='TCDB family')
SRCORD = c(KEGG=1, module=2, COG=1, PFAM=2, TCDB=3)

# その図の検定の padj（塗りの判定に使う）
figpadj = bind_rows(lapply(names(FIG), function(f) {
  d = if (FIG[[f]]$test=='ORA') ORA5 else GSEA5
  d %>% filter(source %in% FIG[[f]]$src) %>% group_by(source, scheme, term) %>%
    summarise(p_fig = min(padj), .groups='drop') %>% mutate(fig = f) }))

dd = sc %>% left_join(figpadj, by = c('fig','source','scheme','term')) %>%
  mutate(sig = !is.na(p_fig) & p_fig < 0.05,
         up  = score > 0,
         src_lab = factor(SRCLAB[source], levels = SRCLAB[names(sort(SRCORD))]),
         pane = factor(ifelse(scheme=='A','(A) pooled  -  community output',
                                          '(B) per-strain  -  within-strain regulation'),
                       levels = c('(A) pooled  -  community output',
                                  '(B) per-strain  -  within-strain regulation')))

# ★ 行の印: 片方のパネルのみ有意 / 順位検定と食い違い
mark = dd %>% group_by(fig, source, term) %>%
  summarise(flip = length(unique(sig)) > 1,
            dis  = any(sig) & !any(padj_auc < 0.05, na.rm = TRUE), .groups='drop')
dd = dd %>% left_join(mark, by = c('fig','source','term'))

# 株ごとの δ（長形式）
tick = sd_all %>% filter(!is.na(d)) %>%
  mutate(pane = factor(ifelse(scheme=='A','(A) pooled  -  community output',
                                          '(B) per-strain  -  within-strain regulation'),
                       levels = levels(dd$pane)),
         src_lab = factor(SRCLAB[source], levels = levels(dd$src_lab)),
         strain = factor(strain, levels = names(col_strain)))

plots = list()
for (f in names(FIGT)) {
  d1 = dd %>% filter(fig == f)
  # y の並び: ソースごとの塊 → 塊の中はスコア（A）の降順
  # ★ 並び: ソースごとの塊 → 塊の中は「両パネルで有意／非有意が一致する行」を上、
  #   「片方のパネルでのみ有意な行（橙）」を下にまとめ、それぞれの中は δ の昇順。
  #   ggplot の離散軸は level 1 を一番下に描くので、橙の行を先に並べる
  ordk = d1 %>% filter(scheme=='A') %>% arrange(src_lab, desc(flip), score) %>%
         transmute(source, term, src_lab,
                   lab = paste0(substr(term, 1, 52),
                                ifelse(dis, ' [!]', ''), ifelse(flip, ' *', '')))
  d1 = d1 %>% left_join(ordk %>% dplyr::select(source, term, lab), by=c('source','term')) %>%
       mutate(lab = factor(lab, levels = ordk$lab))
  t1 = tick %>% filter(fig == f) %>%
       left_join(ordk %>% dplyr::select(source, term, lab), by=c('source','term')) %>%
       mutate(lab = factor(lab, levels = ordk$lab), y = as.integer(lab))

  nr = nrow(ordk)
  # ★ y は離散軸にする。数値 y ＋ 全体の breaks だと、facet の境界で
  #   隣のソースの行のラベルが1行混入する（実測: KEGG のパネルに module の
  #   先頭行のラベルが出た）。離散軸なら各パネルに属する水準だけが描かれる
  p = ggplot(d1, aes(y = lab)) +
      # 片方のパネルのみ有意の行に橙の帯
      geom_tile(data = d1 %>% filter(flip), aes(x = 0, y = lab),
                width = Inf, height = 0.9, fill = '#eda100', alpha = 0.13,
                inherit.aes = FALSE) +
      geom_vline(xintercept = 0, colour = 'grey55', linewidth = 0.3) +
      # 株ごとの δ（下段の目盛り）。★ 離散軸なので縦棒は shape = 124 の点で描き、
      #   position_nudge で行の下に少しずらす
      geom_point(data = t1, aes(x = d, y = lab, colour = strain), shape = 124,
                 size = 3.1, position = position_nudge(y = -0.27), inherit.aes = FALSE) +
      # 全体の δ
      geom_segment(aes(x = 0, xend = score, y = lab, yend = lab, alpha = sig),
                   colour = 'grey35', linewidth = 0.5) +
      geom_point(aes(x = score, fill = up, alpha = sig), shape = 21, size = 2.6,
                 colour = 'grey25', stroke = 0.4) +
      facet_grid(src_lab ~ pane, scales = 'free_y', space = 'free_y') +
      scale_y_discrete(expand = expansion(add = 0.7)) +
      scale_x_continuous(limits = c(-1.03, 1.03), breaks = c(-1,-0.5,0,0.5,1)) +
      scale_fill_manual(values = c(`TRUE`='#e34948', `FALSE`='#2a78d6'),
                        labels = c(`TRUE`='up in KO', `FALSE`='down in KO'), name = NULL) +
      scale_alpha_manual(values = c(`TRUE`=1, `FALSE`=0.3), name = NULL,
                         labels = c(`TRUE`='padj < 0.05', `FALSE`='n.s.')) +
      scale_colour_manual(values = col_strain, name = 'per-strain delta', drop = FALSE) +
      theme_bw(base_size = 9) +
      theme(panel.grid.major.y = element_blank(), panel.grid.minor = element_blank(),
            axis.text.y = element_text(size = 6.4, family = 'mono'),
            # ★ ソース名を -90° 回転させる（angle = 0 だと右の灰色帯が
            #   ラベルの長さぶん太くなって地図を圧迫する）
            strip.text.y.right = element_text(angle = -90, hjust = 0.5, size = 8.5, face = 'bold'),
            strip.text.x = element_text(size = 9, face = 'bold'),
            strip.background = element_rect(fill = 'grey92', colour = 'grey60'),
            legend.position = 'bottom', legend.box = 'horizontal',
            legend.key.size = unit(0.8, 'lines')) +
      guides(fill = guide_legend(order=1), alpha = guide_legend(order=2),
             colour = guide_legend(order=3, nrow=1, override.aes=list(size=3.4))) +
      labs(x = "Cliff's delta = 2 x (AUC - 0.5)    (negative = down in KO, positive = up in KO)", y = NULL,
           title = sprintf('Figure %s:  %s   (%d terms)', f, FIGT[[f]], nr),
           subtitle = paste0('large dot = term-level delta (filled = significant in this test)',
                             '   |   ticks below = per-strain delta\n',
                             'orange row (*) = significant in one panel only',
                             '   |   [!] = significant in this test but NOT by the rank test on delta'))
  H = max(4.2, nr * 0.21 + 2.6)
  suppressWarnings(ggsave(file.path(dir_p2, sprintf('panel2_static_%s.png', f)), p,
                          width = 15, height = H, dpi = 110, limitsize = FALSE))
  cat(sprintf("  図15-2-%s（%s）: %d 行 → 15 x %.1f inch\n", f, FIGT[[f]], nr, H))
  plots[[f]] = p
}

# ★ 図はセルの出力としても出す。1セル = 4図になるが、どれも同じ形なので
#   セル12 のように分けなくてよい（見出しは図のタイトルに入っている）
for (f in names(FIGT)) {
    options(repr.plot.width = 15,
            repr.plot.height = max(4.2, sum(dd$fig == f) / 2 * 0.21 + 2.6))
    print(plots[[f]])
}


  図15-2-A（ORA  -  KEGG pathway + module）: 36 行 → 15 x 10.2 inch
  図15-2-B（ORA  -  COG + PFAM + TCDB）: 43 行 → 15 x 11.6 inch
  図15-2-C（GSEA  -  KEGG pathway + module）: 55 行 → 15 x 14.1 inch
  図15-2-D（GSEA  -  COG + PFAM + TCDB）: 38 行 → 15 x 10.6 inch


plot without title

plot without title

plot without title

plot without title

### 考察: 「片方のパネルでのみ有意」が 172 行中 74 行

#### 1. (A)/(B) が食い違う行が 43%（橙の行）

**172 行中 74 行**で左右の結論が違う。内訳は ORA 側 24 / 93、GSEA 側 50 / 93 で、
**GSEA のほうが (A)/(B) の差が出やすい。**
橙の行は**菌量の増減（`Bun_8492` WT 54.8% → KO 65.1%）が効いている可能性がある**行で、
図A では `ko03008`・`ko03010`・`ko00970`（翻訳系の上昇）がここに入る。
→ **(A) の上昇を「その菌が発現を上げた」と書いてはいけない**（CLAUDE.md §7 ステージ5）。

#### 2. ★ 検定とスコアが食い違う行が 16（`[!]`）

その図の検定では有意なのに**順位検定（δ）では有意でない**行。最も極端なのが
**`COG C: Energy production`（ORA padj 5.1e-06 / δ −0.040）**で、
株ごとの δ を見ると **`Bun_8492` だけ +0.11 で逆向き**だから打ち消されている。
**ORA は「閾値を超えた遺伝子が偶然より多いか」しか見ない**ので、
**カテゴリ全体としてはほとんど偏っていなくても有意になりうる。**

#### 3. 株ごとの δ で「向きの混在」が1行で読める

| term | 全体 δ | Bth_VPI | Bth_St8 | Bun_8492 | Odo_St23 |
|---|---|---|---|---|---|
| **`ko00190_Oxidative phosphorylation`** | **−0.104** | **+0.133** | **+0.155** | −0.182 | **−0.359** |
| `ko03010_Ribosome` | +0.411 | **−0.11**(n=3) | +0.31(n=3) | +0.48 | +0.44 |
| `ko00030_Pentose phosphate pathway` | −0.322 | −0.34 | −0.44 | −0.28 | −0.48 |

**`ko00190` は全体が弱いのに Bth 2株だけ逆向き** — セル11 で突き止めた所見が図の上で直接見える。
**`ko00030` は4株とも −0.28〜−0.48 に固まり**、一貫した低下だと目盛りの密集で分かる。
→ **目盛りが散る行 = 株で向きが違う / 固まる行 = 4株共通**という読み方ができる。

#### 4. module は pathway より分離が良い（図A・図C）

`ko00190`（δ −0.104）に対し **`M00157` F-type ATPase は −0.820**、
`ko00020`（−0.147）に対し **`M00010` 最初の酸化段階は −0.947**。
**同じ現象が module では強い δ として出る**ので、pathway の弱い δ は
「経路が動いていない」ではなく「経路の中で混ざっている」と読む。

#### 5. この図の限界

- **172 行には重複が含まれる**（`Sod_Fe_N` ＝ `Sod_Fe_C`、`M00157` ⊂ `ko00190`）。
  **行数を所見の数として数えない**
- **図D に TCDB の行は無い**（GSEA 有意 0 本）
- **オペロンによる遺伝子間の非独立性は未補正**なので p 値は出やすめに出る
- 株ごとの δ は **4株そろうのが 87 / 172 行（51%）**。残りは株あたり 2 本未満の株がある


# ステージ6 のまとめ

## 出したもの

| 成果物 | 場所 |
|---|---|
| **2パネル図（日本語版・英語版）** | `panel2.html` / `panel2_en.html`（project 直下。4図をタブで切替、ホバーで実体・該当遺伝子・担い手） |
| 2パネル図の静的版 | `pathway/panel2/panel2_static_{A,B,C,D}.png` |
| 数値 | `pathway/panel2/panel2_scores.csv`（344 行）/ `panel2_strain_delta.csv`（1,376 行） |
| ORA / GSEA の全結果 | `refqc/stage6_ora_all.csv` / `refqc/stage6_gsea_all.csv` |
| pathview 74 枚 ＋ 索引 | `pathway/pathview/` / `refqc/stage6_pathview_index.csv` |
| 遺伝子ごとの棒グラフ | `pathway/genebar/`（KEGG 14 枚 ＋ PFAM/COG 5 枚） |
| ゲノム位置マップ | `pathway/genomepos/`（14 枚） |

**5ソース**（KEGG pathway / KEGG module / COG / PFAM / TCDB）× **ORA / GSEA** × **(A)/(B)**。
有意 term の和集合は **122**（ORA 79 / GSEA 93、両方 50）。

## ★ 結論

**KO で下がる遺伝子は機能的にまとまり、上がる遺伝子はばらばら。**

| 向き | 中身 | 担い手 |
|---|---|---|
| **down**（4株共通） | **F型 ATPase**（18本すべて負）／**PPP 酸化相**（`zwf`・`gnd`・`pgl`）／**酸化ストレス応答**（`sodB`・`katA`・`msrA/B`・thioredoxin）／**Fe-S クラスター**（`sufB/D`）／**重金属排出**（`copA`・`actP`）／**胆汁酸脱共役**（BSH ×4・`7α-HSDH`）／**外膜・表層**（`OmpA` 群） | 遺伝子数では `Odo_St23`、**転写量では `Bun_8492`** |
| **up** | 翻訳・リボソーム（`tuf`・`fusA`）／ヌクレオチド・補酵素・脂質（伸長段階 `fabF/H/Z`） | **`Bun_8492` 単独**（転写量シェア 93〜98%） |

**典型的な「増殖に振った」パターン**で、エネルギー・抗酸化が下がって翻訳が上がる。

## ★ 書くときに外してはいけない点

1. **KEGG のラベルを鵜呑みにしない。**`ko00195 Photosynthesis` = F0F1 ATP 合成酵素、
   `ko04146 Peroxisome` / `ko04068 FoxO` = `sodB` 1遺伝子、`ko04016 MAPK - plant` = `copA`、
   `ko00120/00121 胆汁酸` = BSH。**最終図では実体で書く**（2パネル図はラベルを原文のままにし、
   実体をホバーに出す形にした）
2. **有意 term の本数を所見の数として数えない。**122 行の多くが重複
   （`Sod_Fe_N` ＝ `Sod_Fe_C`、`M00157` ⊂ `ko00190`、`ko04013` ＝ `ko04068` ＝ `ko04146`）
3. **F型ATPase の「向き」は決まらない。**可逆機械で、向きは熱力学が決める（転写量ではない）
4. **(A) の上昇を「その菌が発現を上げた」と書かない。**`Bun_8492` の上昇の約半分は菌量の増加
5. **担い手は遺伝子数ではなく転写量で順位づける。**遺伝子数だと 104 term 中 50 本で1位が入れ替わる
6. **1遺伝子が term を作っていることがある。**50% 以上を1遺伝子が占める term が 37 / 104

## 未決として残すもの

| # | 内容 |
|---|---|
| 1 | **CPS 座位の切り替わり**（CLAUDE.md §8）。`M00922`/`M01011`（`Bth_St8` 単独で上昇）と `M01029`/`M01030`/`M01031`（`Bth_VPI`・`Odo_St23` で低下）として名前が付いた。**座位の定義はステージ7（ゲノム座位の検出）で決める** |
| 2 | **`Bun_8492` の PUL 上昇**（`TC 8.A.46` SusD ×1.47 / `TC 1.B.14` SusC ×1.81）。TCDB では遺伝子が疎すぎるので**座位単位の検出が要る** |
| 3 | **銅排出ポンプ（`copA`/`actP`）の一斉低下**。4株すべてで極めて高発現の遺伝子が下がる。**新しい所見だが機構は不明** |
| 4 | **外膜タンパクは減るのに BAM（×2.42）と Lpt（×1.72）は増える。**`eptA` は lipid A 修飾酵素なので `Lpt` との関係は注目に値するが、**本データから因果は言えない** |
| 5 | **GO が使えない**（付与率 4.5%）。`--go_evidence all` で emapper を再実行すれば数分で得られる。**実行するかは未決** |
| 6 | **COG 棒グラフは作らない**と決定（遺伝子数が 400〜940 で図の高さが 80〜170 inch になるため） |
